# PHOEBE Binary Fitting Utility

Use a built-in sample or upload a CSV, adjust the geometry-specific controls, inspect the trained live preview, then explicitly run native PHOEBE compute or a bounded fit. Supported geometries are detached, semi-detached with either Roche-lobe donor, and contact.

**Scientific contract**

- Trained previews are display-only and run in the notebook kernel. Colab/Jupyter widget changes therefore make a kernel round trip; this is not the browser-local JavaScript panel from the standalone app.
- Only native PHOEBE 2.5.4 results populate final residual statistics or exports. The final native curve always uses 513 phases and 1,500 triangles; the separately displayed fit phase budget applies only to optimizer trials.
- Semi-detached/contact modes build and cache a native PHOEBE phase-zero mesh. Play, speed, phase, and Earth/overhead controls rotate that existing co-rotating mesh; constrained shape changes rebuild it.
- Photometry alone may not uniquely identify mass ratio, especially for partial/grazing eclipses. This utility does not perform joint radial-velocity fitting, uncertainty estimation, disks, streams, or spots.
- Input CSV headers: `phase,flux[,sigma]`, `time,flux[,sigma]`, or magnitude equivalents. Time input uses the period and epoch controls.

Run the cells in order. Re-running the final launch cell safely closes the previous callbacks before registering a new interface.

In [ ]:
# Colab installs into its free CPU runtime; local Jupyter reuses the selected environment.
import importlib.util, subprocess, sys
required = {"phoebe":"phoebe==2.5.4", "ipywidgets":"ipywidgets>=8,<9", "scipy":"scipy", "matplotlib":"matplotlib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


In [ ]:
import json
import numpy as np
import phoebe, ipywidgets
print("PHOEBE", phoebe.__version__, "· ipywidgets", ipywidgets.__version__)


In [ ]:
DETACHED_MODEL=json.loads(r'''{"kind":"discs","table":{"teffs":[3653.9999999999995,3750.0,3875.0,4000.0,4125.0,4250.0,4375.0,4500.0,4625.0,4750.0,4875.0,5000.0,5125.0,5250.0,5375.0,5500.0,5625.0,5750.0,5875.0,6000.0,6125.0,6250.0,6300.0,6375.0,6500.0,6625.0,6750.0,6875.0,6930.000000000001],"gravities":[3.5,3.75,4.0,4.25,4.5,4.75,5.0],"radii":[0.0,0.0980171403295606,0.19509032201612825,0.29028467725446233,0.3826834323650898,0.47139673682599764,0.5555702330196022,0.6343932841636455,0.7071067811865475,0.773010453362737,0.8314696123025452,0.8819212643483549,0.9238795325112867,0.9569403357322089,0.9807852804032304,0.9951847266721968,1.0],"base_logg":[2.7138637552657117,2.5889250186574118],"primary_teff":6300.0,"sma_solar":7.974299795787102,"log_intensity_f32le":"W29bwGfuW8BE61zAnmdewOpnYMAk82LAYAxmwK/AacCfGW7A4iFzwMz8eMAxxH/A+daDwFKViMA/tI7AF3iXwP3AW8A+NFzAoBldwN1yXsBhRGDAS5ViwBxpZcBgzWjAMs1swFB2ccBd73bANFZ9wBJ3gsBvHofANESNwK48lsCfElzAFHpcwP1HXcAcfl7A2CBgwHI3YsDZxWTAEdpnwMaAa8C+ym/A7uF0wDjoesAsF4HAi6eFwCnUi8BEAZXAfVJdwEKtXcDqYV7AUXJfwPHiYMAmu2LAOf9kwCK9Z8AUAmvAnuBuwMSDc8BeEnnAQuF/wF9LhMDUVorAeZOTwFuSXsBw4F7A2HtfwIdmYMAJpWHA2T5jwJk4ZcAzoGfAYoNqwH32bcCbJXLAhTx3wCyUfcAz74LAftmIwK8lksBz7F/ARjNgwDzAYMBHlWHA47ZiwIYsZMAH+2XAPzBowG7ZasARCm7AIu5xwKKxdsCRrnzAO0qCwLL/h8DfJZHAjEZhwB2GYcCgBGLACMRiwL3IY8AzGmXAdL1mwErAaMB6L2vApR1uwKm2ccDAJnbA98h7wEKlgcDlJYfAECaQwPRwU8Ca91PAmgNVwLuWVsC9tVjAEmhbwAKxXsDznWLAozhnwJCKbMCAsnLAfMp5wIMDgcBb34XAJQGMwD2HlMC9w1PAgz9UwNs1VcCNqFbANJxYwDIYW8AAIF7AbsFhwDsHZsBt/GrAtsNwwMp7d8AOaH/ALXuEwH6nisCKZJPAhhZUwGuHVMAdaFXAXrpWwKuCWMBSyFrA/45dwOnkYMDU1WTASW5pwOzUbsAXLXXAF8l8wP8Wg8DXTYnA2EGSwM5IVcBOq1XAWm9WwMWWV8AFJlnAbiRbwCiWXcB9iWDA+wpkwGoqaMCwEW3A5+dywLb/ecDHe4HAqJqHwJK4kMAWe1bAMc9WwJh2V8Arc1jAX8lZwImAW8BRnV3AES5gwCJAY8CL5mbAdU5rwLeicMBVNnfAHcF/wHnnhcBNL4/Ax5BXwPfbV8CdcVjAp1NZwI+GWsDLEVzAL/pdwM9NYMD5GWPAWHFmwCWAasAvc2/A8p51wP65fcCmrITAKteNwHimWMC96FjAomxZwCM0WsC/Q1vADqNcwA1XXsCNbWDA0PNiwCX8ZcDUsWnAqENuwI4HdMDesnvA0nGDwAd/jMCEwEfAuUtIwMhgScDcAUvAeDRNwAIBUMBka1PAcIJXwHlQXMBh4GHAclNowJLBb8ClW3jAyDiBwHdvh8DhqY/AzBpIwDCeSMCyo0nAbi1LwLdATcDP5U/AsyBTwEUAV8Bdj1vAs9lgwPMCZ8D1Jm7Amn92wEo3gMC3cobAydSOwBR1SMCo8EjAnOZJwP9YS8D2TE3Am8pPwALWUsAZflbAQc5awAXTX8BzsmXAWIxswI+jdMCZa37A93WFwLH/jcBKpknAdRRKwLrvSsAUOkzAZ/hNwIYyUMCg7FLAUzVWwOIYWsAipl7AFAZkwH1casAH83HAhFR7wLDdg8DcqYzAgddKwEM4S8DY+EvAKBtNwNmjTsBxmlDAPQNTwIzsVcCEY1nAPnldwLVZYsCiLGjAgEJvwG49eMBqRYLABlSLwGoPTMDdZEzA7g5NwI8PTsBVa0/AtylRwE5PU8B16lXAVQhZwEy6XMCHLGHA9IlmwIMlbcBZp3XAlMeAwGvZicBSR03AdpFNwAUlTsD2A0/A0TJQwP24UcBfm1PAX+hVwCetWMBa+1vAWf9fwEbnZMCGCGvAQxFzwHyTfsDRXojAFRA8wNifPMD3vT3A/mw/wDOzQcDymUTAxyVIwO1mTMBPaFHAMTZXwGX0XcCouGXARbBuwGgkecDK3YLAhsyKwNtxPMDe/DzAihE+wE+yP8A65UHAa7NEwGYhSMAbP0zAfhdRwPm2VsAwQl3AINJkwCWXbcDP5nfA8D2CwAhFisCh0zzA5Fk9wBxlPsCg9z/AQBdCwOPMRMAGHUjAShdMwK7GUMDBN1bA+o9cwJjrY8AGfmzANal2wBeegcCKvYnAxwM+wJ19PsAacD/AY91AwMrKQsCeQEXAGENIwCjhS8DKJlDA2SFVwHf6WsAS0WHAWeZpwHqxc8C4IIDAJZuIwO0zP8BWoT/AF3tAwCXDQcBTfkPAWbRFwClpSMAHq0vA5oZPwPELVMD1ZFnAjbZfwKtOZ8C/uXDAtEZ9wL94h8ANjkDAwu1AwECsQcB3y0LAG1BEwKJBRsBtpEjAG4dLwLH2TsBAA1PA6dhXwLigXcAUrGTAtJRtwAPFecCt24XALehBwC86QsBo3ULAytNDwOMhRcDrzkbAsd9IwDBjS8B9Zk7Aj/pRwN1MVsDjilvAfQliwKhvasBTQ3bAmz6EwBUVL8CVoy/AM78wwLhqMsDmrDTAu483wF4YO8CjWD/AzVxEwJsySsDiAVHAUuNYwE4HYsD+u2zA15x5wF2uhMAXpC/Aoi8wwF9FMcAI6DLAPR41wNLxN8DoZzvAIJI/wKx8RMAUNUrAE+NQwJmgWMAjn2HAijJswNAAecDNYoTAGDMwwLC7MMCLyzHAWGUzwJSPNcDqUzjAc7c7wJ3LP8CMnETAjTdKwETEUMDgXVjA+DZhwBWpa8DJZHjAPheEwIpSMcCl0jHAndEywPNRNMC8WTbAJvI4wBEgPMBB9D/AOntEwB3DScAA9U/AMC9XwA+tX8Bg12nAbYp2wMpjg8D8cTLAmukywK7XM8CNPjXA5CM3wGKQOcCuiDzA5RxAwOhZRMCtTknAvCVPwH8AVsAnI17AqwVowBCwdMBWsILAz/IzwMVdNMCoMjXAn3M2wOIlOMCWUTrAZ/s8wNoyQMDIBETA4YBIwI/STcDLH1TAz7JbwHgUZcA8dnHAtkyBwKNzNcDw0TXAoY02wLGoN8DhJznAyxI7wCFuPcDPSEDAqK9DwBazR8Bif0zAFz9SwHhCWcBGI2LAaDxuwCzSf8AWGiLAU6ciwG/AI8ByaCXAmaYnwISFKsD1Ci7AWEoywEtRN8AELz3AXg9EwPwNTMBXXlXAlVNgwBt+bcBoIH3AUtYiwGdiI8A1eSTAwR0mwEBXKMA6MCvAa64uwCTlMsDa4TfAL7M9wPaDRMASb0zAIKdVwER+YMDAhW3AJgF9wI+SI8B7HSTA+zElwBDTJsDoBynA8NorwOBRL8DxfzPAanI4wFo3PsCN+ETAKNBMwOrvVcD0qGDAZI1twOXhfMBMoSTArSclwCAzJsCDxifAr+gpwK2jLMAJ/S/AWQc0wKrPOMBhZD7Aie9EwE2NTMDGc1XARf1fwGjTbMDgWHzACrAlwN4xJsBFNCfA9bkowHbJKsBqbC3AMqgwwMKONMDrLDnAaZE+wITmRMBySkzAovdUwJZRX8BtGWzA2897wJJXJ8DIzSfAD7kowMcbKsCq+yvAimEuwGFSMcCY3jTA3hI5wIP+PcA2zEPA3p5KwIq5UsA9lFzAdSdpwH97ecAZ/yjAsmkpwNo9KsCZfSvA3y0twKtWL8CQ/DHAbi41wNL4OMCeaz3A6LFCwErzSMBze1DA5NZZwH01ZsAjJ3fAQ0UVwDjOFcDN3hbAHHoYwHmnGsCdcR3Acd4gwPMBJcCz6inAg6gvwE5pNsC/Sz7A2olHwGqBUsD13l/AehdwwIUHFsCljxbAk54XwGo3GcCAYRvAhCcewF+PIcAHrSXA6I4qwEZEMMAD+jbAn80+wFz3R8B30lLAXwhgwPYKcMDHyRbAElEXwFheGMC39BnAhxscwGvdHsBOQCLAGlgmwB0zK8AK4DDAuIo3wH5PP8DeZEjAhSNTwMkxYMBx/m/A18EXwGRGGMA2ThnAWNwawPf3HMCLqx/A9PwiwM7/JsAvwivAyVIxwG7dN8DRfj/AdW5IwCoIU8BD/F/AosdvwOe5GMC3OxnAFD4awPjDG8Bo1B3AqnkgwJm5I8CCpyfAQFEswIfFMcAlMDjAI64/wAt4SMDP7FLAvcZfwNOQb8AzJBrAjp4awBKSG8CAAR3AV/MewElxIcDmgCTAfDQowMaZLMDovzHACdM3wJTzPsAeYEfAb4VRwKI/XsAhXW7AgI4bwGQBHMAQ5hzABz8ewEYSIMDoaCLANEglwHbBKMBM4izASLoxwO51N8AFOT7AMUhGwA8eUMCIuFzAbyltwHFwCMAd9QjAK/0JwMaLC8BaqA3Atl0QwO2xE8COuRfAHIQcwAEiIsA+wyjAgokwwF61OcA/r0TAzj9SwI0OY8C4OAnA47wJwPDDCsASUQzAwGsOwM4eEcBUcBTA6XQYwPY7HcBe1SLAEHApwCssMcCYRzrAqyZFwP6KUsDFFGPA/gAKwKmECsC2igvAXhYNwCYvD8Dm3xHAvC4VwEMwGcDR8x3Au4gjwOMcKsDUzjHA0tk6wBeeRcAt1lLA/RpjwGHiCsAcZQvATWkMwC3yDcBABxDAaLMSwN78FcBC+BnAs7QewDBBJMBUyyrAVXAywCNpO8APE0bAHSVTwGQ2Y8DEwwvAj0UMwORHDcD7zQ7AWt8QwOqGE8D/yhbAQcAawJV1H8Cm+STAxnkrwNURM8B0+DvACIhGwA10U8DLUWPA1fAMwFNvDcAUaw7AOOcPwATrEcAIgRTAbK8XwF+KG8CuICDATYElwN3ZK8BLSDPAsgY8wKF2RsDQV1PAxD5jwOYdDsAXmQ7ARY4PwHUAEcCu9hLAJXsVwNiTGMB9VBzAxssgwPMIJsD1OSzAwH4zwPAUPMA5ZUbAkztTwLwrY8Dtc/q/HHT7v/9x/b/xOQDAPkQCwKHiBMACGwjAmQEMwPylEMBnGBbAQYkcwCQbJMC6EC3Aydc3wFhKRcBmR1bA6+z7v0Hs/L9y6P6/8PMAwLD8AsBAmQXAoc8IwDe0DMCfVhHAtcYWwBg0HcAJwCTArqotwFVdOMDrqEXA3FxWwOpl/b9mZP6/cy8AwO6tAcAitQPA4E8GwECECcDUZg3AQgcSwAJ1F8Du3h3A72QlwKNELsDh4jjAfgdGwFFyVsClAf+/oP7/vxH7AMBSdwLAlXsEwLsSB8AEQwrALCEOwL+8EsDrJBjAP4gewGEFJsBd1y7AdV85wLtfRsCCkVbAsE4AwG3MAMCvxgHAtkADwAlCBcCV1QfAyAELwITbDsA8chPA1NQYwJAxH8DSpSbAF2ovwAncOcD5t0bAs7BWwExSAcD3zQHAGcQCwOY3BMDGMAbAgLkIwH7YC8Byog/AvyYUwF10GcCjuR/AjhQnwG2+L8BlFjrAtNtGwGe+VsDoVQLAgM8CwILBA8AVLwXAgh8HwGudCcAzrwzAYGkQwELbFMDlExrAt0EgwEuDJ8DEEjDAwVA6wHD/RsAazFbA+QbkvwD+5L+o6ea/OdDpv0TA7b8Wz/K/Lwj5v6RJAMDdxwTAzQ4KwEVPEMDFrBfAFmwgwFQAK8DiVDjAQIBJwGho5b+9Xua/A0nov5ot679AG++/ZSf0v9td+r+E8wDAR3EFwAu4CsAf+BDA6FMYwMQNIcAAlCvA2cY4wPOkScDXyea/er/nv16o6b/7iuy/PHbwv7R/9b+Hs/u/ZZ0BwLIaBsBJYQvA+aARwAv7GMBzryHAqycswM84OcCmyUnAhz7ovwgz6b+pGeu/7vjtv9bf8b8a5Pa/VBL9vxZKAsDLxAbApggMwCpFEsBtmhnAlkUiwNurLMBamjnAoOxJwDiz6b+Wpuq/9Irsv+Fm779vSfO/gEj4vyJx/r/H9gLA424HwAOwDMBa6RLAzjkawLnbIsALMC3A5Ps5wJsPSsCGZ+u/NFnsvzo67r8mEfG/De30v/Dj+b+PAQDAhboDwNAsCMBtZw3AapkTwNLgGsApdiPAKrYtwJhfOsAJPkrA0xvtv9ML7r9/6e+/a7vyv6yQ9r9gf/u/jsoAwEN+BMC96gjA1x4OwHlJFMDVhxvAmBAkwEk8LsBMwzrAeGxKwO4T0b/mAtK/g97Tv/+s1r+6fNq/x2Lfv1hq5b/5uey/t3D1v/+t/79o6AXAFg8NwEKOFcBi2B/AZ9oswMy5PcD5R9K/ITbTvyMQ1b9P3Ne/NKnbvweM4L9ikOa/yd3tv7mT9r++aADA+XoGwEmiDcAcIBbAT2MgwJZPLcDr8z3AA3zTv11p1L/DQda/nwvZv63V3L9IteG/bbbnv5gB77+8tve/fPoAwIoNB8B8NQ7A97EWwD3uIMDExC3ACi4+wFfI1L+TtNW/v4rXv1RR2r8mF96/xvHiv8nt6L96NPC/9uX4v5OQAcCHogfA+cgOwENBF8DRciHAgjAuwDllPsCsFNa/yf/Wv7vT2L8Il9u/n1jfv0Qu5L8mJeq/XGfxvzAV+r+rJgLAhDcIwHVcD8CP0BfAZvchwECcLsBonD7AspjXv0iC2L8vU9q/8xHdv4vN4L/0m+W/xIrrv3TE8r9Hafu/78sCwObXCMAl9w/AGGMYwB17IsCdAy/A69E+wLgc2b/GBNq/otLbv92M3r94QuK/pAnnv2Lw7L+MIfS/Xr38vzNxA8BHeAnA1ZEQwKD1GMDU/iLA+2ovwG4HP8DkIL6/zAe/v1/TwL/FicO/LznHv3f2y7+AzNG/quDYv7RR4b9jPuu/FwP3v2hxAsBusArAcLAUwOxfIcBX8zHAiie/v4YNwL9D18G/BIvEvyc3yL+q8My/6sLSv4nU2b/kROK/4TLsv6b797+r8ALAdTILwJ8yFcBT2CHA4kIywC8uwL9AE8G/KNvCv0OMxb8eNcm/3OrNv1O5079nyNq/Ezjjv18n7b829Pi/7m8DwHu0C8DOtBXAulAiwG2SMsAoUsG/HjbCv9X7w7+5qca/dk7Kv3L/zr8+ydS/ydTbv1dC5L8CMe6/yP/5v4X3A8DwPAzAxzkWwKvGIsDS3TLAIHbCv/xYw7+CHMW/L8fHv89ny78JFNC/KtnVvyrh3L+aTOW/pTrvv1sL+78cfwTAZcUMwMG+FsCbPCPANikzwN7Jw79bq8S/JGzGv78Syb8Jrsy/+VPRv2kS17/eE96/7njmv+Bg8L/DLPy/eA0FwAZQDcAQQBfAo6cjwM1lM8CdHcW/uf3Fv8W7x79QXsq/RPTNv+mT0r+oS9i/kkbfv0Gl578ch/G/K079v9WbBcCo2g3AX8EXwKoSJMBkojPAtdOtvxO0rr+icrC/gBWzv7iqtr8JR7u/RPXAv8XZx7/1EtC/Ab/Zv4E25b8auvK/E2QBwIodC8AnbhfABoQnwPW1rr8ola+/YlGxv9fws7+8gbe/4Bi8v3vBwb9Loci/ddfQv1mC2r/F++W/B4TzvxPMAcCYhwvA79MXwLLPJ8A2mK+/PHawvyEwsr8vzLS/v1i4v7fqvL+zjcK/0WjJv/Sb0b+yRdu/CsHmv/RN9L8UNALAp/ELwLg5GMBdGyjAD5Wwv7xxsb/yKLO/CsG1v3pIub9i1L2/L3HDvy1Hyr+GdtK/DB7cv6Oa57/ZK/W/paUCwHFjDMD1oxjA82kowOeRsb88bbK/wyG0v+a1tr81OLq/Db6+v6xUxL+IJcu/GFHTv2X23L88dOi/vgn2vzUXA8A81QzAMg4ZwIm4KMDHuLK/jZKzv/tDtb+E07e/xU+7v3rOv7+OXcW/JyfMv8RL1L/k6t2/N2Xpvz359r96jgPA5EgNwMd0GcAo/ijApt+zv963tL80Zra/IvG4v1VnvL/n3sC/b2bGv8Uozb9wRtW/Y9/evzNW6r+76Pe/vwUEwIy8DcBc2xnAxkMpwIaGnb9aYJ6/5RGgvzuhor9BHKa/nJeqvwcesL/g0ra/NtS+v54/yL/radO/ZpHgv3Av8L+kigHAYXwNwLUUHcBhRJ6/yRyfv4DLoL+rVqO/UcymvxdBq78NwLC/Dm63vwZqv7/S0ci/5PvTv7km4b9ky/C/kdwBwIvPDcCBXB3APAKfvzjZn78ahaG/Ggykv2B8p7+R6qu/E2KxvzsJuL/W/7+/BmTJv96N1L8NvOG/WWfxv38uAsC1Ig7ATKQdwPbXn79ZraC/D1aiv1zYpL9+Qqi/UqmsvyAZsr+Qubi/tarAvxULyr9+NdW/qGjiv7Ic8r8bjQLAP4EOwBT2HcCvraC/e4GhvwQno7+dpKW/mwipvxJorb8u0LK/5Wm5v5VVwb8lssq/Hd3Vv0QV478L0vK/t+sCwMnfDsDcRx7Ar6ehv755or/TG6S/SZSmv4Hxqb/8SK6/sqizv286ur+aHsK/53TLv6yd1r+I1+O/3Znzv7hRA8DsQQ/AgpYewK+hor8CcqO/oxClv/WDp79m2qq/5SmvvzeBtL/4Cru/n+fCv6o3zL86Xte/zZnkv69h9L+5twPADqQPwCjlHsAQFI+/geiPv1OPkb+aDpS/LnSXv77Um78mO6G/Msqnv6yfr7+V2bi/+MnDvyqs0L9e9N+/sGzyv2XbBMCg9xPAfsePv02akL/gPZK/V7iUv6UXmL9icJy/ps2hv7RTqL9GIbC/b1S5v0dBxL/bI9G/ZXDgv1jv8r+GHgXACzMUwO16kL8aTJG/bOySvxNilb8bu5i/Bwydvydgor813ai/4KKwv0nPub+WuMS/jJvRv2vs4L8BcvO/p2EFwHZuFMAePpG/cw2Svxqqk79KGpa/FWyZvya0nb+t/qK/uHKpvykwsb90Vbq/3zvFv6sg0r8deOG/fQX0vwitBcAksxTAUAGSv83Okr/JZ5S/gNKWvw8dmr9FXJ6/M52jvzsIqr9zvbG/ntu6vye/xb/KpdK/zwPiv/mY9L9q+AXA0/cUwJTckr8OqJO/CT2Vv8Whl79j5Jq/HRqfv9pQpL/Ksaq/C12yv3Byu7+fUca/jjjTv9ac4r9uOvW/yUkGwK0+FcDXt5O/ToGUv0gSlr8KcZi/tqubv/XXn7+ABKW/WVurv6P8sr9CCby/FuTGv1LL07/eNeO/5Nv1vyibBsCIhRXAmaGAv6dwgb/BDIO/+XuFvxzMiL/gEY2/RViSv4TBmL8ha6C/jHOpvwUqtL/uxsC/TbnPvxnE4b/UdPi/i9oKwJtKgb/RF4K/P7CDvwMahr/4Yom/rp+Nv0Dbkr9ZOZm/htigvwzXqb+qhrS//SDBv2UV0L+OJeK/Atv4v5UJC8Cd84G/+76Cv75ThL8NuIa/1fmJv3wtjr86XpO/L7GZv+tFob+NOqq/T+O0vwt7wb99cdC/AofivzFB+b+fOAvAR6SCv41tg78m/oS/OFyHv6yVir/6vo6/OeSTv+Armr+dtaG/0p+qv0BCtb+u2MG/h9PQv8Pw4r+jsfm/NHALwPFUg78gHIS/jqiFv2MAiL+DMYu/d1CPvzhqlL+Rppq/UCWivxcFq78xobW/UTbCv5I10b+FWuO/FSL6v8mnC8B4EYS/XdaEvz5ehr9Br4i/RdeLvz7rj78B+ZS/JSmbv3ybor/5b6u/kgW2v5SZwr/Qn9G/bdHjv0yj+r/Y5gvA/82Ev5qQhb/uE4e/IF6JvwZ9jL8FhpC/yoeVv7mrm7+oEaO/29qrv/Jptr/Y/MK/DQrSv1VI5L+CJPu/5yUMwIj5Zr8jjmi/brNrv6N1cL928Ha/l01/v8PRhL9UG4u/yaCSv1GBm7/dCqa/CHSyv8kmwb9p3tK/0h3pv3bZAsBNV2i/Duhpv6wFbb9xvHG/HCh4v/c4gL81WIW/RJWLv5UOk7/34pu/m2Kmv/rFsr/Zd8G/ujLTv4B26b86AwPAEbVpv/hBa7/qV26/PgNzv8Ffeb8iy4C/qN6FvzQPjL9gfJO/nEScv1q6pr/tF7O/6cjBvwqH078vz+m//iwDwA8Ua7+InGy/qalvv75HdL9yknq/SVmBvx9ghr/Ig4y/6OOTv++enL8JC6e/w2Ozv9sVwr+02dO/7yjqvwdbA8ANc2y/F/dtv2j7cL89jHW/I8V7v2/ngb+W4Ya/XPiMv29LlL9B+Zy/uVunv5mvs7/MYsK/XyzUv6+C6r8RiQPAR9Ztv01Vb7+jT3K/etF2vzv2fL+9c4K/TWCHv9Zojb/vrJS/fUydv1+lp7/99LO/A63Cv4CD1L8W5+q/cLwDwIE5b7+Ds3C/3qNzv7YWeL9UJ36/CwCDvwTfh79P2Y2/bw6Vv7ifnb8G76e/YTq0vzv3wr+h2tS/fUvrv8/vA8Ddr0y/+DpOv1lNUb9U81W/tEhcv253ZL+Blm6/R+p6v3HWhL8Xj42/tOuXvyEhpL9FlLK/uvjDv9DG2b/CsPW/YxlOv3mgT7/ZqlK/3ERXv0aKXb9/pGW/Vqpvv1zie7+kRIW/4e6Nv4w+mL/4aqS/Tdqyv+Y/xL/+Edq/vfn1v+mCT7/5BVG/WghUv2SWWL/Zy16/kdFmvyy+cL9x2ny/1rKFv6tOjr9kkZi/zrSkv1Ygs78Sh8S/LF3av7hC9r+Q31C/9F1SvwdXVb8M11m/jPlfvzDnZ78LuHG/YLd9vzMShr8Mno6/09OYv9jupL8uWLO/pcLEvzqg2r+0i/a/ODxSv++1U7+0pVa/tBdbv0AnYb/P/Gi/6bFyv06Ufr+PcYa/bO2Ov0EWmb/iKKW/BpCzvzn+xL9I49q/sNT2v56JU7/h/VS/yuJXv3BEXL/uPWK/evhpvzOPc78MUX+/Y76GvwEpj78tRZm/Z1Clvze6s7+TNcW/4Crbvw8k978E11S/0kVWv+AfWb8tcV2/m1RjvyT0ar99bHS/5QaAvzcLh7+WZI+/GXSZv+t3pb9o5LO/7WzFv3hy279tc/e/X140v/rgNb9v4ji/OG89v0+jQ78LqUu/CJhVv0i1Yb+5OnC/47SAvzruir+Q/Ja/ZD+lvxJitr8M0Mu/3i7nv23kNb/4Yje/V1w6v/vcPr/GAEW/ZfFMvxzGVr+XxWK/bSxxv14dgb+ORou/50eXv22Cpb/hoba/6RDMv1tv5797aje/9uQ4vz/WO7+/SkC/PF5Gv785Tr8w9Fe/5tVjvyEecr/ahYG/4Z6Lvz6Tl793xaW/sOG2v8dRzL/Yr+e/F944v65TOr89Oz2/CqFBv46gR792Yk+/4P5Yv3nAZL935nK/TteBv+Tfi7+ux5e/1POlvz4Rt7+OiMy/YvDnv7NROr9nwju/O6A+v1X3Qr/g4ki/LotQv5EJWr8Mq2W/zK5zv8Iogr/nIIy/H/yXvzAipr/NQLe/Vr/Mv+0w6L/Nozu/3Q49v3bhP7+gJ0S/PvxJvzOIUb9R5lq/o2Rmv8NBdL+nXoK/SkeMv9cXmL+aPaa/m2e3v0D3zL/fc+i/5/U8v1JbPr+xIkG/61dFv5wVS785hVK/EsNbvzoeZ7+61HS/jJSCv61tjL+QM5i/BFmmv2mOt78qL82/0Lbov+IMHL/8hh2/hncgvxvrJL/r/Sq/qdoyv5CZPL9IgEi/kMhWv1+1Z79/4Xu//9eJv4Lql79qy6i/R9m9v/ms2L93rx2/dyUfv9UNIr8bdSa/RXcsv0s+NL/i4T2/0qhJv5PPV7+3l2i/Hp18v9ckir+NKpi/3AOpv9QPvr/45Ni/DVIfv/LDIL8kpCO/Gv8nv5/wLb/toTW/NCo/v1vRSr+W1li/EHppv71Yfb+vcYq/mGqYv048qb9hRr6/9xzZv57cIL9oSSK/cx8lvwhrKb+QRy+/vd02v7ZFQL+TyUu/iKhZvyEhar/r132/haCKv3mPmL/YX6m/4nC+vxBV2b8vZyK/384jv8KaJr/11iq/gJ4wv4wZOL84YUG/ysFMv3t6Wr8xyGq/GVd+v1vPir9atJi/YYOpv2Sbvr8pjdm//L0jv9kfJb8i4Ce/zwosv466Mb/tFzm/bz1Cvzp4Tb/BBlu/myhrv86Sfr9I34q//cCYv6SZqb+fw76/rsPZv8oUJb/ScCa/giUpv6k+Lb+d1jK/TRY6v6cZQ7+qLk6/B5NbvwWJa7+Dzn6/NO+Kv6DNmL/mr6m/2+u+vzP62b9TWQW/ncsGv5msCb8NCQ6/ZP0Tv920G7+RSCW/j/8wvzoUP7+eyk+/6r9jv1FWe7+4nou/F1Kcv9cUsb8gbcu/fyUHv8GTCL+3bAu/CL0Pv92gFb+nQh2/27omv/9QMr/gQUC/R9BQv/aYZL8CBXy/hOOLv8WInL/JQrG/xZfLv6zxCL/lWwq/1SwNvwNxEb9VRBe/cdAevyYtKL9vojO/hm9Bv/DVUb8CcmW/srN8v1AojL9yv5y/vHCxv2nCy793mQq/nv4Lv3XFDr9R+hK/p7gYv24pIL8SZSm/drU0v/dZQr8CklK/gQBmv28Zfb+oTIy/Gt6cv/GSsb848cu/QkEMv1ehDb8VXhC/n4MUv/ksGr9sgiG//5wqv3zINb9pREO/FE5TvwCPZr8sf32/AHGMv8H8nL8ltbG/ByDMvwKxDb8+Cw+/WrwRv0nQFb+mYRu/1ZgivyKQK78dlDa/YuNDv1y9U79Q1Ga/UaF9v2d6jL91C52/2tKxv9VMzL/BIA+/JXUQv54aE7/zHBe/U5Ycvz6vI79Fgyy/vV83v1qCRL+jLFS/oRlnv3bDfb/Og4y/KRqdv4/wsb+iecy/h0vdvnsg4L5Zw+W+/k3uvrz5+b4SjwS/kvcNv9d+Gb/kXye/3d83v1aeS7+j/GK/3KV+v8TYj79mUKS/Ry2+vw434b4WBOS+MZfpvuwJ8r7olP2+A0cGv9STD78s+Rq/LbQov9cIOb/OlEy/VcBjv/Y4f7+uDZC/v3Wkv5FKvr+UIuW+sefnvglr7b7axfW+C5gAv/T+B78XMBG/gnMcv3YIKr/QMTq/R4tNvweEZL8RzH+/l0KQvxebpL/bZ76/oKzovqdn677s1vC+NhP5vr4pAr8gdQm/boQSv1ihHb9mCyu/5AI7vxYpTr/U8WS/1wmAv1xckL//tKS/YI2+v6027L6d5+6+zkL0vpNg/L5yuwO/S+sKv8XYE78vzx6/Vw4sv/fTO7/mxk6/oV9lv6UtgL8gdpC/586kv+Wyvr8OSO++Ru3xviIx976HK/++vQgFv7wZDL/V4hS//68fvwPALL8cUjy/0xVPvxKEZb/QM4C/Rn2QvxXipL/71b6/cFnyvu7y9L51H/q+PfsAvwlWBr8uSA2/5OwVv9CQIL+ucS2/QdA8v79kT7+CqGW/+zmAv2yEkL9D9aS/EPm+v9ozzL69As++iZnUvjYS3b4/pui+ra33vp8uBb+moxC/fnEev+rdLr8nikK/99dZv0Vzdb8ENou/cpqfv3hSub84QtC+bwnTvvaQ2L5u8uC+mWbsvsJD+76I3ga/BzISv9jZH7+jGjC/S5NDvymsWr/yEna/mW6Lv3TAn79Tbbm/l1DUviEQ175iiNy+ptLkvvIm8L7X2f6+cY4Iv2jAE78yQiG/XFcxv2+cRL9cgFu/n7J2vy6ni7915p+/L4i5v+76176RsNq+HBXgvmNB6L5sbPO+hfQAv4b0Cb8CABW/rVYiv4E5Mr/lSUW/s/tbv1EEd7/lw4u/2QCgvwGrub9Fpdu+AVHevtah474hsOu+5bH2vh58Ar+cWgu/nD8WvylrI7+nGzO/W/dFvwp3XL8DVne/nOCLvz0boL/Uzbm/NtXevll14b4Zr+a+YZruvjps+b6XugO/sXQMv0AwF79BLCS/Oqgzv+xSRr9TpVy/Emh3v/jni7/QK6C/Uey5vycF4r6xmeS+W7zpvqCE8b6PJvy+EfkEv8eODb/lIBi/Wu0kv840NL99rka/nNNcvyF6d79U74u/Yzygv84Kur9XkLK+IFa1vtHaur6LOMO+BanOvv2E3b5mAvC+3loDv+YLEb/+WiG/4es0v/UgTL9ip2e/5EGEv4SJmL9DCrK/+dK2vnWRub6cB7++MU/HviKh0r5cVOG+K53zvk8HBb9ZkhK/VrUivwYRNr/oDU2/61lov/p/hL+DsJi/dyGyv5sVu77JzL2+ZzTDvthly74/mda+vCPlvvA3977Bswa/zBgUv60PJL8rNje/2/pNv3QMab8QvoS/gdeYv6w4sr9j8L6+8J3BvmTyxr6nBs++0xHavjhn6L5WOfq+AA4Iv5hHFb9uCyW/G/s3v4CKTr9HbWm/M9+Ev5/ymL9zV7K/KsvCvhdvxb5hsMq+d6fSvmeK3b61quu+vTr9vkBoCb9kdha/LgcmvwvAOL8mGk+/Gs5pv1YAhb++DZm/OXayv/Ioxr52wci+C+zNvqjA1b5ZdOC+wVfuvvme/76icAq/n04Xv2epJr+TLjm/NVdPv77oab8DCIW/aRqZv9KNsr+5hsm+1BPMvrQn0b7Z2di+S17jvswE8b6bAQG/BHkLv9smGL+hSye/Gp05v0SUT79iA2q/sA+FvxQnmb9qpbK/JtWHvsaLir5J8o++FyOYvk5Yo77W67G+qRXEvstt2r7Sb/W+H9YKv2s5Hr9GRTW/56hQvwlWcb+jwoy/P+elv+RujL7UHo++B3iUvneUnL5brae+sxq2vq0SyL7lKt6+DOH4vtVhDL89jR+/e1s2v+B6Ub+O5HG/R+uMv174pb+iCJG+4bGTvsT9mL7XBaG+aAKsvo9Jur6xD8y+/+fhvkZS/L6L7Q2/D+Egv69xN7/YTFK/E3Nyv+sTjb9+Caa/JTSVvjnUl77cDZ2+GfqkvirQr74y5L2+0GnPvlH15L6TB/+++BMPvx/NIb8tIzi/4sZSvxXEcr9AMI2/hSGmv6hfmb6R9pu+9B2hvlvuqL7rnbO+1H7Bvu/D0r6jAui+cN4Av2U6EL8vuSK/q9Q4v+tAU78XFXO/lUyNv405pr/VCZ2+ppWfvvSmpL7JVay+ONe2vgl8xL5JeNW+iWLqvjzdAb+zABG/U0cjv1gqOb/caVO/gCVzv71Sjb+pRaa/A7Sgvrs0o77zL6i+N72vvoQQur4+ece+oizYvm/C7L4J3AK/AMcRv3bVI78FgDm/zZJTv+k1c7/lWI2/xFGmvzXPPr4QHkS+mK5Ovo21Xr7IpnS+tpuIvs5tmr70ZbC+gAPLvq7a6r7/ogi/1Yofv4PaOr+RfFu/p8GBv9Otmr/Dmki+wN1NvqlWWL57OWi+dvh9vnwijb6Zx56+0YS0voPazr4QW+6+ASwKv3HUIL9R2ju/qy5cv4D1gb8mwpq/UWZSvnCdV766/mG+ar1xvhOlg75CqZG+ZCGjvq6juL6FsdK+c9vxvgK1C78NHiK/H9o8v8XgXL9YKYK/edaavxpjW77ziWC+2MpqvhpYer410Ie+uqaVvrjkpr50ILy+QNzVvlij9L5C3wy/Bgwjv0GKPb+PW12/mVGCv3fvmr/kX2S+dnZpvveWc75meYG+V/uLvjGkmb4MqKq+Op2/vvwG2b49a/e+ggkOv//5I79kOj6/WtZdv9l5gr91CJu/HYBsvn2Bcb7Od3u+sEmFvk+fj77xDZ2+IMqtvsVqwr4acdu+EmH5vlPIDr9BeiS/yYM+vzb6Xb+6goK/ahCbv1agdL6EjHm+U6yBvvoZib5GQ5O+sHegvjXssL5QOMW+N9vdvudW+74jhw+/g/okvy/NPr8SHl6/mouCv2AYm7876Nu9KEnmvTvx+r3sJA2+9JwivimXPr7ki2G+HF6Gvi+XoL4eCcC+JxnmvmTQCb8eDCW/GaNFv1iBbb9odI+/fK/wvbH7+r1EvQe+CUoXvjeWLL6JVEi+CPlqvr3eir7506S+d/LDvoqV6b5nTQu/wjkmv8h4Rr9y/22/7ouPv167Ar4d1we+6wESviZvIb56jza+6RFSvixmdL5dX4++wxCpvtDbx77sEe2+asoMv2ZnJ792Tke/i31uv3Wjj7/rXQy+dGsRvvh5G74EvCq+gKA/voPSWr4/v3y+l0uTvu6wrL7AHsu+y+Lvvt/0Db+hTSi/CfNHv+Plbr9pvY+/eQAWvsv/Gr4G8iS+4Qg0voaxSL4dk2O+KIyCvtE3l74YUbC+sGHOvqmz8r5UHw+/3DMpv52XSL86Tm+/XdePv5DsHr6v1yO+taEtvi97PL7MzlC+sT9rvvgbhr4Ac5q+uyezvr/A0L6lkvS+KsoPv7adKb/szki/bWVvvyzbj7+n2Ce+kq8svmRRNr597US+EexYvkTscr7Hq4m+MK6dvl3+tb7PH9O+oXH2vgB1EL+QByq/OwZJv6B8b7/73o+/QGMGvcClGr1u+EK9CxeAvbIJqr3Nv+C9jJYSvv3uPL5IeXC+ADeXvlnYvL65Cuq+jCwQvxjOML8muli/JP2Ev/opMb1jS0W9NVxtvT8Wlb2hwb69zhX1vWd/HL5ifUa+cZN5vmJ7m75+usC+23HtvpqREb9m1TG/zVpZv0Ibhb+z8Fu9BfFvvf7fi710Faq9kHnTvei1BL5CaCa+yAtQvs1Wgb7Fv5++o5zEvvzY8L6n9hK/tNwyv3T7Wb9gOYW/+4+CvXd3jL3WLaC9Txi+vX8U571HPQ6+gpQvvovEWL5CbYW+/36jvoDtx76brPO+HRcUvwmwM7/WgFq/0leFv5wnl71s9qC9rnu0vSob0r1tr/q9psQXvsHAOL5PfWG+toOJvjk+p75cPsu+OoD2vpI3Fb9fgzS/OAZbv0R2hb/3d6q9SSK0vepex71cj+S9sEQGvp9KIL5vxEC+/OFovmnWjL54Haq+a53NvhtS+L4b2RW/zeY0v207W7/4gYW/Usi9vSVOx70mQtq9jQP3vaoxD76Y0Ci+HchIvqlGcL4dKZC+t/ysvnv8z778I/q+o3oWvztKNb+jcFu/q42FvwAAAAC5U6C7eLZvvIX48LyQgku9VOubvVmx370g0hm+Z/ZMvvNAhb6Ysaq+dr7XvmH9Br+Soye/tJVPv0RigL9mQC28l/R8vIDLzbzuGSO9JqB1vc+hsL3u7/O9oJ4jvjhXVr5krYm+g8CuvplV277Qegi/tsAov49FUL9Ig4C/ZkCtvKnf1Lzi3RG9m7dNvd7ej71JWMW9QRcEviBrLb4KuF++1RmOvm/Psr687N6+PvgJv9ndKb9q9VC/TaSAv135AL3xmhS9V607vRv7dr2iH6S98RXZvWygDb6Whja+Ck1ovr8Pkr6cWLa+G/jhvlMyC7/CxSq/k4dRv7nEgL+HUiu9DsY+vc18Zb1OH5C9ZWC4vZrT7L2WKRe+C6I/vgricL6pBZa+yeG5vnoD5b5obAy/qq0rv7wZUr8k5YC/RD5TvUFsZr1LTIa9UUOjvSDxyr3cnv69nZEfvpRvR76g9He+Tx2ZvjN5vL4CDOe+cCYNv4IkLL8mXFK/UfSAvwIqe706CYe9MNqZvVNntr3agd29DzUIvqT5J74cPU++NQd/vvQ0nL6dEL++ihTpvnjgDb9Zmyy/kJ5Sv34Dgb8="},"schema":1,"label":"discs-16","accepted":true,"purpose":"approximate display only","keys":["inclination","radius_primary","radius_secondary","temperature_ratio"],"bounds":[[65.0,90.0],[0.1,0.32],[0.08,0.29],[0.58,1.1]],"fixed":{"mass_ratio":0.75,"eccentricity":0.0,"omega":90.0,"period":1.8,"epoch":0.0,"distortion":false,"irradiation":false},"n":256,"manifest":"fcb79abab4e5e9fd5207753d1d718e144f2f2884b747871950f0b7fc85d30bd6","pairs":{"fixed":{"radius_primary":0.21,"radius_secondary":0.17,"temperature_ratio":0.84,"mass_ratio":0.75,"period":1.8,"epoch":0.0,"distortion":false,"irradiation":false},"nominal_omega":90.0,"nominal_inclination":84.5,"n":256,"omega":[0.0,22.0,45.0,68.0,90.0,112.0,135.0,158.0,180.0,202.0,225.0,248.0,270.0,292.0,315.0,338.0,360.0],"inclination":[65.0,67.4,69.9,72.3,74.8,77.2,79.6,82.1,84.5,85.2,85.9,86.6,87.2,87.9,88.6,89.3,90.0],"eccentricity":[0.0,0.11,0.24],"omega_flux":"...","inclination_flux":"...","contacts":[[-0.028405510656974213,0.028460579921375757,0.4706264219792996,0.5293849608327826],[-0.028405510656974213,0.02846057992137575,0.47062642197929955,0.5293849608327825],[-0.028405510656974213,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.028405510656974206,0.028460579921375757,0.4706264219792996,0.5293849608327826],[-0.028405510656974206,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.028405510656974206,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.028405510656974206,0.028460579921375757,0.4706264219792996,0.5293849608327826],[-0.0284055106569742,0.028460579921375764,0.47062642197929955,0.5293849608327826],[-0.028405510656974206,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.028405510656974206,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.0284055106569742,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.0284055106569742,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.028405510656974216,0.028460579921375757,0.47062642197929955,0.5293849608327825],[-0.028405510656974216,0.028460579921375746,0.4706264219792996,0.5293849608327825],[-0.028405510656974203,0.028460579921375746,0.47062642197929955,0.5293849608327826],[-0.028405510656974237,0.028460579921375757,0.47062642197929955,0.5293849608327826],[-0.0284055106569742,0.02846057992137577,0.47062642197929955,0.5293849608327826],[-0.028389519239623903,0.028108102816138632,0.5408456017334138,0.5992624224214897],[-0.027182420611253484,0.02696190560547814,0.5345600151046144,0.5955289197475467],[-0.026191161161522526,0.026045065691982987,0.5181024621169419,0.581435697040328],[-0.025549577396585414,0.025494587236079065,0.493943643383842,0.5589600440241463],[-0.02531050754731387,0.025349752095668164,0.46722543113463044,0.532804948007869],[-0.02545772388685301,0.025589404846118814,0.44103791015459615,0.5060825028015432],[-0.02600285527926476,0.026232622399392377,0.41858622781057236,0.4819214611465979],[-0.026914639177753334,0.02723064600259894,0.40447894926171774,0.46545068730234207],[-0.02805369554458723,0.028440162763095765,0.40074674051290576,0.45917082831336287],[-0.02931111782700272,0.029720724083121118,0.40691336635865505,0.46309002825205237],[-0.03054281455985357,0.030906031760765933,0.4231608087497215,0.4774328397420689],[-0.03145560138127454,0.031708206664990544,0.4470528199280599,0.5001181078855625],[-0.03183777040364144,0.03192318860013479,0.47367483498141794,0.5263257391510032],[-0.03162936219763074,0.03153790631195957,0.4998883247112795,0.5529480399009623],[-0.030838939471589763,0.030609687832705023,0.5225882361190513,0.5768410517161465],[-0.029658309960284877,0.029376699030543303,0.5369177894322574,0.5930934692009356],[-0.02838951923962388,0.028108102816138604,0.5408456017334138,0.5992624224214897],[-0.027851985236131294,0.027215938433245563,0.623098163415244,0.6801882749727074],[-0.025399488255815887,0.024943315706401393,0.6101075675856403,0.6728091911692673],[-0.023502689860667062,0.02321825657039312,0.5751860290681524,0.6438152621023316],[-0.022379456761018227,0.022249270437816927,0.5223035948659075,0.5955179443857394],[-0.021979737593674258,0.021988571605432636,0.46259208288499015,0.5374720581067602],[-0.02223195366088892,0.022386928651006454,0.4045075859518488,0.4777551275912723],[-0.02319142821004337,0.023525693652125097,0.3561888100580529,0.42486358391439843],[-0.024907528683261286,0.025431465182867125,0.32719736550007583,0.38992055155738553],[-0.027173368442602992,0.027899966225885357,0.31980352247478483,0.3769240056220717],[-0.029938024234126677,0.030828132571637808,0.3324910135912816,0.3849339059348215],[-0.032952024404630444,0.03386848172768663,0.36630796179381425,0.4151938124250263],[-0.035407254901949944,0.036045411545154206,0.41792037151669337,0.4646804274479189],[-0.036516748010634176,0.03661334780567327,0.47694321698718906,0.5230363955838379],[-0.03594610128967075,0.03549947746687594,0.5353236768741614,0.5820545807072252],[-0.03378410437404831,0.03303583240856712,0.5848024750508164,0.6336685947648399],[-0.030763820498922165,0.030001926134784043,0.615086751097843,0.6675013862870747],[-0.027851985236131294,0.0272159384332456,0.623098163415244,0.6801882749727075]]}}''')
SEMI_MODEL=json.loads(r'''{"schema":1,"name":"semi-detached-rbf-fourier-v1","accepted":true,"source":"native PHOEBE 2.5.4 semi-detached constraints","keys":["inclination","mass_ratio","free_fill","temperature_ratio"],"bounds":{"inclination":[65.0,90.0],"mass_ratio":[0.4,1.3],"free_fill":[0.25,0.82],"temperature_ratio":[0.76,1.14]},"fixed":{"period":1.8,"eccentricity":0.0,"irradiation":false},"phases":256,"harmonics":48,"donors":{"primary":{"gamma":0.25,"ridge":0.0001,"centers":[[-0.97,-0.8894302286207676,-0.2205538172274829,0.8511843383312225],[-0.2036622278392315,-0.97,0.7415184397250416,-0.2105125654488803],[-0.863114608451724,-0.28785725496709347,-0.97,0.23339262232184388],[0.5052373837679625,0.8228742890059948,0.428452817723155,-0.9699999999999998],[0.97,-0.244116872549057,0.8084134124219418,-0.6272280253469944],[-0.5168621316552162,0.9699999999999998,-0.29428473114967335,0.29908879287540935],[-0.4239379484206438,-0.5800591204315423,0.9699999999999998,-0.2609051465988158],[0.1919723767787218,0.06018267571926117,-0.6061561144888401,0.9699999999999998],[-0.97,-0.48699818179011345,0.13769532181322575,0.4455612190067768],[-0.2712365388870239,-0.97,-0.6518869493156672,-0.6174829956144094],[-0.6786175314337015,-0.8385521154850721,-0.97,0.5636122897267344],[0.9379239324480295,0.30367619171738625,-0.4632741417735815,-0.9699999999999998],[0.97,-0.6304347440600395,-0.8349929675459862,-0.04451747983694099],[-0.9574875421822071,0.9699999999999998,0.31396539881825447,0.9039067085832362],[-0.11639060638844967,-0.044503899291157945,0.9699999999999998,-0.9110998325049875],[0.25161640904843807,0.5642468556761742,0.5004155114293096,0.9699999999999998],[-0.97,-0.11514488980174065,-0.7009203471243382,-0.3718970660120252],[-0.027172710746526718,-0.97,0.21407437324523904,0.7000687196850783],[-0.9302203338593245,-0.7167140822857618,-0.97,-0.7399299498647449],[0.7011835556477308,0.1735297478735447,0.9012245200574398,-0.9699999999999998],[0.97,-0.7527752444148064,0.2731569688767195,0.14943831600248791],[-0.7129962965846062,0.9699999999999998,-0.7512138765305281,-0.7900776676833627],[-0.3684204686433077,-0.4168291408568622,0.9699999999999998,0.7660102304071192],[0.015425043180584908,0.9439039453864098,-0.06330484338104736,0.9699999999999998],[-0.97,-0.5092845000326632,0.6576187089085579,-0.9561137091368436],[-0.46765510737895966,-0.97,-0.1796209327876569,0.0966920517385006],[-0.6227741036564112,-0.15772678516805172,-0.97,-0.08830975182354428],[0.7609285805374384,0.7010217122733595,-0.9902430400252343,-0.9699999999999998],[0.97,-0.3737751543521881,-0.37053932808339596,0.5535725187510252],[-0.7804274596273899,0.9699999999999998,0.8573261704295874,-0.381683170795441],[-0.18402439914643764,-0.959702143445611,0.9699999999999998,0.43722941167652674],[0.448212718591094,0.4325243681669235,0.0430073831230402,0.9699999999999998],[-0.97,-0.25557674653828155,0.9728725925087929,-0.13518585078418255],[-0.1478442084044218,-0.97,-0.48700318858027447,0.807016100734472],[-0.8010138981044292,-0.9196440055966377,-0.97,-0.7529775965958831],[0.5651411786675453,0.4393793921917679,-0.6751565858721732,-0.9699999999999998],[0.97,-0.6118579041212797,-0.06073313020169735,0.3591114114969971],[-0.5846438724547625,0.9699999999999998,0.5397666003555059,-0.7184701561927797],[-0.49019012600183487,-0.2134099267423153,0.9699999999999998,0.7254343200474977],[0.1284056343138218,0.6785416919738054,0.35286084748804547,0.9699999999999998],[-0.97,-0.8690052572637796,-0.8836685754358768,-0.5368185881525278],[-0.33041256107389927,-0.97,0.40469422936439514,0.39623833447694823],[-0.7354094982147217,-0.45494942367076874,-0.97,-0.41876748763024796],[0.8823530338704586,0.935080548748374,0.7182803712785246,-0.9699999999999998],[0.97,-0.014539873227476963,0.08096337132155873,0.9378930870443576],[-0.8941619601100683,0.9699999999999998,-0.5667736511677504,-0.109784346073866],[-0.04763682559132576,-0.6629688031971455,0.9699999999999998,0.07131070457398914],[0.31866107881069183,0.19797814823687077,-0.2554188687354326,0.9699999999999998],[0.2916990499943495,-0.7486407700926065,0.44567290879786015,0.6417251117527487],[-0.0900708120316267,0.14331120252609253,-0.9676140937954187,-0.2823949810117484],[-0.999401118606329,-0.08456967771053303,0.6344707515090706,0.2736921980977052],[0.6346879526972771,0.5573842134326699,-0.1554983165115117,-0.6643619779497387],[0.9799642134457827,-0.3844171520322561,-0.5179066583514214,-0.8642146214842794],[-0.654247747734189,0.9746061228215692,0.004754532128572686,0.1923557575792072],[-0.3112015053629875,-0.782861340790987,-0.33176742121577263,-0.2476426772773267],[0.07044686749577522,0.32616383768618107,0.8175757899880409,0.8257839623838663],[0.23389036394655704,-0.1278792191296816,-0.41066989488899697,0.061514109373093095],[-0.3994458708912134,0.733067609369755,0.9238849598914383,-0.8895942438393835],[-0.5569489449262619,-0.5419312193989754,-0.7240388374775647,0.9293180964887144],[0.8250444047152996,0.06901371292769931,0.2381675224751234,-0.007398320361971966],[0.5451981779187918,-0.9900428783148527,0.6235914789140224,-0.46402454748749733],[-0.836672967299819,0.3999874480068686,-0.10158735513687123,0.604633407667279],[-0.24569809809327126,-0.3416101597249508,0.3118974268436432,-0.580367684364319],[0.38775983452796936,0.7984182555228472,-0.7909328080713749,0.4709763955324888],[0.40617504715919495,-0.19280547834932804,0.33361678943037965,-0.8920065164566042],[-0.22792423143982887,0.6652424186468124,-0.8155208230018617,0.03261713497340635],[-0.822496360167861,-0.6068575084209441,0.5199715197086336,-0.023469928652048222],[0.559283709153533,0.001188492402434349,-0.002911243587732204,0.9140806961804628],[0.8419370763003826,-0.9241400565952063,-0.6402274090796708,0.6140095256268978],[-0.5397201031446457,0.46683607622981094,0.14953431673347928,-0.44208784215152264],[-0.38667641021311283,-0.27570730820298206,-0.4516413193196058,0.4979107081890102],[0.24754435755312443,0.8652669135481119,0.961865661665797,-0.5759889353066683],[0.08874334767460823,-0.6806932408362627,-0.3059368059039117,-0.31124798581004176],[-0.2933012768626213,0.2083593234419825,0.7966890223324299,0.6393873635679486],[-0.6397082675248384,-0.01662217825651191,-0.61784253641963,-0.6795250996947289],[0.9936792422086,0.622432304546237,0.10755906254053094,0.2576643284410236],[0.6514552161097527,-0.4513881187886001,0.7219814453274012,0.21424490958452203],[-0.9820544980466366,0.9085814394056799,-0.24001828767359246,-0.8549165967851879],[-0.0769318025559187,-0.8498322777450085,0.4088280703872442,0.8300843723118307],[0.3049911241978407,0.2601391840726137,-0.9259474780410529,-0.22075614891946316],[0.33271656930446625,-0.8107930738478899,0.8691155891865494,0.3849045690149071],[-0.03342827036976814,0.3301812633872032,-0.3588302116841078,-0.5567938312888144],[-0.8763561453670263,-0.3967372700572013,0.05477008409798145,0.5031997505575412],[0.7421079184859991,0.9942389857023957,-0.545524211600423,-0.4250891022384168],[0.8959779404103756,-0.0717754941433667,-0.16009918227791786,-0.10931638441979863],[-0.7226090431213379,0.5362701453268528,0.6772204935550692,0.9687381871044638],[-0.31315272487699986,-0.7202043943107126,-0.970816671848297,-0.9751664046198132],[0.05287065543234348,0.1378431487828493,0.4888516031205654,0.08458805829286598],[0.14209157600998878,-0.3146128747612239,-0.7706363704055548,0.78708459623158],[-0.4756196513772011,0.7953582257032397,0.2535779979079962,-0.14644533768296264],[-0.566712649539113,-0.9786801636219025,-0.08281509019434463,0.16856060735881306],[0.7996556777507067,0.3812890406697991,0.5647172126919033,-0.7779212594032285],[0.5784032121300697,-0.5518452133983374,0.20230803638696648,-0.6876728404313324],[-0.7878429405391216,0.08720938488841057,-0.7125304751098156,0.3595010340213778],[-0.13046563975512981,-0.15339720621705055,0.8894238360226152,-0.3215938936918973],[0.4873673152178526,0.7356554809957743,-0.398732654750347,0.7434222362935539],[0.46095643006265163,-0.5762859843671322,-0.5786124672740698,0.12167670018970966],[-0.17045718990266323,0.033471735194325225,0.06930365227162838,-0.9497855342924597],[-0.7635071314871311,-0.22472823970019806,-0.2671623323112726,0.990213258191943],[0.6204698383808136,0.635027576237917,0.7569398973137142,-0.06832200288772594],[0.7751938793808222,-0.30579710751771927,0.3852221444249153,-0.39970444329082944],[-0.6086609158664942,0.8334708791226151,-0.903320025652647,0.5402845442295074],[-0.44932667911052704,-0.8917241264134645,0.698834974318743,-0.5158032309263946],[0.18220866844058037,0.4975419417023661,-0.21589335054159164,0.40638348087668374],[0.028530674055218697,-0.04927258566021919,0.5592311304062607,0.7063188012689356],[-0.3552228081971407,0.5919761750847101,-0.04119619540870212,-0.34695997089147546],[-0.6961937472224236,-0.650841748341918,0.37235969491302967,0.3380417171865704],[0.9359727092087269,0.24040892347693443,-0.8552383799105883,-0.7286829762160776],[0.7158193569630384,-0.8176709860563278,-0.4752786196768283,-0.80329111777246],[-0.9164700191468,0.2901002373546362,0.9845244884490967,0.13146101310849212],[-0.008970644325017929,-0.4817249123007059,-0.663344293832779,-0.187451684847474],[0.3746613785624504,0.87604820728302,0.17362966760992982,0.7656214311718936],[0.26890199445188046,-0.4269549734890462,-0.1211891993880273,-0.6014045886695385],[-0.09944023750722408,0.9623114559799433,0.6040697880089283,0.4607958737760778],[-0.9438902921974659,-0.7785088773816824,-0.8105079643428327,-0.4831094369292259],[0.6718885749578476,0.3607594631612301,0.29247111827135064,0.5925005730241539],[0.9556410554796457,-0.6895013898611067,0.9121756646782162,0.8769620135426526],[-0.6602600160986185,0.170254310593009,-0.42245913483202446,-0.048881793394685125],[-0.25709426403045654,-0.10357057489454768,0.22655357979238055,0.011112023144960403],[0.11112627014517784,0.5061870217323301,-0.7358013596385717,-0.9330318924039603],[0.20938432402908802,-0.9466353319585323,0.016364481300115363,-0.19535624980926547],[-0.4071063119918108,0.41113767214119434,-0.5061401426792145,0.8672149609774351],[-0.5036353841423988,-0.3450623285025358,0.829273462295532,-0.8138802088797092],[0.8644422926008701,0.7627087719738483,-0.31996655836701404,0.23573906905949116],[0.5230722855776548,-0.1859352067112925,-0.9480351824313402,0.2947987578809268],[-0.8448825012892485,0.7053158264607193,0.4650954622775314,-0.654128981754184],[-0.18988187238574028,-0.5218774843961,-0.13588508777320385,0.6608776748180392],[0.426730252802372,0.11937163025140762,0.6539810579270124,-0.27020780928432975],[0.43099589832127094,-0.461416881531477,-0.9776134919375181,0.5726211722940202],[-0.20299671776592731,0.9259268213063478,0.4996176194399595,-0.49445130676031124],[-0.8559831902384758,-0.8598610740154982,-0.16503636725246906,0.4406722132116554],[0.5254458375275135,0.27748453989624977,0.6860009897500279,-0.612502437084913],[0.8677917700260878,-0.6711526438593864,0.04991112649440743,-0.9214247819036245],[-0.513758746907115,0.19052554480731487,-0.5366960652172567,0.03078364208340667],[-0.41924383863806725,-0.007081540301442146,0.8622710518538952,-0.03800722397863876],[0.21462588757276535,0.6045985445380211,-0.3481423705816268,0.8973659351468084],[0.12147972546517849,-0.910235408693552,0.8806586023420095,0.224124329164624],[-0.2606718000024557,0.44561486877501033,-0.393810724839568,-0.833544198423624],[-0.665729645639658,-0.2618026938289405,0.1915572825819254,0.8563327621668573],[0.9680086672306061,0.8440456800162792,-0.7057489100843668,-0.21575254201889016],[0.6852329466491938,-0.20622172951698303,-0.0935181938111781,-0.25032077915966533],[-0.9483836684376001,0.6869517918676138,0.5715770050883291,0.6722119152545925],[-0.10204152762889862,-0.6202737186104059,-0.7794797793030739,-0.6339976917952302],[0.2802322022616863,0.022897884249687195,0.2584522105753422,0.30588897690176986],[0.3613624591380358,-0.5344998277723788,-0.44283030927181244,-0.07917594909667935],[-0.0044007617980241776,0.07718061842024326,0.9570200257003307,0.9698456954210994],[-0.9139143005013466,-0.1360518541187048,-0.629491914063692,-0.9613717831671236],[0.7044125236570835,0.7256266884505749,0.1426459401845932,0.10204167850315549],[0.9255432318896055,-0.3324466645717622,0.5307509172707794,0.4265430234372618],[-0.6926621329039335,0.8048988338559868,-0.009725259616970794,-0.5046842824667691],[-0.34967703744769096,-0.9965139124542475,0.34235338680446126,0.5602000653743742],[0.016209103167057037,0.3908296674489977,-0.82041029445827,-0.38834123499691464],[0.1789353396743536,-0.09299669787287712,0.4157011844217777,-0.7488348893821237],[-0.4391572680324316,0.550174789503217,-0.9366677179932594,0.3269151393324137],[-0.596599418669939,-0.7414256315678359,0.7268118187785149,-0.36686786264181126],[0.7699063867330551,0.15174776688218117,-0.24881404265761387,0.6949479635804889],[0.6161586288362741,-0.7890837118029594,-0.6129378024488688,0.7764677479863169],[-0.7504689041525126,0.3167650233954191,0.09880721010267735,-0.16707649268209923],[-0.15931105613708496,-0.37502786703407764,-0.2991075273603202,0.14303963258862518],[0.4586593769490719,0.9808227643370628,0.7858943697065113,-0.7836484666913746],[0.49877287819981575,-0.8311102334409952,0.23139918223023415,-0.4137564115226269],[-0.13302212208509445,0.31184786558151245,-0.744612343609333,0.5230847541242836],[-0.792413579300046,-0.4951641410589217,0.9190640673041344,-0.5457054004073143],[0.5917008686810732,0.8977958764880898,-0.43319466337561596,0.4050335939973586],[0.8119766041636467,-0.03534517250955094,-0.803693974390626,0.06492251902818658],[-0.572259571403265,0.5707169435918333,0.2816917542368176,-0.9867831710726024],[-0.47915240935981274,-0.6369143612682818,-0.1162996944040059,0.9483401067554948],[0.1525204163044691,0.21914965845644474,0.5953331645578148,-0.12020084820687738],[0.058156996965408325,-0.296294765546918,-0.12516482174396515,-0.7582480907440183],[-0.3252148665487766,0.8156600967049599,0.6471079103648663,0.18016974441707156],[-0.7327790912240744,-0.8822217658162117,-0.9392394609749317,-0.12603962793946244],[0.8992501255124807,0.47973120026290417,0.46026511490345023,0.7979614306241274],[0.7444042079150677,-0.5862767230719327,0.8380253482609987,0.7320819161832337],[-0.8875035494565964,0.05079435184597991,-0.3248713184148073,-0.34147126786410775],[-0.04646776057779789,-0.23471900448203098,0.027159107849001884,0.3484049737453463],[0.3370270226150751,0.6523501593619585,-0.5129693876951933,-0.7077942360192537],[0.30261868610978127,-0.16418756730854533,0.6880689319223168,0.9344541002064946],[-0.06583044677972794,0.6918765679001808,-0.20909649692475796,-0.01262587681412719],[-0.9689389783889055,-0.5001298263669014,0.3764416817575693,0.05225823633372784],[0.6471906919032335,0.10593241266906239,-0.8983828667551279,-0.8804299235343932],[0.9884384647011757,-0.9678945671766996,-0.2759905122220516,-0.5871173013001683],[-0.6275695078074932,0.4250650890171528,0.7617988809943199,0.47775653004646235],[-0.28317667357623577,-0.3663215897977353,-0.5893002822995186,-0.45346022956073284],[0.08539466373622417,0.7766361553221943,0.07614815607666992,0.59409960731864],[0.23517797887325287,-0.7073121611028907,-0.6682661902159452,0.2687850650399921],[-0.3812059946358204,0.17975664138793945,0.18239487521350384,-0.6593632921576502],[-0.536141773685813,-0.12138132750988029,-0.48206021077930916,0.6507521215826273],[0.8315848615020514,0.5156893935054541,0.9952752757817505,-0.29133043810725245],[0.5479541681706905,-0.40963236056268226,0.36549992859363556,-0.2411219235509635],[-0.8198939561843872,0.952320721000433,-0.8445353098213673,0.8192326910793784],[-0.2234297338873148,-0.7611862905323505,0.5543568842113016,-0.8745500687509773],[0.3928313497453928,0.3507686946541071,-0.03235276043415081,0.2026606872677803]],"weights":[[5.423342818217751,0.019250879163358922,-0.4561359730296321,-0.25750940568570846,-0.3442317824033717,-0.1243438522742938,-0.07450589007533912,-0.20388031702846782,-0.19294325137976887,-0.11996484220164223,-0.14482727334435236,-0.11684854785257787,0.004109528664148391,-0.1963834051591602,-0.036310648316465514,-0.13289697736982164,-0.15841306840820701,-0.1601919812887668,-0.1205531710535156,-0.16819356356528312,-0.026896442132035618,0.04324679696900057,-0.09855543475548645,-0.037903977612351404,-0.08166949450872495,-0.0018124735755709969,-0.16976064836041826,-0.044166074302662665,-0.2091910981206352,-0.1004307283547759,-0.07450075778640575,-0.050611976941311226,-0.04788737063226131,-0.028806594438780656,0.01735779429702179,-0.006753335295933608,0.007276010511705595,-0.02833679722761795,-0.0552614891054281,-0.006369537958650157,0.03438097482836449,-0.01600371694031801,0.13293445928278652,0.0009129480312540815,0.16609463020707083,0.014631446461906504,0.0740335772864931,0.05811638565616931,-0.022809587805142327],[0.7972603713014692,-1.3206982918809635,-0.5563083687938223,-1.175036070480108,-0.9850645784451872,-1.3470860681063412,-0.6559843042252167,-1.2338919266555852,-0.4590733712048694,-1.167943756949047,-0.15878300203657733,-0.7951328603094906,0.2600008843268492,-0.7423825711329746,0.5054409917780828,-0.4274102427368152,0.8121466643386003,-0.33893261795305685,0.6868853420440703,-0.1361009325339029,0.5332391211151686,-0.06341874766544808,0.5768325889887563,0.1249880501886498,0.6568185744847639,0.0531475349749512,0.583457641511101,0.098974571034374,0.06776668466943421,0.17009337443022904,-0.1740674056316127,0.10690914329592695,0.010958955730322108,0.09096350373787168,0.01881105832397283,-0.013022007068120992,0.016339674836588762,-0.009755465565279255,-0.3339932199285703,0.04146514304651061,-0.37528963982390434,-0.008707278621411787,-0.2230955037730394,0.011156330399743326,-0.00620631615242932,0.08872952791798674,-0.17028301700096132,-0.0575969948030958,-0.22768595783302029],[1.3909589468467696,0.2363267127089971,0.5145300818067905,-0.17830719603131814,-0.5361185094154297,-0.3421170786258469,0.15107518568341025,-0.7472039236457994,-0.016517336673461246,-0.3595796340414189,0.5171470598464699,-0.3226850100360155,1.0613622248567804,-0.3831086281351941,0.9349913733561734,-0.03591324798062537,0.4362797271473635,0.25639477985370124,-0.13047648271287546,0.44802189851730956,-0.7528896086463787,0.22046504623569402,-1.0460541584911305,0.12102745067358531,-0.7353710945187091,-0.06948380337169142,-0.08805815036831816,-0.045320721720965386,0.3674707848086481,-0.06906866471885865,0.4892429232772339,0.034012009867266164,0.12406656935970155,-0.040523765599937075,-0.2956128695747587,-0.07224749053808156,-0.43147166726994013,-0.08263340016858192,-0.2230731624337913,0.05504824278761141,0.07009932988338716,-0.05553319345969257,0.2009181707253463,-0.07045955632750382,0.24490930001333994,-0.02067825671142877,0.2577289301954604,-0.01903189134013972,0.09264123026855216],[1.1838059953215325,0.30431197211164046,-0.7295505468517838,-0.14291168042909275,-0.656532987028962,-0.40029598204983063,-0.6860870042425851,-0.6035116659409352,-0.9189064019667966,-0.5324031748845306,-0.8182127788791463,-0.4558141760997148,-0.5870381084276876,-0.45862536905467555,-0.5432894945202477,-0.349317957771678,-0.5476594752964549,-0.199080826132973,-0.3730151553351524,-0.14563599492002896,-0.29658014264220456,-0.19711812534631246,-0.4023159766638631,-0.30865013661247576,-0.5674665052099719,-0.2871153341392409,-0.4137258580682869,-0.12595040056414894,-0.0843925697132599,-0.022733953777032027,0.2043232065461867,-0.0075648762897081775,0.07148659271765095,-0.1070819657743385,-0.08061834839212882,-0.08702607584099967,-0.023125393761836994,-0.01924924275999106,0.22523561431882272,0.038924635115741624,0.3762895715018359,0.051665837718199874,0.2025421761521332,0.026043299673051724,-0.03206940908606681,-0.001053835008246125,-0.04402961074797391,0.04867576683581648,0.087782726321697],[1.9449016985694276,-0.2489176426907353,-0.7276484857677407,-0.10712492536281298,0.1261425948508909,0.06636408597575333,-0.4807144049704696,-0.009761065138112467,-0.38445894266490016,-0.045536981016958106,-0.419348810064999,-0.02466101136272137,-0.32829334166909374,0.04731597371350296,-0.040783668228572045,0.025111790670305424,0.16097286594544022,0.013856429476447333,0.05909222865927032,0.14952438386265487,0.24097973460440983,0.12312196670402092,0.2574886706142047,0.14316195940205398,0.3384960130965838,0.12285760018277453,0.47370478300441937,0.09923577571927483,0.5050116845546179,0.10580127071655555,0.3905782749304914,0.12451456776355752,0.2631239320136564,0.15082019471546984,0.18525441118923858,0.05851660472975336,0.12121361276328152,0.03817066241391983,0.018164234600862948,0.05472813293784254,-0.08547498907418234,0.00656363058042215,-0.15594145948516142,0.019364834102514803,-0.3150951117076106,-0.0009283829416295849,-0.3301100568571253,-0.06711284309047971,-0.2960118206385763],[-2.592481084209879,-1.444219763691592,1.389662134325755,-1.4223079210382885,1.0145212014469467,-1.0780666117366702,1.0388935551536067,-0.9149773097042063,0.7000057713980845,-0.6195326636809779,0.4796982226519357,-0.2996557546695119,0.5123830731319706,-0.04110325341295,0.7076229866364003,-0.11092191688550113,1.1284353854236824,-0.09075668353309858,1.5693047379799776,-0.29847170130149014,1.2623976878835563,-0.04893333944972869,0.1153137997664938,0.0679418732800494,-0.6823530918934467,0.1576417406111835,-0.601040777538258,0.0227422514335447,-0.051239051907103865,-0.08386647910209287,0.6694236397693474,-0.055164439088796424,0.5607363001145881,-0.04938390947455401,0.06213804437091023,-0.001277471315704371,-0.446193184452521,0.09525826180928033,-0.28703698121385113,-0.02116929169226692,0.13287358277754685,-0.015029583946268446,0.2248802162083122,-0.03465427234445796,0.19078900420244846,-0.006355674839233433,-0.04936322734961625,-0.07133082293972423,-0.12822386140723044],[2.462121726172569,-1.160040154512448,-0.17659676754349066,-0.5630203078160944,0.5099141713875203,-0.21855798444014232,0.7384814342294358,0.018157477184006603,0.19299261344630048,0.0019544674338237834,0.03934357049990487,0.07197085244983091,0.38420913381787336,0.14256558232054814,-0.09986158726219346,-0.11686757887076632,0.2002672889902165,-0.06294889963997734,-0.11960930959597108,-0.28937762556673463,-0.32989825718162835,-0.36918772818700335,-0.15433896714974193,-0.24621749830850814,0.21101967964118248,-0.03403777181630988,0.07295709019061426,-0.2764806694417497,-0.00860198908499612,-0.06782772888018494,-0.05996048410290565,-0.10266247196264323,0.06502530730528477,-0.15635947252170643,0.12359608433489422,-0.08058187960500744,0.28628841187598714,-0.0030151621016459356,-0.17188089678416107,0.0037041304253779767,-0.05758641411846484,0.06584384368820372,0.23614696344172814,0.09859280556341968,0.12623778792061932,0.10373815001773162,-0.14306927923189192,-0.047016596559767566,0.02821651703883748],[6.177560657032837,-0.13053196912587853,-0.87472691150243,0.0867338084991574,0.3210389482854039,0.3357041237519127,-0.1676230157470978,0.1624766137123019,0.06312326976305631,-0.0040097525026667,0.21194288109493134,-0.16872904058362997,0.025705211166183805,-0.09841291349965098,0.20661762192965352,-0.03992562499790423,0.17517745545503804,-0.11350004744382444,-0.1037309397289328,0.05950167995486572,-0.10454670784205308,0.2019583721843943,-0.059666196811138605,0.1742947863846511,-0.22385940297219348,0.1749141424697144,-0.014772186559607518,0.18575391887153125,-0.2533644287696612,0.23905588788609627,-0.6595459955212667,0.2946600750797238,-0.7661768482437562,0.30309514404472726,-0.7237118809264169,0.23149713743465528,-0.29824232937112455,-0.06376033730979559,0.10584576005567307,-0.22068236323521007,0.15729678049872237,-0.17885649118585056,0.028105594500871432,-0.20876541673151697,0.031094043767951534,-0.10867243320199277,0.30182369574869,-0.2509334136212643,0.551183283224885],[3.1415689747025572,1.26309994344091,-2.527584249256628,1.1267010284948529,-1.8544258017217026,1.0068857110947294,-1.575826416776206,0.8786980974853591,-1.325697434405809,0.6746168297114588,-1.5866427666399088,0.5039949050160507,-1.3988219052383952,0.42626289610527635,-0.8591501966802071,0.1990327891432318,-0.37392310310585947,0.047118917517249344,0.19811922898248852,-0.13520712824579956,0.2743914724034453,-0.11703169217066768,0.03017998211323042,-0.06553572201990264,-0.010982216788301304,-0.005286814857633881,0.018104108120696325,0.013158002809145974,0.0779849452353796,-0.054693712155985494,0.31294526723765653,0.018164660165708935,0.06372308398023163,-0.13965829107217897,-0.17299158359129296,0.03964473264364943,-0.11722548535523411,0.030346504254931808,-0.08357647378527913,0.038346059074159645,-0.0050958275462766045,0.0483037327472442,-0.07972436312958885,-0.01100425194303762,-0.07579092273652091,-0.04360210251801818,-0.09324060090998422,-0.010917007481758143,0.0786107618259894],[-0.3446967205771022,0.3893427634770198,1.3709018968606728,0.0003550490745571827,0.49212299621683364,0.17497925541730988,1.1623517271819601,0.04842619059274594,0.8116919402382521,0.2362474512011536,0.47247090294764654,0.3807827207312148,0.5516663170433318,0.3409298678016676,0.1764058259813424,0.17799064536828296,0.019195954796145127,0.06514008730091546,-0.12325589430661231,0.022664767750306892,-0.33019222328527637,-0.1876197741380897,-0.21502965228509557,0.05337024010678193,-0.171696001872931,-0.12662989037826894,-0.15062050133245825,-0.025075387091853894,0.057783527339387235,0.033381028759146586,-0.2082349514997541,-0.01107622235847632,-0.19766664699405825,-0.09133654024170633,-0.09968304304607047,-0.021165216961439613,0.024509085140961142,0.10940555370499777,0.1644530136101964,-0.08958964903399289,0.01804013509188605,0.00397849475923782,0.037532724660883615,0.010209166643344364,-0.07957201483911581,0.03531690401899274,0.016361772473175242,0.010137353754435467,0.05045151681974414],[3.423521303781322,-0.2885469753834513,0.5733685631989417,-0.06500119701805743,1.1818263795466415,-0.04673595322606095,0.6736539679743156,0.3023043686692326,0.9376315919167728,0.07367499762839258,0.7993790797976572,0.15782511391335963,0.36201372182730746,0.22234920936432467,0.2981575612350111,0.16633944658140715,0.17494186051961097,0.11978150126724753,0.12819399176520005,0.14400539676775315,0.13460830921881084,0.0407902427455873,0.13321448384238002,0.10503436139143052,0.0755603099475168,0.07428441786837446,-0.14634101728233373,0.10259769876611882,-0.17498783287358394,0.07978690968276593,-0.3129523395898972,0.027714780570769884,-0.2883234940850104,-0.025535420303713675,-0.059069462799682754,0.03694919095537682,0.14910703390639354,0.03274394097086595,0.26106193944687967,-0.1113593019402827,0.041842515773279174,-0.03599944996222927,-0.05788099992191292,0.05301928649329997,-0.11276666303764281,0.02201180301533044,-0.06718161603538057,0.011011834177764161,0.03532150124945821],[2.58894512953253,-0.08452963112285351,-1.3979307909205032,-0.2002519967195493,-0.9988400933879121,-0.16240427145267064,-0.984467692854587,-0.2404342626235461,-0.8211518155288141,-0.31025115816539905,-0.6117547415073006,-0.3135024552586985,-0.5391732140896701,-0.3278994499394871,-0.45274069610602696,-0.22317926703411325,-0.5958951570162536,-0.22841846320309428,-0.417943328770286,-0.21200969335148284,-0.38050132851159946,-0.22970567396956307,-0.34211573073219803,-0.17009477264106435,-0.172662765901303,-0.2199284339193519,-0.07056787987346956,-0.22318063760018106,-0.0938484300314077,-0.2758528258927678,-0.2691302562943359,-0.1886186990964387,-0.39533605203484945,-0.1513592011100742,-0.05532323819095109,-0.02980819435761166,0.12380038153106801,0.017704717179044954,0.1981102003591299,-0.08673225417420119,0.17982423572988998,0.028806614925840763,0.06319275328628071,0.0026827106599118587,0.08983093375176229,0.012677280991305889,0.0836333176503122,0.08809197645528431,0.06985005138969409],[3.0971887226829886,-0.5293918446797974,1.0841370877763272,-0.4124079470941041,1.7010901054529226,-0.47482851833435513,1.518577610982207,-0.2332443111049091,1.7300285592626556,-0.2539204992466508,1.923681818266442,0.0738966187256995,1.429738195640724,0.13951977481652006,1.1177047853428923,0.30564256225518693,0.5198286367180386,0.3458107058763032,-0.028658108400620958,0.3798229594195711,-0.3977088158878547,0.29982150969178045,-0.7873027902476704,0.23157924376423172,-0.9193700045096842,0.09524231609475063,-0.9647641164989216,-0.04404425560775324,-0.853936775173312,-0.10944734231706563,-0.7050229166302193,-0.19500987473097373,-0.45435538185737373,-0.14019805917621295,-0.06467045015841805,-0.1450904062314973,0.14376349087862694,-0.07843170417331774,0.365477560102884,-0.07599808330253181,0.42828061295420333,-0.036646786817271806,0.47950729528133684,-0.004223275827343261,0.48414431140887515,0.0017127171025427279,0.2748096887340635,0.022369070290167985,0.028928839159131258],[0.30568567271364405,-0.18170793796438664,-0.05830673106878428,-0.11971756868386815,-0.33678136188838415,-0.08665684795699748,-0.39679913547937085,-0.0379018653392861,-0.5242835806619731,0.021280671749060568,-0.5558123826462339,0.06590445816894432,-0.4991399253770197,-0.008748927357970747,-0.44124535938657455,-0.07039239288546018,-0.19149192188731914,-0.14415317687419726,-0.04160163446342245,-0.18019816038488837,0.0394323263845616,-0.15563462208177367,0.07555061317752652,-0.07420795909887408,0.12786164083966378,-0.07761237226537564,0.13667475242798743,-0.053921306676471055,0.1364853494464708,-0.01605262284632337,0.0011830932459241468,-0.020177665345619985,-0.06357200403709148,0.013206948006313375,-0.12741313833983625,0.04708876431694436,-0.06244128662026844,-0.007259571325676985,0.0277586883278248,-0.0020659358420687165,0.06676304406069666,-0.02662406816455718,-0.012108361437636102,-0.02658815337849226,-0.10165161845572881,-0.004023766084313717,-0.09263000263962684,-0.03658951295393008,-0.0024514553632318456],[5.799622356701771,1.0723200131341286,1.457238938381313,0.8414725850456007,1.6794710817322327,0.8636640319913362,1.6467862484597382,0.704595216698493,1.5256661057286554,0.5120670931300407,1.4137315120561254,0.5123023195220586,1.4303494684192448,0.49608501503883085,1.170950301825018,0.35569822003650703,0.9323375377191495,0.26604038811248615,0.598041645426427,0.3303981886591455,0.5941429666289834,0.32987847833325207,0.36436852726331187,0.3360111525765141,0.26491462343809313,0.3928816359271435,0.24359225416888294,0.13552867017569434,0.24541908661781414,0.08598317766107531,0.16586463581090602,0.11488351647454279,0.26802943364445164,0.22014653377876214,0.10130192902976776,0.036732711731357934,0.017274617371792565,-0.022068004430234097,0.10757765240391139,0.07358474230621959,0.09231178432835185,-0.02051551852879899,0.029461006618030958,0.031813925237005096,-0.1250169735958387,0.031620211042652016,-0.08961680929761241,-0.06718829978229408,-0.08792700850369518],[-0.8765010923725965,-0.1202375360301128,0.48917768981970977,0.3483476749835441,0.08531572262603027,0.3107699666761432,0.24229303728951854,0.19280559201376288,0.0007512854519690152,0.498602487992713,-0.3230894697942587,0.7779675464903621,-0.18850836417787767,0.5916262043750826,-0.06984229016979222,0.5866863120820208,-0.061044138509039704,0.34815034515507765,-0.22306213076855633,0.5332475975223318,-1.060813778606536,0.7331529117663607,-1.4604338801791188,0.845087041758376,-1.2286773194955412,0.4571275818862829,-0.5480519134216301,0.3434545789285548,-0.7389819104270603,0.24844228542268476,-1.0993256469855939,0.360725879796388,-1.5100703732619494,0.2634314828701061,-0.9250447838271355,-0.07736145349318649,-0.16590193915024815,-0.37002657899763625,0.15993349531699352,-0.2775996071382427,-0.20767996988926624,-0.0765096970945498,-0.4672549045960975,0.02396966734197905,-0.12511381897973856,-0.17138691630495487,0.4037459946402887,-0.21702987495632786,0.4244645626465346],[2.8313015843169858,-0.41463647395062664,-1.697066174058598,-0.24094979386813076,-0.8183796335840811,0.1654590460112075,-0.9785071367225356,-0.014423621865685042,-0.5209385380792858,0.6065940366605385,-0.11293044060992212,0.2795450638898533,0.6869189812777043,0.2130622210860931,1.1281259043132328,-0.23605016041603322,0.6879163718994975,-0.5556036497502546,0.13266116483290705,-0.4065147317997062,-0.10244447623356055,-0.028241871832683757,0.2178040064029547,0.15763716956378895,0.5071350913651033,0.33079404472739976,0.8950622210084791,0.12051091840218968,0.49608178011163756,-0.16778388799382488,0.09871730438937681,-0.006283835120822599,-0.23430460500363004,0.1334547059946394,-0.35785319518398173,0.1373511082706276,-0.31830704489739786,-0.00030373037897483604,-0.2582670420943559,0.09296114574710149,0.17188131012663183,0.0651605789832658,0.14969672606004547,-0.054463041741000036,-0.0006879969686241231,-0.04035146993559915,-0.17598682581744257,0.0012844198797914222,-0.007832653436357492],[3.9398820937599712,-1.3702712687523357,2.612975772179555,-1.8854680446141119,2.703041254268907,-1.381766362139097,2.9821456681859826,-1.3330591881411276,2.010008346785077,-0.8723450544680381,1.476751375470606,-0.2633264118035381,1.4840693349970089,-0.0964964255190807,1.2335283779351331,-0.10502892201096317,1.4064124570160448,-0.12578728392879807,1.2750888133308087,-0.11831494918423308,0.9391758586168555,-0.041540254401307035,0.6700685001045105,0.300739011258164,0.23163395144834387,0.3782265973944492,-0.15202864208888067,0.10234713407236226,-0.18549981615662545,0.1235322437083868,-0.14231663846060094,-0.041588994243457446,-0.03128218419532175,0.12903099249621772,-0.18820705345019006,0.24731867966964968,-0.46420030452339267,0.056088459128231585,-0.3660078695533639,-0.01840242178732907,-0.04535046850197002,-0.032065904609193466,0.020087687779889285,-0.05309296611466596,-0.07850056287852178,-0.014504935896872692,-0.12652085329566867,-0.05490611159355224,0.016732604202012568],[1.8395080629246225,0.0747259463972852,0.2675714825665276,-0.09349090467539566,0.8145813226941856,0.03984130557862725,0.1230540912248109,-0.16130221833955066,0.5170001711118022,0.10072225071356072,0.33579929812899173,0.02496232409441881,0.7562716630485095,0.30757237749160127,0.22512232621956657,0.3120137978458294,0.24792089504948314,-0.15304930957072727,-0.1604964147529872,-0.05579286985520508,-0.22432810561819055,-0.04303175154836374,0.11254128862772242,-0.22896454081736303,-0.18862567046504264,0.07095219404246515,0.25565432729773147,-0.041867059749100266,-0.011648356632779494,-0.16275537186398586,0.07158444796545621,0.02992587864028815,-0.12795777224919685,0.013078195371226897,0.2940016891360608,-0.1250072156119558,-0.07552474671345269,-0.0674701038671215,0.12966878870783963,0.17000058736570586,0.08979810327298632,0.15788552772272532,-0.008810033161032283,-0.14222374035451246,-0.10174528171722683,-0.04893826546572152,0.1778899521450661,0.12852339317187284,0.1474762262450949],[0.6037264968707222,-1.6994473411795354,-1.320909589048385,-1.1896183309927926,-1.2438876341432068,-0.7207663923727956,-0.42196320114634056,-0.23576008606145663,-0.24710914296724446,-0.0629935657192081,-0.21632619137261525,-0.03171259876545458,-0.2941954080915909,-0.09276219613195794,-0.38046325008882775,-0.022966021858141013,-0.43238700218024056,-0.03897433933478542,-0.2819099947734349,-0.11470044719072421,-0.3664495313554213,-0.043707511553634934,-0.3631771364500499,-0.02803434720388953,-0.5609362034104524,-0.16812118628034192,-0.7240568127462157,-0.14215971175842051,-0.771544750203095,-0.15708167358984965,-0.554531250901466,-0.15648591378014184,-0.42887771962219706,-0.18621479406444238,-0.33224576066749745,-0.14145342360780058,-0.3547342725381663,-0.1535439307901156,-0.3386698982358149,-0.1900723674329255,-0.19705148485355073,-0.051834905674878175,0.03863747263007134,-0.05427697392237093,0.24862926311752626,-0.051788699795652385,0.2514318132399798,0.009089478007686993,0.20230232955255584],[-0.04373559944910828,0.06991083779146468,-1.1470155520404277,0.2050839374728109,-1.0027996020148073,0.4213241708407559,-1.7467536012718459,0.6673426003571544,-1.513052633975965,0.3929496352885055,-1.566552678699684,0.5245678220191242,-1.9231327943946122,0.36830596378379493,-1.6519683846877804,0.24030661206013007,-1.4927060588390697,0.014100621894902863,-1.3420624446653466,-0.09349210940311099,-1.3169601383292417,-0.29877940914578244,-1.0863634361152619,-0.3207700443743992,-0.7650776930959091,-0.38544902609867726,-0.638779374170806,-0.22961282920648862,-0.25879311914578745,-0.2157060368931081,0.04883258230496095,-0.1497684078113602,0.3342271561184703,-0.17998772862842036,0.5006189598150608,-0.12156091474092649,0.50647917615282,-0.06243827909613843,0.4163208211934332,0.05710494811637486,0.1820526596183948,0.03620468392528879,0.17626912599508807,0.028322634329209923,0.30443043717487633,0.0575855394772499,0.14080641183951656,0.052185487420692483,-0.1926888930348054],[1.6501175596566293,0.349238837671009,0.5379928738742018,0.3017236185232244,0.5402178537192688,0.20081802020115871,0.550743084698974,0.19792257594446133,0.4604521298301768,0.14244026166254603,0.5061309812452792,0.1250421821418096,0.4575507221904927,0.07848428013404263,0.2894591479034266,-0.021909863345583112,0.19059856644101447,-0.05948121019835533,-0.08059332070195684,-0.10926890088004676,-0.12568189888642478,-0.05154203471691853,-0.10839655261861939,0.008785962528174311,0.01921052488781745,-0.016436654415077186,0.04748921334447862,-0.07399781027278869,0.0350147288303255,-0.09851099721584163,-0.07204751585391585,-0.0923062267115159,-0.05338165879182921,-0.03231818637941657,-0.035561913551096724,0.020118363541312044,0.045679718946082096,0.006538867603030632,0.059096770383864114,0.034843203723758556,0.0425535340196428,0.018597500976122,0.012026947795437058,0.0267132723409955,-0.018270405375263914,0.024346588226998634,-0.040025894195695466,0.01911998140547102,-0.0354434210795821],[-0.44672775096670786,-1.0633822371987713,2.2278796969568617,-1.0002124946175677,1.77654943705246,-1.058619459556985,1.9878431749364447,-0.7743231716889698,1.6099330568603158,-0.844981984610038,1.462417122981953,-0.40902512677427333,1.414424326163656,-0.20524377246484574,0.9474483406898404,-0.22160699193481195,1.3686953090014702,-0.05536174563556897,1.0037666809282835,-0.10466735590007843,0.7855905215884917,0.07014070411469671,0.38799888101190766,0.1585727426441575,0.183106854774745,0.26975861346689406,-0.06944558167775963,0.18177310601904498,-0.09452599580801843,0.20432760402157776,-0.12320423841655166,0.12633893616538008,-0.16643475693808973,0.14669698544698473,-0.2731416771596239,0.10791976246690463,-0.2556305320933719,0.07482051149962948,-0.3434242645951502,-0.017547330823752726,-0.21349504856222976,-0.07862685676574432,0.1107442553153657,0.04841953433843129,0.1530747041202584,-0.04976814173419114,-0.12682272486187907,-0.07026032693232356,0.01473941135289677],[2.211232735230698,0.11936124567743249,0.1202143697925596,0.005027604670091361,0.025593086568918875,-0.11558065294001331,0.39951798595982246,-0.11645493720985034,0.546236817387307,-0.184092548527433,0.8605732087948168,-0.34625909183903747,0.8955930354732279,-0.36120287931143896,0.8883314529673314,-0.33587903254184837,0.4534419132695382,-0.05565242170031652,0.12611834662497262,0.16218317609614652,0.085650778857324,0.14470554266202135,0.36435890025221884,0.04733423261952851,0.5957504981725564,0.041492406145124935,0.4264017475432676,0.05442662847918716,-0.046993000237042934,0.1387479966699459,-0.33753025614826526,0.14749764617628477,-0.09131524533776081,0.02887284890608064,0.33961284785206447,-0.019990443352166917,0.4234548159553735,0.03927591618781157,0.21125494525624922,0.09076429820638522,-0.2103609763795698,0.1864945074150412,-0.10747003116898182,0.12093137359087854,0.10405819039694096,0.04688547063759674,0.17416079942610818,0.060011496500063925,-0.037297707544392454],[3.6904765824031713,-0.0765519827652408,0.259764056398605,-0.07260749609993866,0.5274480253197609,-0.026863949045178456,0.43190683713790895,-0.04752519487561325,0.3646610610023401,-0.04565221643505862,0.26683561229770547,-0.003450032307331118,0.2731707669608938,0.06133741127951006,0.21384558000413229,0.03318903909948914,0.1946711250785058,0.013029376823763457,0.055344057469653686,-0.009342426238704038,0.026693837513875537,-0.015457814266333664,0.002856639868637043,-0.004927402655359986,0.0566380927370044,0.045281308067682075,0.03945416662731851,-0.005058607132877861,-0.003420588448068584,0.025287117779085343,0.0020325594827880116,0.020228184468186142,0.07934103569403202,0.0035939769331172792,0.05934911926270452,-0.02316152449006592,0.0050132952475795065,-0.004586036617483666,-0.06174348094376698,0.014534360148292713,0.016551256824266725,0.028907001767243708,0.09644928202126844,0.007932342359280474,0.006649949681307626,0.00691986337643478,-0.06177349105434326,-0.026104931096445856,-0.051138639081893277],[1.244309170233435,0.7118658119763532,-1.0071907892937095,0.3398459353446655,-1.5001840062561376,0.44564414019105386,-0.5469786743132283,0.28654266573600007,-0.4779549565126374,0.18988181860001738,-0.7934970815123377,0.16347154371421285,-0.24250002544759255,-0.05453095499684289,-0.0673289886700656,-0.09137545034330621,-0.31741802718930295,-0.16531188565078445,0.12746644442591704,0.04327233208359432,0.054567003390739326,-0.12934260477443432,0.06272841170899553,-0.05290976304860482,0.1881321646701638,0.031063397262657777,0.07972269236091022,-0.07235689862819578,-0.04899164850242737,-0.014658500956604769,-0.07849806430427414,0.01555396864394665,-0.0921904296518845,-0.03340947806821648,-0.04207888767631632,0.03871120006109893,-0.06891676822705153,0.05122725557349497,-0.1184682095741016,0.07530125272784287,0.01820310553543961,0.04838946284094328,0.20325178991968715,0.03104289757132827,0.20464452380094858,-0.0027374945318915994,0.21614472188764589,0.00195063699463672,-0.055245630385442376],[2.342875039656637,-1.3196868844852956,-2.0876701755258398,-0.4485163041867335,0.7749033685688378,0.2644779184859452,-1.1770185926671792,0.4145446648112206,-0.798041178440575,0.7665113128630483,-1.49032116428071,0.3494202426064734,-1.7624859696487485,0.6033021766503973,-1.3439946130828326,0.2772831156755543,-0.794674383619442,-0.27966271066596143,-0.46301241775124313,-0.4631077829655677,-0.27492334078981695,-0.029945779495239647,0.19108944213536527,-0.3215230272641524,-0.5220377446825367,-0.011804092581531304,-0.004829027306221834,-0.01565979379316475,-0.5198470538792334,-0.2836943257275805,-0.10343391920196982,-0.1581269253679333,0.004165721265940337,0.2019406532033062,0.29704326513144674,-0.04710720561123118,0.09332549846258663,0.018266827950503137,0.16532886439430253,0.28823867017983174,0.5129796358355351,0.06585980649344898,0.028425765979798807,-0.2603488514147295,-0.057675610783269306,-0.17216255692767535,-0.18411535331921255,0.09878970262508899,0.03931033570871769],[5.574414709773794,-0.3214399488887583,-0.756634169481324,-0.09104535555151201,0.07904438161059685,0.17273438691118656,0.02105171586961774,0.012120238737786051,-0.06878031066316738,-0.010563746711202693,-0.14723407059979435,-0.09174580805523089,-0.18569686840038227,-0.12206970594418777,-0.09271373623024964,-0.13239182210420875,-0.15515716175628963,-0.1008113642195943,-0.06360075969117432,-0.028721676101331087,0.03683146285893612,0.03342318256551572,0.06599933036776567,0.07385145373856597,0.10252334440291067,-0.0013608800810436898,0.04719212486511983,0.01666563067477776,0.022451987859555496,0.06991660011458974,0.05703095523987653,0.12154558272394862,0.05011213244913577,0.13170792069717702,0.0489058569837217,0.037918993388299446,-0.08999335032717806,-0.05659886419598754,-0.14842535356982026,-0.08964199356028364,-0.13098701245367836,-0.0358868058403768,-0.06607594733778023,-0.04096724960282699,-0.013672012326343835,-0.04219544601927441,0.0033265420863626597,-0.04360530076661112,-0.02857156098806038],[-3.857679020045911,0.20573836378703203,-0.5199514052955593,-0.2664800956380319,-0.8976538133030643,-0.803518983393778,-0.1747417169465292,-0.679727657026995,0.1611979961083365,-0.7359476993458773,0.6870396873586997,-0.6196916727620704,0.9429442547464995,-0.6958782720067913,1.1066179413316193,-0.4776842176151678,1.2622618293958117,-0.4333994833617956,0.9329670018114528,-0.455254173697451,0.8083017526412293,-0.40131833371149406,0.9294306677211369,-0.4658183021104175,1.4561187900180883,-0.4555422170904056,1.4731702795610617,-0.4910954675320468,1.3780889093242197,-0.3267090600898915,1.246989125134842,-0.23067287783825616,1.1664221244955193,-0.10975849173585006,0.7527793823983285,0.11091488706310752,0.257687205086715,0.2330897498630019,-0.3266384820556833,0.43696417131126325,-0.6961483599633105,0.3905633317893468,-0.6060415052418322,0.18142432972872508,-0.45408596633679443,0.06745112883170817,-0.409271489071667,0.004267478085032217,-0.5738814008323734],[3.4960212040916314,0.22863113734599552,-0.5189999913833836,0.20910532031253082,-0.3706222113073621,0.22023931178527364,-0.15970133795060512,0.19206703563707422,0.13399068488409413,0.1595247371941544,0.16699977415727882,0.15231928243669873,0.16701813347052746,0.10947011369316759,0.08913395084856984,0.08925655021522474,0.0322539616997349,0.030516193830605987,-0.08146380921232732,0.05205566614861308,-0.19933071733821392,-0.00251688604206872,-0.33977555884555066,0.00792633331138334,-0.3234594377145041,0.02911348267408482,-0.2691971258833447,0.041430647431288985,-0.29016263536659215,-0.015173918648596709,-0.30905883038701526,-0.0459506870475285,-0.2321847875054438,-0.059704553684192016,-0.12474717427578634,-0.029101839792610922,0.06403170024174416,-0.010543905393968817,0.07804042244318701,-0.023802448624053545,0.06291151543592584,-0.03866840703878616,0.06920810126557038,-0.011818597205706141,0.08997828202058064,-0.007791203429934105,0.04832449014082622,0.0317048176483979,-0.0026185468102044233],[6.404175355343661,0.7386956200037345,0.2610276692578382,0.5079501359186477,0.6217570092234259,0.4448487052022704,0.8178697633533507,0.26614481775517823,0.6068737083169932,0.2576230475989268,0.6398730981514092,0.14647504490225524,0.7619676989242604,-0.06606276273818491,0.8115892105173447,0.10856983038498336,0.3437504258668805,0.1519912541644018,-0.09517572537409737,0.24526707609155438,-0.07478809628845862,0.10741318472405482,0.206718195105441,0.06322091268554919,0.43014173185012144,-0.019210802833654696,0.26606191231001286,0.05218215971833508,-0.1716737086103,0.16517522315750982,-0.10785807497085721,0.120662727258853,0.19252563387165525,0.056494221077885674,0.3249614201212889,0.018728237157600965,0.15905159639462096,-0.02781306441660201,-0.04018003072196477,0.06210060906223194,-0.10442249043686594,0.0869116336432787,-0.09337125359761654,-0.04010688203647257,-0.033203781010330544,0.05898163931705309,-0.06814732354544466,0.009443542840698744,-0.25493347433555935],[-0.936488555713838,0.6106549285666223,-3.3174013896907595,0.310466973312945,-2.3477769928855285,0.5112976321089758,-2.13253254078462,0.27747579838377057,-2.0646128935308496,0.1985745457100225,-1.8392903143681985,-0.16857678573948,-1.6273508502684917,-0.022868571747791777,-1.044804950832937,-0.32979874976442863,-0.6040284436686242,-0.4145695691968788,-0.10480482757118373,-0.7142270488133978,0.7323854601594063,-0.9898178346638778,1.2408012024284203,-1.0233114941701265,1.138734076377795,-0.7460343360685115,1.1391295231733698,-0.7078943445914962,1.330969809869228,-0.8150345749728695,1.8944436886213976,-0.910672534652875,2.142409540995306,-0.7454798349876829,1.3605350136734187,-0.15227200516969785,0.295971427529506,0.1960261647166381,-0.12835561609087265,0.25813917278028115,0.2177553959527552,0.029361430734804298,0.8151074749105716,-0.20393228556135914,0.5129205723881205,0.03596655780062932,-0.1979200148496934,0.27550621496844935,-0.6753852848942133],[0.2741359503182742,0.49053159646095945,-0.16268378747047005,0.2938517235826893,-0.7029689860286926,0.11564168204472548,-0.40639758453202984,-0.1475641100987603,-0.04759053748497992,0.06369919130918579,0.06659997977964113,-0.25416358324460214,-0.03595147550570346,-0.178169897771547,0.07080143087319658,-0.02719351391925396,0.058155182108974246,0.012680455988836195,0.08474407832394863,-0.013009530337406289,0.170774142796463,0.022981697390510935,0.09490179436817327,-0.009416807391743309,0.15173747393011602,-0.09064725973113048,0.07033707197301606,0.04721981884770841,0.10957011786151215,0.03207457467860531,0.1782924873126848,0.01937318164418388,0.08303235525717104,-0.11069820422068827,0.10382211241209982,-0.04879226971174778,0.0926189072183028,-0.010175274406927781,0.08272171733179488,0.00674864958383907,0.12777660901194632,0.10993440727842407,-0.029877497183226926,0.0086760086133911,-0.1263885065824043,-0.06515836047656788,-0.02022211392279335,0.05616078127537173,0.06402556830493293],[2.145175606949743,-0.6351379149872372,1.2848105714335314,-0.831649029500337,1.1295842448723143,-0.9200689108696292,1.5310499747699569,-1.1220389672250715,1.734252220589238,-0.9471751379783794,1.915075347719628,-0.8950472681443256,1.8707081000841312,-0.6009292122131202,1.4539492306654112,-0.2152009968952722,0.9650969138481736,0.14194651915609746,0.235365187779797,0.35764674628234466,-0.2834018212677607,0.4109865003571344,-0.3656192346142221,0.38670354802198403,-0.3221809038454667,0.18672381558953932,-0.18760329084978805,0.13765594635736408,-0.1123421530737351,0.06549054813690258,-0.21213647941718639,0.060722320863045305,-0.0908912015504733,0.011586101522834904,-0.045223393231419814,-0.08889626809969525,0.11821733959069755,-0.08562058858936909,0.14935124913655937,-0.012749901563014868,0.13708329229521665,-0.03550234606680189,-0.019635774505805084,-0.025652730006176225,-0.09750950095083866,0.003741420138581083,-0.1316917084447951,0.008421205295792909,-0.17199756139182698],[0.0008802610172073158,0.28871701795844895,-0.8836595798740213,0.2226762633038652,-1.126925679668545,0.16281864211502217,-1.1616984507286419,-0.059794923263385424,-1.4016047195642354,-0.07162127010200298,-0.9495718559629229,-0.12346653211233677,-1.0667536381507892,-0.3699364345007451,-0.4320737495217436,-0.2565525938334097,-0.46278606979199355,-0.1632055833346308,-0.2141549968284365,-0.012662621006676044,-0.152505558300776,0.11580379759953183,-0.25391607905448355,0.04583026715981008,0.1454045891527585,0.05291702717396015,0.21947951870883092,0.014290465075558353,0.2800679305286295,0.019299335314655344,0.3467070295326824,0.02074229705464017,0.23325677308642123,0.12535770425153056,0.06665196935405832,0.08321329155766484,0.07350465946570965,-0.07080552712008832,-0.08184208582226256,-0.03477963985600476,-0.09040726282454856,-0.09155029009317019,-0.036129804606002146,0.04409173864698319,0.045473491564495705,0.009419865268234608,-0.04530135348964638,-0.046283721900706745,-0.0654099955644677],[-7.310161477231045,2.2743822963847804,4.492956507303166,1.9382957943535462,3.188266786406628,1.4155225796081519,3.3482677488741373,1.4543138495219694,2.992366437297519,1.357562811986552,2.8552303911393078,1.3314751847503792,2.2966431025373804,1.203299465238408,1.4876045833944143,0.6631803591311269,1.0685664050992993,0.39989459651569503,-0.06859887139463389,0.009069551044885084,-0.6335223582190971,-0.2151953270085472,-0.8294386526314478,-0.3834463737954165,-0.8405653290616781,-0.13749818700531802,-0.6512809820670414,-0.21335572125230864,-0.3904337751961068,-0.1515622732119497,-0.4514375289382912,-0.3178235427599539,-0.1765530261451962,-0.14912847028548146,-0.14527752850230272,-0.00512926631090922,0.45251939395484453,0.17106781102465646,0.69304123891303,0.4991772214676053,0.6502141864640434,0.14434439567391144,0.4362707176120902,0.3427811408197752,0.20643315616965682,0.3373619889254515,0.13316012432179672,0.06919666859146466,0.061220733216443515],[-1.0679051938994364,0.09943043752206124,-0.4667126634605429,0.3004493967148109,-0.05016854944726902,0.09479279332605298,0.5110216392856517,-0.16097023677273017,0.32500432279862007,0.07839310606349591,0.19944487389964174,-0.18474762911615006,0.3601393152410302,0.16952619366496005,-0.010946789778360761,0.1506915137286596,-0.36300201313722197,0.3506888049253937,0.06542376536912241,0.39073197726086706,0.635455535140484,0.3675289975138438,0.7991108144063901,0.24105442368637514,0.5910745898816503,0.18058503432852702,0.9139269888898061,0.055543979748434655,0.9227726372012812,0.0758769930888067,0.6184360206401011,0.050417531921325885,-0.07529873612244957,0.2251676381842922,-0.40471313747346815,0.13327565489587687,-0.42941040221796484,0.1190301137988991,-0.07421204580209857,-0.14464508585779648,0.01821203182730653,-0.08696335952934513,-0.2727539692712377,0.052415023350594996,-0.6100430131271395,-0.0040969025731670905,-0.3485924113829474,-0.07152359204474838,0.34580110347609405],[2.023662034528711,0.5971446571959124,0.09625853784592639,0.3729127645385945,0.355520309451615,0.41957085353736434,0.32082765538470004,0.3292845255753415,0.1373525080775201,0.24278159015570627,0.3563469510287745,0.21107231408199456,0.5236905171508146,0.10899275627404204,0.3810102946523648,-0.04641657391266848,-0.03085936662289024,-0.011100679187315367,-0.28458847991233044,-0.009882344076614766,-0.15161467855510521,0.17127214654433073,0.28647799750954916,0.2763478865059503,0.5552368266406998,0.12904317669700216,0.3627809433848547,-0.10752351967959903,-0.02681324243581905,-0.07927900532909779,-0.12573053916440452,0.057058447861266785,0.1610075618646382,0.168792456358176,0.340971180001815,0.06476110507642097,0.17463295641239135,-0.005455278882548634,-0.08486137685271351,-0.017040226509054287,-0.1245043669285764,-0.0023482755858443933,-0.08976971412564438,0.03470669113374626,0.04657474429829278,0.040716796617580035,-0.015574922642607663,-0.08861611954593816,-0.15620202358793875],[1.196991898286194,0.37227623037837587,-2.063670207504485,0.5208213655118277,-1.3531948706560952,0.8293427393351196,-1.647912403415289,0.4202136778914351,-1.465600092810197,0.6869029900826361,-1.4807875557839247,0.4292062722127798,-1.4273427571626536,0.44920319419745336,-0.7955173429990283,0.2394470998147379,-1.0081554956903076,-0.021491310875015077,-0.5509247890553218,0.0980076091796196,-0.7923586379752864,0.06437178878370516,-0.6777721704990638,0.14471856032210328,-0.4177226262389408,-0.07296032366359952,0.08934128760708848,-0.08757116806947142,0.17234763742303064,-0.2016344128816764,-0.10252862239379262,-0.047391855864824535,-0.30675068626361196,-0.08831770915288209,-0.14519987801925194,-0.09055521107916328,0.07566291288429113,-0.11514125069351525,0.3112248904907318,-0.0653368208187251,0.11008430566228733,0.011079379145612907,-0.18534261890300402,-0.00857142069495809,-0.17453426399365513,-0.0067740416766288655,0.1258560946614931,0.006786462407766407,0.11454510306317604],[0.4169062523609917,-2.0771761498886288,4.367800197026632,-2.2197947065333015,4.182253797211425,-1.9946869525324047,3.6171646118762824,-1.7415682580919214,3.3988603219652322,-1.5357095686744069,3.205180482170106,-1.0197718343543765,2.2915556524867897,-0.4878118636747948,1.7526349504221859,-0.03473704322014941,1.5177492723699,0.026025253069858777,1.4887575500215136,-0.06060739399239511,1.1673980829213546,0.07689949954357343,0.5736885175480825,0.22447953594485195,-0.27397047130562024,0.4276864509299965,-0.5105254591834782,0.38791208232182356,-0.022024647702463748,0.20516253712717414,0.15939916168437215,0.1194644212222972,-0.10414286361516985,0.2622074068020341,-0.6209466900704284,0.29848829587182707,-0.7054426513000704,0.1881459616373608,-0.21845844319924526,-0.033282697707596176,0.1899810912975095,-0.11649828040397137,0.04640545623387764,-0.007535888776710161,-0.3196118835686218,0.1562285414032694,-0.3525993479521841,-0.011594582020234901,0.10785950868069039],[7.286397652022798,-0.6024467984669846,0.004979250159308713,-0.2816506381273826,-0.5980632908848623,-0.3105665189891053,0.03936824518180487,0.13955632020060127,0.20425245145522047,-0.017524653540374552,-3.372949941165451e-05,0.06546223294578513,0.041912532424940585,0.20483191040838036,-0.036267176489104276,-0.13418592186271547,0.19692087654330911,0.2323469491214547,-0.1476848712629162,0.11426352700449101,-0.10238328037803067,-0.2420652369846285,-0.11709669189789548,0.16251990007204312,-0.015368938561207903,-0.13764297685778137,0.01574288841935685,-0.1375857128102125,0.15560783607818912,0.08295769745513189,-0.3239532981927278,-0.0585085828541492,-0.09564502930381774,-0.1340230086075472,-0.3865109810915913,-0.016029970218322854,0.058129604609510215,0.14081620880659568,0.07201903116560272,-0.0728354102698186,0.15828438254907468,-0.018475464233664186,0.020004636273898948,0.06702407378978857,-0.028151427758690364,-0.010862756993688716,-0.06883195089490865,-0.04971355242839205,-0.05879063848460434],[-4.2744910127022875,0.20417203748041488,0.12795338275598506,0.4067506699955153,-0.10394484126584948,0.3955911763564055,-0.46360947637601846,0.4448325252427522,-0.4656051994836995,0.2598064708193009,-0.5448508202286164,0.21896445572504505,-0.45634687208224683,0.1944068520175187,-0.4082876605404033,0.11488387324733546,-0.23696934547687337,0.15591123042747268,-0.10787108804293076,0.09212756614369925,-0.3379061596246886,-0.0008584889127093429,-0.38797718548159293,0.00882317449010998,-0.26390799781044244,-0.02598133315564976,0.07716732655965562,-0.07247264323639836,0.33336193036702694,-0.005628154712512652,0.08848925773560108,0.025310432863507284,-0.20917617297961202,0.01664129046767354,-0.25471063185288134,-0.03282177830222685,0.10209536967631826,-0.05946846370175471,0.1599569876706705,-0.05873237242312826,0.08968430553175509,-0.0033520227497948795,-0.2102670720900472,0.06692040420105105,-0.09973485575046087,0.023868678770469633,0.03963888685007485,0.001469074161885837,0.17349243056241495],[-2.6373593020887225,0.8086200569087469,1.3100751646766853,0.34181935930746377,-0.686636604504868,-0.18657451822275375,1.1197003310717706,0.43002222841598886,0.49984260353894655,-0.5184156460554726,0.32939769018818293,0.17353410806975297,-0.5500912186115088,-0.17274354912473888,-0.5193932313229274,-0.3691966963448849,-0.3848810263895296,0.17969504794230534,0.11337578105159042,0.03678337126787488,0.2465999447504125,-0.2773613485017325,-0.39779315163083095,0.30933393081740185,0.5848034872465786,-0.12005280857958674,-0.4899371238191437,-0.023485839508320682,0.2693604309593232,0.4088033030224079,-0.4654890645394855,-0.010883547345368666,-0.10593237569813467,-0.29103725258091523,-0.1652228390565619,0.13380663830374373,0.5524918003388338,0.27582485833557635,0.25119311918732423,-0.5015287584791587,-0.5606631232866549,-0.10683343182359258,-0.024037573935714176,0.48332360481070497,0.05807230683143209,0.20146912984006013,0.09874889696856087,-0.19034995181400402,-0.25828251937752333],[4.44495600164427,-1.414955691794482,-1.6244030043510478,-1.0949752612917945,-0.7980170667813482,-0.7547461478313678,-0.8988361751485868,-0.5898329637794099,-0.7718811680015235,-0.4567252366284141,-0.72420517148924,-0.42170785195273713,-0.736037420590843,-0.39951430718797853,-0.6544790548778271,-0.3935354439173603,-0.46781689590950437,-0.3559237474695478,-0.4042805433666323,-0.34885597991651557,-0.29997454449526245,-0.21611121695197047,-0.1632969803412257,-0.18569448995212157,-0.1667491809670919,-0.18505817024009333,-0.21404389612564148,-0.1533514786507885,-0.18463914010133894,-0.11285582469630648,-0.12505865640853267,-0.08185952219321008,-0.037369197696790535,-0.07035669259012099,-0.08466925650656298,-0.08165364027659955,-0.10445852495224756,-0.0822803118349702,-0.13192286447611748,-0.05106562965982145,-0.06224054684062341,-0.032375045990719574,0.024892159350631833,-0.013025083160450503,0.00575389227335335,-0.028403150369126037,0.02045884903035552,0.026674427001925724,0.04029041466779709],[3.026366585031748,0.17278364973489221,-2.4129948137337487,0.431723183135266,-1.902559089886636,0.7350751995042245,-2.1653955179663327,0.35864385530438253,-1.7810825643544106,0.18290878240496505,-1.4103481494061083,-0.09125083086299114,-1.018412612132488,-0.12370912522483629,-0.8300843659877409,-0.07223670984827674,-0.6166688121447937,-0.0539840432998094,-0.39147105987824016,-0.09870874981167914,-0.30308406591636267,-0.10935064647829046,-0.2014748953994989,0.018933420160461227,-0.23651566428918136,-0.05495294190393313,-0.16994925259846422,0.13366839346195017,-0.13727637101632126,0.13371555586401843,-0.12007213455225973,0.16791692097827407,-0.24078090624612813,0.14071876997521057,-0.312389435958082,0.04655562840919616,-0.1284302716645099,-0.08603143690434027,0.13029135390060456,-0.14491792023283623,0.16174633137087185,-0.08314947510988153,-0.04535579460911537,-0.017915989758978693,-0.19309585915592178,-0.008828157650974104,-0.0164788873234102,-0.05916624831029908,0.3141410749676305],[-0.9381166514303322,0.16150511774739423,-0.07428830425418205,0.17599978907507077,-0.3028159174808066,0.19158133346770578,-0.2587943668996062,0.1555797445018822,-0.28192419189226137,0.15472797348016099,-0.2836883015640779,0.14168375778512668,-0.2145665605898706,0.109138391334842,-0.232195041130418,0.06454216519842805,-0.4709883611989317,0.089428666025591,-0.23880520139031483,0.1152376113966689,-0.18135076384031693,0.10408982570344201,0.008382944533180024,0.13876898761230266,0.05039595617018469,0.08105826958763962,-0.04983199819682019,0.020695327108548055,-0.07812718742876593,-0.045905424208339686,0.012029770221634569,-0.002862081741602734,0.08108202329448276,-0.01712416441592002,0.021827978746041254,0.01396890577805569,-0.09480032491210043,0.03240894243026661,-0.0893612641745774,0.009009170681855113,-0.00810457899666707,-0.03422300533633138,0.10373865512434499,-0.05272116951802942,0.1929559911654233,-0.009632366716582564,0.09737478496119138,-0.0008850007055980603,-0.006991112428763898],[-2.743917588413142,1.7958340415679774,-0.03451240769351387,1.1922479123938756,-0.01895994287372011,0.6768980553032873,-0.2280709714939785,0.7945916595690059,-0.10625183542740305,0.48751718530292465,-0.34581836757785916,0.5273089439352394,-1.0604557662871332,0.25583716065320045,-1.0481638086619003,0.35438670602717004,-1.3233799908530681,0.08646333031497852,-1.0413359790537604,0.027601215053617918,-0.9502829000716829,-0.005280647316967984,-1.1059457492973388,-0.14790662676637423,-1.3435781735791177,-0.16179461017962446,-1.0366647436824212,-0.03102248707145017,-0.3491616151829245,-0.15796644495034745,-0.04485218781286925,-0.17370537322880308,-0.13400526541945773,-0.06368876809178053,-0.15264802454247117,0.03956181097181671,-0.03273546543358995,-0.046835105715261796,0.4443299022796799,0.036067747117221785,0.42463392655870275,0.03396136489597169,0.0249563261198752,-0.08379226686323026,-0.005762132944959537,-0.06675240614228486,0.24494896099717917,0.04803968500470932,0.15476656992639692],[1.5146607095305187,0.9870549148444926,-1.3332853748558702,0.810622178673066,-2.4199272262389337,-0.276170752134281,-0.9597038734386174,0.029332521644125153,-0.4997426395867043,-0.28226239089139515,-0.37343520435691285,-0.5917012410073252,0.48733781973687723,-0.5630863690490588,0.5010860156776876,-0.6191150996755729,0.6486109606339069,-0.3700812823810971,1.1523514889047846,-0.5471921125585233,1.5023124070790208,-0.8061029770675074,1.7635459250079515,-0.7228005798989757,2.361142987123899,-0.6229681985821105,1.5771257559924492,-0.14292798166868495,0.6745603957691252,0.26083269814680454,0.09132419639933625,0.3820969386452461,0.28078706072438164,0.2841881733315947,0.27261567698231454,0.132917657494979,-0.1711839524493481,0.4085002312003596,-1.060850423113997,0.625960315384828,-1.5735592603581998,0.7224931708350738,-1.2385239310059484,0.4183316772835404,-0.7769738172361594,-0.013588292317486806,-0.34346323585783306,0.047322875242595944,-0.2918006494582025],[1.2757791399122225,-0.914049496046003,4.594001215303491,-1.2587859464613245,3.375928138467096,-1.5162276105041113,3.2932984201013356,-1.0876941447582653,3.2002329523491726,-0.9302426283421656,3.319133148031569,-0.7377462886931492,3.4961060392974157,-1.005406698416598,3.275918909916313,-0.46702361035347784,2.406286670006646,-0.12581960644032636,0.9242758948390692,0.34378224723151085,0.10434847181870024,0.42223797962226073,0.01620432230545429,0.3140033293113125,0.47372303747538175,-0.007061299440032257,0.3092095082401296,0.0779763086955412,-0.6780393366905766,0.23150875223431758,-1.2763068310379828,0.24772721081668628,-0.8835872589234209,0.038926258827870104,-0.14006914153217065,-0.26132220275001966,0.21783732274615975,-0.2639148316382558,-0.14307667844137092,0.0060773383687279655,-0.37229867642608805,0.16880543261536168,-0.1455291976056404,-0.02062508178591971,0.18432861212845966,-0.011912801276803654,0.2562972379236136,-0.036176873049047424,-0.10195083782528022],[8.085995235227223,-0.5979177326821422,-1.3772457398735096,-0.5895755898585452,-1.002844197577349,-0.42600006207997887,-0.659617873979069,-0.4788837972769932,-0.37319968988952795,-0.2845263111047107,0.21091063920005163,-0.298115873909288,0.6276285005309843,-0.36906104101634674,0.6208900543285454,-0.3000962402367196,0.6891558368083318,-0.27334626015689967,0.6944133248279064,-0.1587884871987029,0.6690038024381161,0.04204984537396324,0.7443428127919484,0.34332881836061924,0.6518777595150842,0.27897155553768505,0.4072245329418646,0.16672715718576642,0.14223642946612547,0.05864431835896147,0.13286800161151974,0.23589719414300814,0.17839364235282093,0.13666763140474086,0.038520884735101665,0.09975080395358508,-0.2943940216943878,-0.10434112718231307,-0.44129531011810097,-0.058693020257956266,-0.5563615502654768,-0.12749873103684842,-0.2810700699148267,-0.09350885927633877,0.12418629559309459,0.03050919649598391,0.25988844331052263,0.025887082664545903,0.22852108852560427],[-0.7355399239213751,0.03153855031525907,-0.13693051948049262,0.25557124056409203,-0.22254967769673567,0.47322032973911343,-1.1036533035145608,0.6445156213831152,-0.7236396876894122,0.36590336498855397,-0.6857605274782319,0.28468291862105655,-0.6837329025016646,0.14629361771471297,-0.7857696957572237,0.017252378789477097,-0.3988929304824169,-0.008770013986165948,-0.09963349911456457,-0.12186755598274167,0.3495936524398726,-0.25300393746172006,0.27234881624167867,-0.30913070094634143,0.04544161144409271,-0.1088678225621507,-0.20814988531129333,-0.0315637928040891,-0.16615062106774953,0.009748256806888865,0.12491017016217024,-0.0015926774431435926,0.23321757565708587,-0.02518360382989045,0.02195842113529287,0.018939050174044927,-0.20913675380297994,0.14398505665509928,-0.17944636053360816,-0.014480511855174927,0.008681702598506504,-0.05379744845255105,0.3188662479512858,0.004051402493287386,0.2072867482584031,0.04359120008252137,-0.11066033060684985,-0.013446681088040648,-0.1409512432382954],[2.960639474845399,-1.3216589411632143,-0.42145334563160336,-0.5134142866237839,0.41725397113582124,0.03208888508972469,0.7960351435895777,0.24863352044621057,0.8609581801073004,0.48750600437092884,0.6332754468834912,0.4185092089460732,0.37845710270408034,0.36037374113274157,0.3733508890001288,0.6022790213431782,0.4914922727122421,0.5917845002993074,0.5681399912633641,0.8231730995364547,0.7568178224274714,0.9225491855335001,0.7518852420764756,0.8527371887081837,0.07684454957744528,0.29428248026496934,-0.4480960163342215,0.055533844229934834,-0.5164697447893011,-0.006660312790048801,-0.1069069224207299,0.02199059916071229,0.10808177806287383,0.13385374060292277,-0.3941020527115926,-0.2888775336794942,-1.1584587933323771,-0.5835762363978637,-1.417895936460332,-0.5662135001852032,-1.1389098430474567,-0.32136754039938076,-0.4993561117376843,-0.2782755095394103,-0.2302159181752262,-0.284570083073445,-0.06669457036893768,-0.3223904394228584,-0.09538714754558782],[-0.5212177120429132,-0.42718351597667453,0.6012102042819816,-0.19600437601160525,0.590309669490609,-0.0034321108036660213,0.8792608871715387,0.057736324466777116,0.7102388305822365,0.3151315746295639,0.3165047145508152,0.2983703125359412,0.49300395632983046,0.3744315400033413,0.35843219192676934,0.2707436171812342,0.2007784306819226,0.2215970449881074,-0.041659049349784656,0.10760609657558438,-0.031965191788627396,0.11788270248495934,0.17851043856438076,0.09359713161844622,0.15495185177162984,0.05435461021567668,0.20672148464698958,0.0519350787781698,0.18673165809023834,0.14738548718818995,0.2432017736414491,0.15332139986558121,0.33233814288383173,0.17620635161956366,0.15372776441351424,0.05616223937812135,-0.010057216085773703,-0.013019951367797154,-0.16517920447356263,-0.03290600115532319,-0.21824100318090717,-0.06410341978260564,-0.20544445854747864,-0.050919545400007475,-0.19172979336310983,-0.017249286263887224,-0.11525086332834011,-0.07190727416189856,-0.12465659592268732],[1.5131528140284651,0.7167443167489975,-1.634328351022402,0.8732089902983781,-0.8537178133711089,0.7253123761488793,-1.1215900977310176,0.6884608081115373,-0.6371484673875019,0.42277471786457843,-0.5170333096249325,0.2743631547994124,-0.5259905298675247,0.17022736578748543,-0.49007965633294415,0.18310277214088316,-0.4621976281064186,0.059409198678636976,-0.9126488323418236,0.07314237114392025,-0.7408434375867675,-0.04427814778427087,-0.1872143691581445,-0.11565913085341675,0.35160576864104803,-0.1327109362394548,0.5061745430690479,-0.03636072568628332,0.43827589834908215,0.03187260571507428,-0.03106740278530518,-0.04443962751709565,-0.26340187785096414,0.03959481451493126,-0.0005964789842875519,0.04104480622814263,0.3890554406773967,-0.01580262737320268,0.42142597275600613,0.030726825328562114,0.043694697703462376,0.03314566871593619,-0.21637223846077278,0.05585587611464962,-0.40328061294086726,0.03632596405907057,-0.19644030001289536,0.1255323494441631,0.11469023138408652],[-0.5870694496198436,-2.0605472656029025,-2.349460864532528,-0.8760930357087804,-0.15004936909762914,-1.0038693465506123,-1.81255622275149,-0.3234375306330116,-0.768663955724619,-0.6833987295751498,-0.5612144529929054,-0.8918761939390711,-0.4820739562641859,-0.2643593522117008,-0.25268348574314276,-0.44546910021043445,0.2205240206969013,-0.3205084050217249,-0.25643234164198897,-0.3315937270150297,-0.0011244939172640987,-0.011847109235243354,0.03515913029581282,0.07086129931864746,0.1526367240236143,0.2545306566476098,0.3654612546667339,0.11458384565278262,0.2877966869214117,-0.06579656990051796,-0.018292686444889017,0.12395922082208254,0.043216733816496565,0.16536547223395476,-0.07356500262516312,0.07148901243860617,0.09422222650217955,-0.0678265336948993,-0.04481866461742154,-0.05764670759586694,-0.10708679167844325,-0.04478810263902582,-0.3076521209413153,-0.015901086581001683,-0.03852345247351962,-0.06181813859729158,0.13360124719986227,0.011872853984637239,0.10307145779294777],[0.972312166397483,-0.2264023417441958,0.5342978240300829,-0.015387938496774086,0.817173021237253,0.3197161149912152,0.5277603580996862,0.26094257422474276,0.4187888150209022,0.13791365147869963,0.5387713035332045,-0.22024100193837043,0.9043530982438016,-0.1690354226106296,0.7286161374452309,-0.4437151924832398,0.5215495877276379,-0.05020600027092659,-0.023758457241594942,0.16684801690669757,0.18936723325091967,0.21643335509567094,0.5613854554850888,0.017238419052881763,0.8649690151303646,-0.0006837245482103194,0.26738448670164305,-0.038842643710730346,-0.30229751818176565,0.22528758890401868,-0.42769939890757197,0.19308063850203908,0.25024248633295176,-0.07328435214452408,0.48142641580595663,-0.07685423585619133,0.2869402264013982,0.045376869139829086,-0.3337692388083751,0.1431585316071974,-0.27149716059255175,0.03103466315113342,0.14451229110245256,-0.07212135715835756,0.23418151968474163,-0.157010246454904,-0.09694958141897637,-0.01150531190600815,-0.2443693912507581],[0.09742952871927392,-2.507847738064085,3.2116620083820155,-1.49571807601983,4.695433988044682,-1.375413009592449,3.1608120859276427,-0.6029754403032,4.425674124116793,-1.1010695772445385,4.765029136380866,-0.2865067172091268,4.427566596517209,-0.2522605880389927,4.400581902326381,0.4722473131185655,4.113134716173285,0.2908088741014187,3.418140229931081,0.5349264015882415,2.632609694967371,0.8660699516216844,1.5029261465273527,0.7984835213587612,0.7887959938138224,0.8329387708689209,0.10674008321525495,0.8977760549679379,-1.0282103309328834,0.7400733059891017,-1.8973338847507109,0.31276987163258296,-2.0050417222840924,0.18540776161501746,-1.182438245423843,-0.0014874093684597545,-0.38703841697228036,-0.158703337254277,-0.14001737821514715,-0.34150531117122307,-0.5469484724275976,-0.2137136153963962,-0.516287486529684,-0.31275498581582517,0.08707001612648659,-0.24337567514739716,0.6179555752585771,-0.27094347617458797,0.5997142608386775],[-0.7273585879207344,-1.1302404175569367,-1.5130976673540033,-0.8963807996934975,-1.1913030512464216,-0.6401676289724356,-1.154806950387579,-0.4235005583842655,-0.9930596929701584,-0.2717152283577414,-1.042264133954812,-0.2596125616898854,-1.111713488069221,-0.21802888220248104,-0.9578344990920932,-0.14655146712884665,-0.6886878371573368,-0.056954411181574155,-0.4428288893510144,-0.07218562247924862,-0.4252130117623271,-0.19718513254056327,-0.43291622660574225,-0.2621039481677356,-0.37516632845485787,-0.1986369739791332,-0.22249350789901953,-0.04996449083551003,-0.05494264477686431,-0.009910854727461636,-0.017840584671895053,-0.09284407863243634,-0.08222793198517897,-0.14900628650769307,0.0007748708686726882,-0.09770988848557433,0.07970103680913443,-0.028525941118084805,0.08593787156704477,-0.04243455950472395,0.05511866621063144,-0.008108725506083976,0.07012343938540555,-0.017093945598620295,0.05806536923064749,0.0013641205624634737,0.07494035798096976,0.01798369073796183,0.07470838803093177],[-5.109411052334576,1.4049979434921884,-2.3260760984263364,1.210768146676758,-3.0266500200858424,0.8369214810020699,-2.1774276868263427,0.7034753657666646,-2.552329126777525,0.6017105757864039,-2.619678035333637,0.42242799306567286,-2.019269131364436,0.13665822063201977,-1.4120152679613824,-0.2124007354708671,-0.5953191595633113,-0.4937072996158533,0.2046517814833856,-0.730741031648412,0.6713152310052743,-0.609695464793679,0.836021894880981,-0.5096902711862893,0.7596026307802796,-0.19597810202533966,0.5108927157242285,-0.16099964484689822,0.27651017858063476,0.0016772507997305749,0.3650018620108933,-0.001230962820770041,0.465485602146961,0.040973757636571205,0.060141416660391016,0.11480793748613433,-0.3202821901547007,0.1358901640587625,-0.7482234048939777,0.24556959504104323,-0.5245458042250009,0.14147687618918092,-0.19149593066914586,-0.07702955344558651,0.10476028925936387,-0.08823175939440848,0.20619043038000726,-0.07124895654838674,0.07612035326508751],[6.586573356418888,-0.4795897732761377,-0.2000631595133443,-0.4315245203913233,1.2315841601325666,-0.3197546420967711,1.8992144320960769,0.0231917949764571,2.511321303195367,0.027120732747833266,1.9788913390658214,0.12196188023654132,2.1068082764097116,0.476403546518838,1.8686232859286147,0.48658267290877205,1.344539692304677,0.4758062497735674,0.8769966472282109,0.46226625019954465,0.5323466501882222,0.4493964775380255,0.749456442452371,0.5311190214685989,0.24673595622929106,0.3522178341204955,0.24299972011312335,0.47365917573476424,0.09689359225234626,0.2803539946220384,-0.41601329632337736,0.24476490924966268,-0.9794064988488695,0.015465135495048798,-1.27060219429306,-0.18766373781287946,-1.1570026562168576,-0.14319740163694183,-0.7345135509326549,-0.33999723611374477,-0.7328304528664351,-0.3092065600147036,-0.8446496822889142,-0.4485606948885358,-0.6075210734651147,-0.3909807858494484,-0.10191770918008912,-0.337357871076854,0.5212619656896307],[5.161314623777404,0.37452825645284343,1.6827635401119605,0.19148450327502445,1.2113703239167783,0.20521735447057843,1.4542312733848168,0.14688277179230824,0.9447938089291082,0.2521406385747004,0.7520932355144555,0.18835119613322218,0.5833775661774868,0.16388268129174255,0.34015210432270354,0.13135888922916175,0.14622262645112538,0.13128188186380255,0.1415897756553024,0.07349716813050482,0.09042972828042167,0.15114215821084087,0.024982338540977766,0.07132145340276966,-0.10413259233517716,0.05958609413978201,-0.2292885489716827,0.006076028326962857,-0.2547793558148204,0.00955751785023655,-0.09702337178076488,0.027784818472346946,-0.009311479010921843,0.011317430482207902,-0.05217219256490241,-0.03103585330345844,-0.19269146135525717,-0.07042988230980235,-0.12070906451336595,-0.06277837760969175,0.023032027555283524,0.03433070832467538,0.12163353155946825,0.021695307355385864,0.13696655742228842,-0.01334534124757897,0.09336130907208537,0.011761696492179374,0.11850829371212544],[-3.2281757864651293,1.2470813283199718,-3.770120496709246,1.0113639164197366,-2.981245996961508,0.7823305671961169,-2.347100221792193,0.23048823658894665,-1.74006896892618,0.12450378810058568,-1.1605384806479024,-0.21776479467457743,-0.6046567830099975,-0.3808313922041612,-0.025158616183886345,-0.6917703492843236,0.19411099452875968,-0.8007363836476944,0.42277336843075336,-0.6814006835210834,0.2651964757726109,-0.4912392347137044,0.008568699511103214,-0.3450033625251688,0.11056919261065937,-0.220008311429523,0.2721521030823079,-0.11513409630164237,0.2707557395048063,-0.12577154006080485,0.08728618304464224,0.04041514660155081,0.0004209441532626047,0.07765865745548885,-0.1181612361854435,0.059116677721532354,0.0052193194232833555,0.041982969968101705,-0.023166630578204433,0.03835232420319603,0.034716345221287366,0.04576386498371172,-0.0028208766152895215,0.03424658359413266,0.04721629056785645,0.03723153394362026,0.11691993972755022,-0.008258421802892263,0.12516760733934965],[-6.5730861638312,1.7367184528458899,2.4170051491820277,1.468367740244438,0.5623878729209362,1.5698374323778075,0.9836512188602935,0.7828820958411615,0.04126643723209622,1.0099501090410097,-0.22139282291710824,0.7792888154461375,-0.48830138376246146,0.49110668225383414,-0.7980517819493533,0.19113306395655175,-0.8432857335168653,0.06156093182279676,-0.784553791289853,-0.0267433568224748,-0.8642151541824182,-0.3336151382862885,-0.9402037302635037,-0.09171960235932719,-0.7683117776032223,-0.17721207907096057,-0.6124184758525613,-0.1807683581334153,0.13176535182438762,-0.026987485658226944,0.39284516932875085,0.047462092676197616,0.35105134925151815,-0.1276509551189712,-0.15328598306544197,-0.07383554167537808,-0.05313302860683533,0.008802330269675083,0.28325787067029223,0.19549166244541505,0.4873463163690314,0.1234951951130418,0.3084480734908409,0.00823921474922048,-0.28991423963151153,0.10885356926595113,-0.28865282905091005,-0.018408203116614723,-0.037532938087902964],[3.585162232555492,-0.4331482363405996,0.7497617184168528,-0.42591016592125985,1.2753702852075741,-0.2466657037584643,1.0825205015867259,-0.040624735487131726,0.7945838752277374,0.06838286295778565,0.2777631467545374,0.17562722717137946,0.20819782859645058,0.3764187638942948,-0.1373873815471335,0.40656489045580974,-0.2272577574944223,0.47038487061993134,0.04135067856152962,0.2817977603315553,-0.07112138491948727,0.15474058141108948,-0.34688600453890794,0.16739885296501292,-0.8420317021257866,0.15717662529591966,-1.0376786928525654,0.03615630598005187,-0.7304253995305497,-0.25967543677031824,0.04846858812654802,-0.2767071848115154,0.36846039358737315,-0.3452216026858026,0.09962696104732192,-0.12659853978612054,-0.24878780370172068,-0.01141880927607836,-0.13946547027808304,0.0032688942835113676,0.3934833863956555,-0.11769121840732549,0.6705494205611309,-0.17048675437688177,0.6888349352241281,-0.020698317022644026,0.15302040295312094,0.15183009329208683,-0.05793614925636155],[2.049523207400895,-2.033620860747415,-2.052080597565464,-1.3653071919370066,-0.29884094745735534,-1.2997946634128914,-1.4808494153149825,-0.830683073881189,-0.5767654571100739,-0.7912817915960835,-0.6833963451033034,-0.48372323284083485,-0.2960387269926689,-0.14447297810152027,0.3066168072776222,0.34324584984423373,0.6947816551601287,0.5127628509616292,0.9370861009823656,0.7641847426788129,1.3286841210966154,0.6801529413746892,1.476648577858878,0.7136212274697815,1.8523367266013326,0.7902163201893234,1.496884410007626,0.7021192838051932,1.3054044044553832,0.6252528825888488,1.099240583892635,0.63748185788341,0.6490334577187385,0.29986675543537006,0.09781179305101032,0.16304577443027204,-0.03061851796262833,0.15947464320772523,-0.19209226105248278,0.11011664863018489,-0.16602092620504302,-0.07535111636975687,-0.16563945254910536,-0.20852173894191312,-0.4183003786524762,-0.1631058910960847,-0.2719900350506907,-0.11437853136333516,-0.2334364071933898],[-2.518189277674254,0.4105344560418576,-2.704744712475217,0.43544201290930346,-2.578987429198419,0.4791763550414369,-2.465629429285356,0.09094034923567602,-2.0436645461562177,-0.14830645104675313,-1.686800730769651,-0.34617789100467083,-1.6144060939532479,-0.45500429352918786,-1.115100922102596,-0.23490813090040882,-0.37636129545184416,-0.1857520690280903,-0.14856508414548014,0.03467144386681816,0.28417708220014715,0.0006665726496509558,0.1069382300014677,-0.2130563645719901,-0.3597813237546394,-0.0872464842912048,-0.6283357074685327,-0.026341989353863678,-0.3477477993399869,0.24502180359895706,0.26432753692462563,-0.042553204831321946,0.4807317660944543,0.07195870594923019,0.43138711659560824,-0.17173575681984576,-0.008563011464710711,-0.002457368748752907,-0.2890588901441614,0.028096673241952565,-0.37993237707998956,0.02810575810727296,-0.29051672277399204,0.07151909143262272,-0.31988972232324264,-0.030918195671365126,0.18135990769887472,0.020300635977589285,0.2602971177904153],[-2.0367812212527383,-0.15511011537932315,-0.16591693888912998,-0.2625039869637706,-0.3189507236383732,-0.22695714077136517,0.38737015082008613,-0.2622513670779734,-0.2283022275589961,-0.39154262673348095,0.15770134353906268,-0.013878131265335726,0.5610256230093671,-0.17613127888525712,0.33660722378072855,-0.2511645779423509,-0.09645458511183076,-0.15851759882748634,0.01561007639573722,0.1378719381440777,-0.028810084041787636,0.05562427967180601,0.13816077252092993,0.14931960567067376,-0.06766701342254196,0.06122536379049416,-0.07834704564549132,-0.0026478094866645378,0.059512816961120645,0.08076512622640399,-0.08329189263144429,-0.048737922618651355,0.08392290378086856,0.15429397658266766,0.14447381734267478,-7.27897851469479e-05,-0.09231598769392263,-0.08566311523931222,0.024980516305276323,-0.02738814455749773,0.22608981295231045,-0.04490011038470263,0.3099283221432398,0.09802375125500912,-0.04684382304252863,0.08718794322077995,-0.15425202118094553,0.019687633665672197,0.00880302728506353],[2.7575170818772605,-1.9053324441294597,4.643051315246354,-2.058700082048102,5.1809337960974196,-1.9702566539365995,5.342508824205649,-1.9276554359389957,4.972958498250176,-1.6600781489102672,4.628510152100778,-1.0891245935241973,3.9180364080140926,-0.90988125148505,3.1904518156457495,-0.6672241814104748,2.8537946253911763,-0.3321599938166434,2.1375848824859194,0.051529892617507726,1.3066100826816671,0.6315104718643562,0.2912239117805029,0.8076276886716695,-0.853964084891812,1.1647707653699724,-1.3149744046103797,0.7658503962191373,-0.9946540508313791,0.5795882203339645,-0.8486066719295311,0.37187999132420524,-1.138377926212038,0.3909988623257927,-1.2415653768945418,0.3911273039484298,-0.9664087261710634,0.16644705107959232,-0.5937079660479327,-0.15193470058145953,0.005033421991485554,-0.4081331952486605,-0.07263447047498434,-0.20482819004329247,0.04794567586090654,-0.11344050533293563,-0.04000124425779106,-0.21917870749140173,0.028794600470786063],[5.392350237047573,0.40051750308247563,0.3075833434074778,0.09661161787082057,0.11316559923797832,0.1543846958225609,0.6117834396543324,-0.20492744666171114,0.8182929107267785,-0.4155450849167823,1.205375550353745,-0.5617780168198928,1.7490555470647855,-0.7544547684767006,1.7388261564337133,-0.7236488218660885,1.7810468581300858,-0.4732712188100444,1.257365979070018,-0.16632203683475916,0.584581064625405,0.20959309163136897,-0.10352174054701442,0.32671544599732655,-0.6106000756486873,0.4769313455984634,-1.040947116915479,0.4590380438298705,-1.245980027164524,0.37391447289145,-1.0978651883142572,0.260748902970568,-0.7332180920290553,-0.004409524243302296,-0.31278187536776486,-0.12028437583935199,0.1252522371916734,-0.28175115279006796,0.20643296880815437,-0.2487375127266469,0.23547523171304333,-0.15084800893127154,0.1446810237706203,0.007292037700049945,0.01861257582711353,0.08078468683457164,-0.09362490170181613,0.05066783161040215,-0.19099801656800625],[-4.372150202663823,-1.520245810459882,-2.5053120331353926,-1.2653095701317665,-1.458611639800674,-0.5190641034166691,-1.118811203385977,-0.9076124406206348,-0.7336639040131909,-0.3298446076121486,-0.37898928119593894,-0.46862656380246764,-0.019248116268573805,-0.35844177593338544,0.4205393851390748,-0.20946813798641734,0.09517598991664374,-0.05234571694915927,0.6095251508982812,0.020951742284977225,0.5084632497261012,0.22380793426300652,0.542640241190739,0.44323200462539003,0.9011921678211947,0.2985829118994494,0.47687433910296384,0.07590944643500971,-0.20932372915814448,0.04829292603106227,-0.385074552703078,-0.0174348818028828,-0.2319514791978165,0.05280168295310498,0.08773262506186472,0.007769204327336212,0.15664564488399133,0.023335999079748874,0.033820678335392484,-0.19343758960954183,-0.17445416373835254,-0.07857761612465752,-0.2168647195773661,-0.1909148033874493,-0.016510370392169283,0.05918766539069442,0.10731030544110501,-0.014514549953494467,0.24385084846716204],[-2.956616522786043,0.8914717143490687,1.8328527953521176,0.4475945013538907,0.25213111007585126,0.7415883335651193,0.5914832404202621,0.3819926109286825,0.093984227694433,0.7113172299668383,-0.30479040583967126,0.8070117126970289,0.10229464618528139,0.39228463202602915,-0.07853336296737949,0.10357821343929294,-0.0732038596276762,0.04487812007191142,-0.2925872661130216,0.17621983437264852,-0.6681525163939352,-0.04261490443420661,-0.7177807116585939,0.2951008790964388,0.0431854907765419,-0.17416710184257023,0.550858759558924,-0.2900424864640084,0.8402920113187368,-0.07703764092629845,-0.20901219354003725,0.10530300968767546,-0.8402250820066653,0.2908104300371824,-0.8698699286503828,0.2475505588648298,-0.14030363236374366,-0.049755129370174934,0.45492783631938877,-0.18897713850751008,0.7039554137509341,-0.07960030349701579,0.046065663524314814,-0.09250705262598446,-0.34235901232735255,0.02352026481481823,-0.32232002229772927,-0.005596187199447617,-0.0017702920596704749],[1.4433706879463233,0.4184387202463014,0.9931952016912521,0.33168434232857313,1.5309520221750377,0.07440405932814335,1.218434310426019,-0.16515416761594046,0.7065080990939171,-0.1902685211568767,0.6107086172229552,-0.2116121221336443,0.8787374665744083,-0.21059546787255207,0.9209296547231949,-0.07646177946942107,0.9348854236575296,-0.0029771493245350253,0.8993932265101233,0.0024523791517359183,1.0475083641944132,0.2423470874256901,1.3366761438897583,0.24732132487832081,1.2520345131955366,0.15326659833778694,1.040357189397777,0.1964281649610525,0.7819466132794496,0.268653159694086,0.6940131709977897,0.33501359819449505,0.4626749855078854,0.2612669027886969,0.07210438767392752,0.171711157877103,-0.16598484879917647,0.08926197222959356,-0.19690561539807164,0.12361891065777579,-0.13374829251949363,0.0694590771551478,-0.3131833608711818,0.03245384469531337,-0.3930847074839618,-0.018891690007365042,-0.3212528495917693,0.007107060130128042,-0.10611142533974875],[2.869272298354015,-1.1328547476418886,0.4620630375795132,-0.9353048353538874,1.7611659638000714,-0.7907930250197528,0.5327470634038405,-0.18466264836780918,1.4991025737365766,-0.4398010910632294,2.01637409417549,0.18756206462744102,1.291857905812544,-0.1904661949248185,1.42771112695409,0.07677194980936945,1.1059680238876461,0.03207664831951983,1.3806846285008203,0.21773215857228795,0.9566320544788904,0.29371237636673125,0.723138383077196,0.21110435248586956,-0.13949779019282899,0.1748376582590973,-0.6436541976089436,-0.06020940214835677,-0.7911504977631416,0.1783573042022495,-0.5337214965476227,-0.07902293196651737,-0.3243976799544201,0.08898009764789871,-0.09414258981400421,-0.09753492616343626,-0.374925475223824,-0.16893084119674553,-0.5405607813052035,-0.1166677584200716,-0.05181937848650512,-0.18630530995023445,0.25906863261706686,-0.04607894529190864,0.545302632972675,-0.035375587659670145,0.35301833076585326,-0.1025313840622433,0.03974828394710856],[-3.284484810199205,-1.05380600254658,1.8722661458166263,-0.7590043710334277,0.2641679551108801,-1.1661306899696569,0.5991077384712952,-0.6157901908462649,1.1498774026780398,-0.8365022140156555,1.098317048488205,-0.5059068040189701,1.1474284158376795,-0.7877175908605331,1.0008022197685178,-0.6309742877836976,0.9994507026850384,-0.5771927556034997,0.849643625695541,-0.4759009722727016,0.4149581801559369,-0.1253586653046032,0.4606045324651735,-0.1839099260391329,0.40483834349452436,-0.14118753165037198,0.5399977799943577,-0.09249053947584235,0.20780556118788757,-0.11400729161979684,-0.00889114138981854,-0.005546278096227164,-0.21049921765515436,0.11331087824566136,0.15801010212248057,0.007557370138058992,0.3540246801203996,0.02819583967416891,0.1765082099257713,0.017663245951208786,-0.21507344832521602,0.054958349085141155,-0.12323350978970735,0.10904356010860675,0.041096401037357885,0.1595582079179318,0.13917819118909341,0.04990602153691796,-0.18527210589023013],[-10.953232187922922,1.0474567032990791,3.104115015481553,1.3525557112156787,2.025207758070624,0.6964987098472187,1.7559233156636307,0.545211606002744,1.3554535955465985,0.27515925670123353,1.088363027158363,0.16609405662166446,0.7438615209667323,0.21238636042547615,0.3937310208653122,0.5541546513253296,0.28128054193037144,0.4677794999942926,0.16336852627034112,0.5399529894262268,0.23184227713277752,0.3999857603274788,0.14200544462017947,-0.07747440012056907,-0.35302598692246934,-0.09360944447117323,-0.40213426144667985,-0.22618144393988113,-0.8469452415760825,-0.0021119627560659453,-0.3140001027539434,0.13155425447702826,0.057961981955545396,0.14811899526790473,0.24012919014846018,-0.026832128975601156,-0.022274277788777684,-0.2216233214756384,-0.08959738466152378,0.008517982581276467,-0.18387084757293706,-0.16992957357850857,-0.2783267198272994,-0.23586239195268097,-0.1625745858062792,-0.05550766196749285,0.10469172325970272,0.07203170686597996,0.2843923619589491],[1.4332772769463438,0.39064176043618976,0.6660202720355082,0.6927404147299825,0.31816701685544685,0.6637453136658072,0.13565697137891494,0.3787401012773302,-0.04486953098365925,0.4183015439570635,-0.10717021049029413,0.22514041355649272,0.05947732045636239,0.1845786281900167,-0.2470679953714523,0.24060097260362923,-0.10239731857939031,0.2032808888603668,0.01706174745610839,0.04352309200024642,0.17660823651677496,-0.005756490877987722,0.4331362637696259,0.026231218653582463,0.3093851530518214,0.07338912510670062,0.15985983928580783,0.13072243209045495,-0.05331096757653134,0.11461606089412532,-0.10999363146045679,0.051295379101867766,-0.13857832807439643,0.11044973449941221,-0.4571546042211048,0.19046776643328606,-0.6254610567700655,0.04103141872476562,-0.7949320246200047,-0.043803956070055675,-0.6545581901195728,-0.1230287659898451,-0.5748920661758211,-0.11644656580111724,-0.6831988533965562,-0.11506651004479292,-0.6613681459598352,-0.12029583001529885,-0.45118478420882124],[3.2726227473026315,0.21846640735355002,-0.8025579933719904,0.18055924500411313,-0.15973907195587023,-0.14238306042820434,0.6091227807126228,-0.5457836181512359,0.7140511095251549,-0.4122229286193659,0.9136891605038574,-0.30006249902184945,1.1244410681785506,-0.1567846446833419,1.1472578171862853,-0.10955413094090223,1.3396019154730388,-0.07178657714759225,1.4654935848549993,-0.07785562876878312,1.1775373733149697,-0.012162799701103228,1.24241145080329,0.19846462047435376,1.300626569897599,0.194290695036521,1.2520606830955312,0.21975897372838762,0.80405271363827,0.2971718051116885,0.8992883428632322,0.19424434187988532,0.6375078519782479,0.3134620565139989,0.31645715278362746,0.29375701226396395,-0.01794780580977506,0.23697502561689623,-0.19413708982680644,0.21869233881986153,-0.2769680941605097,0.17991754043598643,-0.28788600772419143,0.11988420223155907,-0.3688035547911582,0.19992723362756884,-0.8742255880509908,0.06634147933087443,-0.7942465394785458],[4.254849274460738,-0.47618569849718134,-0.7595641612729115,-0.44952364523294325,-0.28347635257365933,-0.3030130532509153,-0.2079263864387269,-0.21652930721893618,-0.10894799281091001,-0.16321662840957138,-0.20960155174698583,-0.18112352024680414,-0.2825728694624353,-0.19706611262053517,-0.2750339217955153,-0.09358547362509131,-0.25106802407989254,-0.017259380758657496,0.06676932152300587,0.07057050612467955,0.09527158473211358,0.062369113435036656,0.03112156247594408,-0.02418661922931478,-0.16667961283325314,-0.034147645258615855,-0.1993773137875069,-0.017196317051913733,-0.2511050618287911,0.039568294184096436,-0.10118429475023984,0.0657235872133408,-0.05512511912606487,0.02653169864455788,0.0689231507151871,-0.034293859923294874,0.031715353647861315,-0.016054012764893433,0.004081391497547845,-0.09243433245987068,-0.07369088173048542,-0.026487114352495303,-0.07471014590191002,-0.012741704858210317,-0.04231525839019135,0.009189972222795537,0.07091216269006807,-0.009971475903077167,0.10767310325386463],[-0.0730383139278101,1.437225819275336,-3.9410499795029184,1.55419333812536,-3.121600045438072,1.1078023196504987,-2.721352942882527,0.7081412410364731,-2.001008046697779,0.5401186403634467,-1.805042914479182,0.1587946093635555,-2.226421268862363,0.2903094375350639,-1.8286091446335313,-0.02923047125738349,-1.290301458772016,-0.28833069567650826,-0.41153834429166997,-0.4106526847655497,-0.2039226052292833,-0.12488398439510236,-0.33681676049883874,-0.22761815471819413,-0.2559468969581179,-0.08452309306759377,0.11070378605097535,-0.12224503114849894,0.5419603344470534,-0.27524830933718986,0.8155110944042776,-0.13265091849311977,0.5556514009226647,-0.0678880458738559,0.4086913882781689,0.05985834402317193,0.31492588810310645,0.12743595656441373,0.3578711678132197,0.05751014495863658,0.1839927326831064,0.007273274430897838,0.10817889807085712,0.03866085896094524,0.018875054333608855,0.0426215073422777,-0.0603950561789682,0.08991273543061529,-0.19850722707796084],[-2.311501749650009,1.8265404275328738,6.530898508124793,1.339343486380736,4.067794895481558,0.40321428771861856,4.23176261685572,0.7584297421155972,3.409745328219734,0.7713648870367457,2.628308462161093,0.8158909481044123,1.9015670162460576,0.73240780121345,0.5999573560562111,0.7868141061296321,-0.6755231579508126,0.7677175747655559,-1.6646484181766599,0.46137558513791205,-2.2686478205648934,0.27988583842931236,-2.2925128738528944,-0.021030163084560637,-1.7071580638948476,-0.22916735960619777,-1.526425236636492,-0.4261854187404075,-1.3191299212547354,-0.3763714330744863,-1.1089209199762844,-0.5456383941878694,-0.31970403865586317,-0.34279124978888537,0.29725476302823844,-0.34501467342311787,0.843000152578713,-0.12236453093905393,0.8998123513420638,0.004829827102254728,0.7074539862269904,0.19167458958574868,0.6234977973247752,0.19370533026856265,0.40867097802565544,0.09161359340918104,0.08196671492508496,0.1111940183529968,-0.27360494832092996],[-1.209549397579315,-0.6257023340013959,-0.9144905526922897,-0.07108509656611174,-0.2141892101147115,0.09372955078958854,-0.9418062890990339,0.44946261998699555,-0.5224500473728891,0.13088115635059475,-0.5818683028785422,0.08398263999523137,-0.3117665253457959,-0.016188763418866624,-0.21916254116673825,0.19548419529449312,-0.06782815637560331,-0.002621445244076443,-0.19651563610002593,-0.11046368557368455,0.4693484628881369,-0.08126458786027028,0.18489999432965656,-0.05321149783865978,0.28745985247801575,0.04933493043423548,0.20319331580943062,-0.04577863822353926,0.28345571306234324,-0.12836048110669412,0.08277496610944596,0.05864587932165238,0.17807887377935286,-0.01529882149823299,-0.13067414385956316,-0.016053225051513777,0.03534196687345682,0.0892706670074709,0.06364667655380467,-0.05645140260628064,0.23890708635083838,0.04982654273629347,0.05602992381328129,-0.01672636258316175,0.07196881936964057,0.1079278510710798,0.13495531739807531,-0.03793160398316819,0.45931078426929656],[4.9283455691108005,0.1576318564778383,-4.421357643418179,0.009215340511762213,-3.490548357392322,0.09806634335538614,-3.283677164085336,0.1116670462570666,-3.60063308562917,-0.03028774715226535,-3.4083915123476616,-0.34805151839200693,-3.5816111808668016,-0.28032083717322526,-3.2125734608048786,-0.8941861366733311,-2.8753148222196154,-0.6998089530076809,-2.3830259226109005,-0.9332025251663408,-1.7111344279131773,-0.8885649192467946,-1.1791886504959053,-0.5724806393164755,-0.378431501625147,-0.55185963494215,0.07911202941540459,-0.40749218065377685,0.57869602603442,-0.330877064643907,0.2990153025739164,-0.26791576015403923,0.2611835244798851,-0.14227500536700977,0.34662336344567307,0.17621223321403562,0.7074600972416174,0.35062503067896184,0.7069733527778274,0.2411787462545512,0.5200896713716958,0.022826364222693966,-0.008240394661482013,0.10546540227104219,-0.2673566431305126,0.1161569358830084,-0.020357141119471144,0.26154993530554105,0.2721275258067129],[-6.077803530537253,0.01596442465382643,0.8109977633411932,0.0639516119912477,0.5187962807903084,-0.3661903794009686,1.0860256599291196,0.01830467256624483,0.7465433006634754,-0.1128380370337156,0.2645806278038148,0.11676966509855674,0.38218557888113686,0.15467747095865964,0.46779205971699317,0.10812882398179452,0.30154741748876246,-0.061687702935806873,0.2237041883172875,0.05655014761601488,-0.22998147084234602,0.2728963634796953,-0.25837450145465046,0.29028427499176446,-0.3209549646207116,0.19829924021405995,0.03208561398444792,0.15292886029987643,0.12639989753188066,-0.06545726688919015,-0.05882052621483676,-0.06628234335542402,-0.20175022380565288,0.1576077760502058,-0.07514681025220428,0.04269864911782155,-0.06692100260059027,-0.08373985993798412,0.21824683333347739,-0.013722879177946382,-0.011215640851025424,-0.07072429369272595,-0.24055055570928038,-0.00505441886307119,-0.21714953475605303,0.037571743477336585,-0.08990947457792882,-0.05047034496946241,0.05054318075598455],[-0.7884373255328471,1.1256257589949619,0.9920815651297009,0.6628782448850649,0.547795256555382,0.35283082182538295,0.2991070197380583,0.4021813835854581,0.10997290158190277,0.29563696869472433,0.13730944830191738,0.3440227043287745,0.020832435026961763,0.39996177645468645,-0.07071696270082178,0.21111017272815302,-0.23769881734161943,0.20482215149864494,-0.2900430695195045,0.10689469063039912,-0.4208503000069429,-0.0043967828733428,-0.3845537966596258,-0.07085061558851631,-0.24264290371641126,0.046481586971478236,-0.24469726780811582,0.002767629145385131,-0.3413980127227798,-0.07847044825001985,-0.4821719072727065,-0.10793518169981565,-0.482313320207129,-0.13385400513605408,-0.23098037472930155,-0.0592982734496149,-0.012412557757187562,0.04430052655024089,0.09348064380537624,0.05715647969552122,0.12966352510739415,0.028414515008590133,0.18785046154080126,0.06611141266990157,0.35533259748318113,0.07892158129687254,0.3518586526320958,0.051595664654315365,0.3378066244991904],[1.290880740376655,0.21965959414925135,-1.5975498800475234,0.2619061225996072,-1.2903440177925265,0.34077131153656637,-1.873818044330808,0.21804992480676294,-1.9787826259638515,-0.2435560424946288,-1.8245491087230965,-0.16474798978775568,-1.924204228006114,-0.37081613046619505,-1.711326922465304,-0.6810395076541537,-1.5987643995334078,-0.832375413745779,-1.4542327223879716,-0.6604205568967693,-1.2770047881308622,-0.7313108521164691,-1.2193275014737788,-0.49191621913991973,-0.9384813467108643,-0.2558599853672434,-0.9340266231447872,-0.1908291226652776,-0.7177994007108568,-0.1056097847101069,-0.31484960817947527,-0.16979376196968374,0.2636663394957041,0.025479456770182218,0.7570350714204558,0.09643143389221566,0.8595378777353981,0.09371694617452318,0.707294985437981,0.2482949767967726,0.7338347082951043,0.18653023211986552,0.886961842165245,0.22309005731522957,0.6045327973302661,0.2444503692012782,0.04290214439813728,0.18854793607154693,-0.5052291292767339],[0.31244120147007687,0.9785787860285882,-3.1258771792348767,0.9499687349748296,-2.4516120367128718,0.9328471482885908,-2.288184679669145,0.7669940889657528,-1.797791204970498,0.3690319836468406,-1.5224812468129263,0.21957244894052805,-1.3326257973909523,0.2092472143135112,-1.0202447323385142,-0.020834668420958057,-0.69988814204121,-0.15796780114501285,-0.48840423052206006,-0.2711221278490579,-0.18607049561596103,-0.3135572716335391,-0.14907980964902226,-0.3125624118142429,0.023904978649331728,-0.24356264658234258,0.14807908404887737,-0.19431961115689472,0.34015935111023476,-0.1863989160171846,0.3988867661496129,-0.1491488338841659,0.3232910829388159,-0.09252171981225751,0.2213685163542014,-0.011190548053416549,0.16481760828768582,0.05536976145420263,0.10563985607179563,0.008742987109721254,0.019510172578678344,0.02444294992300817,0.0587602089140663,0.04536754586529761,0.0019625719127168537,0.011550604670129102,-0.11489242091135175,0.030001198147831237,-0.13421399560274394],[2.5185160669291573,-0.6816071813130959,0.30790205260698794,-0.38722899649984577,0.6665841476641657,-0.2564836251860379,0.7401129894795134,-0.11573456931761641,0.8237208256086788,-0.08204356436819779,0.6050205557808532,-0.06045519814076005,0.6969071741491076,0.04912898151801917,0.5453226563264544,0.05952693988673523,0.5329658611683973,0.14546475735723727,0.46226617108704543,0.05336305753492864,0.3728999983621306,0.10617601542548892,0.25329681357013967,0.1753852261727558,-0.09693761124272622,0.12702995558655233,-0.2871003259017581,0.09616350040071393,-0.3297846704760366,0.055745206914118724,-0.2793016854987036,0.05093881030918632,-0.15614881515771167,0.034335281761742054,-0.20249122441811748,-0.015264121194178967,-0.20916837914321684,-0.01392701509877,-0.2125689982433648,-0.03282986692297357,0.010734600431530265,-0.037677700266440156,0.0834333199368672,-0.017855522098235304,0.13281314416468393,-0.02986763262415747,0.03531961814917753,-0.05565920601233279,0.06042482696097777],[5.542596133278359,1.3163149668401812,2.8812964141729696,0.7560532386627238,2.7878033580280075,0.838634550176809,3.2835654387747817,0.6159151553446142,2.465802736294513,0.570000446532267,2.0542385714408917,0.5799612776669133,1.7371897905629123,0.6601575654019103,1.442733923757539,0.7437308042290012,1.2308613428527975,0.6354998173659272,0.9002760844790767,0.5310046421642622,0.5577897687579375,0.34873114905878705,-0.01803275099791251,0.38774743985300814,-0.6044384513883506,0.30442927023915517,-0.6127849012844317,0.2622704591620301,-0.003379271148870677,0.17863960884198843,-0.04486018555227281,0.12761392441797376,-0.11903790609933661,-0.0493011455021851,-0.5462268891062066,-0.04294874582147531,-0.4327270755300651,-0.017165671496848386,-0.0415589293558428,-0.02253053587922151,0.09579675056134708,-0.021616010500305167,-0.12798347561059428,-0.08619267906768184,-0.17125339799407924,-0.07706173014493396,-0.07260393453558361,0.0369284377873187,0.2258665841784392],[2.024804053832496,-1.11982412743061,4.9849800300198455,-1.6075061711007015,4.8593016534363365,-1.4115206525699138,4.494865660258821,-1.559529704467922,4.538630966567195,-1.4234725977121903,4.75209497458132,-0.8640530958750359,3.9982911044139473,-0.8432982028357189,2.837485276094642,-0.1553572619561836,1.6612526147317164,0.355801304852314,0.39195622515704037,0.7876404102054542,-0.9306953641337414,1.0060860098381172,-1.7057004355589673,0.8406417304630694,-2.004363144924511,0.565714019686681,-1.798396945317429,0.21311941100710832,-1.1406285847503208,-0.18641330380705468,-0.5320643470565644,-0.37497808351375456,-0.17394314529061902,-0.3808336092041127,0.6570406611109474,-0.5458152021569243,1.061944387720789,-0.5713345582320468,1.31702654990722,-0.37537011192917213,1.004103746656119,-0.17624351988197326,0.3906416225701548,0.25467607328799957,-0.14012506894045573,0.31844911230868456,-0.47864764206208865,0.3379502282609637,-0.5897569659665366],[1.0837513885619425,-0.02607438385750431,0.011085860775982273,-0.22612055643476628,-0.24459785095906686,-0.2652055912635832,0.28405587415596134,-0.22957399435711617,0.05975451247489063,-0.3587719891170853,0.24648515112895733,-0.45808743171453875,0.6156268519009478,-0.6211072306821408,0.34151895016120104,-0.4628914058783772,-0.15491330102527623,-0.14766092365679337,0.24240831195622878,0.161599060939549,0.7199656280785176,0.31797988808458844,1.1277042849088135,0.29543622894787286,0.9276043210584393,0.06839781188238324,0.17480046032193042,0.010030568134463753,-0.9148831916011454,0.0889590001790093,-0.6226839417199482,0.1969391519435025,0.071393546310453,0.13982504958673603,0.5339377606240662,0.01425469089574734,0.07823926707584825,-0.025726257621739492,-0.6389829337353015,-0.15391494483469095,-0.69019187016186,-0.15705124729284756,-0.3235442162415523,-0.08673196689341105,0.3297217378842135,-0.04055296863674637,0.4052292664805248,-0.19399777452738606,-0.044994038579728314],[3.292117675504269,1.1010921050273104,0.2093356980639984,0.5870189120543785,-0.10217681977758516,0.44439119344683903,0.30091277357385166,0.3644088017687358,-0.023710538602853787,0.37525506405751524,0.11048458286919775,0.3029603718849561,0.08415598281727941,0.062386340237178366,-0.06561153562414584,0.20195027375405236,-0.23095005098316873,0.08828978327246369,-0.42542767660837627,-0.1019975406554176,-0.19737628460628542,0.11716587255774506,-0.001887893904461986,-0.14157612036818226,0.21224407235542853,-0.15129176114254467,-0.0014070575219185795,0.024333938778471915,-0.31667502945959747,-0.05981053017582591,-0.0033404736623849703,-0.010427082689724209,0.04191710689819821,-0.1507877610651013,0.25430509796939194,0.00996860233414728,0.1065351371053448,-0.008156606660350266,-0.04287513130241728,0.07372404288330876,-0.24625461394820222,0.04559604292913977,-0.005101501527013422,-0.03942454073292665,0.17398654585068027,0.03147092032134543,0.01939363081107892,0.004294659050378978,0.026366438921602255],[-6.700211193176537,3.816555638961511,5.364985467405561,2.93363715390068,3.154650170949474,2.048843750289365,2.7547591009363486,1.4799744799229764,2.1428866053397657,1.1338980123491673,1.9125550883475924,1.0170914745082027,2.016611818058097,1.0260003372265745,1.933200920331372,0.8589682794965677,1.7169425739057689,0.7864255097944185,1.5197844935641904,0.7633937239154358,1.4053680888239042,0.6386944991064898,1.3128000473698653,0.5868668327170677,1.7333324494390974,0.8149326722348191,1.9324300412041828,0.7891805712811275,1.5648340976377146,0.7442033875663453,1.0057560613299952,0.6311386184838609,0.8321251899596177,0.5736725165401059,0.8991915042088268,0.5229213077390197,0.8546684284320469,0.5112686215335464,0.7314413732897687,0.4751819580070419,0.2795264176637962,0.2424694187799532,-0.0509833530451752,0.1609736704625072,-0.028746921543956076,0.2121927472872822,-0.08931529123696864,0.042250732219001635,-0.1897049952216371],[-8.568902986859923,1.3921344850849109,1.9400254184281283,0.9010504258464148,0.36687935975482183,0.9927777119427539,1.6161657168121035,0.8075211873100556,1.0259055669427515,1.3040258120476231,0.8084023029174247,1.0028929799584092,1.2341161683256048,1.21444642418858,0.929656541349915,0.8694153959586465,0.2831728308904405,0.731837516257035,0.11413195451717655,0.526451694373214,-0.009225400850150908,0.043103205046286376,-0.18588828959444983,-0.19177802370263208,-0.8632664745285137,-0.498819275339884,-0.6632491629994822,-0.5151476909168885,-0.6090805739837446,-0.4975741600524259,-0.8581192275490622,-0.6530322980688889,-0.790919592983187,-0.39085029234544305,-0.2986860485628304,-0.3368278376608683,-0.25061437050866187,-0.20667807920472844,-0.09551617555029028,-0.1084363253175821,0.3378537471901937,0.12438642133921694,0.39512877883141245,0.27191653537585037,0.3721636945982877,0.3615154341496142,0.26976180703477665,0.38027214631675527,0.572527687221273],[-3.9486508549547965,0.5741342140058476,-0.05581838567770127,0.5675022827871089,-0.4082674641670333,0.4257938834026372,-0.07627412702729762,0.4576503097112614,0.3094162823493374,-0.4555065342428701,0.6187534816728376,-0.4125223340489189,0.19007961836407325,-0.39104264192852217,-0.44870456973485856,-0.0367702460350242,-0.6300989910386212,0.18996583332067996,-0.5699017234227034,0.11024936392582436,0.7188424330483186,-0.19743693994670325,1.5250223602173827,-0.19450572264397264,1.311853178861131,-0.15985917239502406,-0.14539381381506683,-0.006693394472403664,-0.8482288255209085,0.34026337926597366,-1.0404373614469158,0.028255546295997976,-0.23309384372230105,-0.07558889639207358,0.640037174544769,-0.11660159196652976,0.8542560047769169,-0.10672822114745903,0.5032891341327602,-0.12694296210870107,-0.13507296496205876,0.024358737905122333,-0.3048963473045958,0.1885539486764518,-0.3715964838057476,0.16494310135164889,-0.1335182510192731,0.09625528079578972,-0.07702879635324772],[1.9520801564188768,-0.01303900823570736,1.610501597576234,0.28622225589783507,1.8254511077080833,0.1334830953930023,0.8111524748640843,-0.05208485324531071,1.1269371960474153,-0.022152414944208988,1.1805652698589473,0.010723350112657673,0.871216165959638,0.09545724850115772,1.1347156657813264,0.19945615949059026,0.7301142864475479,0.12814104989482433,0.5519092263548329,0.23363816805792414,0.5333150258613258,0.3550489070204158,0.7527376643996723,0.33959233977103187,0.9119775874172819,0.1382996691726019,0.5309737133167051,0.32302902638219433,-0.04286839485846113,0.1535747327331708,-0.1336522052921958,0.27478491145805156,0.06390145119813509,0.12549968715851073,0.19012473572006838,0.12033519923696244,-0.1131894089404607,0.09408243441197546,-0.11094095269662203,-0.07718029322961836,-0.1383826462677475,-0.0021057017512075035,0.14842003871543658,-0.018292151332280744,0.10935994699077904,0.051513679331537485,-0.05133936692678079,0.001603609397085464,-0.08565112383427269],[-0.3029566198806333,-2.3600328310817966,4.298492060665004,-2.3192145308658443,4.251956203924685,-2.039280754444427,3.8794089289780653,-1.9801286627095458,3.5868035627993407,-1.6930029734489167,3.319072235148637,-1.2066271150019985,2.9538584805215127,-1.01083718375632,2.783029592844361,-0.8070669863222936,2.43461770487658,-0.6713471874150729,2.0320373702181835,-0.3699362434716061,1.390070760786199,-0.03575521910401174,0.7210494489303462,0.22634276711530021,0.38849769711237847,0.49122882436042653,0.3435377359546079,0.4604585488876288,0.3227632381767982,0.5285177975466525,-0.1540044153906255,0.600440356401882,-0.6878921864673859,0.7136725428377027,-0.87139097840364,0.562940664899345,-0.735622019402747,0.5015486152076993,-0.6562463308555108,0.317480322361756,-0.8501344305522475,0.30611207458574147,-1.0531200077842624,0.24179239189300877,-1.0366864911429827,0.05916014339852852,-0.6042463922873617,-0.20720799319176625,-0.20177971340690165],[-7.623002341951122,1.676075042119772,-0.0017468524984943078,0.22173303950152315,-1.676694766298479,0.8173043947752174,-1.0649240285640875,0.2377880820154654,-1.178357943703578,0.6423921791194497,-2.689087987698483,0.0745013254470661,-1.6050924697360238,0.17041971969538947,-1.6170439668652783,0.17525146826846416,-1.0880028492009057,-0.05035175369223633,-0.45214925108399134,-0.26728752205204614,-1.41989610732338,-0.4307143368851197,0.3662880513358281,0.21869318504808502,0.23416183409920774,-0.003798939243464678,0.5132796946234335,0.4929176199301324,0.8698794872637187,-0.2914719410678863,0.24375765957774745,0.14309443793375864,0.6114293750714304,0.06422457258475318,-0.3601409587346705,0.017927499992868707,-0.5846904872838492,-0.21938420498133226,-0.6430843334927399,0.05443282709281,-0.43103387602926385,-0.07306956889325633,-0.6564894456388526,-0.40250418406077587,-0.31582452593940724,-0.29935375283906807,-0.011019250834174285,0.1344690258235729,0.36685786070188325],[7.416319326773258,0.7269155901424256,1.9927734323055646,0.7877003055285032,2.7462110391411567,0.8533102281445804,2.6966970172879265,0.7358998360940953,2.4623849433009304,0.7309396425697569,2.2932522403052764,0.3249881237622797,1.6549450316372847,0.4355796803654484,1.3473643994693452,-0.022937540824423154,0.7662609088462538,-0.15125227904724373,0.3954375374687303,-0.1162813970050972,0.06834357067586956,0.05056325670934972,-0.15701847982048123,0.20650091870470016,-0.45240344298521096,-0.16648218944299556,-0.7513865027338221,-0.3320781788419913,-0.8633378706419004,-0.26538355893598065,-0.7463618833932596,-0.07127496497875638,-0.5239216259025441,-0.18222862947103016,-0.22690011967504392,-0.08020923738517259,-0.22250307897561805,-0.1853097102531053,-0.223981951641021,-0.2791540211333592,-0.19978651386347182,-0.02701197657372403,-0.010258264269035953,0.0019032681792567805,0.14621941414953887,0.1917032193633699,0.18571555797695555,0.051683672207587024,0.06866537756455945],[4.436513062322045,-0.6703789685621968,0.7355470264173871,-0.5131844207209154,1.1812058759902402,-0.39982722597460496,0.5704369479511076,-0.45985640963468194,0.7654090984515314,-0.2009769626099114,0.7313309850729842,-0.07899775221209834,0.06342787927481473,0.25893886635448043,-0.07604521670908887,0.21857022534241546,0.024604888166530753,0.20476409766885081,-0.026369878263271265,0.23743731186306688,-0.30243156107474173,0.1959338940497259,-0.4435805142734954,0.31764149012339493,-0.4776936046076258,0.09374945330098046,-0.12051798319384492,0.11292775028494499,0.23093541370206133,-0.12737992158563674,0.185126405280029,-0.04442615504884937,-0.06822884453404832,-0.08767166546545527,0.0017564420245329568,-0.021988123533379107,0.1877763961417904,0.017652625299895388,0.5198874108974468,-0.1797217856207368,0.33822351108871007,-0.08517480795989904,0.22346530900741113,0.08507950014782104,-0.01083411548568961,0.059904576339764266,-0.24227323103464823,0.060018153953392564,-0.18987863409061997],[-2.8145739872946214,0.2955513082179653,-0.40616292224036804,0.5965898572493861,-1.2489646106700012,0.787602936299338,-1.710281353997882,0.5341283413206468,-1.3857432762914361,0.6112987821866505,-1.1679215733659125,0.6214010013376405,-1.2274238442808383,0.535442694852438,-1.2218645303058293,0.3676290970798323,-1.4617660446022571,0.3847593772683209,-1.9909466603064492,0.4337812596692411,-2.067971830019569,0.4696482429427787,-1.5057317063009052,0.5067252047885628,-1.031126725534022,0.36237005287567675,-1.1144983928797236,0.2509526922894282,-1.8960495918361118,0.010727277953614864,-2.084059225792567,-0.16978492166770248,-1.2977972620371208,-0.2058610459742549,-0.32727267513555525,-0.2518477459819783,-0.020219869214778804,-0.3829236138571593,-0.25838733856287566,-0.44021456979120005,-0.22828117520124747,-0.29949225879537444,0.3617618169047306,-0.16533284854120334,0.610993872602544,-0.11444552051056056,0.41931052298810106,-0.06289388919574315,0.15586415126407946],[-1.3355310592779943,0.13492702504176876,0.9164150642370013,-0.16596158619019225,-0.508517777172521,-0.7005518432826694,0.737808906032902,-0.342811111885868,0.5737433551885432,-0.09541823151392724,0.9303650378586427,-0.1647783591805348,0.551342538698758,-0.6547852851467374,0.07943396959859819,-0.260849139195877,0.1026340845673751,-0.08340401200358594,0.3434746072485606,-0.37440569182106953,0.013202662922773356,0.07019460500470741,0.1165555048602476,-0.08509353597950917,0.10600338603646149,-0.20319862939276453,-0.5117067182888205,-0.25824396190146676,-0.744729944870327,-0.30287480669370676,-0.7148444865439642,-0.359322291545667,-0.6797043593219315,-0.44160727436486674,-0.6701211318782242,-0.22796371545699326,-0.30523486793799376,-0.2652786715984838,-0.23594514760329516,-0.37866958799049016,-0.23678638996256868,-0.15599986480988912,-0.03830034581778009,-0.010581263337347602,0.2913214152955193,-0.08636003054339624,0.6484320935121074,0.11447267383279126,0.784329735069268],[-1.5189684131551122,-0.43701383723633036,1.2511245109567042,-0.4363876756103194,1.0440255854956981,-0.5230093684885438,0.622663149054811,-0.26718419591486287,0.4333241409072442,0.03671898242249816,0.26869091837869014,0.08273644155794872,-0.0207992485347919,0.0556174066816234,-0.05206675182456866,0.0813129560361054,0.040411641213698044,-0.0013330979348510966,-0.1990794504776376,-0.023014139092956926,-0.15407009983334094,-0.005744368637299851,0.07393933473884762,0.07053303701624919,0.3191650773478343,-0.10653599736751505,0.5022658878238292,-0.054209102206802084,0.27067241083175536,0.0065601222303138034,-0.3160120243365816,0.06782429683871835,-0.5188768916373737,0.14484989877305454,-0.3671120695150817,0.06644710751730351,0.050982557296309516,-0.11194948364288071,0.24505574629823298,-0.04397620649812956,0.2996311468435831,-0.08994999681479457,0.23988809207783182,-0.01494674046848718,0.12623308207676665,-0.02382305986797067,0.08089855489555062,0.002927961928481092,0.0026783718032032423],[4.275965297003566,0.5100261934003092,0.03217757216056711,0.17095592169111654,0.28323536124845333,0.5025482449170406,0.3704873188427869,0.31369143543661,0.28162476672298825,0.31051823644617127,0.44928005706035823,0.12171867452379195,0.2661814977973106,0.2322073307047773,0.2586057884543408,0.09964323418302419,-0.02730064095328946,0.0180149574441277,0.21782905897074903,0.21749757483661403,0.2370789261055473,-0.06275411437495203,0.22876682233110987,0.21277255406632137,0.06333021346847409,0.12004743996349178,-0.063719276591138,0.06977969744980951,0.08106464251932016,-0.02172315094314381,0.21090772261776025,0.16749454332070074,0.163234337724282,0.03047275001805821,0.05840075085610277,0.020000829183271108,-0.08632584184133564,0.07506346690532356,-0.058758450858457406,-0.044767366122389185,0.001267735469450432,0.09320061941355956,0.11399573504614968,0.0494133253709974,-0.12353471475863004,-0.0434416625137052,-0.01715991671699394,0.027619677829174443,0.0261265211239911],[-6.189416634874157,1.1656809726464772,-1.9759004003489773,0.7617633227445164,-2.8028168896182466,0.6780140855458996,-2.7291694639039634,0.2448627703265752,-2.3907676303946226,0.2585012057644875,-2.0145583791717665,0.19682835441840832,-1.9957425447866752,0.17410795467610427,-1.810779857377482,0.21317169144653722,-1.570881379262216,0.09577372610938406,-2.103766037155784,0.13271356395961573,-1.845606770653925,0.14424454441315987,-0.805587053292286,-0.12305037057858231,-0.12722866078242637,-0.36554571951808945,0.6151452688362826,-0.21043321777179566,-0.14152461239842473,-0.07045711713253666,-0.7439345934296848,-0.05888528963077945,-0.9399042177292333,-0.022447109715852284,-0.054590906129270624,-0.2334548783788439,0.567248680573028,-0.33475413259534137,0.7284964355716341,-0.15874294716725865,-0.1356313396197786,0.004396486661481531,-0.5617794726901111,-0.02698595147255735,-0.3390766859065581,-0.039524570560766595,0.10881801994625798,-0.18494989117903135,0.5539097598912633],[4.961479753070002,-0.7252135994775798,3.9470493432060016,-0.8245454443130681,3.7368519611490334,-0.16626966195167256,3.0452155182089853,-0.2312572237221157,2.070023340299234,0.05181055064253149,1.3694964413007902,0.3218888774665932,0.32768208876785,0.6779299529316254,-0.10055677660693016,0.6086782443494555,-0.22548396683835906,0.3864264479894722,-0.09361282163558286,0.18828214173023583,-0.5099777181458442,0.16576475091585594,-0.9087553269182582,0.4385050347411577,-1.0695351999956806,0.3277401048044242,-0.3906893310359353,-0.029206383801706454,0.18274247113091915,-0.08308246671030622,0.08516149581137517,-0.05930118285378689,-0.3931719308302102,0.11497991470983246,-0.35954873033205814,0.1045056495487175,-0.04184924835849219,-0.03784254168532749,0.37354428399153805,-0.2315911005244003,0.1769784761608178,-0.04273305384444517,-0.16205819991491674,0.04688557184720914,-0.15679444934554987,-0.027756292668972407,0.14976456538277683,-0.16258062311623458,0.3625134508487244],[4.819366498268612,0.11839053399359471,1.3926430246248211,0.42161718262749937,2.277723612386523,0.7032606709242772,2.1733177233091863,0.7289892587826633,2.0903528210502107,0.5943491504114483,2.023543272011197,0.6148543664049854,1.9273451013005887,0.3834709757103583,1.4097856078710527,0.17799754298190154,1.079962411746597,0.1736231311575374,0.40779800351251777,0.06207571474898705,0.5055886387140335,0.2517340879827288,0.3217578436910188,0.26218854117368395,0.4650218401900744,0.28015290765349893,0.2581707957293418,0.0012922694541983245,-0.24364508651374658,-0.12116034833424012,-0.5439389112931252,-0.05781602673159351,-0.45189997536993115,0.0188042374021584,0.08945034601009759,0.008319634117048435,0.24845231391869738,0.13894508284777415,0.22058680976403247,-0.03039857444756862,0.030098086628720298,-0.004459952707777304,0.0009316877740053472,0.13474601746991724,0.10431190913618477,0.05959101550636266,0.1442121149308886,0.05620088032420079,0.1182487165621055],[-0.9757274617159662,-1.5508010655308049,2.2040948827274973,-1.745427523362414,2.2833870024827148,-1.5295565867981267,1.5940906827952908,-1.371448740042189,2.209738944800284,-1.013840461208281,2.451022884735993,-1.017542916449297,1.7726867577201058,-0.6531816413311865,1.1288692028716178,-0.1916043632091325,0.8972432564926757,-0.006351040183555284,0.452030445292097,0.03911133919767551,0.6389726434875849,0.15550205915029128,0.45615281494333926,0.02219158284210197,0.07981275725721804,0.0151620342269936,-0.41758722446751806,0.08394564300466528,-0.5044616216505045,-0.12509014493119636,0.04876569173714021,-0.08884541387162462,0.46920392439675473,-0.06676378149779885,0.28433997241096715,-0.018423680193397702,-0.22763672865947326,0.186969850948204,-0.2617344203042408,0.019397649328910253,-0.09629252130571801,0.03701290602074215,0.23236164198977025,-0.13460664564080324,0.12051211144265787,-0.049057196822382826,-0.10138369216715745,0.007408759793370382,-0.30167189401050737],[-4.474562263106576,-0.9529207874581507,0.7353144258823583,-1.1489246291708657,-0.2017508580757338,-1.4394181746534076,-0.10286721165458781,-0.892019137862225,0.13849229665376653,-0.8456469064308074,0.19492528034450096,-0.5822779704589257,0.4245859545219503,-0.23771810047716854,0.5125460600260087,-0.03116768334161963,0.7729548429593838,0.20703617100610314,0.9070856976444344,0.30416124213161316,0.7352140331792946,0.34734062277447814,0.5798639125240119,0.2974712985671792,0.25019426340817685,0.40078430321653524,0.065592758941295,0.33246161796268087,-0.12075236563004112,0.16513176371463814,-0.08303382208708585,0.13424282066925314,-0.022711888007278035,-0.039828664120126984,-0.22427639950473507,-0.038509318940055166,-0.2873665545901762,-0.03174279556307573,-0.20992231670591519,0.08610441693289167,-0.13689301686082433,-0.0005118741647064738,0.06212724699220773,-0.05617095536678433,0.19309509561220337,-0.09490069634544437,0.16351756425634104,-0.03523352418020371,0.20177029848669834],[-0.5188410053792823,0.7101077378104835,-0.5132833940840293,0.745496007828992,-0.4448856894470448,0.5113503333520395,-1.024376259760449,0.4057183130994387,-1.0058298843841385,0.010240298111105977,-0.9326853212710815,-0.07510053720407138,-1.2173660228627425,-0.2510343463093564,-0.971452280659986,-0.21074467339568292,-0.7422136968563131,-0.19221965028949672,-0.42251836652443614,-0.08944960463632397,-0.24362454479684822,-0.03529962356061922,-0.21750105230149988,-0.04239667396301691,0.059992294004826696,-0.0485037203058547,0.11260709538648958,-0.07460432966603073,0.1576943277502098,-0.05635430668839519,0.08844527422302113,-0.002637110844623597,0.04794354190222091,-0.03414500449183106,0.14953325149380012,-0.023413267298862252,0.32302317186805624,-0.07758929004544428,0.2579298519528389,-0.033290659304615826,0.08678100442269895,0.03151437667963355,0.06797679511749648,0.14400811788755633,0.0662234422867235,0.081273563310671,0.04406850825084355,0.04535605677806539,-0.015063026720358016],[4.1621899945779495,-0.26222014018173173,1.4176523608978056,-0.1793139769956364,2.079745893233522,-0.18867161098751875,1.894253604311084,0.030696451259969157,1.2746527192152661,-0.027314814843876108,0.9330944163622874,0.21249575975022725,0.6451229096065557,0.17360259534271938,0.4099900090420433,0.11083411414369056,0.26152029273702043,0.03747224132174006,-0.007985905525511122,0.0016457228803777987,-0.21916584107168377,0.11709546445928301,-0.11331946031185666,0.12398098413813724,-0.10349711768889211,0.05665063145232348,0.010876557284283791,-0.08911968846117856,0.11020975959896659,-0.07398948315876457,-0.07215512884626785,-0.04159800932140518,-0.030985885495858803,0.12829759236289304,0.01675440989751475,0.06100044592140747,0.032583468952594716,-0.015783479263439093,0.0050088059529844195,0.028235371001059616,-0.04905783217632118,-0.0027226068638716155,-0.10047837534407453,0.0016314052408162231,-0.06759664698030388,0.03502435817953658,-0.008258380577958467,-0.08179344316864638,-0.013530409498811973],[3.2090862129721014,-0.5581479753630604,-0.4538720163854124,-0.6628818165256395,-0.5958600660413917,-0.5143485800772345,1.5565993724545562,-0.44645442410623504,1.2105901162496329,-0.5008344578059902,1.0637088087682838,-0.5020440500739464,1.5247513279196514,-0.09449931519499612,1.6613903208823912,-0.07589123536677506,1.8048340129129947,0.15572144897766432,1.3401222385966216,0.5397502807548435,0.9806708349575561,0.2808077782692752,0.45410411716964577,0.21938288104165135,0.2592212698256392,0.29256898859938374,-0.21804082189640675,0.11180608346589935,-0.4464346941820863,-0.015475102511375658,-0.49593377306341385,0.07553788119153822,-0.4639641417550955,-0.11789476433868576,-0.3011851899298507,-0.08963355471401661,-0.21120631913841278,-0.06268555027163385,0.09036219963462498,-0.07459501387668525,0.40740931562439603,-0.14868817861026526,0.5529408330342447,0.034472656067663414,0.18555305674932232,0.0356949511157315,-0.04612126155023399,0.07396966268575886,-0.021029440469882945],[1.3139731481597132,2.7184753636894916,-5.187279977358735,2.5488579059209298,-5.009580110936888,2.339505598531159,-4.66585698908311,1.9238868778579914,-4.259734692631984,1.70136632543618,-4.314264527257974,1.3667135667207857,-3.65900780006358,1.1397634141232331,-3.4382047390552533,0.8465046970642144,-2.7356890339514544,0.6530020583486684,-2.153211694717641,0.2918410900499563,-1.6878623630386467,0.07301637063410345,-1.8376659790014538,0.011993586147456665,-2.0303418719620416,-0.03318433410544418,-1.990313453236526,-0.10563374248138238,-1.197626498668158,-0.23108930683219436,-0.25303640065115734,-0.3625276667913551,0.15265205088512765,-0.4627627970637386,-0.10045073701493241,-0.3177035736682634,-0.31331616475853824,-0.19260618201576016,-0.013279457335948451,-0.27388624444834586,0.6404659629035007,-0.3540780341629608,0.8630223513760221,-0.366393549057731,0.6096195258495255,-0.18349310263643517,0.27871028860912894,0.035649457738857096,0.27572556512459023],[-0.45074231019492184,-0.06565270435502238,1.2013709316143986,0.00977096241625425,0.21023742380221896,0.07044844232159529,1.6506568001794268,0.17896349256116298,1.2741210749274499,0.45013281603143424,0.9211782150300248,0.2664173710846574,1.6753453447621738,0.7111714546692977,1.5820092973735935,0.49383410332817174,1.2524128878359735,0.46000119482667073,1.381527142857826,0.3999166891079272,1.3323631200543493,0.6010367020360754,1.3618318511827223,0.7656995371055254,0.8171362701171166,0.26080727568754963,0.12572672569037088,0.24009598306293395,-0.12242839221543951,0.0990042796988919,-0.03252899126322951,0.19528418421517083,-0.22487117202414253,-0.17899719384167356,-0.55754779529073,-0.07409325184482125,-0.7646637707262919,-0.11312013987078275,-0.6779236907771499,-0.15924257498248565,-0.7951098839379428,-0.03860346839417946,-0.3977651090568517,-0.2772353956761844,0.09210495254211139,-0.019009003508812187,0.042621111877249505,-0.26275975511797667,-0.03716222631002315],[4.042357513769633,0.24934168548788774,-0.8407441795617563,-0.1503959911467663,-0.18816886012394923,-0.23532399831036946,0.3020774948448649,-0.24882734755969843,0.187422644701391,-0.17541518434474482,0.23633775242743507,-0.25505864702910763,0.33766817761943646,-0.36391626273062533,0.6439197353832175,-0.5172655043230596,0.5985855244626296,-0.6983321159736383,0.6824259983015842,-0.6890697269574775,0.6513642816457597,-0.5956382089077035,0.8622674816903414,-0.4938141097689282,1.2622232848711847,-0.41630431457857847,1.4886826681157856,-0.41558116045552984,1.2969552768739747,-0.3013437998120917,0.8212179671385305,-0.11768099096118274,0.5287912302077525,0.10013492830293283,0.3710941663602972,0.11371142069402011,0.3834429834773052,0.12389144065442129,0.18154460468493488,0.1992526754731148,-0.20037966520512188,0.18314369210208892,-0.4304492603524044,0.12896864528561366,-0.31780431267259235,0.15631226392618813,-0.1065068829935577,0.0621054889368135,-0.11028263246966027],[-8.591754607833357,0.24990926559249688,0.9654238009320147,0.29739354896080256,1.8308542032583661,0.063517197826985,1.359906886220461,0.06884130713038003,1.2026444747013876,-0.07093556838357903,1.167969680275756,-0.11041262977893675,0.5397540826722191,-0.23836731825488544,0.45178346198974717,0.2560594275492626,0.1043442867173931,0.1607918268684971,0.6982359655832392,0.09260506192289328,0.6340435130842116,0.3430999651573037,0.4008934931248646,-0.1507664784720456,-0.1026561273917744,-0.022582006932673527,-0.6327679364263853,0.15494667456599734,-0.814850676221966,0.0038378461825986196,-0.07794344215406623,0.015592710195421602,-0.04084546362884228,0.03439838423117041,0.1892435892620361,0.010230819568187395,-0.09698911566355106,-0.08872950340265769,-0.08810006393257253,0.042640943458997785,-0.07922899228363231,-0.019415937943049875,-0.010420501722559122,-0.048874299618270216,0.040478863495738454,0.03549536449003078,-0.08439590354857476,0.0239998041998535,0.04038771540398157],[-6.91414636528192,-0.9778012090997774,0.8139416660931305,-1.2322140938958048,1.1047695032847593,-0.9706792706925341,0.857819136477324,-0.7566180510055884,0.8287685730080943,-0.4070254187778804,1.5197399052021383,-0.571259861030876,1.4141373262166053,-0.8864920406409805,1.9103522153868115,-0.6227653450651445,1.6100596773793323,-0.8352630839709458,1.865377828825537,-0.6414934442136486,1.7022920532822239,-0.5384589835148693,1.5078692867438652,-0.34359396195708086,1.5025633177746776,-0.3131576795596185,1.632139782752305,-0.09669981409478688,1.4321497651769586,-0.12594793434044474,0.8664426333748896,0.14586929582727198,0.044219871684338836,0.46976572479995476,-0.2928396806376228,0.4490713836970462,-0.11566278529842086,0.3268924078959589,0.21760689445915066,0.14817648877897682,-0.3997101486449351,0.2390771729188278,-0.8259195952431047,0.34758815836478785,-0.953819660670046,0.2811564670585613,-0.34690028647981574,0.17617380095987784,0.1679868168957844],[6.904571656625122,0.7687331704841092,-1.5794923608256264,0.32322951925110854,-0.4574217531754342,0.0005935048092273669,-0.1452929443848929,0.06742341755461784,-0.07698347030926579,0.10134425148464002,-0.31582254014912087,0.2682427556464747,-0.422214188397115,0.22594528444905512,-0.27018991400546405,0.18857901737553204,-0.18417201740591582,0.05845492934772497,-0.17287842621159924,0.04622217580229454,-0.11883985652035628,0.0954244077612034,-0.3013778862630642,0.12312066655568567,-0.15770968953139622,0.07329249671651418,0.05956240850335323,-0.023406976047880727,0.14762427972159095,-0.11916190909291487,0.05975607677441665,-0.053437192638688485,0.055911068818922556,-0.025332480184446164,0.06097821124473142,-0.02013934110386746,0.09242229809676972,-0.0071253973905812725,0.17364156737236375,-0.01225714211885523,0.10335816757448765,0.021380972151940642,0.020775031921110687,-0.045374845282620084,0.10856373568323359,0.005885249878824132,0.0846240149984856,0.010395083080017428,0.04547478196168025],[4.299747461982542,-0.30660064799505377,2.691840310423603,-0.5744551304492014,1.9708698092606103,-0.5377281917567813,1.9652090188355116,-0.36565731230633125,1.4629548844384967,-0.27286482313322213,1.2829242327210977,0.1317936853654054,0.7925244800965643,0.2357997508148967,0.6966248754618153,0.2756565605346906,0.7386337273131518,0.20121746451707875,0.6165538532962598,0.13905624128979144,0.43334954718937047,0.17875269122351115,-0.2289398996170397,0.12258722497194424,-0.7480819721957899,0.08287163048685535,-0.5808696539162718,0.060352598648709746,0.061242736981476094,0.029552817099140545,0.42276486381851336,-0.07794666905859877,0.46249311600463605,-0.21244400456293364,0.09326169409968726,-0.03947678305565644,-0.3644183526597763,0.00836854053983175,-0.201655611314976,0.06553433184042205,-0.09122505546649168,0.0474687342936441,0.1790719942017616,-0.09321881914118742,0.16560957105195834,-0.10045715446355281,0.11300927423206471,0.050288226341967016,-0.05239654974095319],[5.401844135476147,-0.6138059784941322,2.569380566717303,-0.028937495915849154,4.329458745956432,-0.019974356649370573,2.810855877649648,0.04150489034258593,3.0405339641476523,-0.19408125876694532,2.181312360286825,-0.1261150744496807,1.5281965937370938,0.003950717122483062,0.9981359968300691,0.15186281884357125,0.2094139106367791,0.16234816887009715,-0.37784351817095413,0.2180703790372755,-0.13307469876274872,0.3276585658480172,0.2829835763750152,-0.05823007720655513,0.20513758414500247,-0.031363590380382965,-0.1642719655917712,0.07495734238789813,-0.33567691877481554,0.06660488940193758,0.2848483703990489,0.04944080024438569,0.6224088013149304,0.041486427813670027,0.599476991298971,-0.03895363272082418,-0.0180725335467055,0.061434325658168844,-0.37973579904221205,0.13868300765763397,-0.3143778773047593,0.09962774954301594,0.007202753464350137,0.1658630488542391,0.001953123347859174,0.02408655268051491,-0.3004960191299622,0.025441721371697302,-0.4272517919486848],[1.4028606161738273,-1.5303839617025625,-1.43701398344972,-1.5688262295042408,-1.6232224348789706,-1.5673559999092992,-1.4381203754635568,-1.0309921445487327,-1.1849169754396842,-0.8010277675695208,-1.1161107024903387,-0.5855980859004275,-0.8342361387285597,-0.33589375617384015,-0.5032962904589756,0.09490334381069343,-0.05287717870372859,0.22461734062912475,0.48378728715862757,0.4712704105446236,0.6346170150118751,0.5280099776382672,0.627955862650327,0.3828467193942875,0.34851353374019933,0.2782286639776071,0.18289874061888278,0.08094827368077344,-0.0827496799785984,0.0042515200502545114,0.05399663115959296,0.1525027946135066,0.13440308834546172,0.03813797255517845,0.0459656899820138,-0.10102589125432873,-0.11065243767559088,-0.1620380170806133,-0.27877824142239377,-0.22804693568384052,-0.3360691274957833,-0.061827252175909925,-0.12799038794666653,-0.10594308717755842,-0.006543218411607054,-0.12236572798639664,0.07246161518585398,-0.01887922497515114,0.11838401393375753],[5.865133989025375,0.4970263542589652,0.15251550503250863,0.3077758912108637,0.5233003064972066,0.3458587937556904,0.891485746320188,0.15890285152396316,0.1321282077629111,0.4720332864930797,0.23031029989558818,0.2777679602173649,-0.2569969557843963,0.31792838507197024,-0.43422999136573515,0.30166661953235874,-0.24475115812161657,0.44961472598641566,-0.1867850763135296,0.29261784606688235,-0.16687535483671542,0.20206683268738698,-0.4392748647407824,-0.15393564926156558,-0.5110432989641315,-0.2594291489197759,-0.35805127111551976,-0.13798660967447376,-0.056771487317792066,-0.17804986399066433,0.2273524295764662,-0.29385830016554676,0.040882343524931336,-0.12121628479559289,-0.16095532544478688,-0.12815317870661017,-0.19510794263607847,-0.1335754430133562,0.23624383488125889,-0.08776661797416663,0.35678354021859604,0.01592825141541812,0.11777516359523474,0.021613602472838965,-0.22012696631781403,-0.0414720024211871,-0.3217020436656397,0.11064790524253157,0.06821567517679686],[-0.9549251469839196,-1.565122535035917,3.6963290681522207,-1.4617572826543643,2.7570886453359207,-1.0577808843218253,2.13431875448549,-0.4885535658425541,1.3059364113247056,-0.004335628337903482,0.4480245909632871,0.4965614060659541,0.44755230123268125,0.4419988358618136,0.3250730977438699,0.5576149883546367,0.23358164491222788,0.48011989283353673,0.14921322291268385,0.5611510261058498,-0.11117269692931679,0.5519743760613899,-0.5401588213531576,0.6353936843397736,-0.9817423265820471,0.5897030787840952,-1.0392706495076975,0.3355453473067881,-0.6695428159011501,0.0739881143644105,-0.15757278236300157,-0.09524400844809869,-0.15763156596079228,-0.039415029594510265,-0.2649012728814064,-0.06864746649548183,-0.44028477161496127,-0.021376948308337366,-0.20900052397082722,-0.033790849166918324,0.1253706928058841,-0.1829947339581691,0.24597260692246428,-0.18498159722531277,0.1681873028761156,-0.06308245148212335,-0.00902683628517685,0.03796088204244215,-0.08402006437978222],[-8.240603241834075,-0.09885804700745984,-0.639202447045043,-0.28673957128282396,-1.5775524719535319,-0.5054901213468151,-1.107206044667375,-0.6284288339482168,-1.0709466358005089,-0.28027949005208397,-0.76479945732313,-0.4643817695311024,-0.8022009972023222,-0.48171587601650745,-0.6336462333717691,-0.26040821629309774,-0.6024112564804238,-0.13332332354809412,-0.3451621204727147,-0.24129235216602013,-0.3403875534111785,-0.2014108729072393,-0.33691171197818004,-0.16971975894877225,-0.23436276864141428,-0.35538863538365906,-0.24594061864825406,-0.07119037827064822,-0.15883753973167117,-0.040345450908960634,-0.2099351047285338,-0.12896468639802783,-0.4435523778298634,-0.21978978063335308,-0.0666882097814725,0.02049683248474269,0.08754751782911004,0.06589030564206,0.010315873649822638,-0.12839624246216472,0.035482441950921165,0.1271465784973088,-0.04406341166774564,-0.005581158213213603,0.07558897069864141,-0.12073610561636784,0.08543461284581141,0.09054341146510303,0.14888555883336105],[0.396191091062374,1.4875501709638672,0.3305533790949159,1.0655101489690317,0.0770965612467504,0.9143588054558832,0.047397023480691164,0.9500215164132852,-0.03148449598544034,0.7969895957585373,-0.24313829599034378,0.710123255074477,-0.29284885539760447,0.6239891445498847,-0.32646673035518886,0.393123659814293,-0.7159180622458231,0.5109923805852028,-0.9109340387315638,0.5605488064460165,-1.1474754506812497,0.5323185515972951,-1.1643075643963892,0.39668393682483627,-1.0594102045313147,0.22897327958636646,-1.3090585234044718,-0.06639571754874815,-1.4603949965526983,-0.22558594575276975,-1.5349343555627113,-0.1867902912141464,-0.9676433636900419,-0.3002043397509881,-0.2876271773550196,-0.3324138888666024,0.0007884060442376442,-0.28243438966747947,0.17188531335217572,-0.2568401416772708,0.3485845860988504,-0.2191702573455749,0.7254338393247434,-0.1687813171599803,1.4384222780562375,-0.11228804770162058,1.4750274954289297,0.016611458118067742,1.2160984068615923],[-0.07548089060390856,-1.0861563086742532,-1.4906510941112527,-0.3031141444611619,0.2175214997037326,-0.022731631021980368,-1.479258099179422,0.3883785428254391,-0.8998510052609051,-0.13680780864656483,-0.8330313251442902,0.23862601678171944,-1.4704602966171985,-0.026975861490147395,-1.270381525516767,0.21856736059834972,-1.0999181349586613,-0.052667809434601295,-0.9884412909192754,0.13758095433975712,-0.2307195127159664,0.06328909241659711,0.14318107125581667,-0.18552447512365924,-0.004111115632293577,0.04119770873916864,0.02576083047017093,-0.12808054671214048,0.10494483443154572,0.12114766056323635,0.24560532996777668,-0.07433525701917999,0.18422728713518738,-0.04477343704991707,0.2657913617832457,-0.016113971250340118,0.20001434187805953,-0.07347280655115118,0.030757505986133967,0.048716709216001707,-0.05714398887207821,0.0462189586028167,0.025519724020105834,0.10167582328839579,-0.15741407610623764,0.030615603239934473,-0.16540664120732132,0.01441343614763467,-0.17599656129144295],[-4.130213841288981,0.511165367268482,2.095553663552278,0.38603532643221655,1.3905667025901212,0.14030926472491193,0.9907559440047201,0.05818660869254561,0.6184970604183703,-0.06636010442322955,0.3104557394248474,-0.14669149241348764,0.18535511742261288,-0.08764576255932237,0.1182357039395186,-0.16971602138852152,0.04652547023018249,-0.2573137214111342,0.18404642761573256,-0.28858018175007233,0.1961371071649627,-0.23396635556580805,0.09442422975135537,-0.22076587607249623,-0.028766159165311667,-0.11714714483363711,-0.007654016718882004,-0.07251327070779175,0.32258952041129507,-0.05569605487801101,0.5311473151187703,-3.991877684459924e-05,0.2867942197880662,-0.03098335119967762,0.0013446401255902967,0.06931040979193544,-0.1715298653299716,0.05522461630691402,0.026500379193677277,0.08070323600735349,0.07066892496428126,0.11070307887897289,0.016854180248229027,0.04245572877882529,-0.16716763616236865,0.05432270641566249,-0.1638305786290149,0.06133968568294655,-0.016817022907335794],[-1.9730634047306663,1.592334904067149,-5.580126863497336,1.6763139909004825,-5.165010697997981,1.3776926789702026,-3.6513210193720123,1.4372506684298507,-3.263179521354355,0.959809533736387,-2.7108356232784976,0.5192186623379854,-1.5423423853909661,0.34146601976808505,-0.9739393408314226,-0.07038113834567852,-0.3687697101380657,-0.42230385864404185,-0.020957803850510694,-0.6462723907345119,0.9132653742017381,-0.5885024491872907,0.8151734807067004,-0.6226604509069626,0.08663537032261742,-0.03734807731963411,-0.5035808664868737,0.1659978337215298,-0.45096258645105985,0.0491705086480709,0.1661160002871547,0.04877913365351752,0.33261829600873055,-0.3253095532121376,0.045463613525737,-0.09825857080681392,-0.011056767324603017,0.12072844247578102,-0.011258531997264256,0.02090197951498272,-0.010603668792233194,-0.03600324441851932,0.3645547370056687,0.04169782102826588,0.458484957249091,0.06390425769937387,0.21857019809393732,0.1734774431453842,-0.033796087943525156],[-0.2660076822357705,-1.735103165302389,-3.679698304300126,-1.3163885677667824,-3.120396715939204,-0.5642925056983581,-3.133940559656661,-0.30378026671691566,-2.682403986266993,-0.12509333532192596,-2.775634087870507,-0.37465877954092625,-2.4700984818381237,-0.2826388621165129,-2.351265341011865,-0.6032429555761246,-2.6005968755710955,-0.5226195000500651,-2.4245612848306224,-0.6130901175552714,-1.7765016410416277,-0.46174552460491625,-1.0734817737729114,-0.4694739584217122,-0.22824550992159065,-0.42801635662737786,0.02925568912858405,-0.45847983039907975,0.08453291114622184,-0.46310943179532205,0.20312127593414395,-0.15791192577573918,0.7722861655495599,-0.07422544219919121,1.153825927993831,-0.009401090647138134,0.9406718242654333,0.1735986236635653,0.5383394467915684,0.22589976173098322,0.2899714889184608,0.21278141672243317,0.3866726395925085,0.19999042361013267,0.2576793681965547,0.11862519540437401,0.18035506210152688,0.06981912355102689,-0.24443965349670205],[1.2948412021593017,0.7178600002177794,-1.2745574265484476,0.26332265735689836,-2.265310652196474,0.12379069900252895,-0.7952458103579688,-0.13498621711533473,-1.2722255138918266,0.2365319767772808,-1.4670015819638114,-0.20899431376331007,-0.41298389198990315,0.0416720424733871,-0.16962205490860308,-0.31038035175714784,0.21802890797849717,-0.18088828435803092,0.4983297415492643,-0.3208848427399529,0.6022687136294171,-0.3750341594997392,0.7088507780595157,-0.1829541857738233,0.8674878560994274,-0.21829191336589335,0.6433924904657237,-0.00856624599957126,0.3040029849492024,0.059000832195175074,0.06916730087985933,0.21593258518831335,-0.0020126862199421835,0.14946711868400867,-0.35978206598733337,0.19249761328893744,-0.45930199281154727,0.25156786870269826,-0.46004614115296416,0.11579852094711206,-0.25860231740345835,0.05649955772999502,-0.04676176154971313,-0.08970880506687362,0.07911070173716747,-0.09161744109384419,0.0530789843622711,-0.09131459013413493,0.11097092542436789],[1.534875572981896,-0.28871223887837855,0.8830665643580742,-0.2233220872486277,0.08516636523191239,-0.44773307831401055,-0.2678019434515022,-0.47701789647379306,-0.38568374714713916,-0.3658202708019838,-0.6377526381157216,-0.29650811988075904,-1.181745107720026,-0.1596011178083628,-1.085367484793072,-0.04248319874151344,-0.5687386448204069,-0.05921328439462285,-0.047160777847442555,-0.101199065626198,-0.1641045318912357,-0.17448355372064342,-0.7813392149715785,-0.3750656272884307,-1.412617377510551,-0.35105909445747707,-1.1652183229033302,-0.10642987468189048,-0.26419298417379317,-0.05026380323717873,0.2692956797240724,-0.12664535317575207,0.04739669371505525,-0.20408047521712108,-0.4391779216532447,-0.1727703562934405,-0.434445694131455,-0.08856623299473128,0.09665774637254462,-0.0012265566997932572,0.3492302641175181,0.038483821145955194,0.37722480766816713,-0.03366089473886411,0.11001545082472414,-0.02122274641853063,0.1583971448360058,0.13288317283714035,0.3568232527798241],[-6.482530152611906,-1.2341182855902828,1.8331732830927683,-0.573649187606216,1.6495028465283437,-0.5509679013722416,0.734156558393445,-0.4918960096437613,0.7684939591879952,-0.3441763533061908,0.8312923079679907,-0.17961506026816118,0.24103428814040936,0.09574590702915728,0.18012391536127237,-0.053011266505909475,0.5139371655483329,0.11301894612564159,0.20797317154201814,0.21686424096027843,-0.07264937509823488,-0.07830362622821459,-0.17259266735810044,0.11323548426044952,-0.18014659027989186,0.058154378942748663,0.11046860984867807,-0.041646690464974136,0.44262280771693835,0.19708146701369228,0.08567153805928614,0.022901137089084634,0.14422170546922464,0.11783101616124446,0.017836427530222725,-0.07137026858109495,0.07345669436336702,0.031512485715641234,0.18146097777473166,-0.047429563690381614,0.14139399453146803,-0.012718842134642454,-0.049043636597217416,0.026323042459691085,-0.26795280795908616,-0.016767670464093228,-0.12471446804772861,-0.062370404586635626,-0.10749956154849723],[-9.367207621850955,0.32027836293612,0.4075849316674207,0.08135344572515282,-0.8757052987368816,-0.31258684780023105,-1.4270674526551237,-0.9111720234516145,-1.4433454359179023,-0.838877808798281,-0.9969727124436438,-0.7834752329223724,-1.101853457639719,-0.7842755385735524,-1.073813063432078,-1.0234479764678361,-1.0695899000054447,-1.073091210802391,-1.2933803006928475,-1.1138393948531673,-1.3928600621147806,-1.0861078610196195,-1.2110794526826238,-0.8751724394819048,-1.0805141637887588,-0.726220567538468,-1.1072076582545898,-0.8067322003524661,-1.0377584067762833,-0.751229420111054,-0.9582570368952188,-0.6407478484010912,-0.6880679997742227,-0.40828740377390926,-0.37696204491132107,-0.21316678630994385,-0.10367325449263187,-0.19883048996910974,-0.22913713571117872,-0.1276212239413788,-0.02363888288552972,-0.07802351143667942,0.16862702866779508,0.09371357839080977,0.09276108685879507,0.009932727542024176,-0.11785003248880384,0.04241865350732178,-0.11763236705460144],[4.388456318120001,0.31377046402621434,-0.03688258015412444,0.21254268441174312,0.4384311462799245,0.1876843725478228,0.30434741509169594,0.1453344400765725,0.4151284751585786,-0.09628697379157931,0.5845148906036253,-0.09795714598013187,0.1732764383398754,-0.22376609937948896,-0.12660799241781884,-0.3756968669831767,-0.2474505364490099,-0.3801630254391329,-0.41320515959798193,-0.37506863749816793,-0.6807698765669836,-0.3407045195365777,-0.9128413611092647,-0.25091577708913654,-0.9510022851680765,-0.17979505008276903,-0.9126034695377376,-0.10873354525993989,-0.6179901946709879,-0.1174439865463286,-0.26025319034794525,-0.0739890842588338,-0.016027630683074234,-0.04938388847510191,0.023781181597811066,0.006858084154958315,0.07990630483706734,0.021801182258355298,0.2832480914274618,0.039675044674036736,0.2884019267276525,-0.006369143917992584,0.1605945971123566,-0.05666250884739753,0.08040678750944914,-0.08197283197469084,-0.06571330777621966,-0.07801217259651087,-0.13569277269627147],[1.4293816017462246,0.7644516151152385,4.714380679569372,0.1480792792025684,2.7959624618135024,-1.0437632169637707,2.8802514069960465,-0.2858357980728121,2.106048547943084,-0.1431087570989192,1.3730892086232769,-0.11337392244875642,1.3719415699996576,-0.0168349017904778,0.7832862096911397,0.23298801261110527,0.56467466592159,0.41999539980727724,0.5846647120776035,0.5338469701378908,-0.23965579480051408,0.34211658332503575,-0.7808396645989265,0.24495270477193098,-0.8634827390431135,0.12324352120405212,-0.5266226230392267,0.005666283093651999,0.08127524582997,-0.17400894779989634,1.030117872402234,-0.00817472989714415,0.620656115833989,-0.1008369675920281,-0.2773328594408607,0.03544106477773191,-0.39725851676495744,-0.13109495807660532,-0.4287994362013001,-0.12030076678096158,0.010658842420811455,0.0349744121565564,0.36064086122248823,0.13885795821364105,0.3617115421704376,0.02239150291225825,-0.1837345139121932,-0.1024253674819606,-0.33177333644153256],[-5.45933785440629,-0.3880771924957835,-1.97226611067807,-0.7817549501933726,-2.593188331178647,-0.645308541921452,-1.7018169741341802,-0.7098302025127213,-1.6413037606609115,-0.5703436479677702,-1.2937448083581768,-0.6294887902852344,-0.4633057208715542,-0.494112512248922,-0.02596957297560842,-0.45239951346539137,0.17692480858524526,-0.1614570945537062,0.393377401938143,0.012478186851579086,0.57817565088499,-0.11522291509841247,0.5168059934422604,-0.06310152055578502,0.49811983575658597,0.15520848490595524,0.6146804762386366,-0.0435104125119642,0.1557054871141678,0.08236367389479468,0.027891223314250483,0.09333911189458333,-0.03812092409624407,0.0658148855821897,-0.08144993075202964,-0.05311126666169471,0.015421761905054414,-0.03236226880672852,-0.3068393046045786,-0.0034210108280157206,-0.34493225798041177,-0.10004225087530427,-0.31800490860187297,-0.013903136249585162,-0.026528933715849523,-0.07766654356223458,0.12942267741912775,0.016658126051430695,-0.07157001787896011],[-6.9988825440016305,2.490110319052085,-5.0279657695400175,2.633965318117654,-5.0647382692323,2.6438593134998754,-5.177735753833289,2.1592856206519513,-4.891356486971044,1.829294278357568,-4.324230061901542,1.3844176158624326,-3.817815986548331,0.6780353287383692,-3.388086765933607,0.433353351399847,-2.7522505550554834,-0.08115416689908297,-2.247371525558716,-0.2971453709850888,-1.739432827067083,-0.32238110081136273,-0.9088186081238945,-0.5941535880153971,0.12920813786211965,-0.8919858033560402,0.8858510284849693,-0.8985599612523799,1.0938937796185486,-0.6229929566313934,0.8609056914257691,-0.3782875401370915,0.49837785051558225,-0.2905125915953414,0.8053036110169415,-0.3391250418836836,1.3419398998766647,-0.4815957443881312,1.382709138582891,-0.32548983451958363,1.0099738302079513,-0.19710331830612982,0.23480337603921278,0.12150997117497682,0.007441939914519642,0.1816522048001145,0.15967872295471686,0.08147617256377901,0.3532757099215741],[-4.570785010623168,-0.3689477351782253,-1.6453747136946517,-0.4592395304254001,-2.1334900874414013,-0.01847087963802501,-1.8037665004867636,-0.21380502024036677,-1.747615134490504,0.1591273945780969,-1.5199462516975066,0.009740067499806515,-1.4268209099986113,0.2944309394871199,-1.3445958929383752,-0.250686520965664,-1.0325105600093973,-0.05781054143989819,-0.43082937813909905,-0.04760020998318422,-0.4379520657741419,-0.015341198020974351,-0.215845870722094,-0.10962606922191184,-0.39371297764896573,-0.11473265129969251,-0.31305094690289903,-0.1339600168156911,0.004623415762391748,-0.1678361315614896,0.047006057371940704,-0.21923138875876996,-0.266106021490105,-0.17082271818973535,-0.07792218576941724,-0.10740275190803321,-0.14064741385314988,-0.0557900600894168,0.03571494289074403,-0.06910833860017691,-0.021639729772395876,-0.22087497111723442,0.19706137402105572,0.012794047747578285,0.0948131824706147,-0.19818861685330022,0.16400473620185116,0.028586467989247355,0.11701321627314193],[-0.3254230048116642,0.9829797566870879,1.0796212406725705,0.7877995687093393,0.9095304371921431,0.9459833412596923,0.4400490345511488,0.40132960269247475,0.40923137211422994,0.2679955264131262,0.22113474815436288,0.203654075786522,0.14054052044243845,0.10259493896267782,0.050612103318875265,-0.017033597881933673,-0.4626505133563431,-0.15275927880541165,-0.6288256997649067,-0.2971293519404211,-0.9015589939127304,-0.5110367367494109,-0.8409407576850298,-0.5123746905017202,-0.5440214749021635,-0.25744255403222205,-0.4546678996381863,0.09151562880167502,0.028067987990074542,0.17598754905857836,0.07845922302703816,-0.1069098633217181,-0.1174282356994417,-0.1385783216680985,0.010597215676731685,-0.13407061282550628,0.015673016204009474,0.06163660811379456,0.2798700838099403,0.16091086025680582,0.3900280008391022,0.24085483576759747,0.42014838282451106,0.1254811390597242,0.2258483205754474,0.08146396666553875,-0.0029312101963087905,-0.04509259371811041,-0.24586225627473504],[0.3781720651728139,-1.01186030709287,3.444037186147617,-1.3208085504842528,1.8640398073634008,-1.4169319810296068,2.7176721281334393,-1.3048460609878592,1.9431642800873337,-1.0179456823794673,2.2316090734050253,-0.79465053515472,2.5128885435918984,-0.9936405829136652,1.8628188013105744,-0.3783626146277118,1.156773766059587,0.08265547018207799,0.2748040011300371,0.21359644204169465,0.3494937442008941,0.10568677631900321,0.7787652581735853,-0.02726097855804031,0.8673985652963087,0.09083186328121191,0.20920792260301255,-0.0409708855508134,-0.2986744761409805,0.41259428112659635,-0.5930377569388265,0.15304084567350468,-0.059486133153189256,0.10297773183696164,0.10248105954821228,-0.041030983506423,0.06561164488796752,-0.12202517597301546,-0.4243806711098721,0.21868143550017913,-0.21180699487874208,0.10605928348614078,-0.2649405995661692,-0.10936995595929469,-0.07994206742183398,-0.0596015973947818,0.2789452935528503,0.004075180188407316,0.0891224271501112],[0.3327047695328941,-0.32386715501959507,-1.3010749229234817,-0.12268366654264219,-0.9124529202124844,0.033367616734518706,-0.7987825590569726,-0.11621677091016093,-0.6277888242300053,-0.04846607678988547,-0.5551075281060943,-0.10560568536920054,-0.4371690476951276,-0.23663589714704333,-0.3648235277377314,-0.1406889246664374,-0.19308621550834532,-0.2692256880694847,-0.2432386341586305,-0.2751721174851685,-0.0683997300274088,-0.30355074789599507,0.037142081662586646,-0.2884435321868845,0.11054257080453816,-0.2710203884358073,0.3326976965594124,-0.14610129460048285,0.5765549770547524,-0.01051368206220107,0.6815528449212735,0.00362203904468761,0.6003921926301826,0.03798661164301413,0.39545002935692497,0.06680373233329982,0.33630973018279775,0.09337151709521148,0.25581989984469583,0.09431477298009333,0.10711311886182923,0.1080250045365378,-0.12603759723556426,0.06973438122586077,-0.5010053007603765,0.06089961589319958,-0.5364794942407488,0.036115455242238484,-0.5116107264889236],[4.556621799767406,0.5039597437551487,0.9653403934282366,0.8508602486653858,2.4622236520170766,1.045934059718521,0.14480332800237875,0.7669531434405807,0.20141880552134767,0.5981698979354677,0.38635582287102116,0.4916409814661911,-0.8775264061519025,0.1746465896314567,-0.7711247512040079,0.40114113183367484,-0.733123172581156,0.1808454420776292,-1.1301262148123814,0.08235637399143246,-1.0821951836196648,-0.07915779449474607,-1.4934367342782382,-0.6073261864441644,-1.345844114952327,-0.44990636370137177,-1.0829487557843247,-0.43919404640685816,-0.632664380254513,-0.22245264766873105,0.09028979713917083,-0.12000745005477634,0.12308662867320815,-0.2508952745289001,0.1318273018776271,-0.12602555518801978,0.2849061312825039,-0.03307165928868305,0.4616189644962489,0.23300575457374234,0.7441575859255036,0.25187339988113566,0.6630892981901445,0.29484638802265933,0.25220622587280805,0.19109893553349613,-0.09556328011733016,0.2233642294505822,-0.17502266801268526],[2.7041973742082144,-0.5931743728586356,3.764718987822072,-0.6233490438637979,3.3058475379422223,-0.5750251251572881,3.1027470033918845,-0.33517062368422784,2.590214544917083,-0.0872167802962495,2.154536483160837,-0.043274630339886384,1.8333347890911167,0.04330893579262463,1.4173862620499662,0.3408888705860236,0.7240626299417222,0.578750712726497,0.33839496631183336,0.7080149862669013,0.010815615881314487,0.6601172675969156,0.06202438848807128,0.44338308317254127,-0.15792540906103875,0.349696482090209,-0.2712773241038378,0.20058194220422018,-0.5495732439860512,0.17626692523773987,-0.36294088743899644,0.17152266931236904,-0.1683096127707513,0.038955939886680765,0.08572225451886252,-0.10591289526782802,-0.009974261824735278,-0.04898462838582357,-0.22209025268554933,-0.020874503443608063,-0.2500241826984823,0.007822746590134606,-0.056471184956872006,-0.0009115282185779644,0.21933750441347877,-0.040501863390658144,0.30334245643602564,0.008939676052405747,0.06891146024726398],[-2.403838725444749,0.6377179168637183,-0.6586928734621916,0.5617203411971711,0.016681032215606184,0.25705528891912444,-0.7817634257605864,0.3599439296541202,-0.40223530212185143,0.3733413961025429,0.1045949670446036,0.12037728623754489,-0.1415954873470889,0.15964470988066579,-0.13150255544713904,0.15138966056114184,-0.37946180446931566,-0.02281856356263237,-0.5488541392863463,-0.07847269834713734,-0.45494502792634256,-0.09828809143379734,-0.3709702044367543,-0.5506850579377148,-0.14345919810169053,-0.32721747844713844,0.2089258202905333,-0.2599877765397619,0.005564328248539355,-0.28879798063588646,0.37386477289466813,-0.1881463747185944,0.28015706559338444,-0.10372852289335373,0.30757180554194496,0.040817485544719125,0.23112564646920267,0.018355885976172315,0.2717145711904997,0.3145383948996839,0.1662449049552285,0.16426523782348096,-0.17502911188898276,-0.0782019338101374,-0.22193983543445028,0.021324658193796506,-0.10803121390838474,0.17116274873153653,-0.10466627914102836],[2.700967125757995,1.3779570264560104,-7.273521030129866,1.5562313666883696,-6.069495241883248,0.8405098134945278,-6.076674640471627,0.618043590051242,-5.616916436312347,0.376277194322991,-5.209547138295031,0.1914646199963354,-4.55408498905585,-0.001969135804497263,-3.8934718491287637,-0.11707376614489974,-3.036656046359849,-0.21574230522273755,-2.2730758881440774,-0.5561626246783018,-1.4405527515083072,-0.5619661493377222,-1.1099155818382702,-0.7009666581262416,-0.844519897466951,-0.47806181109932827,-0.31801156511895884,-0.4769592019684174,0.5457413055521759,-0.5643746743036372,0.9644743862424674,-0.44765518166457235,0.7508629613408971,-0.26026556604394646,0.14957292191091956,-0.07323718862059224,0.045927606166154156,-0.1161802114480934,0.6261290223870667,-0.06708547919802857,0.904383305160835,-0.1024745580962669,0.35097945300113736,0.008829595627427602,-0.24488592628223627,0.12033571066647551,-0.03812347949902321,-0.08474686477211842,0.4114619102513549],[0.43123325933415085,0.8493684940894624,2.6687946102768487,0.7582748482495534,0.5361253864738004,0.5451277428527674,1.8841081181122392,0.43917618089873056,0.6238022636277926,0.640669087659537,0.04970839868931475,0.2753739256700959,0.19231120352375833,0.4491426584688542,-0.6616938022986851,-0.3279849072724554,-0.9248273095375579,-0.13534556774207862,-1.4014720478339187,-0.3452310481667289,-0.7831306548565012,-0.292244443398242,-1.2762409983022962,-0.19964171145440485,-0.9352961193122782,-0.11351059831844751,-0.9290279025092656,-0.3007680695465816,-0.4266780493294671,0.038911246697563184,0.4957718200622137,0.10815566055988003,0.8424273070500972,0.00016523280229481543,0.9259581615816508,0.1389935677783863,0.7795940406672794,0.37321449355280495,0.6233286729130412,0.053920191779752304,0.37611573487602046,0.03404015344254415,0.36677143031056536,0.17633668232169877,-0.0346935072335079,0.1824922153019757,-0.44413759862340885,-0.024822866316200634,-0.6626392916928329],[5.624279835628187,-0.035442586925497475,0.06681665900366175,-0.470825947962108,1.1384290985535732,-0.7520489497910735,1.3308756555715306,-0.6167444566399409,1.0978563999936135,-0.37820030424534234,1.1054099216371744,-0.47791523646106904,0.916369179057786,-0.5133991822438813,0.7417921078050468,-0.12121540415954375,0.5085099699099029,0.043698035577187724,0.5085061655789597,0.011293608958179283,0.7191756888407559,-0.16430245637415303,0.6716494716753307,-0.17482208994175494,0.06718851815435815,0.06213784092608645,-0.29254378579709533,0.18688838767971735,-0.1916571410965434,0.10946786905721527,0.20263860419670754,-0.06324229494666904,0.19651718271054197,0.006420443402367474,-0.136061224275987,0.10052587221086381,-0.4475735289079952,0.13672644580521048,-0.24068305278192334,0.061106755228568635,0.1785474484950242,-0.023834452239458657,0.10013590346174345,-0.06973976861195766,-0.048429913087427244,0.05409986137148042,-0.025240141957982756,0.03365589484516097,0.04379724094749513],[0.9352547388722335,-0.27525113703737564,-1.0153837697795074,-0.3778022068001349,-0.9062077370016635,-0.22959788272088566,-0.6487355696241639,-0.18101890686613242,-0.6980289186265408,-0.14517393905793696,-0.549974887143609,-0.1258830134608355,-0.48429100932828023,-0.17623901038696238,-0.2708466274121573,-0.23159123121651118,-0.19496704358099776,-0.06660709488629431,-0.09282752037669369,-0.08502082096088401,-0.07824849668076235,-0.07587476551719609,-0.05995371067778749,0.06383060456103183,0.13573544449213892,-0.03503209505055266,0.15205048194812187,0.06997992400998636,0.35241022562812246,0.018274675086326325,0.18324576140716534,-0.044452676658414104,0.07049208543312507,-0.048705273329582924,-0.1165346285369734,0.030359383322737826,0.04935754533432865,0.06522363281884988,0.08042544194521416,0.0023252396659774965,0.0884715142341781,-0.0031517323999355066,0.034385368751652705,0.049561610704736185,0.0270983563529399,0.0032247091979818064,-0.09463250122863709,-0.024121154349194523,-0.119125397304666],[-3.257284947128218,-2.4389030262609843,0.04257969088765316,-2.4482350328957057,0.2769339208138765,-2.1892365791541657,0.6305689721810681,-1.8540695569346077,0.5127213402532012,-1.7413652731468348,0.6034692660966491,-1.3373033463703743,0.5508063465961208,-1.109683569920893,0.892953771365403,-0.8774686117251917,1.2290830807027504,-0.7776917931151127,1.640476511152533,-0.7527135015614576,1.4838722037358512,-0.6143828752188271,0.7726219346215709,-0.5184140055900932,0.5301219614822327,-0.3751301567469094,0.8500551552365072,-0.17767829878800429,1.5615735082307771,0.045129562589873946,2.028459276319381,0.14487903342241518,1.4726547156216074,0.09238195381667631,0.9700359675525677,0.17021536904154155,0.8067338183823518,0.4142444355588814,0.9399539938533258,0.5380991917695686,0.8171862432496128,0.457424007642371,0.2465588301021092,0.3913171719961431,-0.1802617663940357,0.2929702522046965,-0.345923117758546,0.21628458776826934,-0.39431669980085227],[-3.9964981456525654,0.11669003351777557,2.4832078928431707,0.09753533216267224,0.8447166314889749,0.40381455629575136,0.31236561443120253,0.539299001264107,-0.1045387614641932,0.5222569695891282,-0.4491140936764468,0.5293301006671969,-0.5271818668206629,0.3436782195336625,-0.31195201061204586,0.18830289745495515,-0.07966968324080848,0.036377476577645854,-0.30776572997637813,0.03693987311734298,-0.23977297507058268,0.21388218252898333,-0.6533699684306064,0.15861214362860745,-0.6257092554321926,0.184488430791435,-0.9491367588236455,-0.001680315677029626,-1.1333485901749585,-0.04386676599581882,-1.1728085458723028,0.02300006459301608,-0.49503120696872926,-0.14553278553027327,0.04253104112447013,-0.1503648251565843,0.1303403015497824,-0.1510342890562605,-0.1049578404975676,-0.045233973675883166,0.1787006542272732,-0.07251793892601058,0.6392556084775034,-0.1579393169239825,0.9455256687254555,-0.1314927629332442,0.8343981257854454,0.00851358631587182,0.3252991702778742],[-10.143455531139935,0.04102262696429537,0.264360100245284,0.02680528132799684,-0.5395879541374706,-0.2920497187600229,-0.5046224185282484,-0.08662282384785067,-0.434877489143844,-0.04937216205593613,-0.4631740452202238,0.07326333570997058,-0.4960246210195486,0.3509581192544454,-0.10849357904402564,0.4153430867973738,0.6716584553539053,0.2867771138176982,0.17651149179508283,0.0959272057339802,0.0035694163366474293,-0.2827017962065339,-0.23721240003685073,-0.3181010114600096,-0.1905840079634873,-0.16010437909335154,0.1446504232388864,0.19013438288641335,0.6073907704299678,0.20709659852921403,0.3918637360668032,0.01821614578214091,0.11511640662744,-0.11454877272122393,-0.32609614812977405,-0.048369574689959734,-0.11310369012548419,-0.0932903863544316,-0.06673403897303863,0.1671814821505028,0.18661678731220804,0.0850630789972545,0.17518373060619402,0.05775640121468632,0.0022385727523665305,-0.11071317729220395,-0.23385347299778964,-0.03171459892454755,-0.2614166288428968],[-2.5110884660675805,0.22818000818837547,-0.8189809430799225,0.7509656481722794,-0.8803809686600556,0.6323192674733279,-1.350077124983879,0.7903536361310557,-1.1272956664559464,0.5885061507845726,-1.2198795809040364,0.47482596044521697,-1.2651744377179275,0.41674369455385996,-1.1552235266110347,0.2879136576259165,-1.0435882775199068,0.14911785792229876,-0.5870054851852775,-0.044356332488899426,-0.21340963899847076,-0.19598221579351924,-0.04861854465694423,-0.17541534132232642,-0.20399516448629676,-0.12839856601706026,-0.08100242511663722,0.017012920452975994,0.12304801632506995,-0.03783751908374856,0.09659086001513953,0.002441810225588472,-0.03330929549404426,0.004634851342438949,-0.20915332017391203,-0.03245849959247182,-0.11171802006443157,0.03279177521274112,-0.007512535561449381,-0.004261492059768295,0.05410177560414128,-0.05326081617539499,-0.0031423389052756615,0.0232150616355372,-0.055427399421262374,-0.025867615121731795,0.018788049728522653,-0.011449806311575376,0.18054639696378139],[-4.884643443935546,-1.0147644645026839,-1.5694435707746335,-0.9481530981436888,-0.2152796816892469,-0.353821517669237,-0.2681109822877214,-0.5236890886737066,0.035018052033774415,-0.435245771018448,0.30954807374781207,-0.19070314320230297,0.42348257614897744,0.0018598703468549593,0.9130803740228532,-0.09741159486261727,0.9997497751531094,0.03535387205745058,1.1362347024775212,0.13891919776640843,1.1904125002262158,0.032859482502943016,0.9760397271980742,0.023899487310845426,0.50089422324614,-0.11063403557241586,0.5546570487341509,0.08486767716676237,0.7274863791349326,0.029527232344694215,0.3934260859545382,0.22635688877070242,-0.31165513378234905,0.15260606470990246,-0.7063934012184643,0.1352464148054444,-0.7690028303184094,-0.0027250027109317824,-0.43185487580127185,-0.03674849820764083,0.15589420907781626,-0.06918212100551485,0.18928779347523744,0.02682455462797673,0.10332571576863087,-0.012096616945849868,-0.10870196403459237,-0.2047883376764832,-0.13770322114023262],[1.6865833098953804,0.273074629268805,-2.091293486841004,0.068916840617234,-1.238035852031144,0.30471476156309313,-1.9086209493648136,-0.16259287688470783,-2.4367916796262907,-0.5293322027440516,-2.4616580036374573,-0.6691060307711261,-2.8614200693636844,-0.8065342494020987,-2.7407034088796753,-0.8016330931757035,-1.9826497574498938,-0.7385162042323905,-1.5627347359245483,-0.9632921226735948,-1.6082560114470323,-0.9996025937568591,-1.3714294793390358,-1.1490304368142132,-1.6370579687020093,-0.9310336129674345,-0.7979528179059635,-0.3667987201940171,-0.16634701969306662,-0.1321362017632482,0.12508676171570024,-0.30862723876919407,0.01242983467185128,-0.16535383589065444,-0.06338389398131922,-0.07917298695273943,0.07522794449525184,0.05875515444523432,0.4417431579158952,0.12101417559404523,0.3290576783748742,0.012960041239710153,-0.1601107243912765,-0.07207325437313648,-0.05533357246435137,-0.042838578964468174,-0.0800903158437909,0.04121193280372615,0.049207457689436386],[0.3825190628326333,0.0073549634812240024,0.31613660673821353,0.1758936436000587,0.381513146414032,0.29210995446993465,0.5229986606375402,0.2247440248587709,0.40265679816779604,0.03531414442180231,0.1955290011602998,0.12162282170265516,0.4470501156106246,-0.019451631914902994,0.32967561100152337,-0.1393318566342336,0.0413795684906129,-0.10990822117350699,0.04077470959789229,-0.08151278424471148,-0.23591459616968277,-0.025234196503296266,-0.11506653969625741,-0.04945068946818089,0.5786618040190542,0.07234503632781755,0.662018024811952,0.1234423324023275,0.17067257357616222,0.13889931607148429,-0.5211560721339508,0.16956636880512904,-0.5668418267433384,-0.048116305236306174,0.022146942533669202,0.025335398448648603,0.6298736733640034,0.06203589635305195,0.5637593289850944,-0.09739079031872387,-0.08477719697938231,0.08335961303863262,-0.522529462444533,0.06093038916337387,-0.45118973452880273,0.06499573111540835,-0.3130733435218614,-0.09664664165146603,0.012363643780723497],[-4.885781528477159,0.6684461802101147,1.933277381145905,0.8539711937203295,1.5823482387261383,0.5114830398357008,0.9028993211904267,0.44069102063869126,1.3391783647079014,0.5685603050706525,0.6723383884191964,0.23760892391031632,0.25963407667743665,0.28291992623709583,0.3075957844647739,0.32758842979363606,0.6056537116652371,0.273933109092459,0.3611263588093342,-0.18290908018763447,0.40432409744906245,0.3289768998536811,-0.2775972088682975,-0.08078707125361803,-0.36127843953533256,0.01519981048594805,-0.10239615650374419,0.017417484267311694,0.16725095225298833,-0.0732950209442581,0.05337564534013342,-0.15033286854834302,-0.26832779419385505,-0.010307109143106369,-0.274617689998288,0.06511619117752741,-0.12546698126289818,-0.0193136886763366,0.3416687150778266,-0.05943435675202087,0.3514646594584842,-0.02596204574999519,-0.13365589601647082,-0.1169933370921707,-0.10347644120601862,-0.1028673961671928,0.04357630245018575,-0.0020114018601152333,0.1403151287473243],[-1.3916314622519586,0.08419801544025184,4.515857162050813,0.24429744306422543,3.011792739824693,0.3641173531699187,2.345738277851521,1.0089941774315176,1.4017596583066056,1.3927658649961867,0.27649596678315336,1.8308745638641155,-0.5966631355913616,1.86881526011389,-1.2826975705960937,1.8254658447842746,-2.0835681040831333,1.7319410332440028,-2.6000524842173935,1.6724524251750719,-3.0829042854346356,1.289058215279271,-3.2694791242583445,0.9809933614155286,-3.3520034087173,0.5640360244862304,-3.115934888476158,0.3164142286301722,-2.731028429316232,-0.03056300981668883,-1.8155625922305365,-0.3345249268940169,-0.6824136603339022,-0.7030363552033582,0.30759506134604153,-0.8452603523515614,0.7268974984084987,-0.6687845439051031,1.0111576932734339,-0.4912158617059139,1.3010778237155276,-0.3404242437453803,1.6217681007775269,-0.24418025040816477,1.5393772744872498,-0.05326230291797185,0.8161569241475755,0.21151669507160623,0.009550170764920513],[-4.626117456918806,-0.3191762042905612,-1.4550091886078598,0.4849802601406015,-0.8521133980778935,0.7992883310080868,-2.2682247278428345,1.3688564136867116,-2.5654925932632113,1.4609754188761015,-3.1611247814082066,1.547828450404896,-3.9356902050242057,1.5827230496172755,-3.5066731665376705,1.1894532972707546,-3.1146177306092513,0.5319876771918298,-1.9717391100440356,-0.0013223016886241207,-0.7431992261101661,-0.5331805464795517,0.17484038354877254,-0.6388651012251708,0.7066970857324472,-0.6492702241236004,1.0890191024399594,-0.5525928683913229,1.245844040027547,-0.42321634483319276,1.0997757313294056,-0.2817487499783245,0.6541131084499809,-0.013918412279171025,0.2781226227454088,0.13931540420173658,-0.18509191445992002,0.32400224191584737,-0.21971371039627458,0.2579019066234298,-0.14973746033391616,0.09886793933414158,0.060021488585752694,-0.07656151592615634,0.26722631487282905,-0.11969205661560013,0.3700402252559644,-0.02154216250200488,0.39182317130811334],[4.40901030735954,-1.0239266185447473,-1.752140542822225,-0.772162913623672,-1.1407355874809486,-0.47860421201428205,-1.0277424137923448,-0.5782701678826582,-1.0805742210819826,-0.2698640380852438,-0.6569273125998069,-0.17631623790376072,-0.7489507311803173,-0.16852921393273385,-0.2550874154548507,-0.3475803589880852,-0.06751965867071338,-0.32748234491774736,-0.47202525476841894,-0.34242615419689565,-0.9103068165714507,-0.1841503540774577,-0.6390465857214778,-0.0637380566215772,-0.12040657081178241,-0.11437354451527701,0.6771317278725861,-0.011458348178639888,0.9427647093442225,-0.041318486985423494,0.03428253026731651,-0.08864868608985164,-0.5252211423500136,0.024329294374446888,-0.4820044010746125,-0.0012829062426516296,0.02007858145420085,-0.01636496694127053,0.5162444428473677,0.06511402764942259,0.4222023487020168,0.042556695283776966,-0.005565961211744675,-0.030763248136989406,-0.37965070508043486,-0.07288689682599285,-0.2137971867283881,0.009504864264031878,-0.03232029955661439],[1.5438278992742587,0.7115099489657437,-0.6926266892781736,0.0205874032509511,-3.0283863695895783,-0.09093890669705898,-1.8977845574983343,-0.5962436848094121,-2.19155331503662,-0.16118513093265263,-1.2929487902032817,-0.05997094169768121,-1.0387316580625656,-0.38894183783789843,-0.6024103013593084,-0.3136505998927138,-0.563567754962837,-0.36983589901326597,-0.23881880568528402,-0.5551038078101671,-0.14047576017320348,-0.41108739803075006,0.23896936313546197,-0.1590982351981502,0.8435217522093722,-0.3049520241901709,0.7452994130426293,-0.17317421483003967,0.1749439673965562,-0.08625548926875914,-0.679056422976926,0.04312630135248257,-0.582160112421447,0.044070106467893014,0.05500013279682244,-0.03858475685874416,0.5979978860260005,-0.00820706555126195,0.3867773856848199,-0.013398053516379203,-0.2583463196481815,0.05828356226268554,-0.4941183925259162,0.10953657817002087,-0.18513345177316182,0.12512244828414065,0.3373135126419609,0.13345142836399873,0.4219079959368219],[-3.7619898307056125,0.460472015097767,0.6223278900469708,0.578276623498456,0.24000934264778323,0.7786196143576998,0.7922644178739062,1.0729775857906994,1.2988001203148571,0.9267187205373763,1.2975437180931417,0.8473627457536567,1.1528527411753269,0.8115128265155443,1.0171605022785157,0.6570337408941354,0.7313370098240229,0.46490399513048236,0.39352711111625055,0.4464364621852254,0.1345931103081819,0.22153224154050993,-0.03942655539517612,0.30882640573407905,0.23175976026008185,0.34817016891746827,0.2522126298310565,0.14496710669562055,-0.011408984622427041,-0.020352100838829327,-0.40307594630026033,-0.06389823081791153,-0.3402943879330485,0.04394561947466309,0.0759099803482137,0.09804775999073173,0.24620889659253578,0.08109299982574653,0.06985470858448059,-0.030358902204311405,-0.21010635701722158,-0.033779714064650496,-0.14894856481132374,-0.015729803661865997,0.06682715887522159,0.026019973310040356,0.06141346093666984,-0.09338201997283305,-0.1455023685590674],[-9.733468954720845,0.24086778745099394,-1.0433325121998223,0.9545720618963051,0.2858428235581618,0.694012377164013,-1.4222092533560309,0.6592396611472013,-0.19746618805464297,-0.15554532691528697,-0.50173260437735,-0.11301193956183997,-0.6421449285530367,-0.2739330830392509,-0.3486945912979564,-0.18969214972342843,-0.5209403448080027,-0.49020294043784074,-1.1475743938322918,-0.4987600043498857,-1.09251355343044,-0.4164786497047368,-0.7236511741869609,-0.1020195271042155,-0.01528832289583354,0.16356932496523977,0.24242286833637566,0.15218133111262144,0.35814543656719494,0.15731590326807934,0.24906517875581413,0.35581534623240035,0.3841844517622779,0.22598405175959335,0.5707386730293826,0.3299384592350711,0.9259564027855467,0.3675709130535438,0.5142494519000981,0.2486395630366118,-0.004548815506705735,-0.0015659509581642513,-0.10675022195070695,-0.08016695308132461,-0.19254144737941065,0.0024183361921237168,-0.1109253560752442,-0.15484066003403563,-0.5555251666429072],[-4.230977305124725,1.329629707021781,-4.51976596038412,1.1463366757168552,-4.8320684348525775,0.728730335389402,-3.544951121387664,0.48499939828642297,-3.053696998203812,0.4000812613624255,-2.106370185188911,-0.031647624376436825,-1.682281366504477,0.16051503613062726,-0.705451202240911,-0.43718923706427465,-0.11785465239313746,-0.48360142900003256,0.6110154017957857,-0.7946671230204865,1.1323167928687352,-0.9235532616558866,1.0140001984680138,-0.45787863887613384,0.6997644960166947,-0.28743645079359054,0.40942085582497184,-0.07411406715100573,1.0340402622772968,-0.15782037962659173,1.395480970496567,-0.07718834779645461,1.295454836012855,-0.06803907691956486,0.1216611902979893,0.26309073495381474,-0.9291782466475651,0.47318748511477393,-1.0961187359065676,0.3781165120488659,-0.452867303029683,0.12844444129687446,0.28299717220452336,-0.06883550914695531,0.4411737574688399,-0.11699759363976374,0.25113229166716783,0.06567124709779096,-0.1772954010727784],[0.8863837174284908,-0.3590774432354064,-1.1024772052372862,-0.22941235247577701,-0.5147029388859032,-0.11819557634702849,-0.786364835233521,0.1930622839917608,-0.7105375826134609,-0.1621871690461937,-0.6099883615652042,0.18098624084291784,-0.5506204119728384,0.0483744626897323,-0.46001724271093336,0.009129843408558626,-0.4343428824919371,-0.02673403623978789,-0.20126476497484594,0.20269034266539387,-0.17059026243502726,0.09013279183205714,0.07139316816429184,0.07662317890812449,-0.17349225607644747,0.08440554958773334,-0.09923984269597073,0.06881136276986421,-0.15026867120033935,-0.01114673422710549,-0.04906607888380698,0.049172980051783624,0.08844190678165374,0.17079432378138984,-0.12183016689006122,0.007803867934268354,-0.21383934595169413,-0.0220537268556511,-0.052461054040479925,0.07728433934725593,-0.24340125327357104,-0.19182276235028456,-0.15433671293629245,-0.03323603654598232,0.08132145654897224,0.07391591266418834,0.11827563066223339,-0.03523380060469059,-0.07856439739488752],[2.2633420152475114,-0.7963457242075337,-1.0876171065262172,-0.4925390448174979,-0.8190122441752351,-0.5563781214497961,-0.528388635052599,-0.4839522254314471,-0.1078379741932119,-0.5166406566402554,0.48238821572806206,-0.5653153492360486,0.49127369341181426,-0.5278157854731198,0.7671071168824316,-0.41746733306731826,0.9713097993756921,-0.46140662575005137,0.8421669584037836,-0.30372847446445506,1.034532126679682,-0.12258495295152538,1.240024718842263,-0.015079121037973561,1.4425920838184838,-0.05392848665073397,1.5143774748777976,0.12492424045073568,1.1109840270034903,0.2506144008413135,0.4855963884851585,0.20986246525551808,0.01576487124762949,0.3340285487615059,-0.09167066202845062,0.1685680587589512,-0.0552506209025876,-0.09642913525899169,-0.18955551514294855,-0.09331252056908154,-0.5428924342169146,-0.012433182705835419,-0.6931080909930828,0.028646929711423966,-0.888690710109575,-0.06380593555433468,-0.5471073989421555,-0.15908656084842007,-0.3328108551084157],[-0.4585472462026844,0.7349474748734242,-2.316341015844739,0.6566609346564635,-2.3747114095290596,0.8766979268493531,-1.7012469471765557,0.1899961973521985,-1.9858261414520404,0.3901468784949687,-1.6763805369763805,-0.161538422991136,-1.1325481797605446,-0.18912351307303746,-0.5479597557814466,-0.5484744970830587,0.16385625657674754,-0.49912042299069725,1.0257628951901916,-0.5975878473331303,1.3533217173449261,-0.3788162683791918,1.4182955326635363,-0.17556882143639824,1.5304236513107425,-0.10029632213784817,1.6229396379737153,0.12620149757135982,1.3632832045422594,0.15946579915228656,0.8345447846696964,0.37104852466500804,0.020646579738510255,0.3981140983205473,-0.3607228332050826,0.31477208243037436,-0.4819886419539929,0.17583788340588294,-0.5072108279305916,-0.07665712973981158,-0.5711331454146913,-0.13800039282095522,-0.9119396225450181,-0.052151775085533356,-0.7698324894820713,-0.0022107845401559616,-0.21117572503316911,-0.1176247140094898,0.3597368592512318],[-0.028617243863188434,0.20989404931719663,-0.023704370955521675,0.3616567285416257,-0.14974976708323925,0.29475331044609115,-0.09028774879076157,0.0520811908379588,-0.1271802689375203,0.05928861542384088,0.09040177262180855,0.18517063237065687,0.2457022592953697,0.10813454923945937,0.28004094288265335,0.21408475272683752,0.3925350229691551,0.1879868378176082,0.22823327894187972,0.1999696059194271,0.30066534913254733,0.2634082507872595,0.3914090337348435,0.3074688855275989,0.4874948692831693,0.3010814601557884,0.47644060903038526,0.1718142779139081,0.2755561365742755,0.04697487665437101,0.002877180890120368,0.03502580770944519,0.00832310752920797,0.1191163487448012,-0.07596178010542473,0.09534539460696322,-0.06819787378533729,0.07045252313975421,-0.12839479728688952,0.06933234194209575,-0.15879875877505747,-0.0966401968969674,-0.14196103345619834,-0.04837260633414242,-0.05930820215495722,-0.08251622687709835,-0.008270037377185175,0.01691079187390997,0.02149961593887077],[-4.170256507930938,0.5232146198906894,-0.16393066920380842,0.8142111743301211,0.06830548138228762,1.0432043199419303,-1.0050136787234836,1.1525679017351333,-0.6343651879164027,0.7408436572162516,-0.4960463733393674,0.4820176759087097,-0.26175938698429757,0.24752558339216318,-0.14831925425457906,0.27947804192859715,-0.49636446239342974,0.3892692985327971,-0.5612853366958752,0.46418436034939337,-0.44830367543300575,0.05466212679683059,-0.16055165927028595,-0.05533032792094619,0.10295431771912476,-0.25929977497095774,0.19633253840677334,-0.0012199285979416657,-0.03927744989951468,0.05986508692049685,-0.10716493495773584,0.06602097664012208,-0.19279405994052706,0.02981727762885585,-0.0053846758121402025,-0.04818111103053901,-0.0020345374049237064,-0.09410574372689304,0.1054572521715708,-0.0921767973616619,-0.0426636861990742,0.005204985027326551,-0.12139462663874086,0.02679861167928029,-0.12608544824287615,4.4542862448804174e-05,-0.03571525452800839,-0.09275235129188626,0.2134687331214676],[4.3499753310182445,0.8923977149922865,3.0533433492205675,0.8565897672122497,2.2720370557772127,0.4954105146997067,2.785381829544753,1.354443412805701,2.650755924215874,1.055014490362268,2.446928315425049,1.2076718302944485,1.891068778213393,0.8306159548219304,1.5074083871005608,1.0356698579539396,1.8463544602997792,0.6779580371012403,1.5817868368411478,0.6402847811905402,1.0111455907064,0.4312385499464091,0.25004517329708803,0.33930821858049653,-0.06618350991194222,0.22974071684933906,-0.21096021144968602,0.2799778442296767,0.11195662596619312,0.38834721291780516,-0.03603578704992839,0.0651688318307115,-0.5125178301567495,0.095741648845405,-0.45135326088085825,0.05654902301973391,-0.12085251804830219,-0.054016883607845435,-0.15782747878745385,0.025330233034445995,-0.28033415848249615,0.0026647174248083123,-0.28659489552534484,-0.08755559900688106,0.07255336677006034,-0.046970259814277805,0.18618390532119156,-0.023016283319813157,0.09146687690426854],[0.8747026186495638,-1.8012651618497446,-1.5997222231856434,-2.109030528425573,-2.763238808324123,-2.0475996963874388,-2.086114473530317,-1.8049327917393836,-2.1171109640529076,-1.638768164136376,-1.520628277245599,-0.929813200736944,-1.3222477571984814,-1.27242212684994,-0.9398297382035197,-0.652017798356885,-0.22969684939262042,-0.44004730123951785,0.0965997215945934,-0.23734864137790815,0.38574279918686083,0.08775309580730414,0.34296137267417987,0.041960584620597666,1.0405367532089453,0.5226912119607379,1.1986050602631335,0.34427712033833724,0.6660484872714098,0.10247451016542343,0.2535873966329404,0.14246436675204902,0.48170815696738806,0.5733900814847355,0.6424251011174688,0.267741772373369,0.2976024816073098,0.16046195405694924,-0.1228489296592664,0.004326211298007975,-0.33029463960878963,0.022397717630247924,-0.002494086756158125,0.23356896943341987,0.04511425766113056,-0.29609904315461155,0.04289692532071049,-0.013989366707653413,0.01173334345149707],[-6.463753516079932,-0.9537233399318156,-0.9217567632833488,-0.6388174843453891,-1.2060800369195892,-0.3454291021446878,-1.5972308551894405,-0.33015026576786743,-1.358171153830198,-0.28071853669257724,-1.2500445172323527,-0.16902773837795834,-1.1726811002734685,0.031531382773718274,-0.9941594641677683,0.017453943425786735,-0.5569063864824845,0.13054250840521334,-0.25658484901746365,0.01849368558236402,-0.13460192519258796,-0.07642159661811992,-0.3824961967492378,-0.06609571082821128,-0.6754317563753661,0.1420558122425958,-0.5649938912691767,0.09503388089195208,-0.001852891344684779,0.044183891630666945,0.27035499345085506,-0.016843394802347894,0.04979584579164123,-0.09269528815570828,-0.1348747027611207,-0.030916306721037602,-0.3569226953497298,0.07083312007495232,-0.07757085819409543,-0.030910757589254274,0.20182781289503626,-0.0263692035286092,0.30525888811280527,-0.0010592077980049637,0.02440101507663042,-0.08905199226729753,-0.09867789423496574,-0.08341943182523275,-0.013350425381699277],[-2.192253642878038,0.16822227948871102,-0.7402771284994699,-0.027657482882498526,-0.7065453564539004,0.2742088786483834,-0.9423779065113204,-0.212260683937948,-1.0760129279395392,0.1508556676689495,-0.9949563025214871,-0.06483898357358678,-0.40774102514190874,-0.045403017832785046,-0.3934011951028161,-0.017537148567932526,-0.17419764941017543,-0.2119263772813639,-0.05880447471819264,-0.22774196382580328,0.11525109937730431,0.018829303804518995,0.36283302038246273,-0.08183580309291548,0.02376931485757779,0.08920385327645387,0.2617676881389472,0.03706680950698069,0.11533518908266303,-0.13256884166299768,0.3835896386796918,0.02146309095712675,0.322798363432569,0.1275622380832786,0.2131592683215405,-0.030336044515711827,-0.1845861264655949,-0.11660749665712937,-0.2365550832449341,0.1296605541799836,-0.001999412897674979,0.03829777875922068,0.0008095164462993096,-0.128037825003858,0.033293303200433,-0.0527112503197226,-0.028430425339930204,0.04982594259362838,0.019369807490761556],[2.725692783469549,0.16583458591199557,0.16716583832588433,0.04346394997550454,0.2441998755442247,-0.28938827474317463,0.46044789868812036,-0.10245688809541825,0.5797110634201093,-0.30641016477652216,0.5495933294801825,-0.22278997385813562,0.3922852113932353,-0.154929803701261,0.283365917739565,-0.21547909430313245,0.22882391398951357,-0.14662953097594267,0.05565186410059617,-0.08777520680863032,0.19305685213027357,-0.09963537591448281,0.47082934475377664,-0.2279415128507268,0.8044110279764881,-0.17338889457099635,0.7730840641380717,-0.22482792303171187,0.6108125502268742,-0.0541211504249657,0.35243133905219115,-0.013836778725333829,0.39047971034772855,-0.006297228598948881,0.5848033528554442,-0.02722920165745988,0.5134222748777416,0.06217643177441161,0.17472437577582814,0.18660037891924108,-0.07333836449617466,0.20937967647407515,-0.10478708360591307,0.14794459737360469,0.06521410171180538,0.08603849792666537,0.016696441338702746,0.014873230591579311,-0.18338817959198764],[-8.159565850516012,-1.3086421409889855,2.6377499068501336,-0.7485338459636534,1.5312617870949408,-0.4121575458036704,1.5042633608129659,-0.675576210844781,1.226511980753243,-0.5974981361175864,1.6395147475926226,-0.8719601334082326,1.5904131856041872,-0.6169661173351922,0.9938124518425152,-0.4748740133785915,0.6387532227184818,-0.07417630676737964,0.6969766867849055,-0.08466801625043092,0.358415227575748,-0.27430201400317744,0.7153357927159222,-0.30498919361464916,0.043800794380375464,-0.10640520853441052,-0.31970371513431756,0.13873509042150273,-0.2489171369237756,0.2844676745732143,0.10285384871118815,0.024485903535197907,-0.1314234610057972,0.04138818974151085,-0.34785477303848833,0.10476887434168942,-0.46447367356857333,0.06917423796455245,-0.4401639508657287,0.024247791909176794,-0.2635686940590726,-0.09029775817695454,-0.2065172513733582,0.10417719795040864,-0.5175078995725493,-0.04234505101761111,-0.34071792062633016,-0.026274824573171793,-0.13054750991762026],[1.7563967748510212,-0.043788874671164546,-0.04582694235463773,-0.10333524033790602,-0.42523538494438295,0.00605540522829735,-0.5989968381637308,-0.019736961788151878,-0.5544001462619946,-0.3980578168100528,-0.6754607462639278,-0.25366088219974425,-1.0237620550515765,-0.21899273231730476,-1.4611120290988613,0.10136030723710619,-0.8179197310788194,0.2988290709607618,-0.19740094278369563,0.16523521194019478,0.3165253126811375,-0.06573670781014142,0.06751395578370448,-0.17305165580354948,-0.31903469398967393,-0.19672211953230334,-0.8370743582665842,-0.12155668367695015,-0.6541605845721484,0.07329279316315161,-0.19502437628643401,0.02861657309614102,0.25372694583296074,-0.009717739014730947,0.4331343751325685,-0.13002361394332437,0.15585953429128108,0.07235635911771847,-0.07865194233907764,-0.08926405018584833,-0.27614899061032644,-0.07481635074880867,-0.12084068399183888,0.047174784556725295,0.1297283507361658,0.040710087249755,0.2903494167759993,-0.012063612323446089,0.07512806086740105],[7.114479097978732,0.43986813621188414,-3.6230187792417077,0.528607838672622,-1.5670151858764654,1.0303257493058036,-2.3880688141670374,0.4715988730477871,-2.3465561489628537,0.5254097913540273,-2.0721811992338703,-0.136877172263341,-2.5189321394030486,0.05206456670711842,-2.1180938911464975,-0.20649725956474457,-1.9267771770447149,-0.011482186638496164,-1.0975796678284198,0.1069395864657042,-0.8521167413989253,-0.22504791984809291,-0.15440539726118618,-0.159019534623725,-0.07787673424129207,-0.3113772580151196,-0.2783345702805371,-0.10913704890730776,-0.3444737251083285,-0.1461432617907464,0.16164772068093794,-0.22811532104204627,0.4828138875255926,-0.14142508308935012,0.5472915519113599,-0.01703115516776897,-0.002533451980699363,0.03428052546671163,-0.3753712220823569,0.08129065578698423,0.04722591489499006,-0.06091280596363825,0.5072981440962677,-0.0502296530898074,0.27182831199416424,-0.03895504722766172,-0.21716980294084484,0.11840751541141971,-0.40691482140930596],[-6.517326054211479,-1.3901110515502486,-0.7801283369265342,-0.8531087048925026,-1.0541984584290043,-0.7166347342134124,-1.1880195606173158,-0.7309538105676313,-0.8240331086319261,-0.7182591697152928,-0.8436483415830094,-0.7357145972277475,-0.3063686320486125,-0.7132690928632307,-0.011256487236012419,-0.4759249925927306,0.353624644372131,-0.4005084173122096,0.7941213507520911,-0.411580962814223,0.9989039504901258,-0.15798507526980798,1.0370438459099944,0.13695877640389345,1.1360039315316297,0.24110687380975268,1.0870420760569768,0.4003756115875516,0.6523965049343357,0.3655093410708545,0.5673396403985956,0.31399977005697516,0.4045501474790428,0.18041705355557056,0.3708455324735649,0.31637430505885106,0.11260127991510327,0.22687746794612534,-0.13515013501765893,-0.025301973697057296,-0.29298740573613685,-0.10292369814548276,-0.4965570671179758,-0.2564362254676375,-0.3560933873545593,-0.20267065402047144,-0.37887422345852445,-0.0014392513751890486,-0.24784173558862158],[-8.625392156180656,2.1416192571033235,-4.52772066646021,2.2699217306531922,-5.487801140484644,2.190912505063551,-5.7185321611164195,2.0651695166129427,-4.990757445720184,1.6138128723881693,-4.860590011392139,1.5423445458403768,-4.2491049856074765,1.2007004372241052,-3.5005795316315154,0.7312448834750506,-2.9914834133907147,0.27098956444049976,-2.4533246656930046,0.0801132364908587,-1.7829455984713791,-0.08030548255176571,-1.2257231918113654,-0.24038168783687675,-0.1379556545257996,-0.5681584535999894,0.3287087225334189,-0.5172873815029876,0.42088082186451525,-0.48010167791984143,0.30555311012339853,-0.25391002364342363,0.6011144934060181,-0.28950264154338595,0.7757255117218335,-0.3046373073246848,0.7847292931512213,-0.19328477781648468,0.5515404080874845,-0.022957036110106317,0.18499171927635907,0.07196728761344913,0.17975669679152823,0.02414050698685047,0.22726485134394073,0.04098049585902074,0.019835073129685275,0.11152755883153274,-0.1407876206227505],[9.208959338857936,1.7819652099878052,1.0794552802271302,1.4917325089159772,1.227687496247338,1.3255877243127274,1.0996400322415743,1.0188573482501933,0.8126325840411653,1.1162523412910448,0.5857315828540349,0.835149605574758,0.3721492804095659,0.4281778010475444,-0.13984996575026637,0.39315355988477485,-1.130695064382441,-0.05548520750531089,-1.4469145212197252,0.09057021100834646,-1.995955901655668,-0.026745643489666602,-1.1271178521298693,-0.028390113219192586,-0.5218186651400704,0.04389854088292573,-0.0886431895434734,-0.198939385185676,-0.17573724351806422,-0.28797039788679984,-0.893020619925908,0.005272378212445427,-0.7137432990743249,0.061837476215318675,-0.5205042415390904,0.08710224159911356,0.24382003018950402,-0.1939378153026458,0.5632208435876482,-0.036962347630639374,0.22950832062154292,-0.020987986743342037,-0.0030413177812283723,-0.022214365754000015,-0.31798098014674564,0.19090320589436635,-0.20703605059686697,0.1436885526793554,-0.009338790329678118],[4.0760103021295135,0.2870192323690755,-0.631064280346432,0.17615621489979075,-0.5623384295543015,0.21854392582168275,-0.28598472438353967,0.06883124160231409,-0.5447439767563859,0.2147803562531755,-0.5009686941456235,0.0960877891420049,-0.2982322116382332,0.08842755205351582,-0.04812355771556858,0.058583755577635545,-0.23018835468142645,0.005274766996890387,-0.4702373483509772,-0.007646695210999141,-0.6429874126005103,-0.08521946380731549,-0.34183452496609146,0.00914435435738864,0.18291241493549493,-0.02351725315144318,0.35321133250764064,-0.06524389168427583,0.038782808369089164,-0.0008533201105959085,-0.29511738702817153,0.0704652185680732,-0.44274983943501833,-0.0694289877551643,-0.07082560146073129,0.02716192868118676,0.4531426962130991,-0.06988304766511293,0.11447201046895532,-0.04133515735512907,-0.22856687610414184,-0.0018148623139245805,-0.2856531354019522,0.010905298907214738,0.17181517461802134,-0.01971956127763494,0.34639884199566184,0.023308075157368394,0.13028718495796426],[2.9436830643522773,0.6422475902353784,-2.0739033701595972,0.6310313235190492,-1.2956664336545787,0.8468170246290463,-1.6157632031852047,0.3240033171704331,-1.6231181347077266,0.19733487211746864,-1.484346862610634,0.01457613126573323,-1.4518415648672602,-0.2872370448173638,-1.1523282965293804,-0.3201481976721813,-0.9464447293128694,-0.4277915810946059,-0.5794303403281,-0.3429346183648629,-0.2742717731763192,-0.34593037036253776,-0.1427325464177998,-0.21287345291036652,0.06560603280028329,-0.23499095041928375,0.15809273515621122,-0.07640081449222887,0.34081487423386864,3.356405793600786e-05,0.3042590390225067,-0.058395026762711924,0.07390039509733877,-0.03619878796029124,0.12874291350807698,0.004874305611188228,-0.00421513719203346,0.04540343968904163,-0.06276459145086513,-0.1352379097218671,-0.07287339923799174,-0.016011738055287494,-0.16251951557168964,-0.07302516204277774,-0.23658580942782226,-0.038522741875151784,-0.15903931567993634,0.032803428697333104,-0.06970045906296966],[0.2980912737478862,-0.8296316785186362,-0.6427204863515076,-0.8914250095235464,-0.9714488049023732,-0.7696497050477924,-0.5926510471725782,-0.7515608124210662,-0.6478291888567441,-0.4240870980419725,-0.5268448631371088,-0.36284572056997727,-0.03876888228266671,-0.2053746976009089,0.2026999001866629,-0.05915113180675422,0.5451846994586311,0.027994749558571282,0.7982750550362806,0.06584018360204556,1.0005668051275494,0.09823721746860833,1.1084380614099383,0.11446117398840731,1.0229712739106371,0.17258674693289286,0.8979225558731299,0.1954637270667622,0.5616872491573598,0.1729443864428733,0.2566895894881551,0.10519137762399028,-0.037996327030182145,0.07723751915650003,-0.24829788863532332,0.053479389456215394,-0.4326896341889629,0.07650729606583725,-0.49564110146082496,0.047803779749784864,-0.3570463551735352,-0.002120017983974356,-0.26331015745395636,-0.08249554743240392,-0.18401740934486974,-0.06980927995655285,-0.021811135343939227,-0.02473439753901136,0.14360864587519034],[-8.300249343264058,1.307700478587467,-3.0625290335219346,1.1391021407312762,-3.294898155233214,0.891236667128604,-2.9899501962513537,0.3169509447447099,-2.5921666294235406,0.3709904144482073,-2.046437934548891,0.18251975911187684,-1.8083485649513584,0.2479322045391074,-1.6149795117986692,0.23775160783945773,-1.201487910496634,0.08461326669167159,-0.8025068024142308,-0.06600031376755909,-0.3351055022065746,-0.24495191855501755,-0.4432747297573758,-0.0890677773441755,-0.27698938695381875,-0.18589164669469269,-0.43601370541984125,0.08939884866113353,-0.14366379748483085,0.03826485101785969,0.1393007040118802,-0.08812712163869542,0.21948859948394026,-0.17198953853443394,-0.11086899453456989,0.02050503993681583,-0.11791687959942294,-0.0887959447180046,0.2697669642619105,-0.03400273776615365,0.49613029673131503,-0.013320322565740726,0.2398584684593239,-0.06255397093707882,-0.08673677663940102,-0.005072179203087242,-0.24646093107929024,0.08903948010323126,0.15592925991677584],[-2.1417817928981484,0.988009792957976,-1.7662926752195751,0.7658676581718723,-1.6988305601328255,1.0662546265058508,-2.461830521233104,-0.03714905518011773,-2.4631219982993464,0.3206796320797686,-2.5230646759695774,0.05198102853517479,-1.8297852913402342,-0.1572225912266284,-1.5406267211473712,-0.06486922233703846,-1.3582219756317824,-0.06663004541469894,-1.1804948183380144,-0.14340018207300986,-1.0270643215034148,-0.3738819690628357,-0.2678476564678714,-0.11521381692784076,-0.027752948195921927,-0.09932813497281069,0.7694994349869266,0.18110120748340522,1.2698329712989462,0.22225447555425684,0.8083757709011603,-0.08083459408624898,0.15798349190133695,-0.08487454329147126,-0.21618612230411574,-0.12755192836788695,-0.044984551895841725,0.04966589656533224,0.17091700951259575,0.10658049837061052,0.3035354031371352,0.14237145309118954,-0.058572006612954745,-0.029943625744251842,-0.36227676988453816,-0.011146078267471285,-0.35176877149513863,-0.0759401611327447,-0.08178688062154565],[-1.2973787945599624,-0.8687428916572396,5.566281296043477,-0.266667374579095,5.360587035740227,-0.29372255615670834,4.201324181364183,0.2415751942159489,3.7913516078941654,-0.042848345265625185,3.1334035352503125,0.09265898704167921,2.395257667353401,0.24594752029242498,1.390346748369013,0.5184966472910222,0.5173089888765109,0.8897419266437805,-0.18085922313029082,1.026851275008674,-0.17061323794567565,0.7263734811423721,-0.008964793019586869,0.569396865256466,-0.29110343224825064,0.4202019389324055,-1.1499560439048029,0.5846996610592256,-1.6743656595419494,0.4556630011850031,-1.4590542900554184,0.18771345359910768,-0.6304142705152,-0.08908204728938832,-0.25968943064646915,-0.14668404416425643,-0.32327075450687376,-0.1135655513824655,-0.781144735038388,-0.045848424138825776,-0.2555964931484182,-0.11654790640765672,0.23699848404407614,-0.11204964520324688,0.6426017296879472,-0.2343891565265116,0.23096802962892898,-0.08775159060187689,-0.3204811150774356],[-4.825921398454521,-0.6145263316081738,0.02269585476234022,0.4241655720352896,0.7429730675838994,-0.3278045855184548,-0.6290910234496226,0.09445946547092957,-0.38764651810324585,-0.32103526846953084,0.357495851483561,0.08568057172370108,-0.763651904513346,-0.25798614363097117,-0.5968414099559022,0.018919333166347273,-0.7900483595172313,0.05011238510352672,-0.4991207174146122,0.001726415338778473,0.2300169124008952,0.15273689915247626,-0.33619780653660375,-0.308645651105339,-0.02357992892689292,-0.21608832300062877,0.29380448277154053,-0.3518494411604469,0.28351111730831574,0.10329461090967468,0.5966996696459484,-0.14160053480106394,0.21777159276797306,0.03792943687680752,0.3073245349337435,0.04513862156727929,0.0666229844836216,0.11231109548244349,-0.198126283358015,0.06264089316779448,-0.4283274461691099,0.19068361660869587,-0.20644205214557126,0.23404251257519348,-0.09824384275992172,0.07264229898874064,0.10327743834343643,-0.24093419709687888,0.07473523973838352],[-2.711839155511196,0.006530795397283844,-0.8753482879623673,-0.14311820916305293,-1.1131858251643016,-0.8438388193243903,-0.658787288493429,0.6572600883365399,0.5410737982768451,0.22470728074625093,0.34247297001371496,0.502754466326908,1.0373334968485228,0.5351134255195401,0.7681543294234736,0.5325853789002869,0.6913414136639875,0.32511047819372163,0.5685056038897484,0.44805332490234795,0.5315943235454892,0.3190484640865052,0.5179768519414353,0.05990521694807176,-0.003841323981356335,0.2132229044256521,-0.10065105224467616,0.0746904096726351,-0.37980796520531696,-0.07520257665073618,-0.06951444271087269,0.02693968778691138,0.044965154402115286,0.036029899283574016,0.042165958088220094,-0.0018970639270325408,0.09158792794053928,-0.14430071249779927,-0.11502844715502657,-0.03436893018589878,0.012574409937061238,-0.03784940854551831,0.16222450460800852,0.10633217909711176,0.2832244766777134,0.002016464711389161,0.08965326565048864,-0.11029324157137782,-0.11945068301118307],[1.6286922866402744,-2.0235484905482717,-0.05073975861504386,-1.2099768018058308,2.3962064361945883,-0.6157062973960726,0.9082052329825421,-0.21114761206067056,1.171280951084236,-0.08985451609879483,0.6209049355853842,-0.14143993531915114,-0.1434243046616769,0.24904541753932338,-0.2082880750827714,0.2313662583266918,-0.2927482089188147,0.12153238430140414,-0.15065338569316172,-0.09472121914639615,-0.2411772750283573,0.31839244554148377,-0.33157713404140127,-0.16236115529719308,-0.2868870904008335,0.06288764319173994,0.15043387711041373,-0.12041306088978875,-0.02173121024320151,-0.15580990065368983,0.344659055521178,-0.0683704531814798,0.3562481789588286,0.03324902116993882,0.2522880554685944,0.013724326017044451,-0.08183899458855695,0.15108186909489868,-0.348277146517526,0.13213756751450254,-0.33714657268992676,0.06353854098132487,-0.01927218987900034,0.002653609590836964,0.20777004127143117,-0.1033829659738503,0.14255485031385698,-0.14494075179466992,0.12436224979107197],[3.678476732694851,0.18727661917283425,-0.8128550889578594,-0.04825734627733828,0.09875432762735419,-0.21963416041267414,0.3070549983163091,-0.1654670526838539,0.1704233814748779,-0.24743717481414637,0.025482716892630396,-0.2455529974638497,-0.061552926235718564,-0.20240592499663318,-0.1543859823382323,-0.03957925476903596,-0.08682414088683724,0.005238299529128314,0.31394219613799584,0.002378447326053664,0.3759666751059026,-0.14027164077096496,-0.05339987066167518,-0.2638098115800093,-0.5203173133424368,-0.2464087936495921,-0.4136522241851026,-0.17303624465898526,0.2716149249275296,-0.08196017387023723,0.6948998761138198,-0.05582892336332709,0.4005258102178124,-0.08647884318359683,-0.08886336409965766,-0.06212671703583714,-0.15967982557516622,0.025259696632809414,0.2315369207331512,0.05285284302887685,0.4066133126321376,0.02643880491582449,0.1150007751547409,-0.03123592977480737,-0.06172998232390838,0.00969710312129149,0.03378169534267771,0.030422419050209374,0.1973979271030937],[0.9631881406637907,-0.9806953644467001,-0.8160755999851745,-1.4412071407341571,-1.0627244346383635,-1.4369031475810317,0.20047302238290882,-1.4439795541637037,-0.01740358214349193,-1.1508317110058623,0.5041788439200002,-0.853478699149376,1.6129299472566678,-0.4540986972485172,1.6780855879808696,-0.452022783770971,1.8414914369721582,-0.037771003678876994,2.0885761126486035,0.5507135972705537,1.9522257219288583,0.40532109567382407,1.5409048693887828,0.6583543377617865,1.2163443612542577,0.7427597810058852,1.4854955833130934,0.5837556474143146,0.8921744641814049,0.5216259165275151,0.12036486566983642,0.3946045054899316,-0.22586792011937576,0.4661419299933237,-0.3065030971264551,0.1934662967246348,-0.45307631315389674,-0.03861240198112309,-0.3769141367552142,-0.07018183377116034,-0.42050486989838487,-0.16262505279127343,-0.6769481089374135,-0.16321027846738614,-0.4337271399978781,-0.20373275587415218,0.016163194428369772,-0.1963802660752534,0.06059302803011539],[3.565868837319612,0.3927854554484129,-0.8856087061666432,0.3157512403300207,-0.4214300086899971,0.27589330042518967,-0.19234946272195338,0.09468438895504515,-0.0027758328686024554,-0.06394171193671305,0.15291305814364567,-0.13744173172341612,0.19050796365509115,-0.07132377551778661,0.10398074178870068,-0.015268457482448393,-0.024184640772144054,0.047823023461842896,-0.06136313432592942,0.05677050254341694,-0.08711034765438835,0.005952829390193339,-0.09190043729400943,-0.0836868543827259,-0.17562476873916572,-0.02134434115845643,-0.19610194762375657,-0.017388522652236157,-0.1096795394750421,-0.021556311553079757,0.13098064644015792,-0.03985847534309357,0.21839224662590476,-0.050473868004169024,0.12585837639852265,-0.03296970304630147,-0.060534116907098735,0.017804735978441462,-0.1664577216359507,0.030013999943490115,-0.15588915352061758,0.055170297573223344,-0.1011706531046019,0.018259984310941557,-0.0471339384356366,0.0028347957218353564,-0.008728715917804844,0.006247661651328167,0.01422848942621687],[-0.812314134119041,-0.36277835761605115,-1.0455450435554632,-0.21046926657941115,-0.8378827399227469,-0.2880286836731878,-0.8790123772635745,-0.07381726908139633,-0.7141964099500889,-0.13400028480211643,-0.6593135422769214,-0.04660842787802144,-0.4790358090120466,-0.11607544412760058,-0.37680977251820824,-0.08366506847273764,-0.26753355480182617,-0.022221173374590255,-0.24037817975493997,-0.043145236784672504,-0.12666704757868166,0.010958276657145076,0.058662300481663474,-0.15874627259539162,0.16124774252973967,-0.053165616004725455,0.16416172182427172,-0.030390026330825257,0.004186645875610611,0.0015741999837853276,-0.008568851794120877,-0.09848249165389517,0.010513805295006352,-0.03152522865793203,0.07292395148110971,-0.013101067546144825,0.14113882236993885,-0.03627208442969373,0.047820732276806284,0.05017155349622105,-0.06758966824299584,-0.10307594251760445,-0.11134920808880111,-0.010027382854242902,0.0764618560535355,0.040565321677818206,0.08652813672530879,0.01827548788987545,-0.016425798289693747],[-3.989682642050149,-0.9190217509439432,-1.8518914574970984,-0.9696977258502791,-2.185235346761134,-0.9967950255981419,-1.2422700880362922,-1.1957526541987282,-1.1293400360599597,-0.9980798492002037,-0.9744174080369972,-1.0849074295044931,0.21592831546249905,-0.6957991057105416,0.4483316609186723,-0.8892999309296423,1.1181554071322695,-0.36763851070132486,1.688985109404518,-0.41291914626254955,2.130137330470074,-0.3807959567944768,1.8247935850801777,-0.3114755645714467,1.7235321778749404,0.0013888126606791317,1.2114647340293063,-0.1726687104330822,1.2533791649833668,0.12921969985416087,1.6170333327192716,0.17457933421068855,1.7161738638149524,0.14780865413934768,1.0515919861777205,0.26492607829153364,0.05347539083072003,0.43338256924642365,-0.6251179810630595,0.3503313861618387,-0.1322068485158659,0.2233519327858274,0.31436673504984114,0.11133647105467206,0.2365213558076574,0.06596276031470652,-0.31801743476887157,0.12241149150985839,-0.761874987644998]],"nearest_scaled_distance_max":0.7055220985873271,"tune_rms":0.0013420283458752602,"test_metrics":[{"regime":"all","n":64,"rms":0.0014337349015822376,"curve_rms_p95":0.0028656965366520284,"abs_p95":0.002247034468110476,"worst_abs":0.020073186020201583},{"regime":"grazing","n":17,"rms":0.0009434777835747526,"curve_rms_p95":0.0015681286746222216,"abs_p95":0.0018271255363991005,"worst_abs":0.011469228670860532},{"regime":"near_free_lobe","n":12,"rms":0.0017745302168363994,"curve_rms_p95":0.003404099542285593,"abs_p95":0.0028338764128746467,"worst_abs":0.017208996358746842},{"regime":"unequal_mass","n":24,"rms":0.001578997947556666,"curve_rms_p95":0.0028656965366520297,"abs_p95":0.002911720099025794,"worst_abs":0.017208996358746842}],"response_metrics":[{"anchor":0,"parameter":"inclination","native_response_rms":0.0038146448633453265,"difference_rms":0.00013891954647888796,"cosine":0.9994344382109519,"relative_norm_error":0.014545017716882066},{"anchor":0,"parameter":"mass_ratio","native_response_rms":0.0017999994709697108,"difference_rms":0.0005397561371251154,"cosine":0.9638626254088193,"relative_norm_error":0.17379617105897746},{"anchor":0,"parameter":"free_fill","native_response_rms":0.0029526631297968255,"difference_rms":0.0003232284211974435,"cosine":0.9973063067109461,"relative_norm_error":0.0785687156324185},{"anchor":0,"parameter":"temperature_ratio","native_response_rms":0.002667336231374198,"difference_rms":0.00021386990599918217,"cosine":0.9972870496466714,"relative_norm_error":0.02907486741308267},{"anchor":1,"parameter":"inclination","native_response_rms":0.002769675702960215,"difference_rms":0.0005842401404163622,"cosine":0.9814204567978351,"relative_norm_error":0.06907044133261848},{"anchor":1,"parameter":"mass_ratio","native_response_rms":0.0007082840436591075,"difference_rms":0.0005338490868039662,"cosine":0.8481196561281216,"relative_norm_error":0.38421985488520827},{"anchor":1,"parameter":"free_fill","native_response_rms":0.0021365439299288066,"difference_rms":0.000667680091622931,"cosine":0.9589684407728877,"relative_norm_error":0.17248375172963515},{"anchor":1,"parameter":"temperature_ratio","native_response_rms":0.008025035705077803,"difference_rms":0.0007169922150317482,"cosine":0.99668260361189,"relative_norm_error":0.04017711614882279},{"anchor":2,"parameter":"inclination","native_response_rms":0.0005496572765763394,"difference_rms":0.00020848581781524644,"cosine":0.9292054413062082,"relative_norm_error":0.014600780410399894},{"anchor":2,"parameter":"mass_ratio","native_response_rms":0.003271690158755787,"difference_rms":0.0007548057990091232,"cosine":0.9801339457574936,"relative_norm_error":0.09798483646308694},{"anchor":2,"parameter":"free_fill","native_response_rms":0.005011252713064205,"difference_rms":0.0005262037587632038,"cosine":0.9970565905431921,"relative_norm_error":0.06880459450689819},{"anchor":2,"parameter":"temperature_ratio","native_response_rms":0.0027486877396028626,"difference_rms":0.00030039243659771595,"cosine":0.994911110458097,"relative_norm_error":0.03723717436328075},{"anchor":3,"parameter":"inclination","native_response_rms":0.004595045434196302,"difference_rms":0.00029225183331796665,"cosine":0.9979754931564052,"relative_norm_error":0.0025114961357378864},{"anchor":3,"parameter":"mass_ratio","native_response_rms":0.001049070394671133,"difference_rms":0.00031792369577634324,"cosine":0.9534653563077795,"relative_norm_error":0.015921395228844094},{"anchor":3,"parameter":"free_fill","native_response_rms":0.004432998955487954,"difference_rms":0.0002340216835547168,"cosine":0.9987953899832387,"relative_norm_error":0.018265945423254396},{"anchor":3,"parameter":"temperature_ratio","native_response_rms":0.0045719930992180625,"difference_rms":0.00015157875796854858,"cosine":0.9997459939724364,"relative_norm_error":0.024061096451620925}],"lobe_q":[0.401200155261904,0.4015687393955887,0.4044807047583163,0.4085198030807078,0.4095939263701439,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.4144474447704852,0.4181340354494751,0.42313442658632994,0.42401410061866046,0.42785461237654093,0.4341369745321572,0.436160197481513,0.4366842893883586,0.4403940660879016,0.44172012927010657,0.4436000878922641,0.448724143113941,0.4497563971206546,0.45300020538270475,0.4589476342312992,0.46306251669302584,0.4675754750147462,0.46899757720530033,0.47068903017789127,0.47265154803171755,0.47600039495155216,0.4820480562746525,0.4824000266380608,0.48496310114860536,0.48514311676844957,0.4949123296886683,0.4968438160605729,0.49771239664405587,0.4996710051782429,0.5074661692604423,0.5105645062401891,0.5112511400133372,0.5131116534583271,0.5163584657944739,0.5243000575341284,0.5259080225601793,0.5266363395377994,0.5274786629714072,0.5296443640254438,0.5317095275036992,0.539724374562502,0.5436880416236818,0.544123256020248,0.5479813102632761,0.5516640385612845,0.5547733516432345,0.557121213246137,0.5633885374292732,0.5663043651729822,0.5667034233920276,0.5708768266253174,0.5711175087839365,0.5726112561300397,0.5746639431454241,0.5769141212105752,0.5861754746176303,0.588973395805806,0.589429580885917,0.5906713070347905,0.5936794446781278,0.6007482749409974,0.6016696539707482,0.6061309512704611,0.607956948876381,0.6094750775024296,0.6131054541096091,0.615155132021755,0.6208219749853016,0.6249415781348944,0.6271761365234853,0.630850818194449,0.6332237894646824,0.6359695768915117,0.6413411245681346,0.6423624033108354,0.6452727593481541,0.64687534654513,0.6504027280025184,0.6551620128564537,0.6578702619299293,0.6624268866144121,0.665665437746793,0.6663374233059585,0.6714682284742595,0.6770122815854848,0.6783866723068059,0.6785431783646345,0.6812374598346651,0.6818011805415154,0.6851552845910192,0.6887970044277609,0.6947219521738589,0.6962754281237722,0.700399000942707,0.7053848385810852,0.7069962125271558,0.7084242063574493,0.7123913016170264,0.7166673555038869,0.7169374480843544,0.720464235264808,0.7212874529883266,0.7259317113086581,0.7264086969196797,0.7321887877769768,0.7349904640577734,0.7377593285404146,0.7401474073529244,0.7421246614307165,0.7443764479830861,0.7488722921349109,0.7539655329659581,0.7572002217173577,0.7573885622434318,0.7632375347428024,0.7663291569799184,0.7702925759367645,0.7761155947111547,0.7782653483562172,0.7790229466743768,0.7801729907281697,0.7809712572023273,0.7887766656465829,0.7924543513916433,0.7953784026205539,0.7981847995892167,0.8033932412974536,0.8074523681774736,0.8081514859572053,0.8088636308908463,0.8091161441057921,0.8119436450302602,0.817701027635485,0.8234039513394237,0.8243927761912346,0.8278273364529014,0.829973245318979,0.8340946723707021,0.8370315521024168,0.8425200197845697,0.8434570570476354,0.8468133068643511,0.8494025446474552,0.8505348215810955,0.8519465863704682,0.8603040479123593,0.8650622808374464,0.8683455687016248,0.872857458330691,0.8753588965162635,0.8770822040736675,0.8810561708174647,0.8844454880803824,0.8847312782891095,0.8887703690677882,0.8892442231997848,0.8947696642950178,0.8964276525191963,0.8976695857010781,0.9037172336131335,0.9068495944142342,0.9120294169522822,0.9144900411367417,0.9182864950969816,0.9184806359000505,0.9229724223725498,0.9232147096656264,0.9266144397668541,0.9280883865430951,0.9308904886245728,0.9349316849373281,0.9357364951632917,0.9376713552512229,0.9390901667065918,0.9437616955488921,0.9486173463054002,0.9581840155646205,0.9600162534974516,0.9670626328326762,0.9743501561693848,0.9748267084360123,0.9748680429533124,0.9773685356602073,0.9805451068095863,0.9866542862728238,0.9893904697149992,0.9903315395116806,0.9907765749841929,0.9925442605279386,0.9967737269587815,0.9985815685242415,1.0024597074836494,1.0078459125943482,1.0123417584225536,1.017653219588101,1.0203806799836457,1.0215800683014096,1.025873350352049,1.029994351603091,1.0350119524635375,1.0407594352029266,1.0412792900577188,1.0446359656751156,1.0461826170794666,1.0477207264862956,1.0505266909487547,1.060076234303415,1.0608849831856788,1.0619128643535078,1.065879040118307,1.0688689798116684,1.0738938737660648,1.076349457912147,1.0777841597795486,1.0820602270774544,1.0882400391623377,1.0908095367252826,1.0913215653970838,1.0975786552764477,1.1008228960447015,1.1039110850542784,1.1042038464918733,1.1044624374248087,1.106822624616325,1.1163892787881196,1.1220693450421095,1.1300945370458066,1.131107745692134,1.1319534185342492,1.1357624093070626,1.1435575717128814,1.1493590883910656,1.150558942742646,1.1553437613882125,1.1577281077392398,1.1591283063404263,1.1613444555550814,1.1647285031154753,1.1654597705230119,1.1673921219073238,1.1724716759286822,1.1734718666411936,1.1765320098027587,1.17867192896083,1.1798804242163898,1.1810449664480984,1.1902496783062815,1.1932189473882318,1.1994862698949875,1.207911201566458,1.208748641703278,1.2092882149852813,1.2122044752351941,1.2136896587908268,1.217047043517232,1.2202934300526977,1.2250618956051769,1.2270874017849565,1.2298205560073256,1.2331139516085385,1.2368341637775302,1.2393701110966504,1.244221693277359,1.2540081444196405,1.254613379854709,1.2551636132411659,1.258861647732556,1.2596281518228354,1.2666670695878566,1.2707862469367683,1.271654430218041,1.2743682867847383,1.2747567754238844,1.2785443244501948,1.2826988952234388,1.2830401551909745,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2885727552697062,1.2913702439516783,1.2966854945756494,1.2974075435660781],"lobe_fraction":[0.30285678420799295,0.30292981940154246,0.30350483884455426,0.3042966481841868,0.30450609701082254,0.3052638526533139,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30526385265331396,0.30544673498811215,0.3061549822883495,0.3071071910042994,0.30727371302849277,0.30799729220449346,0.30916908495927037,0.3095433793246244,0.3096400941075366,0.3103218680055493,0.31056437855152674,0.31090712117644287,0.311835039320697,0.31202087251741395,0.31260247643263184,0.3136596212388261,0.3143841771392724,0.315172497394529,0.3154195544637975,0.315712567728906,0.3160514052464481,0.3166268133338521,0.3176571642517128,0.3177167864070938,0.3181498318864638,0.3181801722727339,0.3198123116293099,0.32013170499821825,0.32027498495853773,0.320597282664249,0.3218692729232528,0.32237015388112605,0.3224807991973714,0.32277995916794733,0.32329978166474155,0.3245593903956357,0.3248124083599163,0.32492678990296514,0.32505890464868065,0.3253977448633423,0.32571973424671163,0.32695914155616457,0.3275661616835633,0.32763257770384985,0.32821931903724144,0.3287760417768864,0.3292435585797439,0.32959506818679996,0.3305270729125559,0.330957591838823,0.3310163612643638,0.3316288168196156,0.3316640176156869,0.33188219417641035,0.3321811966579936,0.3325078919424548,0.3338408460230667,0.3342399015104428,0.3343048071263177,0.3344812553071372,0.3349073599453993,0.3359012319212935,0.3360300172900897,0.3366511452251289,0.33690420919378133,0.33711409579377954,0.3376141432134397,0.337895311979879,0.33866838744124494,0.33922648082195506,0.3395278422422254,0.340021364179059,0.34033871112160324,0.34070460404732594,0.34141636442663303,0.34155109114187454,0.3419339843575826,0.3421441705387443,0.34260517769824594,0.34322367941879484,0.34357385420034964,0.34416013949145463,0.3445746547197306,0.3446604398632856,0.34531290566525064,0.3460129465899448,0.34618569824211415,0.34620535019140025,0.34654302959500644,0.34661353096014075,0.34703193692790923,0.3474841549669572,0.3482153512929485,0.34840614248287394,0.34891074063093486,0.34951731355101656,0.34971253096642296,0.34988519901253506,0.35036325426074055,0.3508758744307943,0.3509081613440118,0.3513287551548247,0.3514266634601067,0.35197714919136847,0.3520335071834538,0.3527138199960896,0.35304183934211153,0.35336491366152484,0.3536426821237162,0.35387205569767843,0.35413260733428603,0.3546506988079519,0.35523425613876,0.3556030265215086,0.3556244544409318,0.3562875221335425,0.35663615144211086,0.35708123509367756,0.3577314061287034,0.35797032255060207,0.35805437716468197,0.35818183183751534,0.35827020046661356,0.3591299796620747,0.3595324137524347,0.3598511738610631,0.36015610982118795,0.3607194757989129,0.36115623213328624,0.36123125476975115,0.36130761474079415,0.36133467587881907,0.36163716815967334,0.362250156018237,0.36285348615857266,0.36295770984356107,0.36331883544020105,0.36354377401136023,0.3639743077686446,0.3642799189923186,0.364848435004406,0.36494515887841655,0.36529079817806864,0.36555659382350325,0.3656725940454413,0.36581702992550474,0.3666676162374817,0.36714851789300673,0.36747894110097645,0.367931146595509,0.3681809320083516,0.3683526355276971,0.3687474097291901,0.3690828165056136,0.36911104436413616,0.36950909810311433,0.36955568770266706,0.37009727577356516,0.3702591878951728,0.3703802896795411,0.370967808047219,0.37127069153167575,0.37176944852646343,0.37200546527381584,0.3723684666064728,0.3723869923755721,0.37281461398364246,0.3728376253669958,0.3731599309893517,0.3732993272670717,0.3735637696361411,0.3739438583114104,0.3740193724321961,0.3742006723474228,0.37433339817593053,0.3747691015366834,0.37521987044732985,0.37610177317483307,0.37626974779123934,0.3769129869741282,0.37757368777563355,0.377616733184324,0.3776204658765263,0.37784599903641036,0.3781317408710807,0.37867887306006637,0.37892290395332023,0.3790066899361655,0.3790462868930603,0.3792034034482385,0.37957827689565204,0.37973806184183706,0.38007992339407487,0.3805526873807935,0.38094550489796924,0.38140749790893586,0.38164386136677436,0.3817476148139519,0.3821180781862151,0.3824723183840113,0.38290184948696465,0.38339148042306653,0.38343564253413887,0.38372030020208425,0.3838511741234667,0.3839811460735885,0.38421779446467225,0.3850187755912483,0.38508630015205253,0.38517205115591413,0.385502202816167,0.38575033131638564,0.3861658755143736,0.3863682829627922,0.3864863415143881,0.3868373380305216,0.38734230970014405,0.3875514797992247,0.3875931062574919,0.3881003138397235,0.3883622279914881,0.3886108708745046,0.3886344083763298,0.38865519375196994,0.38884469297569835,0.389608927308786,0.39005977044697776,0.39069310351560055,0.3907727625990417,0.3908391984471558,0.3911378531154656,0.39174612199148173,0.392196289743065,0.3922891245330953,0.3926584272943413,0.3928419174070422,0.39294950497948955,0.3931195373026281,0.3933785849706144,0.3934344697411889,0.3935819841082793,0.39396865516161467,0.3940446059361559,0.3942766020395652,0.3944384948979266,0.39452979917877623,0.3946176989914134,0.39530959742846655,0.3955317098320997,0.39599881383880886,0.3966230902612797,0.3966849175331957,0.39672473209174886,0.3969396284678312,0.3970488816108468,0.3972953905000587,0.3975331360299778,0.39788126178806144,0.3980287459156395,0.3982273898614286,0.39846619477136874,0.39873521952464047,0.3989181639217252,0.39926716847466825,0.3999672421851926,0.4000103665881896,0.40004955476578313,0.4003125084055333,0.40036691953776754,0.4008651135276423,0.40115543312302415,0.4012165080644739,0.4014071660401691,0.40143442701235116,0.4016997914619786,0.4019900081769305,0.4020138068963618,0.40225474791410715,0.40225474791410715,0.4022547479141072,0.4022547479141072,0.40225474791410715,0.4022547479141072,0.4022547479141072,0.40225474791410715,0.4022547479141072,0.4023987965801867,0.4025928606570421,0.4029604789565138,0.40301030674284616]},"secondary":{"gamma":0.4,"ridge":0.0001,"centers":[[-0.97,-0.24363139644265164,0.5564106088131666,0.4560375362634663],[-0.5889463014900684,-0.97,-0.585610993206501,-0.41895436495542515],[-0.37486283108592033,-0.8777111899107695,-0.97,0.981685599312186],[0.6220479663461447,0.6024029422551394,-0.4744742810726167,-0.9699999999999998],[0.97,-0.5033933725208043,-0.7558481730520724,-0.5696521308273077],[-0.05455758608877659,0.9699999999999998,0.8550597969442606,0.5553407240658998],[-0.7686358410865068,-0.3679417334496976,0.9699999999999998,-0.11776589229702938],[0.01582111045718193,0.1434302069246769,0.2127769906073811,0.9699999999999998],[-0.97,-0.855015592649579,-0.07894633710384369,0.6830103229731324],[-0.9689220879226923,-0.97,0.06127389334142208,-0.6920120809227228],[-0.19228236563503742,-0.015827417373657116,-0.97,0.12952593713998772],[0.9450817964971066,0.49432334303855896,0.887483252212405,-0.9699999999999998],[0.97,-0.392128862440586,0.35636110790073894,-0.29570932313799847],[-0.3875896967947483,0.9699999999999998,-0.25244222208857536,0.32931403443217233],[-0.6642338000237942,-0.7287216279655696,0.9699999999999998,-0.7666437756270168],[0.41703333519399166,0.7501713242381809,-0.6728164441883564,0.9699999999999998],[-0.97,-0.6780271753668785,0.066239258274436,-0.02443109080195449],[-0.7032652255147696,-0.97,-0.04466967284679402,0.10057700797915481],[-0.4891832750290632,-0.4511145893484355,-0.97,-0.5378457736223937],[0.7419828623533249,0.038978850468992965,-0.9060482904314995,-0.9699999999999998],[0.97,-0.07452318258583568,-0.374918382614851,0.8873790036886933],[-0.18449996411800385,0.9699999999999998,0.2671022582799196,-0.9876281414180992],[-0.8985795006155968,-0.8046415597200394,0.9699999999999998,0.42520297318696976],[0.15137898363173008,0.7167095690965652,0.6601170990616083,0.9699999999999998],[-0.97,-0.2858866211026908,-0.5417509749531747,-0.8599585499614478],[-0.8546043038368225,-0.97,0.5670514088124035,0.7650190461426976],[-0.07796413451433182,-0.5927860252559184,-0.97,-0.3275051899254321],[0.8251494560390711,0.9268556423485279,0.49108842574059985,-0.9699999999999998],[0.97,-0.9675158448517323,0.7724548522382972,0.2238220423460009],[-0.25764535926282406,0.9699999999999998,-0.867766324430704,-0.15115460008382786],[-0.534288777038455,-0.16113382391631603,0.9699999999999998,0.7138248588889844],[0.28147396445274353,0.32725015468895435,-0.19812491163611412,0.9699999999999998],[-0.97,-0.911699840798974,-0.9451328963041306,-0.48093653097748745],[-0.5044119972735643,-0.97,0.9157455172389748,0.3940562941133974],[-0.28154044784605503,-0.21695182472467445,-0.97,-0.9567865785211326],[0.5345842428505421,0.30097376555204414,0.027159882709383965,-0.9699999999999998],[0.97,-0.3433520197868347,0.746026175096631,0.6189718600362539],[-0.10185946151614189,0.9699999999999998,-0.6460249833762646,-0.506020011380315],[-0.8247277624905109,-0.53532244451344,0.9699999999999998,0.06844612956047103],[0.07777121476829052,0.9513605963438752,-0.28900693729519844,0.9699999999999998],[-0.97,-0.054104145616292953,0.420669911429286,-0.6307609062641859],[-0.9372452162206173,-0.97,-0.43816298991441716,0.7442624811083081],[-0.15181544795632362,-0.8250547852367163,-0.97,-0.18177538737654708],[0.8987566959112883,0.6628484819084406,-0.6121642887592316,-0.9699999999999998],[0.97,-0.700577212497592,-0.14353448525071144,0.2659275159239771],[-0.4564989972859621,0.9699999999999998,0.24665617384016492,-0.3590949177742002],[-0.7419311795383692,-0.42855891957879066,0.9699999999999998,0.7964256089180706],[0.48887205496430397,0.0905868969857695,0.8270476181060074,0.9699999999999998],[0.3872786406427622,-0.4791370611637832,-0.4353308752179146,0.03859459608793214],[-0.6403385922312737,0.0007021147757770851,0.45672124437987804,-0.08641257882118225],[-0.41746659576892853,-0.6417193002998829,-0.5153259932994843,0.5236822944134472],[0.6644075233489275,0.8701332323253155,0.595551585778594,-0.6012953165918586],[0.9690991789102554,-0.7662426345050335,0.12692933715879917,-0.9239967744797467],[-0.22215902619063854,0.7448539920151234,-0.23394811525940906,0.9510113541036844],[-0.9450266417115927,-0.10460594855248939,0.8003511149436235,-0.38858523592352856],[0.1919679418206215,0.37600415386259534,-0.841701116412878,0.48639142140746117],[0.05436474084854126,-0.6174976974725723,0.9578384961932895,0.8262704666703942],[-0.8013202380388975,0.8928669840097427,-0.9323512166738509,-0.7987061459571121],[-0.015891989693045616,-0.2538353521376848,0.10024024359881878,0.3611932396888735],[0.7689354941248894,0.22848325781524204,-0.00859349220991168,-0.26381484419107404],[0.5892416331917048,-0.12726735509932052,-0.727467518299818,-0.21454134956002213],[-0.33619699627161026,0.35183987580239795,0.631366329267621,0.16043621674180053],[-0.62163046002388,-0.9940734729170799,-0.338484812527895,-0.7231055255979302],[0.3746743071824312,0.5194875672459602,0.30170477367937565,0.6519016046077006],[0.3462057840079069,-0.5618322920054196,-0.2678147163242103,-0.7827659565955403],[-0.5953594781458378,0.9582905638962984,0.3714289478957655,0.8422421272844074],[-0.31780001148581505,-0.31730956211686145,-0.6565283667296171,-0.2795719280838964],[0.5686471965163946,0.17086085304617882,0.7012982852756975,0.3454046845436103],[0.7639561928808689,-0.19270315393805504,0.044924356043339,0.17591864243149757],[-0.013109782710671425,0.29617715254426,-0.0629024077206849,-0.19908848777413335],[-0.790664928033948,-0.9364614989608526,0.9027952402830126,0.6366022843867536],[0.04151248559355736,0.582966485992074,-0.9864482972770929,-0.7383752819150686],[0.21262510493397713,-0.4195680916309358,0.8702811039984226,-0.07038083672523499],[-0.9634868446737528,0.07004415988922119,-0.7707639317959547,0.05459677428007148],[-0.2483710665255785,-0.7091044653207064,0.19664010033011414,-0.6170218829065564],[0.9975083284080029,0.8086150120943785,-0.1632914301007986,0.5079852920025587],[0.6771971266716719,-0.8355724606662989,-0.569413596764207,0.9665242973715069],[-0.4280587323009968,0.6852823402732611,0.5405178070068362,-0.9084523599594829],[-0.6431796662509441,-0.04307729750871658,-0.48963412828743447,0.4711836539208891],[0.39231727086007595,0.443384610116482,0.4014110192656515,-0.4038244746625421],[0.4782246872782707,-0.10976680554449547,-0.8517210464924574,0.7289526704698801],[-0.7290862407535315,0.400865023955703,0.7561047710478306,-0.6460392456501723],[-0.45152716152369976,-0.7615759298205375,-0.21325472556054592,0.2087094448506832],[0.7006646990776062,0.7204605117440224,0.1759897731244564,-0.16631394252181075],[0.8803518880158663,-0.6351536959409713,0.5821196436882017,-0.37486249208450284],[-0.13121357560157776,0.8442357033491135,-0.5571249667555093,0.25013037770986557],[-0.9087693467736244,-0.4861812610179186,0.4749825224280355,-0.8126165810972453],[0.1579065416008234,0.027112631127238274,-0.3828431684523821,0.8124058526009323],[0.08060494251549244,-0.9712715558707714,0.2512109689414501,0.3788380362093444],[-0.8297584801912308,0.5083615072071554,-0.3587213959544897,-0.4961843527853491],[-0.11464124172925949,-0.14957130514085293,0.6750929243862627,0.9337592441588642],[0.8654887322336435,0.3625023607164619,-0.7159512769430877,-0.9412335809320213],[0.5608037486672401,-0.2740099113434554,-0.05958487279713154,-0.5217570681124923],[-0.30995739065110683,0.23686558194458485,0.08145920187234879,0.603266274556518],[-0.5250771027058363,-0.5968407280743122,-0.8900954220443963,-0.04053498059511185],[0.2759242206811905,0.8839754872024062,0.9698370546102526,0.08445689082145691],[0.3096463344991207,-0.34740758687257767,0.23064178600907348,0.7803202737122781],[-0.5607519615441561,0.13441552966833115,-0.12721454910933971,-0.844686886295676],[-0.27635748125612736,-0.5234486255794764,0.8421464227139948,0.28201763704419114],[0.5252508260309696,0.986419728025794,-0.7970675583928823,-0.3429599590599538],[0.8299355711787939,-0.9001383651047945,-0.45320128835737694,-0.13539560884237256],[-0.08104157075285912,0.6129425596445801,0.43601280450820923,0.23961250483989693],[-0.8654335774481297,-0.22071036696434032,-0.5960884410887957,-0.6771253515034914],[0.11432681791484356,0.2579156756401062,0.5117677673697472,0.6978512909263366],[0.13102073036134243,-0.7356468345969915,-0.6301026549190283,0.11578668281435966],[-0.8799300603568554,0.7767011914402246,0.7297991923987865,-0.009189944714307452],[-0.1579774208366871,-0.38568636402487744,-0.30401368252933025,0.5716160032898183],[0.909068999812007,0.09464913979172729,0.33706103637814544,-0.5533920954912903],[0.7293815724551678,-0.01104128733277332,0.9306908026337621,-0.9718996677547693],[-0.4782912451773882,0.4700489304959772,-0.960383789613843,0.9030779134482139],[-0.7002471927553415,-0.8602993916720152,0.011177048087119834,-0.4658082574605942],[0.45133863762021065,0.6512784231454136,-0.09872491098940372,0.4091988876461976],[0.4278700742870569,-0.7934897430241108,0.6467088274657726,-0.7079608533531426],[-0.6767796948552132,0.6939181499183178,-0.7425043191760778,0.6670319866389038],[-0.39238375425338745,-0.08516183309257042,0.28544681146740913,-0.22970123589038827],[0.6434751320630312,0.4347467441111805,-0.3224103581160307,0.14532116800546624],[0.9637865722179413,-0.45951694808900356,-0.9160326290875672,0.31382302567362785],[-0.21269585005939007,0.05914863385260083,0.9418246522545815,-0.311168860644102],[-0.9970866348594427,-0.6691576056182383,-0.023874411359429137,0.8736560139805074],[0.2481781467795372,0.819508779793978,0.11533860862255096,-0.7513673435896638],[0.012795254588127136,-0.18601662106812,-0.24919956736266613,-0.4349946901202204],[-0.7639012020081282,0.3324043434113264,0.1418759860098362,0.4400286823511126],[-0.0419489499181509,-0.9431423991918564,-0.825534200295806,-0.8776026237756012],[0.7908421233296394,0.5467451661825182,0.784366812556982,0.9973892774432898],[0.5955333653837442,-0.5668646618723869,0.44049274921417236,0.5398191977292301],[-0.3466389402747154,0.9202986136078837,-0.41940813697874557,-0.5852032210677861],[-0.5685955137014389,-0.31227138079702865,0.6107424125075338,0.02247287705540635],[0.3174887914210558,0.20885837636888027,-0.5303332526236774,-0.1025199778378012],[0.3214761409908533,-0.8168002087622881,0.48146981000900246,0.13933585770428203],[-0.572353906929493,0.6704013328999281,-0.3783558551222086,-0.23568656109273423],[-0.35827475786209106,-0.06236071884632122,0.5932028517127037,0.6732614524662499],[0.6073979903012514,0.4587462693452833,-0.5479480344802141,-0.7017314024269582],[0.7870799005031586,-0.43592943996191036,-0.20431828312575817,-0.7763955220580098],[-0.037957800552248955,0.08251429349184036,0.1868327297270298,0.8486278504133224],[-0.7520403768867254,-0.6932085100561378,-0.8469779845327139,-0.2858965564519169],[0.0011632665991783142,0.7966407891362906,0.7628475688397887,0.33909534476697445],[0.23636119067668915,-0.20955254510045063,-0.8789770770817996,0.9757785908877843],[-0.9855290334671736,0.3091358058154583,0.9789859429001808,-0.899213295429945],[-0.2088859435170889,-0.9191009383648634,-0.05311746336519707,0.4618834946304564],[0.959717221558094,0.569527180865407,0.08598981797695138,-0.4131398629397155],[0.6550365630537271,-0.5435485597699881,0.6798561029136183,-0.11192278005182721],[-0.40420451015233994,0.9438975993543863,-0.7092510666698217,0.013070059940218703],[-0.680845245718956,-0.33512790128588676,0.26011203601956345,-0.5755562633275983],[0.4316761512309313,0.18475790694355965,-0.3478511106222868,0.5494661405682568],[0.4394586943089962,-0.26105125807225704,0.8975292220711708,-0.31769550777971745],[-0.6886266935616732,0.22001619078218937,-0.993622737005353,0.30728207342326685],[-0.47454805113375187,-0.6102800332009792,0.03652612864971183,-0.8697071820497513],[0.7253790237009525,0.9013360477983954,-0.07329846359789383,0.7552999630570407],[0.9206856694072485,-0.9856645204126835,-0.6671725120395423,0.70407335087657],[-0.1698535643517971,0.5266452394425869,0.692652206867933,-0.6709032766520978],[-0.8839364089071751,-0.13567425124347232,-0.27475632540881645,0.23363457061350346],[0.13476775400340557,0.34468402154743694,0.3663955219089987,-0.14137352816760518],[0.11838085018098354,-0.6500347033143042,-0.498066360130906,-0.543752543628216],[-0.8692588023841381,0.863007955253124,0.39103984832763694,0.5812555700540543],[-0.09261437132954597,-0.47081960178911675,-0.5746608693152666,-0.018585370853543393],[0.8417373280972242,0.007829209789633973,0.5333032235503199,0.10639127157628536],[0.521429430693388,-0.09729667194187641,0.18968093767762184,0.4310458470135927],[-0.2723072487860918,0.3845036756247282,-0.16828304342925537,-0.4439613129943607],[-0.5489464048296213,-0.7735501527786255,0.8596697710454464,0.8814751096069817],[0.29806970432400703,0.7363564670085905,-0.7794365640729666,-0.9935024864971638],[0.272099070250988,-0.02396155521273613,-0.018635587766766437,-0.17205497808754477],[-0.5210229959338903,0.4868909306824205,0.12254688888788223,0.2029683645814655],[-0.2981557082384825,-0.846883213147521,-0.9076072070747614,-0.6405423060059547],[0.5492308735847473,0.6339710298925638,0.9521868675947189,0.734449565410614],[0.8695390094071627,-0.7212312202900648,0.29606471583247207,0.7866451181471354],[-0.11846259236335754,0.7583638150244949,-0.3137294854968786,-0.8383772708475588],[-0.841335155069828,-0.39962132647633564,0.6536766774952412,0.27564692683517955],[0.09241045452654362,0.11247534677386284,-0.7375056874006987,-0.34934589825570594],[0.16952708549797535,-0.8852342311292887,0.6191474199295044,-0.9625993072986602],[-0.9206492453813553,0.5941171403974297,-0.5199282709509135,0.9123935624957085],[-0.13521616905927658,-0.2361104115843773,0.445767246186733,-0.47506281174719334],[0.884095037356019,0.2772064879536631,-0.41222736425697815,0.3999596219509838],[0.6887886300683022,-0.3598398119211197,-0.818601308390498,0.0743208471685648],[-0.43991041742265224,0.15076901018619537,0.7893931902945042,-0.05067106895148765],[-0.7253392916172743,-0.511497052386403,-0.23856196366250526,0.6131582222878931],[0.4742182642221451,0.9705774169415233,0.15051385387778282,-0.5118651650846002],[0.4038748275488615,-0.5856900811195374,-0.602549547329545,0.3709138873964548],[-0.6549966670572758,0.9352027475833893,0.5072456002235413,-0.2540627699345348],[-0.43212803825736046,-0.29298056848347176,-0.46431067772209655,0.8164888285100462],[0.6810070779174566,0.1934583317488432,0.4268707260489464,-0.808519300073385],[0.9856874980032444,-0.16966314055025566,0.8332371152937414,-0.7328862547874451],[-0.23680971004068851,0.3199721183627844,-0.8079444151371717,0.6420913562178607],[-0.9596806932240725,-0.9589847102761269,0.22587158903479554,-0.20482170023024115],[0.20855962857604027,0.5586967393755911,-0.13392344675958145,0.17018547467887402],[0.03775401785969734,-0.44267494790256023,8.491426706314087e-05,0.5178693272173411],[-0.7866776529699564,0.04622836597263835,-0.10790862329304218,-0.607137802988291],[-0.001245083287358284,-0.6864649988710878,0.9241971820592878,0.044468553736805916],[0.7523204497992992,0.8329249583184719,-0.9648795817047358,-0.08050901256501686],[0.572638301178813,-0.8118271194398403,-0.3087496962398292,-0.3827102798968557],[-0.3215622790157795,0.7083337642252445,0.330326955765486,0.49229780398309253],[-0.606991421431303,-0.06733562611043475,-0.6390308868139982,-0.9298106506466864],[0.35806665383279324,0.42081178165972255,0.7189627774059775,0.9451659619808197]],"weights":[[2.6231741781360265,-0.5380226274913077,-0.1554842058974197,-0.3900154392925403,-0.10203112370467601,-0.15397656474316276,-0.05296765490244023,-0.3618131676815667,0.07380966998101803,-0.3018348898795119,0.12407201770663964,-0.23990630980011457,0.19505339173995703,-0.08779754818910894,0.24760970293368567,-0.01715308195367894,0.250582488021392,0.18572231082541765,0.1649408515246526,0.07550966154285203,-0.03466025266736457,0.03148411698391021,-0.16145095339748608,-0.047004036846421134,0.007268433723522237,-0.005741109185745415,0.26913918038221857,0.047300775176989426,0.16456534871006442,0.08855581418817766,0.05534429301104763,-0.007780038085581594,-0.1119330767003008,-0.03482484345612862,-0.16902032885487495,-0.07750317455265186,0.056774591824124324,-0.044403817643741475,0.06600883916405924,-0.0013678777176075838,-0.04115940172694026,-0.02416434936336694,-0.14910932790512224,-0.03306130288981807,-0.12012861569799586,-0.01724618041714198,0.06436537521124887,0.021207418086362662,0.15063309307498463],[-2.5696467292379186,-1.3630848458075107,0.5079755188021094,-0.857212940286163,-0.31968486656364137,-0.0903882122175554,0.11214351412589221,0.1018234487617818,0.26108897939492537,0.20692415276288284,-0.0464408779288924,0.28430550497988516,-0.29109258782986924,0.1152305241629487,-0.6296947912884596,-0.06005930455056077,-0.7964344699489405,-0.23806349595478424,-0.36083327747409943,-0.18655154673226734,0.3083166646821609,0.17176952774765541,0.9630648864523703,0.3627840094476478,1.029105745450249,0.4158020328736306,0.40611780932270425,0.12338653514381864,-0.3719925576962666,-0.15856299380245742,-0.8119017249293189,-0.3208258563326552,-0.6681376179665421,-0.18637764853156655,-0.14622761501845585,0.04504542654298232,0.24141121163555673,0.12605020901397293,0.30569280660179476,0.0816913538213122,0.1319868216078188,-0.002120460164766465,-0.03626109790727302,0.012893037002182828,-0.07552923677150548,-0.03763790119538899,-0.019546536208543182,0.019860273158409467,0.06657748725435235],[1.6754759574061777,-0.07043745828948618,-0.3309508773094529,-0.07733555267117466,-0.06837420387875026,-0.06604353940507528,-0.06740014172341677,-0.061446092302996524,-0.09330059908585864,-0.03580093415866938,-0.10958914931172016,-0.019479533830232015,-0.1346840611924652,3.269424053542003e-05,-0.1484501082099367,0.015005375477819686,-0.14185670030839698,0.03494681225788545,-0.12633016333708857,0.03331339915403939,-0.10955423158180855,0.03507805047690569,-0.08926186960731519,0.01775606552417787,-0.05948606408867084,0.01710560502092087,-0.017570577969712353,-0.001819758583158435,0.025449706544161642,0.00716633468333306,0.026470530993270625,0.03113566484585798,0.011578872601249977,0.017635788881337808,0.0031852892481351756,0.005440626505615557,0.007929539840838412,-0.016117418073740793,0.0299248076499001,-0.017163199472517814,0.043383931608070775,-0.01418149139120723,0.02782799473285028,-0.0030593720304100008,0.00758071766674696,-0.0006444573577925084,-0.0032180128963091876,-0.0030710121568525628,-0.0035032920854923336],[-8.934080613172185,-0.6687573095475674,2.2457452243754306,-0.18529241046912526,0.9185161033805798,0.3390260084770219,1.1437621245432952,0.8325648279242647,1.2252467808545318,0.8860176473528587,0.9915713414493083,0.5987413163050099,0.38740638846107206,0.1252264376140183,-0.35728860017552866,-0.5001102063407676,-1.0812688901658862,-0.8948248759931757,-1.5701549716101637,-1.1291784194180192,-1.6464057811318924,-1.1010030348176192,-1.5334768523289959,-0.9194403620944058,-1.2351247021759848,-0.7220163591927239,-1.0755499307017644,-0.64233125174609,-0.9805818663364645,-0.5100851644633266,-0.7197705756729234,-0.20583682287101904,-0.26202227868504574,0.29123480894930287,0.36826228097005764,0.6979796442300376,0.737745780266501,0.8246129567976084,0.7371454595536575,0.6399653106470977,0.5735945632143028,0.4478376504925046,0.5957259228333758,0.4633801623814916,0.7132351590828793,0.4586346059397457,0.6231897469200162,0.2646591047739161,0.3460267528351864],[1.5970591085186971,0.16765264068387178,-0.5171560993953616,0.009003855490153678,-0.023110466973331977,-0.16273511896476292,-0.2563602205159539,-0.1905474637492379,-0.28881571438716475,-0.1911340995165756,-0.3212787421165055,-0.21278118667658927,-0.30584859986204505,-0.22550159586731866,-0.309458463935978,-0.2100638196352409,-0.28061363035924203,-0.18729006069525786,-0.2827412410173453,-0.16808784171262267,-0.21478241332426828,-0.16607629843965568,-0.20457156513712477,-0.13670511949832395,-0.19188364502494465,-0.11090315170614665,-0.17276213416577899,-0.07353622106942581,-0.11034590915466319,-0.02208406695364716,-0.04715070557052269,0.004506711403039548,0.010783045128598493,0.0389630377162163,0.05676010755674293,0.08375657108157589,0.087915808773885,0.1228922590673892,0.16375377252903536,0.14935998692125688,0.20320033586686276,0.14651809465667767,0.22254577471140277,0.11814289356412688,0.1660144691524043,0.07297224907551991,0.10079591540593265,0.03760534737183269,0.06251518084009089],[3.303956879634005,0.055076554748987476,-0.8766500256346813,0.06222176732821288,-0.33664074104533676,0.09828765973855842,-0.1222644718987532,-0.01627616413360794,0.0559510942877936,-0.06939128898693986,0.059493190655788245,-0.030757867893924002,0.07003053565497297,-0.008653740544812464,0.10385818417776506,-0.0057916648131527545,0.07036958148950546,0.010025131572152473,0.08957895600034567,0.05028808565143522,0.02346509129177438,0.07342091463852096,-0.026722025853564865,0.027151717150854945,-0.036111062614107285,-0.012521426844479927,-0.055109953116025075,-0.05005155171235937,-0.020988780491749213,0.0030933460973409863,-0.03024220842569646,0.008841221828299217,-0.09775603762401963,0.015850406561484093,-0.16334058982519392,-0.0075088077138186875,-0.10628500891379541,-0.0019933991253012974,-0.01491283676857345,-0.0020840756709646323,0.015876090056681315,-2.8231169792057687e-05,-0.05267255437535123,-0.019527489259447688,-0.14267394071959325,-0.022656460673029625,-0.08040964781519462,-0.014757862472100055,0.03975139968601184],[-0.9794714890462811,0.3097212771326995,1.0143800269585526,0.0792812507839124,1.3748791032686185,0.1054996425517975,1.1556257495230065,0.04914473957244103,0.8666613592903873,0.11240220025618274,0.7751665186258837,-0.06318215925630792,0.46593085787338945,-0.2762569612801721,0.5309383873125509,-0.1431095980552356,0.5426300652223124,-0.10108517027975776,0.10277601146774244,-0.14717821474917517,-0.21984602324818828,-0.20383626611918698,-0.34013708301277856,-0.21893771784605218,-0.3601033807464414,-0.3443386738291913,-0.1496293791607436,-0.15436564176716872,-0.06758750164552171,-0.05720190700326686,-0.2515080259899455,-0.19459909930176783,-0.370778031443995,-0.10731847795997329,-0.34488522506624997,-0.06351676622329906,-0.04342234350119519,0.1020610586308867,0.07467034809707022,0.013398282792517159,-0.02690886820527679,0.016762325993873034,-0.17541981795574565,-0.16314502928966343,-0.13762556972763465,-0.11403213286586693,0.0547846808784465,-0.013460358680144823,0.055833845104159445],[0.07906637359702531,0.4590860810692464,-0.4451355936900542,0.8210252125746703,0.059155346619344316,0.3311186242680273,-0.03452247022212978,0.09558070651449826,0.1003509827525796,-0.04024564160034135,0.07276436534828586,-0.08808241348765918,-0.07510227263379618,-0.08898100576468837,-0.050008145884614635,0.008442535166908498,-0.09781736256552101,-0.0196040407925703,-0.34832877186653505,-0.08365594898259203,-0.2304403276360846,-0.005684247520249705,-0.1853915864133119,-0.004750854924123697,-0.25057927512660455,0.11168389959064126,-0.22934076842930742,-0.015034064535122155,-0.12676781669585935,0.05678127743891276,-0.13962344661971562,0.05789538966205997,-0.27300061313145174,-0.004067905911668653,-0.16421202657311015,0.023642600890028814,-0.15839332956310231,0.006707776537621315,-0.16519133335413638,0.017901868147944915,0.025088389993666388,0.02208375857109236,-0.05989171226526174,-0.015395113480066428,-0.09455957630739582,0.10082554732742947,0.013721249971882074,0.08724773647667748,0.11044352948269387],[-5.055755589047825,-0.7206083309111824,-0.4225166871880195,-0.522535948048074,-0.64946347407524,-0.4134186966524173,-0.3938139957519514,-0.5982379832552737,-0.49083876915227725,-0.35186372698070006,-0.520827997858972,-0.29670538180253125,-0.4833219653323522,-0.1164139890537122,-0.5229138039106404,-0.0830496727659573,-0.24030199633035174,-0.09568470710306973,0.15151755917926046,-0.07629294499701036,0.3477126063272745,-0.02740902821129437,0.20340785528691227,0.10015589251447597,-0.08177225472904835,0.09982702447324698,-0.1599065770988243,0.15063202461641553,-0.16648304795402508,0.09118521008973826,-0.061048003972929595,0.07786013036221727,-0.011058910413223677,0.03713490596601691,0.008527992020079594,0.004163733942370774,-0.04935489461249509,-0.03262264090049513,0.03944620138134334,-0.06411170970763068,0.10982555258613014,0.006977971692888137,0.18327182964588057,0.05644404170161283,0.02606606692806763,0.026976024933210403,-0.07401547265527937,0.05204917292078631,-0.12158612169950361],[3.85837451483479,0.5623518605430512,0.3287956450503896,0.45814054791867825,0.9379879956977202,0.36867333608552255,0.7573907704967798,0.327764717301595,0.5726020228856348,0.21641000516805675,0.31780006294042995,0.06135094506972588,-0.02219369191716244,-0.14270803249040293,-0.3132541335251292,-0.27987188030835025,-0.4119195838720056,-0.2914913433106357,-0.34638393585634714,-0.2255187055946542,-0.11021644989924999,-0.09030709798253726,0.0798166939418235,0.029510349464001294,0.12582506980985603,0.03019832556073467,-0.022085295094968413,-0.044763695150748274,-0.1385444770380582,-0.10738324863414633,-0.14703250797411713,-0.09317340234815542,-0.08292392434341839,-0.03408205972258581,-0.004278365708869782,-0.018766079211287715,0.0036572228790020685,0.01770009290901142,-0.015867011602527608,0.0042005875189405515,-0.019501982957391667,-0.0019882644426286074,-0.01094024195459489,0.007379415481132199,0.006727497542079074,-0.011997326334338683,-0.013826758646427574,-0.0050048117131498305,-0.0323330871049595],[4.440054750484865,0.903486134829869,-1.2145865424698405,0.9821224998149091,-0.0052076770989253965,0.45607522366027714,-0.24381450866207968,0.2432315260991905,-0.4392449513403212,0.12434046014230347,-0.4407626889660115,0.054479673910636864,-0.5533930206303967,0.01180984697565888,-0.5972357903404542,-0.10825461713210376,-0.6774555754301066,-0.10545234072606736,-0.8291142111279673,-0.24077019047857554,-0.8388341107813654,-0.40248341604836363,-0.5175209761162409,-0.26424567625439166,-0.2198969314929594,-0.11416523710716314,0.05635113903738686,0.046019649760387256,0.023860526065800906,0.005314051140118332,-0.011160327619205983,-0.03214575679612354,0.16917291858114528,0.028892293440306856,0.43790186650499896,0.07444043606519804,0.5398526964280383,0.17523046349685978,0.41943497985662026,0.12705382430393572,-0.005291505169193894,0.07549675745825263,-0.22411181651672027,-0.024657017839267797,-0.2769174461157761,-0.09334870525410278,-0.12202822366152342,-0.09051974317867868,0.053795816715991915],[1.8819564048728163,0.021323725331339728,-0.0865727040188132,0.10077023122969787,0.1116265327122413,0.14388555787533389,0.13314423227504998,0.14416911073687355,0.17442888727335626,0.13249520155846226,0.2049375973183321,0.15005290753179193,0.2458539045234355,0.13944945701670822,0.30063113864054425,0.16370346648520417,0.34210042249297656,0.17859693509417418,0.3583160483545192,0.18165206355506278,0.30433332790848294,0.1523178570884268,0.16558629142957035,0.08299752257815705,0.035611129758755085,0.0017096920585380673,0.0337646345533489,0.02778709029971035,0.1474736139767838,0.10504836760140775,0.21753871737049607,0.12656192283017842,0.12330821268068895,0.05707466899747101,-0.05524704568700179,-0.034941038305071434,-0.12891574256259106,-0.05266748415018611,-0.06503651363961438,0.005331227150023176,0.04160162311143943,0.06024165941103364,0.06507052815522717,0.04529341806935644,0.00018016758989855121,-0.005508894665572956,-0.045534289136465664,-0.02621233016379464,-0.03571050672115267],[1.3822378619081779,-0.6341483646098031,1.0501217200314852,0.1880329505245737,0.2918915287909177,0.49750660883352227,0.5955646360608094,0.23130456783704195,0.7793133142111832,0.0005983567307286517,0.6616451028415207,0.0701912136566075,0.7058795725643893,0.22837621968249808,0.5216069802697528,0.26284658369331293,0.42915487796647767,0.32932017002428343,0.41125258755011207,0.4036048956381889,0.4836939983889459,0.45206311014542483,0.6846397557818271,0.5205299460740013,0.8253312007844941,0.679181342606364,1.202058191642414,0.754581522336257,1.3872720874498314,0.8275341928600488,1.4112134203412725,0.8162066883291891,1.269352881772003,0.7253757900931725,1.2398674547869404,0.6777130541074459,1.425671412563907,0.7360553192980245,1.6500820543390509,0.6631206297410323,1.6738107618879297,0.5362503512992425,1.4714171938568867,0.35487449592103343,0.9682662126161339,0.20848710392492414,0.8731796555882367,0.13161632737930481,0.9338487166335829],[4.156030300273039,0.00047888524421953426,-0.19841242971566267,0.08133548742424414,0.2386145538244809,-0.11963040416536429,0.16282326241179804,-0.0204461309304373,0.30393409168726104,-0.014138543870165056,0.3290633923076273,-0.0301953477362217,0.28331189931844947,-0.059604189110739135,0.31076159119485075,-0.003063212587563156,0.2863587337996788,-0.03963509982391101,0.14136538930870254,0.03566066201512428,0.11462457493455741,0.025740076087194544,0.09931105419405467,0.05317633067687042,0.0890268549902691,0.005930575530175497,0.21016186232416947,-0.004725714419402759,0.2787725721769946,-0.00305041509956703,0.14304326532467773,0.04106338773968321,0.03897046010868669,0.03325029086957302,0.08059306833015228,0.04020203423600088,0.13135380472618116,0.021351377730022467,0.12629479755228948,-0.012546035457128188,-0.004081094831202124,-0.07531832301644965,-0.13913720746842606,-0.10473897545134925,-0.14734521547717244,-0.05955651960813768,-0.022799412208839763,0.004780675853923482,0.021971343338420733],[1.3492280739648927,-0.2286317150203031,-0.19221458527458143,-0.15679722791281006,0.017937102024915295,-0.11205216179782493,0.04701056183591678,-0.09525491289582844,0.06993946160906332,-0.10267749280599214,0.02037455299378666,-0.08244551121514124,0.0190512569947961,-0.06707277946642998,-0.012593989602580251,-0.05920749652329184,-0.031967269832413395,-0.060790311922833705,-0.04899761254031107,-0.07492950844540892,-0.065491679870893,-0.0625195541053205,-0.03715890674185373,-0.040683351332970255,0.03385936405191061,0.018505065990779498,0.06534134899279118,0.041411264905451654,0.06131662260073772,0.03808620816799872,0.028257629313217843,0.002994125230020943,0.004304719825951998,-0.001208050509523816,0.02947941508172887,0.024608482976587315,0.0634352953085163,0.040207815309490844,0.0729572957660826,0.04210987251007558,0.06879502849900714,0.022651767130862376,0.058772723781469756,0.022159443802838925,0.054404202580996264,0.010470574019672383,0.03721157462760214,0.020070013615429115,0.015939433976286707],[1.5820161819478973,-0.006645926026673561,-0.10783937109256282,-0.027723759513876612,0.15795444316641172,-0.13009693960778115,0.03847022437409665,-0.097340729619705,-0.00574765322827194,-0.09048737881339027,-0.056817518174075744,-0.04753553049001561,-0.13176056730803462,-0.04882658398455313,-0.17654520124123943,-0.03739706456723568,-0.2203438887560244,-0.015173790570516802,-0.218832962728868,0.009147158230354432,-0.19148863757521098,0.03962700881068394,-0.12187293527943653,0.06712233807333431,-0.061381409604268335,0.06336075803584111,0.03613191995762454,0.0647683955257851,0.08164770780466948,0.03491070725370762,0.06392278321064158,0.02057857599982844,0.03014692077984392,0.009160653940459682,0.01755814078506068,-0.007195098130676317,0.03598642964186156,0.006134182791662659,0.07278904937899197,-0.00015652922267826223,0.055319057353770046,0.006780953851258032,-0.0006652270578773096,-0.01370591053094397,-0.051793984644123554,-0.018544398001119917,-0.024664115374970222,-0.000970478489897614,0.003514079712850743],[0.7612362004261487,1.057226035361405,1.184385393571138,0.9240902982374192,1.5223401227855877,0.9768339978500895,1.3845889235320352,0.9301497173745246,1.191794831932569,0.5912719682353497,0.8463101728401351,0.3641924405035683,0.6111508978425444,0.11329210127639047,0.49711906424121105,-0.007211381488035426,0.5477667506666566,-0.019657507871136213,0.35831483155019017,0.08607591679456762,0.3585859019153959,0.020432183004761996,0.21890845301435435,-0.11465648313667215,0.13981235615644255,-0.15852505104893566,0.019364522559019236,-0.1979623707906635,-0.08764185213159711,-0.12158687144694018,-0.08543054657527972,-0.07890279271917874,-0.09676348081732705,-0.038980439189815905,-0.06588198018108012,-0.07828148488777349,-0.017362314091779917,-0.037047960986733276,-0.055904504011939596,-0.021989160521033772,-0.04809067635846475,0.011590439194857477,-0.08710638006088078,-0.022768243928664497,-0.016662175513380867,1.5573465253652686e-05,-0.053958926626929825,-0.01803945080934105,0.0005370447087427747],[0.735990707055007,0.6114446943596938,0.706457776164741,0.3289637969187044,0.9344238382565214,0.011620218948410525,0.41039191211910886,-0.010971373205512724,0.19777883203881688,-0.22401013061899772,0.04277494137400796,-0.2792526578837036,-0.3155722252207239,-0.46427736364646516,-0.5689811621893256,-0.4662713891982505,-0.5870394842534515,-0.4826026129176545,-0.5093681842944195,-0.3105072201035183,-0.3402036520726787,-0.21425457858258268,-0.3425363961409452,-0.044435467134591115,-0.350684890246881,-0.019303751135292646,-0.443252648380266,-0.09268991810714673,-0.299074619115419,-0.05563308807007934,0.003068048206748082,0.08001969386042002,0.2548186858104036,0.08948085204461,0.3401850248938904,0.10112152520767921,0.10783230994384069,-0.00992768564778755,-0.10393234622628075,0.015145128959173677,-0.1697310248378934,-0.06138738585668213,-0.06582144501875178,-0.004526457876822125,0.1257286761775766,0.08896716249866125,0.23310544074346642,-0.0186859841321921,0.16034544306535714],[3.7368616377228054,0.08258734315748306,2.553157921332238,-0.3374327166440825,1.0838640652166756,0.3393451536007687,1.164251660846798,0.46832331461574556,1.3129277445683483,0.4000969727136547,1.1020942146092156,0.45310318063543076,0.8149835923062508,0.2892324713607632,0.5395232135205323,0.170524621005135,0.260691462607837,-0.14312517776924144,0.060749601823789516,-0.35054481169234897,-0.20967648445159176,-0.42552750508901266,-0.4768723114573617,-0.36976635229081817,-0.38082467445167756,-0.029082846274638325,-0.11390300996447024,0.22118929810093504,0.2706302451866304,0.30085183364366014,0.31738799220300484,0.046509093852820864,-0.11512120645407305,-0.16946151788609845,-0.5066091253851984,-0.2986915352049708,-0.5602522472847791,-0.19666359507551828,-0.29197044258583116,-0.06565256001162983,0.07817391063272965,0.06265099239559344,0.2412426116688582,0.1274732212987963,0.16040846769925218,0.016821919693602284,-0.006610357655770783,-0.010717506787595464,-0.11742327064758665],[3.055482652985699,1.3760528417917302,1.2997358007940163,1.2552892794396666,1.589134820906085,1.1064281593704983,1.4589189671737544,0.9731879179661249,1.2601336552807254,0.779958656037258,1.0380458387436124,0.5994250215719642,0.744004741960507,0.39927637365813795,0.5100366197872577,0.19797974939179355,0.32009052656212866,0.09374449738144744,0.21529196809088855,0.048422011390194204,0.16587451265453143,0.039574109205596585,0.13617458575627278,0.015041011861302096,0.08783519974727354,-0.0005852981356381758,0.06884562679891175,0.024172590451701996,0.09488675507401315,0.07300152369275723,0.1676069924929226,0.1197242816614324,0.1687221478406769,0.06016405814453443,0.041869398885631536,-0.08011945698841749,-0.1482917728259242,-0.16309673436923142,-0.24245262225470512,-0.1892115707264312,-0.18130446645747123,-0.07049549288968283,-0.06235842514121558,0.0010573220169501586,-0.002883783469454132,0.0007606759405242024,-0.023046360370927235,-0.03364326558069444,-0.08443288391182253],[1.3147013219614139,-0.2768927854987002,0.8591826984372005,-0.18675416405662276,0.8793642263911633,-0.10096588536018954,0.8582150678725825,-0.09963095217376913,0.8068680476598734,-0.09539579269969303,0.6859505570544723,-0.09458534262261717,0.5912840034438803,-0.10122011134693017,0.46929834359093203,-0.10982418229613805,0.3449098405373354,-0.09540950006574268,0.25897999730826254,-0.08350257746293462,0.19192680763222503,-0.07589106648912916,0.18240139431569102,-0.06247292220928702,0.09748190618466114,-0.04626741528235852,0.020678475408244034,-0.05442804912181061,-0.06604882824851604,-0.014761003469934935,-0.10804369396134683,0.01508859360809706,-0.11060153349029858,0.015293020564615518,-0.10310281093389714,0.028282420991978446,-0.0562005243727073,0.032426978167580264,-0.04010604735943043,0.03149261089357774,0.035872955092180595,0.051845376350998965,0.06638512743432755,0.04435993283202242,0.1004868181039206,0.025065832728481296,0.09594654458056033,0.007781268002389183,0.07863506778722351],[2.6064057567550196,-0.9919241255880905,0.20928725352392055,-0.5647045338874018,0.16801377231936246,-0.18554631953654582,0.41744863542227006,0.0012672471285649085,0.5695668884404755,0.07089293581094126,0.5353678637858458,0.02011743932400084,0.2848808567832876,-0.0715132984124201,0.06105514004742456,-0.1879981303956107,-0.19443441481131366,-0.23519390082861827,-0.3610317723393752,-0.2719230165232557,-0.45509170201904153,-0.23822926568961234,-0.4319349922345318,-0.22956865178016475,-0.3862954773932826,-0.16128579693902326,-0.2740887067674903,-0.10626486171857144,-0.18187999722932502,-0.039764109215498875,-0.14574039477172776,-0.009975772962335637,-0.11158694131723798,0.02805416335472499,-0.026771677752079295,0.1116018615208485,0.09902788915764617,0.12810987767098625,0.15381212696038346,0.11733958421028876,0.1606866047786084,0.0710344275357187,0.11005856871680567,0.06365355371393223,0.08073652777592738,0.039514992441791894,0.06941409389832665,-0.005639759877905952,0.05946706566132375],[2.952366893791927,0.21591799762990785,-1.9591557050055206,-0.11566697740900561,-0.46898678092150925,-0.206756163228752,-1.142311359984286,-0.3002670996725339,-0.8893914719568528,-0.06893241442767448,-0.8201551601594769,0.16742098563998883,-0.9022821283636425,-0.0763286297857998,-0.525419694111934,-0.14033838132618487,-0.31196544341261734,-0.0868231384664234,-0.4659231217019986,0.05695175175611695,-0.33371225715613606,0.10566697174749623,-0.09879124489187678,0.11432523430653425,0.006313538303491286,0.08811524893611285,0.025074889214900465,0.050617103646658904,0.11806145520165706,0.13554635499493836,0.16510326634630781,0.04611091000968611,0.10571013633751895,0.09341962704348139,0.04729056666809438,0.013205123194932203,0.15482701991945214,0.02350157272431595,0.07087163825673137,0.05921407756781325,0.06432380201188291,0.07729627524983171,0.13598225254284907,-0.03912284515743376,0.11871716529078088,-0.08573845751856236,0.06715718570708373,0.052554815488564235,-0.028715474948359954],[5.648778068633908,0.4674433398550906,-0.9206361003472953,0.4066237192765206,0.057110838154581484,0.014368694392178496,-0.05625857065069061,0.14816384768530122,-0.06313995917191205,0.032474904140531426,0.008308554529781227,-0.10009057734498603,-0.10963008836715672,-0.16721378299299022,-0.0367354550890627,-0.17483176170696868,0.018114387878133426,-0.09803186586203101,-0.017881197062351074,-0.08188244340295188,0.06743322296085044,-0.08028437972767268,0.07506222589225028,-0.05706398171654223,0.008515822402616512,-0.09306996890331533,0.1177709097295654,-0.04901021744303108,0.12658993971818136,-0.03651155241942515,0.09931462839274116,-0.024713205680717814,0.06860543580313225,0.010131408653036716,0.13236008811338565,-0.016421603333678328,0.13665408376569893,0.02201161569668902,0.16146321520104528,0.011230728999233653,0.06567978146416231,0.0038099659969850932,0.005605648323036613,0.01812552688489361,0.03039015704439506,-0.0016679664799897985,0.0658308711856621,0.025614824550903324,0.02893046426093532],[4.584777411056068,-0.1370338503047123,0.2719474160153396,-0.18612356222517024,0.2218950728970797,-0.07613686773868512,0.3924076047160565,0.051121819172289015,0.5287266318539914,0.14229470708042927,0.46700659297595526,0.11404610466588497,0.2442659643128216,-0.011755572736804096,-0.019649047735818274,-0.14140489908779458,-0.2777258765329733,-0.31299046586410517,-0.4675144407413907,-0.30230739395231354,-0.4476380873818821,-0.253422961024667,-0.2560369454193186,-0.10012031597188734,-0.10705777294365533,0.02490353463238481,0.050997071340414334,0.061698279634676646,0.05140062100735812,0.08364598709181341,0.04199003415397298,0.06841340357857646,-0.03445980717354033,0.015337928339814136,-0.032229264543137154,0.02921152042252386,-0.08062396055916096,-0.0150027758952814,-0.10528437048127665,-0.031302728557799094,-0.09184499967415793,-0.028888724784776342,-0.035853615175568176,-0.0007068234868295538,0.024384029748205888,0.049385506938333205,0.08594885355052433,0.057877409878272425,0.11485260632683494],[0.7789567042425014,0.038645388500330086,-0.10540263359557225,0.20167325892419952,0.06211029776589941,0.08534230841893599,-0.2441554082741645,0.141348001418698,-0.11939485010025483,0.13261514674749056,-0.07186693600939344,0.18301345526904725,-0.09487816016115269,0.071987747279559,0.06330538906122654,0.03252396959509255,-0.01592907939430549,0.0024537315163303884,-0.1888303063795272,0.036981900351924626,-0.20201163330373675,0.0006474013016751889,-0.0381068425573787,-0.031209511818858744,0.07706136353527739,-0.009194660660272518,0.015189429042463988,-0.0321208539762472,0.0658845105695786,-0.04409756272280377,-0.002728557670509196,-0.053317970519538106,-0.06751776637852847,-0.05224196186490589,-0.036734187431890435,-0.07958441065286402,-0.0018493716242953618,-0.047286505831408226,-0.06430965490064634,0.009423840848292401,-0.13910395061938996,-0.006845234412423254,-0.09723427477905465,-0.02913771593454625,0.014315845729367088,-0.04576082034815599,0.0169346675264471,0.0007043359901099918,-0.008718582612914444],[-6.737157499965014,-1.608126816624112,0.05172261988393469,-0.6285337137364235,-1.8892661761972287,-0.1764391820891088,-1.4173959264607812,-0.01695151361356509,-1.1627896682037377,0.020666891225062385,-1.015167804294885,0.15631769337009566,-0.5689999138369758,0.28182732737410293,-0.20262999713837113,0.39267405186345916,0.193588166197095,0.49969926758848304,0.7012860536347577,0.54507096710501,1.0411149218639981,0.7029033931361016,1.016932744467319,0.4832583973147325,0.640006540138892,0.1162924551271411,-0.09182535695609609,-0.2673362778421055,-0.6197931595649603,-0.4826011290277857,-0.5540135154026089,-0.2743193946498353,0.01657677679748761,0.10241882269721896,0.5516740331092994,0.3635068169909455,0.6821794537492605,0.2517018527377379,0.1971872797065906,0.0011643556510012492,-0.40854457623057133,-0.2917658847206822,-0.6351711072009557,-0.27133817563526313,-0.4145104102287054,-0.05974987790889257,0.10298385945359441,0.1170875953481945,0.37474724973793927],[2.5648731536220892,-1.1907306279282275,-0.9839184497208442,-0.8661009453521676,-0.5717064579908862,-0.44792511396671386,-0.220516124559991,-0.2016738639286931,-0.07429914178866404,-0.08065476234388362,-0.08296681505762496,-0.07742500293252041,-0.08082786842174576,-0.07567021340025598,-0.1365631266163251,-0.060887755414484476,-0.14850523738466034,-0.08226403620895834,-0.15013923884479258,-0.11683334096042168,-0.16523107386987998,-0.1471719847855383,-0.13935509089704548,-0.11152590841664978,-0.03813767327013035,-0.04178209301183767,-0.009768636150161896,-0.033581501010429574,-0.1039247407107168,-0.09395340116424702,-0.19813237797246563,-0.17610824493417856,-0.19384763476534075,-0.1330548124144016,-0.07705812752741993,-0.03485267000027045,0.06499930771691322,0.025037502427489245,0.06344989352012222,0.015433617604805785,-0.04998705175623222,-0.053866535848666096,-0.1443127129829605,-0.06540310361847891,-0.09879147273200159,-0.018338145249990584,0.014991682703690808,0.03575424531082821,0.07365235907429855],[2.997068258432188,0.06732995407881229,0.5006734737324505,0.06348951798653085,0.7197559313502603,0.039696992495802,0.7044749190536657,0.01703761910974493,0.6932222223926737,-0.01704571899892293,0.5981505939507736,-0.029953333193857904,0.547132003606716,-0.025800443894470084,0.5110350570753163,-0.010598079338490525,0.5135602522591969,0.011514161720104324,0.49612607576915385,0.03441075258938515,0.4884261992856761,0.05270691419558966,0.4564544591277506,0.045221554945511014,0.38474307269721947,0.05259273446843101,0.35624849443359774,0.03968111460615548,0.3392048781532642,0.050042446843991624,0.3452581094583533,0.06969653366850428,0.34278032941387954,0.06841164900263258,0.32423433938752855,0.06016551280795953,0.2730781175637677,0.053611919710475305,0.2184655271665459,0.021463844824002595,0.1941537940242881,0.01305471339598696,0.18222112350401104,0.0066532520230043884,0.15445501741402623,-0.0016726227472190099,0.11841808178075372,-0.0016281569439955115,0.08198666111756014],[1.1434183979235468,0.5074769359276092,-0.1549377468340315,0.17641090113949656,-0.1799980359156227,-0.010034359532709484,-0.3741440322913303,-0.03974891848002702,-0.28217113568631313,-0.013483626701935578,-0.19069341102864995,-0.011491856979436997,-0.18018792206368947,0.044082368059958826,-0.08225914128880064,0.05682742708094778,-0.024879535133574594,0.009939342401178781,-0.11604178950220181,-0.008538061895366796,-0.04040166571868007,-0.033264864413712764,0.05940028821405297,0.06618576829461512,0.14807430186666037,0.15465219490434698,0.28801222818196953,0.1857481093172914,0.31418113871606523,0.14070243265917395,0.21707295299836873,0.0648181457541563,0.1971769503130345,0.055643529807302465,0.3408986920392705,0.10929403000942477,0.38540570783310807,0.15516934708372684,0.35912085863647114,0.11030499489753587,0.1251851942278844,-0.04164817327902128,-0.0520996983904986,-0.16454605747133205,-0.16003791033768663,-0.14805414415244592,-0.05852226674432401,-0.06324966973380881,-0.020796521602975286],[2.4292365115396266,-0.48299053085574584,-0.7527756786668974,-0.641033668670291,-0.1105125927867741,-0.4733029636649875,-0.1900646353767984,-0.4839081500109827,-0.17619450550852414,-0.340169446525401,-0.10635422909944618,-0.23304300549143184,-0.12563795417934762,-0.20299190425483318,-0.15096246746227596,-0.17837727791450617,-0.23967924642081098,-0.14838422692122394,-0.30680585015947887,-0.11885027798176816,-0.28908146239434673,-0.08594481523769815,-0.2878364617945206,-0.01953118043858984,-0.2271976870960035,0.006878212231151095,-0.17823663837404088,0.03379753746011775,-0.18593319435890335,0.13914357879118552,-0.00854956344298788,0.06041057740419467,0.12800621151824715,0.05268614157668423,0.039312440180644254,0.029448613586503157,0.004909424521960485,0.027003335754305583,-0.05830141855134358,0.05433242458596516,0.03422798970433426,0.08540061521953973,0.1455914021840791,0.049122624008187746,0.15037042057854644,0.02086608860987231,0.08883910918393083,0.01069495405195766,0.02900799155867295],[-1.4440427721747542,0.7440976645644304,-0.044254372182293354,0.3927396733755127,0.01140219175851941,0.2828166556381622,0.22840813454570846,0.4306042277370885,0.2420521192319188,0.4025636377930549,0.47002028619854225,0.3363731089944287,0.5422581667745593,0.1928914025413118,0.5338020759921172,0.20062685899344587,0.5167815196810018,0.002847463317623664,0.42522461139994644,-0.008490330628192946,0.4741284898590884,-0.12394399350675481,0.3188521112890178,-0.0738222547003072,0.37033982178308483,-0.08368858303445778,0.3442572408568518,-0.030450530690046367,0.29223442141712475,-0.1335206430622643,0.2526011449301875,-0.19617619541430206,0.1449525007641949,-0.18930800384522392,0.0636673851936086,-0.10268738655252635,-0.056986730834246074,-0.1276213859883815,-0.012121559795174968,-0.05899269907316879,-0.1150494448960894,-0.15298282728033502,-0.06152824502551722,-0.09722442721362676,-0.11293642501861136,-0.0722387373898005,-0.18241322239737282,-0.06640958831138262,-0.1968515318133537],[3.2200283326121064,-0.8099134824828367,-1.080432347103676,-0.6040486963393071,-0.9317119867306469,-0.4460839600056107,-0.8756691624890569,-0.42212339492146284,-0.6510551144992038,-0.3393534145343724,-0.4363285462318378,-0.15604946016543944,-0.013589468996342766,0.06997966106931405,0.4109836087888405,0.26069725168621993,0.7130685172658989,0.33484671112487924,0.7398363105493765,0.26382480129574304,0.5198830210733215,0.08993586600925445,0.09873153945223684,-0.05090272060327325,-0.20233940010440563,-0.1424884008430728,-0.2431726706881784,-0.07513934263001763,-0.10435477386277438,0.01898585039165862,0.054735580406945095,0.04614150919650903,0.0760625461009413,0.02810875305350335,0.02370920115847821,-0.017882173904090717,-0.0340190971899476,-0.0324293737547633,-0.0031706151709880426,-0.021130165103772698,0.07129760398701024,0.0231878640531984,0.11622476292068827,0.04062543361773225,0.0840558566412649,-0.0032980544398033287,0.009305601279706612,0.00012079453413910747,-0.022918675551189843],[-1.3873767038582288,0.24449343524204997,0.9670716621762426,0.3436384652888434,0.8283870594937186,0.25019964296523706,0.7446700993090455,0.16314917851987898,0.8289320101454651,0.1309327780136467,0.6218161951940111,0.21308604067689327,0.43255559868952,0.11640664637999094,0.43267317797716576,0.08440920411654418,0.4531296360508569,0.06326550205964239,0.3132082035374816,0.1080585502583986,0.2887900831941948,0.07080302900556842,0.34787889546594153,0.03345516970395575,0.2936685382227079,0.030463228685965935,0.18290175563660385,-0.012095262443842536,0.08921955162971132,-0.050980606118120914,-0.0606532294300859,-0.07767845442553657,-0.1728210308236513,-0.0554050990948291,-0.13807004249095964,-0.13525770713052404,-0.047392365598944386,-0.08667967489272613,-0.062117165140098914,-0.10883081916920288,-0.17167897533455065,-0.048735526393785776,-0.14210674787017363,-0.06250558138589145,-0.06865211227785675,-0.06500959190589281,-0.08483525399209668,-0.015180989253949865,-0.023530886465453326],[1.2867912460513893,-0.6747652382233993,-2.31412008979505,-0.673477474462508,-1.1641442686284709,-0.9302736476798322,-1.0361684342360997,-0.7569107809303094,-0.8431808081303624,-0.5023368413930844,-0.5274022402309507,-0.31326567197331434,-0.27331654837331176,-0.14040217383483583,-0.10368638037651823,-0.03764547967221035,-0.06965964802757994,-0.0011687485735218893,-0.044441637163063916,0.05380493446283102,0.04329936193282905,0.10143796860195345,0.27856107592311546,0.1705401234958231,0.3761724798987392,0.15019757410273024,0.36421676601606834,0.05828199383223521,0.0945060573234794,-0.06334932292754963,-0.1522347126102332,-0.12177702639130777,-0.17057356087461795,-0.06809084123408735,-0.021209374153003387,0.06649577057610484,0.16748256772164652,0.08605273087356824,0.21057127266161024,0.09621761572421142,0.1012901127298845,0.014311251892560158,-0.028302431701327513,-0.03262243084461103,-0.06911723900187418,-0.03148731432799473,-0.042162218252606166,-0.008874727263933624,0.003458636520244304],[9.174511716884924,-0.8613036114260968,-1.173721598362086,-0.6537751057640808,-0.4865684088119865,-0.23870126856849957,-0.2835507254079769,-0.15197293166503967,0.00524750798239974,-0.025170424201860084,0.0917346742067561,0.1175921498847086,0.3120732848353149,0.20676515899534087,0.5335779853509598,0.39125280015561575,0.8794990306450097,0.6015294501823027,1.0788903506776153,0.734928094901602,1.1164008024289207,0.7659394119363137,0.8590106879655501,0.44413653293017014,0.3316733518329237,0.05682264638259862,0.0714425148001788,-0.05303858968126401,0.24075841139478887,0.07311998915026093,0.3316431173543381,0.13842592780149135,0.16799822733097078,-0.1803136994214913,-0.3889391899747076,-0.5817021573447317,-0.7864288828213194,-0.7433655559259421,-0.7393920393650965,-0.5456272614927435,-0.3247223089014662,-0.2200767601302836,-0.10478466824342964,-0.1394235850471666,-0.2528810730425331,-0.2192747376395237,-0.34702697054733495,-0.2092798937957175,-0.2089759098975043],[-0.6152804641715972,-0.2447623209415844,-0.023367055237715997,-0.3052337086302394,-0.19798175038126348,-0.19016218078171876,-0.20464197996974234,-0.16655495654288913,-0.2541283359102922,-0.08691828691264017,-0.274068131511871,-0.014819871477462861,-0.2645280860724647,0.0022718431756192964,-0.3088804023880064,-0.014457639743650838,-0.30077911185373823,-0.08981901357314236,-0.2696692312404602,-0.08011441201424628,-0.27095194562587344,-0.04926829626400554,-0.31290928676203655,-0.02608583173971668,-0.32128631517833806,-0.0005135446263243629,-0.36097177362290755,-0.007868769011656767,-0.3073041589073371,-0.020123066471838116,-0.23909179788951176,-0.02938133769237485,-0.21259980986659488,-0.03758607458340077,-0.23242843576657457,-0.03815988772469272,-0.2513065184291006,-0.029072284250154484,-0.21850625804129487,-0.004343804056050063,-0.18301597550498785,0.0014075429446935442,-0.13557824115717204,-0.007936743289070571,-0.12108786431552435,-0.010490204218325692,-0.11001657955222409,0.003936942132892043,-0.08466475014504954],[2.040239848571919,-1.798076532903617,-1.4981835178502716,-0.5537375090238451,-1.1634712582004192,-0.0041863047979795705,-0.08468121830817357,0.5065233014719293,0.48301171824093536,0.8055275499729737,1.0628747250272061,0.9340207210507532,1.233300768209889,0.8270885644688477,1.292019790850581,0.5829190640576617,1.1086522902096811,0.42476798937465704,1.2417974883934724,0.3737645018831817,1.070804696548151,0.361092542857677,1.0026033298352655,0.19634566258818578,0.7069583170785708,0.06644466539304866,0.3089259599260548,-0.14454663301358572,-0.05197039399869768,-0.13998107592748504,-0.07812937802038694,-0.11656967030434022,-0.024926468408832447,0.05037374676519854,-0.0960921955974177,0.06790769411987073,-0.15252583617975546,0.04113455052018126,-0.18892213125932972,0.049994410966651186,-0.08966413115763419,0.0669906060828529,0.011841823190528995,0.13985409976817795,0.07803517921735009,0.07670739807414978,0.06592761721535725,0.028368156688918802,0.0389402247151908],[1.0083787767214627,-0.508076396054712,0.30223945062441293,0.058400552784676624,-0.5717684179416537,0.19131678782377282,0.007041192256425541,0.2154178611538955,0.14250880400412402,-0.039286630714946334,0.18806761272666678,-0.050497506483065724,0.3969306581758668,0.2866801700135642,0.11963953205877632,0.23983651991826824,0.0727595736945565,0.19427418529347196,0.4452019989792941,0.15778446731227702,0.6192701258740447,0.12001781607754135,0.43467320155660977,0.07342951144025674,0.2811055211246608,0.18460523016186478,0.04616037324925776,0.004350595669276184,-0.06052102794691815,-0.14894943471778604,0.040390117201390185,0.05269936271574209,0.1288259356679616,-0.03109030931552238,0.12688749958834197,-0.04395520050455508,-0.15925567313888359,-0.1512893710883734,-0.16387372716876517,-0.11131906768119598,-0.17343290187634436,-0.12373128721443996,-0.10987176199321756,0.0969851496272598,-0.09486212412939715,0.14251714001166665,-0.12496362066932185,-0.017887181860495534,0.01627278194103891],[1.5581342784528136,0.2750232122655775,-0.9370680834482834,0.44754030569262016,-0.7270704125848378,0.4350538910002273,-0.46773707765735073,0.37681518928053687,-0.27775807687358384,0.3513549496331249,-0.16244513409840272,0.28760216794468796,0.02805309814488321,0.2647293593083142,0.17534539711053684,0.23736239337259543,0.2678554788491258,0.1921513606537865,0.35470449612861155,0.11172741694132116,0.3936864962213097,0.0134913805270919,0.34768689920698675,-0.04337728721609888,0.30274976103687334,-0.06737212380727349,0.15911416926763455,-0.07986425304893281,0.11219975017677333,-0.042723782194653594,0.1525748419450538,-0.04203609118619272,0.16599102902909932,-0.05786880351962659,0.12185065436111028,-0.061077237108124106,0.011755663469336083,-0.046129468063494786,-0.1657301943483569,-0.05132944774812319,-0.13338940041816022,-0.031817500362630924,-0.04181673040787105,-0.005845913227405385,0.04742291763664977,0.013135615418339918,-0.01687662406277133,0.012112036334272939,-0.09026991019471746],[1.7933758995188978,-0.4884576222104603,0.9668880911966312,0.2637485379188684,1.3060606722874406,0.24683975522299897,1.2423563978241696,0.11226870640569625,0.9283870741026142,-0.014793452613793693,0.6102076233831829,-0.1263340644541707,0.4078077127377004,-0.11242690890399742,0.20144807330134784,-0.15556567464078352,-0.025420737707118418,-0.21848960446924875,-0.22376042769155133,-0.3609216384840539,-0.3522072723353986,-0.36003862707274664,-0.2262834006477732,-0.21100907229829807,0.027192283793771428,-0.09276484178578429,0.002336233927984179,-0.041817512743388324,-0.28036046756736527,-0.15310783566176525,-0.5131453330441946,-0.24743024379356587,-0.42543493322988385,-0.11854062182489289,-0.04859756693005118,0.10369483695810813,0.25580372085602987,0.18479988290598068,0.24983824512144012,0.11593077583341385,-0.021780951164167683,-0.03847015100829615,-0.1947690810950413,-0.0710881143186954,-0.12619624297101337,-0.039661934455723004,0.013283669093998682,0.04303877443967846,0.07908636157942472],[3.180244499306891,0.2873430166496017,0.007225086775895534,0.2290334418056332,0.3462469719392869,0.12854809973555442,0.2870527054338533,0.15999099937295014,0.32884506234482286,0.0762378848361783,0.36847409049246826,0.027417823188424662,0.30470037247672943,-0.03673180120966295,0.24530253807505106,-0.038357471898262406,0.021257996206584757,-0.028355777700107297,-0.2007504676900743,-0.031809011411235606,-0.2676866446999327,-0.042595911261013655,-0.1537707856225645,-0.057227158204562145,0.034261437942767224,-0.057316317936158434,0.14116362015832054,-0.037947665315607504,0.12261550589900332,-0.03478226981159582,-0.00036930441083919083,-0.03930492632276746,-0.10214573934332302,-0.03244084821039289,-0.1404250643136035,-0.006010917570368029,-0.07663335572861868,0.011741025428080662,-0.020982193962346886,0.04512060813486045,0.011509991086454088,0.028338026712881215,0.006049458977096736,0.0032557631507726006,0.01763047740377446,0.00027462695692147103,0.011044020227215356,-0.013623917161081462,-0.004478837065217745],[5.8252815055039315,0.37860454838477525,-0.63440799873796,-0.08322409798281981,-0.06149310725875637,-0.23998202715383016,-0.14051210569639241,-0.21248711594648814,-0.06598342354807088,-0.13536316463230075,0.18423091352502666,-0.02191813886912249,0.37649157243801096,0.07524751566605589,0.47040831844713726,0.13323487656017363,0.45091431791437636,0.004641624980732242,0.23289322115484823,-0.05992036735246057,-0.013895811004802854,-0.25359917440383245,-0.15665636110876274,-0.25745326779038746,-0.04689204309944701,-0.1335960607467383,0.2828883940767092,0.08029269961620376,0.4609506416444512,0.23463923854146382,0.35556566856146127,0.12670976650330829,0.16832571811026265,-0.03190314663024065,-0.026590090118178405,-0.08513018532992367,0.09262272957677524,-0.03305823969170079,0.22428309457947054,0.02560143123161185,0.19232357659580876,0.007182089750991476,0.0059334138589078106,-0.031208925715941763,-0.1860561394864502,-0.07340233434339968,-0.2104398573490874,-0.05424932897472231,-0.09785249132632087],[5.20502377559856,1.061561825073788,0.12440521837270987,0.8887040853453327,0.8502918548172207,0.6160866820869061,0.804397954704093,0.4423989549088611,0.7292824798278001,0.3759924203997505,0.7788916195870006,0.3588391486338199,0.7968653681893719,0.36313541048844383,0.8484233641312638,0.3718159172131338,0.8985770109463698,0.4239115872923779,0.9284434900255942,0.49744346259631217,0.9431277091272506,0.5521374731678211,0.9197159653134985,0.46406640534181104,0.7595066843822228,0.31078212086654594,0.5585921949283852,0.17258440622540067,0.3785650872702922,0.11525246501090591,0.2575059049775881,0.08176007990559228,0.1188915980575338,-0.06994900068311012,-0.15119514371178822,-0.3060319496605736,-0.4613133366719966,-0.4760565378228399,-0.6219201486077506,-0.4854411423487651,-0.5521599221298452,-0.3099710364674325,-0.3551900711752633,-0.1740101789748736,-0.23366529280474121,-0.10759205161280368,-0.19235533989925355,-0.0723470981924256,-0.2038206709426304],[-3.210754308311657,-0.14700585935904645,0.2263122778652798,-0.657292312396305,0.22777946052346384,-0.7794181575767047,-0.3115225445548593,-0.574403569898215,-0.48656870995038953,-0.4073524329425272,-0.5030599900347399,-0.3158176631380582,-0.48352761544805495,-0.22913620927617195,-0.423323178206444,-0.1939502553206567,-0.39217616043415565,-0.1841401290883955,-0.46633536040035517,-0.15339938228208705,-0.44095337874694757,-0.21671318966068714,-0.4882554996695365,-0.17816551903384575,-0.4918561139623222,-0.17077258147643398,-0.4991011536288525,-0.11493630048073328,-0.3880541651282574,-0.06892092799323661,-0.12643348093958473,0.0055429343202499926,0.08418220222021959,0.04679436560034597,0.19232007508209434,0.018019700171294856,0.07318283968378077,-0.048295974809424616,-0.08666001431012854,-0.04161292992194904,-0.12389968823553356,-0.011988401319100297,0.04733232994923645,0.05554514961556854,0.20279380341313644,0.05892759009868476,0.20289214893484017,0.03725316557048183,0.03422641923143705],[1.4005293547512692,-0.1150156081579783,-0.20599869346890848,-0.12552685871422184,-0.018814693330532838,0.019483555838644168,0.23390745244234162,0.17455888174276893,0.2396072797330749,0.2540736705870656,0.2762182705424456,0.2527754798311519,0.21219472630081526,0.2223741844442932,-0.10289150981731363,0.014830504199398485,-0.24333912234590677,-0.0882894351986676,-0.14295273058207114,-0.04243286596342505,0.13187443598115187,0.037164746914105574,0.3537360656932106,0.10236274747281773,0.26706711194229626,0.13252418702010482,-0.14500057039458075,-0.07518114670025705,-0.583975065638866,-0.2582549391757202,-0.443584259510721,-0.1278123639294419,-0.01968061889415273,0.12161347907289902,0.3882716097964024,0.1999825699573855,0.1953271883161629,0.12401773116709959,-0.09748270914443588,-0.061295564922706464,-0.24717336408961538,-0.08383464983345774,-0.012037784546239718,0.038221712424549345,0.26912697002147495,0.13999873675546917,0.32152366541530564,0.11733925994673583,0.1266763579119359],[-0.6858531442363047,0.3858569443254267,1.1525737211176128,0.5833998467437477,0.7429990264017998,0.34878537139200916,0.7629441539599515,0.39968611241740726,0.6429723806190243,0.2941064921710695,0.4376382498638896,0.2147535468846123,0.34137304847459315,0.15502469942689226,0.34478667254907475,0.10788244762287971,0.32521013380093466,0.018381402920779704,0.289971980263383,0.014099458804213781,0.2483775776055541,-0.01566316878157114,0.3243368840011865,-0.040574923316124535,0.26252506457027713,-0.015634863803353292,0.12918320474739475,-0.02617078624683461,0.09141712535713306,-0.11497703521203768,-0.09649429613258867,-0.06567122352626587,-0.16586042435929646,-0.08100664240516807,-0.06263457890277502,-0.0609188208401381,-0.05746837893871119,-0.036879918552646884,-0.05158189527706603,-0.051898321868674435,-0.10267583764655716,-0.051001512658371065,-0.12306372914586745,-0.023563354534991074,-0.06360817074940703,-0.018002991259504664,-0.07584097928000715,-0.018966260611603523,-0.07517955327777455],[3.3008533993913045,0.5103398646828116,-0.9029064526106317,0.3645059163803557,-0.07282355494102839,0.10179440134406283,-0.09271557527995032,0.1771048424969847,0.06944623040212503,0.0448991573467465,0.14899773867751775,-0.043734149122062554,0.0675973115861792,-0.12975872413972286,0.083286749798603,-0.11729558615333725,0.0844378320363175,-0.02948890000905183,0.09848855184974004,0.018114065886359405,0.2425348401246576,0.02903252892957014,0.3048448391888198,0.03733835667630325,0.20316139128769609,0.01252167308346902,0.29398310474948997,0.026752402011515526,0.31017909210034317,0.038677908331496626,0.3106009855268815,0.04033618499353798,0.23039151275611408,0.06777226243313021,0.2676245535197306,0.03989551627331274,0.30645316524932087,0.03440630898923025,0.38205184267534237,-0.005863108819166588,0.30501268527711617,-0.020173774625820862,0.15335186104284135,0.022269286273207213,0.07747609179048633,-0.006548274815500396,0.11476428429564829,0.01124705332615222,0.14475966316348615],[3.8853095213873994,0.3077121676625353,-1.7011124551506231,0.7100489905498916,-1.776240758489476,1.0136797826080968,-1.4291732440402927,0.8822301473138416,-1.1956758924801587,0.8425199369201763,-1.3152280136959118,0.7826211557729026,-0.8914592931774894,0.6787049369149809,-0.9375147873728711,0.543783108289389,-0.5644571642402562,0.49893429265938655,-0.17758019092099994,0.4963293650131452,0.4593997583519148,0.49687066122448414,0.8072936132244659,0.6309460486270755,1.2865480269723952,0.6020737893381175,1.142389652369716,0.3535999096438604,0.6312099604392404,0.23964581801899612,-0.05539416019863061,-0.06691070002826066,-0.10356189535008584,-0.029403769036780502,0.3902213312620634,0.13738981106290163,1.1200582665795784,0.3390312898493802,1.2212809739496129,0.18480792626081533,0.5198972750691547,-0.10885915746408689,-0.22355998769445246,-0.2373742972910795,-0.5961004223825549,-0.24625156094356024,-0.29843864331637454,-0.0492045822918578,0.3669033024822739],[5.40530420153447,-1.0255002598340488,0.03147747107538646,-0.7520104284636002,0.25693582878419957,-0.5770511315692102,0.03721898573946799,-0.36910858656614537,-0.08715484273897363,-0.29577468398464984,-0.015856754161155166,-0.08988641372039292,-0.18799488138933892,0.041048204631171065,-0.21749773384797455,-0.04246137695931114,-0.13546785444391282,-0.14045661853166871,-0.6084214749197918,-0.22153964631555367,-0.2866017390255657,-0.23526960330907748,-0.05695082662740912,-0.17344551611265474,-0.004871415601330233,0.14379219666433438,-0.13342196817682628,0.05755683533782629,-0.17644484506742047,-0.049938726027739265,-0.1496189089713611,0.0186743578381588,0.13992596161640158,0.008114934239213403,0.24939766597203494,-0.020627741114427688,0.11994648959096217,0.04952165059035824,-0.0961451120372577,-0.10209588278414589,-0.16251529854724886,0.01204422104973882,0.13682704236354526,2.8421632900398662e-05,0.2622771006182626,0.09240515493139494,0.19225150440991354,-0.020120159666505426,-0.026485575400773155],[-8.326198512804957,0.5971383099542313,1.489561079598703,0.4867738833556677,0.04886486533063035,0.485206040941319,-0.32942355649890226,0.5056308951735284,-0.1854881731548736,0.42838519835658145,-0.36028322139295926,0.4872769740907903,-0.23030195749533616,0.3391267944132156,-0.2257780270643633,0.19330273755445823,0.18348181961134896,-0.11865954772650679,0.8249124740447062,-0.19306612009512308,0.9998481171722201,-0.14397743145657982,0.7310024813278815,-0.009059034854184361,-0.052658764187243204,0.10927270293513253,-0.9057821795172436,0.13727448251339971,-1.0305475096343,0.08607087713065667,-0.35950492562199865,-0.029037310915868433,0.4757534588425303,0.03417412794125795,0.8902344272637044,0.039465997434527486,0.5615929560006689,0.09968529625383457,-0.2522220785930152,-0.07535448837166767,-0.7051745208624016,-0.15174887540440896,-0.5931766473917387,-0.13021035457260774,-0.18816774975220352,-0.1363336028867341,0.13141938298330408,0.010883594698275937,0.1976788740551081],[-5.549990458203409,1.4658909489320706,1.5060699640670578,0.9723428476972145,0.7222621119824196,0.38622013884883566,0.2810480275048599,0.07868706937224901,0.005276149210881105,-0.06855851190708571,-0.002931401335767599,-0.11805434127331614,-0.09505336943208363,-0.0770475293575551,-0.051624210979742124,-0.08364928623365253,-0.059565140547028605,-0.020839866069187,-0.08860889812031542,0.06640455659124533,0.0029591869589966004,0.17555884148765985,0.1849450433357956,0.21580316051448933,0.16682828479691258,0.176727653778079,0.04056329430943574,0.055979120254213297,-0.011257134188623943,-0.007070698202989962,0.03681476056257197,0.14260649985000834,0.22463024968954357,0.17204398278300845,0.2946887939543284,0.09343348869672564,0.027695104802020518,-0.06580135350940115,-0.18143305116960584,-0.1678276213352393,-0.13130483297184878,-0.08532127870872398,0.1072243119466066,0.028708637162658184,0.19479503028436923,0.031990814045174815,0.03179943083608965,-0.04111776846359822,-0.14118752026306808],[4.3178587615340405,-0.5440289821529952,-0.4077320117153606,-0.4575639686987652,-0.21419541413498044,-0.4304823938802622,-0.24419140896538014,-0.3804981512109555,-0.23674293802446963,-0.3206018192017148,-0.21997398794474493,-0.2855754823368579,-0.22509702026373926,-0.2889375010336051,-0.19875055259883917,-0.2646994935788933,-0.14281256014955462,-0.20500449330290402,-0.09921795924137307,-0.1261391894355053,-0.019656192051664637,-0.06228740408173303,0.007323065542007913,-0.04097012586570921,-0.0002901111852573801,-0.03864533757469212,0.037399281655991934,0.020809600683012068,0.17677613986812377,0.12094875828159961,0.29720412675151797,0.19086263335444467,0.32886951234135486,0.1647755300903225,0.25807946187409136,0.0945925649535107,0.1563004265844995,0.07407328544361835,0.18243012932650585,0.08945555211263972,0.24960544732690942,0.13185755236768126,0.27770526743914586,0.12273706288444818,0.1696921946834946,0.04271583883144377,0.05521985504759264,-0.016165957910778544,0.012824879068659735],[-2.6038028902714463,-0.4210524601750327,1.5710802739929037,-0.6330252199896849,1.0749409345204912,-0.5229006529782027,0.7477481384198046,-0.4870641181508885,0.4717495988603836,-0.43197352141152956,0.3232758350987581,-0.3947641694552816,0.11901107380886412,-0.35982835167615085,-0.054878414743965584,-0.3581797777843352,-0.1218280261359647,-0.26490011400767305,-0.23976856124706816,-0.14530984354743212,-0.2998584843634741,0.03276865888070315,-0.2736575194364652,0.07267112102195135,-0.2353770451783524,0.0855040311958473,-0.16064513865562738,0.06516124313063353,-0.18932811160662982,0.04404576673749351,-0.2601165754956275,0.08733027629392576,-0.21961155348577385,0.12171161772771157,-0.14530157962740142,0.10469606138303036,-0.003687158590470813,0.08047112225368767,0.15117986244699355,0.06805858945478468,0.12858862646898564,0.057365345306106284,0.062029828458150076,0.0387864915045464,-0.004648669282263587,0.013104186501302727,0.07677311434910258,-0.014736521243488794,0.1381004856265376],[-1.2336454370827743,0.8094318933367413,-0.6776400115841864,0.5597027316182057,-0.5969278580281144,0.3949640459463275,-0.7461012006710483,0.45584207895029066,-0.5437386739276263,0.520619247490382,-0.4168978057505714,0.5845151971919464,-0.21340199349163774,0.45999262196903506,-0.08639757153259625,0.34460572636457915,-0.00307302285472475,0.26334469378986963,0.218053843527702,0.2674770451332077,0.27930305887914525,0.22501187571508166,0.22349995806301384,0.1663119846824606,0.1396207584063528,0.09921596996275762,0.12802641689464322,0.05282598469743062,0.2488834290775149,0.1313044637602275,0.3983463040439649,0.12659054395179717,0.30330330644538034,0.04104856097608819,0.07112611702012941,-0.08890432807483598,-0.12042869291634245,-0.15958160281088704,-0.14170782567541218,-0.020542869837483522,0.11441652926621464,0.05736131800120109,0.21516931776333964,0.09487505333356912,0.10330740591666973,0.002266853407920079,-0.08357378420767772,-0.013020465852053386,-0.14968416252131717],[1.7685394732459438,-0.11867715899809288,1.2033890225744053,0.1791232753619684,1.164747434299701,0.30000132318822514,0.9763754764761672,-0.03274865515871507,0.8100011122723619,-0.18549388619593,0.5440807028611363,-0.4035118660400636,0.3928230538633734,-0.41587338822600123,0.13564506621705086,-0.47866436008607327,0.22082190516608124,-0.3330866393271066,0.05908258517427281,-0.2392374342083883,0.10859019956395442,-0.15667589351778138,0.038765677278802486,-0.1166238417494698,-0.06653247084611964,-0.09885382158335433,-0.20091176321298254,-0.06197887017145715,-0.2562081547948777,0.011958930939364138,-0.24720437866207123,0.09795851558793493,-0.14011381189354258,0.1104233486444555,-0.05493075273981445,0.10906583262654489,0.057889578103908795,0.06855054322107311,0.12637207555440735,0.03693574118367708,0.18406904741433855,0.026535752476142305,0.17079028108720992,0.024115158730495856,0.18553334414718278,0.0639726040840857,0.0868425031579098,0.036444294697288646,0.05998591443363313],[1.9665031895803577,0.19622921810386781,-0.4672215110221525,0.18313808827006492,-0.20415142177252968,0.0903504833545416,0.05375093709003032,0.03525436996573968,-0.0020579696791710286,0.052739922121264986,-0.02463222721681673,0.06407481339471471,0.005111185382134896,0.09293706143020915,0.07904034406581201,0.10851447409607845,0.0949196086943641,0.04654547905620286,0.04968067472801086,0.03945248270153167,-0.016521467313987948,0.014692963021268577,-0.024014840889709975,-0.01751211461588999,-0.029449344195098714,-0.024455602319116497,-0.09160244616677822,-0.011503862523437087,-0.08023565867028315,-0.024273611852202552,-0.05121461012314676,-0.024086864737781266,-0.004618546870734986,-0.05423673998917685,-0.024318806274060734,-0.041520368480530555,-0.09893276036137277,-0.026219959272665138,-0.11450284419727905,-0.0021149733074921654,-0.06973230143084261,-0.02952482500689381,-0.017614018257857966,-0.02221110725562641,-0.037586651088123195,-0.004157782016404089,-0.057679270579961854,-0.01306676751273532,-0.039245564041523],[2.237879141867149,0.0741576450137705,-0.40873951297193634,0.04963141554962798,-0.030749762299864047,-0.023395200889219075,0.017912998602081804,0.014879887250488522,0.016340186038565022,0.01777978998538072,0.02573165739521587,0.017450561126112447,0.003709315970705136,-0.011324427651299563,-0.03446855721716009,-0.034668967152800836,-0.07748260080340849,-0.05838354766834135,-0.0773951197845016,-0.03947442418041636,-0.0628696106883943,-0.011018828842177543,-0.0005202872144066918,0.032741772420739536,0.05399065990911249,0.055316036325788784,0.09251807300418044,0.053180065632950704,0.07349453021957154,0.025789613860786778,0.038340803994992005,-0.0038727920845253974,-0.010481313728768208,-0.01248297242697024,-0.036517463454285706,-0.01665753966874161,-0.027087614829282188,-0.005453505180089998,0.0038208196365382813,0.004597381248438097,0.022742351786870722,0.00878463887952798,0.018869259647690144,0.005372103168307344,-0.010880607154199048,-0.005122844493459792,-0.030448779827998006,-0.01256912660402306,-0.032572921112933216],[-7.40016680584912,0.44247676194385266,1.1361080664023444,0.93087405105311,0.5958800770549455,0.7968551909298012,0.8386833894743134,0.43434629106393435,0.7094618787394334,0.5505040378343411,0.6542378917273282,0.4249836796713551,0.5095316577987684,0.31146945389042296,0.6463464035935758,0.21251042452363328,0.5466758316603209,-0.061522191221730374,0.45058627998392337,-0.17180427804321322,0.4825322716593582,-0.17967609360133288,0.4589729603284497,-0.030222394296852247,0.3339847292129687,0.0021773256345883276,0.08689977991586785,0.015392465142549439,-0.16620230742105785,0.02706235671178566,-0.29113262870279605,-0.29385360719847725,-0.04662364739239524,-0.17540198738993654,-0.024070221055179593,-0.14527024831863325,-0.256866617229647,0.015772398769338253,-0.3632600659658029,-0.07767808725039166,-0.3634885046059159,-0.028719853272351173,-0.1517935303789285,-0.09832122124224793,-0.10301867745651419,0.001391927544094077,-0.21442137521533822,-0.005728751509842166,-0.25856035171297],[5.539428400043049,-0.9526161638385181,-1.4110304210620317,0.12694699062641618,0.5588545056036143,0.15783345601738455,1.154442869559822,0.5886562358039552,1.3134915890044383,0.6732278393475045,1.4339363943068424,0.6739156868061987,1.4363344691181172,0.5387946771559251,1.1319981837949142,0.3605527299390913,0.7892148279042525,0.0313058657590336,0.38133648832553135,-0.18993378333134706,0.23750169389105572,-0.3933303537639044,0.0988012836108196,-0.1752077476111855,0.35900265233519485,-0.13900891254341416,0.2517512955996478,-0.027745224899791032,-0.1761218359113839,-0.17008439292899916,-0.5443352045370131,-0.29635001100931435,-0.9349062848259916,-0.26502349950542753,-1.167878907268354,-0.2913899647542926,-1.1379611503437053,-0.16209650961349295,-0.8635518341810807,0.005118742686961663,-0.5622838340805172,0.26802315203814875,-0.18391881717027522,0.3849559271113458,-0.07446755643674452,0.4019504849768809,-0.035076928589958214,0.30034696062314203,0.04260510169946736],[-2.0428469290203597,-2.227759997373614,0.5852744080335469,-0.08669678938768863,-0.801951307407852,0.2564467959201478,-0.41795085581815333,0.24948238508869142,-0.47229235454806134,0.16630313021522644,-0.4524164846723924,0.02215223806570883,-0.5278228257876328,-0.09849884333559017,-0.5839206415396218,-0.26974451205925093,-0.5119673739143351,-0.26409802184430425,-0.6419471535157588,-0.2629862687032958,-0.4188558064393371,-0.1198631937768401,-0.3494511397249739,-0.09936835651910264,-0.20844156499238303,-0.20914953436491973,-0.2142808669478256,-0.281957976544772,-0.16572587014915713,-0.2638042690329639,-0.21472267667646985,-0.12165270611922493,0.012333614715676742,0.0356014690663803,0.3153890011837332,0.11031443273880973,0.446049371975995,0.18979866925255504,0.380131170451418,0.09700474651539243,0.12573541443093106,-0.1119171096332553,0.09821617489559278,-0.10527587919990741,-0.06700928259120377,-0.026574233823503206,-0.010397801829171584,-0.017107058709720818,0.06763000299078813],[1.2473022589166192,0.6263413187869664,-1.2596139246920401,0.8287573308197862,-0.26405482193215374,0.3689516982156209,-0.29365477318435895,0.43778429044887923,-0.12176325520219011,0.3888004527648594,0.04965492699862005,0.3837004921720052,0.04152356910528166,0.31828159170894404,0.2789463707617601,0.40867142185829175,0.32583052038817306,0.26478849943008614,0.4398078325026712,0.39132501794397584,0.5891449924502341,0.2739547621017408,0.5243346631758004,0.24272893276010007,0.20799193159922544,0.059295587836547335,0.19594238336637576,-0.0494076602717054,0.3891566861210197,-0.044176781821100915,0.37013234080016566,0.03120735184608919,0.07027576182488655,-0.014129465909705272,-0.16861925540951284,-0.1742389657964346,-0.29060417333415467,-0.13123965083781883,-0.10293298250988415,-0.04511828084581189,0.1282363039265823,-0.034352566312115924,0.03661197813715636,-0.07641768977024234,-0.23214872185062257,-0.06145046248905975,-0.15402920867936293,-0.03480045274836412,-0.056713473824062756],[0.6706856947050573,1.1021271697167956,1.3066370252479342,0.5552587601426175,0.1160276761908642,0.3245120783017413,0.04345965355767533,0.21844917212861675,0.020374798833849912,0.10446575053938295,0.13193389507661482,0.0008096428057901746,0.2379271378099781,0.04908417491838365,0.3798814407412657,0.1049801739662997,0.4829307074650554,0.19727925254326661,0.34922205162595144,0.24569607133177787,0.11913559080848221,0.14152555983181372,-0.15906392817009432,-0.04048553937378219,-0.2638301367092663,-0.1635299540772133,-0.09121771062089964,-0.1452336374479672,0.12936437469379214,-0.06595964005033982,0.2580646854466897,0.07215860370709938,0.2675292205590156,0.10641934939077323,0.10344610786626675,0.05612400691781455,0.01728554727380779,-0.0012350556752682798,-0.1025556757106409,-0.07589897865345592,-0.186784192623018,-0.12413519002747392,-0.23163536258367917,-0.12141530020472889,-0.14605190367407067,-0.013176397020191853,0.009317451420648791,0.016173755584883143,0.08170799806609419],[-2.447870148444613,-0.018183342800729353,-1.0720307378615543,0.1319224894784011,-1.818433169832306,0.35857319452237224,-1.9695721737599723,0.42592435072290763,-1.8070737704619424,0.3405399950900712,-1.7099451595741992,0.45974505742118316,-1.296971493859839,0.42202640299492933,-1.2465118107672062,0.4025584978431777,-1.1303128374025035,0.3547274404384145,-0.8409392288723986,0.4389484182568897,-0.5568702911338701,0.38819159770292705,-0.3751895474748911,0.41065289303989616,-0.08648573776390157,0.325805528805621,0.20603495997117363,0.24696029256641053,0.36081838694026724,0.17330697986288865,0.3142592506567859,0.14513581018413202,0.11937228629256653,0.10989900661066715,0.16431131530687587,0.009703177446137265,0.42505632879048,0.011771420136826362,0.6250311994370771,-0.018517552274732375,0.5838434780990327,0.006013972320397484,0.10425176966076519,-0.010222142568027207,-0.1675367815941912,-0.12828002369889258,0.023651511454416976,-0.041157394414766214,0.23571719532968435],[-4.508893208183082,-1.6350994860471195,-1.5887415711743724,-1.2302357423133208,-1.537102174392384,-1.0016140647033325,-1.3984962603285664,-0.856847360937305,-1.3997819581682749,-0.8024259538186234,-1.5354040131440099,-0.8914002901587805,-1.5980567598640705,-1.1065048178299344,-2.04979804195255,-1.3307936728529797,-2.2135756029283216,-1.4836358267462741,-2.0610006052664773,-1.2104209830732793,-1.4151219153120322,-0.748179867752769,-0.9317316652865463,-0.21248207494360266,-0.41793894949096183,0.011757460630184471,-0.46344062977793027,-0.06800995886855546,-0.7003159788108274,-0.2469175810288785,-0.8157482907049374,-0.3161948281210162,-0.6199545974545323,-0.07776750969860931,-0.23384494733150366,0.2736516349402995,0.15663650530763437,0.42228405510484995,0.2797588167189965,0.38385379062256103,0.10189135623280679,0.13882192776827051,-0.05429328490076827,-0.00021527633146549291,-0.05276068649186804,0.12644047146502754,0.09764112628108355,0.2052033278148581,0.2084643232372332],[0.8643942458710272,0.1019240485370379,-0.12126779303232725,0.03345631970420031,-0.10704141937381109,0.08031048946102241,-0.11904710244879953,0.03515147009673343,-0.13014285233581804,0.029629702158563486,-0.08145318981508984,0.08310887091853275,0.0015067474670824312,0.0874110563359101,0.026959884788428878,0.12351467035112865,0.031795768967839165,0.07008581979896582,0.06911673131040443,0.07630073642204968,-0.019130606327735635,-0.013428120433050857,-0.021572365138378184,0.01754077416989324,0.01801981735915376,0.020801154492943848,0.08846908771294047,0.03670709156383526,0.11433086821031749,0.014640611279586695,0.1089978236208825,-0.026734737899840055,0.05051936986714537,-0.03117285171878176,-0.021514814690629887,-0.01792875394407436,-0.024575568814092848,-0.021464570150362253,0.0019443999553567198,-0.016948995992970974,0.0002615293953168804,-0.005243023217765289,-0.016445303319885116,0.004867603516567844,-0.02733743164533564,0.007774056184335679,-0.04814014903648181,0.007300239272186609,-0.01784474052054354],[2.799572047605141,-0.1353966462809647,-1.1568357794944129,-0.07847105942968935,-0.7713745551564595,-0.6875725645859058,-1.2908948086592318,-0.8242580980270732,-1.4853575052625203,-0.7891267459683516,-1.5145337997697628,-0.7763318245650915,-1.1250908994077244,-0.5120246234540887,-0.5750523633180344,-0.16279299237199812,0.34803635532245214,0.008921153097629175,0.5928651832111712,0.2558705917455929,0.7796787792156221,-0.04711049797136851,0.016923983165885403,-0.1295587596466347,-1.0360923424641206,-0.649061205221333,-1.576253330903378,-0.6532250564655396,-1.4824336353310794,-0.2977739359615031,-0.4117001149414969,0.33403504360424274,0.7738815516676406,0.5998783698271635,1.3748861555280851,0.6826949195569196,0.9274985700161846,0.2587320868043596,0.14278359502093294,-0.05936480017407191,-0.3566943306683001,-0.20139404552151471,-0.25211380282942303,-0.12220482705904483,0.14932829592689612,0.06443683047732364,0.43691287593369466,0.22264419930253543,0.5952245291022004],[-2.0678831516288336,0.07946935942539218,0.8764612506702957,0.41663105262917327,0.3388620704523319,0.3059637006153231,0.1791399970025933,0.13531871604519086,0.03591336048902536,-0.07338730709367186,-0.008542302756068603,-0.26737647752009247,-0.2627184923698975,-0.32312463348718734,-0.41282790817128057,-0.3471766584312239,-0.5575709396506495,-0.30522218221945663,-0.48210251504213586,-0.3694441420337417,-0.4070751410166739,-0.38026443114539116,-0.3466647994768259,-0.376120942007689,-0.40731166911262745,-0.3466650383761353,-0.6371055893838578,-0.1993310137662236,-0.8533080042654659,-0.11631387177928958,-0.6445086788203584,-0.05787767569958132,-0.30588872406168316,-0.055415642464537986,-0.2285715597967029,-0.07999678258370495,-0.4526869001095203,-0.007470808951646811,-0.7255962235918906,0.05493839031461617,-0.49467825058943327,0.0988203816076909,-0.11863343224056924,0.037113835807384754,0.035346575633491834,0.05354014188672505,-0.047037532198322844,0.06086159336825036,-0.2322062293353187],[7.804300189949602,0.3701647257170138,1.8465927854413327,0.414936440067407,2.4167044759470073,0.02280758820823619,2.4962739192045493,-0.09196606428187028,2.4445906478089525,-0.1250678215948826,2.479915106927517,-0.19670615415908999,2.181750499723869,-0.39288449502398337,2.0119442498581983,-0.36206432537465405,1.842528108563843,-0.45263317374805023,1.7504493266612253,-0.5247275046402531,1.3295718271009525,-0.4732350091793862,0.9274217531444585,-0.6261489808986768,0.40420240344228164,-0.5619076138420072,-0.0681263010188319,-0.5864336792546663,-0.3606786444129516,-0.6346473347863003,-0.43774333412299954,-0.6397259158128566,-0.4745788976432847,-0.6066797965228216,-0.7635186743163113,-0.4428630136466659,-1.1531195518831863,-0.35463668467668763,-1.340297658643747,-0.2880975963194035,-1.1091784939300813,-0.2326154277155789,-0.9423774140913832,-0.25903499249225836,-0.7515463617224512,-0.16536346924603237,-0.8355713876612634,-0.04071070573591626,-0.865593661405331],[-8.013794035101503,-1.2462071009683056,2.927922116182169,-0.6190469985504128,0.5722671348190901,-0.09052982234984866,0.8165181489457684,-0.5292914949885824,0.15515334691111823,-0.43325370935071916,0.1871502217840884,-0.8406721646415604,-0.2674793282168068,-0.659645573348215,-0.15934324739908656,-0.4758044292266835,-0.1303006782559227,-0.2781333048369006,-0.07999838203306216,-0.3329295115983012,-0.04105993184105239,-0.07306762703213424,-0.3057438479157019,-0.3429093037179277,-0.38527439720859846,-0.1666985265808899,-0.6743223934905013,-0.10773667348775433,-0.2809608776221279,-0.0029889812136012586,-0.20845030942769782,0.09695528902394518,0.2579616100541241,0.36782917797503056,0.4972912200996763,0.2755935417056912,0.3366047148184006,0.12089666807145777,-0.06887269087212941,0.04636322195512061,0.0860525131293767,-0.10516997069519647,0.1651030101087606,0.18658833336905567,0.4103620846850694,0.04679060778569178,0.22541483768727633,0.15912412641282495,0.1077673809210713],[0.49080083783973105,-0.33818011419619465,0.9142851522299567,-0.31453310372948423,-0.23548175078177205,-0.05045822553706522,0.5990771318722128,0.09503722381157402,0.18016433768001922,-0.030889390234510743,0.259989584491167,-0.3327982768862152,0.5150016112150942,-0.04659228513857093,0.1787734507186365,0.06496645555210973,0.05220277765035791,0.05748604656681344,0.33467572198512807,-0.11288976258799648,0.20897377651324506,-0.13282903187918868,-0.14486777230106748,-0.10726613525734525,-0.2005334063172631,-0.143196209334051,-0.060150423162180645,-0.029696469121935132,-0.1305668493851167,-0.03404715210853408,-0.04966875216582462,0.0154396059642042,0.0974432864873895,-0.03030588691309054,0.06271053820290402,0.10834133352137672,-0.07794400391654684,0.07989399641397045,0.043725215130610784,0.03858414993169893,0.18277917726904833,-0.00491873066314424,0.033643696660004045,0.07285859690275923,-0.08792401549179829,0.10915843778219782,-0.019752043538135974,-0.049859952858066936,0.04963777317412321],[-4.030808463069853,-0.11662375554353474,1.7188936935950612,0.08636461810736162,0.8546254439692388,0.18234481961783214,0.678888829298845,0.050621968398835794,0.3552244869850408,-0.12151687939517568,0.05139205073070114,-0.2647001511546318,-0.16776888349303756,-0.30433769309908915,-0.3097314894902149,-0.3106877460207605,-0.33913428204780705,-0.2386962965501359,-0.40549603691402275,-0.1938441863328299,-0.41210831490975713,-0.18074548321004139,-0.5381538831583618,-0.2509008362864321,-0.6254761020603886,-0.33371979080683717,-0.6623352996934715,-0.3238438660812917,-0.5763287861112442,-0.2195415917014736,-0.3418215235638795,-0.007091164358890642,-0.07926276542404219,0.11157256167752541,0.08660525718896679,0.1368183473643712,0.06722288747634547,0.03257023686050618,-0.04414874907044579,-0.04313462161478005,-0.08385621546656101,-0.020458821881208178,0.038193219142843765,0.10343883057189975,0.20172695634085813,0.19105536210964838,0.25798390173347646,0.12959318240331905,0.20506817892335125],[5.1862702492806685,0.1772292169207373,0.5778090919188301,0.3207959886786569,1.3497497589712886,0.22250987431730315,1.4070322819020167,0.09901615664465484,1.6341594701840731,-0.0202205741439472,1.3580285655912958,0.09977732960156482,1.3102643084861976,0.11136101776886605,0.9369339578412459,0.14648050277045888,0.8295545488115899,-0.061958251131919696,0.5323577169794612,-0.17743766138257588,0.616312333339123,-0.2472179998929274,0.7637621275999504,-0.031070644786334208,0.819179259940808,0.2356012735495491,0.7191656821296172,0.11507303985587033,0.2972091363432693,-0.11524163563552964,-0.061141861344419714,-0.2874858778178373,-0.015931382876998367,-0.1165027174576492,0.27821869715629455,0.057653812459612305,0.4056618232165138,0.1851120816041051,0.22147172824316708,-0.161675635034579,-0.18589333147495343,-0.18370168921693814,-0.16356930091099295,-0.13412509104633671,0.05112670390952035,0.029092310079111835,0.07988839712173432,0.04673859404105218,0.11824540403195825],[2.1654266927597146,0.39925106378816316,-0.2832827308528586,0.12927423402457877,-0.041661691329480395,0.13794056270928892,-0.20745738686826606,-0.0550414207605991,-0.18078766526356774,-0.1512640509934248,-0.11680845264160455,-0.10867439055460439,0.05593324732619403,-0.04583559654993336,0.17503110807098798,0.02266811492144059,0.15736907782943646,-0.03260429575868429,-0.06549115581531012,-0.09822139771927828,-0.24124639015965624,-0.24195696690828092,-0.28341051508414145,-0.17451572590139616,-0.22120910665564714,-0.08137136125443503,-0.01889596541453431,0.0246530872376192,0.04634332156613019,0.04836434391092957,0.1237915842686173,0.025228861052541657,0.1578400946602413,0.04150138898818693,0.23326247327678332,0.06157049946933878,0.18495408246111533,0.07604759902719403,0.05763800947785408,0.020034817138127287,-0.02445127324804775,0.009068437453317625,-0.0052721199695579014,0.010140868411956267,0.05780907042654004,0.01825718524663824,0.0779685294193569,0.007177924381570182,0.05103029252886234],[0.7401784190719651,1.4674412828638361,1.3453900763492603,1.1322130633157412,1.5696221514023714,0.9135102477045004,1.568268397176324,0.9868774271009264,1.3660030408179065,0.8910957651155578,1.0975693784651654,0.7251977938923879,0.744769842105914,0.3487477614869585,0.24151890059634082,-0.11855433607512839,-0.08500213038083416,-0.4162288882170327,-0.12754079670380514,-0.3166806653597204,0.39935846774013567,0.09585650931177707,0.9027753605782789,0.3068746271783399,0.8226163630162239,0.1311488795985218,0.12924990336909867,-0.33549557199432173,-0.5298453898299357,-0.5785838556723009,-0.6875512114576212,-0.4053167253417012,-0.21944926094935846,-0.12657753404764743,0.20060431458158987,0.03419202576365118,0.1914347403163869,-0.04376561870482309,-0.2532190449211542,-0.2665782765830967,-0.533801538574085,-0.33228189889260595,-0.5225063267494374,-0.22214359832738081,-0.2173499478901977,-0.05160716386850205,-0.0480442138549336,0.005851194422753882,-0.045589208503305084],[3.5153039452390793,0.3484306471299212,-0.4528295484536536,0.46026471210183084,-0.07628727075042227,0.2670470310207554,-0.060986148032751604,0.29182316097645045,-0.05414720608943565,0.23941096091894098,0.00591912156403298,0.22384030733012594,0.14136691521784042,0.14878859692303978,0.17417345059930187,0.13610761600445262,0.23370074512731412,0.03507760966979811,0.22104473157231497,0.023048277288821056,0.3340965640111044,-0.03038879275786456,0.30728901343962145,-0.022433751300105227,0.31966349288815016,-0.06832179454748737,0.22952865154685115,-0.07482753313475794,0.1109193768833475,-0.11624595162317333,0.06982184385310557,-0.12453469029711046,0.04060186230711519,-0.13501501618971473,0.0035599337845028264,-0.1346492385043539,-0.07160328426578272,-0.10118087473347986,-0.1548622213418559,-0.07819952972974747,-0.15990394381468995,-0.05714423109500005,-0.12122783664356733,-0.04103996745511203,-0.15162571129126026,-0.023428512976832874,-0.1537059347440398,0.008212004481510935,-0.15951876094648737],[-0.5377684156421336,0.02983513443692205,0.0401465335397635,0.04556180854906695,-0.28163565823364867,0.03316944622364117,-0.2083333896305352,-0.04757659397921853,-0.16849393394072626,-0.08233784354432531,-0.07908606149774165,-0.06796729715059381,-0.03733378874496067,-0.047837675088659264,0.030802051552466212,0.004331050341065016,0.04540221849318088,0.0029342784466104745,0.06420655073130474,0.028606540165500405,0.028530598256719445,0.007483026922881425,-0.005788793576327449,0.024244800859528397,0.01344803721293976,-0.00794555712383802,0.06857696682126835,0.00876019742821035,0.10930624728977532,0.003174183272967214,0.12351221392053735,-0.019465679166629155,0.09029600431233821,-0.020275660205370916,0.08479344622844331,-0.06275951387483968,0.046162852337060405,-0.08352651767690014,-0.023162176527239675,-0.0817678815161445,-0.09791590779150759,-0.08499339701531373,-0.0963045524515927,-0.05864089102439085,-0.07165183268472909,-0.042762101239789706,-0.055970647849461784,-0.023336371831797344,-0.07585778241648691],[0.11028010675258362,1.0837797295951448,0.021766941853998908,0.8505543817733098,-0.08869542114611885,0.532100927004956,-0.3039550380388629,0.3432130763399515,-0.36233920242893386,0.27866857485078894,-0.3568978355728553,0.28584627233342,-0.11324427653268285,0.24847078482995516,0.04919654462045077,0.2748347699999277,0.24765698518315094,0.26690260053978326,0.41328146811005395,0.3067528859034395,0.4790554092040027,0.2659148883758846,0.432236948595432,0.20380903167854353,0.3401146794778342,0.05026539118966878,0.19055843579739148,-0.015172291870619761,0.1509284377221683,-0.03698943351691548,0.12291840963698647,-0.004715182356892972,0.08933405758462362,-0.03716296570041534,-0.00901887201949731,-0.1255819894346905,-0.0905986762633692,-0.1170262265231355,-0.11591714423110423,-0.0649609624949965,-0.07932069497388348,-0.03009185443236127,-0.061820307930953104,-0.04944687989409448,-0.1316119431503879,-0.0622690230947983,-0.11444947965352645,-0.021252725173376348,-0.06764442440015575],[-0.5215965085140826,0.19192808353472968,0.3067703274587289,-0.2948226868300511,0.6635743424423819,0.02147075760567643,0.6705194084017392,-0.06582942122668271,0.4825321305301755,-0.07689109509033545,0.27413943383810097,0.004836543365310605,0.11623344944742901,0.07248117975158067,-0.09739400174154017,-0.035443104336542186,0.19521117936085425,0.043936246284019924,-0.06802115839702884,0.00575502395628843,0.07969361552517884,0.010769055788835967,0.1860310391871212,-0.0850459936824471,0.15597044692703033,0.07697845725153876,0.07356229134118125,0.07238245251063447,-0.11828379393814854,0.056587889962737295,-0.17488749209397797,0.06797335386688788,0.011646351662053578,0.049425336196681886,0.07110228306988142,-0.028520479205780432,0.18752425445323606,0.01969327826588678,0.10863193196272482,-0.0756966319519695,-0.020643636267550065,0.0031265303155676664,-0.12665115096130267,-0.04478734845938665,-0.15162094352598648,0.041081938638893,-0.06540403137716198,0.00792314976404059,0.0707680595861929],[6.854895663715429,-3.273932810967494,-3.6864360991674743,-1.6712368282216492,-3.262280175946293,-0.6607966863032015,-2.3579681502981886,-0.5493101590786772,-1.6669739705732547,-0.23460713581582834,-1.4217705509105856,0.0891787239914442,-0.8964207842177703,0.1797956593163882,-1.002363109317472,0.07371610829681484,-1.1695383833262225,-0.16919314393766752,-1.0138738761902428,-0.30888386926033656,-0.8733629694055589,-0.137733634602263,-0.36242848011661,-0.0312797027510312,-0.02966398198133924,0.014136101459654379,0.030049068254481844,-0.10057220793324215,-0.2917053330017011,-0.16559682809380835,-0.47340629965018394,-0.35182236355686336,-0.3007221942309827,-0.3018050855630616,-0.0001605058803748119,-0.15332903165531117,0.2787998308168336,-0.012972942263535753,0.10899587565951312,-0.07381944251492835,-0.002555908802300415,-0.14554424466443303,-0.06830322746960081,-0.13485975864168714,-0.015705895598195407,-0.040694625615805786,0.1887040640508581,0.018478893685638605,0.23035093520633754],[-0.49445262223653513,-0.11001652315766626,-0.6894006569051534,-0.4210187920695706,-0.8015676678446831,-0.4657664384746299,-0.7122826615180976,-0.3659045343770666,-0.7989663589167707,-0.22566698030893395,-0.7196435480116661,-0.12728057393446024,-0.6355317816861947,-0.03750165015895679,-0.5060616838831753,0.1382330089441541,-0.4867306454818971,0.08193360950564495,-0.4405949106255676,0.12082677151192597,-0.46532530838279335,0.12140994605060107,-0.5245547252425747,0.1063299101575937,-0.34404299568588453,0.11798289865239935,-0.16247942515533026,0.0638499893759633,0.10775037961196625,0.07099116555452985,0.2859919510805621,0.05752146670464966,0.2564902669167389,0.00438395698651002,0.2513298842916553,-0.059943720491112754,0.07925384712715984,-0.11175453539669798,0.07924667765244357,-0.09325319206946983,0.12429943880464189,-0.1260088749613315,0.21524722109294117,-0.04400240407512187,0.0960394285445216,-0.06329266818735219,-0.0725911181492333,-0.03823694747116533,-0.1419135765929307],[-11.723880424681354,-0.4485370349744487,0.6027454209318598,-0.5061656042674738,-0.29021042205188563,-0.5092666107094077,-0.18747994208821409,-0.5557467983431964,-0.20751530195099455,-0.46501153779424015,-0.12666276552175879,-0.4939109422522117,-0.2473605926707668,-0.43262893445992706,-0.37277789994694877,-0.38886132919028965,-0.413436748160493,-0.294430656943604,-0.28544271596597326,-0.07877396225253999,-0.005501419170184152,0.07636424585388879,0.08738180073128259,0.074286858092969,-0.0524684456309053,-0.002380974558821411,-0.14083572354063575,-0.05593146946496605,-0.11000137354049332,0.03677609915438983,0.024846119207607435,0.1244179614469183,0.1353338134153463,0.0923261726160501,0.103536312991359,0.024868130109837676,0.002385917897530832,-0.019560688125683783,0.05416475498381448,0.009088255060285055,0.11731957602399991,0.026527492465345654,0.15890801470285332,0.017344010172452146,0.08708081338251478,0.007446016130420804,-0.005197813864867019,-0.05271989909499436,0.015488296764480235],[11.74031170272649,0.6526525268551365,-2.420572956343378,0.27217245865108014,-0.6154944621512297,-0.05893820365271646,-0.7490010251795066,-0.05112646431723963,-0.19965599807513715,0.2230363864059479,0.2199608980427425,0.3103266006857806,0.7662183018326406,0.47993568640071066,0.9453276577217021,0.5502696452519075,1.0725450148154798,0.7251435843887707,0.7715819678268807,0.4675038629391103,0.601835142821813,0.3001867099050108,0.25222496864345834,0.20771099966334466,0.24581596962458271,-0.024238872068214762,0.6082099867563004,0.1011783133597767,0.8314481355370394,-0.029500109066036336,0.6137149993690335,-0.11121166177697688,0.19554671101223453,-0.028069636221969816,-0.2268932964489676,-0.12763194075276504,-0.04325667272948508,0.05376498715948817,0.16221508026747838,-0.10480386254724473,0.15785545047747176,-0.06502620872090992,0.07868664969592636,-0.10374450272852703,-0.2080056912485571,-0.24898637552258557,-0.286967859266349,0.01922988438558338,-0.2799854138115137],[-0.985089499244554,-0.3570719011159305,1.0869110615847972,-0.46828554142612355,0.8536572985190208,-0.1860550690677151,1.0729599755530124,0.06111941193050959,1.021813457380213,0.029074386510955078,0.9411049601620232,-0.09023183868928066,0.6755532391078324,-0.09124449144832976,0.7231568888429712,-0.05811548341064191,0.7901643454060945,0.09806960479170793,0.8139890584726288,0.17353130115462867,0.6261653666026114,0.1613225741467784,0.4152302844540494,0.0950778284500415,0.29839430520358057,0.08961115567925884,0.26958775039666427,0.05393676433298335,0.13334688386626867,-9.708478456667468e-05,0.07410433475587759,-0.014051311758240894,-0.06356063110308638,0.08434460002127131,-0.02991927998316975,0.11089836526346313,0.05469104904495529,0.02892928117451642,-0.014673667793394981,0.0023616639220953054,-0.3337708854243524,-0.09240770029736661,-0.4613090878798071,-0.10018056752047452,-0.2879026849714176,-0.033917413676569176,-0.08765529073662387,0.0465702057816026,0.05400935194653229],[-6.187818829168152,0.596524345326053,-1.5533942435043886,0.045335827982024166,-1.4165916395542735,-0.14699275445018192,-1.5575617518470533,-0.04787740915355878,-1.6934021486295316,0.056457974620608975,-1.5878718611933549,-0.08826113219536466,-1.7058787922059007,-0.2995153592092023,-1.764040391401902,-0.418056111945419,-1.7961115752503343,-0.5097186607136969,-1.6301474011605357,-0.5541123023019414,-1.4864510174555252,-0.551716811546761,-1.561872820684124,-0.510032237986796,-1.5618340378979831,-0.6142659677936879,-1.8007147048841088,-0.6776717434544118,-1.8534469931716817,-0.7076916930452521,-1.8385794371187176,-0.7536095357514422,-1.7636971607228058,-0.6474084802828293,-1.7104056184733223,-0.5591270700713146,-1.6316300351371678,-0.5274344389176067,-1.5414929757987461,-0.4344632534659031,-1.4784161685077226,-0.3746173696408058,-1.451995746448891,-0.307511636350983,-1.0789039023170883,-0.15990858748413367,-0.890862311512962,-0.06960615673136189,-0.7429525590470731],[-3.2774291574360643,0.5187469196975224,-0.14345323535402088,0.21036344329603002,-0.4484267657893011,0.20370583787817656,-0.2505733104820313,0.25655007335850943,-0.4113070224029689,0.1885020827281429,-0.28123426047033734,0.17086065924684296,-0.30971071167148145,0.14399520094978724,-0.34805075106384625,0.055604673243402375,-0.5148441347639713,0.0031788038816522876,-0.629914705285025,-0.07307574109372374,-0.8464644960389036,-0.09270721091349367,-0.8025990216476198,-0.11631114778116623,-0.5507051359456827,0.011684946404284009,-0.27447303890941677,0.07689578212640387,-0.23197933028079043,0.0573536915917141,-0.32211081545722964,-0.0633790235148734,-0.5237636487394272,-0.06488234797794386,-0.4952728566247996,-0.035765715616096316,-0.1777221704368571,-0.019447329678915683,0.19653235953743586,0.14061242237332522,0.2218260513818572,0.16059079770008516,0.049079629457958907,0.11451081217779668,-0.1780518945704401,-0.029912975180210088,-0.12118751999619412,-0.05074703227170326,0.0887714048049816],[-4.2229304413469215,0.09476749814619431,0.45883573410432144,-0.04040995744957603,-0.31506128929611327,-0.13422384453342098,-0.3372065744383404,-0.11911671698636914,-0.34406304783443403,-0.08990274470238196,-0.1659895072751612,-0.03578685789200399,-0.01838091398770205,-0.0025406671187437053,0.12538738757391982,0.06956541940104093,0.2552185912846605,0.11494073028097399,0.29751644824237844,0.22956860778906857,0.3230528646141376,0.16709888302751513,0.13936381344270551,0.06540610306148928,-0.08805780649141344,-0.0875276378692522,-0.13422802072940307,-0.10846889106207284,-0.04258182072789414,-0.031176280800825237,0.11445401865990061,0.08578900844564231,0.1752369902257611,0.050900098484857915,0.03745011072324056,-0.030072304492966294,-0.11194949986539339,-0.10785756321633348,-0.1462285031731702,-0.08461167840606412,-0.09589592797314006,-0.011980688004030017,-0.0030597720068017854,-0.01726086079441346,0.004766641214814959,0.016870549403553182,-0.02693895702752247,-0.0404165805378913,-0.030930328861963258],[-2.1743530653213585,-1.071410972994646,-3.123083685170244,-0.5282288268275628,-3.23683212849275,0.035291519137316756,-3.263639050485857,-0.04150466716747026,-2.9209820550279137,0.09598639226100383,-2.927475182062844,0.1354672385971177,-2.722805498389968,0.3449250642125445,-2.5004911105328547,0.23425870523137196,-2.1176707948477307,0.5267872051840397,-1.6917608098824422,0.5118489840868347,-1.5772700902646377,0.5804204404052006,-1.0233475403691563,0.3869581368647339,-0.7132861036562667,0.38321935146993025,-0.4346921288983152,0.09189593936746264,-0.12739327887518614,0.20118427786078164,-0.14632386633564434,0.2633034207693773,0.015372527062747388,0.23476283474826162,0.34623351368888344,0.06236321044246061,0.6371303449789599,0.02377563466823378,0.6678063350231446,-0.08269766808472528,0.5040130259837259,0.01782163444115812,0.06561259485106982,-0.028519243123857462,0.07547026725556927,-0.07425757537296965,0.04032298226640696,-0.03547286763425329,0.19605403345517927],[6.637170569345164,0.5842002036886734,-2.9628302056990803,0.14026939615427844,-0.44067738188123906,-0.43667349026548163,-1.4429527149559498,-0.2025823901064874,-1.0308358765896593,-0.05912111557190257,-0.636338183648182,-0.03139624134253426,-0.4573467201782081,-0.19382614866448447,-0.6197179421703919,-0.24345053467342562,-0.7175652447277394,-0.32530152099297915,-0.4985874859381317,-0.3734045753779006,-0.0559477196154505,-0.2015972728947876,0.03385375750659657,0.12881417324543956,-0.23862704288277042,0.09704149363767274,-0.5368457298868204,0.08266762297141192,-0.494869553835726,-0.05190822206253354,-0.14428065722910344,-0.06846342020132345,0.28045520250189543,0.014123576870072334,0.4128399271423686,0.10974320713942265,0.14131095563825458,0.08519737991520526,-0.26079852172166373,-0.004941384220163077,-0.27105970638666765,-0.05548283142178383,0.06681240856922464,0.02739491664021512,0.28414008295569604,-0.059194232033827295,0.1712131557898675,0.000942194050341978,-0.22734916250033468],[-1.9817797973077298,-0.6438249281446583,0.5675013404930838,-0.21078886348167677,-0.18998193787568293,0.11617521004926289,-0.011320817706204335,0.16884807050706943,0.21314093234797174,0.31504031376567776,0.26084276896451614,0.3318615509500174,0.22203576392233185,0.3616213925122548,0.12718942487629958,0.25276555342408186,0.08171345059509384,0.22771154699773274,0.1542001891528567,0.16693266402770207,0.3200840324281636,0.24428639552007358,0.44943101754156556,0.18129197366550356,0.334707801735023,0.12369183317174498,-0.0046644397823014435,-0.048041973947135726,-0.2348401178535516,-0.09938547917877695,-0.25678459243494955,-0.02321051788087064,0.008907366477268985,0.09951582212251799,0.31385546046664703,0.20469521452181788,0.3653720141619484,0.13808825116478965,0.17072698932512606,0.01742417177527222,-0.12075826864243733,-0.1044627235006304,-0.2340624892678841,-0.09132959825676012,-0.0611077077107726,-0.042638193995522294,0.17180930988785345,0.06679368430128327,0.2705990159267643],[-0.28668472932608047,-0.07510495668064195,0.3262630422441595,0.23107588280524002,-0.4267968723637316,0.16864123478089516,-0.3981872658382336,0.19494361618925074,-0.39178379029038146,0.10404740199331282,-0.20791338722111483,0.03430665062707099,-0.06324466755771352,0.06381259993961046,0.026895332760219885,0.11767443168705145,-0.062091730011371374,0.1898344026116847,0.00266294527875819,0.12502848627098065,-0.12094655133315382,0.05278906981944813,-0.0693416658176666,0.017653581677754375,0.16417496745775292,-0.03463450581934787,0.37563517587884415,-0.021536552954495882,0.28945053453931163,-0.040872031672614224,-0.013440936135939299,-0.03182497530135253,-0.23413339110135561,-0.03902945563993718,-0.12013188353242087,-0.03059194805208057,0.11632555649658122,-0.026744458997698915,0.22405892413289066,0.017581003818025152,0.048275236344522646,0.04695737306013659,-0.16292323246863247,-0.0011427653073675442,-0.09909884265908418,-0.06330234216559064,0.024853327853521425,-0.06058948565450067,0.064656399168196],[-4.0316353410251935,-2.926449129029125,-3.061124490494548,-2.7275391294582296,-3.4335054781171865,-2.3971289699067753,-3.139721296977644,-2.0679042033121036,-2.743461851440178,-1.675580362661828,-2.373636238762341,-1.2845290043416817,-1.856759400458556,-0.9134321761645168,-1.42991023959532,-0.5336852223025338,-1.0717513781310262,-0.35306510067899144,-0.8124351135546577,-0.25791070990431425,-0.6679990777252948,-0.2134795508255242,-0.539777675479356,-0.07661474976346748,-0.323229476928566,0.059037055879676,-0.10397969558822306,0.15768620511253498,0.015564487248386142,0.11759030814035912,-0.034603093038447816,-0.011667476457578749,-0.09501412064083416,0.022319449869205712,0.024819456100716224,0.19466735881325384,0.3359213369512012,0.36375911285562157,0.5535366320360365,0.4246908269489283,0.4896068155231765,0.2258444580619858,0.23785155960494286,0.05881914992199735,0.06399867095124645,0.012885920144350868,0.06961644291347549,0.06110038137654552,0.21081333130799393],[14.305228748467107,0.011741839344197664,-2.03888244420537,-0.25702063543960124,-1.133969203026864,-0.4192995768391928,-1.297561323720347,-0.7184823477613254,-1.3798439061438417,-0.7558593040931615,-1.208730065058672,-0.7909413543130648,-1.1056543244411785,-0.6612628829989828,-0.27502250828889113,-0.2203627425268882,0.3828291855684185,0.1265218851994324,0.7684815186571254,0.37912686956050007,0.8367389040929805,0.4202778984505836,0.5773980959958754,0.16922442987042088,-0.1125506865663541,-0.1737864913014752,-0.03663814772728048,0.07045655337663839,0.8023444347662376,0.6435940672191766,1.6363002975059713,1.150604931833013,2.086373428401726,1.0053708368146186,1.6725632162808315,0.5180927667025705,0.7236325589933308,0.14355701914217817,0.4243312677696108,0.07585297796813455,0.7383386769657045,0.365243041199042,1.158873064216466,0.5124528827392695,1.0296383408529555,0.23859758447248533,0.48972775885626996,-0.07994441512525409,-0.08869528980397977],[6.47610263508995,-0.2756138692911559,0.8922186723734603,0.36874301647254837,1.0120086676774773,-0.12356764221807896,0.6940442466446382,0.041011623355646475,0.838297686975513,-0.08921052620992552,0.5820585272146869,-0.10704783481172615,0.7985236641575695,-0.0968960337472874,0.7715072774280217,-0.03432929071427268,0.6645817687597868,0.04387417046933925,0.468655371510305,-0.04742944147960182,0.323070345573152,-0.13528033503827322,0.13456608827909275,-0.24391417946172705,0.3155583962462721,-0.14110289188794065,0.23682010209405105,-0.06728902033649589,0.029557806511070307,-0.042278617537178015,-0.031467223183256655,0.047675144910156814,-0.17205104794597673,-0.034872500327834864,0.18299399567321006,-0.02031021499152234,0.23965754119155272,-0.007441292174135287,0.0851021750760653,0.010372192439770169,-0.15799614668318665,0.049039107336185696,-0.1838099309005297,-0.0007658703424142523,0.16210675953568232,-0.024086822220277502,0.3063678890010774,0.02558229900238021,0.19988533609385264],[-0.8689107773510734,-1.1837377731848626,0.16645340292000566,-0.615010667485388,-0.05566862153758473,0.02206327486757693,0.3441771727109266,-0.10994705478577667,0.49934663266641605,-0.11974902814396139,0.19288812047981016,-0.1552829959346285,0.16855264715804394,-0.09153767150472909,0.14733830074522844,0.02283057630513159,0.18806579932597678,0.28715291061982584,0.34068714262389815,0.3864462441917949,0.6566443904514904,0.4643242572576619,0.8658738461445447,0.42392193421483015,0.9935766785215101,0.2797926283842967,0.8550515648890036,0.10423235306549178,0.2938258369560254,0.005123561717021992,-0.4078294849670951,-0.21436786731919735,-1.0096452569258567,-0.26641100466813766,-1.0214869909139839,-0.29631400461722285,-0.62463025698406,-0.12293424370809894,0.14128912006025024,0.05108532400738185,0.6822511783385444,0.254355699332575,0.7340524178840535,0.2360743957865747,0.3950073889367449,0.08420105063491497,-0.08467932976312909,-0.032141273857496135,-0.19090434079724597],[4.313730516912974,-0.4787102613198605,-1.5038938656290404,-0.5156738334377164,-0.6310004078369241,-0.29887341504273796,-0.4408417868038905,-0.1928570401632539,-0.2908968235181861,-0.14120241014459745,-0.3275168388324264,-0.11858245786977259,-0.30040431420884967,-0.12123012248908076,-0.2978248759310092,-0.11860481107558393,-0.25899872620137077,-0.09750129210028011,-0.16103766734459174,-0.03883087085197265,-0.09389950724688556,-0.05415671903683155,-0.19004774546044975,-0.08435645887722315,-0.24273001233647118,-0.112263172585552,-0.19844708451952917,-0.07167008619922242,-0.04586534608764175,0.012922060364894935,0.07442563289341955,0.004402472548789714,-0.015361320896398717,-0.01616330766145509,-0.15929955919130775,-0.04440097208460122,-0.09393507998938311,0.023915173443912332,0.09939074670390488,0.09511034288380464,0.16009002780026446,0.07637560388169894,0.04396025108081749,0.004773124582952229,-0.07744986890676286,-0.010057985716013764,-0.05661126326335587,0.01806736891196453,0.053722312591603615],[-0.4604617331651072,-0.6268266126726764,1.8974292420250143,-1.2114019241027327,1.9496469537078496,-1.1145199456908863,1.2189179927033615,-0.7362758788426316,0.8287112126924923,-0.44704589776935233,0.5345875210703476,-0.3138390331730701,0.5641038088744205,-0.24975100217104135,0.5233709777666723,-0.281060437821522,0.840970789457081,-0.3491835827801281,0.7358694709580554,-0.2514865702942699,0.7646620158350298,-0.20300861906676732,0.2540822019856914,-0.1699837287736548,0.0502411364473274,-0.03194962302153229,-0.15449469837151503,-0.02282395136334364,0.015712594607733745,-0.07608331766766971,0.39315996874695863,-0.0031209136559113624,0.6068197053939054,0.056879201755877754,0.42642754723286025,0.09013413354883924,-0.013407742807866975,0.11549719300778419,-0.161437797515346,4.651012602007852e-07,-0.05125883199036269,0.015407262252713307,0.20109747561683716,0.025020116000249528,0.2689991009929525,0.08437917311169815,0.07139755703104103,0.07688573319658253,-0.1447643736873762],[0.10826143990759558,0.5577513420288595,-0.1090842972846096,0.5118446583417973,0.40809624357268826,0.48840603296163404,0.5861380730397707,0.5633692968572194,0.5810175638145912,0.6833994167638934,0.7232556327202664,0.6988353315578714,0.6406617461426816,0.6053836145707319,0.35503389181805445,0.21698437705856496,-0.1398019924920725,-0.2027879160545576,-0.43323816912671426,-0.39909441850369104,-0.4516232057811924,-0.373721819628327,-0.1109131953982654,-0.0904911194676282,0.24966737114526888,0.18128944134260347,0.37180659765232776,0.14719660899342898,0.055665493814116396,0.048375314404828264,-0.23601904923891862,-0.20411695378006708,-0.3192414270202081,-0.15049360167857856,-0.23069257716442979,0.013482995971003378,0.060600902864151415,0.14951939118274327,0.3088616563105035,0.27357972749625453,0.3942857572263205,0.2420388529437334,0.29206644052802483,0.16040089299529833,0.1632998934734928,0.010705947993829007,-0.09270532089151942,-0.01262324588922147,-0.1441179615084878],[-5.3675507287629145,0.04937519528829249,2.1117574671654507,0.23031874806361982,0.8269904863715748,0.3578292154241771,0.6409683518171754,0.4441044669496389,0.7723727002173996,0.37799641083004765,0.7893997504354209,0.3647680321405429,0.8321365583971998,0.39482996813927823,0.9274396574251833,0.39708624801377534,0.9637414496504154,0.5045946885931568,0.8403064722143221,0.3523544297567489,0.4860303849032407,0.25642313547006373,0.31674549190463663,-0.02013450896661009,0.2679936944807305,-0.023479572129591654,0.44625559718382357,-0.03092768225318442,0.5494437685629636,-0.02602468361566113,0.34494944117822396,-0.03480082353970196,-0.011322308608080374,-0.1148295820962678,-0.06169693206902225,-0.15594197475448884,0.07647901043418735,-0.13816371583280868,0.055335216263909595,-0.10064071123654493,0.011859237539826772,-0.08211303040801422,-0.09872204838227903,-0.07496235555404865,-0.029446584507033965,-0.06332381682512606,0.06532928318548843,0.001341354353808394,0.03847484607168005],[-1.7331638428146385,-0.13213681474220249,0.3184924731764726,-0.5039313070518843,0.11921190068563703,-0.6227045014605579,-0.2733377067894703,-0.7669206704300487,-0.5895259249292224,-0.8459953487432978,-0.9218551984288325,-0.908930234755242,-1.1440907795770023,-0.8354691535138794,-1.2967502146994896,-0.7167155168924184,-1.2633185991584015,-0.5308577308583191,-1.2584849127410103,-0.36954101159641967,-1.0293742587872639,-0.19869487050470724,-0.8202934149651564,-0.030056831259123052,-0.5827210968200681,0.03553165500159693,-0.3697680351569207,0.11956702634441264,-0.165608776441594,0.09864853896733328,-0.05071291692640281,0.13501954407102648,0.053458389785387084,0.10120376737311221,0.14196190805229023,0.05276871353189205,0.12217063743372848,0.010051392022762189,0.0921616716332992,-0.05076350895399468,0.049853313753708724,-0.05445812169989148,0.07203650005029953,-0.03215679318725789,0.08000144693674657,0.019931261688830738,0.04317903429402188,0.01571032182041718,0.024023986684433455],[0.19372697726717727,-0.2966561265909926,0.40771267325749744,0.17805539758231795,-0.8808942704234735,0.5311735742391076,-0.047190203664122195,0.4026728386464796,0.1296294558116297,0.29089065287033433,0.3211357307865727,0.324582755265294,0.4787936077343179,0.3079054976312532,0.6498800425612025,0.36070305280200227,0.6939677595505895,0.29043868653104543,0.6878396034855331,0.2563675986329349,0.5061389302557809,0.24015933423763242,0.4057603263194279,0.1688801463163178,0.45100154103993567,0.121356032117748,0.5136048717068671,0.088341632497969,0.4736789646499423,0.06610319524157417,0.25659993236875495,-0.006060792668163722,-0.003633150661755839,-0.10232608844871245,-0.18660573437531408,-0.19375979193545553,-0.23387135759525965,-0.1975702480989417,-0.19746644864647495,-0.1955906683705716,-0.19746876480745573,-0.17184104400878203,-0.24797632758953536,-0.1404529308625,-0.3127727222470868,-0.10345337232913557,-0.33566947135395364,-0.10832183095488218,-0.26205809875762415],[4.759894464781316,0.7743400014194849,0.045885178106805244,0.07947884026448535,0.37712308907619835,-0.07210777803867006,0.4870102350293328,-0.37703883840675934,0.09130105726392326,-0.5852530580529355,-0.06353250660693674,-0.6063550982403819,-0.3909792234429208,-0.5394687725376954,-0.3717348594007264,-0.5019438575928168,-0.47822216604282,-0.4753397828621336,-0.34472025964548186,-0.25829499692434615,-0.6083706420079034,-0.22136756283172232,-0.8604756248816107,-0.24695398823280598,-0.9545032606051134,-0.12102325466717459,-0.7229236500274396,0.07274818297256115,-0.26275863758048296,0.26008287384065343,0.15302400523865808,0.18315402555555307,-0.08080488417225631,0.19791783275958696,-0.4409683699998676,0.1324844941623598,-0.3795156865567776,0.15202072074611442,0.09787038926551665,0.18682522858809777,0.252527463486481,0.21090627917319388,0.16458953736302795,0.18268604130002122,-0.08688887687811683,0.03286518195083474,-0.08766378448685874,-0.06921963623247748,0.19838431033099013],[-8.126626857418634,-0.9354215641961309,-1.848863223361031,-0.10242426331608011,-0.5978174552271456,-0.14797891400089502,-0.4672604109200722,-0.07596096326169335,-0.4238847648024455,0.09778117552669138,-0.30654967754033763,0.2031306533518651,-0.15243353869021542,0.23282508341728309,-0.2870016636926448,0.05280597574376217,-0.39588053982789173,0.04424507490445789,-0.0936818741197629,0.04289443274655632,0.41719844691309255,0.3673738927654686,1.0299259722305933,0.5403866735552534,1.284856492789804,0.4762610728985411,0.8029152367055802,0.11141461382655651,-0.06736656527594935,-0.2816222648790302,-0.8400731094248912,-0.5440709152272086,-0.805980753583173,-0.3729638744083799,-0.41608844948424817,-0.1276988847674413,0.18582608393764571,0.10849807102415782,0.45624460874512873,0.1816327340649074,0.3426702403028235,0.1027029944441805,0.07964742326321253,0.008537918428865528,-0.11079879521816695,-0.10385941885581358,-0.2277381050861341,-0.09260215539130746,-0.2408013502378668],[10.268699534429675,-1.0126895106905673,-0.47858596640784246,-0.8246552416659039,1.3597486506136522,-0.9514820942729847,1.4890404986802015,-0.6277509797702041,1.6669904826671358,-0.4439839759636157,1.8553071043959941,-0.3807024964380774,1.5966621357396047,-0.4055623125015629,1.6334931529511116,-0.37534634164948977,1.4667838043869146,-0.276264492804897,0.8430965824627519,-0.2219506310471721,0.3063002899757028,-0.16845156430199368,0.35827422094827927,-0.15032416252182387,0.539171515099613,-0.04373062717571525,0.904366759896541,-0.23688131728491205,0.744354561811279,-0.1968209754086764,0.034949479233296304,-0.20670388281551438,-0.39148530662478165,-0.11890829352444704,-0.33591362986566253,-0.010036223262507091,0.06439592611804563,-0.09200366165719744,0.24459998120144663,-0.08809564620918452,-0.14150114556295304,-0.08743523394282032,-0.419207899043352,-0.01593310023320392,-0.18234885440111706,0.04836203864997293,-0.059244354982102496,0.02569037439580768,0.036912367415910724],[0.6687296548291688,0.05793145332105999,-0.8829889582480457,-0.6402884454962378,0.314260274926802,-1.362957881801815,-0.35778455328156217,-1.2012205555692095,-0.6166749131520082,-0.9972811503470104,-0.6363478112206139,-0.9825611721238803,-1.0484351496988509,-0.908105943732006,-1.1888369111822312,-0.8145207965071601,-1.1222176327469895,-0.6441882199732827,-1.2719058613263279,-0.47887960715974043,-1.2007639930887897,-0.44021232587681813,-1.3929217841972712,-0.23257001211479436,-1.53444991701729,-0.2334702620485067,-1.4824116842029134,-0.2280443589963886,-1.0545357764002943,-0.19026910096906202,-0.27788236562203095,0.08862897960907146,0.06795495731614952,0.19900016271178617,0.20265297739566607,0.13520895604140432,-0.3686877616243849,-0.07512757080558516,-0.7438562693580877,-0.07463295733106129,-0.6484577423443503,0.026443138950967206,-0.036155255138239764,0.10604584399721707,0.47193208303967693,0.17215167378535665,0.45917737434857026,0.13682868042186308,0.12662076909594702],[2.606919572587687,0.3244837235214463,0.5296174817151704,0.13605301083854782,0.47643197660453146,0.19870164990984054,0.43706660009106024,0.18988420786836957,0.3718471063159624,0.15852684774126533,0.2996593785444341,0.09789844603116292,0.2219470658910908,0.04767932858961727,0.04309293555488907,-0.01935621480654846,-0.008215808236163953,-0.09695216479491535,-0.07571690794437724,-0.0814615568669244,-0.039863153810652044,-0.11270055439253365,-0.03359931055323806,-0.06469411630355468,-0.07237608778856278,-0.04665373717518868,-0.12995536614581824,-0.06606999373462834,-0.19654707340445918,-0.12620382317198525,-0.14355888670263736,-0.07107288167834988,-0.046611119326162874,0.0065874293994901975,0.053800813296975065,0.07509330382119955,0.03573082699453113,0.06087546196189147,0.005250103094693665,0.013962585762342798,-0.04843999064937119,-0.014956235606286571,-0.013154924103806575,-0.004014982908989117,0.015209072056305234,0.042870313775597226,0.0455209991995554,0.029868453846677547,0.03856015561870744],[10.243027949421174,1.7785106434008002,-0.7668934906145125,-0.6998235710564457,1.5222555464191372,-1.2140064513001056,1.0545943152011186,-0.9566278998367973,0.932653667223789,-0.8001682005824948,1.278941614346473,-0.7764236869286596,0.7625342546305699,-0.7548526833342206,0.2149417684024945,-0.6215623876965805,-0.28539744688834434,-0.2070617246436971,-1.165703737997881,0.024278414135296775,-1.3143515967434667,0.13033472582032687,-1.3231723122550647,0.04285809787028087,-0.5127822024127884,0.014079108996447544,0.5007888335494842,-0.0753736054989199,0.9644000277219925,0.013644734593841315,0.48258238659704905,0.11357920851675929,-0.7247777955924228,-0.053472242008679,-1.349257631014367,0.016673664925168655,-0.873034497078157,-0.21294422431582818,0.22518366728408695,0.06320040255335876,0.8673426807641532,0.08424554772097025,0.6033021229443776,0.05297602710448381,-0.19940280053372367,0.041433610100561714,-0.6102892023865439,-0.13353217741095036,-0.3065450163272438],[-1.523790044197243,0.2929377353931601,0.7884027859165034,-0.38864548973200147,0.5220956724422428,-0.20530364290476535,0.17918782542670292,0.0009401873472977582,0.024865965609074902,0.13907083555066646,-0.17617124257579903,0.029759842076919712,-0.2714707921060329,-0.1937569726445011,-0.441253455291799,-0.4039459559633567,-0.5011367142516628,-0.4879678168165209,-0.5092369348567553,-0.4331111994870046,-0.2694865661700994,-0.17760468373352306,-0.24255594635381658,-0.20217884557338828,-0.28148879949594824,-0.39046190822614457,-0.6191404276828814,-0.6195816857256495,-0.7922787082563014,-0.7114785697656966,-0.9969799432040806,-0.7700576958095351,-1.0070875521531586,-0.8046546211109009,-1.0738311237422027,-0.8510796978911275,-1.0612974580689445,-0.8920064529990478,-1.2313958215414011,-0.7843619024351304,-1.3985316135499313,-0.7440428079148266,-1.4891456091962774,-0.7074601292760893,-1.2724042137850449,-0.6013381172555227,-1.0678022383521566,-0.3407407606572264,-0.8134590319893399],[-0.3344262980468233,-0.7720993689817944,-0.9365915204467262,-0.7333285031131699,-0.6478530739730578,-0.47562499925321555,-0.5917114396934177,-0.29161377911108144,-0.5356561081950041,-0.2030468994951796,-0.5426734138862682,-0.1636283801866228,-0.4903732916546205,-0.17458193554021822,-0.5518726363200768,-0.24753275556571055,-0.6142038237211189,-0.3066099381135932,-0.5834821778948754,-0.29112230263283373,-0.39082127522418164,-0.1500192640198763,-0.22096285474524294,-0.07881433174978023,-0.16908798606945175,-0.12521120250408713,-0.290776596813486,-0.21418157776944227,-0.41363229881466773,-0.2541914081979848,-0.45427053616314683,-0.21704361701346048,-0.32729142428918423,-0.16980165509994774,-0.23292733371298707,-0.14220019082399235,-0.18731927247416533,-0.15068066964016244,-0.22776167355654103,-0.15547562972643192,-0.2820204522901485,-0.14591410669556498,-0.28840308783140145,-0.12835770008964958,-0.207672872723875,-0.09225487278218429,-0.1477332295272644,-0.050652501131912525,-0.11442196507670782],[3.2951809382673276,-0.005897853954584007,-0.4020027204576826,-0.09760806060857168,-0.05523583861905527,-0.14947649335888885,-0.09562682887111586,-0.17952182180488135,-0.08407601128677397,-0.16867021945148997,-0.08303182379665286,-0.1253031236512919,-0.04275055013971722,-0.08192031582350004,-0.07243391705104135,-0.07165074719554525,-0.0448722546509325,-0.025238458451860575,-0.11215947845081023,-0.023099398203409803,-0.18328569389316562,-0.059157192079217526,-0.1586863338472425,-0.08688256773751665,-0.08055406287528359,-0.061272605368461824,0.013266403623702813,-0.003066817972663683,-0.023379118137086496,0.027819731133061184,-0.07819123358474199,0.03014234840719291,-0.11279843478424598,-0.0006438238995613845,-0.0542004204754923,-0.023164670286544665,0.04608478100696434,-0.034569133685054126,0.08655098069712025,-0.004798717861733243,0.02560079314067354,0.03460795516534294,-0.05082686654803612,0.01624870871044237,-0.062148528750853645,0.0044516466460165684,-0.049209127641061126,-0.000969001198115172,0.008945549889923195],[-7.087049433555171,-2.4826242488578556,-2.9473971567137665,-1.5685365706741954,-2.4055625094854434,-1.3595777276579242,-2.2130636870655858,-1.379279827921472,-2.0273133268012726,-1.1572830861967551,-1.4840446943041574,-0.8295940462505635,-0.8188937036722996,-0.35824166014439807,-0.031648256291261584,0.14157468416933264,0.35191867035808383,0.3219435982540162,0.34091629867517015,0.2365442847515092,-0.24879300292782702,-0.2865824238995374,-0.8620787363957834,-0.43053673494468975,-0.8804291542491838,-0.26969679964882604,-0.30465151788489575,0.2153976537310522,0.28143275779483906,0.5533856117946242,0.5299940987217298,0.36710947706037983,0.13477391916796364,0.04671496948984534,-0.25303588100954166,-0.13289128952202864,-0.37526387789063415,-0.11613862838082117,0.02900912181663436,0.20135508596495602,0.4084806970863849,0.36911958582367205,0.5663838881151493,0.2548800173787029,0.2995423985014524,0.10499691235942234,-0.06583760984469199,-0.024503565716528907,-0.13367422291961198],[-3.066479241050623,0.8684657295649775,2.0298177280915097,0.7530748296906994,2.7787788447776482,0.18546639080685515,2.6390299242428537,0.149030970411323,2.3531802173377314,-0.008650139792927064,2.143542562600958,-0.0019120171312791369,1.9773549216478663,-0.22138434896575634,1.8113911811553196,-0.17560136160278855,1.4518239925049286,-0.38067738535669116,1.156959933482119,-0.5938593928126294,1.06459793982647,-0.5187830064255667,0.8952397211943103,-0.5002838374027736,0.8417210980326377,-0.5369050013654086,0.3634966195153659,-0.4453705099687115,-0.11086612594539857,-0.4364729173925262,-0.4377065741473838,-0.49933789970076997,-0.5341269285847993,-0.4293960281184895,-0.7089198664417901,-0.2734527534382369,-0.8425081916583217,-0.15311223139421104,-0.8393058358289898,-0.015884869589705104,-0.59771749011534,0.0791980360180092,-0.27225155731967965,0.04665659954848903,-0.270211030840702,0.1868246125688079,-0.4836702311598761,0.13561279861182465,-0.42448858930428046],[-2.0717254304959916,-0.14277612930372308,2.0426106519762834,0.20446762854743586,1.5932637173614537,0.2972331063394336,1.555592160716983,0.19071283864515948,1.5905292747872737,0.08674225499537683,1.4399514893093457,0.24465065274767367,1.6638841452969213,0.45485052229434775,1.7010395093450639,0.5886321316711927,1.67902393721462,0.5555579632125988,1.4429006666143842,0.4023525838875631,1.1686968593402243,0.2790716783896201,1.068456039456386,0.21898687418140736,1.0635277078723457,0.2752610909331885,1.0790954275631273,0.25334922574412866,0.9318097886502665,0.20718457410895075,0.6790538970491636,0.11197098955194563,0.5690800358280799,0.044603637435768596,0.60293982482608,0.0924798287701842,0.6748460003851674,0.0788705496539946,0.5323848426932926,0.039783884288812896,0.37597223546426917,-0.02556760942794884,0.37241692884222893,-0.010121416110923882,0.33398599754283403,-0.028715865224768,0.33426937529503103,0.00849189988606721,0.23600747504392838],[-1.5861722205841742,0.3125815501365842,0.23622292158736521,0.4043082438739495,0.1514504857545074,0.34232370152518704,0.1724471179435416,0.3169072230692087,0.12052406223665937,0.2957224656842851,0.13788241423682984,0.2043273407284119,0.07324579654452268,0.1745091835998415,0.10665545567002192,0.09915866379965668,0.13733395621738176,0.08975832400609383,0.11056717125491211,0.022573945774054565,0.14168650228762458,-0.04058413881410923,0.147960980747265,-0.020886965998627202,0.07071419290437234,-0.0056320790585064,0.05203720559309495,0.020145994808542157,0.029936561584878826,0.022765662100999305,0.05066787491269292,-0.025640195578372393,0.12436268490609821,0.0024603830041864507,0.11232975062183936,0.007620487640876383,0.03824759316400706,0.07516506816502085,0.07277969511988006,0.0691655658542815,0.06342300400803445,0.05760305750786698,0.10159599271419102,0.03248116473438757,0.05005965602380155,-0.0021646368065822656,-0.011509055854222454,-0.02364416404694536,-0.03184606261271252],[-6.493828346029983,-0.0376813117655123,0.4804169445745948,0.6444615946208857,0.000888357846829656,0.8483695690990066,0.22926339354698458,0.9580880957037036,0.3566357578121759,0.9650492707046914,0.5759191653184227,1.1436751740281321,1.0928461760057853,1.1529939649361989,0.9970839247181181,1.0413094669269751,0.9102356931193679,0.6899543911680863,1.1359341701264922,0.6181848770914647,1.2371408193900924,0.6225719028861951,1.5378354521493323,0.7582278901700633,1.4918810273455942,0.4961408729561274,1.215343749579203,0.264701951566155,0.6072159599736026,0.0368017771086668,0.3010191901339516,0.009843249337972088,0.2952607363529362,-0.050284791111720115,0.3562204747440613,-0.03631501980907751,0.07708434294504712,-0.2357545793050827,-0.0867018939556997,-0.16609251992054033,-0.17766529313925952,-0.078547115715836,-0.0438056982039276,0.01036577295328844,0.04634834871011534,0.03425428518012386,-0.04265825168138412,0.01951282355558427,-0.21217254899363788],[-6.194653177548348,-1.193202964096195,2.230780843132706,-1.4586310959272901,0.25744334076162206,-0.5650964062852232,0.3473081727061214,-0.8343266451432765,0.3292720185090017,-0.4977869032353552,0.12127686372851289,-0.5599596355969908,-0.18783367109611795,-0.3724600670117466,-0.5208697600254558,-0.3994233544776442,-0.9885332031366761,-0.25858814692606685,-1.2080659670044065,-0.07910158867298496,-1.514993249527014,0.1036421292733074,-1.1556530181796232,0.15571366495312491,-1.0582605848231355,0.5033342684609068,-0.6725745930681453,0.32673964340328604,-0.17254044638808586,0.5114935870433142,0.08524044340546862,0.5490250706362194,0.3086069648320029,0.43457440268361014,0.6562338937051236,0.3011793110316058,0.8556091891691997,0.15420827269967913,0.7175917851253995,0.011158596190610668,0.7574103922512228,-0.1600177439970889,0.6380531696744874,-0.11858928050784869,0.6680432265296906,-0.3057093856013416,0.6397636222037579,-0.3011400919841622,0.4375198696447826],[2.2677974757757733,-0.4338061694387425,-0.980345752908403,0.29012597520982714,-1.0819064832746974,0.3202323439117837,-0.8747369891833788,0.31094075590918985,-0.6920186361855342,0.2664052261842055,-0.5248212648424612,0.3020231495501864,-0.2968795217528361,0.3045943242179216,-0.1244383054176802,0.29743000126534935,0.08226050637019448,0.2813565548173405,0.18758774510853513,0.21554696102436724,0.29333987445807047,0.17057765696373334,0.2716817832549683,0.14521551719979403,0.28041222693465107,0.059348884070847825,0.2753991799881524,0.038429262625770365,0.2212190242256597,0.003189307726311237,0.16674093660798034,-0.03700353140690389,0.15261720223886413,-0.028292436647630113,0.1519394594016532,-0.04582442127465242,0.1467734357048712,-0.02701344691640393,0.05932689873805496,-0.03823961470426048,-0.10506207837309847,-0.08565492233632035,-0.15926195949320401,-0.09259171093341602,-0.21110381540325696,-0.04536204052729669,-0.1145456317379661,-0.02184161395032785,-0.025172042157525925],[-1.1354426353395093,0.5720177565537757,-0.36347797256916087,0.3814077740278559,-0.38209333238629273,0.18311225970084635,-0.5193275414920105,0.01796718421761432,-0.6563369400686977,0.005633212588545338,-0.4873026110449092,0.10688257282125423,-0.3257942589571813,0.22720343523499215,-0.16879184956896467,0.23906543208618816,-0.17661024847015983,0.09001037945751761,-0.29225286948031953,0.13331737058122248,-0.1529380637478603,0.1077252606050755,0.1704884809821953,0.27347613901064477,0.27125093620605817,0.28505847027767783,0.17150513631794864,0.1710125901782202,-0.053359996512153694,0.005992271727215364,-0.06977147820094912,0.06946184242983602,0.16107381241336385,0.08293485399808316,0.25546052424643856,0.07865472447695548,0.07511021096821567,-0.016232475077381753,-0.12403278474382491,-0.03361034111988764,-0.1047161303184616,-0.03550069479307295,0.14663431071478364,0.04116959047848086,0.10670638338604158,0.08139328061093737,0.006236293669303243,-0.03218718965522549,-0.09509819987921173],[4.008725271198603,0.5597263443663729,0.5067730562641415,0.15273066609015584,0.6500496887775171,0.10263497419300918,0.5492135626141443,0.19189458638680834,0.5214660371929879,0.11868703680823503,0.41177588415837474,0.12261415380145825,0.38933900525927206,0.08164434153217905,0.3144828136309071,0.07283090130110637,0.3347952370762355,0.08846481776023217,0.24997474367486744,0.07727287607266933,0.19634925815037357,0.08538161459587237,0.10734841463777486,0.016420348339394375,0.059445720855817216,-0.006790913473701855,0.0040338135446155495,-0.03832144382659259,-0.015672928186902118,-0.009481947035606973,0.051663588367054,-0.0009167515247739983,0.09656893778030563,0.014289803776207973,0.06060267999480436,0.003117622010277418,0.045452966876522706,0.019392851270647993,-0.017077514136203575,-0.01753731496444269,-0.032701576691929336,-0.03174676991243792,-0.060203725374388124,-0.044602210428634576,-0.05031030056891038,-0.028773273439504053,-0.0111420229704976,-0.0316512495963629,0.045459028367955186],[-7.476794383358062,2.8798942841646205,1.1019148720966414,1.1584521901275997,0.5371958331504474,0.32065264211917166,-0.17881858141520962,-0.12471535234534141,-0.8409332788059216,-0.48389978802028333,-0.9884180744732396,-0.5999556660686018,-0.9413136765422068,-0.45833554067611365,-0.4341221823815134,-0.15788654058423274,0.11009595890490456,0.20541812242112417,0.5800567149732829,0.5497408430481638,0.7455556972830443,0.5321149428524715,0.5362886888946469,0.39181130056825947,0.18103113609309388,0.17468551630636972,0.1487790638013566,0.18120974229457928,0.4337391192550975,0.3624124928714594,0.7936569633169872,0.4923738612490186,0.6404930121729953,0.28703538279322677,0.11235392450368058,-0.09949843193006842,-0.31456109890315365,-0.22778167203365407,-0.1686534566958129,-0.06588787086002189,0.024184673902434988,0.1111709627001802,0.10560239147066909,-0.0014798583810120476,-0.07069375415031529,-0.1000862657204384,-0.36180946548342174,-0.11717236394680138,-0.29268430997997164],[1.6132656556427152,0.17500730584042484,1.696111072261045,0.6480622478147529,1.628841857107832,0.4202279659001152,1.6315195109650291,0.15684053550941499,1.2196558404547373,0.1230446949861443,0.9147499393772314,-0.1221972434718557,0.25213072483286636,-0.19566669466239844,-0.44086722318123894,-0.46372202584112715,-0.927133078217003,-0.40692150987574943,-1.216795476042146,-0.39995500989789434,-1.0098030495485304,-0.2433458197941691,-0.36292184320076726,-0.0055758909098111865,0.3398806961892741,0.24440347825357953,0.548724814788536,0.10620383603393405,-0.13238245026883774,-0.2526779035748261,-0.8911170850835916,-0.5833461258676708,-1.042174634778419,-0.29605030012475814,-0.39338324157697047,0.04960391820734574,0.4652766331907979,0.40436592528688803,0.7456690518026542,0.24028144831183665,0.002966019456158455,-0.06247703678877901,-0.5116856520952211,-0.2107912278392123,-0.5187834230755396,-0.058756276215731755,0.11903430716965586,0.11472554057261226,0.6885996972267102],[-4.04794060857938,0.7856186128831937,-0.4937664710497874,0.4122958967781105,0.09872099514850291,0.05039350733046121,0.055930931614831016,0.20225954347648803,-0.07833072576950992,0.2091109477573732,0.07056678299567912,0.14664361150673355,-0.054675583889567704,0.07037372523380463,0.12740375919892621,0.11475752199422233,0.21368538723041688,-0.0651148930520759,0.05021410672557187,0.09553048019149597,-0.09719272040840202,-0.012679163707167132,-0.13233502938720584,-0.006899767005202991,-0.19386587340931735,-0.0266528529106334,0.04741770913503893,-0.03528328471897867,0.36587331144400187,-0.07136018947767807,0.22279520664703006,0.010909244947507458,0.12620529080824935,0.014320682723630313,0.010415361540031289,0.009151767358977589,-0.12124815844018041,-0.029488975259190715,-0.02134747944914983,-0.050170950836743104,0.04380339796894213,-0.02571490984087699,0.021793323350072356,-0.0113615304420512,-0.0201473981935327,-0.026089006169196096,-0.13275529486127813,-0.008823510181461863,-0.18477344922102606],[2.562169108073277,-1.2647146795499433,-0.044259136240362026,-1.0220039020723684,-0.9860662134908001,-0.6659373540236861,-0.7090797105981085,-0.5726302706065325,-0.5764075978899026,-0.430181768371604,-0.3880430084154638,-0.2054092577323163,-0.13310704889511746,0.051076708153824556,0.21120552648508772,0.26874552336035656,0.44435981179864936,0.33116250988857526,0.5773152407744248,0.31100580814648027,0.45688551841492153,0.21111396765628487,0.24620353089525207,0.0485578182160374,0.02667842201070898,-0.03318124267182199,-0.05311667264302521,-0.040524162051422355,0.03864152808331117,0.05308785277022381,0.1641953324879221,0.1087876557819177,0.24775684869645434,0.10065598336536272,0.22378523207441428,0.05449300020621292,0.11739081202702334,-0.03506009349439191,0.004130308888518239,-0.08403096247423937,-0.059434147148670645,-0.09818908793999409,-0.03129409852558497,-0.031979150089974144,0.022789402836163687,-0.00899376713406906,0.06667228276685351,0.020540506839644635,0.06705013941331076],[3.109427431780658,0.06456293270826562,-0.2638077039280705,0.15880485538372338,0.13018975821517667,0.009238020608737662,0.17486769580338343,0.009658823417869916,0.23134541637060452,-0.015078747789579256,0.29437674057778657,-0.05547033812869812,0.24503353283394466,-0.07387316696912397,0.27708676456590736,-0.08124615427894948,0.2612600466518922,-0.05601422518010423,0.20132863658797964,-0.06949594326504345,0.14124654110827112,-0.07303490240111689,0.16627050325908163,-0.07154823712732233,0.1955439378653861,-0.08267484747228332,0.22335733379026504,-0.09097526373477555,0.14715983439421698,-0.05992271122721848,0.04672712463735886,-0.04374604051423341,0.03840381475762198,-0.03014339413714209,0.0583087201220204,-0.03566218026824115,0.0603007883603791,-0.024199551892684523,-0.002023422191540829,-0.01318886063935657,-0.05425451829731865,0.0060207575707154504,-0.038366020396940786,0.004252128765184038,0.006275844257001227,0.01156615775330875,-0.004097092899244064,0.005257452849391073,-0.03426303634234836],[-8.17212089204858,-0.35800626277156977,1.6937489120163605,-0.20865951707278085,1.1871760722331044,-0.13394173089085412,0.7294338612439975,-0.14560305650921565,0.6217690892837197,-0.21577858131205727,0.5223716715169865,-0.2206492903462346,0.5446759637243874,-0.23742760654581613,0.49292339564032395,-0.2758429115728633,0.289386869687135,-0.09297660033629095,0.16074618393730372,-0.09216377823675911,0.0027009952788325424,-0.1858329398960922,0.010851018495784555,-0.13626435008621807,0.09018515268948238,-0.27133531361442975,0.14256329226045578,-0.29613662944794383,0.008572843044064193,-0.15119740535841444,-0.27406622914860723,-0.13014871174976517,-0.4293727710547953,-0.06838826344389903,-0.44884284058999563,-0.060227181091092014,-0.119064756364206,-0.14730055907002562,-0.10396702862066047,-0.07290594522414695,-0.11384505751115757,0.021415853073348475,-0.26755478625931356,0.010241883630422947,-0.1804585272505498,-0.013843565210832468,-0.09630463598930816,-0.006267711238836781,-0.048263881872575305],[-3.0939944726323065,3.4056757885282556,3.3823574264440257,1.905254675551277,2.407743145704842,1.3879895213610454,1.1816410382614437,0.6594322292521482,0.5693685971315652,0.10633459873547955,-0.10239225577692149,-0.289743806699827,-0.29236488951165523,-0.3674301923960579,-0.23242880734589275,-0.1530146608228839,-0.05270481672797532,0.009283291119643376,-0.34743613997854317,-0.11315921780825298,-0.5916257912843885,-0.36443806201240614,-0.9374267363384068,-0.4705932253977213,-0.909572722373886,-0.46077633642753185,-0.4527589989164869,-0.1456369254589675,0.14849488314728165,0.13559801509975908,0.24959803122877047,0.16485458638150086,0.04179908950541494,-0.0817253355573979,-0.14540584149389787,-0.14902018507180503,-0.001538678611867843,-0.04645946441108566,0.2797300438356759,0.12959465272687604,0.43034531773541296,0.21099757428493723,0.27044635969178227,0.03242493238633797,-0.03164531702138865,-0.09286612337428668,-0.217745863229328,-0.1212378520198639,-0.09912963589704833],[6.618957561264979,0.5713143136969935,-1.3967263544210111,-0.1587657593283246,0.15057312615183938,-0.46012605354368363,-0.0291802570323726,-0.7057721996404857,-0.24103436739552217,-0.4920479477516631,-0.4226377512387442,-0.5347377475145748,-0.6850197412711275,-0.6084188860023906,-0.7928501255519604,-0.6989860039244314,-0.8328762323013049,-0.5951107961541198,-0.7685396628406924,-0.45549863313290806,-0.6618800431618131,-0.358500638662284,-0.6795165400255619,-0.21986076217595743,-0.43699111512352895,-0.03856219848324636,-0.18330005972157862,0.1095142999630686,-0.106090337396997,0.2621221662747342,0.06470938060074999,0.09719217053249683,-0.005216155839753594,0.04796963973857079,-0.1786794934794731,0.07175313520848306,0.08328775819378784,0.13505504665659818,0.2807504188609326,0.32813716306015245,0.5171821420454945,0.23744645469717918,0.3953862790179123,0.17298593350006583,0.08605167051071008,-0.008484279120258035,-0.011999098253773148,0.003315046194263227,0.09874124139108031],[6.331838362695445,1.3124623551687755,0.06919737920202007,0.8859301228754952,0.7977884151022716,0.09871287487360887,0.6259633377525176,0.6855093603827115,0.9543424277792576,0.5861846571129692,1.0530089633525983,0.9332554765359525,1.440811452211855,0.7100665201784061,1.6243179736938604,0.8729623691392953,1.7301602379083099,0.5604425997714843,1.6562349327849322,0.4019096128983786,1.6575478353718172,0.13299122256961238,1.0014318609025474,-0.10239987322519012,0.6219155841509176,-0.5783732143702213,0.41091734652484274,-0.40028543375545983,-0.10572315616186582,-0.516116607787877,-0.23781128608817362,-0.5411915012894902,-0.5674703817910566,-0.46124106927991837,-0.7219088770791297,-0.2907233425062549,-0.567433421031354,-0.19762791685749764,-0.3110764731574011,-0.04018303254264985,-0.5719024658874975,0.02245654835564388,-0.7426920025023659,-0.07433300491330232,-0.8921313376363778,0.07922445048660827,-0.4506400645659119,0.18484516475784152,0.15168266952211795],[-2.3931039117612634,-0.5647203707866938,2.9289928786977546,-0.2838931936022573,0.23054926517300225,0.35519912221516137,1.359257955365787,0.07430827909192089,1.1335953443851703,-0.01646392717752904,0.9327037113168358,-0.02308956574052678,0.6046337990901974,0.14007680013960108,0.5755644097526953,0.24203985556261826,0.7372683356913379,0.2963181936539372,0.6761190079329101,0.39318419728669207,0.2322511124522384,0.24603597866926608,-0.14697725229155117,-0.12005402202090606,-0.1302965920866806,-0.05069803640224503,0.14848033770241895,-0.014046286573073146,0.4457608077252251,0.13322273575227056,0.3373139822180948,0.13592250907119158,-0.19334424456998053,0.0044323479286767576,-0.5225578710862516,-0.14034334247375163,-0.4002092431115345,-0.09622148025542454,0.1586438361312644,0.022085531508631028,0.2992420173896815,0.08490176727211438,0.010204919794592728,0.027513398400403045,-0.31590937892000165,0.05589846121798247,-0.26734575101627067,0.003295662417446324,0.18072141434189115],[-3.073354469796089,-0.938464174683841,-0.9162192731323371,-0.7030113421805184,-0.6749490899839115,-0.7490712265941497,-0.33359629465300705,-0.46066796906895463,-0.5058685710599944,-0.3577145611924333,-0.3226331420452907,-0.2771335160761993,-0.4322579347829917,-0.2259252402735393,-0.3665358620002591,-0.26047622384922203,-0.3224904668401433,-0.29104593258143246,-0.26165112741753827,-0.17396522535882197,-0.2735582900085676,-0.1596231526975036,-0.3301101534278148,-0.09481895874808163,-0.21310425693733084,0.036410671036601526,0.08305642813419016,0.1542973119606904,0.275646961764774,0.12983611233000417,0.2676424506293464,-0.004492574616246633,-0.0286633699153686,-0.08063224942716231,-0.3905502865997227,-0.2013056283779711,-0.49872288821022304,-0.14406354505052943,-0.14799027476899695,0.02463517058024835,0.1537261228212613,0.09857258789955409,0.24958451132308287,0.08227322457400682,-0.006030860992778878,0.015440157374112185,-0.23482342409854032,-0.05342588280296394,-0.28118436862252444],[-5.038183644622958,0.7014224693590929,-0.6503059719641111,0.2660735842214931,-1.19049670114939,0.5643933762241378,-1.1577314282158233,0.580416418800613,-1.1678301630564838,0.5320683479536679,-1.1487432391181096,0.43373091324751223,-0.9552269857039847,0.3380746934543752,-1.052559968636494,0.18376358331587872,-0.5970547020412871,0.044095865531979195,-0.27594506792374834,0.1344182857632858,0.23992104237176465,0.2018693591832402,0.08170268059099803,0.23280147661219958,-0.15924396628927676,0.09551204896411247,-0.4154345603309432,0.016201763760323413,-0.22852540711853017,-0.07652151567805006,0.20666622878532862,0.041016535235453344,0.4270803626511612,0.09724021739601983,0.2868126604446521,0.07944059418285263,-0.01727045370466624,0.033056531615561165,0.02075311514035052,-0.027287713608325218,0.18347968115280128,-0.09916604689934806,0.2026995159541185,-0.023304885508343115,-0.11510655514011744,0.05628034992919018,-0.19569300997737019,0.030725674391098728,-0.05955660965586388],[0.29513439047439577,2.468669257844751,2.286886007349074,2.1412208354703393,2.0543486455179343,1.585626333556714,1.416762688250465,0.7568556417160015,0.758882977017819,0.17961882714056637,0.40280075379107927,-0.10677397354265795,0.3191531339892949,-0.03651782540018112,0.4957579114173926,0.15537420833385315,0.6912715104840459,0.312023695132101,0.8269272484722128,0.3584653247370707,0.6877782697054696,0.22330238461618226,0.604485330125357,0.1086730976791195,0.4279433894394723,0.10396646715199626,0.2739610853581356,0.049812966478654,0.10384830101171698,-0.02401714389379354,0.07974372955847568,-0.04792429879298343,0.2473197505392059,-0.06521228067367771,0.4123729591354887,0.012270916286978649,0.38177159279813877,-0.08554037351234438,0.10642434552492931,-0.16340957053719243,-0.1981170945282672,-0.27253173346902343,-0.42397590872455193,-0.3005024102079971,-0.3494072834766549,-0.2365140532516442,-0.18134270964796256,-0.1261319233727967,-0.1291037589614643],[-9.033623743148706,1.1576088179570612,1.8382534349298405,0.9703332397036608,0.994191335208709,1.0952806622694455,1.358808599630055,1.1850274520832278,1.4759125805822393,1.19493537356162,1.5975893312186675,1.2559836136408022,1.6521120879376412,1.3250122293963316,1.6192075411197744,1.2381203013850086,1.411448741391992,1.0607679354411201,1.2059225750164142,0.759932541147,0.7374521018574204,0.4718458588884888,0.5167238629686052,0.25066831513642834,0.4264855174165684,0.17409125164141587,0.25367650356171717,-0.052375271164419104,-0.18646681505215604,-0.34260354643773316,-0.559615183881665,-0.5363180805290986,-0.7961459226459471,-0.5329320809402985,-0.7313543578027114,-0.43999768198966405,-0.557779758144654,-0.4235548055221286,-0.6789464055408144,-0.4471639809902881,-0.8397334919624242,-0.4869610404521131,-0.9069819629700597,-0.4019114455695819,-0.5842028884322785,-0.20111943268819504,-0.292571589545533,-0.0486864046728102,-0.16012650831360742],[0.9364817089387568,-0.10456029954444516,0.20867704334716095,-0.46722944282284085,-0.1811620680397434,0.13865284636622496,0.34142290967227384,-0.012714741371337837,0.1521470848240901,0.034327059512657694,0.11679410377913481,0.09820596304555107,0.027403787054412536,0.12638596663434137,-0.004380043085205061,-0.021943608857375483,-0.1690365625575808,0.022056180788057596,0.31121493957059776,-0.04642905677562718,0.14873724211184544,-0.10550108008657631,0.2908185771789733,-0.0802794265068264,-0.02577918231805638,-0.18337510059631995,-0.3185931274747057,0.09943230881810142,-0.451168217544949,0.10188746076760673,-0.1714000310091189,-0.07237431287660404,0.30055032976116863,-0.01158322977508325,-0.006587887384676092,-0.046948425110608455,-0.15038386195059972,0.06140922466097634,-0.3386227700762886,0.009199660185655923,-0.2007503243542643,0.017988846860530986,0.23066755146347212,0.07234700569156738,0.17658880317608408,-0.07971148488886419,-0.07399540654524367,-0.07869974417479653,-0.2659015680508461],[-5.0177362887656916,1.8639761256078997,0.8571157664156146,1.0720259318708407,0.7907526322618108,0.13204671509769134,0.5888568644253667,0.17542279763331647,0.13839307562679268,0.18601589138079244,0.34905023185311584,0.032981413618786086,-0.021309012486056777,-0.07142256521475727,-0.2951654688049679,-0.20896136454247166,-0.6557842083174098,-0.2739798181899814,-1.0082396076293072,-0.25252382101226195,-1.251056383360285,-0.2361326352527163,-0.9911866613752695,-0.22252914069476246,-0.6235247692915675,-0.10653764158587366,-0.15608195095448524,-0.021225501207326754,0.3451022307837309,0.051536784442176804,0.7059938710080329,0.3168754176296047,0.9371322433598234,0.3004942948511413,0.7275935024417863,0.23998260513909225,0.3004470527030061,0.0221954428219519,-0.2643488234240931,-0.10006039404468872,-0.6250705060344043,-0.23980889656880905,-0.6277012065173214,-0.2382378841526911,-0.353148766373632,-0.013465109974928844,0.028738321686102252,0.005792103518071147,0.1239004946802458],[-12.146906488996157,0.20606646477573778,3.8084747192803987,0.5017896027665529,1.524615846357262,0.2368527948757264,0.9020575543428689,0.261360463413411,0.4621182730856124,0.2931907427810884,0.4934412370901662,0.1587400365318366,0.4906547584649373,0.107379976160236,0.3776141524415822,0.11281907797374824,0.36659610409789667,0.11364370127570762,0.1835741612170648,-0.04175351564115343,0.19315913406635915,-0.03591957220956648,0.5335484685095614,0.08340351217959371,0.7583185432658355,0.15625159392605997,0.6585653630395774,0.12872397297585314,0.14619952091241856,-0.054454615975240775,-0.15888627211069503,-0.019694594264025766,0.20290152512513032,-0.0056648996470559775,0.7044930076756254,0.07061134466669969,0.554651109319487,-0.04318006135915961,-0.07265268976342597,-0.14224765490406777,-0.3242108365199211,-0.10934594266701468,-0.007014450778947295,0.025150060830784478,0.4226045244333032,0.06600188864909699,0.28695577629645275,0.01011396957903431,-0.1552202231192774],[0.9880912438227061,0.31476024116526613,0.4704401333899937,0.3059603688516016,0.6369937001500068,0.23707121262550945,0.5890851917037909,0.24959871360897057,0.6216778036957725,0.17880475187271738,0.5971961705547445,0.12637140831205032,0.5559649877607584,0.05146384567791858,0.4859497042832135,-0.07499285667892587,0.4405312111080702,-0.07913228389424887,0.35105747215795824,-0.13309698956951296,0.3535924468626692,-0.12097175946805704,0.37015462787372827,-0.11285866196012911,0.252711700512746,-0.08092549209970029,0.07135093288071989,-0.038546283062816544,-0.16649179130468603,-0.06541343072536737,-0.2801849989113625,-0.08006911585428364,-0.236911858181934,-0.04686623819874562,-0.22870545484807808,0.03215742248838096,-0.15193395439358467,0.08600271918400466,-0.17049391916356593,0.08223541220944695,-0.17210060946743336,0.09572996794679313,-0.16373027996301912,0.03869244528652585,-0.0665913593258654,0.051699400483523006,0.052506510301582314,0.028323697057222777,0.08282797011153437],[5.290978097071165,-0.13370863412055306,-0.4508386954689224,-0.11879325760272123,-0.018447808228776764,-0.03169301827320348,0.02533501653635983,0.02043969720438052,0.06099339124229356,0.007029457532931367,0.04699232348663134,0.02336256243206697,0.048291187479297545,0.03134379349202126,0.06281509883435217,0.03508826401697185,0.05462239480423396,0.03609347891794145,0.011909780580155056,-0.01091107456840491,-0.05102834794412099,-0.026623816104136284,-0.04767293094506515,-0.02194341073805996,-0.001519081648815881,0.02076739592672966,0.03353486933551495,0.03562128256810223,0.024167631122122706,0.004123091079100986,-0.01566146719950043,-0.015548224563458269,-0.04777380001971204,-0.012066338960964497,-0.033731011518418265,0.004603842031764794,-0.003578743719215138,0.01982550425560959,-0.011255738596468976,-0.004615910220634104,-0.03388883145442873,-0.005172902028070703,-0.04021240884974202,6.404054609163677e-05,-0.010333990564651214,0.008925051350213828,0.022546867108293976,0.014243702160830147,0.01611501367379805],[-7.4048856818281985,-0.7751569045672276,2.3763532927260362,-0.24382393569693253,-0.023305679435267006,0.4719511347867005,1.0669125510783826,0.3038105863070598,0.5316857544229323,0.17983360952856922,0.1392065900176712,0.09217836305654428,0.15875297637941196,0.284107737237559,0.4809787846356456,0.24845780033693915,0.541246487379538,0.3313918826532219,0.3488575200243709,0.34911400088989025,-0.01660298783592606,0.2406180717504694,0.1112165671396175,-0.10477763871604545,0.45971623302618186,-0.059594442867385966,0.7103483444781166,-0.08604067642311088,0.4395790573921704,0.06565415425631187,-0.0006646662267914167,0.05556071208715435,-0.3133526270691263,-0.009334935671123035,-0.3577730592753955,-0.00638165071724191,-0.03480364964674598,0.014899875770739816,0.39378824793839134,0.08686966859742128,0.47729596805371105,0.11989800966968238,0.0994169810273195,0.008045941679055852,-0.1856868140440059,0.11705194296267882,-0.18341009270271838,-0.0056944059518299495,0.17448463656358257],[-0.5627239142273168,0.3661679620174989,-0.6430180182210944,0.04941498350912671,-0.8443701526044278,-0.16601874175667847,-1.0988863561472413,-0.364465547489279,-1.1009736779287496,-0.3998607448382869,-1.0356179021618614,-0.2858184032125155,-0.9661050913308398,-0.2148541886275435,-0.8690518573379569,-0.16497311963376504,-0.7700954875188214,-0.09832288594730594,-0.592630748717043,-0.05707125613295897,-0.48593119123903966,-0.03516720898887197,-0.44866712377027357,-0.07770647764732484,-0.5745741404318149,-0.08232743608241093,-0.4414373196408471,0.03818304539177252,-0.09572173875175331,0.2121127706299812,0.249443573724115,0.3230751797667252,0.4087181814253106,0.2760287001989123,0.3535069607851969,0.23352645224989374,0.2817123872207811,0.2732680227079472,0.464930006661007,0.299638845272986,0.7190972509294247,0.3251816976767265,0.7371997455178264,0.23036940905571834,0.5081137577707143,0.11961739322172324,0.27897504427607833,0.011398548067887216,0.1581519367967443],[2.2482838542743013,-0.2635655249512296,-2.6254786450937293,-0.2549212524708584,-1.2728917274114913,-0.4366898309340775,-1.518728026643755,-0.11275143102400925,-1.2883774216138244,0.03915558241303128,-0.9581545318838871,0.13174447209427947,-0.6612760348402539,0.11595818969249376,-0.36677336003589467,0.057646693488951524,-0.3898043488857,0.009625743587071984,-0.5444292466850374,-0.06818093758931118,-0.47380898753558837,-0.023742169543236193,-0.16748943729205357,0.08295522802126827,0.03961858961757021,0.09493959804053284,0.11018811738828183,0.18516721073849057,-0.07181641405989503,0.017319033455964004,-0.0617397678460627,0.04488805806461237,0.1728875127985051,0.03660105817224755,0.34652573467394576,0.05145891850679108,0.2442326930081898,0.076390864411308,-0.00765386811947895,0.06269624866105465,-0.016875571656493973,0.045777788142383764,0.2649034212282895,0.09263640255648892,0.3303000246690061,0.05431046180084634,0.24106435296236278,0.007358702398771092,-0.03154316098504821],[-0.036495502332249546,-0.6584170294617518,0.27721368283253595,-0.4847505208532948,0.30032771319897883,-0.27666941943838186,0.19074121943702774,-0.41262316056780796,0.11096274124984218,-0.405225131369347,-0.14650950905271928,-0.30604345289887724,-0.043980842476338836,-0.2675911382129063,-0.13956639523031023,-0.16750794865969693,-0.07211597395813339,-0.06332633678577887,0.13732775630289873,-0.01714540709581647,0.24485300955619177,0.05537278733180951,0.18396991074550606,-0.0062445092624529306,0.04155482883042777,-0.12027290414915015,-0.29818559570838454,-0.18514700527545194,-0.3948414357643014,-0.178336713597223,-0.2120261078858833,-0.03698377123457093,0.00015177278880201874,-0.03340572530501323,-0.03295275083452661,-0.07127557654907322,-0.19866912883735907,-0.1529480716567595,-0.48375553135394794,-0.2377241815314017,-0.34927945303498814,-0.103789484494555,-0.09666537214170318,0.0566104098173874,0.2345544951632706,0.15664171762671147,0.18387210133953852,0.10897227323354708,0.04379120118551034],[1.0039649110825024,0.08232620163812653,-0.05262738025994125,-0.6067478981269225,-1.2604639004680749,-0.21678571443811465,-1.041592887919006,-0.30230006325484005,-0.6065217290712097,-0.202597761698136,-0.6208609729809456,-0.23334356641878098,-0.8939027753219081,-0.2029289074705734,-1.1363899067546013,-0.3756679587253936,-1.2548319169813293,-0.35184152325150847,-1.104359939001351,-0.2869613538867316,-0.6712493062571462,0.12790996384067418,-0.013999906907067376,0.32308094672150844,0.26240960355167486,0.4669758732030734,0.19584539163128878,0.2085743601476363,0.04352219858589879,-0.07344994951364114,-0.19625623923995572,-0.04467161193337902,-0.07312143164729272,0.1792159295681323,0.31840105698771404,0.3269564551526366,0.5608342649595673,0.38734296066566637,0.47827449733616734,0.07629843845213385,0.004957765235047817,-0.19205486791072676,-0.36838419065434463,-0.16577709425028123,-0.277868875657999,-0.062490698642716905,0.17728711343070816,0.1015710453182751,0.43538162343858955],[3.717421614130333,-0.21682594446372694,2.502314067785621,0.28845346800831495,2.4686071315575115,-0.15779358789465686,2.4965901212200614,-0.14539760561422807,2.2998601616394,-0.28932065253490946,2.3124208301399927,-0.2866733833901119,2.293626977324813,-0.32330762550298375,2.3433988257821836,-0.2470971436418347,2.2918474900272914,-0.3477077871495827,2.1271594625292782,-0.4024982230672065,1.780511821673939,-0.6290773163174117,1.1348264543329987,-0.6259541656197275,0.5210177229756972,-0.7120156314244017,0.11888403061791383,-0.38498887025938594,-0.41754686189081136,-0.3392680133673132,-0.4310773947248734,-0.2524514848021433,-0.43551869021624845,-0.130721352081016,-0.6218585200494084,-0.055536740664327545,-0.7251980130510157,0.026715288412077348,-0.9852003488867161,0.1565896417902687,-0.995524014737221,0.25559625314902146,-0.6794043278987817,0.2178395738169746,-0.3692031344776807,0.2182545839193672,0.07480674226000243,0.1873139780122951,0.023451404945503],[-1.407436471956671,0.3987350042485741,1.5999249787511822,0.2498672133992554,0.701469147175045,0.27839585111637355,0.8150503618344345,0.18353066338254986,0.7051436732959696,0.2369347964905667,0.7082250241848028,0.1913371021615951,0.4888136087210882,0.12626382892548219,0.4207221688087565,0.04585190549216938,0.42796980812184116,0.13489014051378043,0.6235185506171972,0.2513483256547533,0.5571463620701587,0.3632059270925084,0.2543298391173204,0.14995483673845753,-0.03283658758103773,-0.011530101778882978,-0.07515996879654491,-0.030386693979298402,0.3248381498364193,0.17227189941010487,0.5146570869281211,0.23002198140407681,0.18615348376421306,0.09288331292267439,-0.23058507688968014,-0.11622977168504865,-0.3060729766737361,-0.16425048460405622,0.013092712603566106,0.046549323387454565,0.2792607665828363,0.11703375810658685,0.06535720206231432,0.043654808674235566,-0.19263753193871871,-0.11477022453947168,-0.2148982074905912,-0.06395792577252356,-0.010060009833385113],[-0.7957320211950681,-0.2937628548821259,0.2832638374675605,-0.13199627666369426,0.13412441637977454,-0.18299881720595154,0.19452064758743653,-0.004686191391850163,0.22055128223451118,0.04184585611588728,0.22266681422097964,-0.009813209774363554,0.15993112055285405,-0.05803699774254454,0.092104121310221,-0.044120458641016304,0.06761592643347743,0.025060651888784955,0.16296670094645554,0.12096366827858501,0.16640810444160964,0.2558408081436302,0.09123561278788711,0.15523337903207515,0.06955893656575463,0.0326879813244794,-0.035744130167851086,-0.06927213895336828,-0.040307211439810565,-0.06340766595777506,-0.1379469888411585,-0.005178641803354971,-0.24990935255400396,-0.02551442684373558,-0.3251392956115648,-0.031112489175919706,-0.23120116347389996,-0.08804701897538514,-0.10028439522524277,-0.02530696600382766,0.012940704861757685,-0.01918088389243091,-0.012553555440631095,-0.007950880259225475,-0.028117066870785582,0.002879426354137667,-0.0223249941626706,0.019564971036494197,-0.022477864705868864],[0.34933880561033526,-0.7617354873274886,-0.8933275769937734,-0.29622393051349694,0.0669582143132397,-0.232370920272818,0.00864855567471145,-0.14911552995268523,0.10207001438843133,-0.0524636875228062,0.11420490123035114,0.1607931178751574,0.3483950521335379,0.3429258187930642,0.48105667498049004,0.4664708220087143,0.48807082274082825,0.42517711877006914,0.4282441213539063,0.18182653662941772,0.21584500880384802,0.06914240853941182,0.14213603012865572,0.01918689473996721,0.2122474174140717,0.12795042806572698,0.27151156481367866,0.22263931074330115,0.2810070106888217,0.20989906249285095,0.17161419588252977,0.06322579451318479,0.06007785448611165,-0.04211983917180043,0.06296019781278066,-0.015889505884521062,0.12281904530080467,-0.006065101469821832,0.1718583115618251,0.061190880450324114,0.24330362877910985,0.07062087963072877,0.2760615983064366,0.08527674152014598,0.18775658215047383,-0.017972235688697286,0.0931870609397223,-0.017772899117675177,-0.033929364251432274],[-2.0271266871742855,-0.5782769648399947,0.4859828539502008,-0.6892744529390018,0.04300093897669933,-0.4262039043270316,-0.0013126564882980894,-0.46323981353613336,-0.02216027828461284,-0.37206660831333105,-0.10008157895276121,-0.34918078421816096,-0.25416602573861163,-0.21953236785719216,-0.31962550985035143,-0.19877435826628184,-0.3545185517698935,-0.07063941967938639,-0.34235619418826135,-0.02280474248618481,-0.4711598698181876,0.04358180211928387,-0.4663871732166459,0.031934554513736245,-0.5489113303780117,0.0903053722042196,-0.4539477298543565,0.09142015497808752,-0.3028574346191152,0.13430901447609822,-0.17604830473727212,0.1609578451825604,-0.09607389315877339,0.1682414569139886,-0.03272435649889988,0.1579574485725242,0.04915790587702143,0.10456912669319607,0.14980786721876613,0.07516556540513479,0.1656458779411365,0.045579576000113975,0.14583674230470556,0.04177448657498932,0.18619137944810515,0.01169364313494052,0.20962224733225812,-0.017994359480552287,0.20682230041935837],[1.977165886046829,0.11434192597725878,-0.12611002779899907,-0.07070098713093377,0.48067676772011064,-0.11926070863950393,0.3190559241959724,-0.07330918292641896,0.24560922632778137,-0.048664800479628276,0.13610906766914202,-0.07887900319425406,0.056406241361119734,-0.09729182641713627,-0.033425241087765144,-0.1346909955452852,-0.07732883946208625,-0.1151182249322565,-0.10078168863712877,-0.10130999399144226,-0.10421565448511388,-0.057001290865932225,-0.09413645973336975,-0.05970408087630893,-0.11962965231013813,-0.018418421826484004,-0.14971456496604987,-0.027530787453305942,-0.14075256770145464,-0.014578218886905362,-0.13232382483226945,0.018755885126319056,-0.1105694863836791,0.02627852850519466,-0.09661107536602061,0.07692158930372162,-0.03742853178328257,0.09611179360462696,0.045453432769265924,0.10235446069836299,0.12102412142697798,0.09629098250132644,0.09770212999378401,0.06333511382769302,0.06979698350219903,0.03561825804624757,0.056762337566801366,0.027162790246768993,0.08015186945388955],[2.0093150504127686,-0.3226175407324691,0.5710227713595615,-0.16295481008718746,0.8503142276697437,0.0851441362912858,0.8862126150009656,0.2464081101563199,0.8584634367315245,0.21039629135125842,0.73112918121206,0.12117371082160693,0.5192359691373417,0.08044427608747717,0.3924878571810678,0.04200578314885947,0.32896588738912785,0.011710772166285853,0.10834277826864137,-0.20225284717405492,-0.13322166257086343,-0.32511255611095513,-0.31465181362115735,-0.31153567989963477,-0.22990665394669563,-0.12139040874803256,-0.08809264159379289,-0.01144078127070387,-0.08234626130337894,-0.033674728162648826,-0.12534189148564598,-0.14991771638051743,-0.15446518249287422,-0.06228060644106354,-0.012377702604764009,0.08925937811148149,0.12478790667420703,0.12093007352745899,0.07927882775605036,0.02687562339697721,-0.04970550750882664,-0.024842477777471863,-0.11037461661722901,0.007007830917840619,0.07031830094739125,0.059627348893607025,0.16111671654578044,0.08644627421971389,0.10014438286027312],[2.3984002474543096,-0.27871449824638156,1.0324499741176167,-0.03502507435652899,0.0815506556999244,-0.13853337572156768,0.18536904550637406,0.01231111929277349,0.22511886469418005,0.13861619928724128,0.05988971573624025,0.08452876252326591,-0.13685112103793332,-0.01654244033670697,-0.34985045396322106,-0.11105535385432687,-0.6053550302281777,-0.16917111227108925,-0.19694800984272476,-0.15654968142508827,0.01740073421190483,0.04019721093242647,0.07169439412947257,0.1368636135965615,0.01974139062733758,0.0638495036711379,-0.1681697181003668,0.058204010935269626,-0.11832718751247626,0.024166730780971958,-0.047790099465862566,-0.022797426792520475,-0.181967068814081,-0.062192508779744034,-0.27847059504848126,-0.0344858721740392,-0.24923662993076443,-0.011029160115343354,0.028229070938945278,0.09519611983638777,0.1807946850644019,0.027504470501863342,0.18470798243323752,0.02328276580746923,-0.021737995893229552,-0.07375461414979517,-0.07991886312971033,-0.02057718771350743,-0.04934165525996693],[1.883345840784537,1.463998936674183,1.2723915196748772,0.1911312295059393,1.8914054283768353,-0.1554768421664617,1.5215536262002305,0.10726684330798482,1.5406183093556125,0.2064190458054399,1.4827215351651797,0.14336185517972863,1.3392953747362093,-0.03469796949324762,1.2268116888919387,-0.02266475325314171,1.2406083546006748,0.04035502804694899,0.7421401618034201,0.10916851166968683,0.6086398889259771,0.04486091674691418,0.13896951244484546,-0.03150759278522296,-0.16734018349670585,-0.24537359219894805,-0.1262822661561051,-0.17817956445222868,0.1279519359375165,-0.11280286031530007,0.022591483869930498,0.04834334383818697,-0.1702529335010325,-0.08353480234674682,-0.4719452791546319,-0.07874878361049376,-0.38981850355525083,-0.07309282328181921,0.03840977275112279,0.058238250437568855,0.13310931013618604,0.10767926733616622,0.00976334443707784,-0.02725175985636861,-0.3214860142685954,-0.055081189056462906,-0.2972692660526821,-0.05672383460092006,-0.10008177331085118],[0.8616515112999401,1.2877902974208228,2.0232940143539446,0.5054062763412674,0.9009205521964792,0.4589105122510436,0.9853677493848028,0.39248790665431504,1.295292352649434,0.38628429262201586,1.67590430640291,0.6390298262902478,1.8787405765233782,0.8260135719730389,2.3205275878138654,1.1395717996028545,2.3554526350693457,1.1778649489971578,1.841209331150951,0.7847285023015571,0.8551206412507284,0.11104798413016652,0.08053990258501359,-0.34582429408258186,-0.18396827713511305,-0.29869854617216385,0.4246663626592256,0.19903812839083537,1.17402727360012,0.5697082135972391,1.3127171609085242,0.4979548374535815,0.5182605106761786,-0.07058983351971919,-0.4803652174973165,-0.6453253862406352,-0.9853637366109343,-0.6611190376850004,-0.6159935671717989,-0.3375894911316305,0.09791709495116291,0.12579853522604836,0.35305774865846573,0.19928922140312014,-0.011619854648114217,-0.11403523503841313,-0.5563399609466039,-0.32656338352339004,-0.6826233195705456],[0.40657812333959836,-0.14170376490372316,0.1099056244772555,0.03446335939927906,0.05224116374226597,-0.020690474599147815,0.0784995995141287,0.013288315252776153,0.059396926095189394,-0.0016675111141844434,0.016540914525349365,-0.045788452136706244,-0.025241792633132674,-0.054143977667784865,-0.03206377294006373,-0.0963263234616553,-0.041795028775223474,-0.022497367365017104,-0.01978077041671496,-0.04175107000573701,0.06088166836725334,0.050121576261522485,0.09330013607852626,0.0047920372212111005,0.0911775788096659,0.007464709426982004,0.007182555043476337,-0.002635903566871729,-0.053746326061877306,0.03547145160338631,-0.061589443705135735,0.045931794098018716,-0.024177341120464914,0.03452727297314807,0.04895810388701377,0.0039390109937346126,0.06378536707773182,0.015976314395425344,0.05208713593184534,0.018873002619747416,0.03642367348078738,0.03131159355772475,0.04296184244601808,0.03099521871761807,0.05618409046771569,-0.0030569562505591568,0.05744924091450589,-0.018298862665773404,0.02621558827175486],[0.7741981628456892,-1.2105334316124792,1.2408780127992618,-0.4209573598601317,0.603474408610927,0.3106131884812361,1.0072908475845146,0.3343757171259622,0.7987372057152361,0.09005186147853138,0.8465986367415447,-0.003938998126558492,0.7934894818797821,-0.09361051664289037,0.9295719633505977,-0.22116865290134652,0.29037167339696984,-0.4457307492647532,0.21126139666036384,-0.48483986282619407,-0.5776262019061678,-0.22117137688713331,-0.5343221249891074,-0.39768582964704846,-0.3482370530738152,-0.08410015839687127,0.0024226524611956697,0.026201857271384438,0.5392871843808306,0.07203015094061105,0.4576269760187981,0.020575638388670846,0.091417072662194,-0.2004378767140103,-0.5490866060852958,-0.252445312097571,-0.7594492503355577,-0.25929641019855376,-0.5649326182846468,-0.020337808608625926,0.12468598933959707,0.08541915744536882,0.35511097582676254,0.19809402418705577,0.4377299709579319,0.1916431188402557,0.14439907997181528,-0.0809866404555228,-0.5117724320661663],[-0.805638874392002,-0.09168705293188115,0.2164982833937969,-0.23171667415681038,0.1523213843891035,-0.16495125198664812,0.13904053477822528,-0.08739895120014841,0.11033914466128224,-0.012796428298637957,0.02054730962510772,0.046469901364136326,0.18678191091081955,0.15878204016833797,0.4166766382540597,0.34949780612406006,0.6761704337429484,0.45669598162655034,0.7503732107881643,0.4860835124410262,0.7123400239016074,0.34150242587315455,0.6648935495013605,0.2566287411204103,0.6792081839441376,0.1856055263726515,0.7095647626401482,0.10656579077352751,0.6779180395799028,0.09072113580986413,0.47113600863480715,0.08309096984019686,0.4616618553102475,0.11837690848735065,0.6314902144500893,0.15976417927888611,0.7586759198927665,0.05670789649033869,0.6840796145405066,-0.07910560853491477,0.32196881766412155,-0.13262904205565765,0.08060155627219398,-0.020260787758029888,0.17990319312264494,0.0050868131593359905,0.286334148401486,-0.04508960609251837,0.23966471137823947],[1.894371339022365,1.270746684351533,-3.601022937985514,0.9539295820713279,-3.292080377694604,0.886447323308477,-2.9353037520393945,0.6765663798384748,-2.779082921706174,0.6354344868148248,-2.4986807538644618,0.49374197845220513,-2.295317393306382,0.5910358058635629,-2.0313153856288095,0.4775246117433556,-1.9084444008312489,0.5136238772557212,-1.8462368032994052,0.5509457524075182,-1.512847397164762,0.7532835857750045,-1.1885936434092645,0.7495605279201015,-0.8095351216154806,0.6526417773893675,-0.14291716528271375,0.6114214016917376,0.5799355830159625,0.5619830239952366,0.8523935724498495,0.544924284344136,0.5566413050310018,0.39552202487504545,0.5334335175305137,0.25205252905816955,0.7202331790236374,0.2200610476011823,1.2252224503720666,0.2635846288765432,1.3040549814824474,0.08433028593585593,1.0985222652016176,0.054490024585134134,0.5103899282999561,-0.03942499947434918,0.3696923976560588,-0.07807053070125462,0.5659774130624112],[9.393493888048788,0.4991644710428571,1.1210577846856504,1.217568098500413,2.3352852006913776,1.0095569709762053,2.284883734085384,1.2649576191548193,2.628148906057803,1.0395543391806294,2.0273992312742912,0.8442064228674416,1.4844664864913453,0.261675170589031,0.7590380998189902,-0.14605983036253098,-0.09737272933232512,-0.5662996132724006,-0.6837598524554892,-0.6553397421025676,-0.7435849043316194,-0.5820392447553651,-0.30006627756670556,-0.2971975303607152,-0.04807992526973604,-0.21559588336557445,0.13865847255300962,-0.18435384724047715,-0.055570453246267984,-0.2013834246518609,-0.30192261514070307,-0.12119705288304415,-0.488487375091747,-0.16473434753251137,-0.19450008490116047,0.12886865890137159,0.2861333450163283,0.32949705840437443,0.5028471539774352,0.2902371473387342,0.2731958135428493,0.12538489105594514,-0.029470349603303046,-0.11889871516933449,-0.28808950751691853,-0.10700308550703187,-0.06821230390714053,-0.1555018103352199,0.09601970570275113],[-0.40483355367270146,0.013718352069332328,-0.8764153545362621,-0.1516192924191939,-0.2010912243918595,-0.12856977377092468,-0.42800822489610146,-0.2605698832961331,-0.20484381373407268,-0.21004327108950677,-0.1623965420300059,-0.11718916683426892,-0.26816098121007964,-0.14567760918832368,-0.32366540432556734,-0.16374412675778277,-0.27700899274767044,-0.09963326289044412,-0.2759759961877594,-0.05666050073196602,-0.16411098583175066,0.007716538751998494,-0.13911052892423376,0.05548133583476223,-0.15321629516084317,0.09164280164583335,-0.07939853249040395,0.050769599204304176,-0.027008022977575347,0.09144926933689763,0.05640155738110866,0.06546085627061846,0.04394722807329541,0.09820725073463024,0.007244549877992078,0.05681153260229881,0.05356683951360375,0.032182865887540366,0.06944854602092591,0.013318754954059575,0.07110683668389436,0.0324263491321025,0.110557384258175,0.016872260614352916,0.07599131108415522,0.013475443736299606,0.05997366990453439,0.044252769068719534,0.03715461926657957],[5.156809130384566,0.7427445388427024,-0.9286212309604082,0.40611354678824885,-0.16108180110219375,0.2344021977530748,-0.17962743903580314,0.21472773430823458,-0.012535925227148007,0.26569461867756794,0.11211459664179269,0.3526167324125675,0.28667861269527456,0.3774612561707971,0.41992031321363993,0.4360679323253921,0.4970549240653296,0.3666936031107714,0.5027392668954221,0.2806291242575549,0.4159519499182438,0.15875511971139988,0.3227734638861986,0.11998745387891095,0.2793159465917372,0.15882237682592135,0.35646902682919746,0.22602137973356123,0.42969711745847394,0.2099423055670878,0.2755054724331444,0.016302125807181194,-0.049940843879106474,-0.20448004671427883,-0.3190716642994394,-0.2907474041821822,-0.2945438408459959,-0.1610158054534195,-0.04560104777198865,0.017712146290735836,0.117953460722002,0.05132907100250581,-0.012113127433010044,-0.10716456682678335,-0.26386858834676585,-0.2330132523025073,-0.3325061600939744,-0.16667529093392727,-0.18686192185825223],[-17.590941995216,0.1739674924511832,-1.2842783967646965,-1.5227855902424008,-2.034764643097643,-1.37641958538284,-2.7574405577562473,-1.0455535358699604,-2.767238574225877,-1.105640432719262,-2.7888956465891828,-0.6681629842030621,-2.350433003309442,-0.32528547747824293,-2.1820710569709454,-0.31736237409462226,-1.9174398642766803,-0.17809767526116654,-1.5367945348385303,-0.12795269669663803,-1.4806107915371212,-0.23743057444210552,-1.25235142442771,-0.0826057522828205,-1.0660796764599827,0.18181411877265904,-1.0465452364575591,0.06117171543338468,-0.9232562880335917,-0.017851185898803537,-0.30171973477998654,0.06490178921201979,-0.06258716992539692,0.07594360503258599,0.05290905917097267,0.16598160493645467,0.011003635228404952,-0.05179882748696865,-0.20186876530910133,-0.07874706484262362,-0.039391783036182605,-0.06960654302911243,0.08882989210149589,-0.07681056252750573,0.35026872897300787,0.00156915949705104,0.1329287687026013,-0.07476004718562763,-0.2524674808285052],[-1.2418921564768814,0.3381186236999305,0.4219591990783181,0.17995240426804196,-0.3878339196697462,0.6209825540700009,-0.5254740022565132,-0.002214292452499946,-0.3644464022782679,-0.1572243873731219,-0.29503355256754227,-0.14129868976491422,0.21641047201943045,-0.07846415230473422,0.22539635355099466,0.022777971187256617,0.21529237192088083,0.30071043508356476,0.435259034481312,0.14418100586838242,0.15819790472644157,0.14478801137332034,0.27902924856642547,0.19114764886456567,0.5166933020261311,0.021179243062254275,0.3885908350051911,-0.01656308218126762,-0.24562068367482778,0.05184978874667782,-0.37068611666249074,-0.14219387187837057,-0.24469415580415826,-0.06130227442458462,-0.009149972873076508,0.012823613728796162,0.34447906357852476,0.010562947732383317,0.011023853668573867,-0.012463789062310035,-0.24998366660920918,-0.026989402587852332,-0.3157386986850868,0.018191420188541634,-0.10001133925504505,0.06701654588186311,0.08140139630395837,0.03765668032890309,0.21637519009197406],[-2.784902398998872,2.596816838912709,-0.6485011395202307,2.3771869937548047,2.3290897911769903,1.4943089733938923,1.614109714506474,1.123953278094131,1.0582173753025184,0.7453592154909374,0.4968158787981295,0.16015038656992125,-0.18100172488026067,-0.359722436017793,-0.9021295317349932,-0.7918161694196384,-1.2980282404062995,-0.7594380251376689,-1.4550035056746387,-0.6206257701502437,-1.117413207265573,-0.35754832152501054,-0.6087785794558735,0.005262359873429602,-0.18020887043366515,0.1062185156832647,-0.04114856157782231,0.05926993303676817,-0.15854897370279383,-0.0977832641461731,-0.30144603309977475,-0.1444801506501467,-0.35144323770974895,-0.09738682585500416,-0.264784303142151,-0.05911483029410022,-0.19181691961182093,0.08599529921445961,-0.025176991207054182,0.14723451717142,0.1469995581012039,0.22965674013483078,0.2307758262194514,0.11941388906676434,0.2093855863709136,0.08120409769525166,0.040042121157292616,-0.023121266602048187,-0.09825957380319524],[-0.1276968891386559,0.32439205772892354,-0.500801575161132,0.18399218780517645,-0.28680653646335147,0.12854064593620898,-0.18796950228077006,0.13157644558957488,-0.17716239348793225,0.14831900119245192,-0.12329094294902895,0.19348710833075142,-0.039360102248972934,0.19536461027377278,0.01899689970212669,0.17552688247073303,0.042659855162953335,0.10991828123486644,0.011788343701759985,0.07078054375052163,0.015916578413414748,0.054616642293935924,-0.04937420250503265,0.06895463824576573,-0.1209811460488288,0.10703585651939666,-0.07909236486787473,0.1018935370501471,0.07604945252553852,0.047023263532353046,0.1304266959055732,-0.009573891020701008,0.028975790859166628,-0.014625914491508416,-0.07016954626128444,0.021337182567550052,-0.06983784162198584,0.010707565400443639,0.06291760518992959,-0.011695011454684351,0.11973201322994305,-0.03070167463476973,0.05809411128027624,-0.009745741106474753,-0.02537374579307991,-0.014116727000510712,-0.023449739630198717,-0.01541902402020896,0.03747914320355098],[1.5192876558119919,0.45177968251509126,-1.4215917066669865,0.6284089143243563,-1.3102878243569294,0.5686436292884965,-0.8775711921536912,0.4646136818854852,-0.6725131162657395,0.3791054674460536,-0.512686945125711,0.3111331613861933,-0.46893855735923756,0.27843887443109694,-0.41182747051577945,0.271502410757284,-0.37448246754484227,0.2375362349832629,-0.2871013337168801,0.17593657790028672,-0.21381366035243043,0.1724101903161469,-0.07268958110902023,0.13570790559596738,0.023489106878414157,0.14403709866119646,0.09264676220996892,0.13935553518144544,0.05783984455864782,0.08515044668266918,-0.014937084815970473,0.02766907653474236,-0.05272594515072611,-0.0020216927539932375,-0.031169841828193857,-0.002815069652102655,-0.004076341117457264,0.028645206995538384,0.05881501917971015,0.015999252669477295,0.04243760907418826,-0.008879447810454437,0.00355863053984552,-0.02055364042635192,-0.05624663239509162,-0.006389944595100927,-0.0408607770358829,-0.015825564698799883,0.022345842810136723],[-1.8364992665800788,-2.975070397374651,-3.4217495340863118,-2.2749249084100267,-2.823169000676472,-2.1457635237619304,-2.42698804415199,-1.879473848477202,-2.1233870284769303,-1.4197008663431956,-1.6107341667126613,-0.8954621399390699,-1.1057710546392379,-0.4560754000400823,-0.8496549710106421,-0.2238493512439536,-0.5892121267001794,-0.11918049201409628,-0.2916831156025232,0.005693765871356664,0.06353763343398293,0.2593261629663076,0.4658474877949528,0.501129523709528,0.737246207376445,0.5622237523267589,0.5003949122210429,0.35615590374874556,-0.01334887377689745,-0.04312708263707328,-0.278511202478483,-0.18412196719665969,-0.07510316160797029,-0.035527425098978736,0.2342254016085703,0.05488477421239789,0.12844118672618923,-0.08238434087392706,-0.3444546166868507,-0.3593542867253225,-0.6934349018971712,-0.4476440489597211,-0.5400923229339873,-0.19820103011707765,-0.06214079237222571,0.08752912231682981,0.2798183031852619,0.19780415935782314,0.15086678488056254],[-8.586979683891935,-0.06375711247201608,0.591476362443328,-0.07021789229098743,-0.22176321487110928,-0.3247645263982293,-0.09099804594496509,-0.09034745314590674,-0.23527234040233155,-0.13825880542611452,-0.1338059079943855,-0.16627161687963546,-0.07996909097767606,-0.19745039969038664,-0.1539352720714756,-0.0679725203694411,-0.3913276394442288,-0.23624340690655052,-0.3242727554558817,-0.1636821478509844,-0.2734321455103154,-0.08945341003387298,0.007140403193584937,0.08214662786982169,-0.12380110685565682,-0.04410110427983779,-0.36151120339591786,-0.023251740136625728,-0.3238671236306469,-0.08202078055098595,-0.24429264763384884,0.031478155820161856,0.048054047472643226,0.10308148475476538,0.25289119428311174,0.2235414275387213,-0.018231748016442742,0.12694139419587258,-0.08102630160516293,0.0059166363453190445,-0.050941873460278654,0.010450988480988235,0.11205061155268445,0.01539145512060434,0.18014446369712941,0.03950359009684664,0.014794331622033443,-0.030474994808957056,-0.14872744195514812],[-1.7756770572050737,0.1389705698243403,-3.013792532143839,-1.107225607577163,-1.8181809379677234,-0.7013184246870301,-1.3981165764627137,-0.4871736987796938,-0.7535665953217883,0.014256484331003044,-0.22688396776679942,0.23828837691922763,0.06870659391607907,0.3636331427631048,0.14106348255571868,0.32951366751194566,0.07643078473178817,0.3492884831402664,0.2594589046879012,0.3516819819096239,0.23036403758124577,0.521791231098507,0.5714419748268577,0.6536830905776037,0.8593393508962713,0.909352055577382,1.0630776507308948,0.802595201023545,1.3430159737397915,0.755123923055283,1.0117186843767718,0.336351527643341,0.24897490792591917,-0.11747779914872478,-0.6296767278247105,-0.5107418426839241,-0.9846949323293533,-0.4340884936218726,-0.6560854229691636,-0.19853928623520742,0.19497889427577583,0.12962368679605824,0.5523563132131675,0.1293988943366717,0.4325746273551369,-0.07980951989246474,-0.21539974170965456,-0.17012141695064908,-0.6850556767348867],[1.599075682288932,0.27402513874693324,-0.7380873963891842,0.0851485790002519,-0.6865559246581667,0.050433280283407114,-0.6595370626782446,0.04472267767472492,-0.663247555720266,0.06374573654093363,-0.5776546758671407,-0.03991795992442142,-0.6964230803955216,-0.12498750439243506,-0.6676936613409203,-0.15062442378360366,-0.622443211302902,-0.09304481937628829,-0.5230852348095986,-0.03173850556238863,-0.4811487180259545,-0.056135653935736204,-0.499178651654138,-0.07808572468615298,-0.5116927779243178,-0.09698682133844051,-0.41459748597621615,-0.04344924430606735,-0.2803071267004854,-0.00661659728536068,-0.16307700348267004,0.017391666528799664,-0.19173897729418632,0.011839971181904082,-0.26807770366068473,-0.05088371633347639,-0.3038163532561451,-0.029912719193941326,-0.19024392653793976,-0.0016488476376473008,-0.10066600398824056,0.041851850970557075,-0.11754941973628627,0.016603570141161244,-0.13922100544120056,0.012837367016538882,-0.14660351475497704,-0.014698694894891024,-0.08129336836726833],[1.8230910019418014,-0.036702628789277325,-0.39961505069933295,-0.014550584994371988,-0.2203617433451295,-0.007787194548754209,-0.20186401972730325,-0.03025994867741824,-0.1448774623345571,-0.03916787188918696,-0.13543434323835188,-0.01942598723067544,-0.11147009157589724,-0.019454787711924326,-0.12104029915225958,-0.006069294952405444,-0.1360623669110527,-0.013286185198328327,-0.07409796878725011,-0.013951135930260648,-0.030100545507482125,0.0011525134420733094,-0.016161799691794234,0.010133736734411269,-0.005821772568700221,0.0038371480508552986,-0.047213189151768094,-0.014296567017330761,-0.03725056461626238,-0.019797239004791172,0.001559435830054462,-0.006869872431707335,-0.0003345534086788221,-0.0075183732265264565,-0.0009316835515405522,0.0045675639376974156,-0.00873354936203343,-0.02063485132758907,-0.059119033394686715,-0.0226590799232441,-0.039091815189100286,-0.027724194458739176,-0.026252199014786655,-0.01637605306234612,-0.006836109447008746,-0.0013552793100677408,-0.0015783183172974038,0.00597231083129337,-0.009205378147863461],[2.1311803888527003,-0.5724430221990725,-0.28322843555108296,-0.27853883441349075,-0.30329755876109543,-0.3181573793884286,-0.3585664918222387,-0.19321561889664854,-0.3733518251440448,-0.29647202425943114,-0.35052126173964104,-0.4985717979538301,-0.38054676844523166,-0.5770400380476858,-0.020756071662058475,-0.26239169606881674,0.332861732067565,0.14608397114890953,0.5509227099676853,0.2687680367081172,0.11124099837594296,-0.09177579796988504,-0.7661067146502598,-0.540918337276073,-1.224195866031292,-0.7960504339113978,-1.0269467211969627,-0.441262543505256,-0.3536024710978091,-0.02440123552710071,0.132925725155571,0.10337758446776887,-0.14774585479607508,-0.10159081341115826,-0.7470263405571653,-0.39042224385622926,-0.9794464968772414,-0.36597139147995705,-0.7307751033768425,-0.09444333786977906,-0.23665111059497018,0.11525361988027892,-0.047924437545533986,0.06943221260954066,-0.12850705648552394,-0.0065383249165313516,-0.3007609205275477,-0.06624771356131623,-0.26358808998005],[-5.154777580841294,0.5255808175426109,-2.0015367710437904,0.22233272780144897,-2.003627053882986,0.1788188220769579,-2.091757280130948,0.16429407862456885,-1.9888602560915642,0.08323663598535185,-1.87012545271466,0.07069315348293528,-1.7615187409637607,0.1038364171045507,-1.537517774479422,0.08509937705766117,-1.265610490947262,0.20532107946642875,-1.0195354762017466,0.19496121612980305,-0.7560143419642994,0.2500412973025036,-0.5185008989590897,0.25643291576032234,-0.2059667416118935,0.21976364255818787,-0.007623205260118975,0.2230619927577692,0.1945212571274984,0.1560364761100753,0.16949391914489578,0.10054749959476494,0.20982140378415542,0.11092649494079414,0.25780751765066406,0.10428198094878842,0.3083022671225827,0.08318464175020425,0.37993678982826556,0.01675202000585585,0.19192901378884705,-0.018502040295148965,0.009524671864320449,-0.04133594815088692,-0.04161702521014805,0.012266947673127474,-0.07388933277659603,0.00517675521850711,-0.008015002313016398],[-1.7764268358774022,1.8753222589757716,2.09091901198755,-0.6581076124665852,3.084448809594687,-0.914554950052776,2.373685067510126,-0.9004661049102233,1.9294922766835862,-0.7869168732010072,1.4903442127648965,-0.8319228149919907,0.8297047502359928,-0.7687451450281481,0.3404085944970568,-0.7021824324310423,-0.18756650198745553,-0.6226369502192037,-0.34522777313582076,-0.4351815873995549,-0.6614644298056525,-0.35880328476995593,-0.6014821345274804,-0.26244215896831663,-0.7438121980886324,-0.016228702834018987,-0.6928766967559592,0.07684770730791764,-0.5642281072788858,0.11853485908223894,-0.3117008447355828,0.1727347390382767,-0.3616060419213128,0.06657982819489097,-0.5073947385614429,0.020619092841241866,-0.6239158163480327,-0.1256747377970446,-0.5307473724235595,-0.06572360170081722,-0.04805043328501732,0.1608651202802658,0.1396827010411541,0.19605508337154115,0.45195952442465637,0.09453610701645444,0.2921804121542431,0.04560619915705797,0.01003058871085604],[3.3545468891002086,-0.8477343973810627,-0.5554448179662634,-0.5175702361073306,-0.8566186346842581,-0.22747167354544678,-0.4517899208064732,-0.1597569651505988,-0.18535470411791063,-0.2454920571239962,-0.4482908363811254,-0.2382206394589332,-0.31256543568936784,-0.21372229949222615,-0.2973048362797805,-0.2397371282285879,-0.26889472447451945,0.04811040635069356,0.05091090044935234,-0.0793805388209546,-0.22140952473168987,0.03959067595245423,-0.3493681750673997,-0.1699817812731562,-0.12172293838632374,-0.1431380336851154,-0.04799566574239771,-0.044585680593588785,0.0049394881287794816,0.007734799239089647,-0.13450831397914098,-0.06500886762708583,-0.2794228691714369,-0.039101801760503506,-0.11876012810121746,0.04999034067406978,0.1700267099067818,0.08312689876914824,0.23864254334056148,0.02208909920928988,-0.02809182797158016,-0.04140254359366703,-0.3464407945417657,-0.04737824714190531,-0.09668461473740186,-0.10735892344379643,0.04852083690160706,0.06819638460738928,0.13954837130658163],[-5.490644348740357,-1.307564099219544,-2.1910802734018473,-0.4280874134393808,-2.17037450291077,-0.08869055757152783,-1.9623963474550083,0.044907873780346635,-1.7115512267455977,0.009123591872882275,-1.2746602199439592,0.07146140069297925,-1.0709359103868223,0.10560950705688571,-0.6159897755257908,0.1135785864710772,-0.7183716632406628,0.13787325672535397,-0.724399563970575,0.08623712516360021,-0.8626156293003192,-0.0020833304759061443,-0.4962298755024462,-0.10148008304801032,-0.16868397085824435,-0.011780794278476949,0.031902292632861436,-0.15319208365314643,0.1110720083889178,-0.08426546546746641,-0.09799330857048988,-0.051271063823193,-0.0997115547118432,-0.038975032918078516,0.0343106849030246,0.043438930539782546,0.053605993884647096,-0.015233412520254518,-0.005492738528656328,0.06467902026963294,-0.03260989518714981,0.06649331589611968,-0.09954297561179219,0.08238750872902244,0.13964478312695766,0.11026979903350538,0.16915740120622744,-0.0068339389409282246,0.10562027868599724],[4.729281739822107,-0.6251859964958214,-0.7638887098674481,0.17935198884264938,-0.21017243702743746,0.3497739673832463,0.015601803838862276,0.445264109921534,0.40706655901170585,0.5715459306293819,0.5367224299962567,0.6757905365735728,0.7577066054373529,0.5725528231854252,0.643538437931842,0.4698919656468348,0.4658919197056483,0.19940056754016716,0.221591544250976,-0.06664211554782953,0.057023489641988646,-0.20059598519120508,-0.07400958341880365,-0.21982615658986407,0.03269978489421386,-0.1724124725380814,0.10837505415155298,-0.07421828799313421,0.10583990982771495,-0.08196617095050263,-0.1583281862798005,-0.2534380418988016,-0.3874291605320572,-0.3231582548645213,-0.4348088898936096,-0.20478608453828304,-0.16198775454146538,-0.03518165212444058,-0.0030677974367641254,0.05812403632963192,0.07080731590863351,0.018230640129508,-0.0835069235715728,-0.022912289872732264,-0.19137895297889151,-0.05268870538063811,-0.045966357465597595,-0.016318118697609935,0.05479442070658103],[-8.231745650113824,-0.6705038597935463,0.88247068527736,0.7948348423123377,-0.4465212247861347,1.2594012290522865,0.09900036057747084,1.1260761818968124,0.38675348250641634,0.9650172100456457,0.49870653448508784,0.9185331051630639,0.7483176154337281,0.798378900463403,0.8973670659019269,0.6596898250526939,0.9099989511777878,0.5389553392406942,0.9208690279864111,0.34084000512283086,0.995123650393847,0.2467022383372118,1.093711267666272,0.12314898269264116,1.0579891241975148,0.005042936213933205,0.9636313013164968,0.030215255097390214,0.715007401415973,0.010022824903983239,0.3370883378058756,-0.09517662914415395,0.21409175667790648,-0.07496701180555336,0.015896211347027175,-0.0717482822581972,0.12051632583912311,0.07824514467682253,0.34105162627747926,0.06125053928725429,0.38722373272028304,0.06850817651976658,0.28168142907848404,0.03297725150782727,0.009741870663753918,0.02031532545432092,-0.1197919433433314,-0.024247705204456252,-0.07959103683586209],[0.061189807117451075,-0.30971347615417244,-0.9522315336105176,-0.1296034517274636,-0.6210539133300571,-0.28913668758654104,-0.717752176028145,-0.30525098997282507,-0.6370671433561965,-0.2889221283107034,-0.5228670413890208,-0.16709287727951894,-0.34658260663129864,-0.0812715634891652,0.05526500653487555,0.10637613975981743,0.1997847709434137,0.1877328161409409,0.1673659015662517,0.128677967663641,-0.0633106668864619,0.08058774708664716,-0.18857316333815916,0.04047299580218723,-0.06247126594620471,0.03056192236684328,0.2607407046057729,0.16152762299849982,0.5621650632228765,0.262464114865945,0.39269363197359847,0.10553752102669318,0.03478631182172284,-0.0985539832536588,-0.3207412922572263,-0.18786207121017315,-0.19683063628234918,-0.1325886383249344,0.003269483797145085,0.0023867511372102023,0.14092217537580864,0.040398245876775214,-0.023953029041081635,-0.026307286109041414,-0.19465099558570054,-0.11578146734439139,-0.24442455354704257,-0.07715567446160547,-0.12984762530100172],[-1.5694922283328783,-0.08354076730329128,1.1599517852843564,0.42078987644583876,0.6846412267804008,0.1618751366983851,0.7230377991349389,0.006327277670644174,0.5246935449785561,0.0096391677993233,0.4649615480083615,-0.12365449491810566,0.5675963191721973,-0.15187206920976706,0.6270266298554509,-0.08828660803978239,0.587334044732252,-0.18591603966227835,0.5661349313542323,-0.11348717885310018,0.5375984446171174,-0.1571191402020801,0.26944317896165837,0.01340307641853432,0.18262714557281956,-0.1371372394284783,0.21208710374556652,0.01391813049767804,0.1973886768336288,-0.03335416745068614,0.24407236490464973,-0.05359558548144941,0.19463235319340474,0.003773948296622479,0.08694731194390118,-0.008372725248761762,-0.04971984516252116,0.0360532337119569,0.012182779585801247,0.05113074278382284,0.02390125986768155,0.010690805281958428,0.17995379561489497,0.025397114881870585,0.14041477807836572,0.005867350556277287,0.06733840619018962,0.058231049167292916,-0.09650180427514761],[-10.819849749210452,2.097345380583822,2.2304319251039084,1.7789642634001488,1.2448128442595308,0.7009602585185857,0.7549994284914525,0.28565120399317,0.37323531949055183,0.03415420313095721,0.35051971979759167,-0.08926006589011705,0.12110893927946169,-0.05000656410672363,0.023320134029396707,-0.04975716723550281,-0.22724477248829006,-0.12476198590845676,-0.45987352994742675,-0.2655935571963829,-0.6474573438077437,-0.488074673224134,-0.46583064546691566,-0.19936684633500817,0.10732418454960493,0.350631217296418,0.5619791495349413,0.6034633214643055,0.48371805299742743,0.44587158768323204,0.3221977188067926,0.27009506575210623,0.3010014620803065,0.49048041455150226,0.8299627053182126,0.8917896864564545,1.2296199859138002,1.1055857230947244,1.2996395106113732,0.8655249560700911,0.9230796774907627,0.47424269651309386,0.6651116983593655,0.35927232762038286,0.6557213563282546,0.3814541339888429,0.6899151381173977,0.30622314106141096,0.5107564109300139],[2.942383033996998,0.018992116349742887,-0.0008746828611734189,0.13943230579848453,0.13723255277024055,0.128709209375463,0.29241129420890066,0.056513763117949305,0.31865574213040443,-0.0055097397672035945,0.32284444933197326,-0.026905006818066336,0.2949343668753145,0.029306162813404348,0.37440649476792587,0.12215541104600637,0.4723419816561866,0.19076462780416642,0.4494422396766256,0.19538249886410405,0.29197800514613526,0.08008660311022053,0.22272887666904814,0.036211761976184094,0.17767825524004985,0.08451992629570629,0.25646832970401007,0.1353594387300588,0.27830182005206844,0.14515101444173567,0.34445847933224005,0.16632989148278246,0.36471773663650414,0.17567589252136898,0.36728201501684754,0.17945183578600565,0.2857207688764486,0.15585659329689464,0.24192061629760309,0.12090358679431354,0.296936484685873,0.13943833186708332,0.39260018529228885,0.1583903414024708,0.355811288732225,0.1674565016920094,0.2818741059750161,0.09767627980296113,0.20548265146674322],[-7.230548499667318,-0.4150653419963663,0.28832475150022185,-0.44948630876418066,-0.7667998576590171,-0.12668546486752474,-0.46514333675356007,0.10437735051903498,-0.24649745040269325,0.18654869798394869,-0.09412116742335996,0.32434028739085596,0.022370482495114353,0.29907097180265624,0.294718357182763,0.43755972889824996,0.16161419989651837,0.1940620340201222,0.4584764901562149,0.17829531031489518,0.5184280784194477,0.1947510256812802,0.4115548794036893,0.22378407025386007,0.2157736395917198,0.15742813540760617,0.06905765534978478,0.011496700956578126,0.27312661745185995,-0.04498676103922979,0.4812197223342058,-0.07382646918651457,0.44911392387587107,-0.054616272364859025,0.22096865131678364,-0.014474975506876238,-0.15994879078072258,-0.036200478724722894,-0.40085595469948354,-0.10170983789810104,-0.213269717337067,-0.19167152462684556,0.04397554455992989,-0.0807301443514537,0.15075846888005684,-0.030136424016834258,0.12640891058894257,0.004925189453462058,-0.05360025071887736],[0.3988116165361686,-0.07178838276614517,0.19799532047264318,-0.15033660461118153,-0.4266277968769206,-0.08125669639917549,-0.1773605982468268,-0.021209055111228926,-0.03280450761275949,0.1464041058879863,0.08269945935673402,0.21779683341891778,0.25683569541454726,0.3446080090433225,0.27931780170833176,0.31138263512755726,0.13961750372826942,0.2558682196124274,0.08357564269255488,0.07725713315951632,-0.03826684831774618,0.13505931519398817,0.0922729157009182,0.09593811910301045,0.16621564043331796,0.11153804645721342,0.23916008253665966,0.0937929987998098,0.14998130340623714,0.007312208395870449,-0.040869297916505495,-0.03897699208988714,-0.07037381722001804,-0.031007758028633547,-0.05853959690306446,0.02391487697917388,0.033629786539370154,0.03308696075730432,0.02354520796132396,-0.04854778778757756,-0.015995377154126363,-0.06398066434647731,-0.07382605468692392,-0.04259557067406942,-0.08812815078062308,-0.040057962598747845,-0.0003239808664577965,0.01666308532171074,-0.00022591346661246018],[8.452036570277583,-0.18116703964950978,-4.474801551197984,0.23883201880550434,-3.4040691660886635,0.1598561960292799,-3.558460343743065,0.1506020514908425,-3.19390750264828,0.2472192551627379,-2.892388262128746,0.292165120222875,-2.5328462976944484,0.4686511818000354,-2.0805173011571827,0.42741868898303054,-1.2169596651366448,0.6480460564526065,-0.597649680996754,0.8138055861771647,-0.003334259210035331,0.6735479430759485,0.4326559842284194,0.7250546617455706,0.4003745212340657,0.5544397034143691,0.49475343918502396,0.33592617235711664,0.7488500887754846,0.2524491830332954,1.0196789703337363,0.3886457061801042,1.4076012856941813,0.26338041762381587,1.415504631688066,0.1019993540986698,0.8340295482709739,0.05174095813748211,0.46766069168416785,-0.22276161412438944,0.29036573692618783,-0.17761010316965498,0.3979344978063862,-0.11229480758568192,0.513643074050528,-0.045056497671474176,0.25818685100408506,-0.07119564021038761,-0.2333392075623575],[5.516142210414371,-0.8802297456601984,-0.24642625710452506,0.057147264408152,-0.0739329549298374,0.014271868227965534,0.1686067406576445,0.048570058092077664,0.3958255410981127,-0.24726685290734182,0.12192622526568153,-0.1781347076641622,-0.11589214949161386,-0.285429080325468,-0.4180074430780777,-0.04577141591092387,-0.6487237112042434,-0.21650968009944616,-0.25641937121283753,-0.20549853346538624,-0.05574563982875985,-0.19616863022180225,0.3805895091973858,-0.07442991648536149,0.10106509816611778,-0.04754947310409785,-0.4287402115764751,0.018446883381079013,-0.8890596697120002,-0.08167075400020105,-0.7842519132854213,0.12307449457692106,-0.22051905742701478,0.04671973283435329,0.43406853273411083,-0.005852699924999264,0.3944274691317122,-0.05773233414870305,-0.28172197677872707,-0.054352040069075146,-0.7392276562345368,-0.11205774688875528,-0.6400076330413051,0.0003733934544638623,0.03837753994990283,-0.052447758136862456,0.6089018256354083,0.03077393126772612,0.44506104294860693],[4.2197790627235525,0.8911757080170831,-2.0216430667211016,-0.4559708050944019,-1.3582023485370143,-0.8197829252868004,-1.5878902505053245,-0.8132510959089947,-1.6039800356028218,-0.9598843873064435,-1.3329983575942805,-0.8562453538077557,-0.8092417278342323,-0.668840970568329,0.07781857984587692,0.03279636136540779,0.9641822501644381,0.5509767884458507,1.200803939042538,0.7380393538570277,0.6303377851027787,0.3179595352103989,-0.5045652135015265,-0.22269362734787318,-1.1171861469271778,-0.4959626717366774,-0.699081394112637,-0.13527370165989352,0.40190853416323397,0.28132147938582663,1.1376478333464448,0.4865413508368436,0.8062876950185492,0.15418138671759213,-0.14446359871084297,-0.3460269010640696,-0.8724519749034042,-0.5291880294120905,-0.912106931481274,-0.3884306862017139,-0.2632389880132161,-0.029636244282120014,0.20621704956770082,0.09677837524943013,0.1992619609762236,0.11687064121953217,-0.049642340794495206,-0.06058695038455933,-0.2738408242427398],[-0.8387462284581453,-0.41942965149451805,-2.8260040797322623,-0.6159796669563236,-2.142879915869666,-0.6271942727814542,-2.381047137978017,-0.4530478049499508,-2.5473453286689884,-0.33278947350959526,-2.2836827011276877,-0.42929266453738846,-2.0738586040200024,-0.44317232582840416,-1.878951077235662,-0.46573506531706643,-1.8693141200303764,-0.4027814017301532,-1.5471519076791374,-0.31577707613652317,-1.2024128015717417,-0.16929538593216042,-0.9531757803125308,0.013479793388579813,-0.7703682215658642,-0.08063301028001847,-0.7408050549403578,-0.015505989203183766,-0.6319358729930024,0.06950605693893637,-0.3029358408980071,0.17410168776835008,0.07720472982679556,0.20446159325672134,0.15898188342628572,0.26260423244364295,0.0373495095451651,0.14283288468711522,-0.0038835149827570804,0.2167622822338241,0.13530867643477684,0.1476590624417161,0.16891472554647508,0.12120269156808473,0.12649151274444437,0.07270414208257767,0.111699944550948,0.019583039313251217,-0.05137739108552811],[1.841698889037059,-0.034604157574286636,-0.5107802262690392,0.18565053679124643,-0.2246313377610047,0.26049750282541356,0.04672236650996357,0.4343452443953258,0.2669774176739664,0.5116215542016661,0.49166522341855806,0.580103223066376,0.6630586636629408,0.5348177150492435,0.795882245969017,0.468221065024992,0.8060359420821354,0.3383130229910415,0.8419485454662978,0.22022736724958317,0.719678117455537,0.09880671696034309,0.5736802507372677,-0.01305581518569641,0.4221370153588421,-0.07514310353049802,0.26485767727246406,-0.12712724049742802,0.11504443017567409,-0.12332699225360345,0.024567034311565795,-0.13957777370372518,-0.06676174611111522,-0.1096536072395661,-0.16694526400719023,-0.07155278785650714,-0.18048810464442316,-0.02742771307952733,-0.15849154420738365,0.02486101786248532,-0.10109618243912175,0.05545720951028464,-0.08180742977389134,0.05119450062736045,-0.06550509911990858,0.020572223389774134,-0.04925800608758299,0.011988647247468617,-0.040055635097201094],[-0.2656143722207897,0.4467460345813714,-0.9409963091813243,0.19206697949384835,0.9840069506236915,-0.25826484208939393,0.11313399621397238,-0.1683176623424553,-0.07043630045417087,-0.1156084456038157,-0.33852437700954513,-0.2341020223306037,-0.5504534400169871,-0.25657944674424654,-0.7832283486303399,-0.3803716619196327,-0.8935606214461551,-0.2966245229017693,-0.8086186308668133,-0.2540978726103485,-0.5252944018922732,-0.11983357832969549,-0.2099616899148754,-0.013821937008873852,-0.19191167528556657,0.029092550386070334,-0.32678188286278703,-0.011528490058358308,-0.5134296000466622,-0.13080069118204446,-0.45664397595224226,-0.11231663625528486,-0.15261312873768224,0.05459691656046473,0.1689212445768208,0.24752009298485628,0.3730121970026841,0.2689204237911584,0.28948068031764806,0.22199765230696236,0.17715414313553127,0.11232888465834621,0.11685845088411527,0.04086759847531028,0.21497419563315062,0.0587786359775255,0.37443804527723223,0.1321814169350628,0.3729707994158091],[-7.585840445894914,-0.9720149434968137,-0.3276108010401974,-0.9858843325406946,-0.7233914828155882,-0.48370565323749964,-0.4239270024964518,-0.2647691515084559,-0.2300558481146948,0.010957711632366907,-0.3299785527979581,0.07967341328662714,-0.43485940456395217,0.13894878090375493,-0.5970610920411568,0.042629384493273074,-0.581587866197503,-0.03128505981113919,-0.5439192183855397,-0.15602736992925834,-0.16063951324353087,-0.11031425397975174,-0.2254481728113976,-0.14553536586737673,-0.4465471857921289,-0.07436634904636205,-0.6513121769015253,-0.06161063148013075,-0.4423477037148511,-0.10246532397346288,-0.1156937900651731,-0.05810404218724591,0.08556869016452617,-0.10214171570692268,-0.06374619999750025,-0.06105454816838158,-0.264605404623201,-0.028958171977899832,-0.26306209209759246,0.036360085417235614,0.02298528887303906,-0.02765588350269649,0.20402808721422963,-0.09783244383562405,0.10482690831012535,-0.0258533794398714,-0.037625573697845555,0.015865305053388255,-0.12987583874128028],[-1.9036702654851232,0.9841528527674774,2.923250688771162,0.8164812636315459,1.017118451386048,0.9109980097989937,0.618341703402464,0.5280180810952815,0.22733083641814017,0.07622538837431846,-0.11592364821315237,-0.1935429098381506,-0.23445416917881623,-0.2336740942493152,-0.01726391860106427,-0.04400134984928957,0.3615643401416833,0.25178573512162716,0.5174951579547667,0.3283815409323351,0.33538018663797164,0.13536961858584395,-0.2545838048652639,-0.21981202699366126,-0.6897544486624654,-0.43474621640819217,-0.748811830562195,-0.2851130183798027,-0.21061046840970798,-0.004419698788194124,0.4061560161230459,0.32842840212193264,0.6262154487335196,0.3177008965664321,0.4559272615019962,0.0887506360715992,0.043337770062066086,-0.04819465797296327,-0.21566854859539583,-0.14362248403344702,-0.24624519205680495,-0.09450912896350833,-0.13464889167554292,-0.053512265339449125,-0.04093798012579777,0.011233365906743524,0.049333247746902884,0.0021598402046232447,0.07506017881434073],[-12.227766818521197,-1.0014151222990555,2.3930067395402888,-1.0019861674556017,0.15261179353959875,-0.1962531350437207,0.1624769341938619,-0.4407957346359342,-0.056006311361943416,-0.17829400234889348,-0.422810090431197,0.03412751444063366,-0.25235169205878766,0.21213673767513525,-0.44244682481715586,0.22896937710631907,-0.4378236089117173,0.09938356057408931,-0.23824293950504102,0.09480800277494363,-0.3029250505850212,0.10234091528501364,-0.3944926896932748,0.04253700206179883,-0.31541773008575025,0.06046239494413975,-0.7065142520241724,0.07129782109937007,-0.75160680718991,0.05145130564984078,-0.49312731468413545,0.08715920976926704,-0.1767529799342809,0.0025696825370326503,-0.23495322670241703,0.01254934365029895,-0.4155177336233994,-0.007510898531856643,-0.6208192907731138,0.03441562255542458,-0.4034298093643456,0.03654285809952636,-0.052180083344154506,-0.038343670859818504,0.01469837508165822,0.01168820211055656,-0.10525182540632046,-0.023556779349660342,-0.18051844092988345]],"nearest_scaled_distance_max":0.6098501362383525,"tune_rms":0.0018704108659055542,"test_metrics":[{"regime":"all","n":64,"rms":0.002035732319762458,"curve_rms_p95":0.0038112522666018177,"abs_p95":0.003110665102278143,"worst_abs":0.03646553721760648},{"regime":"grazing","n":18,"rms":0.0011812625290361476,"curve_rms_p95":0.002375425633896312,"abs_p95":0.0021549597496050934,"worst_abs":0.012168180668448825},{"regime":"near_free_lobe","n":12,"rms":0.001692929463766995,"curve_rms_p95":0.0032817785214537976,"abs_p95":0.0031684794755556917,"worst_abs":0.02238030989251072},{"regime":"unequal_mass","n":23,"rms":0.0027110612335884444,"curve_rms_p95":0.006225733870664622,"abs_p95":0.0037908440245844935,"worst_abs":0.03646553721760648}],"response_metrics":[{"anchor":0,"parameter":"inclination","native_response_rms":0.0057873255261588165,"difference_rms":0.0007593510349466557,"cosine":0.9934322357848101,"relative_norm_error":0.07078218789858774},{"anchor":0,"parameter":"mass_ratio","native_response_rms":0.0021346924239851544,"difference_rms":0.00037714502295545005,"cosine":0.9852545839030297,"relative_norm_error":0.05879492489450509},{"anchor":0,"parameter":"free_fill","native_response_rms":0.0035396827035730563,"difference_rms":0.0006913005063689485,"cosine":0.9821646682716504,"relative_norm_error":0.03498112353319897},{"anchor":0,"parameter":"temperature_ratio","native_response_rms":0.0037940501082601467,"difference_rms":0.00045266999976150926,"cosine":0.9938359995635376,"relative_norm_error":0.05026602366139002},{"anchor":1,"parameter":"inclination","native_response_rms":0.0013336839344153084,"difference_rms":0.0006780284160003292,"cosine":0.945035523587375,"relative_norm_error":0.33432991637640236},{"anchor":1,"parameter":"mass_ratio","native_response_rms":0.0021576525064995983,"difference_rms":0.0004590168100870262,"cosine":0.977617580278515,"relative_norm_error":0.009145507046654493},{"anchor":1,"parameter":"free_fill","native_response_rms":0.0038689083589411215,"difference_rms":0.0005869629881948697,"cosine":0.9934526424350116,"relative_norm_error":0.10637186250790477},{"anchor":1,"parameter":"temperature_ratio","native_response_rms":0.006242065346932287,"difference_rms":0.0010077920186131164,"cosine":0.9900830290611373,"relative_norm_error":0.0894848893173184},{"anchor":2,"parameter":"inclination","native_response_rms":0.004137166739508631,"difference_rms":0.0019036382111203647,"cosine":0.8904147487292993,"relative_norm_error":0.177104582296838},{"anchor":2,"parameter":"mass_ratio","native_response_rms":0.0035289233638413553,"difference_rms":0.0013073006511836467,"cosine":0.9573920931519081,"relative_norm_error":0.27463189293155593},{"anchor":2,"parameter":"free_fill","native_response_rms":0.007019147552894897,"difference_rms":0.0005575891699638529,"cosine":0.9971039735783601,"relative_norm_error":0.020055620708774015},{"anchor":2,"parameter":"temperature_ratio","native_response_rms":0.006823767124350976,"difference_rms":0.00042640138453188975,"cosine":0.9981498131793521,"relative_norm_error":0.012563690158672034},{"anchor":3,"parameter":"inclination","native_response_rms":0.006608269804958515,"difference_rms":0.00019237852148199958,"cosine":0.9996226154111703,"relative_norm_error":0.010014250999079415},{"anchor":3,"parameter":"mass_ratio","native_response_rms":0.00047144341186963664,"difference_rms":6.944873620845953e-05,"cosine":0.989101020321234,"relative_norm_error":0.015516227475122452},{"anchor":3,"parameter":"free_fill","native_response_rms":0.004126182867947012,"difference_rms":0.00018893843946158256,"cosine":0.999134810968546,"relative_norm_error":0.02002520631453988},{"anchor":3,"parameter":"temperature_ratio","native_response_rms":0.00699985614981538,"difference_rms":0.0003479688093108472,"cosine":0.9992640790565759,"relative_norm_error":0.0308847755345182}],"lobe_q":[0.40143804624676704,0.40266693718731406,0.40645096581429246,0.4068260534666479,0.41292779985815287,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.41350000000000003,0.4146178698167205,0.4184568803757429,0.42558592036366466,0.425834452174604,0.4277704723179341,0.42859232546761633,0.434779588226229,0.4364045777358115,0.4397350716404617,0.4449377357028425,0.448325910512358,0.4516445959918201,0.45502996454015376,0.4628652737475932,0.46524298330768943,0.466956836078316,0.4689025540836156,0.4721422971226275,0.4739923927001655,0.4787253466434777,0.4824399060569704,0.48263753624632955,0.48467779625207186,0.48791129812598233,0.4887264432385564,0.48903681337833405,0.49292961563915016,0.5019024312496185,0.505190814472735,0.5072908315807582,0.5085165172815324,0.5110292400233448,0.5141888931393623,0.5189589244313538,0.5198351585306228,0.5220752674154937,0.5254459508694709,0.5291731745004654,0.5309029906056821,0.5347402543760836,0.538056170474738,0.5404655292630196,0.5410907505080105,0.5448877710849047,0.5488790774717928,0.5570430612191558,0.5574843835085631,0.5605746344663203,0.5612263148650527,0.564180836826563,0.5721260361373425,0.5753739850595594,0.5786161856725812,0.5796527245081962,0.5814216723665595,0.5832462886348367,0.584892258886248,0.5864394634962082,0.5949109021574259,0.5971754685975612,0.6012864224612713,0.6054031481035054,0.6069268096238375,0.609104899968952,0.6144481184892356,0.6198263264261187,0.6208201996982098,0.6234729823656381,0.6291287888772786,0.6305147633887828,0.6312184325419367,0.6343883224762976,0.6381311791948975,0.6432173733599484,0.6469984347932041,0.6483218182809651,0.6507962734438479,0.6538317520171404,0.6571484861895442,0.6584541729651392,0.6611943587660789,0.6652711857110263,0.670170403085649,0.6706334579735995,0.6735420119017363,0.6764411361888052,0.6844262199476361,0.6880720846354962,0.6893758561462164,0.6936665859073401,0.6954915910959244,0.699192444421351,0.7014809872955083,0.7072106970474124,0.7089616808108985,0.7094778786413372,0.7096964132972061,0.7141575228422881,0.7181587441824377,0.7213510205037892,0.7266955398954451,0.7325269338674844,0.7357740915380419,0.7373418276198209,0.7403658716008068,0.7418796357698738,0.7437503147870302,0.7506803348660469,0.7523716788738966,0.7546184463426471,0.7557013547047973,0.7610168752260507,0.7632835807278753,0.766292520519346,0.773651586752385,0.7774897792376578,0.7775756996124983,0.7824767090380191,0.7826929126866162,0.7828821819275618,0.7889465869404375,0.7927296902053058,0.8006049375049771,0.8020717445760965,0.8029273231513798,0.8031838353723288,0.8062164976261557,0.8116771751083434,0.8142309490591289,0.816464567836374,0.8196989682503044,0.8219376765191555,0.8219460223801434,0.8256531344726682,0.8306152161210776,0.831480267085135,0.8362029808573426,0.8392173001542688,0.8428776621818543,0.8450314207002521,0.8503159516490997,0.8535231444053353,0.8549142536707223,0.8622006840072572,0.8629964560270309,0.8675404827110469,0.8708027646876872,0.8743244347162544,0.8766168852336704,0.8815198719501496,0.8871314320713282,0.8895101826637983,0.8907641036435963,0.8909757768735289,0.8925921129062773,0.8946535866707563,0.8959224282763898,0.9006139060482383,0.910486988350749,0.9133745137602092,0.9145435931161047,0.917846054583788,0.9182535181753337,0.9257002297788859,0.9264945643022657,0.9268873838707805,0.9331410581246019,0.9356672857888043,0.9370562492869795,0.9439862693659962,0.9490072858519852,0.9528174660168589,0.9549154806882143,0.9565895118750632,0.9618477915413678,0.9660620540380478,0.967657791543752,0.9747429195791484,0.983279718644917,0.9854381944984199,0.987013899255544,0.987072245683521,0.9891111126169563,0.9915165796875954,0.993987453263253,0.9972625696100295,0.9995819545350969,1.0018211472779512,1.0051078096963466,1.006910477951169,1.008327944111079,1.0131260623224079,1.0149163463152946,1.019201869238168,1.0206896838732065,1.0230266540311277,1.0260631468147041,1.0303892607800664,1.0384446527808904,1.0393653017468751,1.0403521660715342,1.0456360348500313,1.0479232919402421,1.0489710952155291,1.049523074552417,1.0564358212053775,1.0615220187231897,1.0691009188070892,1.0724455043673515,1.0741595876403154,1.075463981926441,1.07876267824322,1.083769405260682,1.0869903577491642,1.0884412987157703,1.0960353247821333,1.101413532719016,1.102016992866993,1.1062872313894332,1.1118193344213068,1.1123349186964333,1.116409853845835,1.1173527131788434,1.1210813240148128,1.1222328634001315,1.1241021350957454,1.1258241518400611,1.1352869634516538,1.141941805649549,1.1427150228060783,1.1430752904154362,1.1481176354922353,1.1482818168587983,1.1510626431554556,1.1516805998049677,1.1583770531229676,1.162263167463243,1.1632481567561626,1.16875019390136,1.1725193060934544,1.17420723028481,1.1794685089960695,1.1813604101538657,1.1825089924037457,1.1851842964068056,1.1875770959071814,1.1882075197994708,1.1912637167610227,1.1995155361481011,1.2084883551113308,1.2107613847590983,1.2138767554424703,1.2147064574062825,1.216070473007858,1.2187789509072902,1.2241302990354597,1.2248162312433124,1.229906066507101,1.236359026376158,1.2383535798639058,1.2385905019007624,1.241559954546392,1.247013399284333,1.2477889692410828,1.2517901428043843,1.2554036558605732,1.255601221509278,1.26086205849424,1.2641343761235477,1.2657995769754051,1.2670850390568376,1.2708412364125252,1.274753919709474,1.2752841119654477,1.2755041629076005,1.278112268354744,1.2812307537533343,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2865,1.2867598376236855,1.2938888776116073,1.2948207885026932],"lobe_fraction":[0.46178251576985163,0.46150745317066866,0.46066557504435557,0.46058254085154654,0.45924215668850255,0.4591174522460869,0.4591174522460869,0.45911745224608697,0.4591174522460869,0.4591174522460869,0.45911745224608685,0.45911745224608697,0.45911745224608685,0.4591174522460869,0.45911745224608697,0.4588743104187266,0.45804415333196985,0.4565221080748659,0.4564694977386158,0.45606069850058495,0.4558877081437162,0.4545957049024103,0.4542593701533289,0.45357386164130525,0.4525131519489564,0.4518289167797873,0.4511636330548844,0.4504899268462373,0.4489494369671162,0.4484870499888313,0.4481552081688701,0.44777992894759455,0.44715847165802736,0.4468054722567961,0.445908599401855,0.44521084565629304,0.4451738717590868,0.4447930433696626,0.44419273789226427,0.4440420287003758,0.4439847111799636,0.4432688655286468,0.44164006555479113,0.4410503954099982,0.44067582240628916,0.44045791562591385,0.440012829849048,0.43945624464109456,0.4386224299636794,0.4384700957827046,0.43808181610610675,0.4375007059837627,0.4368624613411654,0.4365677799804457,0.4359175104149547,0.4353593516540671,0.4349559545890692,0.4348515696399205,0.4342202243648905,0.4335613284912199,0.4322284953866737,0.43215700815751734,0.43165803007484416,0.43155315865532545,0.4310792401901406,0.4298171257168454,0.429306276399686,0.4287992335526615,0.42863773748592915,0.4283628041124939,0.42808010512240485,0.4278258527698117,0.42758751722686716,0.4262937590592964,0.4259510820556163,0.4253323683208793,0.4247170811988464,0.42449043291820804,0.4241674404695873,0.423380041831482,0.42259450267421605,0.422450099126786,0.4220658184853428,0.421252064597853,0.4210537913799458,0.42095329704682927,0.4205020053504227,0.4199721009234213,0.41925708525303645,0.4187292865454469,0.41854530162079995,0.41820231809766073,0.41778339296845507,0.417327926840791,0.41714927197115054,0.41677551675269664,0.41622238072169687,0.4155622547520419,0.415500119071847,0.41511083701041057,0.41472453463512043,0.41366928952555526,0.41319169262138383,0.41302153579159234,0.4124638824039126,0.41222777016376194,0.41175092559286164,0.4114573560522825,0.41072668065886436,0.4105046089375818,0.41043924947559646,0.4104115940768744,0.40984896197071885,0.4093474171928658,0.4089493434342232,0.40828695676187693,0.4075699674350644,0.40717328179304213,0.40698241115025335,0.4066154240499144,0.40643230358872423,0.40620654277675305,0.4053753158588141,0.4051736580646292,0.40490650598531985,0.40477803826420417,0.40415021438713694,0.4038838787596325,0.40353160209469513,0.4026760701589599,0.4022332226719907,0.40222333538858596,0.40166123287170724,0.4016365209978373,0.40161489355616115,0.4009248031970585,0.4004971140145764,0.39961360784960814,0.3994500556595949,0.3993548013891422,0.3993262637894312,0.3989895948609917,0.39838671501209866,0.39810622660458445,0.3978616573331155,0.39750874952535425,0.39726533740572256,0.3972644312730005,0.3968628925275156,0.39632837517025093,0.39623553515505916,0.39573046258715894,0.39540966111208936,0.39502173019106473,0.39479429928569776,0.39423884390775843,0.3939035074060141,0.39375846901845113,0.39300282146919385,0.3929207032984327,0.392453316775051,0.39211935797168973,0.39176032548504236,0.391527431287733,0.3910314811081266,0.39046742378969185,0.3902294569953901,0.3901042874459745,0.39008317608932025,0.3899221446772013,0.38971721267888965,0.38959132501114685,0.38912749604638514,0.3881596896073312,0.3878787429133904,0.3877652632280771,0.38744553106690305,0.38740616658292654,0.3866899991885784,0.3866139676127369,0.3865763935628982,0.38598048474085267,0.38574096491837595,0.385609565610587,0.38495704975578104,0.38448745863365935,0.3841328732424831,0.38393827072820985,0.3837833215223702,0.3832984854624234,0.3829119479615834,0.38276605411328624,0.38212136618607684,0.3813512015312909,0.3811576014252687,0.3810165577373567,0.3810113396830416,0.38082920592585345,0.3806148388773743,0.3803952212555437,0.3801050187207252,0.3799001180024329,0.37970278381403316,0.37941399228610023,0.3792560249754981,0.3791320253612176,0.37871366855727484,0.37855811391603766,0.37818694374090916,0.37805847486568683,0.3778570885063504,0.37759615717622697,0.3772258337430882,0.37654070285290253,0.3764627629836969,0.37637929978543594,0.37593386314145194,0.37574179413066566,0.37565395684819947,0.375607722377525,0.37503090639352415,0.3746090884863218,0.373984559837575,0.3737104683847787,0.37357035452385745,0.37346389106527594,0.373195274166942,0.3727892572134743,0.37252912393162535,0.37241221327353313,0.37180304918347484,0.37137437871743756,0.3713264209523717,0.37098786787601024,0.3705513647981477,0.37051080293620986,0.370190934368667,0.3701171029197575,0.3698257892416866,0.3697360320384797,0.369590542809462,0.36945674616670504,0.36872543854198186,0.36821507904058737,0.368155990057,0.3681284733252068,0.3677443294023506,0.367731852231988,0.367520813109901,0.3674739909491792,0.3669683455085543,0.3666763590310622,0.3666025192922818,0.3661913032766185,0.36591081522579666,0.36578552132295683,0.3653962304985879,0.3652567065556725,0.36517211930568005,0.36497544248456654,0.36479994258752335,0.3647537681245209,0.36453029846206936,0.36393002507618405,0.3632823818301315,0.3631191492710488,0.3628959672583524,0.3628366334508119,0.36273918559629126,0.3625460393509992,0.36216579687840794,0.3621171887012994,0.3617574252285482,0.36130363972997187,0.3611639012125332,0.36114731876441264,0.36093977644667036,0.36056002937796466,0.36050617037656374,0.36022889081327475,0.3599793066410836,0.35996568348225805,0.3596037811020898,0.35937950410234476,0.35926561866693696,0.359177816081816,0.3589218087761212,0.3586560146045808,0.3586200665620713,0.3586051514914065,0.35842858803707517,0.3582179897451002,0.35786341757934337,0.3578634175793433,0.3578634175793434,0.3578634175793433,0.3578634175793434,0.35786341757934337,0.35786341757934337,0.35786341757934337,0.35786341757934337,0.3578459740349475,0.3573688829686537,0.35730673030154697]}},"training":{"train_count_per_donor":192,"tune_count_per_donor":48,"test_count_per_donor":64,"seeds":{"train":53117,"tune":84191,"test":116731},"native_phases":129,"triangles":800,"primary_mass_solar":1.2,"primary_teff_k":6300.0,"passband":"Johnson:V","native_artifact_sha256":"34e614ac4fd0ce2101fd1bc54bbf1e03940c59d5531aae877bcf7e436a8e28cd","baseline_commit":"b7ce074c8231cc8db9bc2e4016b667071daaa845"}}''')
CONTACT_MODEL=json.loads(r'''{"schema":2,"name":"contact-rbf-fourier-v2","accepted":true,"source":"native PHOEBE 2.5.4 contact envelopes","keys":["inclination","mass_ratio","fillout_factor","temperature_ratio"],"bounds":{"inclination":[65.0,90.0],"mass_ratio":[0.45,1.2],"fillout_factor":[0.1,0.5],"temperature_ratio":[0.78,1.12]},"fixed":{"period":1.8,"eccentricity":0.0,"irradiation":false},"phases":256,"harmonics":48,"gamma":0.2,"ridge":0.0001,"centers":[[-0.98,-0.8119016531854869,-0.9421845134347677,-0.24485049210488785],[-0.3175862990319729,-0.98,0.9786640424281359,0.08054748177528381],[-0.9217839781194925,-0.23281661048531543,-0.98,-0.7643834985792638],[0.41841956228017807,0.5084587391465902,0.3041487447917466,-0.9800000000000002],[0.98,-0.3109845817089082,0.6499821804463866,0.6620982736349106],[-0.5071391612291336,0.9799999999999998,-0.6791916154325008,-0.5129243638366459],[-0.23200239054858685,-0.7338519450277092,0.98,0.33006878755986735],[0.7324677519500256,0.009494543075561301,-0.10367651842534564,0.9799999999999995],[-0.98,-0.0828351601958276,0.1734589748084543,-0.7031756844371555],[-0.055033616721630096,-0.98,-0.1557229571044445,0.6179669648408885],[-0.705596199259162,-0.8807025384157896,-0.98,-0.2852089889347551],[0.20419980958104134,0.3467748239636421,-0.5645827408879995,-0.9800000000000002],[0.98,-0.5837522316724064,-0.4812310244888067,0.13993179798126243],[-0.8692648448050022,0.9799999999999998,0.4396257270127537,-0.03861743770539772],[-0.39159351773560047,-0.37966720387339603,0.98,0.8704026099294422],[0.8939358294010162,0.8457390200346708,0.7807353101670742,0.9799999999999995],[-0.98,-0.4618021342903377,-0.24374435096979163,0.3089473359286783],[-0.44080557487905025,-0.98,0.21039941906928994,-0.3951327297836543],[-0.790771447122097,-0.5096701271831992,-0.98,0.6956640873104329],[0.2911451365798712,0.23355444334447362,0.5099139045923948,-0.9800000000000002],[0.98,-0.9628417640924454,0.41089236550033137,-0.8467853311449292],[-0.6260705310851336,0.9799999999999998,-0.38492625392973423,0.9490763805806632],[-0.1052468754351139,-0.00875638984143745,0.98,-0.1475645899772644],[0.6019128579646349,0.7326411753892899,-0.835442416369915,0.9799999999999995],[-0.98,-0.675122030079365,0.8874936234205963,0.7723844014108179],[-0.18559798039495945,-0.98,-0.9083320591598749,-0.9357107225805521],[-0.5828249715268612,-0.34603667072951805,-0.98,0.22160133160650752],[0.0852589551359415,0.8794917687773705,-0.37447310239076614,-0.9800000000000002],[0.98,-0.17408240027725708,-0.7046653255820274,-0.32194874621927716],[-0.9965488743036985,0.9799999999999998,0.749485567212105,0.4701459817588325],[-0.2721337527036667,-0.8469504080712796,0.98,-0.6852281838655473],[0.7707069646567106,0.38040503673255444,0.033374933525919914,0.9799999999999995],[-0.98,-0.20110230892896663,0.5695806760340929,0.011226885020732658],[-0.2880826685577631,-0.98,-0.6031904872506857,-0.1599047761410477],[-0.9493496529757977,-0.778234912082553,-0.98,0.999510137364269],[0.45286701060831547,0.4942915216088295,-0.17675365135073662,-0.9800000000000002],[0.98,-0.7001814115792515,-0.7754850648343562,-0.5822450947016479],[-0.5981638412922621,0.9799999999999998,0.8017050884664059,0.7470438852906225],[-0.13900941610336304,-0.2792664282023908,0.98,-0.4064658395946026],[0.6384523790329695,0.9953235220164061,0.47817231900990054,0.9799999999999995],[-0.98,-0.9124616924673319,-0.29890094324946404,0.5329907014966011],[-0.08941948972642422,-0.98,0.2782973460853102,-0.6338245403021576],[-0.6731481030583382,-0.11654669418931018,-0.98,0.4587737750262024],[0.17463568411767483,0.6421741005033255,0.939139695838094,-0.9800000000000002],[0.98,-0.41338258981704723,0.10868834145367146,-0.12359359674155712],[-0.7753100860863924,0.9799999999999998,-0.06409101746976387,0.20928307622671105],[-0.48751674965023994,-0.6155151780694725,0.98,-0.8853141628205775],[0.9850219655781984,0.14114210009574868,-0.6532793007791042,0.9799999999999995],[-0.98,-0.5414287820458414,0.36921766400337264,-0.45078148134052753],[-0.40689460188150406,-0.98,-0.3330034986138344,0.3489712066948414],[-0.8227140661329031,-0.49551418237388145,-0.98,-0.5562484785914419],[0.322187427431345,0.7789536044001579,-0.8844411764293909,-0.9800000000000002],[0.98,-0.04247221536934387,-0.03837912715971492,0.8934274949133398],[-0.7215073294937611,0.9799999999999998,0.009423134848475456,-0.8067615646868947],[-0.007872181013226509,-0.9946042560040951,0.98,0.10045698471367381],[0.5112978667020798,0.27804416231811024,0.7079548090696339,0.9799999999999995],[-0.98,-0.31432280875742447,-0.5149194691330194,-0.9757650811225173],[-0.21657977625727654,-0.98,0.5328898560255766,0.8280636407434938],[-0.553811589255929,-0.6414548344910147,-0.98,-0.018731780350208616],[0.05128640681505203,0.11532442830502965,0.2470466494560244,-0.9800000000000002],[0.98,-0.8132793754339219,0.830199792981148,0.43009172752499536],[-0.9059953391551971,0.9799999999999998,-0.8720284923911095,-0.2662693727761505],[-0.3646251242607832,-0.14236476086080085,0.98,0.5746209193021057],[0.8662043772637844,0.6162338703870773,-0.40784128941595554,0.9799999999999995],[-0.98,-0.2534207981079818,0.4599520321935415,0.8336443398147828],[-0.34785922057926655,-0.98,-0.4922762829810381,-0.9974661022424696],[-0.8895578570663929,-0.7055850401520731,-0.98,0.157860856503248],[0.3881466258317232,0.0510671157389877,-0.7863899916410446,-0.9800000000000002],[0.98,-0.7525068186223508,-0.1363726258277893,-0.260627239942551],[-0.5374111887067556,0.9799999999999998,0.1614367887377739,0.4083280805498357],[-0.1997772827744484,-0.20662489719688903,0.98,-0.7489062491804364],[0.7021957095712423,0.5521065033972263,0.6172924544662237,0.9799999999999995],[-0.98,-0.6017499752342703,-0.6598456650972366,0.374090248718858],[-0.02476063184440136,-0.98,0.6379565075039864,-0.4607567787170411],[-0.7378222644329071,-0.43086806870996963,-0.98,0.6358074210584161],[0.23447280935943127,0.8437324874103069,0.07818729244172551,-0.9800000000000002],[0.98,-0.1026639547199012,0.9989871587604284,-0.7815800160169601],[-0.8389928732067347,0.9799999999999998,-0.9532339740544558,0.8833908829838044],[-0.423818688839674,-0.9298282116651535,0.98,-0.2073596809059386],[0.924207815900445,0.3426930997520685,-0.26297280192375194,0.9799999999999995],[-0.98,-0.972660792991519,0.730139497667551,-0.6438006497919563],[-0.471080482006073,-0.98,-0.692639771848917,0.5581108089536431],[-0.7585434895008802,-0.05177854746580135,-0.98,-0.3508334215730431],[0.26087024435400963,0.7068309355527163,-0.02352692000567913,-0.9800000000000002],[0.98,-0.4736963622272016,-0.9286550562828779,0.19924729131162167],[-0.6563444249331951,0.9799999999999998,0.8985432032495742,-0.09841309860348724],[-0.07301981188356876,-0.5508612263947728,0.98,0.804717555642128],[0.5716389790177345,0.20591314509510972,0.31768645718693733,0.9799999999999995],[-0.98,-0.14032233133912098,-0.4052526783198118,-0.18160684779286418],[-0.15532313659787178,-0.98,0.42193750478327274,0.016808411106467247],[-0.6150529850274324,-0.8424872104078532,-0.98,-0.8261403944343327],[0.11553378403186798,0.43015631660819054,0.8567058779299261,-0.9800000000000002],[0.98,-0.6392867621034386,0.19104920700192451,0.7252813037484882],[-0.9662749245762825,0.9799999999999998,-0.23172204568982124,-0.5766037739813328],[-0.30436075292527676,-0.34340453147888195,0.98,0.26825234293937705],[0.8009808994829655,0.9310741070657969,-0.5469843130558729,0.9799999999999995],[-0.98,-0.7354063279926779,-0.08344087190926097,-0.9105368480086328],[-0.25780973210930824,-0.98,0.12071005068719409,0.7623399663716559],[-0.9815757591277361,-0.2812882158905269,-0.98,-0.07850407250225533],[0.48313993215560913,0.9441073946654797,0.662900067865849,-0.9800000000000002],[0.98,-0.23449616692960273,0.25797056406736374,0.4952574390918012],[-0.5678917989134789,0.9799999999999998,-0.2878552079200746,-0.3319314792752269],[-0.1712345089763403,-0.7823318988084794,0.98,0.5147871710360044],[0.6687244065105915,0.4451505932956934,-0.9956958163529634,0.9799999999999995],[-0.98,-0.40112215094268333,0.7813824489712715,-0.38948289304971695],[-0.11969248950481415,-0.98,-0.7644360810518265,0.28719148226082303],[-0.6409220527857542,-0.5739578679203989,-0.98,-0.6199493426829576],[0.1443626992404461,0.16939374990761258,-0.4566515702754259,-0.9800000000000002],[0.98,-0.9020323120057583,-0.6225369144231081,0.9546646345406771],[-0.8055820725858212,0.9799999999999998,0.5816068258136511,-0.8684788867831232],[-0.4552915934473276,-0.07291418500244629,0.98,0.03669359162449837],[0.9547499939799309,0.6683505512773991,0.13942172378301598,0.9799999999999995],[-0.98,-0.022033069282770268,-0.8517059721052647,0.5962119679898019],[-0.3766197096556425,-0.98,0.8191506899893284,-0.6975269280374049],[-0.8549420386552811,-0.9448682311922312,-0.98,0.3969954475760453],[0.35246233455836773,0.28249209746718384,0.4019598457962279,-0.9800000000000002],[0.98,-0.5230648275464775,0.5522364024072888,-0.060311835259199476],[-0.6912334505468607,0.9799999999999998,-0.5269454997032881,0.14552114717662312],[-0.040099259465932846,-0.44394710659980785,0.98,-0.9470328222960231],[0.5415717605501413,0.7815704885870218,-0.19410594180226326,0.9799999999999995],[-0.98,-0.8723079990595579,0.0287728700786829,0.07050414197146915],[-0.246854605153203,-0.98,-0.05040095560252689,-0.21967745944857564],[-0.5215835608541965,-0.16819026321172725,-0.98,0.9337869659066196],[0.021011563017964363,0.5731964576989412,-0.7331862784922123,-0.9800000000000002],[0.98,-0.37127624079585086,-0.3126765973865986,-0.5229081772267818],[-0.936269273981452,0.9799999999999998,0.3581720478832726,0.6872105877846484],[-0.33239810913801193,-0.6691113878041508,0.98,-0.4721285086125132],[0.8359304275363684,0.07411806657910325,0.9253560844808819,0.9799999999999995],[0.8427159693092108,-0.11935745552182209,-0.34376275539398193,-0.2938544601202012],[-0.34124328941106796,0.6532851066440344,0.3270634934306147,0.3790634069591765],[-0.9293959941715002,-0.9172231052070856,-0.9349040258675814,-0.7118252012878659],[0.42595522478222847,0.3261207751929758,0.8942779209464788,0.6095339171588416],[0.027873100712895393,-0.6202747728675606,0.059920286759734154,0.8629729691892858],[-0.530352745205164,0.15420289337635018,-0.01923111267387856,-0.9642875790596009],[-0.2400273997336626,-0.41618753969669353,0.7188094034790995,0.13250621780753158],[0.7405688762664795,0.825084725394845,-0.7020473331212997,-0.04632058180868648],[0.5347862783819437,-0.7671253662556411,0.5208145584911108,-0.7562234699726107],[-0.03125401958823204,0.491483174264431,-0.5583287049084902,0.9205262307077644],[-0.6981066707521677,-0.18803811818361293,0.19532879441976525,-0.23669403232634023],[0.19654261320829391,0.5373649913817644,-0.2255198061466217,0.08849703893065453],[0.3456007372587919,-0.2662080489099027,-0.820344939827919,0.33700729347765424],[-0.8461729139089584,0.990565387532115,0.850473113358021,-0.4861809760332104],[-0.3834469746798277,-0.6890736836940052,-0.3958613295108081,0.669040363281965],[0.8859571926295757,0.03840104117989518,0.4333124216645958,-0.5057142768055198],[0.9615752920508385,-0.6540273763239386,-0.5911533515900373,0.22857106290757678],[-0.4640855733305216,0.12057221494615056,0.6130280736833813,-0.06524497643113092],[-0.7987224422395229,-0.32493981160223495,-0.12501289322972298,0.7793505489826202],[0.29929455555975437,0.9164546690881252,0.17081280425190926,-0.9285242315381765],[0.15123407170176506,-0.15298751555383217,0.7500596679747105,-0.6770383305847644],[-0.6496699582785368,0.6195328235626221,-0.7957965172827244,0.5288413371890786],[-0.11290885508060455,-0.8258537948131562,0.46616948582232,-0.31375532411038864],[0.6093766521662474,0.4173681680113077,-0.48798139207065105,0.47805808484554246],[0.6618990488350391,-0.49026779644191276,0.2890794761478901,0.6882089283317323],[-0.1624405886977911,0.7663834169507027,-0.25676964595913887,-0.6020232923328876],[-0.5747514888644218,-0.5381340608000758,0.9895644132047896,0.3014881163835532],[0.07723107002675533,0.1892187464982269,-0.964579401537776,-0.4028027262538669],[0.47626861929893494,-0.991307657212019,-0.1146110724657774,-0.15624556317925475],[-0.9728277940303087,0.26742280833423115,0.08956323005259015,0.0539542790502312],[-0.26459330692887306,-0.037220077589154354,-0.6640957333147526,-0.8554622586816546],[0.7631206829100847,0.6883052475750446,0.631722841411829,0.9406712055206292],[0.8097938932478428,-0.8835075180977583,0.2222270295023916,0.4659927953034635],[-0.31117466650903225,0.3578801676630974,-0.2005828619003296,-0.3651587292551993],[-0.9574962630867958,-0.08759472519159328,0.5613835956901312,0.5402133055031297],[0.4608455952256918,0.6869983095675707,-0.515814708545804,-0.6268794704228637],[0.12427030503749847,-0.38442864641547214,-0.4363692123442888,-0.8773582950234414],[-0.6219433713704348,0.8588017802685499,0.39085957966744944,0.7916685659438376],[-0.1464988775551319,-0.5865629632025959,-0.8471822589635849,-0.11564129777252652],[0.6461096275597811,0.1859660781919954,0.8255970850586893,0.2174518145620823],[0.5628259256482124,-0.22170891240239154,-0.897324638441205,0.9917999524623158],[-0.06620595790445805,0.5056523587554693,0.929896017536521,-0.843122310936451],[-0.6651231460273266,-0.7988432440906764,-0.32376256585121155,0.0035207606852056816],[0.16653449274599552,0.4578166715800762,0.34900864213705063,-0.16734287329018105],[0.2521337829530239,-0.7207877840846779,0.6987482979893687,-0.41490712016820896],[-0.7516530435532331,0.004730746150016563,-0.7240533977746964,0.25010855682194255],[-0.4799046814441681,-0.2998750060796739,0.02227974496781826,-0.5906904209405185],[0.9774863701313734,0.9588489029556513,-0.05491036362946056,0.7383916266262525],[0.9330339822918177,-0.35861106030643,0.9676337484270334,-0.02714374288916588],[-0.4306156150996685,0.884741522371769,-0.98456403426826,0.1748449485749004],[-0.8302544448524714,-0.6857448145747187,0.25347634032368704,-0.984172997996211],[0.32977376133203506,0.08690621145069577,-0.29433299973607063,0.8193744346499447],[0.24325337819755077,-0.8575673811137677,-0.6284314431250095,0.5669424664229161],[-0.7446647882461548,0.3836983274668455,0.6693473495543003,-0.7307645790278909],[-0.015945730730891228,-0.18665497191250335,-0.09261949174106121,0.42240921407938026],[0.5193256996572018,0.5878158994019032,0.10960877873003505,-0.27373157255351555],[0.6933696027845144,-0.5044179596006872,-0.16751240566372883,-0.548327665776014],[-0.1929802969098091,0.2209785934537647,0.13025932386517525,0.6501381825655699],[-0.5461495574563742,-0.45850556530058395,-0.6160753350704908,-0.44285709969699416],[0.04382267966866493,0.8000961802899837,0.5861458722501998,0.35716737061739035],[0.37936599738895893,-0.005461638793349377,0.49103055335581325,0.10765003971755505],[-0.88271539285779,0.7220217883586884,-0.46116010658442974,-0.19431620463728905],[-0.35667418129742146,-0.9575954079627991,0.7924980260431767,0.9006169661879535],[0.8580548912286758,0.29918649233877637,-0.7553041912615299,-0.7997829001396892],[0.8749410770833492,-0.569195978343487,0.8615225814282899,0.6288638710975649],[-0.3715153317898512,0.15633149631321408,-0.840668048709631,-0.5421980675309894],[-0.897170901298523,-0.39831186644732963,0.42129071243107363,0.3671436067670586],[0.39568319730460644,0.8604164496064186,-0.376511475071311,-0.4679770804941653],[0.060099221765995026,-0.07010971195995819,-0.5463030394166708,-0.21552999876439594],[-0.5606256816536188,0.657244760543108,0.5014686007052662,0.11371983587741852],[-0.20780129358172417,-0.897272240370512,-0.2365655414760114,-0.7897472269833086],[0.7102959547191858,0.3593773078173399,0.21565557643771172,0.8754363562911749],[0.5025611072778702,-0.2943528834730388,-0.00720893405377887,0.16529743373394035],[-0.0009820330888032913,0.9488707445561886,0.04056994803249858,-0.0014749672263859592],[-0.73033182695508,-0.7465153969824316,-0.7130963169038296,0.8410763945430517],[0.22681458480656147,0.026003276929259078,0.7391320429742341,-0.9897546358406546],[0.31337467208504677,-0.7934391498565675,0.33810875937342644,-0.74025241099298],[-0.8158999141305685,0.4479574803262949,-0.3640890531241894,0.5925508439540863],[-0.41567302495241165,-0.24755502305924904,0.8822517711669207,-0.251968864351511],[0.9162301775068045,0.527042418718338,-0.915557598695159,0.41676802001893476],[0.9938023556023836,-0.15757278352975856,0.0775409322232008,-0.8174524698406455],[-0.49435945227742195,0.569903613999486,-0.09526085294783115,0.98225162550807],[-0.7664953637868166,-0.8597359340637923,0.6427719444036486,-0.17292346060276043],[0.2690206617116928,0.3970358297228813,-0.6844182536005974,0.025221893563866837],[0.18346202932298183,-0.6565374452620747,-0.2678147926926613,0.2757158838212488],[-0.6799448505043983,0.06886780634522416,0.30940589308738753,-0.42439412511885166],[-0.08068088255822659,-0.36065300926566135,-0.9525533709675074,0.7327493671327827],[0.5791017450392246,0.8979533892124891,0.9702178668230774,-0.5689269006252289],[0.6296720486134291,-0.9481634292751551,-0.8068461045622825,-0.35902830399572827],[-0.13216665387153625,0.29311097785830476,0.7703826576471331,0.4447174333035946],[-0.6069785039871931,-0.027278978377580754,-0.4759595636278391,-0.6519986391067505],[0.10750501975417137,0.7471965458244085,0.44681975059211254,0.5501884762197737],[0.4440406057983637,-0.44919876754283916,0.6010025274008515,0.7977375499904147],[-0.9425529651343822,0.7941467855125666,-0.571807274594903,-0.8985710237175228],[-0.2968213353306055,-0.5263619031757119,0.18185842037200928,0.1922703776508572],[0.7933955267071724,0.2462789863348005,-0.14533979445695888,-0.1056045740842817],[0.7775688003748655,-0.42610802687704574,-0.7360795401036739,-0.6115188281983135],[-0.2809026390314102,0.830672200769186,0.7180947326123719,0.7138095125555997],[-0.9897213708609343,-0.5989454723894598,-0.043865105137228966,-0.3810478411614897],[0.491117637604475,0.1285396274179218,0.001953439787030442,0.2958392482250931],[0.09204419888556004,-0.9270179569721222,0.9188470263034108,0.04439920932054542],[-0.5916704498231411,0.33158164657652356,-0.8770020063966513,-0.13058425299823306],[-0.17872499860823154,-0.09790203534066688,0.36103082820773147,0.9623657297343018],[0.6763825640082359,0.6274966597557068,-0.34311289712786674,-0.8610514812171459],[0.5950510818511248,-0.7186439260840418,0.37981383688747905,-0.08645109646022331],[-0.0964779295027256,0.05582480318844296,-0.4160424377769233,0.23464868217706636],[-0.6328979749232531,-0.2645280193537475,0.8376227580010895,-0.9184807650744917],[0.13626250624656677,0.9767398312687874,-0.8665282092988491,0.7541776429861788],[0.28435983322560787,-0.21773399598896515,-0.2126045636832714,0.5075746104121215],[-0.7819260284304619,0.5549153573811054,0.2415768913924694,-0.6709012966603043],[-0.44767861627042294,-0.7655714564025403,-0.504765247926116,0.48804178647697016],[0.9472133703529835,0.4777827989310026,0.5410604793578386,-0.33886775001883473],[0.9008069038391113,-0.8318639490753413,-0.45011448301374923,0.9266027696430679],[-0.40034172125160694,0.4268578104674816,0.47070365957915783,-0.7774287331849338],[-0.8624815084040165,-0.12774841859936725,-0.7673297449946404,0.06332303769886494],[0.36004764027893543,0.5977722611278296,0.8118440955877306,-0.2266497239470483],[0.2110254056751728,-0.3308324366807939,0.1422811448574064,-0.4800428468734028],[-0.714389881119132,0.9258258137851954,-0.18686214834451698,0.31573972478508927],[-0.048173688352108,-0.6286693122237923,0.575096292421222,-0.5308256223797798],[0.5496005918830633,0.09869362413883187,-0.5957523379474878,0.6790232080966236],[0.7255966179072857,-0.055075593292713276,0.6813733726739888,0.40466314926743485],[-0.22325424663722515,0.7174517828971148,-0.6477779969573021,-0.30334890075027954],[-0.5139225572347641,-0.9779124837368727,0.09856363944709301,0.6038841921836136],[0.013548744842410088,0.26531960815191247,-0.07229306735098351,-0.6900692358613012],[0.4115940257906914,-0.5561071056872608,-0.97351492382586,-0.9386254120618107],[-0.9129902366548777,0.2184837795794008,0.9473112355917692,0.8534168191254146],[-0.3244461677968502,-0.47699159011244785,-0.3063553050160408,-0.051908962428570016],[0.8277800623327494,0.7643982451409101,0.27282655239105247,0.15419964678585552],[0.8218808230012655,-0.4819654878228904,0.7113989386707544,-0.8655500300228597],[-0.32228539139032364,0.7610103785991669,-0.7407917212694883,0.9673600699752563],[-0.9070753213018179,-0.5591278672218325,0.005053911358117835,-0.12726648710668087],[0.40944797173142433,0.21314265765249707,-0.04274830594658874,0.041577234864235146],[0.011600660160183907,-0.9810553230345249,-0.8800053484737873,0.2901220079511402],[-0.5082970447838306,0.2601006980985401,0.9177663810551171,-0.3767876960337162],[-0.2213227916508913,-0.06017153896391403,-0.33638077788054943,0.7159009575843815],[0.7199573367834091,0.7141858600080013,0.36584042944014095,-0.6150673683732748],[0.5517150852829218,-0.6310797445476057,-0.41911232843995105,-0.3407206572592256],[-0.05411819741129875,0.09469543211162068,0.3786664642393591,0.48842233978211946],[-0.7165211234241724,-0.33519594557583343,-0.8598628770560026,-0.6649378892034291],[0.2169560268521309,0.9237805791199207,0.8423653934150934,0.5001388490200047],[0.3657795172184706,-0.131989909335971,0.23481419123709202,0.7535510268062349],[-0.8643223345279694,0.5956051126122475,-0.21738359145820163,-0.9173736162483695],[-0.4060578029602766,-0.8341522738337518,0.5440942831337456,0.24183075129985787],[0.9026625715196133,0.4227373767644167,-0.503715056926012,-0.09315263293683518],[0.9529304280877113,-0.7598030772060157,0.4738041833043103,0.6725620273500685],[-0.4495351407676935,0.4812307804822922,-0.44899751245975494,-0.5238839089870451],[-0.7876568548381329,-0.21391890943050396,0.8051481302827599,0.31709485873580023],[0.2861997280269861,0.5603165123611689,-0.7720419783145189,-0.4809174481779336],[0.13839277252554893,-0.26083483174443256,-0.18013008125126362,-0.2343257814645766],[-0.6388277392834425,0.9822630193084478,0.14709057472646236,0.06952674128115155],[-0.09859347343444824,-0.7129977736622097,-0.5987555086612701,-0.7772961053997278],[0.6009965222328901,0.0593949072062967,0.574015699326992,0.924997787922621],[0.6744501926004887,-0.09542956575751316,-0.6411125976592303,0.1504031214863062],[-0.173084681853652,0.6322874408215284,0.6861161943525076,-0.0495695322752],[-0.5897232368588448,-0.9225924927741289,-0.0753631442785262,0.85743660852313],[0.08641967736184597,0.3344193138182163,0.09741619974374771,-0.944102296605706],[0.4852036386728287,-0.5943978112190964,0.9503449723124504,-0.6926432847976682],[-0.9875763524323702,0.13125520199537255,-0.972464919090271,0.6069540325552225],[-0.27500252798199654,-0.42351362854242336,0.2660640385001898,-0.29811426438391175],[0.7754070330411196,0.8353409189730883,-0.31113426573574543,0.39992430433630943],[0.791325856000185,-0.5302991084754469,-0.8398012351244688,0.882414443418384],[-0.29083009622991085,0.2439295183867214,0.864373067393899,-0.796229284256697],[-0.9404989704489708,-0.4511829186230899,-0.381473395973444,0.11288160830736138],[0.4380346443504095,0.7898443825542927,0.41434517875313803,-0.21419574134051766],[0.10172832012176514,-0.02938513271510612,0.5065162666141989,-0.4617334567010404],[-0.6051844079047441,0.7430160120129585,-0.5393328554928303,0.35944264940917536],[-0.1263733245432377,-0.9522227868437767,0.21481370367109776,-0.5437631290405986],[0.6278917212039232,0.29088376648724057,-0.2393301110714673,0.6289715990424158],[0.5852003023028374,-0.3639798518270255,0.04556233063340209,0.41907110624015353],[-0.08264424838125706,0.8930404186248779,-0.0002938546240331963,-0.25574454292655024],[-0.6860266588628292,-0.6618175208568575,0.7382346484810118,0.587037615478039],[0.18543928302824497,0.06590818800032117,-0.7159162294119596,-0.7362117748707533],[0.2707694284617901,-0.8648938275873662,-0.36327764205634594,-0.986724745482206],[-0.7723182868212461,0.393953924998641,0.3409037832170727,0.8385272752493624],[-0.4961240142583847,-0.1607776526361705,-0.9205746836960316,-0.006253762170672195],[0.9996104370802641,0.5648688040673733,0.8752510137856009,0.17055699974298455],[0.9179937485605478,-0.24270331487059604,-0.1002226024866103,-0.5639076307415962],[-0.4214802347123623,0.5295759122818708,0.07059545069932938,0.7282108683139079],[-0.8176497165113688,-0.7905403394252063,-0.6835514921694994,-0.4271911699324843],[0.31919851154088974,0.4524439908564091,0.6456222590059044,0.27899369969963983],[0.23289738409221172,-0.7437388878315689,0.3085638489574196,0.02950288541614965],[-0.732309952378273,0.030611954629421012,-0.2705794144421816,-0.1786770448088647],[-0.007079554721713066,-0.2896230295300485,0.9752655848860745,0.9787239320576187],[0.5045235566794872,0.9515261966735125,-0.9455830156803131,-0.8153973687440156],[0.7045035567134619,-0.8937632944434881,0.7694854494184258,-0.1052489206194881],[-0.2060220167040825,0.36520667374134064,-0.8096659500151873,0.19045739062130473],[-0.5548481252044439,-0.06464745104312908,0.4518120661377909,-0.9060316849499939],[0.05830414965748787,0.6611209306865931,-0.46904467046260834,0.8037408776581287],[0.39362827129662037,-0.39272772148251545,-0.5768245458602905,0.5512747485190626],[-0.8911638893187046,0.8641706313937902,0.5940017029643061,-0.6525888815522196],[-0.36944651789963245,-0.565564760938287,-0.14452831633388996,0.43799502030014925],[0.8689508140087128,0.16203872486948945,0.18465363048017003,-0.35180986113846335],[0.8521547727286816,-0.954512657597661,-0.19754541851580143,0.21221336349844933],[-0.3545124065130949,0.3043435551226137,0.22328074090182803,-0.11089863814413536],[-0.876801386475563,-0.0004257522523404278,-0.5225732997059822,0.7942464295774692],[0.37722097150981426,0.7252255957573652,0.5566085577011113,-0.8804319500923156],[0.041875503957271576,-0.4535917714238168,0.39752842485904694,-0.6308954451233151],[-0.5405250731855631,0.8034221846610308,-0.4316229447722435,0.5456863753497605],[-0.19104796275496483,-0.5014572720974686,0.8225311916321518,-0.36136601865291595],[0.6877293232828379,0.22625758498907067,-0.848325515165925,0.4636571798473601],[0.5214411914348602,-0.1824736148118974,0.9366241395473482,0.7382551841437819],[-0.021891118958592415,0.5899212155491114,-0.8925191536545753,-0.589081624522805],[-0.7467950023710728,-0.8553391005843879,0.34600178711116336,0.2577884290367367],[0.24918309040367603,0.3877566009759903,-0.3248468432575464,-0.4211146384477614],[0.33550461009144783,-0.6833945009857418,-0.7209570351988077,-0.16869419254362605],[-0.8320943620055914,0.09084258601069428,0.6998611073940995,0.004391547292471154],[-0.43633269518613815,-0.35430758073925983,-0.061609797179699055,-0.8366649448871611],[0.93489052914083,0.8867246117442846,0.017564043402671814,0.9848620537668471],[0.9832023996859789,-0.30362810380756866,-0.9913225583732128,-0.28091745637357235],[-0.48176029697060585,0.9532636888325214,0.9628588967025284,0.4291145652532582],[-0.7573848683387041,-0.7264941744506361,-0.2912957426160574,-0.7301341481506823],[0.2539745569229126,0.001098705455660598,0.25452998466789745,0.565831502899528],[0.16866575740277767,-0.8046715483069421,0.6662813890725374,0.8134148418903349],[-0.6710537895560265,0.4543066490441561,-0.6295748855918646,-0.9767410513013602],[-0.06832047365605831,-0.22558425180614006,0.11627781018614769,0.17669404111802534],[0.568770457059145,0.5001892521977425,-0.08787315711379073,-0.02752048149704922],[0.6441781651228666,-0.5911705791950228,0.12725252099335194,-0.8018781598657371],[-0.14085958898067474,0.18318849243223645,-0.16859651170670997,0.90416932106018],[-0.619995279237628,-0.38708326779305946,0.5928738228976729,-0.18859514966607094],[0.11864478513598442,0.8540710806846619,-0.6112699024379253,0.10338607989251614],[0.454930717125535,-0.09012713469564926,-0.46785959228873253,0.35385341942310267],[-0.9553502462804317,0.682145532220602,0.4863146804273131,-0.44003893993794907],[-0.3052754644304514,-0.8879931904375553,-0.7522076573222876,0.6546329166740177],[0.8076331540942192,0.3549805339425802,0.7936108876019718,-0.5533181913197045],[0.7610519211739302,-0.03505377471447002,0.32204229943454266,-0.03959435410797596],[-0.2586030960083008,0.6925475690513849,-0.3507675025612116,0.12528396025300026],[-0.9707729201763868,-0.9871883001178503,0.8950856924057007,-0.9653776660561564],[0.47026165947318077,0.26971219480037667,-0.9321125224232674,0.863567026332021],[0.07145349122583866,-0.5341366920620205,-0.020132504403591156,0.6169907636940486],[-0.5729563944041729,0.1916300170123575,0.0570962578058245,-0.7178247142583134],[-0.15664816834032536,-0.4882236309349538,-0.6970584634691477,0.3787115681916482],[0.6601197496056557,0.7707480099052191,0.7257208507508042,-0.2920452877879146],[0.6154741812497377,-0.8206357080489398,-0.5591670647263527,-0.5021827686578033],[-0.1148713119328022,0.4205118454992771,0.5180536210536959,0.6669814474880691],[-0.6557527650147676,-0.1497226618230344,-0.22046814300119888,-0.49046578258275997],[0.15321220457553864,0.6246298532932997,0.2023030947893858,0.34276469238102436],[0.30104432068765163,-0.3215527907013894,0.8455148320645098,0.09129026904702187],[-0.8045462444424629,0.9214293975383043,-0.8272869531065226,-0.2399680335074662],[-0.46584910713136196,-0.6486873310059311,0.43418334424495697,0.9155107755213969],[0.9673824645578861,0.12359403818845727,-0.3930068537592889,-0.7516887858510017],[0.8877217620611191,-0.6917903423309328,0.6138360314071181,0.48284069448709466],[-0.38925506360828876,0.03385444916784741,-0.588361781090498,-0.3190187048166987],[-0.8479216881096363,-0.2708776053041221,0.16579154320061207,0.5258076321333649],[0.35142366774380207,0.9879718124866486,-0.13201782666146755,-0.6744853965938091],[0.2026243843138218,-0.19282996095716964,-0.7908078338950872,-0.9230148177593944],[-0.7000838872045279,0.5348935984075069,0.7569710481911898,0.7753137275576587],[-0.0373525395989418,-0.7699638642370702,-0.48888271674513817,-0.06754424795508363],[0.5367496069520712,0.48705855570733547,0.46334564313292503,0.23234292678535007],[0.7347755990922451,-0.42171178944408905,-0.2517178114503622,0.942240081727505],[-0.23824712447822094,0.8213926143944263,0.29605382867157504,-0.8555738013237715],[-0.5245760977268219,-0.6238457076251509,-0.9654178135097027,0.047706840559840424],[0.026079056784510612,0.14855758659541585,0.9868033044040205,-0.14854079112410568],[0.4239012077450752,-0.920672170817852,0.09043398126959801,-0.4019683133810762],[-0.9233900103718042,0.3203534651547668,-0.11175664886832237,0.30015767365694024],[-0.3391735963523388,-0.12475944869220268,0.6267646122723822,-0.6089975647628305],[0.8367247078567743,0.6494708433747292,-0.6710375752300025,0.6946871709078548]],"weights":[[5.014281232999536,-0.4526693497406797,-5.307393886063246,1.1738514387571755,1.203357672571645,-0.35202035422395545,-0.1875027403447907,-0.318107289803,0.11415402211634977,-0.11441437582021191,-0.417404442355692,-0.11501406193210423,-0.35773002752904803,-0.03962262347385322,0.16312653261655868,-0.17307686782287734,0.18710162219829096,-0.1196153331565651,0.13612626880242115,-0.14402662691958268,0.06615378473584971,0.08806524416872248,0.22707938678772946,-0.028076572061257268,0.14597559705878266,-0.10498421942619657,0.1717368194269021,0.07611522378289587,0.22510716019072177,0.07887956355390596,0.031252057473995835,-0.0756440244882276,-0.3327076373739295,-0.0003684685280881978,0.16552132142445744,0.02979312225344495,0.11290776983528555,0.07465475495152156,-0.029930899979548318,-0.030908077907285073,-0.029985894896617116,0.016863367300238862,0.1437206028462055,0.015019124663193799,0.01299180529509624,-0.18001585632189587,-0.17841489390417578,0.13736876027998757,-0.005205550228787923],[-2.4684385153697956,-0.8939801965896269,3.0106030173282723,0.6748625834202049,0.23706783299055328,-0.3933424736524185,0.5331060113927056,-0.5354430710967127,-0.1614430750559518,0.12528379899522313,0.4294708897196891,0.3024289483706865,0.14807638796273817,-0.04613062626391619,0.013188398549281482,0.34282929843391263,0.42296735974293537,-0.2130543991687474,0.002583337149187682,-0.005228015143655557,0.05891977268118295,0.1153570279734058,-0.12141897292081866,-0.13134218908702927,-0.22836900480258465,-0.22514676343449647,-0.1670104971117641,-0.14869447552453383,-0.24386292170353055,-0.09139237582433926,-0.2999580775883051,0.0018503837531560017,-0.100016454003229,0.06901215287793987,-0.2531610614090302,0.09195254833240916,-0.10622855628472849,0.12192771806111917,-0.057759572281399024,-0.08077922056210256,-0.1660694683905356,-0.059397165992076575,0.038936658320204966,-0.030791027113413506,-0.016500896302019556,-0.029204744410556167,-0.04454719656109065,-0.010631495989517183,0.13345908095988493],[-0.5406660101974821,0.7294484148808255,1.1656705093844586,-0.8985523371146698,-1.3279282368493854,0.5250616774851667,0.20431085628205353,0.5305832599817358,-0.1801717631678425,0.02837001642404787,0.07174176204106872,-0.07141331536399484,-0.49050836737360964,-0.05765861548019763,-0.3251635192956425,-0.20177436602245447,0.013169757819383808,0.18782157313008505,-0.1804111231979778,-0.1241799124882193,-0.2274607414726851,-0.09728784752512702,0.0904358141133111,-0.15123392327844049,-0.20969011954341266,-0.0027446441836468283,0.11580631901036455,0.056920217475384456,-0.004116824995097899,-0.1796505364592417,0.015036602861651182,-0.010807502315037841,-0.0759790084917023,-0.14436917492295834,-0.0011554016970568497,-0.1764922176580126,0.13060948257388558,-0.06976975209504274,-0.09635532936837403,-0.0021615925657535325,-0.048921082887630384,-0.05458534420743531,0.0719552681688901,-0.029881990290806335,0.021924047814624766,-0.054670096398582495,0.04168406937922029,0.06752421335261834,-0.04402441260823711],[12.235379346339604,-7.272376185383933,-9.286647398332365,3.0799012296848116,1.5390194378480622,-0.08391022639527705,-0.027286744856183088,-1.0122313010972914,-0.12555798269664983,-0.42013672529271606,-0.14007601588784507,-0.10348071151733798,-0.2375399120098192,0.2928027661696747,-0.012811390898577454,0.1809500633854769,0.01223950646640937,-0.26271564639752887,-0.48169042444905497,-0.3565753477064377,-0.23656392754404884,-0.3654846521368614,0.40242865231198405,0.3066251288066252,-0.3463869316298451,-0.050947639186671445,-0.024836505418431007,0.4035181759729461,0.26389163306297797,-0.022232967922619168,0.2670783531981021,-0.04990083093545947,0.10579073467521893,0.020680555739669624,0.2435251771909999,0.2840991762348238,0.1351194604910875,0.2550254311697969,-0.019747739751525047,0.20021964829818154,0.2040370531954655,0.08902200645497435,0.18996809390271815,0.13767016415737013,-0.04733803056227123,-0.010901056891973862,-0.26863424695458066,-0.08382236165296136,-0.17242778601869574],[12.45145500249746,-0.014156730195142533,-1.9630408782059179,0.4997490682345559,1.9891104183663353,-0.502860531811002,2.407266689550201,-0.3506907023370205,2.204249068649869,-0.18401299833352863,1.2219228944903706,-0.5046748081798433,1.195689873394289,-0.24660085717518596,1.0876066107951923,-0.5967575373144006,0.7066663321884074,-0.25559445731610014,1.0698722149703264,-0.3299527133958609,0.8117690943020269,-0.37095751051439085,0.5759609751433973,-0.5026546829543455,0.33201108682437913,-0.39936939636768076,0.6457717336186909,-0.42358285022400943,0.47917214076609865,-0.6489303468122584,0.27564791907715697,-0.37011015715387807,0.5813256551312044,-0.36330300394734244,0.3788287661313325,-0.434639825544998,0.5551194485951972,-0.20871976231746406,0.3322061391204957,-0.1879077647846467,0.48584926993714256,-0.14994733705569419,0.28893310698509767,-0.14322518848291949,0.5049638199829944,-0.04103063136339574,0.27003402055739995,-0.10804037844072706,0.27152136751588934],[-2.910696196376702,-2.7499731308443267,1.9003777278130256,3.065534184107451,0.26542983314167196,0.10462215663330335,0.1830993890560032,-0.09206241152825947,-0.39312411533062225,-0.1508370332062928,0.18742147462605874,0.48401347538646716,0.7771043147681951,0.019921314312585776,0.5965876220932073,0.44345005667607934,0.536069092041192,0.21946964440550426,0.4956660933167857,0.08133390105796827,0.3684729647171585,0.13169449990685975,0.27390173995953665,0.05051434848449225,0.3000247705213688,0.1612998511851085,0.17602330330385738,0.23353482927687225,0.5471994743425835,0.13640555784563396,0.16196180249376418,0.33065358798626054,0.2013914005798972,-0.11253547787994204,0.06170361377335409,0.04497011216806739,0.05992023351949261,0.03873939986339195,0.05931853650438766,-0.1852915004333617,0.1679702040569162,0.33011065208911555,-0.014488679069487908,-0.09384875247482571,0.09812036942634676,-0.07256700836199223,-6.384772069003287e-05,0.11477163019242051,-0.07459933918745612],[-14.885018970020159,0.5887639984014733,2.5420530469920872,0.1664586319138968,1.7151025822586339,-0.0018731136641527341,1.0378197698071174,-0.43537118586497886,0.04570651042770974,-0.34047353500168376,0.995098697607411,-0.5233114219856936,0.059803980227019875,0.05345607092882856,0.753074128661878,0.07245295542691978,-0.07351501706180745,-0.27994723675898087,0.608862077324016,0.042940716066428915,-0.24669642231396427,-0.05094089677219199,0.40882163302308994,-0.15418588127237975,0.06231762902009,-0.3175466113950858,0.19739392285280652,-0.06449225828576673,-0.13819996153388084,0.03183681391084009,0.04659390997129678,0.1422938675918131,-0.09924651397428673,0.11081810181591911,0.12385111767531949,0.02120633672029103,0.05249733535319442,0.045205187477938155,0.05395552573051634,0.02722537638051623,-0.09510403817331024,-0.08556103746011247,0.24676220305226235,0.07599282620525483,-0.06880191237952163,0.18013877395309635,-0.0810231966664562,-0.03536899061912328,0.18426890248797254],[-4.877081564715357,2.630095314925087,-0.8964100497574475,-0.7037638495169706,-0.748880800002223,0.4932212747307673,-1.8253159355501154,1.2103673660127081,-1.5147063891582158,0.9167975887329041,-1.1647064911439513,0.7771101215782028,-1.2755839033126934,-0.13257358976985884,-1.137683062348529,0.42179156349384955,-0.9046571612988443,0.5192526557514343,-0.3171889474945575,0.3087777883567567,-0.8361543510636003,0.07067558892297153,-0.5341454590738707,0.6392791984047875,-0.5247854946490965,0.391998894173006,-0.26663382975377004,0.47399421371018413,-0.2858776132430599,0.2773304958142349,-0.3614803341014468,0.2106690240094299,-0.5775216523606148,0.30270655613999875,-0.5487808719699135,0.5085664838971076,-0.6804051072660819,0.36415551118730066,-0.6120941551718576,0.029345230621934115,-0.5204069380360393,0.4750328802226169,-0.676274961439026,0.45084773584531596,-0.46880547465849653,0.104781903951703,-0.4007429884480832,0.2767997897298672,-0.28771722352358586],[-6.244868175235359,0.4291348964973973,4.5931916784462015,-0.08733677993443659,1.1370392261129438,0.05232012454394756,0.7235001792137388,0.4518379204753549,-0.18451898852185847,0.33693715862281765,-0.366056315120928,0.15620787711909861,-0.14575614719809515,0.3132511680914285,0.09922324914903008,0.014946215119292209,0.3249426038002795,-0.051962832409241926,0.2541906988171703,0.021343415424954812,-0.03287438512981496,-0.13643175419422168,-0.0917929629161518,-0.014434817366662397,-0.00621761774645412,-0.16566841828897158,-0.10651855465246049,-0.1075740708879484,0.16341782222364262,-0.015503061676143215,0.2631589075179105,0.11894922847996105,-0.006692459207409478,0.06899354279362642,0.05837255979579828,-0.10771235797542351,0.1385881501445713,0.06410635030599149,0.07068836455940532,-0.007752859648760222,0.18961231273218312,-0.15723359981952295,0.2526806682540607,-0.05275899241366036,-0.04085464375777859,0.10265791809183551,-0.03919321082192743,0.017548277149243997,-0.010366802500618185],[-1.900564287744791,-2.1681638946625292,4.722087104563276,0.7146708999701555,-3.3529815983436255,0.03775236396546053,0.21832068095103224,-0.6354729904281226,-0.7390534767038972,0.5925449196248187,-0.4270190047654771,0.736590510551874,-0.49197006832495527,0.13364647087162726,-0.12201138175165,0.2274973780009692,-0.4094727015772286,-0.7387403502202543,-0.04041614040055359,0.10816305968009772,0.2751478840402719,0.01927508836052139,0.3280135060226229,-0.2847439733399392,0.5346002092197403,-0.020963305446682793,0.2904414305607524,0.06888152224234173,0.24199305705927007,0.03381273412069579,0.34643741553789037,-0.2441207368741803,0.39053172486913323,-0.17141824588452825,0.3281981899991962,-0.1381443061361342,0.038092571963444996,-0.14247051202784491,0.21524292440038142,-0.08132097858897928,0.011664252958879783,0.022352851988663203,0.1870438633504543,0.0508454966819388,0.028699085143448943,-0.0340040371348384,0.0757428281268848,-0.0810783877734694,-0.1895368369638649],[-5.533226582066955,-1.524804021622351,5.335575277043685,-1.2355211406762723,-3.221636356052902,-0.3946108190952627,0.0544601006595918,0.4426018598964837,-0.21695887538693895,0.01693705807063151,0.21553202841533411,0.26201687345019964,-0.11329943369669412,0.23597703388554742,-0.6598252039287206,0.2226658342796947,0.052077475808722636,0.24041114095464863,0.10006547532108695,-0.07171502977340194,0.12460837694765746,-0.14498931001771975,-0.1448753435423225,0.017586205247646283,-0.2666475339563588,0.09674109258803666,-0.09179555670545242,0.16113570345778475,-0.2029870854313703,-0.14318538077581885,0.04641907768696561,-0.11860072188872472,0.009083350124279764,-0.20934887820820025,-0.06266651949048582,-0.001353692614269888,0.05800214094441686,-0.190104723461432,-0.10306486343407631,-0.04172752826432637,0.05746823951753094,-0.21246907268872992,-0.0405578641239034,0.027253108702261282,0.17939990460504734,0.07765314539383578,0.17657236926953204,-0.08132578859681378,-0.08700572915947619],[0.2565737231801121,2.258890756272005,1.6311290517448216,-4.442050510322743,-0.6489576360689608,0.656971593816813,-0.42267520159228406,1.2113809018943578,0.08695944177760374,0.09732702290594919,0.41394833325058816,-0.5576246311114856,-0.5790530159376899,-0.005734286686187907,0.15552830459389874,0.07173054599514965,-0.06351193679825283,0.2238555149036485,0.36695737239052684,0.2902964180274411,-0.2494576772087007,-0.1777570496765326,-0.02849425679445234,-0.2819036191375433,0.02534136913813472,0.3735083023455958,-0.22423146347380665,0.016494527723337562,-0.1465688655018775,0.0360174629435684,0.06907403938484029,0.21844811555804042,0.002212854111818643,-0.21034068272682013,0.11989102347986821,-0.14094894262897403,-0.0032199863324083766,0.4225565313916607,0.19010230743063614,-0.16902160053273352,-0.005962142401291873,-0.08128665931636464,-0.026301576982131796,-0.07766432831767789,0.18704897023257755,0.24996510172534536,0.037017225856240094,-0.012981306304351998,-0.06963289710578559],[-13.107305987367612,1.0982559085361423,6.488345318384207,-0.05331477829254191,-2.2844097233026734,0.13786250453645713,0.4831547832550505,1.0957793712253283,-0.5459063239605451,-0.537191902685239,-0.40983400971429956,0.2717468789044686,-0.4220507852483724,0.020507820976180445,0.068518769587875,-0.028978012412197135,0.27307021613146903,-0.2013801208715653,0.48340748575262893,-0.27965454787695504,0.07142340803508425,0.014209051923575562,0.4281556163344069,-0.0732440718884951,0.6518494996573202,0.08617360457362518,0.5149063240473843,0.07910046427373618,0.5856671674801699,0.0890755805764848,0.5567649989253657,0.09749220197759646,0.7599218696850414,0.04731056804545511,0.09248262619596212,0.17814027078030567,0.6700991352570217,0.22556288162951085,0.08346263400342964,0.17578854470861918,-0.016285038635829703,0.12824580658948906,-0.11394215648865816,0.09274080702835578,-0.006572313893953637,0.1449109779223516,-0.22549977740400146,0.19546035749898935,-0.39574587571867625],[7.09761402769758,-0.10144453190011385,-0.20475033091495626,0.20917879161795594,0.5183191430267113,-0.10006390469253004,-0.599702818966521,0.024079519542501083,-0.4288808913487509,-0.048351760870964586,-0.32752177933174814,0.2185622135479604,-0.01120495311655,-0.17381126107063422,0.1128581103213998,0.3111301441525722,-0.40171731629495383,-0.2566977367403099,0.35678153972821736,-0.015005022754868733,-0.5938167933544156,0.11364049192564973,-0.3178793116832754,0.11814625788171311,-0.41297991553394897,-0.010775338447249993,-0.4044879382628876,0.0983281870662291,-0.46540208830703744,-0.11529517470383566,-0.29440827400950353,0.18210247939506577,-0.06830281407082103,-0.03813182710033386,-0.12738540335920384,0.011343412762693038,-0.19242469648076438,0.05187848842887296,-0.1340164818651651,0.016948515779883103,0.008733473663545422,-0.03189402104666665,-0.04067489346697881,0.02325059095917401,-0.1953352970060922,0.0247478600086966,-0.08335536134747241,-0.010254466247690068,-0.12231828352897833],[5.40088494158457,0.988182429432007,-5.115510779453789,-0.5526274857742437,-1.9131101348002648,0.4237395519379737,-0.1007501182794675,0.3111437444125695,-0.5484304625112958,0.8417237934994597,-0.6067533727026643,0.19042957582144682,-0.7681571676717349,0.3224989988182619,-0.3262014096191512,0.1197442913209028,-0.3056918811783688,0.05442285753251123,-0.41542309906031616,0.17035406174496429,0.16921147732498568,-0.12468320893842909,-0.43955761202512933,-0.00029754854854955725,-0.4392340114996773,0.16239289972599433,-0.16393501610383554,0.04962912717725974,-0.29551492625700204,0.24426580584171095,-0.02293864227912392,-0.10202092829242244,-0.0785145376014492,0.12048260049755559,-0.09237262525830814,-0.12324002671980787,-0.05391636811469638,-0.0017086382361631153,-0.002072242679466047,-0.07473331971712006,-0.2656151510916928,-0.052433898123322886,-0.11837739809584268,0.10184079984872456,-0.1008508285000695,-0.03529665880173969,-0.016885062029007712,0.12160030215960028,-0.09259005877741545],[8.516594363305142,0.6374241025949854,-2.246899031766504,1.071166781265323,-0.015197181193167616,-0.1182659924364104,0.07348191243262332,0.12843010490202453,0.10448507299003902,-0.1389831537914231,0.02317579510539154,0.11580494448826424,0.15569099668883435,0.09545573090620814,-0.12590305410135819,0.12367042372635445,0.09331574677315219,0.1718382816341158,0.031034287790850865,-0.2285177420878025,0.16199685419881182,-0.06351798052464418,0.18870671646664455,0.011575672141253273,0.15988913659799725,0.0957010398650534,0.22327729205027977,0.12004076067266793,0.10267279792025501,-0.09283196294661764,0.030825425450313557,-0.05818562612907906,0.21676748709249885,0.007579422773673771,0.04490112127538672,0.004999287298561143,0.1983770821601609,-0.02616012918121301,0.17075177733586805,-0.0007765770141501421,0.17352561389948812,-0.0029278287408201673,0.29801867047912345,-0.045296193816397205,0.12035490437739102,0.06024615666219598,0.08782769493612062,0.08848673847854596,0.11539389807834846],[4.608462422715419,-0.30142800098480144,-3.446682403204157,0.39313507675156195,0.3370325706419398,0.04856210025757001,-0.7101919925686,0.13050249999523195,-0.3486547111498382,-0.08120805845558199,-0.29891494201488017,-0.3549611300036484,0.16009934262515438,-0.191388316179595,-0.17837634570676875,0.49782377446176723,-0.5749567484606125,-0.21226785095683037,-0.30214101062479426,-0.07062925983007658,-0.1583280120565539,-0.07074495695731685,-0.43718436720003995,0.016476972728623213,-0.11524837350407334,-0.1384261521813982,0.10161124790728959,-0.07607581065494697,-0.15107206841558699,-0.17607499926621523,-0.13359362323754348,0.23100027751882055,-0.1468458519585924,0.018537211931849873,0.007317478161404571,-0.05776065485743389,-0.35673426979668194,-0.022121601206561606,-0.0971842926580275,-0.14076286346555544,-0.14762041860321487,-0.2468043336141138,-0.13204563120651014,0.043641766483790846,0.010422248197339784,0.02883471568178655,0.07607277503378873,0.02123278680048736,-0.22560717943716607],[14.375596206814592,-0.05555376764644847,-1.4527827244768567,-0.26218457246218685,-0.014205427608275194,-0.36568799299783333,-0.4581877551507076,0.08185357734761713,-0.2749698743656687,0.3536741559967882,0.12137965783240826,-0.7796881010361572,0.1168760398569685,-0.2890509998740959,-0.47780161968409984,0.2064889018059326,-0.28946646459745556,0.04092895944943142,-0.3372975495373697,-0.12708996441454626,0.12392812233003046,-0.10605755584339834,-0.49128958325049465,-0.1449559255907806,-0.545400875160627,0.09741267547233093,-0.5032273781619727,-0.054957448033257396,-0.04260748713050723,-0.08124294161861571,-0.16916033557114596,-0.10147599703809808,0.055829157870766796,-0.030406982702426325,-0.20653106299455448,0.12199479885598309,-0.1942693940355983,-0.25191379675932646,0.12007695467619262,0.05629614800036803,-0.027595644628314937,-0.06520015913651268,0.00797359302359995,-0.015745044238106166,-0.08818240116660664,-0.17409040018941815,-0.044488595626616036,-0.24104722963527211,-0.046503397958527995],[-9.873338160602453,1.4221483725496005,1.9067496852987644,0.3165167044198084,-0.8017194197673677,0.23967343188492762,0.005246537816538725,-0.08766292692700368,0.5517291611671554,0.9960794268163211,0.2742058168911647,0.363681174524426,-0.21425925537446283,-0.04615851859489771,-0.27993697268721884,0.23360224947177305,0.13277453490750385,0.3500324849316772,-0.07247416342838348,0.22272164972497777,-0.10718257278008399,0.0759482990148623,-0.2799597391526318,-0.13383508619371778,-0.12566817352663937,-0.24798071529444649,-0.013774131220482785,0.08662085810186253,-0.2560973507174076,0.045999579261851005,0.15594135536512446,-0.02247076007997606,-8.053237907948645e-05,-0.05157250816174599,-0.25779438456191556,0.07306574531001493,-0.11692730046753148,0.002347515511235893,-0.1222775279287564,-0.08691628157345051,0.0237720685812452,0.04261756240797909,-0.05013651461640766,-0.04859960889425068,-0.07578727808793503,-0.1348985396530169,-0.0578795305823085,-0.05374722885716933,0.05944836250007852],[-1.0414143312732054,10.19342547613373,-1.5095867742025344,-4.95232050595206,-1.818616589958189,0.9800463683947587,0.7457807616746863,1.7712869699877234,0.8296690230765672,0.3725768530077954,0.45295662085112326,0.17487422174271694,0.3736062456259916,-0.7000938398986208,-0.6800632178258852,0.45602077130008656,-0.16039329396675622,0.23066511024327593,0.13052167725150865,-0.0756841280968312,0.6028738143479476,-0.13571885925311122,-0.18268724839928996,-0.028335730227157085,-0.21296163372599006,-0.38656879178772685,-0.4182607108400881,0.40408911687216054,-0.0574365715738162,-0.30809618034420655,-0.18511470546423187,0.17015871385352538,-0.2969086406638695,-0.3956708092886217,-0.19547568303507679,-0.3162316393176283,-0.1527680557390052,0.02663257316365463,-0.23828241307978232,0.008762889748647466,-0.09038833273333094,0.3471017168486274,-0.15446146735414548,0.051089627211002166,0.22343795149554216,0.1742368512724102,0.00011795479722058198,0.3013988251691967,-0.12202136175390092],[2.9941738332883174,1.8486490036082837,1.4450779148226478,0.9135178539845533,0.7420935367955607,1.3466417044403465,0.746038021961515,1.2269529758487498,0.375124969300709,0.8014027332052664,0.058674580504567884,0.6873170525309996,-0.07655194216514383,0.5560981341618592,0.0058092105864353824,0.3312913490069285,-0.1780996368245739,0.1308175065954638,-0.30797161395846684,0.03284148745151872,-0.5185273229068699,-0.05428304708501683,-0.4760215397922087,-0.11822749686760384,-0.412915211546922,-0.22116805697109626,-0.3863047475005657,-0.17738073259647133,-0.329077472382621,-0.29499496541890435,-0.28835669725863755,-0.26064617089049863,-0.34453409241030764,-0.3149279461632007,-0.32843271189265955,-0.28831416218327816,-0.2630366927948575,-0.29531433499230053,-0.14744860612509122,-0.24871241333707958,-0.13605083809914648,-0.1837516313141399,-0.08323773180655016,-0.11724482803138932,-0.02316185505737336,-0.1565960695307369,-0.029773760787525794,-0.06809871005316438,0.024893326262851862],[0.43639856746469463,-0.6925806544759885,-0.5239376329797626,-0.4632192490764852,0.1510411015694563,0.036159827583321195,-0.3375833637836457,-0.15671778150469673,0.03915182217529011,0.17064001352855218,-0.07537041629913366,-0.3214253560462974,0.04858023997539697,-0.3362098039500741,0.06423988441098033,-0.25330834431725924,-0.005085403093344098,-0.21507983000240632,0.12017711646915182,0.18123869944952584,0.006949125758326959,-0.153502886098174,0.09894995249714958,0.036328596407539486,0.19052316020302945,-0.06055045707666975,0.2016765599817317,-0.0125776937743456,0.17445461608851448,0.02106496653059912,0.2903708558629328,-0.06982082977247693,0.15902327499709343,-0.11171638691501816,0.17521768053232684,0.11678451175853959,0.0699868413385076,0.009915219708762963,0.08708750052630401,0.04129499909549529,0.14902247530902848,0.14989487971678975,0.1709889167399946,0.050231894926189465,0.07364896890130439,0.13995905524085062,0.07429896668642004,0.02231547770904177,0.06592874358932861],[29.851447227709965,-1.0624823032665867,-7.041300331932226,-1.3248832356658926,-1.7008657198146706,-0.3454065231641634,0.13197259770319808,-0.9196772133959653,-0.04319999708447764,0.06756975511846125,-0.33716864593512574,-0.04703719895227454,0.12779710444078773,-0.3974052472401834,0.02488607226881942,0.2855645985041325,-0.03157761801714124,-0.34526591867725304,-0.018263407219614176,0.2616786040981845,0.3429412032423394,-0.196331242893521,0.07169608372262323,-0.15081472371659205,0.07719146709477273,-0.08606016214795485,-0.3981543717794172,-0.43011896528180293,0.2372622884560134,0.019941601610268374,-0.044104382056157514,0.3368691130748737,0.37790098279873463,0.3395961965929815,-0.07074647839027143,0.0034675268120580188,0.1420709986606285,0.022543173462576168,-0.0011421809677274837,0.23063376344727815,-0.041446654647054514,-0.08409688288654318,-0.08353684396752221,0.040845439532982954,-0.01831227971672126,-0.04945367739506769,0.032014876522198914,-0.004524436765900504,0.09854725185910648],[-6.3185633615989225,1.0379315483112432,4.486257379741099,-1.1072489237276582,0.5699766910620322,0.06746647270981398,0.3251153680021412,-0.11188436760458076,0.8931888801846968,-0.775523226633655,1.1665829923804683,-0.3988413094844499,1.080508924464558,-0.7718243139379252,0.47873549995279646,-0.06225140552882944,1.175622061728746,-0.5815430245120141,1.156762560410593,-0.4514601089319991,0.9376982025192651,-0.425640291508711,0.7323851112369805,-0.3848165630465621,0.4746820114768301,-0.2795047011453917,0.7131710782646639,-0.389432649400544,0.6919623904348755,-0.34563753461961066,0.5555170389305345,-0.2443808885395695,0.5854841058848416,-0.5058502434772856,0.5525603227377712,-0.35788067527800815,0.15316799145358215,-0.43694032404760663,0.5684451840344124,-0.26677168394626094,0.39537019431528614,-0.06596249107418473,0.3541371631495557,-0.2568811581308637,0.3562996868986289,-0.2974321103495996,0.5246901544385942,-0.22794215523187117,0.28597617040280865],[10.48311319833307,0.2228147016885127,-2.456891164860942,0.16269399273768517,-0.2827595928429926,-0.24241903666204961,-0.052410918696081187,-0.11490626454128983,-0.5501230218892659,-0.06122297535378767,-0.1782301425609954,-0.2739897507843026,-0.1812343423711932,0.13281634373713322,-0.117778836620909,0.06378811611410616,0.006569648504210745,0.009309243625105898,-0.1223159899866171,-0.04671241039735655,-0.11962297795086127,0.004643906519628327,-0.09127490713377663,0.05613715066584907,-0.13341318951192535,0.0010311553435960903,0.015211700111025319,-0.10516985548036765,-0.12099140681439008,-0.040658799931154364,-0.09602457916756031,0.012899767915107592,0.03673414651810027,0.02924236168210052,0.003840184384712505,-0.018707178619642696,0.04554411207174013,0.05609341199025274,-0.050956951938242484,-0.03619243436330452,-0.032420548390085326,-0.005399638273478838,0.03596089078867319,-0.04622883417252079,-0.05942605669399031,-0.12346572881829736,-0.04391414684192114,-0.02592508087239473,-0.011149266002011094],[-2.381082944311457,0.5110611858450035,-1.7267241398388256,0.6132994623937081,1.1601988696940513,0.11316864660589372,0.08813961974298301,0.39451363712576176,-0.09612026804719294,0.3928288140965161,-0.29580295850144966,0.2940428944021121,-0.2004087039903324,-0.21944769978038672,-0.1255437000776112,-0.03655641493824912,-0.2613611914178802,0.09859355532629516,-0.04187010188438096,-0.04197349300809002,-0.04598099548733716,0.3127270513807257,0.13833755923528698,0.11809842294807528,-0.016696416920201986,-0.11978084627604027,-0.12840187481539772,-0.0004927637755723843,-0.15763207886026542,-0.007293794820757087,-0.22577342212423118,0.046218533536538826,-0.0624288214840409,0.04900057310091457,-0.03093670708502949,0.006706159353947704,-0.009489151482787558,0.022195708915647715,0.0734694441057394,-0.014940562958619669,-0.02369195610204194,0.027866297124150667,0.012742346813208266,0.006548369885373468,0.0504928047375103,-0.03995196251017374,0.06069393507553766,0.04641079697642686,0.053213855782519186],[1.7786110635205015,1.1619916431938226,-1.2811918640213051,-0.9476349603243902,-0.5262553399010543,0.5669719898825283,-0.867082927859605,0.3854177596935765,0.5132601715313279,-0.30142568265759934,-0.053146988301662416,0.20750718056506076,-0.23556837427249755,0.17225097054963917,-0.20228900502605648,0.09783543404691815,-0.13784451868355338,-0.030660313552390482,-0.13944742496248466,0.012853260072463594,-0.10510810705546689,-0.017322584915231296,-0.05144639855613978,-0.05189156508284446,0.11926705161505352,-0.05835441788422815,-0.08008234774427672,-0.01946076400263424,-0.11631835807418976,-0.03674155753971586,0.2665553431019888,-0.09263374411745265,-0.032262209553409395,0.007992995339757994,0.027292543793372356,-0.11411447658549174,0.15169584743511202,0.031426473251827396,0.19557466409577107,0.05176838223447272,0.031971124185078525,0.07840870390115783,0.10560809048070496,0.25669240052589243,0.04461686529678335,0.06962610668733502,0.09143938670087258,0.09213389533319674,0.03963707449499501],[1.396302289206498,-1.545081172741025,-1.8892219759431075,0.5371329727520446,0.6039257971761778,0.23205664179671218,0.0037197830803464833,-0.117554787595725,0.004693939778823692,0.28567142449739324,-0.5424184811791208,0.46942583503317337,0.0015287735807973707,0.09057217768322127,-0.16606657155610755,0.004938096694955511,-0.08624701181815166,-0.07921372733288914,0.30771761301598854,0.3109942833973955,-0.23161923140236013,0.017074290435877826,0.1961083665105691,0.03377027989588898,0.1686481673584592,0.14762704017517908,0.19727806113691082,0.03063342504703408,-0.11769043605034964,-0.25217553592257697,-0.22732788387650366,-0.4561977251070135,-0.203334769209635,-0.37571597172860344,0.043787154026634266,0.033243841726325433,0.15465147961179349,-0.025618050107472732,-0.16396229721216668,0.09670339459422911,-0.17790765742914547,-0.2117451694289317,-0.1504305307172082,0.06296872250112404,-0.04968462761080132,-0.028581763476194594,-0.07430292183989579,-0.019122525409947744,0.07690959941758949],[3.810617987844623,0.05329564721050919,1.070784744677837,-0.6832458537066007,3.5163104428089347,0.16884875544065336,1.1625839490831786,-0.3450003105190921,1.651822058387316,-0.14840893382684947,1.2190277614864378,-0.0047737311374926265,0.9706638476473906,0.1280360506590184,0.9086323529544805,0.08220396530314847,1.0629198643307274,0.37044055946146814,0.6045462553166386,0.0033116122235241617,0.9243648429275996,-0.18042863811707646,0.5904574013537912,0.018954301885388082,0.5056396684415315,-0.03661945745633753,0.3905614830511725,-0.23638249384038965,0.7155341537112212,-0.2855034741103305,0.292822698693087,-0.2795625194535789,0.5920558711988914,-0.024127936064442083,0.8433013746138336,0.034452089282562534,0.6229235576954407,-0.1417259603356485,0.8856549329448611,-0.0943340088511282,1.2714774554508101,-0.014646975776784248,1.1616574007318863,0.22202068822357768,1.0211524646590864,0.2588474806223174,1.1386361906913174,0.1808771342047205,1.3588978884354588],[-0.08043677140431515,0.30937571061977576,0.6982436593412928,-0.11329369039870334,-1.147853889868083,0.6048747946954914,-0.294938016808283,-0.28849180680620323,-0.5491244123029763,0.12556214494653675,-0.2699912604447837,-0.17185638252083543,-0.017884842733897836,0.018594542009398696,-0.19902899651631673,-0.021164016723306707,-0.04833738514539849,0.1685891850416087,-0.15291768482885737,0.1400253598165755,0.016392424538278244,0.0687200017295339,-0.194255220788406,0.22877302959962378,-0.05372812280513654,-0.07610615183265403,0.07126469136180749,-0.14073213983595617,-0.0069589428140884,0.061074206038487996,0.008661996440354891,-0.018112810628716728,-0.07144023991398875,-0.03281670811663421,0.062072118474181995,-0.0030283976155494998,-0.1445806680115205,0.00817958997624797,-0.04582194945408857,-0.08954684072668082,-0.0187244117635397,0.018094864407888874,-0.031334492860762495,0.05472695870247389,-0.017786429207716156,0.0656105786394858,-0.04754358668552789,-0.030969280735979418,0.026095831617207785],[-6.37231119526636,6.131318133582365,4.060189790707913,-4.426558182561243,-2.3433116882772422,-0.587534225579644,-0.7915650057135218,0.1262622391100555,0.023873933972712973,0.6805966898425561,1.2412888391579162,0.45093600306396114,1.094460537724632,-0.38796719907748745,-0.004843297493632567,-0.24500671393482673,-0.2207053976571082,-0.2761700523133828,-0.05988968803293575,-0.059573360327757126,0.04431921886842762,0.08937698968609979,0.4004811790267968,0.0918381657116009,-0.2636116578274636,-0.14037878764750433,0.32661642364075416,0.18515763171797492,0.05623770292783196,-0.0655783052357299,0.2419676057377725,-0.09219493700350957,0.21179225766358745,-0.17388929097837902,0.04092070695579119,-0.00045353546010776957,0.04033712715963545,0.03337491384482646,0.14286046934146598,0.09777135225389151,-0.07421800921010553,-0.12542538736606915,0.002695614896326689,0.19442977205796327,0.14015047695540242,0.06878211936683146,0.1807192379315058,0.05404403450176418,0.07265207360783571],[-6.590676682258656,-1.103700531984108,-2.8889641274255204,1.479547914856812,-1.1006138558871885,1.0712087366208072,-0.7427699070336957,-0.2299943105297509,-0.4705978934604902,0.3956674537894547,-0.9147482973962261,0.1396013711768516,-0.49185169439009313,0.2907955313157038,-0.7348603179878376,-0.06114567908848419,-0.17992802150011863,0.3966578783601375,-0.089607564617489,-0.0919121263963347,0.27464354740022234,0.09976055407151388,-0.18168199397372,-0.09918138280247468,0.10240969752571957,0.029124032344037733,0.4709610350116012,-0.21833897692352816,0.3378330651323463,-0.01989322536727424,0.2260788303020244,-0.26870371057614384,0.0629470777511204,-0.022703924203958354,0.08572131510228496,0.04806327478105518,0.4816062262385952,-0.01751418688681309,0.17212563027395258,-0.0017272995851362015,0.328994434853536,-0.29445364788488404,0.4373821594134314,-0.14049605963183753,-0.03198468738886324,0.0649535557181772,0.25340418567766615,-0.013590870822618619,0.29148956789511143],[-4.535720549066132,2.6429297305532993,-1.7576568459266266,-1.413778529020215,1.5123792233113043,-0.8378211291426694,-0.07009419324612111,0.0372442575340695,-0.46391088549932236,-0.2077774075759457,0.5458151704471446,-0.009414016325874342,-0.27282687889579976,-0.28646940515062636,-0.1094314065548422,-0.10453277313519668,0.2270777578020058,0.06885085522716593,-0.2422897874847201,0.34120008424277043,0.3550720956096323,-0.08169065487457111,-0.2608155468565058,-0.4261460078889091,0.05803694258790486,0.11204929396384328,0.011023804619851993,0.3203736486437282,0.03633184092333166,0.12191892481818867,0.07443282265724946,0.25103529375482253,-0.14748731934577405,-0.08990974422796504,0.16791358071887436,0.08090860240797064,0.03880881341371363,0.03519028536316566,-0.16778663226468893,-0.018608332613107185,-0.006626679921689602,-0.014371958506855503,0.09109304791790827,-0.21700157507878623,-0.05615084279647847,-0.017162170610384265,0.03442262588878835,-0.11900088813373129,0.06880773309664558],[-5.36999242327677,2.005069345149996,1.9868249688482678,0.2262622652003756,-0.9295521909592322,0.5757020401849331,-1.0499691660780517,-0.32292790792615433,0.06430788336604874,0.08154589277641391,0.1061153556956663,0.12282473838018551,-0.45203959500879826,0.24277126082505004,-0.20541243980976243,0.03291289604894472,-0.398501917104691,-0.16188411307003017,-0.34523840612737383,0.0008092494917892801,-0.3106899659210959,0.05770038685154721,-0.0503948532815551,0.10354372624378487,-0.1149844840677594,0.07413387682309626,-0.13101170991535838,0.032416399293040556,-0.2239472786137973,0.0785718461930696,-0.15148429560851423,-0.08247239430080514,-0.010585649093236445,-0.01318923628389277,-0.16911605976614003,0.08287656821593585,-0.36768468595263615,0.03406113506760104,-0.13428032992553213,0.01451248171544774,-0.06670998708968139,-0.05271354409582176,-0.22033371962806594,0.018764699718979778,-0.2253950238608439,-0.11961424324328344,-0.09166212690358143,0.018132699209654234,-0.09501124052177597],[9.258195998845833,0.5431192046039736,-2.2380946171936635,-0.2295240558408462,0.32756380141999275,0.06418001361007192,0.20580270340359344,0.1020358706104597,-0.13506338452133793,-0.2858156357247683,0.10073149157670715,-0.1391108403294337,0.1996370116151393,-0.15818664598061552,-0.05727904387023875,0.03502640133603923,-0.07921175310338738,-0.01879963229007585,0.051984207740737226,-0.0777162187239554,0.002927775740180044,-0.051379568974565075,0.0954335871357229,0.03823849618482949,0.060020715923154194,0.030902989285666628,0.022831942822412504,0.04809018194206734,0.008069329385187287,0.056824084577463116,-0.02043941116867351,0.012629015233194733,-0.0022906423400293713,0.04593769222791726,0.059361008624598866,0.0773413569836411,0.005728751166181452,0.06747926064125313,0.11235853422811741,-0.025295165807873936,0.09788912086451995,-0.007191841474689799,0.01654755592669848,0.05438744487483041,0.01746731637611013,-0.014232378599293143,0.03936867295390045,0.03166232652703151,-0.005508757318994144],[7.0677165816205,-4.102062089804602,-2.3429622603347067,5.379439818939144,-0.11181653659407724,0.4846223824665217,-1.6404181020858835,-1.732841389109969,-0.6031558230584442,-0.5194653055518605,0.201421252997626,0.4781648380576753,0.6262588469460277,1.4973571912283774,0.3075950445221449,0.5104663667094705,0.4823354867336179,-0.09852803261478808,0.13793154063004118,0.39526140369595875,0.7764681791962812,0.5939682032795394,0.7226516662281562,0.6517609553608319,0.4103686748904645,0.6657107384503459,-0.1849016190993801,-0.026748103977270433,-0.0466777495556274,-0.07867635594694965,0.05135799034151011,0.007746702265153879,0.15192389130066933,0.10439706915008054,-0.19350996193677925,0.0525890522969574,0.10793167063859249,-0.029639161859968347,0.2380473800113264,0.07965887201891406,0.13337498427082722,0.0898619848051049,0.032618792584671936,0.26498944358972526,-0.020249304101392116,-0.15141163717526,0.05518260923940241,0.06609600992555631,0.11502364472984672],[7.978100994978331,1.361829746301561,-3.1599228759159548,2.6274529409312914,0.9854916521373629,0.9252190058743514,-0.212031804024484,0.7530937537103295,-0.6560936121518237,0.9948690019390837,-0.4662185597415265,1.3374715202184668,0.019173035749868374,0.5179989981950033,-0.07633040695319619,0.772280868183078,-0.2317004370755063,1.0155433802705829,-0.3136814922656362,0.5907116527568654,-0.3235209715621432,0.5480487718273623,-0.2052213291375476,0.3894144023283398,-0.5265690685741647,0.2189632400625251,-0.3917472216779231,0.37377297770164036,-0.539194915327009,0.11364348570236248,-0.5610672706969567,0.07578750863593971,-0.32974536840843366,0.11080647676585004,-0.5167678241371692,0.012250217479508444,-0.5654956956905413,0.10870958121301447,-0.4758983085307221,-0.02882013251566114,-0.6050698204312899,-0.14133008006646253,-0.47640931786116,-0.1487670160150073,-0.3631320275538683,-0.1583852303358774,-0.602305457280033,-0.14656997124879467,-0.4713322050738888],[12.485565997859036,1.8894640071698123,-2.321188598958014,-1.139418822228868,0.08669130741200336,-0.3337066272619747,0.2486039930937783,0.013583890254022225,0.5541868082838681,0.015141032313918812,0.4851372885522299,0.06582896657755077,0.02046489368308534,-0.3097616988221736,0.5198357310501928,-0.06182620127782591,0.24755434002706492,-0.2429925247491331,0.0922426148846919,-0.2528905770788,0.12001988037022564,-0.21525042291602103,0.3912328067004558,-0.12634129879162756,0.2819737706907731,-0.04361311838390759,0.04245000399080033,-0.1531178056212777,0.18198591141944892,0.14138972552007803,0.1961457159433754,0.12674238623976042,0.28086504968151454,0.04531088186794806,0.15866270686586043,-0.03858722930351854,0.2589586018516068,-0.017057299195314096,0.15650436992186537,0.02345914113765996,0.007879113643078707,-0.023541603800809604,-0.023776418484612875,-0.01049002297461529,0.14140461350652953,0.03338285268025114,0.06042470075072106,0.05200571554955676,0.0007257618776781576],[-5.897568160484643,-1.0693036983797968,1.2328711797365726,0.349175388719977,0.7677511613848729,0.2377663986776424,0.742905025875198,0.37868518313872923,0.40025354954379916,0.19117960176079646,0.4762107988294357,-0.04063837034687323,0.4934266395056809,-0.10935416632626868,0.20184986397186913,0.05028565361901397,0.16848388524574823,0.05339414713357195,-0.4703776675402423,0.04724854186724135,0.12925785993386582,-0.1524291187928066,0.08833962484303749,-0.20478175510251054,0.06863985709208931,0.09649990078981154,0.6196440232841257,0.12973804048081458,0.007475882454284841,-0.0893684805674606,0.10307857824196881,0.1471153548565893,-0.050542629603452925,-0.07653230326132819,-0.007596664126698377,-0.06044719557042332,-0.18293000530418546,-0.16040376548914417,-0.12179869574291466,0.028504602338103074,-0.0844323531128993,-0.08484759256365007,0.08858102986584639,0.32340452332765646,0.037850122178263015,0.2202768126270449,-0.10396594893253262,-0.0045539807580964,-0.051183854160593276],[-4.004379016805756,-0.002192637762120789,-0.840116954471469,0.4060432086144924,-1.221577004322075,0.3367897517431251,-1.0976522973915226,0.7066487817705961,-0.6664612375045408,-0.09653060667316092,-0.38511104290845116,0.34351802852193747,-0.7371116239098724,0.35319912618280774,-0.7092632214014468,-0.02528439830315105,-0.46709645747847867,0.22839362980019634,-0.7123830214699721,0.19535471477002755,-0.6150652787733679,0.22284084975605328,-0.5076838584325726,0.2169795285555363,-0.7818577314332357,-0.00238624779315828,-0.5639904105886161,0.1770392438788926,-0.5359237668539608,0.12563531112622228,-0.58669276075142,0.1351169139583737,-0.7185295200338045,0.15914469627946307,-0.509863534726878,0.16065907730415843,-0.5149907227813242,0.0986245265737324,-0.5572831181030913,0.06020288996699601,-0.47836374573200746,-0.015481658162276524,-0.6368689536589074,0.12070866281088818,-0.6217409627379024,0.07932150530065059,-0.4182559972664663,0.08466580026133604,-0.3882067892219564],[8.004546387142463,-0.8566803082090508,-3.6541652331817573,-0.4976529876367628,0.2626991449560657,0.25440351249699883,-0.22359437504154606,-0.18268139288856622,0.20049221127800745,-0.1737494639269225,-0.1532205929180791,-0.3715511411609783,-0.3011751038199303,-0.33472012253845623,-0.05808484505379162,0.11269894195780823,-0.23169055737996572,0.08338637160657213,-0.07687198561048932,-0.2351863198627895,0.0706674123321087,0.1046259960809721,0.300559344129049,0.12732608344757398,0.07245306965199282,-0.000725658029472383,0.22450703829129465,0.1681898933455429,0.06560233075695633,-0.034434844864267254,0.01539270339376516,0.06501915287882921,0.28300821588290603,-0.2540664916660402,0.12141893486353249,0.2558086385485861,0.059755544364881126,-0.1134118715028419,0.15090355704649047,-0.19516403972720656,0.07370787251976907,0.17040667158070053,0.2051332287264721,-0.04449593906037591,0.007725041034336094,0.044891977377989424,0.1450564407824123,-0.012202230849467496,0.09290483082125535],[-2.631809429751071,1.4821202918041574,-0.21501774604731866,-0.2307658252550166,-0.6835324312743211,0.39775300639759714,-0.5048763145463807,0.2017726012409835,-0.36248290614224105,-0.28024763926177526,-0.27558644532910326,0.2644371528938834,-0.35548332037758285,-0.10007626788772303,-0.17222288609625402,-0.38918406927590626,0.32935034420944664,-0.12047610415181927,0.024668002602126076,-0.04510826200187791,-0.1860153243363953,-0.23993163365276268,0.3363338786958108,0.008392114961686173,0.3568318939948851,0.07217616086470097,0.13925857948032677,0.1579404894188724,-0.13867818910119054,0.08011646000704686,0.12365372741994828,0.21036311421446532,-0.3423278788855995,-0.1144488942356058,0.2564596633786886,-0.07580260557590417,-0.004487160258980837,0.06049314312371016,0.06467692971482221,0.14907705694803122,-0.05231280276752468,0.15713466766719325,-0.1652677206453819,-0.10222639203003626,-0.16878869911123884,-0.010034426541917163,-0.2092371834821757,-0.011517211983672516,-0.06780597912787628],[5.830659005108461,0.36990396780976215,-1.9715291461786855,0.053436000000143404,-0.5818887917351044,0.580789179722271,0.06585664622757054,0.49862681149240834,-0.4570353356892987,-0.39842541714117546,-0.2374892856645587,0.07246617901190125,-0.06206429151321613,0.35195308575485346,-0.32842410818774925,-0.15355765621927592,-0.03551106375321046,0.44066902983853473,0.1958752047028358,-0.06657579392446487,-0.09063981985020872,0.052534875996665974,-0.2346246011098565,0.11213918269683953,-0.014252338712575287,0.12733966261706836,0.04851258571060641,0.1751485628173445,-0.11017746121291408,-0.12191361526457366,-0.17791989152796467,0.018725853958656034,-0.08310935970829153,-0.07644285805659817,0.16919264126731068,0.2568315634744987,-0.18089172339821807,0.04989506229971607,-0.1555739248454364,0.075442239859483,-0.057043061987467704,-0.020352238379657318,-0.0010884549586649653,0.07304878951680123,0.12421753098947082,-0.1769674460443774,-0.13470327714477345,0.009328409601749226,-0.001433128492508785],[-12.448820931782532,1.643164510609668,19.149509278436593,-1.9173832253132426,-0.1367954263777828,-0.09469358182116222,-0.8094321194770993,0.36708475388070333,-1.4080209349086523,0.6647847521405962,-0.4717321154270438,-0.13538472899804344,0.599105697133279,0.4192062594621733,0.5408327132221439,-0.240199222573784,0.6412625529947261,0.07877050837361516,0.05434977154561599,0.24519561421775513,-0.23532846966344326,0.5023143325473604,-0.4208776750652106,0.24057873621259457,-0.034496573651495284,-0.17170950305371402,-0.19536708479362472,-0.18900860950597376,0.33927673517953455,0.1476108191905431,0.16695500430962085,0.07718341756332493,-0.2868428591899266,-0.009015894461868268,-0.23806327717860484,0.08835060379881182,0.010061132561385983,0.06733892558597679,-0.11908704586566796,0.2688579722281109,0.07440818269395404,0.05351761698973573,-0.13350839761601294,0.013009897111296829,-0.08018722415837927,-0.10354901818871738,-0.0549842639654188,0.025311391692859873,-0.04594179415283953],[6.607357126569758,3.0497815731252516,1.8650512127112018,0.6995858557534423,1.8757623109419042,0.7594581133104568,1.2690629766035377,0.48333428918597554,1.9665954843941542,0.0492583431181915,1.9386213332061721,-0.18724931439433737,1.5950219762700966,0.16882301670077618,2.047013088758237,0.129793665481979,1.9636412076000642,0.507513862080674,1.1837716898691821,0.23231930406157306,0.9822998845415218,0.15438521725580487,1.7000882955235113,0.2727243732953677,1.0251242200788715,0.07350481919305779,1.20672102138333,0.34506644190318525,1.1046224116948244,0.13617272760249685,1.0282721317555286,0.22749132851421816,0.8344274111621932,0.09333743823867845,0.7078647579213688,0.2867881886446402,0.44082733153560516,0.4131022171074047,0.4588496485752675,0.20504307154094797,0.353390228783192,0.21387918271499098,0.31920964303733934,0.22731962195346983,0.5394785981322542,0.2600794372245079,0.3908249521006621,0.27887503573487454,-0.0006992774317393155],[-2.6037444295777123,0.652851912903143,-0.5731180070868598,-0.9437946002419714,0.5827460021770201,-0.24790405780916966,-0.09910191782391595,-0.16055748110875012,-0.31376242834365603,-0.05862768011554457,-0.12141472368537413,-0.057115778087306904,-0.0642070403366973,-0.14507286055926952,0.011700686915555852,0.054987034254620835,0.17855665321148048,0.1767177261200206,-0.684662210283151,-0.18975158149632065,-0.19637922211652406,-0.009383310705643265,-0.2460707957048117,-0.24929960453245245,0.05494076116659131,-0.12192420808542516,-0.1084544058489107,-0.16815413181258873,-0.05757671739441136,0.18587847399788363,-0.2411886228361464,-0.19058710094393014,-0.2863330979726743,-0.1772068933718301,0.06043875076476583,0.12945097065077002,-0.2201628424616675,-0.036069614294536895,-0.10440239097573584,0.0866924952359789,-0.14148930321207853,-0.04233699678848189,0.057640205346287714,-0.2495178261918425,0.24735031538858213,0.05267289296165316,-0.10514557898112167,0.0954794332716395,0.033006965038348435],[-6.189510358886916,-1.02780331962344,-2.017486906286004,-0.6864083689266289,0.7071650531693828,-0.22937496739386687,0.7611286648520945,-0.1257436516182333,-0.43239929880932093,-0.06020356088253028,-0.5283534195265495,0.3922452471168162,-0.6220593735352354,-0.10957623120511667,0.3426061681161192,-0.25204726313460574,-0.24775896818115603,-0.13659416344530087,-0.13597313735833347,0.018650311753112433,-0.1825660978885289,0.027564606678859887,-0.2530012245536959,-0.1556819043167487,0.08004551271362027,-0.008895378215975936,0.06242399317951683,0.01396305581177022,-0.18755176933199913,-0.08172537384556476,-0.13832541166552512,-0.19801943503059277,0.037125146362638764,0.10411571369604924,0.019233167631500885,0.09016720396244597,0.10138518773693939,0.13502546742205296,-0.03569600011602402,-0.1869877408314877,0.03938347775014412,0.2472480901197156,-0.03415090801904791,0.012670486264684536,0.13461617010781116,-0.009941261150169741,-0.08254792420775593,0.14586299582262044,-0.03206009385873025],[3.498358338658392,-2.240835427239409,5.825254773287376,-1.6737636495678214,2.84740447260036,-1.4188182602553627,1.2981525944336347,-0.7613839031759562,1.6226771164821343,-0.377476179913793,1.5574438569407134,-0.2613877936272636,1.3170907612583371,-0.0957083020055521,0.6966912201957279,-0.005166813424582955,0.6006550725535693,-0.14542033475218794,0.512655424392925,-0.13925601609465704,0.332204067700133,-0.11685393281255439,0.22161671003280642,-0.06487063155967261,0.25940266793292,-0.04056167928005808,0.06370581222584966,-0.03542715990753582,0.2910980502493035,-0.22237158630542866,0.309243973595761,-0.1993205919594144,0.16094234202824104,-0.1586269785334562,0.3158321891693383,-0.21617243291144253,0.14794816492639082,-0.2953004339851522,0.23716871208830592,-0.24789154738990857,0.27551792361507177,-0.27958194785795043,0.38198198156121815,-0.3735504707640587,0.47697756357158716,-0.3080847832997519,0.39891363596638274,-0.22093297341557058,0.44874535000363935],[5.613610758898267,0.19163888495202172,-5.821281125431666,-0.20591925111528953,-2.006212415479463,-0.8470221001367301,-0.27369767235051506,-1.1921123088237282,0.04690993934553134,-0.37496181409856666,-0.19892885083106118,-0.21341275129331222,-0.04885916311240588,-0.1651901404013071,-0.0814025887712447,0.11214667060490915,-0.13605796629771774,0.4254029551987056,-0.10863094465926292,0.2673863472485611,-0.16389858410289868,0.19398856188922015,0.3183756094130183,0.23539159007564908,0.31160505453287746,0.25696091390485903,0.3303661151851166,0.022121191715886595,-0.34438732622503926,0.03512055585158622,0.09518713007012808,0.08085569622673472,0.007539188642832164,0.13485884317217178,-0.14856382707181134,0.09451424533139614,-0.11841710403533731,-0.03705127981270418,-0.22246607865077508,0.08190305105970726,-0.062081970566824934,0.17109005777473474,-0.16329348273429795,0.23307544109504363,-0.14665752032794538,-0.13047546887064215,-0.03539273253584816,-0.0607635179583791,-0.18015292673373792],[4.368432624695545,1.0604537670588925,-2.860417676798409,-0.30041657580622344,1.4974369652991197,-0.024666764296567348,-0.07908538962710909,0.364045977749987,-0.6641458353907749,-0.06728361266850783,0.7707764273295848,-0.6229638220983439,0.23287059029681528,0.10757184329001596,0.1070550967306869,-0.03387985200709629,0.05781829612005687,0.014991777941779898,-0.03336382557489212,-0.21991159892342357,0.02365109626252285,0.05454827873656845,-0.2119683887787754,-0.12294642444693878,-0.09729339767745361,0.25191578221639327,-0.2339346463864577,-0.048753805068587536,-0.15461316487098406,0.028375348365201846,0.011831510007540817,0.13779109242551793,-0.2556214936738111,0.22676690006757674,-0.28802096725682935,0.08219838094058408,-0.21173973689738626,-0.12333996446577172,-0.13941824094953012,-0.053434148198438385,-0.16973870725952145,-0.05081901204165445,-0.05917197018924721,-0.18392658542685253,0.02185341567098307,0.011488124995898507,-0.2024198447241552,0.09921879614972708,-0.021062646559901286],[-9.591355144986496,1.1676673342164223,2.6145682882353936,-0.1672483938267718,-0.3330013756868273,0.06867584369429297,-0.023222122692032422,0.8200601660684171,0.635526427715116,0.24578852301438653,0.46316703260303943,0.05712376238782517,-0.02285715522322313,-0.0644473561023632,0.1587562754921479,0.2567080495822767,-0.05157541170307632,0.07345543066432739,0.11766452312902376,0.19559196160437956,-0.019510193818061003,0.08006322479597482,-0.13404243856411338,-0.08503100326427378,0.08851806115775115,-0.19771451001855664,-0.011504263806600024,-0.008764813901688049,-0.02526523983241994,0.14780831836646746,0.023261044423567145,-0.05393724053693971,0.05372621733395944,0.06655275981881065,-0.02915077947994287,0.0898351680087138,0.005668546291125055,0.07129928139945899,0.12279852449376022,0.0933652191250471,-0.021998747340348034,0.10559246254849654,-0.010068957843069062,-0.1240709163743513,0.05774991275634816,0.09180374864860531,-0.08983922054963368,0.005587340811330078,0.025635039504127464],[-0.7029075282430247,-4.7456554264483986,-5.404850546235088,-0.6894280445595846,1.6471418341357746,-1.0282120860304107,0.5330485423318342,-0.8480135770328414,1.1833872513548536,0.13019908012302842,-0.24995593237621827,-0.1420408351668032,0.045600777556749315,0.2458158318983654,0.7122633039864265,-0.36187414346556146,-0.03620447192157069,-0.04435881530685761,0.18199931450499288,-0.009555385611755078,0.1758955362623471,-0.099811463748499,0.3405622789048483,0.05963206286561295,-0.21485462944151026,-0.019727865906581984,0.2638902675237141,0.040317840795835204,-0.05222265246459095,-0.14855804500656483,-0.02284713747009251,0.29469518684183205,0.20925875707702485,-0.022748360143571424,0.1124846534541852,-0.0016648550782805273,-0.08339315281393839,-0.19716188234998677,-0.07745317643653583,-0.2578112986345474,-0.0030889276539399027,0.054541911077629054,0.0969334529261871,0.030980811475850627,0.07658245189198357,0.012542549239307421,0.06553024471482478,-0.11031996383143819,0.0271826737243766],[2.851289327758958,-0.23214887722416097,2.4798487251955494,0.3652808759636752,0.2093416367484528,0.1658700663192788,2.7331892431703695,-0.7103720542177033,0.8108123226915858,-0.8600724259041126,1.8887629274074063,-0.1465835401048766,0.5941780980130346,-0.3794709972988182,1.1940021219916983,-0.14113427191950054,0.5595001688145885,-0.44184531929220117,0.46123586654754467,-0.2441875284394301,0.6203595929049138,-0.3515402245438235,0.38541352180097227,-0.5207253632176725,0.6092547695414041,-0.3396829269005283,0.4544218398320116,-0.233999377773691,0.1512202018759881,-0.3215162277811195,0.6592752192262709,-0.5665754066980293,0.8054588162286968,-0.753499505431643,0.9045025285350414,-0.6701607611829605,0.8011622096784737,-0.8152016544268014,0.937424810273722,-0.7283690882134759,0.8811858019825278,-0.6790724705623408,0.868735296562588,-0.6789088660392762,0.7912783172766211,-0.6781172250454681,1.2670299874635063,-0.7778955618806633,1.2487658054787],[-17.61354195890814,-0.6600915568392565,0.4092275360281913,0.1653353740930416,0.4495655338422434,-0.35938421236399415,0.721540119631209,0.5789885937888033,0.4010218845004638,0.38576941998771513,0.9071829072305938,-0.1636720560972178,0.48112255787171054,0.313937010660906,0.3221263513719918,0.11484380442277287,0.3287787654016075,-0.07526010151949444,0.41268189136263406,-0.3369356862892679,0.5552820716752948,0.10849307368009466,0.3536818092621616,0.1607806819671968,0.18375890965541375,0.27784468027162784,0.578641241255689,0.16109107938519637,0.540315906318874,0.33116599376601585,0.5987026203812319,0.04122656577230766,0.1022519555422726,0.15433785116852175,0.0672365348848368,0.207163027254035,0.08781261719566295,0.0040537441964793785,0.03565466978852771,0.13846702293878524,0.05257821254070335,-0.031583345715405435,-0.03491193720060154,0.04716902172636577,0.08982895325340426,0.053795545621100456,0.04827994341054228,-0.03912909742638863,-0.1046076333759793],[-5.26294348570556,-1.163520478004118,0.5565450624946847,0.40943056865572036,0.3673213493956688,-0.07837136216474003,-0.5762587516888867,-0.042483250860213075,-0.7077318001016188,0.34680553786285423,-0.5352382996423943,0.4177955446622004,-0.7059734182072637,-0.10439619278724824,-0.4605859885985707,-0.0876063308518865,-0.41231151456335696,0.005437045982516242,-0.24880642449330986,-0.18056920652153813,-0.15754686656252145,-0.11579278196235568,-0.052183593564781944,-0.07560526866289312,0.255474376789319,0.07085306922560448,0.001499336580997268,-0.1614648067469611,0.12005169082275999,-0.02047795314610293,0.1532984988685497,-0.13907679326352965,-0.05250417772983505,-0.13127863105136703,0.1528899022476798,0.08349940564178318,0.06989193874518183,0.03193782132196901,0.14489501186891215,0.030620168605656146,0.17564007678454854,0.1531775503875118,-0.004095753613864335,0.12004139702503215,0.051796146870598025,0.007986486216447758,-0.1873732216696535,0.05322844987697025,-0.25300463313786375],[-11.468014020277028,-3.419642353059559,10.303072405776142,-0.2286419672230176,1.825155502349613,-1.1520401347606763,2.479295792019232,-1.1543406695877854,2.2538400998493358,-0.41618143774666805,1.9275169110141475,-0.1850392763915332,1.6087273343917834,-0.16586585376759214,1.225467436435611,-0.7943077548285019,1.0913236947668115,0.06484443993978117,0.8458370505302619,-0.22839886093360381,1.2431287633877595,-0.19893567886062483,0.4019068114608914,-0.2911573709647213,0.8103032793586564,-0.6553572704949039,0.9698660494926236,-0.13806744461139162,0.7414043854196438,-0.20455800943996283,0.8112903578608408,-0.2669424501196595,0.5821666103012554,0.11457904036928208,0.3773505865086831,-0.0029894667142698166,0.5748195156681838,-0.0010465531970371871,0.5938285318226088,0.08875888617112836,0.5316042195274351,0.07208221719454397,0.2816575187976534,-0.15778127276002996,0.3664071507584389,0.03430159074346242,0.1771310442877766,-0.012013427878685624,0.10804142058618657],[-2.7303113413342612,-2.2074133704236525,-0.6719373097828875,2.0383984298931996,0.17250325907427186,-0.7427394939720201,0.34569925479254837,-0.32133164707143225,-0.02736941144245492,0.18110522602544196,-0.20104362123007496,0.19260074672873456,-0.09079853436235338,0.12592579281361865,-0.27861374263787453,0.040961828332482124,-0.056428861730237034,-0.06431690183344071,-0.022313591882916598,-0.08941614307953842,0.004142892792716162,0.17247526063001045,-0.12765387646666446,0.22138461479939817,0.03477216288168627,0.11513294422306337,-0.041791656508734114,-0.08432851234393568,0.056250381520255396,0.10802151148612561,-0.03292734781506185,0.04001224824958852,-0.039947070200901574,0.046627850414667445,-0.061386962417572646,0.14425777661691744,-0.07749149258341509,-0.0009743671147372068,0.047527093639142946,-0.024859448197314393,0.015185964283246764,0.01156548036196331,-0.04292515052660191,0.14583234632359537,-0.09086313052758761,-0.0612574602299409,0.056186701158048684,-0.14128773139843526,-0.05285866749154219],[4.412048713577037,1.1235972658957516,1.249399681563331,-0.29818280109737566,-0.19826462133519168,0.20801376846357453,-0.5040190426602054,-0.47680521349536836,0.6652545223043724,-0.2684635490183598,-0.3389232211964017,-0.31415107996243424,0.04258917968266512,0.6013919618255056,-0.21692310027984635,-0.34734242692718953,0.43311022555294054,0.35905031857797887,0.10939528080452275,-0.6214187269104263,-0.0012054731727428993,-0.07259122338422296,0.3164954949320379,-0.07294562091334851,0.21830608057957585,-0.12670986023851963,-0.010006125717618034,-0.12809189429605067,0.18255601541046032,-0.17196344312570783,0.20364348308606106,0.04722027639522394,0.05627161187820589,-0.04202044920874069,0.04075205933557717,0.10173256205169119,0.22286512235458703,-0.07297983047113461,0.12881402495040273,-0.2093201384630227,0.12178753788079792,0.08699783884412797,0.12045156187790497,-0.04991545122025171,-0.005255972266219916,-0.18391817819805367,0.05201777848906208,0.07238666160332646,-0.05164788465693452],[8.29846043595935,-0.7341693794580276,-4.499983968491626,-0.5458531075922558,2.904469961518009,0.929331343914574,-0.7382444701658339,0.22284293139075378,-0.2680544907127525,-0.48461141075239683,-0.2244627315349855,-0.4699648559165121,0.08991273679258054,-0.30157013571360386,0.28114853895167013,-0.008096522657832991,0.1157963339240454,-0.13667971394686643,-0.03933880647140415,-0.28720797091739236,-0.1537878548172624,0.35898389539850856,0.1303907942073203,-0.08201177262172382,-0.13273024696864238,-0.04505997655635382,-0.19916751015263007,0.19219832232270487,0.3431407680327436,0.03245210707751104,-0.1650515196450588,0.18233175241931568,0.022912568450600083,-0.08014574991455192,0.04282861426037229,0.30605284363744645,-0.19293110534922292,-0.05268078676573277,0.20621778313133127,-0.3148951637849684,0.03362375975583423,-0.002825758305663421,-0.06541851387714646,-0.03838451944705409,0.02377334771807317,-0.14944942676176545,-0.0810042770105089,-0.03870701545929818,0.10555161720573837],[24.680110890550495,-0.8454413285499426,-8.676261552202973,-0.7661447636914278,-0.7700882473070063,-1.0036876051273351,-0.6483250709510523,-0.782490687455048,-0.9638784105541677,0.3251823818588613,-0.9811066944895307,0.07504648351365054,-0.26872903764761047,0.6143632664468627,-0.10850576894928585,-0.5421724477216315,-0.8055412393262125,-0.8721614799274483,-0.0035178694023303165,-0.3250102655773947,-0.8184811367449125,-0.011097232261744916,-0.5788443069569712,-0.04762929565600286,-0.12137258069552226,0.11696010552337631,-0.05072757559584282,-0.6124619110317393,0.10541498925392127,-0.011797955221820247,0.11171846128525519,-0.3760386058124272,0.18325227240846806,0.24010366077031856,-0.07181048136409232,-0.057100315370383646,-0.037971483687755,-0.4389758829042207,0.2432080744357578,-0.16082670883653366,-0.5353614054942862,0.07057964671856738,-0.21879837566283955,0.03314785691124612,-0.11472902973119456,-0.1790847440627332,0.03264607473406467,-0.07924503571675974,0.09276534201412133],[-13.536689481037083,0.14178341076726644,5.177940500273885,1.6597881060139652,-2.6223502725663583,0.41182374171032343,-1.7262132142089721,0.8923538374762046,-2.0880364892915986,0.6544048510744495,-1.6481430331799227,0.7801316675535847,-2.3421966432613313,0.48023951759402655,-2.2711972503104616,0.508278941673319,-2.634948437480514,0.5847597748098529,-2.362316250446205,0.45068053968520166,-2.3535109495303037,0.28354016875759974,-2.226427799673725,0.23274816580944158,-2.141703963553814,0.1102659037661275,-2.0068114098926118,0.10986489955569442,-2.021268585322497,0.15843829294883302,-1.6693934023200585,-0.17104532577815343,-1.4620255245593476,0.0033247617398944915,-1.2631412184543953,-0.25160626645314227,-0.9656948778986827,-0.19998757854398275,-1.1229852174791473,-0.038469382421337406,-0.7443225676337616,-0.16531013530587954,-0.4915417962664001,-0.15723421252685338,-0.35572089258056483,-0.21927532254512377,-0.20640947518688013,-0.20491545061377198,0.11717334544528958],[-4.138226815992294,0.7901482938696645,-1.839289342592537,-2.16219611013179,0.3581095392370301,-0.48223848212982473,-0.22852230611078642,0.16429828567865115,-0.15555113367039786,0.3228038667900127,-0.31326747386440007,-0.32895329316781835,-0.36398552666687833,-0.16938544218962476,-0.545821734040007,-0.25833962897024093,-0.37500782305465635,-0.1968460331374762,-0.2750604963693728,-0.05786238018018865,-0.18936837471247292,-0.08377236722125675,-0.2994282998248757,0.12100225346206943,-0.28909882167261597,0.13483020856255631,-0.2044334594249351,-0.13922458569029705,-0.4700709991178194,0.048580238629535755,-0.16158451395628795,-0.18821432338679522,-0.1520571299806327,-0.024994510634957123,-0.13284288180148066,-0.08213997745775185,-0.16858607402875783,-0.1619609248595808,-0.07296228301823456,-0.026531800057721815,0.11309249490655425,-0.027598114574138143,-0.0500732236940658,-0.006868349281768731,0.008715486140771488,0.006751081413695441,-0.21518594990850098,0.02484531003139258,-0.21861143116634604],[16.79118061525938,-1.1194911083819177,-1.84023668474207,0.27448736106730387,-0.7191068955019845,0.04705948880398948,-1.4524834250473775,0.6253206673372474,-1.0186297784218796,-0.06298288232668132,-0.10357456762947193,0.2848536808655992,-0.20468102985534473,-0.2119808753263475,-0.16489612439831403,-0.052887786006416124,-0.28911632363865786,-0.06539387078880911,0.019637370799636723,-0.035986680679834036,0.052181964042737,0.16135101507877841,-0.13361013549714934,0.2449642799587212,-0.018215702052775357,-0.18706682430596253,0.18690298219948226,-0.13613031875995418,0.13626627119524,-0.19800181042102785,0.1835967131502844,0.10882653902252848,-0.0285463913253467,0.18127395328368814,0.1955949976018268,-0.24088758906527333,0.013031094553706235,-0.05145485674435111,-0.12348753467981467,-0.17929515157103823,0.02835847978289334,-0.1813868146008533,-0.020404418316780814,-0.08746620266254464,-0.13891086822402354,-0.00896014554278497,0.026843524356300923,-0.02733435352529305,0.006991463991310191],[-2.4876687429737725,0.4250213258218209,-1.3623879246581623,1.6988046313984166,-0.04296760918408002,0.21602368646679074,-0.8683016798202843,0.466183498762697,-0.9042034197100414,0.22260069627421977,-0.6997155232141529,0.43729963680101636,-1.138338551309709,0.5559190660052176,-0.6894924385010328,0.06935411762945876,-0.03329015919967869,-0.2725730640432445,-0.34494359344271075,0.18202566515756388,-0.36085824602671185,0.1974192720272366,-0.06450733983443072,0.17564243726360312,-0.23736642037754846,-0.11553499694098451,-0.03892973300912381,0.08820661175660846,0.008480460150503194,0.14003424756400537,-0.047545436119177525,0.37796209875477876,-0.017330735127820556,0.3020979048629548,-0.09671510209843337,0.00635000862528096,0.07557108957378436,0.2196015125479589,-0.07374776209862054,0.16612852349394824,0.022599782409699037,0.04832237656596982,-0.09078120873580336,0.33601886736622966,-0.07053850695652081,0.09804788192002156,-0.20655412462240244,0.10856616276499848,-0.1962932403881347],[-6.247245099395346,-0.5274148055388238,3.8576601468065683,-0.8284323654320416,0.48420815662732664,-0.18311592442067948,-0.3252648359196545,-0.061343411844115237,0.43917289053113406,0.07287018147572644,-0.30031139125041534,0.0023120271684578648,-0.18269478052324523,0.08505754134524673,-0.21125298011304275,-0.6444092291915178,0.07496911751500844,-0.3718375221795111,0.3572770946516064,-0.03025615487139382,0.30035448762708306,-0.19127499976492435,0.10056680544028826,-0.016042073391115335,0.1723906227938531,-0.012100859920825184,0.2629437592532864,-0.1430116590403728,0.20026598638128876,0.017486050233553253,0.22304902759605225,0.08518413595331999,0.06323192174195469,0.05913997620242785,0.21048997698154018,-0.08961340528273475,0.14426887108494452,-0.10441020923940018,-0.022555645273503157,0.049469386527905806,0.1468465111394153,-0.09700222351018084,0.04499777034610983,-0.12360402663531532,0.025139888481595887,0.05801167499601195,0.16908076604338845,0.04291625102318562,0.0882781217954994],[7.896910330729202,-1.1713988753864322,-1.5319591534272823,-0.6419832466171622,-1.0934894587417903,-1.0030773435320752,0.8345773657673243,-0.4687055273927571,-0.10529833481668965,-0.03129921980101625,-0.5425678830648163,-0.3020722040969025,-0.2095131865864391,-0.13092866987624044,0.11458205392484455,-0.20415015247281978,0.10142590513151621,0.04557280824515785,-0.019828024110743452,-0.06220781832299996,-0.16558780022230202,-0.3705187767649024,0.11071616195862562,-0.30462733122039315,0.08572588541194474,-0.005597667157884432,-0.00567229817495658,0.012806918431769168,0.11720074178916466,-0.01889924719232679,0.24974083384930298,-0.21792252306008297,0.07862571496599652,-0.024388508868764182,0.07541125734165212,-0.018147120328503512,0.0027005314277954873,0.0694978169714767,-0.027343246863857036,0.06776757567046439,0.07309989205847295,-0.11672252087465387,0.015242641922093769,0.02332356227511912,0.07869010496087071,0.09287518183238153,-0.06900120043107423,0.00711002777462527,-0.04386758035086967],[-2.793748666912232,1.0666164108794511,1.7467299249605188,-0.4740152617962176,-1.7202687456940242,-0.4400259568631263,1.4622915244671877,-0.6609771248533958,0.1453330265603389,0.8488555343529226,0.4668309420517383,0.13751035072728438,0.5815456241736838,-0.0563138618713707,-0.02052765477951914,0.3637043407376749,-0.4069140703727135,0.04370785606451843,0.17743465001097863,0.20790209395035855,-0.3246191730231199,-0.3271108293444115,0.00440997638321283,0.20490091558626775,0.16467366171986791,-0.05759525837315832,0.050447076608534785,-0.22898273849858997,-0.23428896263981033,-0.0011498216764834035,0.04749355155924753,-0.07665009887958428,-0.029874868399795383,-0.166264918765205,-0.014044601361477347,0.11975094097027995,0.13263204639221818,0.2157280635822214,0.029380931006354364,0.1589339201278861,-0.22911306334863996,0.08610965955509302,-0.1534840006718645,-0.13570979737896985,-0.10118494723136029,0.06867397244196236,0.11982857383445569,-0.059507294469735156,-0.0832990348322512],[-5.884240205162677,-2.1075004179195487,1.0050237387683654,4.4275253246584,1.1805369868279159,-1.0612870484627868,1.2813493460793097,-0.7470924589009336,0.4819580442053798,0.32750494252597445,0.8892322715947287,1.2285997922710774,1.096456113821148,0.1579086827086129,0.9043489542034612,0.5211883748969797,0.4155918419772996,0.05458323211997174,0.5173876585368931,0.4753739772714154,0.6611390063574684,0.11417687015696766,0.39602749347138716,0.08674727393010061,0.47085629898821063,0.3281216760532004,0.4127000003968613,0.23165028488855538,0.3518618257269146,0.04096435247383737,0.27165883864800267,0.047150456172553756,-0.12652110348748025,0.16874397611480846,-0.036838637435606,0.1312465749269429,0.09492329148719145,0.05221382492872497,0.0656575211080904,0.31012790261947715,-0.14850438041808,0.06663276773036012,0.08172758285814834,-0.0038111847179347815,-0.20903751103840498,0.1466940324387786,0.1298258377286795,0.010164974627841869,0.18683171845919463],[9.496797964601205,-1.770988526035264,-7.5339550854775155,1.219104430115548,-2.482876593780791,0.41305222947532055,-2.4738723477135225,0.09150104420100638,-1.8997009625012844,0.5690063130698672,-1.3778530453894318,0.32947540518753354,-0.9897674392507686,0.3231277212559084,-1.5600205587159504,0.3810922000270583,-0.9479665815837259,0.20046818634810928,-0.7918304228205078,0.33020635692725947,-0.42194786014024854,0.19720907111578467,-1.0195838983683614,0.17126442545152773,-0.7778123533937881,0.062326442283743795,-0.8412107481986142,0.08222520837399935,-0.8614546739525057,0.0013834787873654004,-1.2024873606646982,-0.08902835082800968,-0.8768451270076766,-0.1203107419798903,-0.8418728469200392,-0.17853882187097506,-0.8908423031973851,-0.19913933902262984,-0.7579645445900122,-0.14714350192463505,-0.7923629366296782,-0.11580633884595652,-0.7162268142888182,-0.20001818453038836,-0.9451383908103471,-0.3070848571050431,-0.6948250548230277,-0.31820981324715086,-0.3531902082608777],[-7.265527754274109,-0.9804362032208951,-2.356083896300182,0.32501482770307977,0.9264193482077825,0.314185777579211,1.0442683651250773,-0.09260086002424019,0.9664148902096562,-0.026649379233371556,0.8754386514395172,0.02640458872236622,0.7870542007320936,-0.00107938104084207,0.6505954520044688,0.15813234140143445,0.8172723036549714,-0.2943663701134296,0.6987775984428098,0.06326333400656126,0.8006042562517784,-0.05828189325336441,0.8039095731527994,0.11536441654876672,0.655403486459572,0.08220643681320262,0.5532381116658153,0.13215302566070114,0.7683967650259061,0.006344343136007138,0.5886349974154379,-0.10265068907741182,0.5476937040604529,0.07691663635359954,0.3515416900910917,-0.11990805506330335,0.32843998910429567,0.18671436799547642,0.3741653411259488,0.05028695637160584,0.15154219467306532,0.12480471813687326,0.0695300793242415,0.10097958390989743,0.22092146038420252,0.11441145782183391,0.16335430220860384,0.11953824934631385,0.20093406555168492],[-24.09879069783657,3.6902045276117494,11.094569736918261,-1.7806896310520781,0.154049158631727,-0.6574569346864172,-0.6035906796027837,0.18347832706761413,-0.5136894902647415,0.28244442714290485,-0.45174175544097306,0.2078841125152336,-0.12911686171445316,-0.31955228292306465,0.053100681471418,-0.5782294965631871,0.07975133081118264,-0.18392580151148677,0.21810788869761152,-0.40777659726056253,0.14791146131204946,0.30171496472139675,-0.10654256946790909,0.004225269025721569,0.23533888070009262,0.1521728521128539,0.06220711770590997,0.19947571102786013,0.10333566656207085,0.14670196279416237,0.4841505759642744,-0.09058571248519781,0.06163852796571248,-0.19268906183270595,-0.07571908036735282,0.21980057709694573,0.16502188993593817,0.24446231504345012,-0.17404217852919798,-0.19973480578293396,-0.23050669236948312,0.08513319252747123,-0.004712827462708917,0.016743680817110786,0.029715586704321718,0.05340193285352778,-0.05342589715685197,0.02580874365597428,0.023286545473157786],[-4.336079466789772,0.5225783617023632,-0.27072188230562577,-0.45160026433475126,1.8881701006607925,-0.6073973553832921,0.2471580070129283,-0.07883901687374283,-0.0852570209348096,-0.18199094069915742,-0.14322592432813536,-0.5367551631417159,0.7606840115218229,-0.04906472845341349,0.9117325440643548,-0.4287355462314068,0.6411353512449025,-0.7301931474106123,0.3040180040724279,-0.025615078036468713,0.3457971586063899,-0.8282477124719393,0.4836180655169705,-0.17097286341367396,0.7699171900678687,-0.48505307369821504,0.6099082523010452,-0.3863262278821122,0.46198432021121194,-0.3483225167758367,0.6312974497638139,-0.22815523851640088,0.5882745209025014,-0.46519726898771524,0.7608949368585423,-0.4755229659843599,0.4779043195323397,-0.24744125444704723,0.5884877130599117,-0.25034585772351514,0.4837938745962182,-0.027631865951154737,0.37526581596472364,-0.07666036114718043,0.6731122634752001,-0.3943909378077885,0.533230129259718,-0.2858836509259527,0.43263035435591246],[1.0631077465652465,2.4114846074044514,0.8927982075295553,-1.2389686937287319,-1.52015753435869,-0.1510574729772645,-0.4463255789947939,-0.16482300162293295,-0.23334403854830083,0.06200190965488749,-0.40339309177046484,0.7367448000674385,-0.42861783248388735,-0.26336435011432047,-0.0509186749689406,-0.4795425165741442,0.09891912474674444,0.15242838109361814,0.042527352179516406,-0.17388589422856784,-0.043190619445277564,-0.0065196242380103665,-0.17225926370473132,0.05505582732390508,-0.24102380424577594,0.19754941623260008,-0.13786957161925534,-0.0666515287763641,-0.006036497921880058,0.07983724511559324,0.08457293163136902,0.05967248948922712,-0.300502427991662,0.09911282104477173,-0.15356945829355462,-0.2811952963889103,0.058132279709563514,0.02719808056906501,0.011497687380843096,0.15369960548740794,-0.019624031221253977,0.00582386924671187,-0.13310338406951375,0.07593876269600756,-0.12467057454641818,0.14418644362635813,-0.12319579772278809,-0.0992855616818582,0.03287245145361795],[7.889867107473086,0.48927044300375305,0.6525372452714003,-0.3004368034556489,-0.03681660737455326,-0.05947789663182303,-0.3686066055327884,-0.15981578484307862,-0.8157003327264775,0.11460520568585525,-0.25250926264898055,0.048500067864474194,0.3501202674873909,-0.030489628993025795,0.10341985832885228,0.37163521776367686,-0.09139974450454633,-0.2764362607604845,-0.17622024422431687,0.23029807407337788,0.33226311932322494,0.035666069193511556,-0.2993046311471252,0.27496502574000325,0.2238689742414931,0.22522228930491314,0.47405041747706533,0.1289951361098246,0.08803801359100209,-0.025966605853371908,-0.1753753863778931,-0.09045720267864119,0.017450741940154394,0.23672886094591467,0.08693038556898877,-0.020994234803305825,-0.05288053582293308,0.11083437020753657,-0.17047629094307554,0.19314530746386394,0.05936031335745973,-0.3612063232466661,-0.11152903831675381,0.169283958276355,-0.0656794563946816,0.041211333991257815,-0.0055901640108277665,0.10220667974186355,0.08100158701393623],[-4.5994707669636625,-2.019027191888715,-0.8353337572529934,0.7450891617889991,2.034802180492687,1.2129077869425517,-1.316035145918056,0.35640194664389746,0.3820044384890469,-0.625774871081579,-0.6312310082746715,0.00792603638053859,-0.12211684309042088,0.5582533409036703,0.2108710061505134,0.25025802076595344,-0.28537169812000274,0.3526533171618629,0.1362868019503944,-0.2751660987789864,0.12969504944335833,-0.07883013218824243,0.29519304067058844,0.07198742347875915,-0.572482796361541,0.3202231473446236,0.1695152640723767,-0.018158095654210914,0.11052439384616851,0.06833480971017628,-0.40930873903211124,0.056262979941306766,-0.09423058252935,-0.00970815467923332,-0.037035171348794076,-0.17686917241591993,0.02842500166077023,-0.18323658196223228,0.047980560794982176,0.020248762459017043,-0.017469839174615354,-0.16387616807283478,-0.056597962275189866,-0.07046839545534134,0.07886884974832363,0.004077478693286839,-0.0414304575025929,-0.07505694585706674,-0.09180648181249047],[1.3644542550822063,4.2432059397902435,0.038153163996064755,-4.44846953700525,-0.5254693636620215,-0.17722821272392725,0.9261874524199599,0.32739863930640684,0.1592914474813428,0.7781325111773876,-0.2106855219229518,0.10198552716899663,-0.4677061571210563,-0.9337948288928425,-0.24563955627703543,-0.7650002483956894,0.16688134449613107,0.07898819286935077,-0.36761932115811313,-0.5336857512611645,-0.06950678940970897,0.15393635976865389,-0.20019740274989553,-0.39624902992456473,0.05246207693723269,-0.3632345492352811,-0.4817499976966859,-0.17986009625702323,-0.17558832421140924,0.024023020990088226,-0.19452529022417822,0.11183334182640753,0.060946740605375534,0.07227772081136989,0.18175093787669258,-0.071505079683932,0.14301960698441363,-0.09008131221632612,0.060295341822756636,-0.18861750524216994,-0.08485275810277865,-0.13247472827564688,0.029618269655266947,-0.34068926921075965,-0.15890578926305796,0.10415487908856975,-0.15939222889639143,-0.08863167114361402,0.09241928103093139],[11.023917780157479,-0.09724152685360016,-1.7327898911573554,-0.8200967804684959,0.02333261834969852,0.028895133351203205,0.018429635912008704,0.2329583517566303,0.12356839834773221,0.13468449801477525,0.1840776417448289,0.023605523379464272,0.09818774410180954,-0.08280510465061072,-0.08753485375557792,-0.07762463798243384,-0.07794671997297055,-0.12283406617049282,-0.0055167888127862046,-0.32069532741403595,-0.2786076595656282,-0.28222988513264996,-0.3281693901016329,-0.11069791242820858,-0.19425603684959586,-0.11331757139139066,0.02159402444449891,-0.06779357626890888,-0.0817331746442973,-0.16332078999793212,-0.06579329916042645,-0.1896718725157061,-0.12872758991861694,0.020301566148927017,-0.05162149919359331,-0.06504965043605498,-0.07170680522165648,-0.04301467205222064,-0.06706970752978234,-0.05240418061158628,-0.014146721744173562,-0.0054187453724900934,0.14246755276997408,-0.017403983883951795,0.20155567691057374,0.012202602589353352,0.2250755321651997,0.10234534645091595,0.2632503482847984],[4.06230477295885,-0.40816824424654236,0.2361378644553203,0.6349593451205693,-0.4231995103782485,0.21308704790193086,0.06718942169925289,-0.24221996909311425,-0.22789633528534758,-0.19424553626936641,-0.06158438630912975,0.024659755644213317,-0.16621655310245312,0.07194267257486625,-0.10671117987157866,0.10556142784720718,-0.09151827551365378,-0.0798801693968763,-0.08359831236290204,-0.21766415728524302,-0.11011528279896389,-0.018420570672975218,-0.17838330837595817,-0.1059042828626342,-0.12712724900004185,-0.04839741500152904,-0.09971479870242428,-0.02132141046440581,-0.10198831565318152,-0.058554047153612995,-0.07394707091817693,-0.04337721945331261,-0.0466301694543989,-0.09689440896556409,-0.06280985678829694,-0.06582416588586687,-0.047560468904396254,-0.14686662995704897,-0.003668292183849252,-0.041675764824775384,-0.0617301320440895,-0.08880902295430246,-0.04069223508143708,-0.043676395289964004,-0.029138665776719926,-0.06593887012851768,-0.010446643388000535,-0.046170047300122884,0.03133112843225894],[30.881211750724944,0.2717441926927312,-9.25484962668765,-1.4018369513703535,-0.10098255612609032,-0.2373932043792887,-0.18557128773764542,0.02330326672721941,0.27232903065800135,-0.5077730594511252,0.34011994261948114,0.053311390371643556,-0.09306776327180269,0.21726890882472438,-0.11042816839305329,0.1606425656211076,0.1873815489840426,0.31294270430883087,-0.05677813390617456,0.089522687308129,0.03182402508647351,-0.31859205796724044,0.03502447487093705,0.24281778488053782,-0.10248087452397822,0.2845232593569182,-0.20299127050569915,0.2736860044818225,-0.1967554318240878,0.23362175282155204,-0.10857039551799517,0.06652754626371851,0.03656742974896216,-0.06799544795907336,-0.0631811426643048,-0.17800116681208059,0.005327785150021501,-0.07367737660496787,-0.17070946868608558,0.06318171188458117,-0.06392410025809744,0.11435418616308383,-0.13394111154039504,-0.020523117927543424,-0.12706450510284,-0.009579429430963545,-0.07276924008003077,-0.029088868390927277,-0.1142352868868542],[2.2106682389482106,-0.38271597928053636,-4.084455723868409,-0.2787733366482376,-1.5644831909438182,0.41516373742607166,-1.5630363285046607,0.5921217832601662,-1.243959368191835,0.3198232822578071,-1.3071948973685232,0.12322610818751521,-1.091973793672423,0.4736913456927971,-1.188976549123713,0.09684627863707643,-1.4989514100264987,0.6023731035498048,-1.1659222567599405,0.13172122025400831,-1.3074125407023427,0.6467409844968409,-0.9188174857959062,0.20176449417268297,-1.4487475373389773,0.35045567511509257,-1.2703840942442883,0.24665420948591257,-1.3598242937849216,0.4715125525097236,-1.4535838383658921,0.4033692918501745,-1.2423250997642976,0.493855496268957,-1.4802725580278242,0.49649641267523037,-1.3011858303046784,0.5618524394091245,-1.1245990523737588,0.4492052012858379,-1.336528755758754,0.4026075907902273,-1.222768971573716,0.4363947405527307,-1.2718549628391107,0.382930940607353,-1.2418897956792088,0.3383249132548001,-1.3504447903842802],[-10.319754177648575,-1.521810533803005,4.118374038249007,0.3893918057535512,0.7066856039024813,0.08653037391479143,-0.03857882410181506,0.09005738453320034,-0.29429044631227147,0.13475663995222295,-0.2463606801926932,-0.14130925006810405,-0.03357878140159264,-0.345681417641424,-0.06603921656492,-0.2822926349324381,0.15089423645153033,-0.21444585407741554,-0.18440787621273796,-0.18221758015239192,0.12263796015918146,-0.17096691060435315,-0.12805185692712193,-0.036212895567791864,0.22108331866701866,-0.10621508912923144,0.16140974848593626,0.003940801227694266,0.23353332936733298,-0.05290464281111839,0.01994223368818285,-0.002660292328936316,0.08700760618351264,0.06337445451616426,0.1652574193384828,0.044344173323229784,0.28626575448545205,0.05059018242879151,0.09732999503811815,-0.11381709747461424,0.06327015477188394,-0.11312206471870609,0.17720845125418633,-0.11662835670828246,0.03939807018791074,0.04007801761550385,0.06580197462475917,0.0033280488660113556,0.013328902463710135],[-1.1199063698309906,0.5898917193123878,-0.5940868735782343,0.34487767186094415,-0.8932173740179321,0.41842510805162125,0.36990518791383814,-0.08887903028900533,0.6795873732276124,-0.268515348312592,-0.057248217172897915,0.2587806428876946,0.20273384107803197,-0.33099369423201724,-0.0017574462794175294,-0.24240433156144367,0.06633253807446937,0.3543322562936689,0.025612991107971148,-0.041278963904492726,-0.2808183126903565,-0.12454442833734115,-0.028556979537540828,-0.004559618027379542,0.024835106132687187,-0.2868654178429965,0.00023773928767751998,0.25852444570967814,-0.04654116561235462,0.03333197304381834,-0.10672832238235731,-0.25218877026228664,0.006063615366083461,-0.07111313001120537,-0.18347603659342096,-0.09854170478170098,0.047252378752198884,0.26507985469795825,-0.039947330447979344,-0.05832863557600566,-0.09536562291889894,0.12209626919637985,-0.16769552509129879,0.08637973539019152,0.08391807621115535,-0.03228400436435102,-0.041310298372020084,-0.02837046964674536,0.023927672634785162],[-0.8594408855710223,0.49895915966741033,0.0016433267636014012,-1.9230448509983311,0.232293752180255,-0.5159457890392782,1.6946848396058307,-0.1934638231322525,0.8312854963523321,0.8017185815245159,-0.26527548588134564,0.3219202653317845,0.5960113706366986,-0.14001198486246402,0.862534716484823,-0.07113857349742875,-0.2046998526685098,-0.04477652685940498,0.6070834754289464,0.19361264436873565,0.06775093972821215,-0.35507871321207146,-0.22430837707245985,0.20737741737552312,0.1632443920291989,0.09920548043054277,-0.1530450086929727,0.10394282425174733,-0.1086321099839239,-0.0362568136973877,-0.16821324880239474,-0.07792176016702837,-0.03215044422277776,-0.11493529919215947,0.07864804601966319,0.09760206089742304,0.13308259812775833,0.2524918116341996,0.06529355575601006,-0.2041546935451044,0.1037089903772082,0.022364012728193636,0.13589715779689718,0.10585194906068472,0.08634104413814625,-0.08447870829607243,0.062399699827980916,-0.0861074563037193,0.21808126019607213],[-0.5145684588082748,-2.3763872188148905,-0.6526782350005063,3.304492713255608,-0.9382703752589551,-1.6414912832301445,0.1805053374150721,-0.9796623443604594,0.6455955632426961,-0.993555051150519,0.5994606625854885,-0.08835751437663344,-0.48841021064477125,-0.10985140896448267,-0.7119883713543361,-0.037168921050332523,0.011841327717205523,0.08941551467335207,0.802313754125186,0.2943261001087263,-0.19860378315086574,-0.05286110795260687,-0.5925709443170671,0.034614960283719284,-0.10126621839164189,-0.15193111321424485,0.028280468754071603,0.1097139162545246,0.1517867276451034,-0.03244550324951833,0.1564751000064886,0.14030376922566065,0.1438575060675525,0.22783488178140246,-0.5825240224259175,-0.4317347314715782,-0.25396430572179185,-0.37029831273171643,-0.09949944738065941,-0.5395725169414435,0.06166120597484972,-0.3155967857568655,-0.0007745876821919022,-0.20269773229455004,-0.10851050714046045,-0.15914687181818477,-0.06787389550749622,-0.06558940483120551,-0.17928673041386697],[5.417653577157721,0.21801667454586673,-3.766372730706363,1.8231896047692013,-0.2937753898361188,-0.4453263368330939,0.5428679377976896,-0.5979152177389144,0.12193044956359834,0.3535654792016265,-0.33059781392962045,-0.10919641671560588,-0.2887952855234545,-0.2933111821273453,-0.5893076976278745,-0.3080629231948987,-0.08280484388253442,-0.17111770584881394,0.002264386139156827,0.09420570451308742,-0.12767849957045727,-0.35536662312683687,-0.3321765913408456,-0.2675390864452706,-0.21775839221585386,-0.2852994426891233,-0.46624781998754056,-0.19630818888159565,0.04711223646818355,0.07343893668891445,0.1272675138654839,-0.16945468410458628,-0.0039559044221448994,-0.29002268329167424,-0.1580477596930513,-0.2340103553029387,-0.19628952803199573,-0.23288179740552709,-0.021096754944112457,-0.15068279322737319,-0.24454778857570814,-0.10310732784985982,-0.14071840425056867,-0.07788299572664036,-0.25984837119323057,-0.04048777631207701,-0.10077739328936745,-0.10610968276610178,-0.10458156983591353],[25.50862739626431,0.01234025974913725,-7.397260823571336,-0.17005779096771204,0.1770581222073331,0.15442681855828413,0.3092471235581159,0.05571276405844334,-0.05426491565563511,0.20342821490499385,0.21637321535738857,0.3903810186421964,0.16412679357680296,0.3078233605207287,0.5763235770441292,-0.20552367583699402,0.23490410016764024,0.06037873528375929,-0.08331686212819275,0.156172424139917,0.16372413936651647,-0.10928575940885359,0.12891657608318588,-0.0702906751650298,0.04336926873505032,0.2137071059204716,0.10504617284828247,0.1704351230804754,0.2968459455152481,0.2698002272602233,0.028037716570808537,0.06867215949933649,0.09887497641790753,-0.1753049346182846,-0.08256979740937327,-0.20690046538207102,0.07660576447623085,-0.08937602768932246,0.08010870009869958,0.04516184626000198,-0.08123923274022425,0.016684562290107785,0.04100454770974007,0.06283396896367752,0.05374966001510787,-0.010819593278718552,0.1163029203560674,0.12010940202632206,0.028425647319299908],[24.135478771183166,-0.0005425436024409725,-1.3631515259918567,-1.070628174241135,0.43144847757778615,0.07821904679166422,-0.9075207856238541,-0.4532309380168043,0.6524946957600404,0.16657938181224713,0.1088463464412329,-0.44755971567217206,0.6738130120014146,-0.20063612303191744,0.11008742709978923,0.4985464759815146,0.7846153761542136,-0.10613977492926471,0.13569566748972733,-0.11298848215471397,0.26420213061541953,-0.003196640934341761,0.10583096456595274,0.21333530373287024,0.15792675541475773,-0.3586105172753036,0.330899748427338,0.0940372548770171,0.2173593126281497,-0.028985431806867974,0.11032855837616193,0.14207769214737911,0.3632435861492822,-0.09526370666126471,0.13138767082516026,0.22266155133549503,0.028123014309385783,0.14133159828074748,0.014403179355418096,0.16131578022159965,0.08098424723918893,0.024732811082635375,-0.0701578540542801,0.11158612158267162,0.1663178484233792,-0.009460116783428143,-0.03814544145010282,-0.004835458044391679,0.193820932559719],[-8.250019577535967,0.0910157060341274,2.893884758433159,-0.393788866194164,2.058223102677224,-0.26277148235894365,2.12214553190671,0.19951747875551512,1.3095730423938816,-0.3787130424964176,1.5094002794893968,-0.14456855972196003,1.4309830214283532,-0.4186046940423209,0.7843566050743518,0.4299735535729473,1.3152244266229542,-0.5360358178888908,0.8010556159232625,-0.0068608347871936546,1.0777963745643993,-0.07762403984513702,0.9410506590668531,-0.0953265505910342,0.5965926227327579,0.34865336128011176,0.3820328869186443,-0.3264535140974964,0.29536463888897835,-0.008741105605789818,0.3847266244129021,-0.06294348469379907,0.6696300356777589,-0.12039983406305908,0.22955609332475704,-0.298257757909861,0.42690749695256097,-0.08605479468657813,0.27806961929002433,-0.17122926039650985,0.24170214801448153,-0.12177823687011473,0.14950360691160963,0.04068113103546195,0.0536334084112836,0.12803740054197474,0.06305650570157828,0.15017643563902072,0.06047455223192234],[0.9846953327051309,-0.07929973982143694,-0.9018061131466298,1.6077145123302434,0.25572681554698157,-0.010325956108796388,-0.9019700531456981,-0.29883962686399734,-0.16137515792630203,-0.08299013428487981,-0.2690387561993099,-0.31175867063005547,0.6510407448775634,0.20676404366764548,0.16114303161896734,-0.07023312070961425,-0.02531025859955352,0.124626485301517,-0.07508191093410897,0.1871995615587477,0.18519795991798715,0.04685616651313926,0.0452894389009844,-0.29316361645130884,0.04023329346694242,-0.11995168188501736,0.18380877154682385,0.21686854426459667,0.29783623295826234,-0.022499813303822256,0.20551681341543496,-0.21972145137521484,-0.03912866699709283,-0.3985798358877293,0.16211186450607643,0.06467028739175198,0.2636905865513515,-0.046574600197287014,0.026012723187384977,0.10684037093361945,-0.02603861018035572,0.1177270250641364,-0.09490060761032136,-0.22889598507800804,0.17027101679148127,-0.08872230854565463,0.1281276794202031,-0.04564304393314622,0.15763204951860255],[-6.590174369801602,0.7810890590400158,1.9400674184582318,0.1969516447902247,-0.48528909102568984,0.1205950097812518,-0.34890907742913524,-0.2866633929145059,0.27975042145678797,0.13052434161384857,-0.10213663464581897,0.01081250197430957,-0.03290195332651636,0.2361167013526617,0.21766039993379785,-0.17273132725774348,0.02686532734926953,0.13230604003101215,0.11122742782723205,-0.0405293796324105,0.10845730373635634,-0.019091359016961455,0.429237504707285,0.12048248579212723,0.07180422270133688,-0.36279638552949284,0.03511678335545664,0.24343683286522233,-0.04692688724568247,-0.021150866361313365,-0.22433518091730947,0.319831738805794,-0.2858654543281837,0.15300688453198547,-0.3229992520195796,-0.03427844182844001,-0.170699512313899,0.10905098440587406,-0.21665293133053082,-0.08822106423454563,-0.23836789722679017,-0.05334589339637891,-0.10957407977064769,-0.05614361943753624,-0.15262871971929776,-0.08602147087056955,-0.26454005381499063,0.10830287831094766,-0.06046320480875837],[2.257196078173026,0.28914349905718156,-0.5391599164786142,-1.1710398061381404,-1.0217964257071948,-0.5008103250602537,-0.002295463924058599,0.2614030410229663,0.5132645488680719,-0.38654282591353817,0.3771920295208618,-0.18550186574950772,-0.24614925087253933,0.09107648274136404,0.12880440919426123,-0.2260964418450566,0.14179194064102946,-0.1380494473692367,-0.03218157359716207,-0.029579054781987274,0.039848128391466835,-0.009623013554332274,-0.0405894438345651,-0.001688257742846301,-0.05861362111821336,0.11508563284134161,0.15213483185591156,0.00014109869612857778,0.10168093488453402,0.03835779434243316,0.04798385202293244,0.04085870680296912,-0.10471482130570788,-0.10602719164385685,0.0643955472720669,-0.03797565040458616,-0.04299509947320756,-0.1354566233469747,0.00953558118678534,-0.03777999070597167,-0.074174226189262,-0.0039881144617883,-0.08047562516607698,-0.06941106763949924,-0.10525183133649507,-0.058201075999364055,-0.16809274397201698,-0.05972542604104718,-0.015855149768966244],[-1.5238692762461994,-4.594193730446593,-15.605841333200829,4.35738264379205,1.3580820794822253,0.41863678344256566,1.6172031323048692,-1.0272437825129597,2.643246568291467,-0.7601992029523478,0.8791226137210133,0.3425151185483117,-0.111880257955741,0.49739576899309357,-0.353484997467618,0.2972985334746347,-0.4317100281300473,-0.13831080020196168,0.3176612596464395,-0.3576770516868301,0.09875874160002324,-0.3934014065839364,0.4851357741565133,-0.16604230904903136,-0.4237036327545361,-0.44921286806913585,0.05993253208843095,-0.07356344821312072,-0.47745966189462014,-0.0769759880965906,-0.07667222465234096,-0.10832020189211536,0.3989158277311603,0.24818624748762586,0.3577121648813603,-0.05078884091141504,0.29387564925819404,-0.05994229607453319,-0.044251736995056794,-0.009949683093458077,0.03213357361220088,0.012378275755490145,0.019934386863781853,-0.02677679878049592,0.07374325133152805,0.18200161419330363,0.2750889538482921,0.16051130422288418,0.08909784019897178],[5.768724673546012,-1.3678888060476557,-2.4055311275179814,0.261202508248915,-1.4891514745096797,0.36512530529223897,-2.7780321717984564,0.7780055518423944,-1.422274901014013,0.8163880742798432,-1.3534480548989603,0.8974532362523938,-0.5192555381886079,0.4168888012955565,-0.555214686903925,0.26768932232192505,0.20490179335951572,0.19833875076731622,-0.17535865028672465,0.20993974379343075,0.39437567387823547,0.24607127741187465,0.35400744072712914,0.22680163334469436,1.0174088164133805,0.015197804047582012,0.5816044365848396,0.06462217466979539,0.59610231479246,0.21993683672516895,0.5836385315501246,0.27584400022424915,0.5262147562396344,0.19145687454274532,0.3136516863022033,0.19857384336666684,0.5367501340245703,0.22120812994657973,-0.06265126443351751,0.3387275443541987,-0.1306293640085656,0.415290994527001,-0.34109126840265447,0.34597193283537614,-0.22369483128653236,0.24548111134722603,-0.47576368268089914,0.354675277119119,-0.8663229692328056],[-0.7090697083493561,0.061146893378876896,-2.88303592562282,-0.25514648557504793,-1.2191761910188048,-0.08671585370882182,-0.36699293967606383,-0.13694167583931924,-0.8679084221657621,-0.3100017454790056,-1.1649368540020544,0.17376348632952018,-0.8041911018828821,-0.20884701901992617,-0.9636915549202283,-0.2674511741914482,-0.712260353635668,0.004325336721670743,-0.9523825262077157,0.10408099650099084,-0.6342264821911954,-0.3278717949572461,-0.5841099460663569,-0.002095821339423946,-0.5953501754779508,-0.20668160820495315,-0.8149557603312354,0.018871957239687705,-0.6550768745940088,-0.13507998141740413,-0.6718118087677014,-0.010259346353315974,-0.26254070787993844,-0.10638537735100513,-0.33942654204707445,-0.24842158027508718,-0.13096032808685737,0.05270255630916614,-0.16596342719392915,-0.13355903540358738,-0.1935707189954805,-0.06460301181920354,-0.19308205746887463,-0.05992688414582926,-0.18310019852689619,-0.09523042678845615,-0.10437748591066233,0.02195086226060912,-0.03825654381208324],[-35.215897291426494,2.6723754799052135,10.581098788668129,0.6119112340708548,0.26991509489706317,-1.3934910707213,0.7759530577534648,-0.5015775351693393,0.48007136773080117,-0.5371914864319899,0.4820183326423946,-0.042762272172519405,-0.1353937864636745,-0.027918740554545567,0.2118413501019882,0.048544263567657994,0.06658473780161107,0.03706002282567431,0.08593589181498124,-0.4755059337664978,-0.006589441118874007,-0.04764671500907918,0.11267755482201251,-0.360482294882043,0.3414868589523949,0.49840043990277855,-0.1327218519752233,0.06555095681032727,0.021536095769625438,-0.11630101219475396,-0.0821728297508033,-0.06762760077918263,-0.03567975809334135,-0.22086415305911544,0.06888093018168087,-0.06882696710008909,-0.011086849888958082,-0.0031041345791150933,-0.03965505436860278,0.14125429137913395,0.029965382551666517,-0.02545284687313352,0.08861615579677226,-0.23588757608548702,-0.16457414979808163,-0.05109333379189489,-0.14796114777232558,-0.08866680783371265,0.012168803115793232],[4.7407372622587625,1.4616933840624364,-3.7565189261660112,0.5583467381259629,-1.2600862525546623,1.0447543779626702,-0.7255903962176098,0.3194464189955089,-1.1076190033833155,0.5068281818725342,-0.8680595642381937,0.16978182488541968,-0.6331422311335847,0.31142966524005294,-0.5303448005972166,0.22255741258256423,-0.902761337156448,0.42610688979675504,-0.7229282986454414,0.12414288783923332,-0.5578979845967296,0.11448020048594926,-0.5409381518491323,0.06407324042939905,-0.41175650685942744,0.09846171243755977,-0.39589490730886284,0.1350343337433475,-0.5242036000456636,0.0485373521533786,-0.5550078056502544,0.04955861042757635,-0.3372362034116629,0.21022153450902872,-0.44268643188234297,0.12874246354321256,-0.06579605338299245,0.19664535001884567,-0.4057705867674357,0.1303872093521791,-0.27383701868481214,0.029636039614261953,-0.31274552844105186,-0.010612576133186757,-0.29639019152383905,0.11273649184125388,-0.25710805800266684,0.1587624513540904,-0.28714614857939885],[15.086254696277427,0.027822952129852294,-2.2252230910365287,-0.6912582197254024,1.0710689589082887,0.5252437168275417,-0.3591171633357421,0.3449619583158625,-0.03567661991365772,-0.21793883372107947,0.16657935897940318,-0.11679198562316973,-0.09829356274271797,-0.07791763234670457,0.6186406541629177,0.13888835434331304,0.20051543885118656,0.04479622011659838,0.2458213755257271,0.24112808860319815,0.19781677083664215,-0.10107843505608816,-0.07377202276843557,-0.16506990965836082,0.09984570623943308,-0.2047026134386928,-0.06482119994273036,0.036291377402587637,0.04510601629875825,0.0142859115441178,-0.13804833106040468,0.13682060878933325,-0.02456041411088375,0.13847367625365778,0.15000551207912766,0.037975746419820364,0.04430486524548052,-0.07597101653258859,0.09234097257097114,-0.04961357831950586,-0.12064674694412503,-0.139676070574355,-0.08615739812074595,-0.02841316125276445,0.12479714942235384,0.07891409724212792,-0.006042494629710535,0.10500341449662318,0.14450193889817242],[2.5109002747732254,0.8807043069019728,-1.8386208770969923,-0.31131492693957996,0.20689870369660424,0.1242961917229454,0.4137463015946017,0.5809564953063182,-0.4266521346611543,-0.33718705317652475,-0.002559708996561644,-0.24309903751444056,-0.14764724988465788,0.15859308029207503,-0.07386986579820479,-0.4100504089334643,-0.12336576194674656,0.38312906680996445,-0.014697700976309774,-0.21950734441123154,-0.08536953750558542,0.05202985341251038,0.05196220663136354,0.13030291932512292,-0.03494761924535063,-0.15221603483399335,-0.13276507448529817,-0.021714721415196404,-0.12364546142035253,-0.14312376135864052,-0.26275848961772,0.0012854272232098545,0.13851610126184205,0.15547041220479274,-0.19094574741747336,-0.009215835734202453,0.15034800130142345,0.040005122833072886,-0.03928412947935024,0.11091794927086274,-0.1472104960208205,0.0725858763367659,-0.004249626389828444,-0.04849284098965141,-0.018102737197708187,0.21173546275585095,-0.22648498180998386,-0.02915660660501313,0.0703074883331017],[6.723908775860702,-1.9797873728514295,-2.3644564650923408,1.3743413246940799,1.0550656762404511,0.3283002210505816,-0.6661140538564669,-0.22628001177654028,-0.10024026780631724,-0.7063391328108879,-0.15068506707896423,-0.06531233585244679,-0.0013663757100773064,0.019681871673965054,-0.2086905329916421,0.10466643129681774,0.17422961471836024,-0.3075117181489192,-0.33145836895823627,-0.11592103121531186,0.3040805183919042,0.20525250702482545,0.035237584023046435,-0.14227960132696,-0.11494126339878379,0.10626072836765915,-0.030698905227859542,0.06109790586853321,0.11646246270641589,-0.0061948118485500685,-0.060350397724441424,0.15568259044717156,0.1429970170106877,0.2227997891664106,0.010029467151594498,-0.05437093215237358,-0.30186915493217026,-0.2468926405541868,-0.027352759005658776,-0.01753061303165201,0.11657306106526771,-0.08573445372618785,-0.12769927111370033,0.047149266062158274,-0.10918748115632737,0.03423473623083755,0.052005058207886916,-0.06092306872448551,-0.0705041791293063],[29.41909608442551,-4.282606068763736,-13.575713825105444,-3.1546160688287346,-2.065537945917434,-1.385251473619406,-1.242301335918898,-0.3836806584421823,-0.7218377070280952,-0.1219438075570858,-0.33187053264665517,-0.603089203740237,-0.3419406493728503,-0.05336355219339533,-0.15430633718507802,-0.19908939876074117,-0.0714327222096167,-0.3028385690155177,-0.14684498985415365,-0.11652833737220639,-0.2831524448439612,-0.08464170777799099,-0.12496123784188727,-0.0610343553943661,0.08314001617261496,0.19537254923352262,-0.0504360002548245,0.08901326119975835,-0.0191572893900258,-0.07226483831729626,-0.26989467914362436,-0.008672656296540328,-0.018985414794308052,0.06638239930048483,0.049374769644057126,0.022659644717963216,-0.09321174213232417,-0.14367285580830216,0.07077178342605275,-0.10080776004206812,-0.10592257715579358,-0.16012583297985183,-0.14479889723907605,-0.06474841150219297,-0.11868376845664658,-0.08988241456708053,-0.08617222640203462,-0.01489719923781567,0.025676382836216188],[-4.796086606173787,-1.1921259687961727,3.559916133048438,-0.20904367490846967,2.237028694283595,-0.9763366222095784,1.3353723659967425,-0.7021472917197347,2.0970423876483206,-0.3151537366401909,0.3463631229543515,-0.6268479210665,0.8551348017354199,-0.33210178288226233,0.2621557964875858,-0.13067761524906701,0.16393618954044,-0.3308171520393439,0.29568052371251813,-0.37908865509549255,0.3014002127587072,-0.4729713019544286,-0.02478994727499129,-0.21670902376399642,-0.08379514494079984,-0.203338461723753,-0.13851267255326377,-0.2897224377628955,0.61297983499826,-0.13443069303219485,0.06641258547527375,-0.1942672582364282,0.2577299296798802,-0.1278412811655981,0.3392164615063324,-0.14950764403449976,0.35729407416362124,-0.10965434704145986,0.5074915145017872,-0.14269395995988013,0.8752707376903738,-0.1926484177427673,0.9365560780657162,-0.03068046094153752,0.680029020015774,-0.0983231278478074,0.6019930379928304,0.009610876414565015,0.8464541136506762],[20.12566994345372,-0.2559190247652689,-3.552003090144406,-1.7839505882263176,0.24304990757719366,0.23610488344547306,-0.33394357345675385,-0.7615877056973378,0.961039640236865,0.2981871606566534,0.7925068658763246,0.34714327386895627,0.12452904463772063,0.3732027457867185,0.7552402588692467,0.05115069815752292,0.4051846938963691,-0.14108091934332004,0.769633945483671,0.5620978590178134,0.16232656796107015,0.30038658966375065,0.8133197052947122,0.017266314539814087,0.37934018339159936,0.28809645518188215,0.36158812353811176,0.046541866793881954,0.08104374632966779,-0.05661869299632468,0.4580782747057296,-0.10598115315038519,0.34855865892391774,0.2977239402277199,0.4420714614716192,0.2215097206268049,0.2269735266637069,-0.0671119128696226,0.3923575244095613,-0.015173855424558383,0.28810346555951233,-0.05322691629271722,0.07555899775298096,0.13934647708996584,0.04806353648546758,0.05831857509920655,0.22151783365168293,-0.03286700492525086,0.2197907577603819],[9.333880475095945,-1.9989491268846882,-2.656831288280326,-0.9481778225696433,-0.08708128159644073,-0.10987845240101672,0.8742881792406413,-0.08445393296700682,0.578537398826092,-0.37021815736615404,0.37305107934230747,-0.23859988623405431,0.5547926662631494,-0.05680660800146389,0.3884416924893597,-0.5145908099539294,0.21427115057409676,0.4900209176475882,0.37132389444031405,-0.09186346921314117,0.40049795808770083,-0.3176048115498376,-0.14378154737247856,-0.0377848957044017,0.06729770293747389,0.11873742641413969,0.04822680295772343,0.04962975632857574,0.11164050002666182,0.3168588921643989,0.05341169451444003,0.11371901509308505,0.08332670015850022,0.015612512947468632,-0.02341022615516356,-0.10665978922912306,0.22943489299692071,-0.05708858133211171,-0.015434229540638778,0.02856831699519075,0.1141798324200238,0.09165687384266578,-0.10619430188119341,0.04508827055515762,0.10403701928081953,-0.052907390540855054,0.24390189086485176,0.02806802318586633,-0.049930310178691144],[3.1224176010047087,0.9624749532120094,-3.103161154355844,2.7441531599733406,-1.783954925930827,0.0602978754491881,-0.06185735531630336,-0.20894742662605542,-0.49051469403279024,0.48430608333171143,-0.548741112767707,0.08906707343131587,-0.31774437780345344,0.2779678390345437,0.07786508264135719,-0.32822500817906386,-0.11462030600738203,-0.049811811855481565,-0.215664447825518,0.27123759523100033,0.1269466551045749,-0.13296793441131993,0.2389818530273387,0.05119417181724159,0.3443236268584015,-0.03555464611686697,0.2542014984421701,0.034541872459784705,0.1098588404401648,0.09131245651704595,0.1889832215594552,-0.05877054378116889,0.24212044943763567,-0.06866226743741861,0.27166203746851797,-0.030324568132677778,0.33019127338938714,0.10256107846902743,0.03775188759760668,0.041943701987801604,0.1469518255230354,-0.022574523612076857,0.2241811608782299,0.0070900164021837206,0.18884308297751917,0.052871524941305786,-0.03438818036268576,-0.06833819233871173,0.22146372208205456],[-1.6258664841595107,-1.716455604292084,0.7326206802746954,1.2514160187479564,0.29767717226576307,0.5751752248109289,0.016580146770885113,0.40280440602409023,-0.03035351797034951,0.057955187694005504,0.10211478380107537,0.24365072977297197,-0.2039970770219024,0.05916153595509652,0.06003826153983924,0.14140425526978545,-0.2977859350700957,-0.21202211463416307,-0.03171854776470926,-0.07120529783860953,-0.1542273135534633,-0.06408871671415808,-0.04481221940695951,0.02632358694050248,-0.40014722898285543,0.08510930639123854,-0.08911601860442003,-0.06409784321362594,-0.04759879523867405,0.0008928345239412922,-0.011474804124325555,-0.1598165731205184,-0.14262375155446283,-0.1822093255370873,-0.15168874503905752,-0.05629503628644898,-0.038527435095289844,-0.00702295152013071,0.10947306999486912,0.07516898809032121,-0.0790068836725863,0.0011336957860101054,-0.09194386983254399,0.09473363102007155,0.05810783064180803,0.11288785710881385,-0.05387254053743966,0.07331287206049066,-0.02633752202563845],[-2.727876657100921,-0.2020598791300897,0.5242364018135748,1.0056564952965834,-0.7066905250442447,0.056772232702817976,-0.0977652035879959,-0.2045170208751413,0.0777949260042575,-0.11672777211988089,-0.14746258160643577,0.06617111970372476,0.04613539758378661,0.010058897295908331,0.054473955022874145,-0.20390079279661336,0.18188337853139153,0.08327158368706584,-0.009100419677518139,0.013648303094299217,0.014313430541803085,-0.014647456771874548,0.06468222028497064,-0.04072146544044715,0.12370623535407417,0.09709758921901628,0.07881196026352147,0.01010170819092339,0.2319843899838182,0.10466811968770119,-0.1818849575711837,0.05233764679601835,-0.028252897897951986,-0.029402849158316736,0.02409947519460921,0.02715146607909236,0.059431702190453684,0.03891816032480413,-0.0739995294751259,0.11782029872688796,0.0793457619700714,0.14157680138913367,-0.026299939380316984,-0.015335922120529954,-0.008034211842546369,0.0007386508984469836,-3.202540911337697e-05,0.05158284818040757,-0.0802078006144799],[7.632825148274116,-4.434044895437038,-5.962769524450894,4.980445496868347,3.888938882198688,-0.17282720855757897,0.30225272271222575,-2.233002339651203,-0.9440661615163954,-0.485766880106703,-0.8900697940092248,0.767617525983347,0.664834873197191,0.3582656503217333,0.32215459185728645,0.19838019300581092,0.0827349287686507,-0.03935429614230533,-0.16951948016402016,-0.05100815089917906,-0.074377774025022,-0.09673838986588539,0.22004067319675952,0.0043966430875599,0.21140567976357005,-0.0661526196321488,0.03085137524693192,-0.15903528298781727,-0.0026712037073793955,0.04750385483042227,0.07425687722370677,0.10887825861663335,0.08299082997166668,0.20503546509304427,0.11284659345695787,-0.07684781273888391,-0.121671827501804,0.33750950447323425,-0.23497190053481484,0.20249542486511823,-0.123793944931879,-0.09409701940752956,-0.09433608993732315,0.049395885182821925,-0.07780098629113835,0.055305389044203145,0.0966311718503157,0.07674172211194813,0.06292564422868824],[-3.61881828077698,1.0812125427466115,0.8022490563908347,0.5218605609120593,0.6788609286497759,-0.017352674606856848,0.6927686788714054,0.09710000929946332,-0.23805680359410636,0.270685308430583,-0.5060796456317288,0.38858766483851664,0.3472448598029644,-0.18370652444292537,0.057804739268800136,-0.29617562780074796,0.0916350272454587,0.26489218819148524,-0.4561706152318098,-0.1407340717328079,-0.18975605813269508,-0.03975049795362626,-0.028143770645950247,-0.39826111211431503,-0.268318408611998,-0.3166197040290838,0.07236694802110245,-0.12411696514476019,0.027074033671367168,0.13906720389416108,-0.13283381719115048,-0.007033179202325456,0.008058169002479037,0.07990877837445536,0.04171460026802328,-0.013036859299271094,0.04591025556474556,-0.08202756760708181,-0.20827794961850915,0.037242998315163506,0.01602116071893971,0.009749166155108962,0.08058193019506055,-0.13958988538732867,0.046505380236755496,-0.13682095002482964,-0.06476334290253326,0.12625074821412813,0.028290992071137517],[14.47786679482427,0.8926505402767316,-1.7081149619257194,-2.9673607207639794,-0.6656367802932742,1.6961522149455555,-1.6413091141878955,0.5088993304695123,-1.5234376210300298,0.17329756257346837,-0.9200069070424521,0.945602210008888,-1.6719970346579622,0.6416799416251728,-1.6518242764990207,0.7939531777616199,-1.3883423923017089,0.6459593944360577,-1.5104451697324566,0.3340091460745203,-1.2935091830555712,0.7068023208730039,-1.3831479094429602,0.6332191856487016,-1.257180454138035,0.2810979239309449,-1.330560954342055,0.18591685205629746,-1.0809565120664686,0.15053033640919422,-0.7883719598029889,0.30785652567677263,-0.9310356064261518,0.13559279596065285,-0.6068654643810445,-0.11381769124499974,-0.41299776451755504,-0.08873292492123314,-0.28079885963858414,-0.1932156964757271,-0.12380970663299727,-0.056659920308478516,0.04344663245154822,-0.18623319737005825,0.03625894020071308,-0.1566174375948181,0.05630579865282181,-0.06642747551842837,0.1216944199052542],[36.289920804733676,-2.5241868589793537,-3.1204357913463125,-0.22563376317706912,-0.8016134329192879,0.2329406801453268,-0.7446481579772092,-0.27804312016733757,-0.415974461650089,-0.14742409671279802,-0.37807512545097743,0.1025396401373455,0.2215523755795216,-0.023540074877538255,0.09985320527481956,-0.17647599139195552,0.15581161102854438,0.02190479585749688,0.0391945771201275,0.19310726380082271,0.03937857137401828,0.23765584672154258,-0.18581819011456782,0.03398019002214598,-0.10926851215198613,0.07225389572911942,0.09708076921339696,-0.008712513087942334,-0.1358453331936205,0.0018582614371870606,0.03318399151992605,0.04247205033693988,-0.007912157490771538,-0.09508423165208156,-0.09183178296743211,-0.012338325438787193,0.12231849516574514,-0.11761597617117232,0.01443709571268927,-0.12364427561959714,0.1137395413693749,0.06664651029534167,0.07260408255550294,0.19011912002244583,0.007572678097057946,0.05293597933175397,0.03801871916461496,0.07443940474855663,0.06879183953719846],[-1.857212941569605,-1.3909016636298281,0.669521952411509,1.1312788856032792,0.21809079808146611,1.0309485465818602,-0.2161831636397943,-0.5150835765912758,0.005414636066411467,0.26341206698806324,0.13731319820164695,-0.26016194427755335,-0.31694336484006613,0.0994860425130467,-0.2796910694638014,-0.0373712219700883,0.06151840403856339,0.05319266589286822,-0.13279106722758466,0.12590138859663721,-0.10844277763186004,-0.07382963223779762,-0.249670258598733,0.06611281763495791,-0.07982724824149912,0.02945847507479314,0.24440025652995373,-0.017447527808996423,-0.3100557500024111,-0.020915226864451717,-0.04856355426203118,-0.08028803836735554,0.06493672972572262,-0.052747702183490314,0.06249092791205893,0.11348272625792263,0.08589730906531375,0.11787390461710075,0.04512939824899665,0.07792784893127727,0.1730656596911391,0.03658120022666275,-0.016502547487636316,0.1361105937442246,0.21966204926971844,-0.058732280657445464,0.11365621314602485,0.007073894216636503,0.16097891052083427],[-10.575793797257722,1.5549675201277526,5.635658800983488,-1.7870802468233293,1.4523191071001558,-0.22957281743409338,0.20759099981603726,0.11140686277608881,0.23706214371862283,-0.06395581980385708,-0.36805017429064535,0.09806667240918156,0.2943167760265217,0.04586301157223069,-0.19613450045321665,0.3388883000611654,-0.2281750855605005,0.4467339647969908,-0.906881590383578,0.6260607876814427,-0.9465390888926917,0.33126539055337567,-1.0064104739811062,0.608463522946456,-1.0769686540960846,0.37809581450068497,-0.8657961401593025,0.6275644718520826,-0.8537650273392191,0.5245448001069186,-1.3249262120796532,0.9126656011618046,-1.0687151071815166,0.511507648066895,-1.040851109877227,0.5853522389688093,-1.1529720998421207,0.5088529613678112,-1.0185732742564166,0.5243527694626897,-1.0324547356160574,0.6253167977805104,-1.0942000246407786,0.50900838087038,-1.0040861360021536,0.5223406923445639,-1.1435656623240738,0.5541177965465077,-1.1366762390685743],[-1.8269089241451593,-3.03426253112788,1.1095674807708253,0.7392470454007906,-0.5286490361172624,-1.15377921478943,-0.3490303802930419,-0.29441177308982663,0.4243494059345714,-0.6024242337176882,-0.21477471048272664,-0.47409129261747907,-0.3730764838039066,-0.33642919513428304,-0.29292763595030075,0.14121622019803554,-0.06441999377057754,-0.19668868082409427,0.1546192625782102,0.2867655105052771,0.16070793613833956,-0.08522286735446942,-0.0303011234417145,-0.32778970154269993,0.09190379974279936,-0.29754855030705624,-0.18273493151975845,-0.17583580519575862,0.38462301448960384,-0.0857391388337532,0.04149666180270506,-0.03991307090715135,0.10001171775761021,0.11980519113172332,0.09623428335094521,0.10050947992319934,-0.09911809081875847,0.1286450664520099,-0.0323211172913454,-0.20100920408900677,0.16390422487344927,-0.08490626914686085,0.032346276697697254,0.0242654041665189,0.0045469224886406125,0.22145059921015584,-0.2524068066185433,0.054516017290988456,0.10089168118433055],[-13.348440189513477,-4.130133640162137,2.0372372040063675,4.058555868867564,1.0905213441394632,0.37748896986957,0.6586536401026175,-0.33499060726191765,0.6324392615371125,-0.7791124867289247,-0.3488907107686563,-0.39001448712219317,-0.6251601561337189,0.15463331879717676,0.03549902803945869,-0.08718806733446177,0.1882321544176233,0.35523341326859087,0.4328079307040889,-0.1253411297449123,0.031067806069956655,0.054649936131876936,0.08874108817759965,-0.26777254573477577,0.21479777761544575,-0.14605258047531913,0.023115508375102893,-0.19547304477934227,0.2878746223576567,0.22568953089509552,0.2853455289023548,0.20957035525126122,0.19275664884689048,0.1113897418838667,0.052061848372798814,0.1764689786761533,0.24763541550830698,-0.014333886445960183,0.1945445855717724,-0.1579845982628593,0.26300374910810886,0.14406005740214525,0.1621734998307283,-0.1272416143470934,0.004382074491651254,0.028993220710519865,-0.013742241586960377,-0.011688255326110505,0.04887496590300962],[1.3947076700328906,-0.425893575449161,0.6180779950467092,0.43952970283303927,1.4483366830909283,0.09355479338705106,-0.03147576629787032,0.5286144325209963,-0.3153028123048104,-0.1593339535851245,-0.04152199576484194,-0.26684785120107796,0.003222823433039831,0.33810382877335576,-0.08933866870490872,-0.11476370430428327,0.3192596560271662,-0.08197934535146077,-0.08962150531915225,0.20509135134384607,-0.0438418562805743,0.13157089746577086,-0.07827872930036706,-0.129996238689648,0.03620996929815174,0.1444692322532646,-0.050727555639864454,0.05949972500904901,0.039831692908394147,-0.029517028569491486,-0.009254962405411769,0.04629904554081068,0.1584639716042418,0.15989898883259596,-0.0016743149131239533,-0.16918090517943984,-0.09510550222219039,-0.05540831223840879,0.004197768722627883,0.05084406174651041,0.027987887563054942,-0.09876897326722746,-0.00020781761651400465,0.010153483542383724,-0.004145987680607341,-0.008184175626855396,-0.019822147898138884,-0.016582927605237167,0.0046650744279772005],[2.0176057171683173,2.6775908461760887,4.129530812216411,-2.3124104611570697,-0.6586539444578325,0.3049842432653736,-0.2710369634294409,2.045713023058364,-0.32169271825189333,0.5944279630514123,-0.31005414389924174,-0.19332730311874843,-0.3416892713800557,0.2835039992953145,0.6138394028970765,0.3175758831704879,0.2091198938926449,0.6400099655537126,0.5438104311841054,0.04370791520870449,-0.29754585786572063,0.3733224414857287,-0.6652553359061496,-0.12193633159411083,0.26018183474936596,0.1002507782409915,0.4497941161079179,-0.2019256843654627,-0.2757963823297097,0.13722594599056004,-0.03417977056954615,-0.34989842753816075,-0.5871146159898514,-0.20889977126244091,0.02237046394566199,-0.03754471162419722,-0.10278458004986715,-0.09749754317706792,-0.03637683603645947,0.11498788440934676,-0.11737195392604502,-0.010130361382194577,-0.12519611251338805,0.014815883282336361,-0.29086000725529,0.2226532509491849,0.045102486193815974,-0.11051646155672963,-0.019683729934302008],[-0.42107919799481247,1.662209633863758,-1.4192079948138132,-0.08078576291902857,0.3533182049199373,0.3791086193449683,1.0594462717134652,1.2257957290285606,-0.23469456014469872,0.990965604130149,-0.06038277839089317,0.7157188382720309,-0.04239790778443818,0.6039730738307357,0.3407024201267145,0.19144480358834398,0.11563039993541917,0.4243012998281487,0.7213608890181886,0.584816491944769,0.23396601312030058,0.33728998585469994,0.2618028937244851,0.42631280681575423,0.6848442900740083,0.44193447817759296,0.9125735467699885,0.2493721913481048,0.16308679237646892,0.3450492425695525,0.644205005435866,0.43576072431948387,0.40143939205288615,0.4179731325351339,0.47718882059650125,0.41259671157130123,0.2930905346691651,0.2984899818059988,0.20490321902616948,0.26412258302930774,-0.09933030506638929,0.32766336179936,-0.06492540694116702,0.1080824194622242,0.12212917574643235,0.14827973132130562,-0.2235732880831318,0.20160993527822782,-0.31695768901230253],[2.1862274660496674,-0.8934668979722011,-1.4649870907584108,0.4019430479065358,0.27997612867183486,-0.03195848511894017,-0.10656475750452399,0.5257774739359826,0.08308095433476151,0.21536991436258396,-0.3949349703196334,-0.27737274703589887,0.4082980617484139,-0.5926411832871912,0.033451853342630726,-0.16234898210374102,-0.01498487669715444,-0.17934428170512823,-0.029680974969457773,-0.2475795093235142,-0.18837463657606957,0.27756012872242514,-0.011180330980337803,0.24772917114635642,0.17061982647206225,-0.20227136806146837,0.38514792298846434,0.002277869290476349,0.24576118484435053,-0.09704484902091302,0.17871862445238532,-0.01779526613869121,0.33882020510972294,0.2481269723850058,0.08825955255503741,-0.14935806850792374,0.14093364006996342,0.06144840627461008,0.19135653582816092,-0.01593853908387169,0.04042213111330063,0.0017378511649101345,0.1959089363672085,0.11910315152487143,0.08331614890823376,0.04939417797984778,0.22500202634625158,0.04686324510505782,0.09419032531586716],[31.112401549652695,-3.4780005248291443,-7.5407458135239205,0.9352078623245432,0.23122354036091167,0.33373215141925233,0.11983527523490212,0.4576947032428217,0.4316612852206129,-0.32633975373828056,0.09283327650501104,-0.3423264608782144,0.4785367216218861,0.24569977806171567,0.14557437060385603,0.39293880944016674,0.6301959698706935,-0.06730352039920443,0.14047490689787895,0.029150971561525928,0.48679610592130196,-0.36629801274984314,0.05177592340753529,0.12790598785617427,0.21794165059346254,0.0048380389142746874,-0.07729634893134295,-0.06457426534787693,0.2969442352592794,-0.09776692365512142,0.016011625577041697,-0.05222684937160561,0.055658308989486566,-0.02605022072425211,0.28500398075632954,-0.1831967350538785,0.10523144495728408,-0.09557027865167766,0.2504344673108574,0.22153077177134453,0.23747677735737707,-0.16523467841093126,0.2027783824538405,-0.07771274272403979,0.17499496565413653,0.034172388510655564,0.18773406080684016,-0.03893426498525549,0.006745099314850752],[-0.1444963562069036,1.1285024413192324,-0.4834471382179017,-0.72138238790116,1.6675346102210615,-0.6339871269592486,0.6882068664927599,-0.2113917718962684,1.3445664229308518,-0.31309618419971114,0.5601535378746023,-0.2541812417295236,0.6601577261333872,-0.3413750510021162,1.0008864528074684,-0.16868798995990633,0.43488257631903277,-0.1637355907785455,0.508755893695627,-0.23331988826758304,0.4090541901838164,-0.2990929823538692,0.3632766249294241,-0.38209246937560953,0.45266915534929425,-0.10264423144581959,0.2446790483202445,-0.09759171855932586,0.2247910245196629,-0.30161723610549107,0.26091114321840725,-0.25460357404979894,0.27275845254686387,-0.4782114750294345,0.3443966351761666,-0.220981183365677,0.20298164723987955,-0.36642311748519235,0.4211603372558039,-0.3159095088062859,0.2843183107752864,-0.11994380152755056,0.46102695755510087,-0.2737558149599262,0.39908104339027506,-0.2581973704057521,0.33066676562906033,-0.3135913034846304,0.5115585116561413],[-3.48302732370366,-1.0012567803266037,2.764852016371223,0.2506007141362647,0.0847134961869223,-0.04233502697248579,0.2173342818666405,-0.14992082701014395,-0.23990531746459062,-0.2622454789916825,0.44299735405296276,0.5206965010063478,-0.2177092341269599,0.1837091577583477,0.20844511470040336,-0.16145935480376544,0.12356707286287354,-0.014926871993492085,0.23075092495683497,0.12260057724858624,0.18812411323517636,-0.1643618687387399,0.35528859158168985,-0.06367446722118578,0.20238893932743104,0.09480177543909223,0.39913746318179444,-0.19978043319461314,0.31809000984680663,-0.10567003936184709,0.26673147686539356,-0.2839872386898007,-0.022435349543944556,0.005995683804412007,0.18263785675133604,-0.1829345933347127,0.24836497999657448,0.00490113504329051,0.13342825738280978,-0.04773762074068966,0.21580326482195478,-0.0954293900089202,0.07385292486883113,0.014508485713878942,0.04392608597049903,0.028778343819378997,-0.025453509841466406,0.032396657921095626,0.009516666230423462],[3.349745207682951,0.6635154352412601,1.2347332402433484,0.3693082763289818,0.13858605081696318,0.5672666400870555,-0.15480751122078693,-0.021815066000749183,-0.4179367532188077,0.03893815924594945,0.28732917737502967,-0.2345451957736751,-0.2348086627957568,0.2540391302150273,0.06854911376684757,-0.09781888357638874,-0.27594837439426256,0.4453923138385478,0.1041915113503008,0.15288692272056081,-0.19302363598295505,0.3994157228956796,-0.04645938279559688,0.3628497701499359,-0.18858844842470862,0.014788885471427819,-0.2937713748449975,0.08489896292397243,-0.5394847891031487,0.04161558009442307,-0.37389758662416367,0.0518918315627178,-0.6017056751734252,0.09448389270689689,-0.1419113179365784,-0.01969502016903157,-0.3547136907571604,0.20570939426116078,-0.3687423977254117,-0.24269644099142484,-0.16631881108967037,0.08771945344661926,-0.10842742771731156,0.1204960831026957,-0.12861956043166878,0.06012986149255888,-0.24907627018934925,-0.0029368481764023057,-0.1760761965679048],[2.928275014233954,0.47211774121388833,-3.25506312512126,-0.13419212777578016,0.753766551041001,0.10628918855109276,-0.7109459561946417,0.12784084234449877,0.15832789660013333,0.22305011814422337,-0.49018906230107023,-0.004059121716456591,-0.3348938338187865,-0.13243170134363969,0.05011523151616108,-0.13526140852983762,0.13449340954237746,-0.3377549215554435,-0.1258323358246533,0.22349492087787395,0.11957961320815258,-0.3070847419113217,-0.09873063724439246,0.09242707598585716,0.03464602979412586,-0.03417853828648859,0.3263669802308617,0.0052688445712557146,0.1972046217636613,0.0008100920923089279,0.152113526609587,0.06065238928175408,0.07462630919453213,-0.07770874855005579,-0.009812221078549104,-0.13655954286957933,0.12151095963035231,0.03942948404420351,-0.009295412723557779,0.011259468083782047,-0.04739259928166989,0.051682011345020276,-0.06432697116948966,-0.03557298025195317,-0.14101699113434396,0.08914881680573143,0.036325764455547895,-0.044035064928258434,-0.1152761511594694],[-0.4214862467124887,5.2707012197911,4.327702057027797,-2.793712558494125,-1.6405733758213885,1.1736918835336496,0.31487168930862697,1.071703237020169,-0.7682379291064955,-0.6310089951249567,0.21292065997920884,-0.4254447312485529,-0.7704459477684045,-0.7353456126992878,-1.176168730803954,-0.11742772769132363,-0.2974742590965692,-0.4020502908679418,-0.8716867994696723,-0.7329025253912197,-0.2642206138119507,-0.4260137437149673,-0.6659570695510341,-0.2965118139904508,-0.2613691660148325,-0.07621371377443178,-0.4638883587905399,-0.4435113040031365,0.21950852602944645,0.05193900040475015,-0.09218371912002792,-0.16842222436277263,-0.237952403005379,0.15994189021448843,0.1184977665176704,-0.00390732371177808,0.11992492160366636,0.030930120940428428,0.2653951454019342,0.33350899795660355,0.29081580839198884,0.14449907447687832,0.09452936700174358,0.02889429459357393,0.13948058155621917,-0.2412035300988531,0.265213541098992,0.029104916954012773,-0.10638283029523267],[-2.6249083413782217,0.5930553867995597,7.26090408512056,-0.30791117952382463,2.5968544042607853,1.5478660556557882,1.744537889838475,1.311751672828817,2.263637060895865,0.7337163209616703,2.896439428373949,0.43850231032542303,2.6690542323676008,0.8514878847016211,1.8431010596520094,1.0947092425716445,1.5803968566556708,0.36973281909196404,1.8319234263215565,0.6717569006591135,1.9683259180449906,0.712962549411451,1.5576070652447447,0.9504282928619252,1.3584204835895717,0.8977454978176164,1.3267398322275448,0.5812938218204341,1.1208492640755228,0.4386912785388716,1.3058610204538068,0.5015693938177074,0.886672603025203,0.8448707068238797,0.5731289918761044,0.7397562125200089,0.7984687799604758,0.3379588658848432,0.5370327624793351,0.4523844154754954,0.3277812296484611,0.4938291687385715,0.30611318801170456,0.3761732893755163,-0.14745609720392136,0.29512700310503076,0.004355499775874431,0.1895834602582721,0.10910889603448544],[2.817547805259219,-0.39705732587888865,-1.3516228614500292,0.22324457950042886,-0.28707725452211724,-0.8459511318792576,-0.9683456922516114,0.05586510620369098,-0.6658633901058787,-0.2799045255769719,-0.6367410350426249,0.20370211935040974,-0.842164343176959,0.058900812198435674,-0.713984366236681,-0.04712682795205958,-0.6772649032437863,-0.013453577013000146,-0.4553575829411617,0.12227265161451972,-0.42858930857265315,0.010119985767931463,-0.15113811834176688,-0.17266269278152477,-0.5136476676499836,-0.011384046016665584,-0.4510838847763254,0.15686501333820088,-0.4535776421452778,-0.2674569397698121,-0.3323943630341821,-0.059916181217730326,-0.20566269593394484,0.0524814824361124,-0.3697737908659515,-0.14104697677223715,-0.06273706982237968,-0.0411750927879496,-0.02649350142317559,-0.020740709884525928,-0.20548605445312385,-0.07512516626796875,-0.031732473351489716,-0.01823585610268291,-0.09926732074804209,-0.14916798607950515,-0.13083435910227134,-0.03124116947316925,-0.17787478269595786],[-6.7065307097147855,0.2828515836291053,3.1254095482378292,0.7658346541391748,0.46994990672223264,0.6378317067123256,0.16759606038464306,0.11189132336112054,-0.056968742879489505,0.5699878147179519,-0.36823580275207596,-0.465229863706303,0.041496553926495594,-0.11249528775987837,0.391681190046948,0.3515935099016312,0.02113179917284767,0.22258839739484743,0.033401334633586334,0.35458952614838507,0.3630771362027715,0.3879517047957144,0.2528416649714773,0.02050266893421974,0.1914428315108495,0.10459780629972787,0.10214199502771436,-0.21877245467813464,0.14291965336176765,-0.308494701250347,0.0709908322949897,0.07599671147187308,-0.25385706041806305,0.27505051654957935,0.07070024325748983,0.11117998405996284,-0.06298231172341884,-0.04055590342363928,-0.13289024322030119,0.07472865749634768,-0.18580919385066808,-0.09066941600754024,-0.043490698179779144,-0.10793212208235613,-0.048285059027920574,-0.16375295345014773,0.06265196172214424,-0.2106233138999668,0.02615271888008235],[-7.810545657505003,1.4467298654107823,0.5879190217986009,-0.19174130192853248,-1.2904177099066338,0.5242980996171003,-1.576292001341656,0.6349245135845434,-1.1747896726746696,0.4936786319758064,-1.1957585055836168,0.3181505412817195,-1.366267849103192,0.24651018638987685,-1.1794070159109133,0.47669240963950005,-1.06085820116182,0.2296950000369925,-1.111767716160445,0.37744852101769155,-1.0882474075441766,0.4641144158181273,-0.9193713957332122,0.34173077486492465,-0.9293950233339238,0.3229026417779256,-0.9278745547608738,0.2271188626470508,-0.8306925848568144,0.3638020456726004,-0.7083979019824689,0.22140178601526517,-0.8054758019889894,0.28117822303311435,-0.5933865421828974,0.06333254123995989,-0.5261064007957619,0.0674316850830566,-0.5148334518496059,0.004322020443918876,-0.5153926091119337,0.06470641705786775,-0.31016890967641164,0.09885986812968903,-0.44190293815535564,0.008081143984128333,-0.2682767271333119,-0.0018107704456400955,-0.10136977744412771],[18.627750302573794,-3.3044351353190313,-5.728077049660396,-2.228181766242528,-3.101944533104391,-1.8116709706127063,-2.7915969413694555,-1.614345653340812,-2.142657916170195,-0.823199712442115,-1.27491642713597,-0.7665801120525149,-0.7030919820840748,-0.918117663144747,-1.3180260021823715,-0.47930525816827024,-1.0306916910633788,-0.7452776030079287,-1.1210196675154316,-0.6719308642381792,-0.9179164316248567,-0.4641254651597389,-1.0610713032607781,-0.3695693591265697,-0.4620585435162983,-0.40704684726501444,-0.8320090541308895,-0.3652561026479145,-0.5635717142990464,-0.1079353044548683,-0.150243972786766,-0.591621796217307,-0.33294976893965694,-0.3397774846188884,-0.41499658294276986,-0.4957542275873638,-0.355444740407465,-0.2512356121973294,-0.08521523986067618,-0.5151195879883184,0.03793453967865852,-0.051115715849463686,0.07367333905808351,-0.20832322262856998,0.11351064632311385,-0.16892900985513318,-0.05513053709987838,-0.3896549090053921,0.11520753416727604],[-23.543225705964133,0.5862245420673352,5.413563711499239,2.4409676602604637,-1.1025298761738203,-0.34488393718853855,0.87039525406496,0.646302752350179,0.3312058024188557,0.37116788524539984,-0.6534539751436594,0.5475858196568668,-0.14659618986342687,0.3143888588269863,0.16644567949164096,-0.16933932824770354,-0.28780014069879106,0.30193983492971077,0.030582574145556696,0.07949272405870825,-0.07581781374571457,-0.04805498211001697,-0.3773681996055905,0.42650070867151074,-0.11143951065184537,-0.16972072726349827,-0.28833548813987747,0.02635758986049203,-0.07044592643938995,-0.08653140142322767,-0.013992372763537838,-0.19171379046803974,-0.22339455607630251,0.05147843860214577,0.06776519200278673,0.08140294220648046,-0.11352103257216647,0.1366149902461507,-0.16130817850647117,0.0286359369453695,0.18478070962412146,0.15061634339814617,0.027926450252723403,-0.06815840974661087,-0.004171335021056904,-0.013190609087713816,-0.16199046270898573,-0.12968551459937822,0.08191394757990313],[3.49747595997573,2.4664120759325416,5.395418557892996,-0.6900213355958627,-0.9206184579445524,0.9519417960552763,-0.7836529778756791,-0.18901199876330826,0.03564898168032069,0.18176812763830577,-0.10099907606345608,-0.1331291366426581,0.3585828200264278,-0.07623682380316385,0.02718700240428476,0.18692677501828472,-0.26843743849692875,0.052483071292449154,0.08252449864658527,0.11412332736161192,0.005393317488166391,-0.060924405383769684,-0.05259397261678466,0.12865779896404395,0.0272733476420393,0.021679827007956946,-0.14555820536369543,-0.03607670558927383,-0.10580346930847391,0.057692449171470414,-0.1472360126714499,0.06527751563315615,0.1442011559855392,0.05396460682763182,-0.19747976598749525,0.08838488173796194,-0.02980904852642016,-0.028397429095545747,0.06204837146646415,-0.058654884472846856,0.023182812525349792,0.10958005297241526,-0.005702188373416757,0.052811655871799826,-0.03706195539090446,0.03709750695125535,0.04767510166861235,-0.013065701048429008,-0.05417799733361104],[22.183627470548082,1.8686379812529648,-9.64541149917119,-0.058172252864821535,-0.5687995802426635,-0.33101123072343847,0.42582264818602744,0.0457560290596785,1.292439315572799,-1.1358726865576023,1.221741997296958,-0.08882686457724973,0.6107154069634697,-0.05243992516145811,0.7119161513264382,-0.1940912495323937,0.7211841186219821,-0.4910118998850797,0.7292682944367507,-0.9987348182862483,0.6375565002272069,-0.5357992404623533,0.6756070978920335,-0.22080880245909018,0.35880283564576193,0.10520679700370936,0.35942399955063276,0.13208841757641093,0.44755024104307234,-0.12599060201359855,0.3697319740989722,0.09368973162032979,0.34219844440207203,-0.2029603117944305,0.23947130697768712,-0.002141122375865383,0.2037843751211978,0.02126195924922328,0.26385245974662175,-0.28146672443041243,0.33076393247450075,-0.06908661677083884,0.15249621548598488,-0.16356013052030577,0.2856459638737286,0.06840016529813393,0.14266712440822604,0.139285924285307,0.12808174985794402],[-6.835890564384176,0.21596497669774825,5.8046274824321955,-0.7084485469038749,0.13806278418506793,0.25118205407319505,0.7211940089367015,-0.7604957423499696,0.7852905370030941,-0.14461042384681205,1.036136627910759,-0.004539355479909995,0.5123848774264061,-0.3521364567821971,0.4806875180135466,0.23362474757247417,0.2548829657580212,0.18601822142853153,0.4374964245129746,-0.013726180651004359,0.24949962684664134,0.15480190615513917,-0.23788812196511522,0.04015572283102831,0.09125049675858357,-0.14998925459102874,-0.12026710149387007,-0.2804937224458297,0.0856048044881223,0.35695845979300994,-0.0883472635911438,-0.0762944256571071,0.1271090749638404,-0.1757025888139418,0.11381891044777569,0.022449379693815236,0.029297480439841005,0.03220635030558909,-0.17251884717512564,-0.08728460969155057,-0.1813388549829856,0.10351094023359331,-0.14118225483414024,-0.03514592410116215,-0.07368195133726464,-0.06961888778686146,0.1392623715023485,-0.0011556243505250934,-0.11509111337287016],[18.502412928932742,-8.228267173681616,-7.1682532951381335,2.716251954101125,-1.3061332807646118,1.2097811267710703,-0.01906079420352834,-1.1020802145783932,0.41108813449474946,-1.8113157704456497,-0.3723604423884696,-0.4195462259986852,-0.10720078294743667,0.08373691303990755,-0.15036823474973798,-0.07945944301054443,-0.3741887066373863,0.3944671302681608,-0.05576677766742517,-0.29068088262860203,-0.016284132779200402,-0.5130566704984484,0.18356652156842637,-0.4762021665731243,-0.38594696670374,0.15476435519547718,-0.2862369744419996,0.32598705077115414,-0.2854166845349536,0.01936698213402721,0.0570073712498216,-0.027679899487707316,-0.039750652627641526,-0.18095038397812757,-0.17985457475691563,-0.009226777001636138,-0.033592647903410294,0.14996362540312438,0.1105904500658012,0.2366243367532318,0.08138861228702131,-0.07309838345350464,0.1316958058579402,-0.1295054345243117,-0.12393551439297715,0.07868190082489351,0.1038557132437823,-0.0235981930071291,0.0146960816077974],[-11.760087247853898,-0.36696904316269746,0.7505650109741088,-0.6902000842795195,0.6910253973448437,-0.21743643534692414,0.20884521097165176,-0.6513813170055501,-0.15050324293415296,-0.3254204987919142,-0.11659173683410044,-0.11289316487371752,-0.30373766932031165,-0.2514797215463635,0.10123473498471601,-0.47741273401468126,-0.0380989461103551,-0.3585330246074232,0.07433331326336357,-0.16928612877105179,0.08572524705901363,-0.12818186873013204,0.19922918365614853,-0.14079096158394258,0.10947265534930889,0.11498320863572166,0.21568496195723202,0.09915671675852386,0.29683884352894013,0.4626528313327867,0.5478189714121715,-0.14650707189270307,0.26106416849875896,0.09065057840860935,0.4261995593194632,0.3272896030474448,0.13339513375562498,0.06660021515838625,0.3032280758131363,-0.09457833075121093,0.24155030177172193,0.08482797554269399,0.08392940370050886,0.0006924087834793404,0.1930765427773637,-0.23985204232679502,0.1918807919770572,-0.24819801324891755,-0.11789598727963758],[-2.579254654363519,-0.9413285200557064,1.6603850925589723,0.4737505583278594,0.6406025609764541,-0.5172489158167929,0.6876849560862063,-0.18538391671780352,0.8825134228851498,0.04518912226001965,0.8291461610575745,0.12790077015304122,0.7412815533792929,-0.050909138906954865,0.7606525347749677,-0.10847045811561794,0.6244399102302545,-0.1817014881455522,1.0393050397958286,0.1043812648044882,0.7835604375727484,-0.06390111526641101,1.0917736158695068,-0.09528120474265175,0.708411537470111,-0.2770401147938537,0.5664658916188635,-0.18744943996134533,0.7966791256087056,0.03622994608420843,0.7805695573532724,0.03053281080707391,0.7108533776435829,-0.15003039560782808,0.9043267325451123,-0.07541770268760652,0.757517318204072,-0.20717908826243214,0.5646175692542844,-0.17564055074327897,0.6103941774473789,-0.020634976425795596,0.7942204146583941,-0.0239632663217357,0.48626471313741404,-0.11756669586817955,0.5416268214935607,-0.16587269499752808,0.419437882183898],[-13.60102357149444,-0.45002640666724986,6.5654335565000315,-0.25646363276387485,-0.18611764946626935,-0.16704593519137093,-1.3879538947008974,-0.8455979572390527,-0.6940794186910464,-1.0687834708923145,-0.0844596455157272,-1.0714145482214892,0.3522013611277567,-0.5457735325694404,-0.7827828037242591,-0.4006376193583233,-0.8018022310690375,-0.38055779625461783,-0.21360060143906304,-0.7871775293241913,0.27043793049464643,-0.2891121519121055,-0.08323734403279667,0.04991011332001971,0.10032674342117376,-0.07507344664426618,-0.3878332116341888,-0.002461374057652269,0.09420875383423033,-0.03867548812724649,-0.05247685106120688,-0.08121559727815879,0.06315407246737789,0.09328137275934069,0.21026366026628798,0.12211317968916582,0.26912881053377746,0.10117859580907192,0.2280713130817242,0.17183626104314806,-0.010652141221720313,-0.07389508694261751,0.2961318229950942,0.12015702135174806,0.009510619411138713,0.09430825522437944,0.4725530024497823,0.1332388396392843,0.174407825776912],[-2.259364517182152,-0.8353126132449914,-2.3061937059063364,-1.4658251510308604,2.455866037544441,-0.12462876978237483,1.1979740930679077,0.07188572030358477,0.2144362539034714,-0.2758610202005097,0.012047198951929596,-0.12630581860137094,-0.21823057694810016,-0.2093548916869717,-0.2764175529054162,-0.358530820500492,-0.3164912040975871,0.03145002297435579,-0.058015148225772015,0.11730421105894544,-0.30974860601941495,0.06566337693620211,-0.11231347987757104,-0.32671704798845314,-0.45322397512905516,-0.016371794745544854,0.07461198208414789,0.41307574262709285,-0.25905998473501174,0.203085523738895,-0.09508988625369762,-0.1334252387466741,0.078472134442272,0.030424782224551437,-0.11150856311439226,0.010329301590792522,0.024029174775495772,-0.35919966971344774,0.2780415733200671,-0.07738904271005924,0.029556045234313284,-0.09111325727909031,-0.008809421739265919,-0.2279434086633482,-0.02041804490257954,-0.0434270990014572,0.20331828373730534,0.022118136887563162,0.04197948833824985],[17.63613208850065,-1.3261445966669003,-0.1779078881791398,2.0770419991592433,0.5953853897769423,1.2293536134125567,-0.6709546774618841,0.6327290689028537,-0.12011541590672209,0.07913512391207116,-0.20375787267735115,-0.2985781617808796,-0.1999698062188411,0.4702200664848634,0.053083914171696646,0.5358045392446718,-0.08962951065186602,0.2455364287921249,-0.22488705741017473,0.13553544069203888,0.22081577265539143,-0.10664890658121678,-0.24496728947443375,0.23918251559462345,-0.6942209291871886,-0.10377773604934387,-0.010836914004937236,-0.026586460953626476,-0.38097927569930823,-0.20567640184267275,-0.11720344389867274,0.1975904609375488,0.15010161742481268,0.17659122610515665,-0.09872129162661081,-0.03166399918608575,-0.14235583527576975,0.1814665280704345,-0.27388980432536353,-0.003186736092083185,-0.23502787594411872,-0.10676637114153904,-0.08203388614670448,-0.016363525073748428,-0.1969596422339872,0.007865042238551037,-0.27850866577502853,0.10078578286256641,0.09302038327030543],[-5.957007130367657,1.3023811183090357,4.475650623116508,1.000949228323369,-0.8072639673934567,0.19185358941649588,0.6996583297256914,-0.38197781560304706,-0.012313732790525464,-0.7361512412582176,0.06429049910097412,0.10511683842444225,-0.4469062763201807,0.0018510591601726258,0.07930389723970672,-0.31263932675883266,-0.48120643818289033,0.11671783239469276,-0.6961751508437061,-0.3115241197298077,-0.6140129950471725,0.020808321516944544,-0.5678499800923786,-0.4329085223147022,-0.30233554448492955,0.16734837114405435,-0.23884592354025036,0.11498962229347603,-0.19975627664928644,-0.2563601230389397,-0.06390972938924595,0.1519573471057322,-0.3333757628598191,-0.25562360835400133,-0.2065928183614017,0.10566363462085529,-0.15150092476420926,-0.07214299281191904,-0.069822120822484,-0.4852819825657242,0.1616912064251264,0.21398594617191494,-0.17384190465805036,-0.19588847491901643,0.3097542404284925,0.11060083522624738,0.018691018341864832,-0.28640436891359683,0.07579507318266558],[4.185432359376708,-2.2885508034613844,-1.3079904876492168,0.5072296894099579,2.840904971873223,-0.041871488074474966,0.5568547727600272,-0.3777417381823635,0.8598811534275864,-0.3659872475059129,0.37122268726860225,0.5944097016656517,0.5712845915702804,0.9218291563163581,0.6560829556681269,0.22203349731249616,0.012893783404590183,0.3885882195046946,-0.2327486785136944,-0.040689328564504815,-0.24335213927010724,0.018200129532582456,-0.006623619180853408,0.28681794263240207,-0.32141194747793567,0.10587695474485387,-0.5031029526249466,0.37345480994913793,-0.3630745142661907,-0.09374574547505386,-0.5564731928437064,0.014460811176153356,-0.3939844466908498,0.09936325523269116,-0.17360729061276428,-0.2012540089110069,0.0032259897354121165,0.13920712223319007,-0.3507820189996592,-0.017944397932046822,-0.21728322046452409,-0.09499442831746285,-0.18172568012597823,-0.05017759881327726,-0.121637845207317,-0.13202869528414773,-0.06683354836169937,-0.20983386029818168,-0.26420748750183093],[-17.991274315279387,1.0469121086572073,4.395255187252202,0.7468161082554238,0.6935184490646036,0.17907851674565373,0.19824718370292896,-0.11907295058141391,0.2905450885320327,-0.17282940349614245,-0.06399286572189333,-0.12020360809954582,-0.05678802037231532,-0.17385685797287306,-0.3528452145916435,0.15900042360633126,-0.16690010222190538,-0.20358866364034772,0.07910369846386367,-0.18552847418137505,0.052082687399887945,-0.0697338541353068,-0.10511154198657408,-0.14286156619212562,-0.15524347202895833,-0.07245925396584621,-0.19895272014381465,-0.18871693707191753,-0.1487734676459859,-0.10028765559054412,0.10900087018409117,-0.004090613119338513,-0.011325755194070749,0.14296941758326195,0.1664778357894122,0.11956986092511816,0.037918469360753916,0.0946301484249107,-0.11464637741167283,-0.060624440896422986,-0.09460467767430476,0.013097141715827666,-0.020165429812910612,-0.059306844126651576,0.002626978503965211,0.011460333573160572,-0.06580925688583009,-0.09413853630315894,-0.008083385412019348],[9.143779290524387,0.778831033056606,-6.049040892131304,-1.8261561169815839,1.179260339967263,0.04126123245657673,-0.015472759036592848,0.0033379853382603684,0.02787468660882008,-0.16121820529964181,-0.045223966130404955,-0.2834656974673144,-0.25975021122175235,0.05866658816368781,-0.4911374023646408,-0.5719962559935383,0.22095535668282604,0.39592610610530826,-0.19083904017212378,-0.4282495978476678,-0.0983102961381212,-0.43317708823732154,0.6022477176878112,0.05656809670457326,0.051801916519768454,0.04963385363983598,-0.14847390507961492,0.2777826519391383,-0.06206176972038721,-0.17009109847693926,-0.08254987253331718,0.09860621920647611,-0.040856909369715776,0.20455612211318056,-3.609972387063997e-05,0.10256603520881294,0.08583111493761754,0.1223077623268104,0.14881885889304594,-0.0629770982571694,0.3346385247424344,0.14614891504061994,-0.06623296213994827,-0.008785917597524173,-0.06960021432680812,-0.03601688950094854,0.043349737554775974,0.22085365003435375,0.17646433661300306],[-15.33187784375018,0.3120055068360342,4.392328321724602,-1.239512087187628,-0.295729514965723,-0.9536667457588994,-0.716221403253654,-0.21190139606035183,-1.4303943447177638,-0.5589407653934055,-0.8274757562830307,-0.872543843066138,-1.1759782109761945,-0.6071115215421748,-1.1426248171240876,-0.5292130686189458,-1.162016056980795,-1.0227844347968198,-1.4304734562270205,-0.7738558297128286,-1.3768916713771713,-0.6042095626447898,-1.0205086050431542,-0.32831870354013737,-1.0156202281691284,-1.1265216943661363,-1.0083348980063869,-0.7815120529416542,-0.6319174249416554,-0.5309886082234998,-0.674739563782056,-0.3908792629640499,-0.6110818034160556,-0.7189356949338978,-0.6052976160351344,-0.5115543195887797,-0.5617543619597962,-0.47673241050489146,-0.3602823381402024,-0.9370964489427109,-0.51453326095965,-0.36143318246103334,-0.2420626711870182,-0.3392128507812733,-0.5457394629825028,-0.5541783214149119,-0.13013858779069243,-0.4293381945281,0.07701977105005743],[-2.540300259768983,-2.426870021647954,-1.691037552567166,-1.8156480537315938,0.666623374573422,-0.15143614551744874,-1.381717929393414,-0.49179234481628753,-0.8697555211312922,-0.18340642137599336,0.007712594703119131,-0.36863586288775413,-0.40890225156674503,-0.13216401028817756,0.11189934932455388,-0.2639272802409302,-0.6075218067189986,-0.3163346567519069,-0.14265557954177097,-0.13421281069142357,0.2792083619270667,-0.03709094167786536,0.3328646331471165,0.11635280871792696,0.08759139913302325,0.07927850067351333,0.41012114938146765,0.07138899635788688,-0.12175150267597853,0.1286417020391122,-0.13892328645262061,0.18243138706038356,-0.013519720565160799,0.3286726387593091,0.18895176766811805,0.2334910329207944,-0.22809679749057155,0.30750791713150344,-0.15014170568686572,0.19587920217732863,-0.19827307999472063,0.20085243186577023,-0.2404156520156054,0.18522916901511116,-0.10256952580703595,0.151389930338825,-0.18580859025165075,0.1079657817651289,-0.17250339552230803],[7.301496343598894,1.9409326731641026,-0.17208178410609767,-0.41715919915252725,-1.5091564804723254,-0.47012181431268274,-0.8298570515427892,0.07579126436273557,-1.2427701899618717,0.49139995282048365,-0.24134045592976308,-0.32272988458653507,-0.5617352474626522,-0.13191910402013396,-0.09247066070838943,-0.19524474208152348,-0.1015906539936362,0.20906402244100727,0.23926543168998055,-0.14876802242180112,0.09214161863690794,-0.04639485043889515,-0.04281662280313581,0.5382214349801934,-0.09114474426755804,-0.17932304925696183,0.2002224487460723,-0.21301876632647007,0.1950344138784813,-0.1481081908707632,0.1535438468992579,0.025509303688229935,0.01514195177390579,-0.10176278569482008,-0.12518969025662693,-0.051522340120714986,0.14602633009964316,-0.1328469503170449,-0.21536074253260187,-0.1297690880667956,-0.1100649280314382,0.03965860331718897,-0.0032053729381851244,0.004327236774701268,0.126282210225811,-0.015082447132298341,0.04092229023002428,0.03966436849176487,0.061109824876794984],[9.480677753561503,0.9710658043833835,-3.3019961415607564,0.6069063305484327,0.8942809520453875,0.2642743664439479,0.4165610505017123,0.37159514251038034,-0.6924987456015805,0.3934768513483038,-0.17696373743870059,-0.14959201297521743,0.5111685134219013,0.1920670768922557,-0.013492633033147784,0.11700150207513886,0.14904300679989352,0.2162017858485284,-0.3413309356011318,0.3193634295855224,0.09639616984561752,0.3866628668193462,-0.1543644417468975,0.07984708449002272,-0.028464514761491196,-0.1750938857232179,-0.0905922549919917,0.1294865711547428,0.13975267947901615,-0.09791553628118324,-0.02934272184855826,0.053376270218814005,-0.007536660868126823,-0.002504908046896099,-0.18046688046944623,-0.20415595222999688,0.06101913255654667,0.010577844114819048,0.10670881095840422,0.11826277320012149,-0.1294699638939087,0.1832282744795818,0.03882770711007369,0.05633070522881491,0.005133392420590557,-0.16907998375127115,0.08067785913988411,-0.14452878903866,-0.028819947269953187],[-4.4758353187984,-1.3028403277733946,4.6665910939867254,3.087715625808243,0.7415747182329648,0.2045675156215136,-0.810787269196867,-0.5199150834393447,-0.18618504347916529,-0.48323787465743845,0.7232013531443297,0.14913549399857973,0.9027385125286473,0.1751492222361099,0.46637768243408223,0.5438162236640821,-0.6305692206958872,-0.009836052159827914,0.02707518913959188,-0.12811375399099756,-0.14743784228388326,-0.3818493089852602,0.15474295488611628,0.17042878009669354,-0.7065619067179844,-0.5406199239273846,0.08015835911456547,-0.0828127693643422,-0.2504400674653874,0.14411163274533906,0.008182235415392501,-0.02192333720157503,-0.20227347986126437,-0.18062526702643766,0.04853904992567567,-0.1537832578459162,-0.28785262467679656,0.07131641857413984,-0.2863455354453021,4.391441550522576e-06,-0.1805398459785193,0.1643432653951076,0.07366889435596695,0.08546308323259824,0.077260336823814,-0.20524778808074223,0.01574891453502096,-0.032654720465361796,-0.10913802536372717],[-36.89415563505177,7.102052123468641,12.215345720217107,2.2991085113646488,2.2479808720436245,0.7791198615141466,0.688021804283944,0.45561100012663913,1.2988971797382882,0.3733752084122668,1.2046126672274078,1.5210921865894187,0.5343137742367008,0.7013023945216041,1.045030970832295,0.39071721409913573,0.8438419720366628,0.44543524258931905,0.6500208713574515,0.4703922052862198,0.070088611260578,0.16943638649797357,0.04657204098507635,0.5773459469826246,0.08991676542890974,-0.45215420682475616,0.1569828936397831,0.011962347518521552,0.16397938732733552,0.3737740392600608,-0.0462928015071138,0.09475837908314595,0.2484355023994057,0.15802744713240943,0.15287972477098422,-0.08133223207426446,-0.07648897328925142,0.017838117871315808,-0.0161721679321566,-0.03628166294848379,0.25999705619618996,-0.017325076928671544,-0.08977181018736005,-0.040819183768614146,-0.011949890150902556,0.08059331821768365,0.16125288998678375,0.2685285949019153,0.14896969427759335],[-0.6655549020314699,-1.4494154636560135,-0.9764027792996613,0.6997641871522122,-0.039114427745456225,0.30410322695531633,-0.09367416083911276,-0.3700072904620358,0.28326036338689703,0.6705452667034179,0.0973857079399031,-0.15946867905661427,0.47494725615515593,-0.1666800972431865,-0.04431415377258435,0.44584568927816987,-0.09267594617097534,0.10306381790703362,0.08155701747644734,0.45264655229380124,0.013640935555786892,0.03679218442901492,-0.05740836660872455,0.043752649191866465,0.14617193227394532,-0.015072379245259268,-0.0379938242323372,-0.0021300946638609167,0.09160755077820454,0.22921839218635584,-0.04005889605799154,0.13906390404194408,0.05680564078713168,0.09752190973216032,-0.039621207964531474,0.10144309772046033,-0.0009226152220459041,0.04953863164733863,0.15242183958978606,0.11369466631288655,0.2220738175142015,0.23338136074371268,0.02119091249800153,-0.10944745038025237,0.046744062041154576,-0.07306841368739489,-0.22406186511753926,-0.07875630258091013,-0.15997084847825133],[23.120378598093744,0.9260101198446903,-5.048492310113865,-2.704460830411425,-0.8869364958798204,0.13658449594163152,-0.19314195618046168,0.43381445655641837,0.17523053371220712,0.026764978515780812,0.5503144160702963,0.1356783068843456,0.22946613047358905,-0.28454807466428045,-0.0877686030338088,0.04878678722675979,0.46036545905508164,0.10346220337362035,0.0848465679189426,0.19842293990957183,-0.07930888572664291,0.05871372362472322,-0.10655858142980172,-0.2661028305083524,0.035360983686863384,-0.08982808049908024,-0.16437890177635156,-0.32759541164206124,0.029084924686713146,0.3078473333287417,-0.18764112149603723,0.09811152789310625,0.15972631488426683,-0.18914271013478276,0.02506269371566229,-0.17077290461275568,0.08062696095070196,-0.14308735070295786,-0.012689040298781673,0.013478043654766446,-0.11580922902839719,-0.0004628522283686246,-0.17411124859540594,-0.0833856276576359,-0.1483023951730953,0.00022711367012132226,0.13109934688912162,-0.0735601320573505,-0.03178477641165928],[-3.132921086721346,0.9752667943716496,-0.1882115490244547,0.06446523570457494,-0.23635974340749524,0.4570478366597531,0.5728181903877805,-0.7957498874445876,1.2038881473631275,0.41092859308567403,0.1763760367428218,-0.06594204593478983,0.5095083507160169,-0.030437907573403856,0.4118837296618671,-0.8955033395270455,0.8006467960971312,-0.047084799548696335,1.3207840818141086,0.35229707021284395,0.6994059990002234,-0.19522884623620218,0.2904530656632011,0.047329967372859015,0.8643474092982222,-0.38173646856701166,0.9129257351670401,-0.45506002957695774,1.0231485242037965,-0.284396080953613,0.6446367268766026,-0.5201876732187818,0.8631012033035542,-0.3259177302294655,0.43754972851338214,-0.2170368822930501,0.4945346388240831,-0.12459547622631528,0.6318827562414686,-0.007603483024267967,0.16888776327873797,-0.16795183738686856,0.589255805373291,-0.18098258323128716,0.5069710908344659,-0.12275388348952855,0.2961539945205188,-0.0426059177544038,0.3173845278012992],[2.9066294125312155,-0.24198327134920783,-4.411766202993627,-2.295814315548256,-4.113119455120807,0.27251569480878635,-3.1012050093195134,0.9481918724027663,-3.8283163243561034,-0.16956983217524327,-2.8174500918768954,-0.14358032003755378,-2.4295991425284003,0.16546918949045575,-2.367858140343409,-0.40910769136695535,-2.123311047212924,0.3181387763017838,-2.0041053592072617,-0.1934723299133836,-1.8113739003088667,0.2076275529328979,-1.1476651511796303,0.027527077462406912,-1.473362894417356,-0.32172324518175754,-1.3101406483898848,-0.5123368280571307,-1.0585598255006479,-0.5809308660057452,-0.9786657659629072,-0.3392696397655024,-0.5535780194924452,-0.23941265986782578,-0.29374061723260236,-0.4337659549792455,-0.21112161664335422,-0.2765479288485852,-0.14674504912176975,-0.3045246428160638,-0.21531239485145576,-0.1787759605407457,0.07799291545854915,-0.20066791765893852,-0.052211168893999245,-0.11692754969862669,-0.07113415177235313,0.016627609841528505,0.07562283906497168],[-1.449602066599522,4.714158592154256,6.345838312025715,-0.49527076225441474,-0.5215422200100726,-0.8983451403186876,-1.0072027214253656,-0.7231228346436125,-1.0949707182914723,0.6081873491151796,-0.2032620753719088,-1.0051534564173108,-0.26553734998799133,0.7108619191966553,0.1868669109721987,-0.8199639600642833,-0.03167443306512509,0.08340097967034638,-0.9403295035104956,0.12798562938414673,0.3106241464920534,-0.4024433145175924,-0.20607359689814753,0.3640840703950598,-0.2877754199951888,-0.5373996732104441,-0.003080258404726517,-0.02026717179667319,-0.05920551638735126,-0.07404709379812711,0.11402996859227431,0.19164600797906517,-0.15976127902311643,-0.02156257324691122,-0.0018370531045651033,-0.2895811325846032,0.08869039285502786,-0.04568424839197837,0.21807857529448652,-0.24187327774049827,-0.09814185714034884,-0.09395063706311356,0.21376207072417344,-0.12803760603491468,-0.2106499813843095,0.1178759588508871,-0.21710667862188068,-0.28383537899692,0.18723478114156003],[-15.948850405282927,2.036452329472722,4.867486614104977,0.6348628373455749,0.1699937589777332,0.5556972575184754,-0.21031024527194694,1.2445739369387558,0.7156069384953807,0.23193287295152962,-0.5667776421098217,0.467415054350993,0.3940582992566209,0.14420214666307654,-0.2010074848262877,-0.3083388190872924,-0.08554253844638256,-0.08319281732304344,0.04444931548311949,0.24805893588653932,-0.05008355061659545,0.4132839961786002,0.29439463739148836,0.26869540469154046,0.08231195019888342,0.3106773988939092,0.08453792877457354,0.256484011840462,0.27100537376722716,0.054027923851380714,-0.09440674295471273,-0.12119974305324666,0.04915988892696773,-0.042847741267773,0.02850811582617747,0.10271192653355848,-0.18906249853103801,-0.09586320328669341,0.012377684243262712,-0.10228137645227545,0.06339104788098333,-0.16237596310061245,0.0886733601741592,-0.11945666831033737,0.012672776625441404,-0.08886982236257267,-0.021491817513152264,0.11577262743445532,-0.0030657092375548094],[6.003731732629878,1.5433228555990761,0.5492411303377305,0.14072112875557882,0.7128122892250105,0.28855337538178055,-1.3134084359097935,-0.21718423740419374,0.4137209339786463,0.14606335517067895,0.40591969119966737,0.2575296295379755,0.01519159871565388,0.3431869889418762,-0.2022357615935495,-0.227559351798194,0.22800460122507987,0.19540960582924805,0.20352345195494753,0.3799133389417901,0.8109165062703363,0.18717206014293802,0.5006444675689936,0.4220677143463243,0.08187319859993918,0.0476436720894285,0.19460348930171467,0.014419301570225095,0.21388554324972142,-0.054926803408173526,-0.3035833692812386,0.15933838977532164,0.2500948813953771,0.16707275659025256,-0.003099448704366606,-0.08093296473972242,0.21432646622928272,-0.1627296574313019,-0.042957632427143205,0.0057591312462787975,0.009037913150529753,0.196422131882604,-0.18474463597619145,-0.10865897047598504,-0.1785540382269065,-0.1123537239984462,-0.2756154409532273,0.0356429675676139,-0.12912271768652733],[-1.4593522862250143,-0.8544346360177828,-0.24346205783801922,-1.5333855854371403,1.2115511204978429,-0.5160808642123368,0.6459445629089663,-0.43144691333501417,-0.11068443849115467,-0.8725825429267967,0.26331700756283877,-0.27146097435902594,0.5309118736764659,-1.0560332758704865,0.1351907532784791,-0.4659050115084884,1.011689722272132,-0.2710294793397365,1.2736844521691326,-0.4782524040736434,1.2700277598936436,-0.25612064539856666,1.1214246068698284,-0.5491483125209453,1.1323797567150948,-0.2930526914100353,1.3992061336410313,-0.2979799665195302,1.3400088389233171,0.24124221802284684,1.532327340345323,0.34721739400029805,1.5713953505456446,0.10708233430515314,1.4334854178424032,0.20883314534948924,1.22593444260543,0.18344606069460015,1.0181776096728365,0.35985812430941083,0.6625436956956706,0.14769231466071325,0.6201281926499966,0.23215948616604418,0.4609322766781344,0.25491041098205447,0.20292556184326827,0.11665735866099917,0.15439110859362717],[4.736105780711627,-0.7671704092633177,-1.2427080245557076,-0.8649914118028431,-0.32021410220717716,-0.7151861823997986,-0.07007727736659343,-0.33718358105446666,0.09029024669463816,-0.14465306017525237,0.1022671997342418,-0.5947762257656246,-0.3313258729705915,-0.46085584317384143,0.10117455295663629,-0.1317932059265861,-0.1965582728246585,0.16203357646068883,0.1128873904514673,-0.0963236608011544,-0.1571070382951323,0.058016530926286905,-0.08585274440260472,0.31566196108806804,0.09620346812000513,0.046886525995603626,0.44749873936293594,-0.02822278975259648,0.267833729047659,0.05474606381049303,0.4266672214671273,-0.0054026882783047075,-0.026204223792665574,0.04442095229554369,0.2090961890101899,-0.15701973283882686,0.1580843286461744,0.01767201919079343,0.32728194970034996,0.000246879328949889,0.06784158261994053,-0.010873836145458879,0.22470756912584516,0.06322612268113188,0.004567993596592871,-0.010396839185737038,0.2515138138271887,-0.017322700264256717,0.10556172213450947],[0.13955738544694724,0.4140666273643232,0.6888656421280727,-0.30350646365480444,-1.8399505781515204,-1.4316266276863028,-1.4444846158827218,-1.182529546802684,-0.2617090443961868,-0.500835935933814,-0.4766894093015813,-0.6377123556774515,-0.4915078627943066,-0.17085260696662202,-0.4573014169558698,-0.7582604210968567,-0.6710570967846683,-0.5936147546370082,-0.3067335836091672,-0.2893257241738317,0.14110011861170654,-0.1933569486438997,-0.13586582656721577,-0.18158983041633986,-0.124862869536821,-0.2898025313752821,-0.00879486914461806,-0.18801998251848676,-0.13913566484097029,-0.26157431362177835,0.17016391882872453,-0.08425864491337724,-0.018749932071054604,-0.11464341448854044,0.06666825644514707,0.005263828352697382,-0.08219664866741788,-0.20790496896156568,-0.1165784135331839,0.024891605869319895,0.14629023593557022,0.16851896122468657,0.06936715302174186,0.17014413127952577,0.13439887021999006,0.07294640711138758,-0.015033328459985893,-0.11803527825011627,0.1711210570358867],[3.0721501411263965,-0.30751385308987506,-2.482866601203555,-1.2832142129480368,-2.177989455252037,1.252680276272669,-1.0115342800044418,-0.30024771686462687,-1.1714289277047183,0.739031477345365,-0.9564968798010909,-0.2835322160908141,-1.1817167290789656,-0.48104474316228524,-0.5704074393098622,0.01082640911709578,-0.6659176433930015,-0.20201067003981746,0.2883279870839297,0.04408643802986367,-0.30903695074308973,0.4757438564734085,0.02008160125037041,-0.40865986744815785,-0.022394659649891346,-0.08818341333602073,-0.09411482864364547,-0.391711953239371,0.2713162484751645,-0.09497400232966549,0.15150081397411272,-0.14986852043094823,0.0746088049244295,-0.2129840942505904,0.17279895685730168,-0.10859037064213355,0.17630176595987035,-0.1959259220974158,0.021914323764600143,-0.07733671194302837,0.06351995585425932,-0.3273343146461802,0.0658782788284579,-0.10014756872784256,0.10188040858624611,-0.09110814731342488,0.2370914339056432,-0.21821600281704098,0.16561695839050275],[-8.534886500134185,-2.97723551124395,3.8064139382288187,-2.047693952260606,3.5206371714728943,-0.9575866863766984,2.9376558165605204,-1.2099193122120129,1.7713742959467222,-0.8634300352899643,1.7071630483748683,-0.8042644565276583,1.7552759256260495,-0.6589319511281132,1.4497235030733393,-0.3983097319630887,0.9234703607635771,-0.006582763061582935,1.2401479743133608,0.010404941059414384,0.944164405740159,-0.16307393088215877,0.4518154068515597,-0.2532212642637525,0.6162641116545406,0.18948659615380908,0.5096189880624336,0.24143341707826402,0.2923897073663675,-0.05199537726552201,0.4982923039985226,-0.016413870936535086,0.06972198470166802,-0.14929805509193922,0.29709730740962975,0.06977981962590568,0.05696678192151112,0.16155974426159905,0.1849200827114771,-0.05515882032291769,0.08077279655993232,0.09468510347180253,-0.1009603475742912,0.022009294161039774,-0.007966414500487419,-0.04676730967996285,-0.03527409416214658,-0.07006273938410247,0.04971804729070006],[16.876722900699026,-1.808811533517408,-8.133531713540181,-2.133335277798344,-1.1058365944169235,-0.8106423599300318,-1.1838909130161452,-0.21941811121560764,-0.42424122610077897,-0.11302402105559956,-0.750714043168629,-0.6867743696713988,-0.712737256245526,-0.16536712270625936,-0.8676990960395329,0.17114084320122158,-0.40966860436094255,-0.3664429985834051,-0.34706384920543915,0.3188343932381845,-0.28733710170345506,0.03116999189049398,-0.5674330033093405,-0.03462574345577246,-0.3026325620098087,-0.10367670320608013,-0.4429685532472192,-0.06410342902124191,-0.08518586232200222,-0.22258500505952042,-0.2092561534992192,0.18641352104699538,-0.24123818631679428,0.0909971894035215,-0.3417891757030215,-0.02091398484534409,-0.16294389342902602,0.289640605026755,-0.3287050134747484,-0.07224026571761975,0.19328099350612435,0.13864554213994118,-0.03838708214575808,-0.008493493909885892,-0.16005756677309854,0.00039000736045250963,0.22509563137251593,0.08235039397331287,0.09981775433604674],[8.84635087802946,-0.8112634330179006,-1.0967898399966296,-1.3407272395005498,-0.27290587191692417,-0.12314002165530838,0.34660376889318206,-0.8286942844511576,0.1043986387118398,-0.17825068231812574,-0.09448999197821564,-0.14422708527977557,0.3714631572705232,-0.2396278642849294,0.048881129804594296,0.10731152497127519,0.32401919361475956,-0.002856134384467537,-0.07890619881921113,-0.5229019007048746,0.20625715974351308,0.18425453746023157,0.331574870850116,-0.06389935012973706,0.29209816933640215,0.015250366843260543,-0.07643975293594815,-0.15048758234624804,0.16129106087465322,-0.16234989775778258,0.15621758091556148,-0.2135095246627071,0.2327553205996955,-0.082644605412078,0.195744381405564,0.0013380783926070004,0.001550642506002849,0.03443438044892271,0.1822354072292787,0.00480980349286814,0.12177071141651108,0.13872519026398333,-0.003248077903798031,0.2373233260326366,0.2119647498024932,-0.010013087671114113,0.08448831394211694,-0.026564601830979733,0.0032630859201045123],[-10.632509825966203,1.6714085940005043,4.328857929900506,0.5964014401676925,2.3690216539367652,0.9734929778894361,0.3260628658613188,0.4086286112192365,0.5647668119855873,0.2602240590027919,0.543845086587624,0.2039228558209217,0.10869127090295407,0.048920471897407614,0.38609412569268403,0.22110013946542173,0.3451379243472417,0.6228949179364094,1.084931862628591,0.12313861092693665,0.31482955565361176,0.6070091400331259,0.019339176332481854,0.53889279864347,0.31356138066316797,0.2786612254462014,0.6368049082036461,0.7974039250423901,0.20849270991123395,0.08067423255274782,0.11276560893371747,0.29826378331014997,0.3413527188841223,-0.049534062396863233,0.34434636011262326,0.13151324804131295,-0.1520538097943392,0.1341907725386813,-0.013572829287624209,0.21652129455280023,0.2088126302919886,0.0721612041155865,0.17195971004268612,0.1931166630582911,0.26112234376990257,0.2797224451253399,0.05032905098527132,0.02556894528828529,-0.07696525641203701],[2.301745996316343,2.799253241242678,2.904059758576508,1.046148377294311,2.7506137223132203,1.6464383250041847,0.005319169313528214,0.6243274816043203,1.005161465194256,0.5464109990731885,0.7530503280265047,0.5585726110473318,0.41494157638783574,0.21937038907868342,0.25744684610410307,0.42122333912870275,-0.11208916984996023,0.46536863883904556,0.20218997763189564,-0.18463423947343527,0.31038369580348707,0.055511259439664215,0.09146793330975844,0.1638837945036665,-0.10788290549522028,-0.40687153308373397,-0.12134583552295178,0.039468777819285296,-0.06898543401536376,-0.14817925442671342,-0.06020480253796054,0.17257763726682304,-0.23376247293101768,-0.12548410774204552,0.06869551943074635,0.20353961348237143,-0.4939680564502529,0.05916292351971964,-0.31155012864039155,-0.24988534582154223,-0.0007183174856959983,-0.012050896258715435,0.021557222704056443,-0.24359648008174736,-0.16706761093605751,-0.15261060845464028,-0.2061547696266226,0.010617470682755068,-0.06629380301612252],[-0.3188111683150931,-2.358129766203932,0.5932405806325919,-0.04874010685298108,1.08853233664031,-0.11605311403397196,1.1297673743523837,-0.07683683098968161,0.749057549222792,-0.289497507135526,0.37574227767121837,-0.5868056770127927,0.7881000265220949,0.1958840692571251,0.3534545378814164,-0.4049765894805086,0.3914825981544697,-0.2199189045039624,0.49207534906394806,-0.513404009004122,0.30091699886374695,0.015178515250709337,0.05810266285309553,-0.28497794235850105,0.5228299867775814,-0.20416431257655593,0.2814585098313143,-0.2478753855911603,0.17813776224455333,-0.027754926325878605,0.20085719939037874,0.019737209427321527,0.08326292571600799,-0.105586074851,0.12086016915565471,-0.0062572573121356615,0.16215490069318714,-0.01191389791026572,0.04156677185970925,0.04060092908015855,0.2114812720898251,-0.11262679316785897,0.02245728847181597,-0.017992202548228697,0.01181156919321962,-0.08900986470423582,0.2717011618032876,0.007265950668602034,0.10268960675247951],[0.8074028423347219,2.362379364532108,1.135994482899664,-1.347120416499888,-1.0955801040416313,0.22488310695428623,0.25804754969544624,0.17937284808435344,1.3804751293049768,0.1487653634256926,0.6748962922178358,-0.12057889041086273,-0.12972797248515802,-0.16090742599682992,0.36087622005189124,-0.1766248955895575,0.05723894817238877,-0.04590937324524222,0.37300585102152045,-0.015051351552321894,0.11193586924555353,-0.06790597905775535,0.11871594845710011,0.1085071569822482,0.06484634774771365,-0.0826952637012171,0.30464839758772816,-0.0654149789825972,0.09517846147325511,-0.0042525891806660795,0.2551298250020933,0.08621475555646058,0.09851589989370947,0.07511851368621619,0.06277739917115753,-0.03040130490555777,0.2755590995752129,0.0027345327630807674,0.16294776716283216,0.07388863141039768,0.20586638250958406,0.004724954964152431,0.06449526396978343,0.03285484888751801,-0.04577505331653229,0.07352790060274382,0.1001410964891293,0.02981500015487849,0.005905290791533039],[-32.90021145089799,0.32814313898593905,9.778102909697147,0.570315136977843,-0.9774958033891806,0.331963718227137,-0.48161545492894153,0.1248724516625938,-0.6536208574300382,0.24684115406735077,-0.3121566433721988,0.13092233058343264,-0.42613623302901593,-0.12203038499691723,-0.058365666526338955,0.43646086390133365,0.13720674140016526,0.26163340345178626,0.34987158577991506,0.6146744113105274,0.04926321720740793,0.05304139246962965,0.08478972683295398,0.3817859692212051,0.2718735611283059,0.1845143896889068,0.14311344234961476,-0.01170669586141656,0.06102987142610742,-0.10369707395751064,0.46230429984382587,0.07868064176261018,0.2453142924109411,0.2156094018799484,0.2681664228980205,0.24947169646001766,0.3096291876110729,0.22828865150534494,0.196830200641802,-0.04017173442443756,0.1746171390877319,-0.18433536437685485,0.2544579741605907,0.013139785359566043,0.17260959048452817,0.06339654139719228,0.10186790709369942,-0.027225860170024478,0.1699786550761814],[-4.505725628703226,2.4091696810923606,-2.7487738884391963,1.8900731116359082,-0.2648929429800914,1.7363008726431624,-1.4878664756893265,1.5563031634729456,-1.2072602509133625,1.0444901080054105,-1.466277627984524,1.1943827416569526,-1.6259324587863013,1.209572901617468,-1.3987097584456663,0.9589332012480539,-1.4182294311302344,1.003158238884509,-1.5686412919039237,0.9699606931013511,-1.3425177280913292,0.7978665156641531,-1.3295729076639822,0.5469180026009476,-1.2548817292140413,0.668466057273312,-1.2367159147823028,0.6265876551382,-1.1298143419173354,0.6579412924002729,-1.0921904533323097,0.4789630905240839,-0.9840106376735827,0.40677817612996114,-1.0551900902661275,0.2351328002188544,-0.8540122094042034,0.2432474362139849,-0.7870837196877146,0.3051250528886338,-0.7677653476310047,0.13734195305596245,-0.6708217417377565,0.13525955143756138,-0.5918184149396798,0.15578810603757984,-0.4779345516748119,-0.02738580886164999,-0.44545397745360277],[9.09949322761513,0.11744603034998215,-15.357975410442924,1.4303997848458943,0.357078479115556,0.7792953882365724,0.6395220107562116,0.23770565172204727,0.16712607688137462,-0.014547486496916949,0.38375711052290645,-0.14415083173419457,-0.5054348356038466,-0.5894046508708047,-0.5238290259203532,-0.11372946210626943,-0.6335497456856669,0.06274663397199942,-0.20790854364378072,-0.08002334882001143,0.06113787290235587,-0.2943327356741341,-0.01146689991510467,-0.020404968674302717,0.14250461510086446,0.5029533350439805,0.06861646367765731,-0.00833091521092192,-0.17630677442231016,0.06215552015232255,-0.34217363296741415,0.1646303463042368,-0.1633316136255712,0.01612286240149232,-0.23203633315769123,-0.024867278264803474,-0.4342129717423418,-0.16209385767954426,0.07531681822644935,-0.0308707087652482,-0.23043144367799082,-0.06042153175710364,-0.12421311675961831,0.20421661011907077,-0.07068307349406708,0.03305517532560997,-0.12309521517046369,-0.05187734736046395,0.059744907256357706],[-7.395866433702967,-1.1614520523778327,7.661520385323322,1.1724987595014225,-1.6522435160667637,-1.6659788975161454,-0.02138167218186157,-0.2330676801592191,1.77335035119476,-0.05918983579998405,-0.9267615103746324,0.6938917799557207,0.8809912336652819,0.08920482131373027,0.06333900044930799,0.6819530660486048,-0.06889342946544817,-0.2113591290663564,-0.30733521303498157,-0.08853835320952609,0.09921897768260127,-0.16656562857698975,-0.34919363642852996,-0.09306444376982188,-0.04711154836097313,-0.36030208820113035,-0.09795943468153548,-0.18887820190221158,-0.21148348934969147,-0.14151145919029529,-0.25192963881542846,0.2772509638901307,0.07314290096977599,-0.018303664139304582,-0.21317831826632322,0.04968522486271834,0.1933491187602379,0.07967284920938923,0.15984510471109314,0.08305001867631412,-0.1357914465318336,0.05163320159424805,0.012027531596364107,-0.09013262052254167,0.1358553354037468,-0.03311472314826687,0.1250284223413399,-0.12214924008626918,0.15887865377218524],[32.94093369901236,-1.1623568541224134,-10.66971632109624,-1.815901516546662,1.8013731818414123,0.380868536555316,-0.8043475968822376,-0.2652380534502705,-0.7698606434523254,-1.1297692840564222,-0.24699271726617755,0.4319550239747934,0.8883537965652734,-0.4046984597961647,-0.1516207973028895,-0.007520984544578388,0.5157352047139875,-0.23758729705422296,-0.683733963735787,0.0023080836203762923,-0.44677161071521865,-0.22029937158941976,0.1878261603381746,-0.14336141852059323,-0.20995827326015068,0.33327866365297404,-0.037984665643794,0.12386704411400523,0.20026615837705047,0.12223511728970103,0.27900946256133413,-0.019808313228093732,-0.06208069110936246,-0.07549126217305673,-0.04591303886741254,0.03507114430719669,0.2707601772933253,-0.0036284746131359717,0.2066716928093649,0.0494154347196545,0.4475266445332227,-0.017756464064913706,0.013665833968421554,-0.03526780889067901,-0.1653152518246873,-0.06541436983838739,0.3455198456785123,0.010640291165903941,-0.10501533557452443],[12.767120224116196,-1.2013817390288126,-3.0949894881401177,2.513383130665102,-0.9005912493652009,1.3828979939379646,0.630895388920046,0.3533277885753068,-0.05830630771894532,0.5101341865887216,-0.4487933864764645,0.13416806450867722,-0.4086588636887956,0.5527457766465678,-0.266963347443833,0.1574017009904157,-0.0519857240558451,-0.01870604150149566,-0.5103335652968772,-0.3534621268471666,-0.4719675594852641,-0.12680827995893168,-0.5985374120964241,-0.1162360615320571,-0.5271573028043033,0.06446357610312828,-0.38589287041340237,0.12430438408553753,-0.3631392513228863,0.018820855094881045,-0.11659035535376894,-0.011055093973390112,-0.30379752936333626,0.027246477292288233,-0.3813563450139511,0.08014532430489256,-0.47038431874762376,-0.23495588746152818,-0.14847519475527773,-0.12043851023601201,-0.26386236194964974,0.3159192684911063,-0.05565685241106823,0.027297706594978752,0.15449417128882656,-0.22293693452034874,-0.1573973311168645,-0.07694318982154633,-0.011075163085541314],[-7.5934827258280135,-1.0575633005974996,2.464212086429042,2.886472955738068,-0.42767272716220617,0.2089647380152788,0.0690324009838111,-0.155355704930274,-0.2980662190062872,0.16340703894415237,0.8414769688885584,0.4348444131696309,-0.43841776316116904,0.41386168665138884,-0.123372242756652,0.5133191869759423,0.04153847116497928,-0.3120408933630569,-0.18190826586543007,0.061911518574676076,-0.02988295486167815,0.3510094899871356,-0.060024625626466106,0.2445184334929994,0.018629639144033757,0.28892667562856983,-0.11556348775422645,0.11195447418419836,-0.1403350610756587,-0.12348443470341981,-0.2428007687510955,-0.04024513685410587,0.06948177618986089,0.11030139575386137,-0.32136985759150377,0.07890459582929983,-0.17279129584523098,0.12106312024067023,-0.20098743225879437,0.031697966461165274,-0.09301267702733397,-0.0915154898693529,-0.16976713467178775,-0.030874882209464192,-0.2483593551048602,-0.11418832879479428,-0.05691490939277552,-0.022868317086638213,-0.21112249620787238],[-7.569240817896627,-0.43003389868153724,0.4342959260119307,0.27766338871150475,-2.4572309742497676,-0.21152270857840427,-0.2259828958068949,-0.8671996810839604,-0.2652128716738424,-0.515498522358831,-0.5974971397691214,-0.3312001085070876,-1.1132186148120047,-0.3581934020661824,-0.4629710093171726,-0.2137838946098721,-0.30739893200248597,-0.5993926463024547,-0.07925127591000827,0.1435434530205882,-0.20181316456974424,-0.17023158047662418,-0.07866494645299325,0.1043957052075015,0.11099252052974488,0.388254263818344,-0.09561210123879069,0.03537004430946796,0.006831437921529884,-0.1380161343593701,0.10962307361590613,0.1981962763367685,-0.07983757167575099,0.15212544765816277,-0.055975003128298895,-0.07072971372213298,-0.10428588029903861,-0.06216078792770184,0.14197561574811476,-0.1128040558406837,0.31420130300042864,-0.1042901552658483,0.0249693405100506,-0.03900908734891618,-0.13482834064324184,0.16964028004190024,0.10502559325856152,-0.023955821226618518,0.1431624497539236],[15.26691437665655,-0.6021367662052919,-3.0381864170668265,-1.1282822721814179,3.235208255251057,-0.5104167373452478,1.9273102620768094,-0.5430566870141763,2.098207817535252,-0.4919872731473868,2.002765345148496,-0.5552965742106831,1.9916170761991088,-0.3923705271520019,1.6122062362802554,-0.3311490595218819,1.6003025691849142,-0.4960768214509065,1.6972978731024853,-0.10410157972543059,1.3330902472801953,-0.3543047745749943,1.0205524112634237,-0.29799900842529076,1.2037840856713178,-0.1703059758929456,1.0291797685554473,-0.20649633813652693,1.1053420109851877,-0.13962644422527365,0.8661387468061034,-0.11867745044369851,0.8328313968066933,-0.24551951064322136,0.7912775782838705,-0.1173721380530345,0.7404605239107832,-0.31955393378694613,0.7280433816740636,-0.23933542187079712,0.6960569975953258,-0.1509141630399279,0.6645444094394292,-0.18034491849715026,0.650476064275582,-0.31550016550216936,0.6652461390713301,-0.21744186065061213,0.6173261095566167],[7.581414414355944,-0.5457180263812639,-3.8551155487810385,0.021710307961575526,0.0337201046201177,0.5291495740910513,-0.3486959818365291,-0.19767108927954263,-0.03735822848659917,-0.05538500965434942,-0.20044818476995957,-0.17445208140302976,0.09805836914836669,-0.09212692636735544,0.14395369130455293,0.11633281943324474,0.24045470047937947,-0.12056073913720922,0.04686265496653374,-0.18601199319997222,0.6214826837689776,-0.09581650559077991,0.514441141395106,-0.13707406448958373,0.33150061147999027,-0.124583633821111,0.3278084352544905,0.08614102409322454,0.6291280555928025,-0.1622735322417049,0.34510099370372027,-0.11549765201830588,0.38901519593977224,-0.1896614529579693,0.3833014765791069,-0.18968216937261723,0.4046251281740239,-0.10022492302785181,0.4624784741953266,0.03949126450864994,0.6000197754083814,-0.16939979328504903,0.3975678224679673,0.01840433938650867,0.34162519251514817,-0.0189560282723706,0.4036028362910456,-0.033080052410975044,0.34598262083797204],[0.9861368990113865,2.316252860260048,-1.0140414145295298,-2.216231839426105,-0.4224878335610008,-0.2322107235923924,0.19975923932893086,0.547953466058226,-0.14741573190144316,-0.004441467225411175,0.32976145487512876,-0.1507728225566222,-0.028609755730787268,-0.056580537102527716,0.10918912575615652,-0.06601789337409737,0.11648981940378018,-0.14845860728884097,0.1578159139096405,0.07864933203282272,0.42341766828539723,0.05411500555579512,0.13954923949697212,-0.24575762123812397,0.2530582713037077,-0.15085705528966603,0.1965797369878125,-0.10531751551203405,0.10388056216949489,-0.16033453156340813,0.13522738121002015,-0.0918026412985542,0.2445027959031923,-0.12994455631223129,0.19047770243618403,-0.04948311885904376,0.22676124827789562,0.006489092291110904,0.15338892018964956,-0.07229811081367712,0.1737091167897952,0.03489808171380897,0.1208914595349265,0.007369062219621721,0.1445770485657771,0.024261257943052804,0.1970314954835109,0.003450869612558063,0.13877093598310747],[-16.12646415117785,3.07116057592372,2.0450153818819055,-0.6243586547838861,-0.3400762200239442,-0.06918854022399659,0.38919979693826146,0.3429594889837813,0.030061596213897278,0.582505891929979,0.21069452853640205,0.3068436791837515,0.2821632078865952,-0.06642934494006168,-0.3847762673944417,-0.020066627688494057,-0.2943310229405197,0.07982046354180794,-0.10551380995175788,-0.1991422871428466,0.028762709162372745,0.06306788424512219,-0.11305041867649353,0.10094755199586117,-0.2822224253965413,0.052067359086350726,-0.0614031060611282,0.13547168700660708,-0.17046435427957,-0.18347065441923752,-0.07320905705345608,-0.16886414589375115,-0.026195023551486608,-0.20715744931921923,-0.12005091701468366,-0.05833917973445584,-0.09369404885947141,0.15378123578064878,-0.21606462621621556,-0.02714558878337359,-0.0233284750431863,0.16525900949075956,0.003337470792014294,0.011681031117408007,-0.06633678808505418,0.04430819434868977,-0.1293303300701052,0.03863388293019302,-0.18261468265749695],[5.71848341426218,-0.16576747526523825,2.3498495336182694,-0.8851363528660242,2.8095530230676817,-1.0214749341361165,2.1718635550847454,-1.560712511543368,1.7688064254192926,-0.43493960582389435,1.2713393856732464,-0.37353291406897515,1.4453149093607098,-0.08791406680167933,1.7924727834766723,-1.165516086370567,0.8745963937287364,-0.8792982345446141,0.3047299142768474,-0.2569540102833151,0.41981872995011776,-0.2788208743092999,0.47801778167789943,-0.08573333726125389,0.42636102054478814,0.1463144967267504,0.2705594794054742,0.012603363463224148,0.5414453781842569,-0.00413082201157652,0.07755524517730837,0.1159985984122233,0.3095627748430154,0.10286241177984762,0.12556061824074866,-0.2494730560980622,0.10252987949444083,0.19257960789342066,0.05749346884443759,-0.051449109182907134,0.43612558242956617,-0.14026395394958346,0.1796393288799701,0.03882999601464805,-0.0301410831480057,0.03767414832218913,-0.0983994637480372,0.10347505246606682,0.12074164097463475],[1.6139990287078663,-0.9552261098031045,-2.371092119404218,-0.6967369790237885,-0.614226512454074,-0.4409145562205887,-3.3150418000750546,0.7539173908142333,-1.1424702397592204,0.7961699179150722,-0.876039484559546,-0.0331160936837838,-1.0440330334913261,0.5211986168124612,-0.9677788523494328,-0.08303829314874298,-0.5086373987404575,0.09855833383733593,-0.6074020987566884,0.010697572426008153,-0.30170182034214205,0.2118587217379993,-0.18285144866719105,0.16971945807091798,-0.14917590899675884,0.21015232215796284,-0.09311060509736371,0.1850980011655218,0.10737467329672727,0.021779010146708255,0.27117712109331676,0.012354196937945486,0.5217103154049396,-0.09975470127518805,0.02481635303922219,0.13322424254777007,0.32516807262835207,-0.12527813863959295,0.31027430682906504,-0.14196096283377407,0.2026356952030245,-0.1946260715691115,0.15075668230310071,-0.03282220502972485,0.3435063994040874,-0.159194597916127,0.4079335189662281,-0.17110540314696135,0.31674711404368927],[3.662891043684435,3.529741731603344,-8.248877408384542,-2.7877915571676946,-0.9439275580756827,-0.4789370419876604,1.102783025857255,0.20870277185818747,0.3464534942970852,0.10286542691438785,1.547858344885555,0.9941533228065411,0.9453822288005259,0.33986001105392116,0.8556676644386508,0.08624589647012604,-0.2206921305202747,-0.29698591620969655,0.23217790758921183,-0.0865811219348583,0.1260006279818178,0.4709512501324561,0.10300440031232125,0.12452426462967685,0.2664192936016262,-0.07704757475288199,0.15807654703374027,0.12512830791584012,-0.014761750369830521,-0.020615571594259457,0.3546916009486426,-0.03345323550747021,0.024473828488821636,0.05451408368218242,0.21603153853627882,0.13793158507223882,0.0721027529028072,-0.09211929453492435,0.008595869493354228,-0.21039601109012054,-0.10862430978436147,-0.03085147575616906,-0.1382148131014961,0.027268734510687058,0.08511436743849644,-0.0732889397736773,0.09923436768544883,-0.09046522307820899,-0.0947651367565621],[-4.844917958969885,0.3483104336321395,-0.03683256479051833,1.4987460130615335,0.04475533796559499,-0.4721792173749262,-0.046665594745969624,-0.42895020320221144,0.24512714699261026,-0.13949741218714576,0.3483310229101635,-0.6005259269446258,-0.06568934268105281,-0.9075353359708952,0.007857377133905313,0.013409203990507025,0.4133424932735851,-0.11855732871865904,0.32844060825126054,-0.12852431409349258,0.23202234935581859,-0.21738582268680884,-0.13680034038934266,0.160343697180633,0.3166170138137027,-0.2710135951243676,0.33056026934881283,0.013102050553207336,0.4661810911249362,-0.1706063952244376,0.13890935719313915,-0.20689839109227223,0.34145908660927016,0.05430052539369424,0.3258880919021974,0.047499167211703996,0.04942867046780584,0.03472637246350086,0.30780802513562733,0.06551944124883897,0.04180331665029552,-0.01836545609530998,0.04059034346683358,0.114641171912832,-0.14044861599016348,-0.008666588631422143,0.23764665213155514,-0.20204876843841935,0.23760590166308634],[20.53076299916145,-5.532413580208496,-5.838963369815104,0.9581023382526764,-0.34834770052149244,-0.16650270501470385,-0.19560323367605845,-0.32892185449137074,0.5531017039884192,-0.37391150844034354,0.6422694232860178,0.4672600551129262,1.5514189483559946,0.8577920920173259,1.002744350733296,0.30490071622093934,1.400130512869357,0.402094631902853,0.6926969788803198,0.11558777871707589,1.0688497230359868,0.21895041019565,0.5054057333770091,0.1742756780987076,0.9135945761941816,0.2530231262858572,0.6024422947368162,0.18719791973123867,0.31461727162479514,0.6543964130085281,0.6596482332583619,0.23615502646598943,0.3743739896989146,0.1453931098790829,0.5859330837849263,0.20222961696955444,0.29204612260773394,0.06846897839624443,0.4810373975644704,-0.0521911991282153,0.08647462852819653,0.24176495326489225,0.18402771667029766,0.24409998411035908,0.4062734409486501,-0.0785933606691424,0.11200728538861583,0.326771964852202,-0.04069602583893651],[-3.7219690028223327,-2.318558698329874,-2.598111281782864,-2.035602237990708,-0.5023305732098692,-1.598675156422326,-1.054704023739951,-0.7153540255422915,-1.4046929560880217,-1.3306718739859715,-1.5461513945286018,-0.7027553958597724,-1.1707186803883178,-0.4829072014271905,-0.8439853612796018,-1.4370988804543436,-0.592135581416736,-0.21631819336386648,-0.7473176758265073,-0.5605859045084349,-0.5200102699323128,-0.14531283568751358,-0.6666073760649328,-0.540483100318917,-0.8676736297192592,-0.47528070425181296,-0.4891113420175437,-0.32298387297605075,-0.7391923249336272,-0.268714855850105,-0.7669506773636428,-0.2718555768994731,-0.5335620227265995,-0.15418788743891676,-0.1881656383074062,-0.2926561495624464,-0.5241728759963614,-0.08347713529248248,-0.3621755139628657,-0.15336144747465047,-0.3035016693558696,-0.38331829536666506,-0.22384718244993973,-0.15099980399776503,-0.01296468343182463,-0.2663196239512134,0.015810330614698977,-0.07434285638764226,-0.15769169044366804],[13.78606558381691,-0.5286994946113975,-8.307463660882869,1.2719414775936284,-2.7571018734605417,1.338498926256747,-2.03600056137281,0.21030739781279104,-1.6507858922541896,0.36440305424474334,-1.7912139786435985,0.01256643038905676,-0.7185569446830953,-0.3526045754437575,-0.9243851192938576,-0.06440091546321165,-0.7858351441265737,0.358490418508249,-0.6500016816563583,0.20298322443281352,-0.3947829069636753,-0.1233678258847378,-0.15471067330391705,-0.24211177742172885,-0.3138661329452577,0.05287651619951777,-0.48347485864196776,-0.060740820146769244,-0.2454799482694804,-0.045075869039584084,-0.05009907547235806,-0.10178651114712883,-0.06577975557268832,-0.08784049110239288,-0.1346860046340047,0.011050019453603517,0.10115203985294244,-0.1391901474771293,-0.002993700067911867,-0.1681326137414957,-0.19173862631820277,0.009673849724924934,-0.09318214495250679,0.0724186718104289,0.04153419623217222,-0.04694379389115815,-0.055555637886173265,-0.02189466443874973,-0.16946598446585842],[-3.7723706164342516,-0.17562179210211787,3.812689365753732,-0.5271952752886623,-0.3401227392985606,0.14911330348879578,-0.38439523856226376,-0.15382599592912877,-0.4118432719921488,-0.4096064064784249,-0.09834317471937735,-0.4605173563516386,-0.5665472444428649,-0.4884632884512711,-0.015354232915040709,-0.30479583595685467,-0.18313136835629615,0.45674827283209446,0.5314313763733423,-0.14488340291080518,-0.6128459555302423,-0.10666144954077557,0.2639126742191453,-0.2864462216826976,0.08749229374431532,-0.12938878663153197,0.25648508004363024,0.19399262153571795,0.2007016469059459,0.03902926746816491,0.3453432229898066,0.1501957473561475,0.2155487054938608,-0.09378219225505796,-0.11597305524678897,0.06413575555494225,0.13385831472249202,-0.052318389960304974,0.12187078817051769,-0.1195803054514667,0.10921177034825638,-0.09545799469541642,0.24474660820428434,0.19546402250867745,0.016696042110656815,-0.019528038767596602,-0.02196957508672149,-0.11741136706893604,-0.1360330505986421],[-27.611249645283735,1.4029831206139398,9.964936455290061,0.39941676104736756,0.7302946756261873,0.5981486546717867,0.2991316518322808,0.2519526493429638,0.11343214115784175,0.08787660927812031,-0.08886702844929532,0.1053606584090137,-0.12915168423232623,0.07160978579947237,-0.6127393750864876,-0.13219644524502766,-0.15379082700547633,0.2401620556654853,-0.40062572349489134,-0.04452227958936127,-0.2922577569445144,-0.28112343473230306,-0.03950317424549488,0.19208281727037174,-0.34323950339442116,-0.05743508224661179,-0.06195238095958386,-0.05708208691641127,-0.42548053646168044,-0.14686567076993046,-0.051236678922409976,0.025438266327634948,-0.5587540846762697,-0.026974897502339328,-0.3334014881763979,-0.012954488464599127,-0.0721097804049791,-0.05816052440393205,-0.2553377239522438,-0.22365232823421022,-0.2972311654051559,-0.01774189806585785,-0.2177131279415639,0.12187851119697814,-0.20032310207287524,-0.05313939291539142,-0.034969757230584674,0.03547382095439521,-0.21644629480788985],[1.658585709650014,2.868893084941966,4.248430882593836,0.7550094955137505,1.1245966175046302,0.7433268275563222,1.9187000759554798,0.7204658951454933,1.462110754354332,0.5434324327482031,1.0376914435314213,-0.14300022719170988,1.412588338879931,0.20636908952981492,1.3383712353164985,0.30362608564699717,1.0423892902010183,0.22975931817578163,0.4410292322469277,0.07896236418137033,0.8306428748887235,0.0391306561666417,0.683243270398778,0.06933321608418927,0.47022735549649564,0.2651483184180422,0.7851541854847311,0.2349743521074808,0.4311311919414191,0.14221834798768043,0.07260842881451007,-0.08468144314289598,0.27352494929061133,-0.21380560527598336,0.26967415433935865,0.16374949526395152,0.34179637711122174,-0.12985443977420863,0.19826431409021894,0.23051615068116976,0.19418370495959147,0.21682068647093392,-0.0551044000402402,0.16460853788229707,0.058186663910358635,-0.08064751047911534,0.08569804698568956,-0.1087202542555483,-0.24588650868879755],[3.7290687489599863,0.8045775750999495,1.4671807457975734,-0.7978294780032892,-0.7947458085251604,-0.3748728912056913,0.8287636857462508,0.23955898679781612,0.10549078487565108,-0.17549237667338685,0.4798389716207812,-0.09316644758784409,0.1680368539667503,0.17061263171913346,-0.08492542147595189,0.4113959146090539,0.3443307746840456,-0.021554129793398415,0.08200278660056476,-0.3788426929985666,0.34284489554673936,0.27715915760694837,-0.038622295087432916,-0.3594925240268966,0.10721160021860665,-0.12937191939356193,0.2476383527613935,-0.3126719283477195,0.03334636216014828,-0.046266728889307655,0.3014074892372117,0.07914661598469738,-0.03594205314871033,-0.02382471126159025,-0.05420327299670112,0.0879267630184967,0.08752062590811302,0.08130053922499413,0.15062647712080177,0.02122037097723158,-0.04518228867722165,0.12067359480748546,0.0685839601629909,0.14453812070035268,0.20062099967889063,-0.018794232733413407,0.06478477165770587,0.04402923704514197,-0.08059105564632234],[-4.675537999775648,0.0909507042948475,0.8943559452481327,-0.2536483493811248,0.025729543280210553,0.18281016339536196,-0.0706898724869209,0.03813111323805594,-0.0766436648440769,-0.047408149893058346,0.44774642617129895,0.19506939504016826,0.09615042702671285,-0.30897131488919516,0.3421657429948462,0.13081618668142186,-0.1481712659152102,-0.331359545654862,0.03683849899070223,0.2003186930856636,-0.05280711392134939,0.1852105724146823,0.010911807257175415,0.09842529499367038,0.05665589857597542,0.11068917577066853,-0.13547154095128122,0.06492439419797309,0.012227705961281454,0.08941236323375386,-0.009036171039339319,-0.07851940585635557,-0.14603340958681213,0.0026398059250863515,0.03227118708934196,-0.019391930438764318,-0.32994462625911697,-0.033623966222334406,-0.0537674143858735,0.0794939787177157,-0.027557261551931996,0.0622263162930378,0.06255898737641984,-0.015427872507220709,-0.06370890312824447,0.09785786584301095,0.03911118104757721,0.0027752613924308255,-0.06788232124089116],[1.8063066128886711,-0.1951657004430188,0.40122064503554133,-0.5195984116243507,-2.437095765466582,-0.5758510225365577,-1.948232027209806,-0.36480547248607764,-2.1226610126007985,-0.5195791486747049,-1.1822241579593376,-0.5932643176005435,-1.0870674215516145,-0.5225897960970444,-1.5531183974347402,-0.5985472014413745,-1.2991898660327041,-0.44649707484458856,-0.8535486451891691,-0.44417850280852317,-0.6731948992469992,-0.24996308323723068,-0.8835775287246765,-0.1494862653527442,-0.6993650051769706,-0.1600473194116665,-0.6372892000177608,-0.24208360895600975,-0.6366808309795604,-0.07405806552191885,-0.4125378815005615,-0.15725728794256608,-0.4889268438804753,-0.07226082902437593,-0.6887995532854763,-0.09227083497338534,-0.5127178102242691,-0.03140420757631374,-0.5631125028865392,-0.08179793953337472,-0.49492841263152576,-0.021172378957724173,-0.5875284504808491,-0.04176205158121285,-0.5192605022132503,-0.08965366350735865,-0.6248100656584054,-0.13140972254618974,-0.6706264356722063],[6.437667749536935,-0.6000885240914254,-2.0503743714368596,-1.9245034265064989,2.107619995739412,-0.815449516389564,2.0246280075544187,-1.3455851253437003,1.3009698297750745,-1.4169601728559647,1.9891638666235278,-1.181382072162836,2.263242585418067,-0.6857132721500998,2.062917796538945,-1.091577888471816,2.3830914095566027,-0.8213006031660401,2.249636001422406,-0.9167328358122975,2.1831619474163055,-0.6591359789401275,1.788682570945951,-0.6119270145436463,2.2978134150995175,-0.3384059117016859,1.7510845631475076,-0.49794519612548493,1.8683761067601723,-0.43607668296148566,1.9753527924136032,-0.32359966191712836,1.5437882269410175,-0.3724337490228657,1.3141147994442903,0.1431831877025227,1.1049181303371651,-0.03805728236621009,1.2031946549299535,-0.24168162198398452,0.8127936718745787,0.08185787170273495,0.6479050282557731,-0.07942486982691109,0.5389748154572744,0.15478902827850435,0.326342931885232,0.24185835132777228,-0.10562204383139472],[3.5632915027239984,-4.594445362571269,-3.7643341607837457,3.5170340264232225,0.846971885285426,-0.4061739387975937,-1.4445224473021032,-1.3763338303709318,-1.075090340663207,-1.2408614763242265,-0.25704730206934323,-0.34187947270193725,-1.0765275189228762,0.23568154369580865,-0.6476509737222389,-0.02467786656057636,0.2459994984675622,-0.3204137867506014,-0.05240091088856338,-0.3364499325318942,-0.3751551828583668,-0.06695757418635438,-0.43998704511567893,-0.25651678054142796,-0.3602985591616161,-0.042530356404819225,0.22605704667769705,0.019522075272139578,-0.18953103859231107,0.121682274838785,0.11289900254978455,-0.044282544519191606,-0.2934597007819307,-0.11333065721172977,-0.23959101080061398,0.0007465071828451923,-0.08100934325447556,0.19625812733476197,0.2280905662312381,-0.012801072910667332,-0.08978090392499015,-0.20599230011675604,0.1298686828471214,0.0802622826604418,-0.02803479596742443,-0.11876308965923181,0.1228167163519348,0.09876386770018898,-0.14652757852514087],[-7.138067015049743,-0.34067886081662657,-0.37236351353267777,0.7547319805434204,-0.5292672333291238,0.3225952444058327,-0.01214199131684026,0.160628312041544,0.20759090127158936,0.47157095292169265,-0.06156579006296167,-0.3585860532044567,-0.18370767335373825,0.36165247258709743,0.30424425339857475,0.11248264191535334,-0.3686904502181064,0.3611381796336744,0.4302358559484211,0.04167884614175701,-0.262289878061412,0.28502880701226874,0.3901826748077966,0.45635812851356433,-0.4434939077462675,-0.26444484629496423,0.014777608554927026,0.2831044409822882,-0.18089338020743176,0.08767765905891564,-0.05886452949745491,-0.0784362523775309,-0.21629786621631716,0.13496684271236856,-0.28416568916430407,-0.04879049406813747,-0.1258703096293588,0.0358923563307946,-0.08342104777165076,-0.14343634761091772,-0.006698092669341647,0.02121543286478454,0.1044053445354556,0.06882987637419617,-0.28510198011257015,0.11836979925618518,-0.2998252472958308,0.07282410136959652,-0.007724822194093461],[11.855556483000562,-0.5776974118946926,-4.842795745641861,-0.5544114852319069,-1.2294864237270142,-1.4523023406449251,-0.23703952395912825,-0.15916633585902173,-0.06701199175687292,-0.08772349391846906,0.07894920596003113,-0.07726792354763933,-0.07800052122524495,-0.367499353676309,-0.24204869599129333,-0.3062096694002261,-0.35286918722621635,-0.4698255018811049,-0.20893149188018703,-0.1764021452260429,-0.5307289482352968,-0.3431930616211586,-0.26660546275330554,0.02322671161783692,-0.4631853222857319,0.4704076153441752,-0.45370723348951914,-0.039396153081689145,-0.5979119446111539,-0.3136647748124148,-0.2659301295001511,0.11865538002962084,-0.14008416578952018,0.3385670689149433,-0.29635129730108856,0.09616938219089702,0.004760081840464296,0.22024755302262092,-0.19051301199120269,-0.08313453528343946,-0.3764661303370229,-0.20278867892142957,-0.25417979140457947,-0.22872956950063406,-0.275453425262188,-0.07044848133827283,-0.3086408896436443,0.04197650428385468,-0.08449214004708638],[5.882388305761384,0.4178145901939725,-0.24746598088555646,-1.6282396160395016,0.31451328922027705,1.0706080111715781,-0.5844006815035261,0.3019876699784728,-0.008474389796777787,0.12028952770205796,0.5619664124114334,-0.5453682228252225,0.5087047558940252,0.12012319229030627,-0.14618187594276966,-0.15338160544999163,-0.06685703955262486,-0.07598182278100096,0.4568718718411994,0.030814603704593234,0.33049729622002827,-0.41996614316872366,0.3360721778799018,-0.11636823008700775,0.204598501987916,0.14783548253751772,0.24931480635278774,-0.15706428328502797,0.21541590073352554,0.05696782055279757,0.2711467055373315,0.02133959028091941,0.1806339545214786,-0.09012681103295125,0.361670379714997,0.04180793600600921,-0.010803999925696441,0.19861565521458854,0.10027365785908515,0.21257224377772252,0.08334869698510347,0.014190820612747108,0.08534240336702703,0.0964506797001121,0.07168908741617737,-0.11948482387161927,0.048164664759556775,0.1888082469148213,0.016653797206443856],[-19.563172442461884,1.5637145817251272,8.214767054114475,-0.8095150694543751,-0.603398276909024,0.05993804331918968,0.4760046175770913,0.13098827621780101,0.9341619850296483,-0.7514426568145607,1.0735692024095096,0.005261373814608553,0.08105856145959216,0.11255289886414316,0.5526399270325061,0.3797116702455635,0.5178779501495536,-0.12634800477468547,-0.11612430813734248,0.17560113996442403,0.2843796359229192,-0.19173658521238499,0.31310084437205793,-0.5911755585215828,0.13594841630999197,0.10999935398869068,0.04709649841267214,-0.14296484054152836,-0.040348322234696046,0.0004236727420965818,-0.03364708607851901,0.06428416864962307,-0.04242818276986534,-0.07111867558240137,-0.03691714767407311,0.17668806540453244,-0.062457514711207744,-0.027472437245081756,0.18876718667564668,-0.08736023725097683,-0.12781716837117785,-0.19121501919023123,-0.1571366992554819,-0.09310499542334823,-0.14205850199080117,0.05305183043966727,-0.11563656630713048,-0.03578931599490514,-0.039613945282535604],[-8.407754978758641,2.0319201522238353,1.9904448780036164,0.16635282851096234,0.4152011610615819,1.133405425491511,-0.8277509877656715,-0.3060576428288367,0.4799627475965691,0.19533336215233232,0.36833477902585027,0.5289940748651711,0.0734441858712864,0.5831021829506027,0.08062161913491062,-0.1464333033054847,-0.22783059977350079,0.3192759841078046,0.03815752027410721,0.13740449991673606,0.5069216525881073,0.38734151350328855,-0.32386782701405414,0.1983723587026342,-0.018259398016925145,-0.057115084350953756,0.27306874415698,0.1112416208345407,-0.008127633805103491,-0.03427021267614295,-0.3158899595248768,0.0010104263101217918,-0.1543790847908872,0.16579335202656761,-0.18665009308338276,0.08351056038766826,-0.1540313078556963,0.10683458213628505,-0.14544525464129568,-0.20021394276353546,-0.3156714291964528,-0.04527419966430078,-0.0453526712547063,-0.08664761327144571,-0.10197139979114087,-0.2249109902232793,-0.008905442461603548,0.06600865110954374,-0.01293319631881049],[28.394884350682567,1.2564721094402749,-10.589864453564324,0.9692137562936385,-2.7898460897771216,-0.7261675615005463,-2.184629558340947,0.45327269480415516,-1.3728277033786578,-0.147550362420447,-0.11932059037478934,0.08739766524470799,-0.671751484070113,0.4013216935607482,-0.47861498053411977,0.27943926469626734,0.1105842729182648,-0.2619122994294002,-0.40039910800335565,0.028037335185689148,-0.2869908567655294,0.2100965008325562,0.29567280220547276,-0.17358595727906334,0.17664698490085695,-0.0586519918827746,-0.20776007111535558,0.4451042564822388,-0.06325702382229677,-0.28108604872400644,0.18953571454127727,-0.1646309894648453,0.039490779928910216,0.09183184681418706,0.11926811617555547,0.17355084013816213,0.024910612134321903,0.06104026730297792,-0.037654480646277994,0.30304198954606965,-0.029081803034870225,0.07907219486887418,-0.029276896242191236,-0.2959662346668696,0.096051482343047,-0.07507318369316916,0.02629909517093246,0.15458665082325893,-0.02853046641258593],[-9.366906496540741,2.792835025454782,2.329308089403308,1.9739075136436866,0.7435032781212793,0.9515401487368627,1.4614778989849428,0.07691415330887627,0.6809247828793015,1.2797878726680936,-0.1336970578912118,0.7581110099876808,0.33805290074699973,0.9020514468501835,-0.2434132158035097,-0.05923174790583233,-0.11223950882688169,-0.04178670940618113,-0.18403978506142488,0.3230035045747067,-0.49740595576400404,0.7984198755539803,-0.5013024565806222,0.5546644536943834,-0.343783133604926,0.5126682633876583,-0.7314144247354664,0.397086885123244,-0.23878266286168054,0.45156430091630057,-0.6075636288930444,0.6093739808656473,-0.7005841811600829,0.018733184724440414,-0.6306268533667198,0.09404661089521069,-0.7218070991291341,-0.1874417247490229,-0.5557516576184456,0.06480956713999819,-0.4576548643865924,-0.15618843630337903,-0.282827629505608,0.01610316631601266,-0.3741708531347124,0.28738250614232846,-0.20127558646257393,-0.044127943539943586,0.0013536562188386449],[2.3301204794389356,0.5149916295189019,-3.0979599119209524,1.8316201393874294,0.9016520721801435,0.7035325813561099,-0.11739972837890973,-0.13677318283387735,-0.636274106165588,0.18825981485076226,-0.4939102709614816,0.2042235659346821,-0.4448935574969053,0.505087693785626,-0.3045953204852044,-0.1778784657435584,-0.19102390954053003,0.18094164433248905,0.0874559898430499,0.09000893046551137,-0.13028722152439132,0.10457728752929339,-0.03718376207128671,0.47146850151678943,-0.5075586559109604,0.06796610954979997,-0.1959594859573479,0.09027308803343173,-0.3776693491970001,0.26193195028636196,-0.4266078070738697,-0.07893775156239745,-0.3683556795648829,0.08428682077872604,-0.2886820051234324,0.22452980451406204,-0.3226091287753691,0.11558443072174439,-0.1297980970241451,0.2390869830635784,-0.5389301516325191,0.04210472721168559,-0.2170303152878619,0.17152413617395249,-0.2079274668360463,0.04170823227045031,-0.2938222185762062,0.015124581493502905,-0.4008603386272951],[-19.785530775444897,-2.270484555834289,10.902897389633788,0.9805584394666055,-2.621595762121172,-1.0121687783431257,-0.1622155317106147,-0.568551704670597,-0.329567533784818,0.4964319240598942,0.05902008015218771,0.30271289953183794,-0.05281892624042367,0.6811789065718463,0.4797460439726846,-0.15333175152602227,0.0853244872100337,-0.4396376474719695,-0.17596009355862993,0.09944843769009079,0.12147375557598888,0.3477177767412168,-0.1002052710989566,0.060385088772198864,0.19667590923178582,0.024929192611139694,-0.33187806948498655,-0.16265191920092584,0.11502698230166031,-0.16955033844041947,0.05231867015021231,-0.09008391198038221,0.07352678033679547,0.03930226962289399,0.07594683764703547,0.01998738508368566,0.09056562248834507,-0.19053116122681849,-0.23910658981914718,0.15528009749661606,-0.1934833156710459,0.056435777393499725,0.10566269980993644,-0.06566087111053011,0.09879220453562094,0.031244453221601746,-0.04357926629402529,0.06235334272463558,0.03279241304127171],[30.06745192469569,-3.921236756022099,-2.7856132108650145,-0.48636842424649296,-1.2550242084344982,-0.2888633789255283,-0.5880817458849051,-0.7657728361410433,-0.931648456255168,0.0863306130058088,0.12863168977163053,0.1777842629194122,0.3423982549027675,-0.10297173745119098,0.2250696133905541,-0.004523818526545212,-0.035065613225994635,0.09039211145815497,-0.3407981637160901,0.3291885284328931,0.05716221392187449,0.2681265022206405,0.3878961753950582,0.020656909341485927,0.18884870149634952,0.2871249009032458,0.2880659628675041,0.2566335924938933,0.18770255221027482,-0.1518957677710659,0.20059577370079185,0.2290682051222346,0.22032379603183372,0.03338204186179204,-0.18856731357412312,0.11422921781403814,-0.13776808266299057,-0.07838181047209743,0.03248158304101781,-0.3263997402603463,0.04378382534201166,-0.13949855958482463,0.24468502600564265,-0.029846455756884917,0.09017555470752375,-0.3204188819009861,-0.14036671429366973,-0.2255579803414286,-0.13866308814864292],[1.8379100697717543,2.4496439317827328,-1.2540775361135386,-1.5286490209485604,-0.2104260857161489,-0.2872480155029522,0.8762690369402474,0.8491470839370598,0.18264570970234928,1.1477460366720402,0.3255965549498915,-0.020631869008546407,-0.20855795253038836,-0.46569791561790824,0.42979372618042555,0.46583666381827704,0.04844415279872545,0.15729209163631633,0.09294628994494299,0.8255146791888262,0.06515947841081052,-0.0474413631173547,-0.006063082001918629,-0.16160675197906169,-0.3552391750615704,0.1975394591679205,0.04823762134134015,-0.10066034440655373,-0.13764508921931431,-0.057133393155888286,0.1671014919824576,-0.07909175132376575,0.14168746904324198,0.029603308751916375,-0.26379040166456685,-0.008012781175397762,-0.07797155786720926,0.14146372725163917,-0.0722812436067956,-0.04538455035233082,-0.03671067584032001,0.17022342551625708,0.09101068765030754,0.02134957068362472,0.05740088384423523,-0.15636005090886634,-0.0008002058245106216,-0.05087922276230476,-0.13492395197251453],[6.083998545739062,0.18057395096480716,-4.401676471330783,-0.27640886737590753,1.5941405627934364,0.3069093521476005,0.34061114563846456,-0.18286330111288354,0.637173156809971,-0.18117060505757165,-0.0901748283143087,0.17567388647927903,0.24950400943897239,0.4589569937528715,-0.0458000344460394,-0.6140602689094261,0.2037318727452753,-0.08985521067019309,-0.12018371114716775,-0.006725929476500374,0.21596424406091583,0.01740649718947482,-0.3195676860630133,0.019712199909072338,-0.4046015036385985,0.27516525578764156,-0.2671289869713664,0.04888385508237598,-0.07383855625023496,0.03900440468821086,-0.1299642714532602,-0.09188058545486896,-0.3591894166388756,0.20102835478196682,-0.31538598124193745,-0.1917930567236602,-0.27155496361646275,-0.11761857866546493,-0.16733482237709052,-0.03547116995907519,-0.16977867696613883,-0.18197568343592235,-0.35426101290960027,0.02992653011941705,-0.2538591681801584,-0.024860480428114742,-0.1369949786395884,0.04334482358638772,-0.15951674870650714],[23.237557543480825,-2.0242239905295563,-10.15843737170963,-0.37111025138559056,-0.5438681294052203,-0.15855658727029118,-0.5294267448563306,-0.35176239506978363,-0.9967669624559321,-0.35340119586452584,-0.20053906519299125,-0.3941446400623174,-0.04757098925986431,-0.07241500338647125,-0.4707677652642076,0.014699820289524317,-0.041213085603583256,0.15842800864419562,0.13874998763484098,0.13388664531756114,-0.39671058855210023,-0.0558271882043477,-0.2505824845022434,0.058415526098857265,-0.1443179068088478,-0.1996543992051148,-0.3905146506261169,-0.3607057301940335,-0.13473467712705933,0.15188610424040672,0.10542539501403617,-0.07777327610122281,-0.16698970850823022,-0.06445267787694042,-0.07047696803809499,0.00817131198890836,0.003751472641333895,-0.10568394335118687,0.02238695133420709,0.07765543998686582,0.1542681317691551,0.09777564493455308,-0.06402084172114528,-0.0469723718359871,-0.16529815750437493,-0.015036870602201179,-0.030359636429766988,-0.006591253541281888,-0.08558184404132604],[1.7761755623631594,0.1185151089567608,-0.7039993176880938,0.6001932870899578,0.6556539843643476,0.29207044993488585,0.955475992618794,0.28478255187517415,-0.06422230192846112,-0.025816818875908497,-0.08926644591163743,0.41467835267140885,-0.4218525668046538,0.3031947585495372,-0.333519633234286,0.18200612167911817,-0.44328449317835,0.271064625978359,-0.5214679655446604,0.0660512891932099,-0.48372175715002214,0.2641464870429958,-0.791194576241698,0.25441136748822135,-0.7319665145486103,0.4197809593134141,-0.4060571376805078,0.4595372587798274,-0.8735522212571385,0.19412435970595898,-0.9452701508598179,0.20579736125821352,-0.5006421756270009,0.38565074481793393,-1.004114959443546,0.22010617016653752,-0.5986587507006165,0.21111823416946432,-0.5181750631280887,0.21331199547529625,-0.7318575447394239,0.14571912377415785,-0.5066642478840276,0.0911947543406407,-0.3394923990016832,0.07028567463123431,-0.6333820166497869,0.1636037843284004,-0.5932512617088389],[-8.685881533074514,4.210194925283741,7.681724916947837,0.8401098486456992,0.41426400968598814,0.6703390642973633,-0.13532116450446377,1.1400329289027382,-0.22241057544591403,0.30914786850811476,0.5282290047228705,0.26979107050160145,1.3339494764113282,0.1347333964143978,0.13056661522887775,0.361470142537471,0.09873734067674,0.2991720801259646,0.30342914518091396,0.12214586749531477,0.2135781345694193,-0.09750316853586351,0.38827867310931313,0.11981397198272785,0.2553337738067952,0.22586773924337636,0.0042967726669293224,0.24398733267899209,0.28248225532362675,0.25307897192945344,0.07314789884186469,0.2368366732754221,0.1194739520277256,0.11762059789917126,0.4134434314726569,0.14829260274759942,0.3066441539464493,0.30328283706491166,0.2924988551783892,0.3953147380860417,0.6374551555308945,0.3442214764670752,0.3092963610047798,0.36413107527890987,0.5280834745581915,0.2120509974757561,0.5761008665156921,0.2714429422347893,0.4133061746259737],[-11.459194771502888,1.5966727416751896,4.783668524774142,0.12248699865900686,-1.0134082291770017,0.7159205569862079,0.5910287854569334,-0.5004411202392254,-0.015957043130828304,0.00470571513609374,0.008540716120530237,-0.3892587502365753,0.014866857653013035,0.01130705627322091,-0.18419915332479014,0.30156978940822216,0.2646958644619533,0.08163527086594798,0.11979143986825033,-0.05111300346981908,0.12034307619552274,0.17944990068875047,-0.2310556072942532,-0.017336233831620685,-0.0388105962644796,0.014371482869894583,-0.01706460754412353,0.15172682040844063,0.25200550318422776,0.11477147543376598,-0.014260474059123332,0.2023422981795126,-0.10022611582713843,0.14920970766165204,-0.01396648132535516,-0.02785497845249239,-0.132847929229767,0.1084132128172429,-0.0877058593992072,-0.13351575860414483,-0.0894285479422473,0.005733772289918577,-0.07685302180722679,0.002124227875252955,-0.03075192717142195,0.022914223690833676,-0.08431306044492763,-0.14489416013188886,-0.04797773135849396],[-1.0939936667014503,2.5954106122802054,-0.9929496108545341,-0.7292028872182165,-0.5700359789067228,0.07680119528441524,0.013291039663144804,0.48516864953662503,0.04375092843050064,0.41379802817045375,-0.24081596774135677,-0.14739990656449767,-0.051462711506874495,0.14578684134196607,0.22195902934732642,-0.2341936414593087,-0.16841929965707095,-0.09442835190631033,0.27904294558073417,-0.007949702623167219,-0.13578243657882594,0.14331160051641675,-0.20715366786850134,-0.10166389961389305,-0.030315868619434017,-0.1268281288791459,-0.15420599693575684,-0.06643735427802273,-0.03291347912805536,-0.15128377419684746,0.07635504765843266,-0.013002146390087703,0.07861248541422215,-0.12152503683030472,-0.018519657235044278,-0.11014566359205583,-0.050787836481517766,-0.09808043359455078,0.10365764074378661,0.11224497305399381,0.025890433527479963,-0.03700451136535738,-0.0022154435946268596,0.11405154416995311,0.12066778109487121,0.06443571858808457,0.1930988640467743,0.17754800181145233,0.09040535388493226],[-3.722869420321281,-2.0029507519032905,0.3949801132094157,0.1227851974715304,0.5191979556638427,-0.2701409641362274,1.6798003842168328,0.09537376324999329,1.0206534152259934,-0.13430890500960727,0.9317732389685998,0.15920875244389485,0.6658597543089431,-0.13524164415658665,0.9852794992560315,-0.27908453252294746,0.6594849135867589,-0.25913477533250595,0.27827827185231696,-0.15162725481184738,0.6600312854749272,-0.6521387915684607,0.07718962326728959,-0.28915704384260676,0.5827754766463163,-0.3166639106242888,0.4619356098930282,-0.31201955138660903,0.8076084734531283,-0.08601040429125292,0.5406877497354496,-0.3614797949971476,0.5849747278955436,-0.15699254886533776,0.3172763363564932,-0.10187864237241968,0.004150330986786965,0.01366779191221775,0.17093664431347866,-0.07875399061392448,0.3844447644093774,-0.1626813307842302,0.1686616068789394,-0.1831282685965873,0.3199809615226035,-0.010113508635206563,0.3745289866730627,-0.08783798610135439,0.20202635297004748],[-11.518407615373581,1.23026663725095,2.1719965213804135,2.8520364149633277,1.1113780904434036,1.0537055995421856,1.3629009457569985,1.1202221386651265,0.7421188438015539,0.19939550119046417,-0.06851934233977822,0.8559551385612449,0.48481934111261454,0.14944953875228717,0.8149943170728766,0.4859319450333165,0.05385006189326359,0.3040875659646824,0.5949668528235366,0.18971827124116836,-0.11499595545151686,-0.09448978589194534,0.09047788973192203,0.352313987937046,-0.08865885550443509,-0.3518788330550878,-0.2397974559661643,-0.0035322612710071716,0.050838589587310724,0.5831966669317247,-0.027165199731357163,0.1224395823125293,0.18307329549127244,0.18273564689696578,-0.029154770374583538,-0.0363944049469233,0.016430575016505133,0.09015962664591119,-0.12782165456428063,0.11157782969945289,0.20895266837304163,0.005771262313868462,0.02118946859168966,-0.018452513971322193,-0.1238858219311688,-0.15358802386459805,0.023734964917770846,-0.03702909912248443,0.10264679441588065],[4.0283788008235675,0.720993345804927,0.14162078014641094,-0.9473006964894436,0.5377938589187009,-0.1669587691344999,0.24722055060900225,0.2869967690353146,-0.040951604100761346,0.38888493905390414,0.2975378722832069,0.9130551900117607,0.5509288955464033,0.5391147768618461,0.08292173601803472,-0.1370340995493339,-0.10047280339339536,0.031501579473589794,-0.5201682373108019,-0.3321204615216557,0.01862731815797567,0.456434822870793,-0.2592144042795375,-0.21559005639447595,0.30384689269875276,-0.3296238419562601,-0.1168411770494646,-0.009505169751567276,-0.04377337432302623,0.20590781101881864,-0.22305820972372012,-0.17114814760260796,0.13048559967420023,-0.15916120187201477,-0.0605838734491228,-0.03935588167258131,-0.13243137430284624,-0.3898604801872278,-0.01464019604560286,-0.0030448970399418883,-0.24312623800622093,0.13463943480013799,0.017513271164482442,0.1503703109446082,0.13942282106860326,-0.08350331399681071,-0.035812968618436,0.0274690814008087,-0.2127367693190982],[0.03887635041055119,0.7348320163855591,5.203515077698169,0.939672881194999,-1.63744787700421,-1.2880401562427446,3.3720755725289733,-0.7814491457355861,-0.5290621409083128,-0.32922712543964655,1.4792265315524726,-0.1829770885122478,0.7286617599381132,-0.6003605200711701,-0.03588696608735502,-0.32272369649642535,0.3365367119487026,-0.3967563278114021,0.47641670245755907,-0.3937790013932297,-0.005514240706647362,0.09214242749750413,0.04819325834287815,-0.20225889203739184,0.1469350659005624,-0.11938999443288129,-0.31681624193134933,-0.10246086532716063,-0.12240414688755867,-0.11676987479451799,0.013358682885639253,-0.2617140745362405,0.06575012700567957,-0.09753853605583115,0.01037784278031566,-0.035171348028728816,-0.24459010785953375,-0.19161567434104682,-0.042334118769445636,-0.08376981138555945,0.07624136143183557,-0.11146211223290667,0.019547628595239157,-0.10795647407484103,0.14947987003580812,-0.04422108562066543,-0.01255634493953605,0.06549033881813954,0.17261238500161888],[-19.97539063597431,-0.03088530736532505,3.4446678340291284,2.412680513344208,1.16070857970935,0.4143683668075047,0.5658473329622661,-0.004308743095317811,0.15649722261826063,-0.5535389462870337,-0.1564441646115593,0.24660283284255635,-0.0002679636653574817,-0.020778803857471474,-0.32829465645238703,0.6042181856728328,-0.04637207156212162,0.10822266215126135,-0.1937034428726389,0.1040101665686448,-0.123263787358975,0.006948731189093335,0.07822169161987691,0.001338962341043583,-0.26251299797017413,-0.22819991564663084,0.02508410521832846,0.1093958496915567,-0.20070941196820408,-0.08579892020445072,-0.021128006096226323,-0.155077902559962,-0.11592125444536289,-0.17635367845186625,-0.12149934928685803,-0.11334547643517547,-0.06816056610676076,0.0004647035330455641,-0.03756463190078587,-0.15582302543711002,-0.04927499694618862,-0.0021627952103606446,-0.00486322050951458,-0.10125562725032783,-0.034734612123723085,0.019630308350676813,-0.04887640884989486,-0.06398605930010899,-0.13928152375669167],[0.36733381317799363,-0.5862755372556757,3.187349452337533,-2.111577877115218,3.719018815983607,0.5052286159177153,1.9991290350997108,0.17653698998795267,1.7504100011998498,-0.624731263882159,2.122551624974171,-0.34950993158137295,1.2981076793156099,-0.13934694676832762,1.0594297660501943,-0.01115487425804466,0.9892887990490602,0.12386331649246561,0.872838264589046,-0.2442988983331942,0.6351227168096197,0.09764042055419228,0.27730451895265235,-0.1685954660459616,0.48939118246204155,-0.02691923747350718,0.448486230492149,0.042244433977907885,0.29024334877755265,-0.17879073165109094,0.2020855422990331,0.0877112140653051,0.21526248483760038,0.04597485692837037,0.3533303586654731,0.05314917296499685,0.08953785561795699,0.03106387471737611,0.06569824015362292,0.007451794246463811,0.14717697978567074,0.03913586340558729,0.1719290883269055,-0.08692920835685188,0.0858619608281413,0.06784226363849635,0.1460225227059479,-0.008256899816389177,0.23872197369777046],[-17.651406403736548,0.315157054564862,2.2977475042751814,-0.15294691805667363,0.10403381885958307,0.5083832647738162,-0.8188767860499666,-0.18045473798858458,-0.461646640723687,0.19575364266130013,-0.7666952318679078,-0.25827381927558546,-0.5848445323837511,0.020248559136390745,-0.9005116001209929,0.3514467308307814,-0.559678661206155,-0.14400753509983918,-0.43464376141078215,0.6134080753033673,-0.39402811141976946,-0.1073373428411801,-0.01156455703539941,0.04652661937222317,-0.2483578245023328,-0.3901467886473488,-0.2955492077855208,-0.20331800006382972,-0.1341226313600042,0.12145999036948753,0.012880420392169611,-0.1497463980045377,-0.2214937606283774,-0.19825512353534547,-0.17176993090875758,-0.26580634173612916,0.04377082963207256,-0.2575911794259316,0.19208292345240716,0.131848939454373,-0.07421899750451559,-0.28545241804886984,-0.05332133055368053,0.23130256557449982,-0.08837777670138733,-0.016094666234226312,-0.10079080421639786,-0.059045816395203364,-0.11581008252362629],[1.946462295575512,4.078585049525194,1.5761291739130436,-5.460327702022447,-2.1727410164092666,-0.1636883911595882,-0.6136377160899095,1.2872681119969338,0.9725007090128205,0.6880411974070547,0.5682779329851722,-0.46077878715906817,-0.5283409352178742,0.08675290915737904,0.5292006016694475,-0.0765509428326939,-0.4426168237159438,-0.1966758248754357,-0.02323315355096642,0.2522589192931283,0.5125424318680197,-0.3775930985731617,0.0016878571097564284,-0.04541220481041848,0.269475096829036,0.1324657452556025,0.2168309103867283,0.13432442338414224,-0.011841687951743535,0.32235634860256457,-0.2049256021073468,0.12269731240655606,-0.0558978454307498,-0.02324098589505855,0.0759490440352959,0.0675506558388077,-0.04727369358567283,-0.12399911103266843,0.11793708771843309,0.04790650286857656,-0.1954454751273081,0.2943974879842192,-0.1937236661118537,-0.04108632109105707,0.08043947325292743,-0.06371445754250701,-0.032602642927033146,0.06723814716340146,-0.10643026546239877],[19.279518546664978,-0.23669595820430533,0.024236688645216976,-0.4085383106445296,-1.087876264940649,-1.0052909688370988,-0.7536632564760827,-0.11303731255674222,-0.1567692270318249,0.10813843224760983,0.028925331656535692,0.4108006686786116,0.10147860731576387,-0.21459693129470905,-0.4736945893107795,0.8495998784101412,-0.4816106137066374,0.1602214664681903,-0.4807672728596187,-0.002537611716796436,-0.7379998501363894,0.42497707345337477,-0.217384887836012,-0.05569101883186531,-0.27671522404219034,0.3707159967748725,-0.43398752042545885,0.04219408469049087,-0.4013252921930433,-0.01882879088458129,-0.47348404624448126,0.2827800098550083,-0.3653108366973645,0.39412722815048506,-0.3905194231908426,0.06122405397313466,-0.22092490573896628,0.01173997198432245,-0.16169461470283958,0.1592354149953091,-0.158743711529368,0.1624237106072512,-0.05412571121614662,0.0016868055493723792,-0.31705975412487525,0.034366171597514324,-0.18467649918479814,0.045274005894613445,-0.3490571844163579],[4.7682542906914405,-1.8757574080372148,2.316219861377732,0.7106175029684086,-1.7298104992449757,0.44505348470351486,-1.6519327890413749,0.4034355533254107,-0.8770998792534371,0.7176386697460752,-0.46951120029389565,0.27319662815471507,-0.5162468422527707,0.8367714517944694,-1.109283866242247,0.8904789263820518,-0.7769482724262965,0.38965965667834496,-0.6826783289019589,0.5811950010183011,-1.1459696704835034,0.35324157308629606,-0.9748400544451649,0.09195031033065082,-0.7459382911933035,-0.01637655581049952,-1.0539170967525266,0.7712172406612873,-0.8169918124878833,0.494367616301584,-0.614133048699787,0.01021601240717586,-0.6628259823415273,0.43307556838910527,-0.6176913493018155,0.11821777540313051,-0.321498102577865,0.006102009924074262,-0.4915718916414304,0.14848500832780126,-0.1638169425855,-0.07885812480405478,-0.04024526747318344,-0.10783746088982034,-0.13073192441884135,0.01717270000784572,-0.28093925661375657,-0.048234973888429275,-0.060021054981356126],[9.1089228183658,1.2032622743435923,0.21933190900004543,-0.46340321269477147,-0.11555596497134196,-0.052951544689031294,-0.5118909537872048,0.1818965332369518,-0.3365534390400753,-0.09725404507413331,-0.13243008604794654,-0.10812184151531919,-0.3726078377259066,0.14027845599166452,0.10381746610671232,-0.02015660919679586,-0.1270550822858608,-0.06790880523713744,-0.048937607674380365,-0.28445618006112067,-0.06482495618340084,0.2134477781858005,0.003947416663798953,-0.251640548418463,0.16571205412953782,-0.09213849319861869,-0.015272066636685224,-0.22203072149187145,0.05438832255010329,0.07489428042375097,0.055517762667851654,-0.0329932745313155,0.04105242767253232,0.0350695593152431,-0.06428777193851842,-0.040461265448286175,-0.025198549208255556,-0.026446305092562458,0.056238740864870594,0.08450895068754401,-0.07066194993825818,-0.012413291364516956,0.10013783285002313,0.08179112290028802,0.0005846025468408917,0.13282366968202794,0.07840886589376313,0.024213455678383823,0.0818864151671911],[-16.912281587702132,-1.9725426571359717,2.155655556489861,0.31012496217166674,1.0520306948216236,-1.0187976815278474,0.6092844612916888,-0.8730343054114987,-0.5866919492629565,0.06936064364569079,0.26746407877102485,0.19184569266379203,-0.4363140232179509,-0.8283231901089054,0.15560414294621347,-0.5623158313326446,0.49564722417525964,-0.09725531892340233,-0.049884904495895874,-0.3261156075241822,0.024339217573926597,0.06042754127008996,0.2885965503427919,-0.2397837974431643,0.3047420581758067,0.1953945502831961,0.12769990730478584,-0.07523794139963064,0.04448507608477683,0.01471031645130869,0.20610190535897688,-0.12312356984983093,0.08459903633332473,-0.24542178396114564,-0.005838804371037123,-0.2269044280399305,0.18529545660353827,-0.11161954719472894,0.2067722403206789,0.08092054819469204,0.1205949150015182,-0.07655116430360817,0.22183913156249022,-0.0622737486517792,0.13631035120362836,0.06030139707324816,0.066406822338355,0.03496607093986659,0.09428665117248389],[8.67024552499961,0.23404192143078598,-2.8991502691388886,-1.131960421860257,-3.236656938003556,-0.011792214087728885,-1.4774358671082857,0.1076679119993291,-1.8054533066582146,0.3933126245003419,-0.5574492134499198,-0.11769461399943563,0.06751596375183055,0.2528595570551349,-0.4374546164792595,0.31469640988282566,-0.3706237173275084,0.647443070468895,-0.2670973896188602,-0.10669234141273128,0.11839483547893484,0.3955697822975505,-0.057175866853654896,-0.21631006666155525,0.19072586754652643,-0.13260935303553714,0.2936947652666159,0.3309262975525459,-0.1196842409261829,0.12220156986306265,0.005317802847588549,0.24375278992731916,0.1108265119162418,0.2043594000093942,0.06080987243202631,0.17764434812863425,0.18763814401322995,0.1540994750314192,0.030956344447474762,0.4542984584050461,-0.009656614180457546,0.015258436302450449,0.17273060366999146,-0.09288044216838255,-0.11877041373534376,0.2027235925561732,-0.1353673164134255,0.002930044597544939,-0.047635958121782146],[-9.177203492929019,-2.649649481814421,-3.24025788764567,-0.5352489921013213,-1.3852750620156675,-1.412602390018549,-1.2561420288658696,-2.240693222332844,-2.432869819379022,-1.8370679803934051,-1.5769594122424773,-1.1351322184622286,-2.1069106395132344,-0.5966605277174019,-0.8770590769072069,-0.996452462822699,-0.9522217048147045,-0.9746416222535469,-0.5544285792121838,-0.9245605308291115,-0.8495456332685629,-0.141312344740373,-0.1818128527348577,-0.32639900010726974,-0.11402278485649384,-0.3877743265066121,-0.37029925278358355,-0.33186921906631023,-0.15802198112999905,0.04758232765300841,-0.15115722339058127,-0.03969868319334895,0.28046658920630035,-0.17014373130994087,0.15404440669946973,-0.04503392519670658,0.11473860527761878,-0.15408185692225113,0.2767529492079624,0.06953766006047277,0.02791327122871564,0.03193057654668875,0.14070234301559173,0.24635462528793753,0.10205502456985258,-0.01827086642484848,0.3036945281227454,0.1854423829285277,0.1689371481863124],[10.079026718378039,-0.6611016323782853,0.1765130090659452,0.6116802201942562,-1.6445873759830816,0.679586390234174,-1.1333126099078743,-0.22488746181317823,-0.6244682648896998,0.16321639182512895,-0.19797995945910327,0.5527452191617018,-0.23972520307301332,0.15856965111754875,-0.15241362650244342,0.33930724834914877,0.06856980312317071,0.635884717512564,0.11545953607087428,0.47968646825058797,0.024741847109890112,0.2702535167420619,-0.19392202439622147,0.6072358433648383,-0.23650657887107676,0.3336419743018639,0.12322055526843352,0.4333705516418253,-0.1644574480335352,0.30114957954515964,-0.12479593248865678,0.15724005925913706,-0.1100892205263786,0.1676259838987524,-0.06703334240150095,0.17100401583302244,-0.22611858297347928,0.08556900208237124,0.09632092601873259,0.012297955363668081,0.051306057734771586,0.15962800060096982,0.28567077929530604,0.10163556021444903,0.1334907344258827,0.1781002737250283,0.025267736070405108,0.043873386387773376,0.28231027003313425],[2.6045461514012422,1.9212575416000879,2.038970206278025,-2.329989252168777,-3.363123020855894,0.0036735501138207747,0.4041718278761163,0.0722931831840147,0.18161181089771142,-0.44357293877004955,0.13461667480821093,0.15272807400490163,0.11316957422173912,0.16445013064698374,0.05114807132210914,-0.16287867702965383,0.23354559261008084,0.14170028532031123,-0.07874250159826818,-0.5463739903101534,0.11980896058788927,0.09667827027518672,0.08142522213282538,-0.14478401010146527,0.009282569169704187,0.057210454253871945,-0.1828095566495357,0.038290166102448286,-0.10681407824285791,0.10112247613053486,0.0154388709381785,-0.16135482602767334,0.18461830609473365,-0.03421155850537785,-0.24399890201645794,0.041544740197276296,-0.25033810428070913,0.11578270485521332,0.17803788202617604,0.010123335331472694,0.0190541915976766,0.17872486787334657,-0.14090218826837456,-0.06884611203709556,0.022643593852262982,0.0831789112380295,-0.026911188233077962,-0.17735254248614052,0.09992647805929457],[-21.302548077903243,2.1772390817618414,9.668915102015948,0.40705845892159237,1.2829515724119338,0.8928550816791831,1.3806391170767904,0.37911488512691904,0.8975773170911925,-0.21928142529012093,1.6365686913514008,0.040944627686827786,0.6183135904974356,0.1666911405826532,0.9814287240149397,0.11646386874401081,1.142784113498482,-0.0001373767481838508,0.7372096093318327,0.25226887250204977,0.5062886334288604,-0.0011612260677404836,0.8097455384784429,0.5609633022910797,0.30239675424184587,0.4671544924500428,0.07308457125340892,-0.02804976933238877,0.48642913776783636,0.38874136956709127,0.3176425824924187,-0.05480282229852351,0.38029415386132087,0.32056751264212,0.3944417061930846,0.39481997736419694,0.07019167377591999,0.05152625305569737,-0.018903944388324816,0.09457745787632822,0.043806778601407755,0.10673808436055215,-0.1435976656828973,0.15273578698639448,0.08081347051592648,0.023888560287058904,0.06695542821941151,-0.0562113232447132,0.04200224712503433],[-2.7931794907516685,-0.06942840727509132,0.30158694973477446,1.4432471665141762,-1.3972880250582886,0.9311016063385991,-1.589687481007221,0.8754610553238515,-0.6089463492957028,0.08778266213225035,-1.7998918814895817,-0.3974601227255178,-0.7730028372368349,0.19651714560744107,-0.7759980977677998,-0.1335430990156077,-0.6440617175003504,0.07926062800887498,-0.5045934519129486,0.22419442734163916,-0.45612940788191847,0.22233003315119057,-0.3670894435910182,0.11568157902783158,-0.24903262455787395,-0.013674837185498513,-0.10283204008721751,0.11013668614258955,0.022479581710503935,-0.22415402609586796,0.10764628571527933,-0.20525015119286877,0.12412002044085026,0.07349045613720395,-0.09755670199835462,-0.07098353283335752,-0.08522970728286197,0.009357381829657261,0.022713770745568775,-0.13343735553828223,0.004891105017446946,-0.16258493561569123,-0.03948155695530291,-0.3507725948924722,-0.014659576035992642,0.014973459958062173,0.06912021545722637,-0.0077501582457133195,0.014757141019742428],[-3.3344137384029793,-0.5040714964922957,-4.185080219174012,2.004470534197498,2.8074035350915962,0.2862805657013337,1.1866743275371978,-0.12130330651289338,0.04905327764459396,-0.07457683018054453,0.10177659991472858,0.6120901263577335,0.6525215654688833,0.5445840551228855,0.4148561933661556,0.08134435868148164,0.3858812519988173,0.34846477819353316,0.18231518600070976,-0.1468299725970208,-0.46454057276946864,0.07076995921576364,0.3680708820394262,0.03346786062010888,-0.0808186664955872,0.12416841790267745,-0.29835192165977065,-0.23168063747258136,0.4134429724910618,0.040923226948791545,-0.04270417536890402,0.19355447574972193,-0.046455535796836624,0.02425089886632763,-0.1364873032955053,-0.05414317855710723,-0.17191813134126266,-0.15868444768534212,-0.13597619073137368,-0.01848400074023744,0.08042388202937419,-0.08055936550182308,-0.23495085163270407,-0.11808806737168875,-0.16978195819976294,0.15545893092478136,-0.05490508113064058,-0.10246545244882921,-0.14418640951581135],[-12.072544171512398,1.2151565385122887,1.9673200387647771,-1.733170302013994,-0.24178601227779897,-0.5974949247384669,0.3378258567513335,0.2698476424400652,0.17415177352956318,-0.4914967108647628,-0.06202321698960703,0.20025304864393753,-0.18080112886068433,-0.40623689455337186,-0.02145019221143901,-0.40275991986334814,-0.05159375792280707,-0.150719729526972,-0.09743660059950086,-0.019700511786838133,-0.4676085657017991,0.278404607783464,0.21339398327532103,0.1786954926664313,-0.03150259924554089,0.05470416720397011,0.10843824972936002,0.027706204416297585,-0.16076885761773266,-0.18970572468273905,0.006360210807254548,0.061282118808274916,-0.268728369488084,-0.1491619377557911,0.20717398591671324,0.07638154321944,0.030674588805831564,-0.1597482754092105,0.1090860239997001,-0.15894220833569525,0.1918424701103418,-0.11516318058223506,0.11832568582029077,0.10779372538626003,0.039456557089013834,-0.04480025656065032,0.06910253714871657,-0.011107999235497018,0.06007326498530175],[-1.231793393768179,2.675884719012196,6.122337469242611,-2.3372853147770876,-1.4514705428217192,-0.3652086417241447,-2.063077258305866,0.7633912409564124,-0.26456479340457634,-0.053719423510460954,-0.24876262240984937,-0.5409356185465228,0.7348907911977602,-0.07062763619801239,0.2894534489845489,0.5266448263009714,0.19400604932770102,0.07360913613218442,-0.1345407565102365,0.19545034101116665,0.8243826482823209,0.10116739828154114,0.8551119415071996,0.17730882243744073,0.5066286007265599,0.12769646226324963,0.6730008837174245,0.3013135226001251,0.5359944635346309,0.5100635251460682,0.5280578777818004,0.26812417743139016,0.37502556617358096,0.44959577154278474,0.5180053946548671,0.4838689359034155,0.32791012900255934,0.16218277518521287,0.4217235780763709,0.3136690736406448,0.5857582997473982,0.32545154486256506,0.4857146838278363,0.17528313925470657,0.46956580524222363,0.2572668389711092,0.41906434315608543,0.19974634219012394,0.4207636820989425],[10.28880808826743,-0.667738167877326,-4.261561904140667,-0.6728202420889123,-3.4260288294103773,-1.0577066776068702,-2.7615496429677235,-0.7187197672422274,-1.7537066157778114,-0.7807364052234103,-1.1416829255244898,-0.41911907754799815,-1.1118282350183373,-0.872916582311689,-1.0877048991804188,-0.627880599581768,-0.9096611944584508,-0.45626778839692034,-0.7453045320519991,-0.8194153163513304,-0.4813832621993597,-0.6904495327738346,-0.7447945920541365,-0.6178990008463974,-0.005530154083053276,-0.6020627555685135,-0.4234812923205595,-0.6309569548234849,0.10763182198519482,-0.36177265528695257,0.24650421349211504,-0.6304565898386455,0.05418252828297746,-0.31297245854470856,0.29673937359716707,-0.27130342358932535,0.36554893956373236,-0.16878479923974984,0.2498877096668005,-0.28129031654102366,0.5373688234650629,-0.28398389889127756,0.38069053896123,-0.19130516653948687,0.29220098227748037,-0.2801351356211725,0.5784483315580241,-0.15063321126905402,0.5845424324490592],[-18.1426341127277,0.5351094437181205,2.758385861593985,1.3159976526902382,-0.968508309406591,-0.8081172787392092,-0.9114764665072801,-0.11166184941952226,-0.3997196460856106,-0.5755778999207856,-0.7551743755460462,-0.004015865236186404,-0.9700063765826092,-0.3336168538327211,-0.3103195586897725,-0.2753883784754018,-0.46115831391531137,-0.28911129218205284,-0.4181687829830247,-0.38662925731820075,-0.31628013897220586,-0.15512437197584678,0.01950965159716133,-0.5539204475846785,0.1592297463173272,-0.5330567151476164,0.18417972746732506,-0.0739676282875514,0.2173911219196512,0.03165317074596831,0.3161733371190392,-0.23614614333644873,0.09046620965973172,-0.1392027655522254,0.0657935236383248,-0.005130573902619049,0.11488446947012462,-0.04801346238411466,0.2787027710487053,0.00937390565784643,0.08594013807668942,-0.09791798960298657,0.10660441808190557,0.08368550932132321,0.06734017345738418,0.15601966654715896,0.3178273725334553,0.10079791152789894,0.0641906278089577],[11.405617330638792,-2.749999565104601,-1.2267354411151576,1.5984863109872745,-0.367707548807814,0.6602397308969717,-0.475499278867594,-0.11512085126344719,0.6586907720634266,-0.03944003050300194,0.2848044069319286,-0.5078404697886248,0.1533632039215476,-0.07093454134247268,-0.34543867553648466,-0.0014945031454866803,0.2702774187469081,0.31267977073527137,0.037713460403816894,0.20406597313481803,-0.1453626680017263,-0.05193439336555104,0.13394282218261117,0.12197586974988239,0.1961719455272344,-0.03443943527264269,-0.2465651967578115,-0.07049500810735426,0.06435719586902267,0.06944303658870071,-0.269019668721332,0.18303587073219443,0.09765847199405535,-0.016841093060839106,-0.00919567227249402,-0.09594419601923077,0.07226930345149404,-0.21957524849060286,0.09756258981822938,0.08398711003657414,-0.1001130089884328,-0.1644915302541218,-0.04517664126214176,-0.08485595363015741,-0.2590939832892422,0.01565070285664165,0.043446645051680395,-0.10985260973864108,0.23890799547848682],[6.116001129556306,-1.4280398660405325,-3.0260954319520543,0.525781615041196,0.9640996614966143,-0.2929131680890885,-0.014739160578318374,-0.23846588446969028,0.08273525879871539,0.28990492500106513,-0.22736620885760156,0.21860574413585085,-0.14185443421983795,0.3531623469967126,-0.1325522456169325,-0.025755731592043595,-0.25840366522436403,0.30265172143972335,-0.2666777649079699,0.2899054268899717,-0.5733606857881305,0.27660406562385953,-0.29151430346834556,0.20140153988141246,-0.44089573068905985,0.23663489750195085,-0.33800515394646363,0.332260321661463,-0.2215151819281584,0.2540752555870669,-0.21093467002672353,0.24572597664677956,-0.4401084443361864,0.2943845865977143,-0.3756533057754368,0.22420898115065538,-0.386132942577294,0.24515630533095006,-0.3513527761415492,0.1600520968591662,-0.2763635588932277,0.09681913557226998,-0.32132083265817324,0.09932540929839767,-0.30133686433438983,0.08936510082722196,-0.34245794915526195,0.04225442153277091,-0.3494298597319416],[4.667642318667732,-1.8408851187142297,1.5823686531025327,-2.2209418691524503,1.1778295641810108,-0.03352562265312753,2.018977120977465,-0.4824128855917758,0.8334376703469806,0.15967228708513595,1.2754664985606088,-0.18302676305936602,0.35920481581334796,0.33971272914898026,-0.18227599193063465,0.12499560980068657,1.3791701414993784,-0.022231876422140313,0.3060558148400292,0.1155300107201486,0.40277899021872454,0.0695280269366156,0.525882691825136,-0.2917555429332258,0.15322404974630616,-0.06547404802687916,0.05253119284148769,0.13139855194463607,-0.02522486278291151,-0.14360056969050092,-0.19097349017101053,-0.05075023312293331,0.030464136392235767,0.1615000691920535,-0.1248568458009125,0.07575133331517366,0.11361184968376703,0.198683011103032,0.029767323790616612,0.0975716176559097,0.0830235033488341,-0.14376948685830032,-0.14398603756698572,-0.029832427483814598,-0.09480651095621741,0.0025312932011195967,-0.287492590036027,0.05116666985393538,-0.2518314354263229],[-14.728670407845147,-3.2861210015771958,3.7155203563390904,4.0317215940116675,3.004554102303346,2.147144434496753,0.4610856571847624,0.2818872021908388,0.25869223280744036,-0.006958937082163835,0.5363336764915682,-0.6259708896049117,1.1116196037979118,-0.48186368974973465,0.21809949439601756,0.7831140533722131,0.3657752927224906,-0.09903356042143067,0.6337202464317685,0.4280916358667782,-0.10068822086453061,0.3451354032010987,0.007881212621706752,-0.004036124997228877,0.15656502321207547,-0.22248619956233662,-0.16618345715479899,-0.27251651013422923,0.09731525754916455,-0.06544083637030618,-0.09569630617059764,-0.09386988950612797,-0.06079394780906391,-0.08232985332386399,-0.04556576896228575,0.05178296499486595,-0.0759464660258213,-0.08024053881528281,0.22336150793298087,-0.03296896624396087,-0.10504114520581093,0.19308219375378974,-0.3810812510414013,0.12289853008324689,0.0862318133359783,0.09037613376085861,0.01029735661043623,-0.00938798075927621,0.04792370604424357],[3.162746612632297,-0.13367905915346304,-2.7316665043617134,-0.6700077550676115,0.5472367724965126,-0.18404129199335023,0.7910478488699372,-0.11725888443844035,0.1466546347080427,-0.44195693447832035,-0.043128554802392444,0.2606211796678777,-0.16028273342780808,0.09117839761353837,-0.004963884728149147,-0.15416680670710392,0.06010987937452143,0.17781981546594774,0.35764204515505854,0.13555719087216733,-0.11501086693155849,-0.124434969022463,-0.07121914780450889,-0.0005920524089176389,-0.03994236386680971,0.0411474044785374,0.18041410496998506,-0.11841865440475059,0.005717744435559847,0.19486306485944319,-0.08772717862130001,-0.09867696168221088,-0.03441106839219981,0.08662263898833997,0.07959591415951503,0.05701464418784445,-0.053132503862603056,-0.03895574067047791,-0.23053256692152335,0.06888627502618497,0.05914217291626494,-0.03645732253593596,-0.08229640002687202,0.04531558692262142,-0.0476637393258093,0.1487857641305837,0.08228359418488954,0.027935361351898252,-0.05085006882848112],[27.335263398439075,-2.170742964062895,-10.313812148529964,2.979907439053797,2.7728881793313156,0.5167634927128886,0.9225424771192644,0.009664020066711151,0.5877600550131228,0.31891141353185115,0.3292996855132816,0.5951412377418535,1.1451117446549337,0.05614430906056788,0.5434127408819159,0.21391042157351298,0.40390911303334354,-0.056958605775611716,-0.2551320101707521,-0.3810841869500863,-0.15671708764400114,0.10927030169104865,-0.6223455834605088,0.04811695018142995,-0.44440183643359116,0.23206060259490224,-0.731101189830318,0.11449922992952344,-0.5973084991180044,-0.2055995471240989,-0.7908751985159607,0.001565695255369939,-0.6255350466985683,-0.19238457706179468,-0.5480685539232235,-0.21806348692081082,-0.7182746938662061,-0.20956638175851242,-0.7140861935383471,-0.29895430001943896,-0.5892956235414948,-0.19538198675452803,-0.8743882735921795,-0.19297323298515132,-0.4761986691677538,-0.00761468292331751,-0.8648586301022751,0.09381975313411264,-0.7326044912552784],[-7.388939061092779,-0.2815006247938199,3.7461208924168057,4.418538425355432,1.8215675416051913,-3.0165387484679935,3.8050212535896293,-0.750930263072207,3.2867634452606604,-0.40524290365610754,2.3896904754556623,-1.1701147651322257,3.1471891646591734,-0.6846965291788311,2.4325500857509086,-0.9154368117940731,1.9675094414347574,-0.5326292940386779,2.0097042364295445,0.04023499206604676,1.4611552174312463,-0.6562808827808686,1.5596035487218605,-0.39177585109352536,1.1066939759522676,0.08876881741900099,1.1842553823239814,0.288315287612811,0.8368525800387984,0.1913002249406553,0.4540618328544336,-0.0897586498326773,0.581188945733465,0.1743783060311717,0.2103910157167058,0.44408418219951473,-0.02469671987058323,0.4177443626612305,-0.11498344707070744,0.40835489877910447,-0.2557040968411747,0.2029271774765296,-0.4080650179262325,0.3687238979095313,-0.3949359173545133,0.2982141792175841,-0.41773498157118427,0.06763340068562806,-0.23515192970376986],[-12.97812870936392,-0.2737701195211495,2.4969136373247918,-0.24112564078055346,2.4532828236888577,-0.7472264449195412,0.3800197932875992,-0.7259065542525536,-0.8484751199917363,-0.04900870803640974,-1.2170796967882156,-0.1844115847019782,-0.38582629911623917,-0.0977830510658622,-0.1604642687315999,-0.624912455263637,0.46838805289657814,-0.1935380042022767,-0.27963213124616776,-0.32633370237485204,0.032392643289584036,-0.1620810589063811,-0.08442409894212222,0.3195535030405665,-0.30767194981586327,0.4603105382425639,-0.01333935236993854,0.1373806141040113,-0.12908761507207858,0.034265170699003526,-0.1974890554691057,0.10321821561181206,-0.09984255561509942,-0.07354720026482545,0.01242220169964462,-0.014115174375060876,0.23345047643600209,-0.026257787235713512,-0.3344488376234934,-0.0933126451117041,-0.23808198699743166,0.04850873944924054,-0.13949204039910987,-0.2284538395069575,0.004483521533715701,0.121640571947791,-0.012326732363101717,-0.048952884210292986,-0.05797191208210424],[-0.6241566898537618,1.0048077337985246,-0.7914109068337292,-1.3577901257367049,1.4500870028678474,1.110087443149026,0.5626058538175951,0.16470289789680406,-0.4409313248110617,-0.09434567980094559,0.39654395003304777,-0.1992932055600736,0.16530007878637074,-0.03022827310507844,0.2800942600579389,-0.09848060138327065,-0.06836344671526297,0.2890556853425442,0.00645764536427005,0.14250851952115298,0.22241515039730753,0.13961484361620288,-0.09018544836979435,0.26833579208154684,-0.14185823380042611,-0.14808769050750292,0.041440298571697974,0.04471679414898418,-0.4492887310743677,-0.06834803490674281,-0.011594475319401913,-0.09689737587074104,0.2626254675244001,-0.08084152150436755,0.1535645915152558,-0.0741193554432022,0.20834342936764863,-0.1880328025555585,0.015408091843332987,-0.07194461094048284,-0.11956718526911646,0.09802618307684452,0.16237434192788078,-0.055680875443931944,-0.06772081939421036,-0.018135275960500553,-0.16988543242472,0.1636428341279025,0.01115069800879783],[19.329609518584324,-0.9892125186405923,-3.292844802337834,-2.3061644310448317,-0.4812613901243544,-0.7762891240602727,-0.0520076376041207,-0.1336626389066795,0.34065264712873583,0.6429819924987972,0.41092882785868295,-0.48593285648896817,0.42769535557185273,-0.0035261998691641606,0.15918569481184128,0.004955053655090714,0.6018062186002587,-0.13926823337315342,0.8476718352270202,-0.21099864587308345,0.474920915410916,0.06282882844169935,-0.014693528990267577,-0.05010775021006856,0.30956773488237094,0.2840824474413732,0.2551376868336529,-0.039030365927364634,0.5597576964271288,0.03255455003616962,-0.024181126815748254,-0.19359349461423064,0.06643565100846185,0.2633318917767955,0.42752497741165685,0.20078907163616846,0.16588291913928238,-0.058121239495575974,-0.07408247395317334,0.07479013521445879,0.16148381242819057,-0.16017660367734063,-0.04712616201840602,-0.07078984288899132,0.15846641975025402,-0.21758684301470366,0.08572970825986195,0.0833513215818139,0.42525872145062504],[7.136020574696563,-0.5838179930990942,-0.9293506466780393,-0.07712002801734621,2.7731571050541803,1.6066927469971621,2.463942839494993,1.9290882891350831,2.5809360691112198,1.3837442210908206,1.0590831557042875,0.857388507283512,1.345371711182885,1.0367990194482706,1.1519845142651528,1.2881992277143604,0.7168116651152031,0.5372642477891972,0.5892977969716935,0.9076112296280782,0.4548649733281118,0.19392489444858035,0.16571340286692884,0.2991388820872185,0.060063998990712415,0.30801681493297306,0.19159861237885056,0.3055984950851223,0.03876110006079596,-0.13833869400963314,0.12061799688421158,0.16812587840891746,0.14281370976334679,0.0016786187515864526,0.0052628958625687925,-0.06366769637597144,0.05874299152564144,-0.04947045620393294,-0.15858937216264915,0.1410156640565438,-0.33389170695750775,-0.15164722953051568,-0.15339185267387667,0.0324446725981752,-0.16434141195963975,-0.009898187977176976,-0.3937761760683958,-0.00842204989698178,-0.3022899661169588],[-7.420637334097929,0.5719204007051505,3.45504738973853,-0.011655309091900348,-0.07307933489924849,-0.4079996842756294,0.8204019756438904,-0.3877861263139827,-0.13053352585650338,0.38664158338002724,-0.24767089964441327,-0.39421405112901403,0.0857391894541208,0.2940039482598282,0.1227680785394306,-0.10340768885326881,0.17708227759135237,0.19017739568938619,0.31387888078821874,-0.10056028673019247,0.12065936887617797,-0.24856208292921336,0.04850199548861864,-0.14582138681333917,0.11102817166315328,0.1259818750676593,-0.019740441026704643,-0.10198933360996554,0.260502035940815,-0.04840893304883819,0.08189705964055592,0.004548796304333441,0.08139535370240443,0.0716951083376172,0.0324915724460171,0.08146035731372005,0.05453039148224693,0.08223437001476774,-0.08561036269788183,-0.0743343510087405,-0.026004893772045394,-0.0002850340703490824,-0.03311144386749441,0.08098373642631687,-0.07227894386559525,0.09363008464851569,0.07954867006173623,-0.09961597116575528,0.0029884611218421605],[-7.076157623982272,-0.01545632613501862,1.8132497461003412,0.5125387700623771,1.8857725170107782,0.41453411857499917,1.2011257113844216,0.1535917697773675,0.5772856376537817,0.5680627441763059,0.6108452277681675,0.5267828838826029,0.5656735493642183,0.01965255518535496,0.21453857604770923,-0.045930162638016026,-0.03600350486602562,-0.3850060653629199,0.2712725337955392,-0.05278530511048658,-0.0678394108113207,0.06715910397375148,0.15143708883293988,-0.022997171190534155,0.1765995849219066,0.12596913053097228,-0.07568610634493525,-0.010325388534185846,0.00502419808211883,-0.27563943578614314,-0.22473507104023477,-0.019077673329133772,0.0654504965456933,-0.28214231822180397,0.13921735631466922,-0.04213716372553489,-0.03793129718463216,0.12226681287291306,-0.09519229789208158,-0.07254939348691365,0.15173852605992144,0.01717773786631273,0.14849215843244354,0.24534006238047837,0.09511848539010137,-0.020681445179857347,0.03145733893216576,0.15791448147601064,0.2731759831345985],[2.3707103212792,-1.877306600417725,-0.843045533147889,1.1856862350494293,-0.18358453120983473,0.29335587574578337,0.9613188725980307,-0.15241827398414468,0.3271222773264073,0.34895840777913817,0.3177630334805373,0.07661809363698467,0.5780005299143383,0.025806090140723204,0.3885994316153447,0.07423655242039805,0.4772087895675701,0.16974932394746797,0.22029482739919662,0.48331294844905714,0.4522267579627444,-0.03269448010709056,0.575777850768769,0.31740179617108816,0.027963256549107727,-0.04913125468350251,0.14746270718207974,-0.09807717571474536,-0.4630683218057483,0.0367792055814209,-0.055533864552720795,0.2140368585375559,0.24624324606180956,0.15096148214221058,0.19491028770071417,0.09388919909105685,0.3443249652089842,0.03796237387294874,0.003254120523224854,-0.10919073440561324,0.05438447166141504,0.12204304594377625,-0.02943080645418395,0.047824247609606414,0.03283173234881262,0.07860721044629673,0.04339891226387389,-0.12714412272512232,-0.17363137868659398],[-20.168523095459,-3.9228276337933083,4.246398042570411,-0.3533929236000927,-2.0396389068071943,0.005909534226060962,-2.6552802556498976,0.08302034664199293,-3.3378919776518043,-0.044431673779913396,-2.701691212981761,0.7095008994902791,-2.1779827683035657,0.2567407887243835,-2.6819658836828535,0.05583254256570671,-2.176667796672278,0.08334696570031624,-1.9173056702932028,0.3943948933485173,-2.108782495123883,0.4162900937157879,-1.6448812614876855,0.10910182188462472,-1.456357829585325,0.33716142334853816,-1.020385771403241,0.22817278452890316,-1.2041607446473834,0.14402447076134806,-1.0110566115439166,0.032247685619997975,-1.3395127013105528,-0.08852615092936549,-1.102998507659357,-0.04470751251507311,-1.019907280480992,-0.10128061028355596,-0.8625986504815951,0.19215349884427288,-0.7267182596981138,0.09979904127543218,-0.6874271925455786,0.013025694822881654,-0.6333623215963291,0.006640851462208957,-0.3286207159479125,-0.2226688177708638,-0.33462912404144424],[-3.7727044186601235,-0.45797996070815566,-0.5899106267888297,-1.7123250156264926,-0.8210056131908814,-0.22346313856422328,0.5454555985336569,-0.033587072169523384,0.46590140729817003,-0.7643185400354562,-0.32101368439101624,-0.12713010218334184,-0.3180669833216565,-0.023213381373551008,-0.2782613032414238,-0.14380329802721173,-0.35354578644873835,0.251950681185027,-0.25211758081732827,-0.5142767946222929,-0.030656405115310665,-0.02278115701141558,-0.6578946909243814,-0.17881813744893008,-0.39676553987187735,0.17454651147403946,-0.03844479497470992,-0.22380174031076974,-0.5412500628342714,-0.15556865089277483,-0.507675133886977,-0.4548466280119216,-0.4210351516969863,0.24409774251264146,-0.05526142620566479,-0.06164132314238242,-0.010143229844916571,-0.003475262916793371,-0.055760586450626815,0.3360336965178039,-0.31199420378875126,-0.14228519731755804,0.07140552313619229,-0.1704851043615394,0.08716872909432735,0.04097008086855601,-0.06306142703966298,0.08255467405081013,-0.21157732258439835],[9.039895184135135,-0.8361384659597709,-2.855336040639914,0.6713248580970667,0.8897810010617869,0.07565094294828101,0.5713561987764757,0.2796394359644026,-0.4459568616085968,0.32645106058508316,0.3945192037292788,-0.047002465818207176,0.09667418204820986,-0.29399716290063127,0.005608159592499376,0.27743862916895046,0.36516105131102133,-0.5967651450477428,0.04718280268311403,0.5581926580007718,0.1687993865829037,0.005372547338590318,-0.30070486356064074,-0.03753270556934698,-0.28034935982267895,0.4120045133780985,0.16418371596842563,-0.10921373178217136,-0.08328216958849675,-0.24255744254092854,0.02410450892295916,0.16730254364509453,-0.2178848658317298,0.03805318732559484,0.010875495200703836,0.08994294193489999,-0.14975449690326376,0.08242258270918006,-0.16654328780326158,0.36383451854114635,0.086154227136798,-0.153268480647404,-0.11520919389690755,0.052276651260047664,-0.10748127634733659,0.03363841779808738,0.010168386639652884,0.029238876496394627,-0.04235753521303695],[-28.309575397568356,-1.251938262759602,11.364302456594958,2.4257796549656248,-1.0641193871157935,-0.006519723497663805,-1.5281412939970154,-1.0723522149688183,-0.9243085019841533,0.31124119104241327,-0.042397262415420245,-0.05134995976418804,-0.25591546434507195,-0.26469641440076624,-0.7304767716271662,0.11009750586293622,-0.382453463144484,0.17568869298416437,-0.538916701698685,-0.5054596010934922,-0.39805933471760463,0.22389405098399937,0.30373309102923485,-0.14708473218351067,0.17448745445286695,0.16147953126849907,-0.20320814960051814,-0.3465439296789681,-0.06732861239399283,0.037794101733949495,0.03149593230227065,-0.10053021929907972,0.06325674799820222,0.13508301906546066,0.17572170522922262,0.36736608935625215,-0.11387598264396076,-0.11570235497585053,0.07650562147827883,0.0678350662812456,-0.324729430759908,-0.026205211105176422,0.14863968865725694,0.062270692285263156,-0.19760454279664297,-0.012389728584027067,0.10442141976456724,-0.16498915661803942,-0.03866690955754909],[3.553802106740726,-1.4135721155692036,-0.7384548690781101,-3.6164558243965086,-2.390645979492117,-0.3327433087967254,-1.4006239466951522,-0.08772207185307783,-0.8285142854115665,-0.24042244242257751,-0.435119524036828,-0.6828509745657134,-0.7648491884549042,-0.3042026463402023,-0.9266856680711698,-0.2691260546581757,-0.45112447388539423,-0.3608127013205551,-0.44283090239502065,-0.13444765809900414,-0.3631340713225143,-0.5287574540951149,-0.4594342097275795,-0.3571326853772023,-0.44221000919200254,-0.19541572397765375,-0.0248582709059047,-0.30680301282834405,-0.013160698092239121,-0.061688194922511014,0.05354350754147102,-0.15347589913417647,-0.09621764783217422,-0.05548466479758245,-0.07163850772074618,-0.05884558711972399,0.004249090851465099,-0.16290340653940133,-0.08021999533373714,-0.01090020776309777,0.10686501078228483,0.06465829161232456,-0.012803360551866684,-0.020159391370976238,0.06658848476485553,-0.056928279327696625,0.07794627434058438,-0.002006537896671,-0.04957938953821838],[-3.2217974024474922,0.3036361529809377,-0.3309032468520177,0.3522939961194245,-0.39962029385933384,-0.04118660456785701,-0.19357668776871353,-0.07973624442912494,0.06379132146702148,-0.2902137631056485,0.20372692280284516,-0.03412554716161239,-0.07600920247253347,-0.1433732845635541,0.0679920749922438,0.08533696057458838,-0.22775065982206485,0.300698475742068,0.05057556998982981,0.006962942818469034,-0.3635290459833883,0.03994929526419662,-0.13422770619373534,-0.04308772153938777,-0.07630670554541631,-0.127188903193939,-0.04142144685882861,0.0536907926502816,-0.072503363299221,-0.026704793837918638,0.008837563913869544,0.05696148513889846,-0.05575007499594051,-0.08508362934134159,-0.11011507679434465,0.07940756576569287,-0.025910300704372252,-0.015009632682816656,-0.04492891787755113,-0.03385323312826739,-0.05553097239384984,-0.06184863853126316,-0.007226933137605564,-0.021089199177761513,0.040021674691223114,-0.0007255795789489042,0.03499285416896464,-0.050830326368396246,-0.05656268582097139],[-1.3710058775563647,-0.6362057336438082,-0.12445607959834744,-0.20981859115198503,0.33798115111138194,-0.7775118320360456,0.2309975648146961,-0.18874461634657946,-0.2422596768653701,-0.4417057313550977,-0.2774021193504664,-0.6217687989528555,-0.48604818560764346,-0.07896050159160768,-0.1869128091090844,-0.36955679483745757,0.11740484470356785,-0.25534423531145006,0.1484558905814331,-0.2860433028367244,0.2921761185285956,0.05609046814702734,0.20838318479218268,-0.38021698832909906,0.29582864026160977,-0.053034843009272734,0.41056193992280704,-0.29626514556937034,0.15021308984653056,0.12617894015187514,0.27685029785985443,0.004180217139592129,0.12650826833397116,-0.0049718096686789314,0.35928523235533555,0.05120471041508424,0.1916191935188415,-0.16606417432251155,0.16172013063768279,-0.027612806582127912,0.1168757297448842,0.0973935420141087,-0.007863347921738033,-0.0535743795188291,0.15527011603109894,-0.06792084993039829,0.10906395028088844,0.0372824260987793,0.05877413819507741],[-16.553978917259684,1.5571416501590947,4.099584524004402,-0.6188133134694248,-2.297006864382943,-0.2464967337342145,-1.9947325073090605,0.8616946500481985,-1.2194755249768832,-0.04119126508336109,-1.3451246446678975,0.33177275007742396,-1.8456974907094081,0.3154424001345377,-1.182025241134378,0.32515248773894806,-1.4317631924744414,0.03568420494992408,-1.0162711857844011,0.2625652166326531,-0.7339948298372453,0.04488455469100231,-0.5086649072261948,0.17498875871515865,-0.7021483831295946,0.26126673534086114,-0.31298909570800776,0.12011127623259801,-0.5457859540201185,0.1474215463283403,-0.21852547991389834,-0.18649700326496035,-0.35007713583899863,0.07250033385946578,-0.423339888639879,0.020937958423946523,-0.038338205817854756,0.1268594361529435,-0.00758775744080046,0.05823861146986803,0.011682252711600097,0.1655516426337916,0.07556748016889496,0.06749190170362383,0.005777967309538313,0.08044191446697423,0.07537905043694894,0.06752796739229283,0.13268526452413165],[7.033316394828893,-4.765964407171743,-4.047161204246289,0.9426487467028061,-1.2089489367988009,-1.0949375766269425,-0.6103046810652577,-1.7354050979944573,0.06441548052601428,-0.7822826809457613,0.2396877500647028,-0.05717209867262961,0.38752081660293475,-0.018935378658595823,0.8808316300925081,0.0428966157606179,1.0375271719773138,0.22479178817841983,0.8080127278800685,0.5120972551225919,1.353194317761218,0.7309173607441245,1.213709348023845,0.36323081008299357,1.0978975070153447,0.5912362867894133,0.7947201390081379,0.32251289674618333,0.7695972957405108,0.537870058473766,0.8284734363795689,0.4454400241570265,0.8186544922530751,0.4306371500444817,0.4377399885665407,0.5541850573727095,0.4738926663440739,0.3763225130571256,0.34988985789004096,0.3166082579537512,0.20006904500091477,0.14621389929219478,0.024052834821768024,0.1386414059717735,-0.0626052309360822,0.16318858026529431,-0.31007031520491607,-0.00844747773656281,-0.2829144812763537],[-10.528654886286958,1.1499227613683964,1.436651071511221,0.3420364284838179,0.06813325339532933,-0.22357729747518842,-0.3728171238458142,0.6678579232827155,0.3685817210543012,0.037061331798551096,-0.04542994199869061,0.36102972509489184,0.06003432748301824,0.3909051889010755,0.11354323414913282,-0.0852764358844549,0.015229015194941836,0.3992355414107671,-0.12614029021473455,-0.03626006962568267,0.1360485838763953,0.26376336870470546,0.26177278621214856,0.27654016552780136,0.22302615798942235,0.19294797451304577,-0.04551131060385756,-0.14830259852799962,-0.06847852794309207,-0.09274752777619477,-0.09147769008985886,0.030547784637783064,0.0017663237976877477,0.25198996168080395,0.04604163184986389,-0.009441115214064879,0.0550909079836398,0.31717408671378317,-0.10759789195109333,-0.03211185796409617,-0.03216139088530463,-0.0620302628888124,-0.054005312738253174,0.1546032663166615,0.010926458403302653,0.08237269658419083,-0.06909345690473205,0.06039165911268452,-0.013208840831927141],[-2.7000600886763775,-0.8947973966261847,0.19919442997263712,1.7675654604658328,2.604120996500125,0.40211607872919375,-0.5651402704046756,0.2009665865067138,0.31257157253901113,0.4815376822764649,-0.2334209859580612,0.13563643314074045,0.15716063678871312,0.29545267200153646,-0.2748842252213516,-0.26010726703624215,-0.14582301470014675,-0.020413598807060552,0.4067681962113572,0.13645618431925954,-0.2200836616495006,0.1362322634400696,0.14119547140199418,-0.3517434642670587,-0.18046689448984293,0.1444271870217518,-0.5047013501033377,-0.23912673702842452,0.17479210226261277,-0.09498531326084886,-0.21084007065681312,0.051838913037444465,-0.03633837586902184,0.13509374043676986,0.1904281362508874,0.3450718080723204,0.06040614218677532,-0.07237286441443858,-0.1207388918237491,-0.04203232086767187,-0.06743481297508926,-0.15918622154828138,0.10121527194227016,-0.012683481416523939,0.035457276340163255,-0.09756202724899893,-0.013236734899886587,0.1207199073849305,0.07159086331686354],[-5.438844963866804,0.8455101098889947,1.8524815954589609,-1.27641497913983,1.431747849942982,-0.32738313184361845,1.8695741073923022,0.796246927314561,2.2632498882942906,0.22793844243653166,1.1789208337665293,-0.9646456690990831,0.8460257995349197,-0.08836232424525689,1.045159430211152,0.40806027131187583,0.8584140463575208,-0.08024091555000341,1.623139462855221,0.0076498883178109526,0.9118397553855664,-0.21352952292456026,0.9896317001558432,0.3159224424200715,0.9865315404669086,-0.03068052849432913,0.8568567602764244,-0.11564944760758128,0.1915731817394653,0.0694677391370338,0.40708215675976483,0.5560621662918556,0.4763701690125337,0.14971320459473986,0.6424317364559672,0.180165726862761,0.5018465122181873,0.27592252695467306,0.33744311339513594,0.07360235701908507,0.4272554295113032,0.08569686076749748,0.16030758213118443,0.2143867359625189,0.05417840324308243,0.10146578086784816,0.2078357314994391,0.3278010848640119,-0.04010437387994744],[-0.45833192346484203,-0.07179694397151248,0.8368161298346515,0.8555884213677405,-1.4319421540135233,-0.22750610648932754,-0.6339014206897441,-0.306684032793723,-0.23404264575283268,-0.06871712055881339,-0.9867471122207165,0.10139216480397285,-0.4215484616052593,0.13784965867281687,-0.2750529214084194,0.2621368415610182,-0.1500106251488088,-0.004694768523457749,-0.20509691182132778,0.11194451137142855,0.0019801595278031176,-0.15361471051460773,-0.04876446384352042,0.06745611785911289,-0.10671655912345818,-0.0619067117634931,0.06608963794748204,0.0009601883204298283,0.045964033789709056,0.024463372742973007,0.08187315518540952,0.05380077913791933,-0.13396043515160558,0.033691405717877834,0.11140594837465824,-0.10844834084350634,0.0615759550244141,0.03248237208950722,-0.0003328919752134873,0.043582560031509704,0.0041091431280992925,0.03730138164157123,0.016817164812018726,0.07960295045689901,-0.02888805921939787,0.03220398787411874,-0.1390873531582582,0.08647501599750335,-0.042923699697506125],[-8.447153757723282,-0.9902249694611466,3.6162563009856226,-0.6712677122092509,-0.38837109913366097,-0.46768302791001093,0.24699011374114072,0.04855246697090588,0.5109289209627387,0.043616321308541706,-0.8939068403947513,-0.12124271417449346,0.1975965026531063,0.3565257485173819,-0.024834101184103643,0.12901001718625046,0.04158438913528926,0.5306938728192475,0.1743988339953806,-0.19688688138349977,-0.2010841647026962,-0.09576824403065613,-0.30784398570518023,0.3512279755941742,-0.25112415913117736,-0.3035309310828902,-0.0680154113223321,0.12242517729330335,-0.24843054711212964,-0.0665333937031065,-0.13023329102848358,-0.03306171653986969,0.03147291070119229,0.07051662907254247,-0.22520895536780822,-0.08489982757122498,-0.21789109881466534,0.045834036191864144,-0.02387469611530238,0.013326587186366944,0.22087281937895947,-0.022963978258349893,0.06301436106457368,0.1920220366145169,-0.1008288322847733,-0.07586645207747494,-0.02956040401953207,0.10928882337990059,-0.1540965114491655],[5.360968326615499,0.813396447474591,-3.3516025653865618,1.7675362103450196,-1.2446916105179173,0.2603592456565015,-2.2697558289761424,0.018974765403754916,-0.523185741783734,-0.09395208597909616,-1.1927004371327368,0.30367187758118247,-0.10699627577442285,0.0768236000122872,-0.3967097051373314,-0.12447939981458059,-0.44094377114492234,-0.1559971200247254,-0.22050153677648043,-0.41657031593372823,-0.1986653911435132,-0.09267819738944949,0.05938595094723919,0.09321360335488252,-0.21616306735960905,-0.02005278984266787,0.25405053836747876,-0.14940885157649103,0.009464649212110002,-0.21360241652444387,0.23697910504752837,-0.05681436379255473,-0.012374048326132432,-0.06027232576649884,0.06364880770717667,0.031272789033144524,0.08804033946215908,-0.24206151724445724,0.3161147434943274,-0.042048201889871725,0.03619912770360819,-0.008573290186912599,0.09283742934674347,0.029950401793914838,0.0807011338751309,0.014090097216319328,-0.02451760116650432,-0.023458343115043172,0.10616123418250951],[-37.887024659120584,4.085564698335085,14.104172475777997,0.28601479714040173,2.435813646816891,1.8516554544100436,3.7877358942319264,-0.22608276947270384,0.5444739984936917,0.4506092763294962,0.8902220151994711,0.3845997810057588,-0.17432736538819696,0.09677287487620083,0.45563780377617774,0.0740623874520714,0.6792112169293073,0.4432680886331939,1.154612278059827,0.5336078136355733,0.5824945886410385,0.4378114462390273,0.5779513603152383,0.2252642992161019,0.5630155213185294,0.10412054215411326,0.9353682901203615,0.06496651557389711,0.7010437231658083,0.33490391357853344,0.6017396073923206,0.33861264051574963,0.6051947295398786,0.2839623372488465,0.5598350556560022,0.45505407991379965,0.45541050166181374,0.2635358793537785,0.5025288048409785,0.41078683197112054,0.38526161005062126,0.09850844493017383,0.5662284970595595,0.20287037784970707,0.4104395105485444,0.15578003309278082,0.5611653032251956,0.1257481231007609,0.6122972370040412],[9.42952051310606,1.1106827392220038,-3.3420031327015507,-1.9990370460722353,-3.0112295233635473,0.5426960363593769,-2.8748417017486423,0.43288157637644326,-1.8628345540385052,-0.8725014070017293,-1.6083900528571693,-0.5709305249190046,-1.3822671856698254,-0.560192426602628,-2.017594920913482,0.29728109552676346,-1.2753419491942812,0.020492145991144333,-1.4764611098253595,-0.5470328426381657,-1.6290150681596396,-0.6923222830626732,-1.446841556004793,-0.514433930598353,-1.1580304425856989,-0.4418567980880555,-1.1728961141996168,-0.17696981745276105,-0.7754119843843111,-0.6286850160692427,-0.9721180881750878,-0.09625630638429925,-1.2234105043770227,-0.6086009782519958,-0.8037189110654153,-0.5416354541546793,-0.9066448185411772,-0.46884990307519203,-0.3354701762995202,-0.44517692633036565,-0.5460816252681852,-0.4566087835649054,-0.6431938504479203,-0.35794963103879435,-0.4196969493603459,-0.40596670978500904,-0.45826760356923246,-0.2689936830844799,-0.40629448938247936],[3.235049728249444,0.2297966499083753,-1.720932150665803,-0.03742372779724784,-1.5580121239791707,-0.6693826141269482,-0.4704291324359528,0.20901392935130925,-1.0415567789461055,0.03268821866636116,-0.4259502635785286,-0.4626830121823054,-0.8369458298573337,0.3463133211562197,-0.0536326734551003,-0.38064814188711527,-0.5322306314688616,-0.3520848914722887,0.18165848574648147,-0.4701446462901581,0.03409260115454649,-0.16176941092877514,-0.002207850186926179,-0.40092867756331924,-0.022169623509772408,-0.034286850731234234,-0.3475190820162715,0.008849069184957835,0.08512389330140521,0.22205573116659547,0.034036522792255254,0.01486483758376463,0.29347524559305493,-0.03055529084110032,0.2569433121349181,0.012248146581406636,-0.10216576837298588,-0.15485463655425286,-0.019356727042335212,0.01870624552746153,0.003465473675525112,0.07764795737400339,0.038779129820073505,-0.14905532359916082,0.13764362622292853,0.1290138795345341,0.20116672430236907,-0.04134238833839916,0.16860686350944598],[-2.9865491293331057,3.162940863531925,1.24962120409189,-2.4512913647837626,-0.48708943736513466,0.06450697630935256,-0.5771103459725317,0.6093928272097566,0.005997855349354156,0.11301208763262413,0.5562708497152041,-0.3334311968450824,0.36935519375604053,0.008945316833716665,-0.15615556808690353,-0.0026255863719698105,0.17124764346564925,0.06780130041537043,-0.01958767253974499,-0.04824367079900267,0.2437852027995062,0.1175962531310567,0.016914128789160134,-0.14799856833267794,0.06021306096046291,0.1163486933388422,-0.18400518011809627,-0.10716169783139885,-0.03206974430283014,-0.1540179875297993,0.03360486274788641,-0.13052044775415064,-0.04346440290895136,0.11141216654678238,0.00437659301763151,0.10061893465225234,-0.13213699020585118,-0.05345296766605958,0.00951425110617938,0.014607522639056134,0.075364436942313,-0.08244775358111239,0.009546289943337042,-0.08034058848411281,0.07492879768839461,0.05071606608971767,-0.07066688874736873,-0.0056490623855426,0.037632575398586096],[1.173006436235361,-0.15435503438682038,-0.2342382232653788,0.15867124619131753,0.5827769609505216,-0.054348148197162896,0.7360680267518019,-0.585772476493842,-0.19255928966015698,0.029008754867715505,-0.3270340219251559,-0.21301159468372016,0.20127304203487134,-0.07709943732814308,0.33045466062628914,-0.0486152135853948,-0.3335328601246759,-0.2735487410797557,-0.20152324592139764,0.27218470037528686,0.049980354623008254,-0.18551004579497204,-0.5482357387432304,-0.10678281062709159,-0.25260995023902216,0.06167177106567634,-0.5623035004906878,-0.18300298561718673,-0.2871536368400408,0.11789641588627152,-0.29757668949265037,-0.04335261655904887,0.027102172175035036,-0.10192823719439409,-0.14623129768476845,-0.06454713515723243,-0.30300201549320754,0.02858810603885943,-0.22345558586786549,0.051717018105721074,-0.24419730437703763,0.047431997719645756,-0.2569196815835943,0.020894114502523767,-0.1580816021305331,-0.008982431851207237,-0.2760275088494471,-0.009844766239070732,-0.20576838669645245],[19.572569568652963,-0.3384476756091446,-2.2911557230064137,0.3266126533763703,1.370564052577758,-0.29972191791985353,1.393938134982615,-0.4073272770262856,1.339270024915032,-0.6318973750551766,1.2331526759499418,0.4873502542408285,0.7211443397546292,0.22223214109900952,1.409549441335462,0.15632261030767902,0.709039112990843,0.013164148764061407,0.49175726635021055,-0.016969772045014073,0.7023885171884835,0.11584702858941928,-0.2380829843370906,0.27574247363691184,0.06981352105326588,0.17621151634612586,0.10436030122760592,0.24741398188457386,-0.019915789446847448,0.36395962390607445,-0.16111777899662302,0.42086248198933357,-0.3155269371082553,0.40063423210718174,-0.3694166474583688,0.15370034989617054,-0.5074222406017042,0.47998649361482815,-0.16504148659540294,0.17713188191180942,-0.2667175552583126,0.09824870222065385,-0.3684830375621605,-0.021251625057867093,-0.3094038948752043,0.1424309598670828,-0.3868864854060579,0.1630521897821212,-0.15805066317876387],[-4.398302167364629,4.046352477307456,2.3177196653049967,-0.374489876049492,-0.13223139144370136,0.05129145284928401,0.4689765575142866,-0.1785007019916503,0.8632357477809911,0.3464082210632475,0.3283557754046463,0.5073043064584399,0.3626057100152912,-0.19011427139407194,0.28301396144744334,-0.06781258600775189,0.28244161240051047,0.1634601632374633,0.18623375875567755,0.16852087049949246,0.08777059582020777,-0.3801699352374854,0.10820171387576429,-0.00646013865507983,0.022039300544546298,0.17149180764350228,0.30893824054746033,0.05297507804918041,0.021022897225409252,-0.38375521203668994,0.019434983965640935,0.11995262045841071,-0.005642855273698222,-0.11623452788988262,0.41251014211388526,0.10983199341283356,-0.04974541051735268,0.09079573835403781,-0.13395642913338673,0.009746493869887069,0.0479663163270382,-0.045092409263329504,0.08342939416946397,-0.11669205443203423,0.01029993931018501,-0.18485030840527883,-0.11051312352986585,-0.10119528476128012,-0.13536014860801007],[-22.698444509829592,0.3338399843868223,5.221932112195009,-1.232765907915661,0.03638734220034646,-0.48027436824363956,0.27136843396228755,-0.03396492080148642,0.25269030338957016,-0.3283224169833128,-0.11487998448381267,-0.3313618901316167,0.314424644183746,-0.05788928949121175,-0.00836368219018353,-0.008678049389367276,0.0811220263596874,-0.3349929428422598,-0.5676614792009244,-0.13466606127808892,-0.005863987686864563,-0.12440863244878277,-0.1387421452920207,0.12894229144254915,-0.4901837358346962,0.13186390127367967,0.20893180594395813,-0.22536233182143628,-0.18320361152120918,-0.2748335919605826,-0.0996807508495088,-0.1252851785397,-0.08910005883420873,-0.16363890198876627,0.0636347410646581,0.12452243302900214,0.04138687735058925,0.11275062606281282,-0.12108343733042395,-0.101586419600853,-0.11735362498132748,0.1592874077596467,-0.17826576525711207,0.04441589159328859,-0.16031699958422763,9.75946603902332e-05,0.002133740888649764,-0.05370457599753912,-0.026430798793363585],[-6.25406880813263,2.8105027711135975,1.177421283335386,2.5023241290688008,1.8476426029019357,0.6563047180671223,1.182919809874532,0.005399070944234487,0.06711430655335976,0.6093641219260683,-0.03342773912621479,0.04758116948793345,-0.06103706633105023,0.020534427141069116,0.0710879911751931,0.1320960756955001,-0.0505936016886374,0.12303864411128577,-0.6517398339926709,0.3011052282567507,-0.5650060635487086,-0.26488756112077855,-0.188347821659752,0.09684876497868292,-0.36659197983368463,0.10028287184948122,-0.6397232496054276,-0.05714594741854438,-0.4132046430484408,-0.014981053729892224,-0.5375515632787478,0.03344221475994875,-0.7683356619310288,-0.16591067297435952,-0.6058427632176668,-0.02032742231484704,-0.6500316108128106,0.220280498890202,-0.6282367388803219,0.06329310516388299,-0.6538459905093311,0.11574553453925411,-0.9810912503741736,0.16892951006381696,-0.6854432856596805,-0.2027294697820205,-0.47976798637608176,-0.05724252464192115,-0.6105670795980677],[10.189445921947597,1.9038895011652486,-2.734079004161908,-0.19595029694966265,-0.7470475085659258,0.7409614686483451,-0.9439763878099517,0.15001571042212028,0.05084946982817834,0.1299922728381954,0.08484775713820651,0.2991580625996801,-0.5498480769768327,0.019959142676561247,0.11101696878235548,0.43797537890232136,-0.19179196284026837,0.26747809424347513,-0.5533463552799722,0.16928364337897964,-0.08509894270324389,0.2903105951617149,-0.20320719578810859,0.35551896206822703,-0.5182091399750127,0.3013758236090901,-0.17557229255338253,0.13547623511847673,-0.36067335253636357,0.28106386383398607,-0.4953158144120055,0.31874731494390995,-0.5160185090140323,0.19881222242603808,-0.6999383853314501,0.21968003303828396,-0.7911152265987321,0.2930817153632867,-0.38235926673547527,0.25245202355841634,-0.32469166480920203,0.07128363562364921,-0.1600623722887884,0.04883091094340144,-0.29748420663283803,0.09709510238990976,-0.10980954259171428,0.08039004435662565,0.025152082425847988],[1.9544302592053997,-3.406487038194865,-5.253914988441579,-0.10620257443840557,1.1933792643770083,-1.1101092613964243,0.4548852771435566,0.44910331379314566,0.32930656591849783,0.07398732611766748,0.12626849935986165,0.3272551619268438,0.2251826086677073,-0.16324752932660622,0.2864281648418397,0.05345178280102336,0.14621548109440102,-0.004817538606246489,0.07647526032390246,-0.06508221330567833,-0.12406648379903493,0.1350889629327214,0.014549533154106498,0.2291846122741221,-0.00929884993369678,-0.43888764813302295,0.04381226294067486,-0.17548463870443196,-0.1505591472737112,0.012091257175068053,-0.06936115182755857,0.11082307967392811,0.04395186248828826,-0.10706510911394763,-0.10669478814484737,-0.1626718895673182,-0.26701889766145176,-0.30920202828215887,-0.3094402812250715,0.06218334476282201,0.01720851097616743,-0.11521968398918492,0.04387324869350625,0.13496709377829538,-0.008775461908035699,-0.06742606979159523,-0.087557880087369,0.06413942108970283,0.012771731749280974],[-17.423265453577198,0.6006730767420835,3.7913429096724602,-0.15617263348950658,1.33591360265845,0.673196850612651,0.5605575191156575,-0.23465451999404183,0.36499375318994276,0.14132457662297998,0.22031997979706214,0.05894398037614198,0.3053069570302244,0.38490477220461544,0.41831590987680084,-0.013128248266340962,0.2491794467679972,-0.4008780949054219,-0.1643722039839599,-0.007065452494254785,-0.058580164920323634,-0.2586894876771247,0.11327194844709018,0.05752589119255264,0.3746116632131705,-0.14766257425718482,-0.14599280155631456,-0.14249713663514907,0.07489693655800915,0.10144204598595087,-0.16072835144345013,0.0779378090114923,-0.08859006745096448,0.09759583542892214,-0.2705315983860392,-0.07644444399405867,0.12911607059518254,0.06608359115612736,0.10671609691658784,0.1293629566638951,0.11358414172151275,0.15319551477871926,0.054205047888442826,0.004924070044681935,-0.0167408489458886,-0.0038733034743258215,0.04941590821921863,-0.02853251222778545,0.011309266599192984],[-2.587511747640844,-0.32586137433375967,0.8982695439295784,2.1158331212817028,0.5874650769066153,1.0810572998901833,-0.0747630232835852,0.0521737045986158,0.5205832229500392,-0.45753485869933036,0.32244123874664427,0.4474235520647682,0.250801733922721,0.4298415888844715,0.6496931210097708,0.36425591585055844,0.17930842209519332,-0.06956811552823103,0.111670386839647,-0.027719353790832207,-0.013058181921665952,0.16035973875753481,0.432579609062958,0.019433268601667428,0.22954333739755814,0.6124449962888237,-0.021985103459337997,0.023267203257306242,0.23875936725499658,0.04016380615514086,0.16870110321830753,-0.15406794693893547,-0.19147018413385972,-0.006668928364412152,0.24365156471320362,-0.03145236418909268,0.3308349666448724,-0.04854220835929272,0.09723579408710986,-0.043132462650089645,0.13723775400312552,0.10645984091694136,-0.09930154075790537,-0.04004214036804321,0.01859689269345463,0.2106389088659781,0.26206315717166984,0.15974291093918716,0.4345842760438145],[21.164572470961282,-1.0677611612641003,-3.5688143696242878,0.8152794710268588,0.5573480128233196,-0.27751418467288325,1.2665938725325145,-0.03628698708227593,1.4086646854620946,0.3363552893125455,1.2303482719535543,0.5001947365262249,1.0498309624211197,0.40952017883473796,0.8032612837231805,0.2679442820267317,0.5393633098545491,0.42723542566185546,0.0844205162054937,0.17015600422352017,0.7965728887497322,0.14766411139108004,0.5616075909637512,0.18324499209196476,0.3512367360800987,0.15392998178369058,0.12526969307842076,-0.025750272530889465,0.07879672773093598,-0.13737831155470148,0.38429678603183487,-0.16496187827211625,0.4976974641525685,-0.0034375408774177736,0.055725761543006534,0.10484325798339368,0.4413792677767708,-0.0799884162298261,-0.002032945806694328,0.0005383055478468751,0.08207411000659867,0.29498457098452074,-0.02133975843956379,-0.03574753529561099,0.03487414319793124,-0.011356319875268873,-0.051866218967368385,-0.1464964978454978,0.004538683334469244],[-0.8430244903562105,0.7071675748373462,-0.16179620986336737,0.46899446437504744,0.7288088368243227,0.448117750984434,0.9411713531786698,-0.04479768521873759,0.599384910138772,-0.13699633627295751,0.7501220157121314,-0.4149032054489404,0.6800621439134755,0.01928500974399353,0.2209439900080603,-0.14653622975372052,0.16878694900988014,0.10183466331594174,0.3145291336344048,-0.34031044678202976,0.6592393997265629,-0.02506158078938416,-0.008716958634249183,-0.008195068374172,0.566268519940514,-0.012236700341744986,0.5859027710651604,0.031598543645893,0.6267551062041724,0.08101681355395408,0.5879821431446578,0.0524675295203566,0.30527673430219443,-0.05854345199917746,0.325264312257636,-0.002816157160257445,0.4360274228245819,-0.25007677899781555,0.05493006412762105,-0.06643164850927546,0.36472416370796845,0.08135121728298873,0.1679266178472304,-0.04116185910749247,0.19841160813558767,-0.05898533172720671,0.12574321135450844,-0.09888367034696456,0.06456040667720948],[-9.91326065265129,-1.2142437681583802,3.4922018178897245,3.06395288914925,1.0980166137139658,0.9224483438750516,1.1444140425893212,0.3023754362503232,-1.179281652949192,0.12895089998734585,-0.16438135482272645,0.10888527089518214,0.7196800120665168,0.5320641427781966,-0.0507014296460467,0.6603676396105596,0.5014317598773577,-0.2757256441002825,-0.13683729153480959,0.25675872802542377,-0.01815430822903357,-0.3238366708037096,-0.5964005921718654,-0.17344503208596448,-0.2664181652658378,0.42865893627873575,-0.27623083132546433,-0.006419931600803322,0.13000452351691916,-0.08983319269941539,-0.07846487283060014,0.04138069330294651,0.029523008521384356,-0.006779729999211297,-0.016827704850205308,-0.18864362713494867,0.29565502204836086,-0.03414199290405467,-0.008151473889508326,0.012977474097529,-0.0019935757621086644,-0.011729404298752551,0.05638692733908406,-0.09811790751918414,-0.08538649704799064,0.021001699193895374,0.09593293215663198,-0.1218693035457997,-0.08436578306126753],[2.0994806825379553,-0.6890661883347498,1.786460063617242,0.7636890152212878,0.45158255095831057,0.12674493510430257,0.13016360310339456,0.12827617668994418,0.5379996556211376,0.08141450511271676,0.3932374884888638,0.060024393784563694,0.39573490246612514,0.2515128687604976,0.5153105157124136,0.043209752530552425,-0.32049209801531087,0.5391296276995191,0.5202815112403436,-0.032311937508628485,0.13940652795546585,-0.1356801316909956,0.17554344015178266,0.49894053723479964,-0.14930219679117815,0.33587956560011645,-0.13702713048951737,0.29631641875995063,0.0009967550385981956,0.34709289743479854,-0.02525159800893751,0.3262386600670611,-0.10870244231004775,0.23511499864214708,-0.037545380243521734,0.5273444190813141,-0.3859217661142565,0.23572950254814792,-0.30382551814358794,0.2308711739575321,-0.3318661441870324,0.11936532482013157,-0.24753752177887187,0.3306418943991259,-0.1469285236292496,0.11528852274261132,-0.18311816579173326,-0.07522457547791246,-0.2785416486102714],[-8.939317957449598,0.7090727365580065,1.9636744927426542,-0.3757458398646892,-1.762714148649516,0.12051551961586981,-1.3103863628529795,-0.13883559090240744,-1.895969268992121,0.3855809173554665,-1.2935851862625967,0.017204539637889513,-1.4906329503742957,-0.03849311839677231,-1.06022091293323,0.18114605483657503,-1.2025543950621453,-0.15016750810563478,-0.9560675734583652,0.2856804433578482,-1.24070240655073,0.31963314415329075,-0.7557483870851579,-0.21942204229992418,-1.1380478894760906,0.12736733317691754,-0.4934228364970187,0.04060535320289991,-0.786430544779037,0.09490652699646634,-0.7101780269929262,0.2168463230446394,-0.7554419147638494,-0.07050535644745148,-0.6099174734828787,0.11612291036924084,-0.7181093841508887,0.00815086854780805,-0.8016573975721835,0.016294230458476378,-0.5719716438657202,0.06465384507093443,-0.3073924972362855,-0.08822263103184566,-0.4678883827212436,-0.26095407538196225,-0.4607897834037685,0.048381079627364536,-0.5652700921839242],[-0.33678167006339604,-1.181401011943793,0.46569579092972563,1.064368948145242,-1.3398940470272087,-0.07781022093707839,-0.7166138506520718,-0.10000214874639227,-0.41175730126679366,-0.17086284087919787,0.004016135902547659,-0.09175575058569864,-0.5726455054330232,-0.24905594627861027,-0.44467782695753133,0.03371445201933379,-0.432285952609499,-0.07884071323973803,-0.3944874806027492,0.10338922585903558,-0.4574171570550041,0.2588975065221381,-0.29937039650995556,0.0544422599943954,-0.1589750844162961,-0.3883136775487704,-0.02198361937599885,0.168574397832278,-0.5530637829176808,-0.2157282819749326,-0.15321970504541743,-0.05924598710056804,-0.2576667182458552,0.012471652856520602,-0.24642219287371245,-0.08804046094602457,-0.29032596879385153,-0.026927433165476952,-0.08647918658599599,-0.024516828014168872,-0.41938824916065365,-0.17136029229217145,-0.24996854996451998,-0.16125897983935494,-0.22716051937376283,-0.03233614749519262,-0.17216825175629238,-0.0883478258503905,-0.08424876046610105],[-4.08122961961178,1.6679034439232283,-0.4506079571183605,0.18903748397395553,0.36121029997549897,0.03347977342582305,-0.6179538008102541,0.3088820361119948,0.6236978004586982,0.14361756654917254,-0.30737576402749855,0.3394825534383294,-0.06958324129073828,-0.022262652308131917,-0.29612042239708997,0.1969384574832499,0.043555095541651406,0.23126947107630627,-0.035151980234255434,0.07960264881266602,-0.20787479984572768,-0.1304636341823386,-0.3366629972134609,-0.014580534557867212,-0.37446666005897333,-0.15696016932239482,-0.07467254464555587,0.0013200182826177784,-0.14127541574895286,0.34642097025077534,-0.13774591267792374,-0.08153712588205375,-0.22993155489793704,0.024079170922869028,-0.07624784085933164,-0.13171301308655503,-0.11616434956091892,0.009503450684332653,-0.050536337930674594,-0.11749424163320925,-0.04520732749883053,-0.18530676364262186,-0.2718210965222654,0.10765368160316931,-0.018772980536733073,-0.05678436594239991,-0.0967617027923932,-0.01688289555219282,-0.0013948585178469592],[-3.4628596959618045,0.2501891538932853,-1.6885380719586918,-1.9414638181217587,0.20244886745577145,-0.2964437120644304,0.32754193246616264,0.7723110519098196,-0.9048042420634044,0.7085397877980817,-0.7541358082603781,-0.5650961373082187,-0.6246607212349679,-1.0263627830529516,-0.3923796795333001,-0.708002987570512,0.33746673254978216,-0.43230015496946134,-0.6973460433729128,0.1531720536874032,-0.3876595622549214,-0.15824177018536426,-0.2557028384153478,0.06647540546120893,0.09101214201088872,-0.33505556944012244,0.28930892676366937,-0.11614095306801564,-0.04832205932139045,-0.12075864284200433,-0.3317116295435422,0.1274706569283466,-0.1328297555927567,-0.16616264185451313,-0.017823702138386674,0.022055852247132513,-0.033615116860114695,0.08763254213192745,-0.14131491076440122,0.151348960114717,0.17172072456718654,-0.2596016999582026,-0.39807352291073567,0.12238293254722984,0.030475439877251174,-0.026548546916234782,0.028729330651981185,0.02450051184250881,0.14966315461389057],[-23.55683175101253,1.6793025275677864,1.5451668555232547,-0.39146784886769787,0.33929728273237114,-0.46750384438142484,-0.24040456118105583,0.24066870104962887,-0.9872924041878554,0.19170108208210326,-1.1511757775327363,0.009732916738745235,-1.5342286026195444,-0.053730029622915856,-1.1460414190470714,-0.3479231323537077,-1.0558532093626658,0.037039211187427666,-0.7446922390728739,-0.14080692471009332,-1.560586660991572,-0.12126391946623201,-0.9897825966035317,-0.20270847461701447,-1.0175678808926927,-0.2839257568565432,-0.7557164807798608,-0.3349279224315945,-1.067711729621763,-0.049868059663010784,-0.9028005785903951,-0.02905948569000752,-0.9117262803209653,-0.24235133641107423,-0.6090145772650651,-0.2673719650330669,-0.489730415025725,-0.17859271158638132,-0.505065899915382,-0.1881011282484256,-0.38401509066378314,0.0028100612547719652,-0.606092301869335,-0.1752974555647373,-0.5389403745942691,-0.1376906746899361,-0.39579144947056655,-0.07150336151206516,-0.17928477078652694],[5.37540127781673,1.9696307118503487,0.27332812357848313,-1.7774593011639337,0.23994968936248853,-0.5796388144524405,0.04011957455400602,-0.010826359414932443,-0.4155386822036913,0.24696925508142997,0.049123955796083814,-0.03125436222353092,0.3499795674496266,-0.1635145971213701,0.3578974826432435,-0.4452345853988429,0.19222361136774668,-0.04898146842713313,-0.15563460068805987,-0.29189219594766946,-0.18932710784697732,-0.02127886930336283,0.194415339654375,0.21252971302706675,0.15270752712809607,-0.08676460162465892,0.2010645886820429,0.032793910450731885,-0.016448060850021852,0.018802420809281007,0.1593822772694954,-0.029170549790847915,-0.029854190488352292,-0.00179077564994865,0.057273446199755676,-0.037181801255939916,0.1984771473351859,0.010402208441879158,0.07046978222428506,0.08849670563195579,-0.09992687096632773,0.10394077659104949,0.13649850716456272,0.005219522953861824,0.10065760087719318,-0.03203547368261097,0.3582853611407125,0.06447055623432128,0.10441356886801348],[26.141707606331284,-3.8965319219177115,-7.138381363062746,-1.3542925540512925,-1.6392104929956055,-0.7055086571358847,-0.5561175907363862,-0.981169359400633,-0.4375559642429614,-0.5588186526934161,-0.06179312688035895,-0.4075461805083646,-0.2528114379680286,-0.6275353070323517,0.19681659068037605,-0.6435668318548093,-0.07992012465770691,0.3717330251435804,0.3184407527749798,-0.1612763962364865,0.1431140561836213,-0.23948842196595735,-0.12482222407220253,-0.2169785499234412,0.013439700439197424,-0.034226281893169036,0.29490732321011337,0.5158179293685423,0.3542177908345192,-0.056935081705004845,0.17519745178474294,-0.0982199867114519,-0.014890252923327909,0.042266605356892084,0.28672520967824977,-0.028552463167806987,0.024187994490925437,0.06960245504264002,0.15353455465639876,-0.1720008048687284,0.10236397354463755,-0.014982743217814738,0.11751923102651116,0.006906197036910593,0.14688764535359275,0.08985949422181662,0.15790110071343924,0.007758104744199352,0.21422543961107654],[-11.279664603448532,-1.5108715340555277,3.494634602895444,-0.34830787053284473,-2.3398652359213012,0.4440078321767718,-1.7927910967709209,0.06821180072881104,-1.586971023830558,0.06714575950192701,-1.1647498801561522,-0.26729304808392096,-0.9613532332584694,0.015463139192089536,-1.187185949795229,0.01053463702058636,-0.8335685038192063,0.1924337196688469,-1.30362434139787,-0.31581378048228903,-0.8519931032091859,0.31199042765183355,-0.6422276294711734,-0.22127726135379014,-0.8514549483713781,-0.08776983490962029,-0.7386245956928988,0.0750664375529321,-0.8614104156176045,-0.17569065057560282,-0.6402407223741816,0.09544069082154374,-0.41043836684473917,0.016468387380325543,-0.4096022572770105,0.03036647056070323,-0.5200361840395287,0.18655085840217833,-0.5245578266196226,0.053449049903793774,-0.39168527902164785,-0.22073212867991376,-0.4196942561552851,-0.05118120502761395,-0.5740894068893547,0.15126564340004442,-0.4677104943744345,0.018834729515114546,-0.28039515514992025],[-5.861638887980279,-0.4625621391556254,-0.46938460532769,-0.3973325997259208,-0.6980969149619494,-0.5058633149698845,0.2930774347180574,-0.8469345709349144,0.5488877752755444,-0.7112215043624347,0.4577220637902769,-0.8803972235337776,1.0050762184826276,-1.025488479154853,1.4141926662285584,-0.8711165508843183,1.4144570980385367,-0.9214389032059808,1.5114428717023096,-0.9539643295587671,1.4868041289491645,-0.7938562483627595,1.500913424329834,-0.750287570048896,1.5511803695980315,-0.6945706091531967,1.673588494010422,-0.7339054438094231,1.3839667078633489,-0.5503085867177412,1.108085401076913,-0.5358170348444947,1.1680720145320416,-0.3421129534984647,1.049957862648107,-0.14135600019399278,0.9435352718311719,-0.1408340636221939,0.7607144854528785,-0.05974624624361697,0.738671636644471,0.0457578577034525,0.4570613272628291,0.10411777548809426,0.33063281804328387,0.03843940322647685,0.31892932558475634,0.20703490255611487,0.10584111721562499],[-16.940069842303053,1.2903894636576514,12.048662611887067,0.11832810412506825,-0.3677168172033757,0.06135324201723694,-0.648699723182639,-0.15099144538323733,-1.3758454089209837,-0.2616798628763259,-0.5132871475719475,-0.262926048311328,-0.666197404626394,-0.42281689680286205,-0.285276274777323,0.2792781211783858,-0.20690177518797226,-0.09138070469558236,-0.3105325037965951,0.15864920235570862,-0.27457005370118015,-0.14595098923532027,0.05112034081264035,-0.3342488996312775,-0.1652232464519694,0.08465603614340363,-0.09676131103543488,-0.016042841898052964,0.12318043624579328,-0.010001208144158665,-0.05728859108964446,-0.1113074177889873,-0.05735028270571215,-0.2000120068160673,0.17448806063257183,-0.060123055535542626,0.03629566118044807,0.23933177285677928,-0.013000048438849943,0.05010107885078515,0.2845503807472609,0.04940878118139871,0.21831274315345914,-0.15324517706320942,-0.17496481204460684,-0.06141470507974936,-0.2014622640972765,-0.133196950699042,-0.18581782367285268],[-9.527376151495316,0.4007043208336438,4.650770680606158,0.45128829265743897,-0.7891359263987983,0.8105983746654717,0.3570696142378983,0.2731359350345394,-0.3963343155201347,0.4892613636018106,0.3927095391811622,-0.1710390484425082,-0.27541322015460395,0.07704140741102179,-0.043650620762110515,-0.2844664830794909,0.21976271564716987,-0.22021016194188495,-0.07652184318892408,0.12026022899351739,-0.08636083231000512,-0.43786344169015307,-0.0527873084870229,0.09994801235271683,0.1607399712585379,-0.09022364800277904,-0.1871158204118735,0.005171582730922176,-0.193030895821284,0.21481338014320295,-0.09467528865120214,0.037556868770283476,-0.08729627255138722,-0.05437444754138897,-0.1731881180735949,-0.03978707541421791,-0.16491744077866177,0.041788141414375965,-0.20705235630423444,-0.030446722352718485,0.0746692509981128,-0.011542823833448094,-0.05539168687135911,0.07072339887199407,0.010600998570331161,-0.14749538181177255,-0.0722577888740617,0.11419274378081917,-0.17499562264199583],[26.742450029131156,-1.9884039128866802,-6.1330887090990505,-1.3788719625934762,-2.175589727048648,-1.0784936257740223,-1.2995959161255786,-0.19891548513402593,0.14978990350992552,0.4427973622704992,-0.1962458335970929,-0.872883835194015,0.22586931388502118,-0.48485988607190617,-0.10163425624794253,0.12424907234847292,0.28649532558405255,-0.22204699826991506,-0.06493337461804095,0.03647607023915991,-0.3735749197412235,-0.2284230430422704,-0.157231342928786,0.09178956799775372,-0.19064513618036713,-0.2494245905986689,-0.3447263941911588,0.12927071809969148,0.39116326032049603,-0.4614716504906178,-0.14004318380920053,-0.2469575509397008,-0.07534249377057935,-0.10937820777693646,-0.07608850887230902,-0.10529360459355454,-0.23293312424831114,-0.12632601994026763,-0.4345145500984228,-0.15818306626223383,0.007170856176910387,-0.1105356382638003,-0.23492760336713603,-0.28878393486507037,-0.18184296088643978,0.18635804924095792,-0.07854657972138081,-0.0995107495876466,0.08527498927701278],[-15.313629753932876,-4.954093635162778,7.475722465786075,5.063225963489923,0.7107634403028852,1.3467494300596285,0.8444075906220084,-0.22500030102190635,1.350978012774716,-0.5655347060678723,0.7753826876646546,-0.5087234365513349,0.7864351801481776,0.4644310905634954,0.2214582641412613,0.4463773468667416,-0.19190748908888103,0.10738311851312582,0.4712619632761608,0.3679161075687985,0.3581201578533056,-0.2026844817929736,0.44292769762116035,-0.16258612780311263,0.3751833876013658,0.1626095289334783,0.26015975194784646,0.2131376040141246,-0.14959804195041035,-0.12055229260970177,0.08469365724925994,0.26341470308455267,-0.022219641858621818,0.06180614608227683,0.029145504833898737,0.1650307050725305,0.08243408857565189,0.4303204703386993,0.3355151171393833,0.21370375186486507,0.21380747301792993,-0.022773827863526004,0.17441775342547955,-0.2719914127969278,0.21321394703428456,-0.0434553042906031,-0.09630209495913088,-0.05902272855362981,0.2693421730649536],[-5.678042848561101,-0.46221443907998383,3.640315439054955,0.5469221672628067,-0.2802372673422005,-0.15415118256211321,-0.33847341585611673,0.34807061354713037,0.23954880415154128,0.1411501179783935,0.421994535479469,0.1348921568211464,0.00038958766195814595,0.5562883019310348,0.6617809587987833,-0.2750382852891633,-0.04202214612361256,0.11646511545261831,0.32459175894708725,0.07193235570947022,0.5001297362756059,-0.10156152156917364,0.4285038315511223,-0.013133395205371936,-0.04038431059163944,0.2138356957766909,-0.16108157715412175,0.27994786510746994,-0.0648456282121602,-0.02688644279721046,-0.1799665818983166,0.12632278577902337,0.012388434255810047,-0.3872832174918392,-0.1527938813861212,0.10547720321550207,0.16758385895235664,0.01289625252682095,-0.05847184198130036,-0.23951113835799015,-0.18162574482105004,0.04744996755520625,-0.08945437611864435,0.0736675439353274,-0.07879217522974569,0.1311764407564202,-0.060393812544790086,0.08920686434258016,0.0038165918363641835],[-2.636851286790086,-2.8989068669076308,-3.020798017908007,0.14844235568903477,1.2519557353803394,-0.9265084735441873,1.2278478205629753,0.2600327168026559,0.8321348420309637,0.07956934402153065,-0.13625959217617703,0.47602053462980326,0.44616196215170717,0.5456040746284404,0.7848827572093646,0.6042623831170922,0.1980370673674827,0.156988651778436,0.08261387788507728,0.39645475000725916,-0.4310821105857242,0.22057909755761462,0.1711670260318879,0.20255578674394778,-0.028891109793248278,0.09246746033940931,-0.2701529424705523,-0.049675279825254895,-0.1362115311003837,0.2514902795110794,0.1699629240562536,0.030419814845680397,-0.07765358149553844,-0.14611001969431098,-0.1658154241926896,0.0275726114116634,-0.08339847929019076,-0.23135217397170652,-0.06795371665314538,0.015770418884353064,-0.08161257227435272,0.032837710139894186,0.03733951642516771,0.05548707729409836,0.09916076807806891,0.22740359653467224,0.09614106954356791,-0.0887980070165358,0.0038527875686538513],[8.378949943910984,-0.05973583461810215,-3.6411923373499775,1.5159809755891402,-2.010204505866154,-0.27409760903764974,0.19133961735231936,0.6789463502796821,0.4145161961937342,-0.16347849044859145,-0.18481523006715542,-0.03048587035901377,0.8735183340979483,-0.4765411886850462,0.17880517004743002,-0.32028492210209336,0.18867264249240523,0.2592549765623009,0.5747896597463882,-0.2079387179425924,0.5400344878137625,-0.3067234094381638,0.3826138945231818,-0.06691268441673373,0.6659603902545176,-0.18289077859506903,0.3788594302684376,-0.24254855959231902,0.5658030667322246,-0.08308671199587925,0.45151328511104216,-0.07710709232126486,0.12523977698311084,-0.11974763049546992,0.42720353191902316,-0.09950627969291821,0.5302832182337223,-0.12361580675947899,0.009171777136392336,-0.1159467118923474,0.1902366619708088,0.20434901393933924,0.14261347654772516,0.07608753304546208,0.2585339894756995,0.05443239307742174,0.03581484370411984,-0.15704244406735582,-0.018558805841688036],[-5.305785088289025,3.966117938740519,-1.0226997314075017,2.819723436957041,-0.6313100551797381,1.6056339531922355,-2.3197185130838553,-0.25999404095584305,-1.6127398773001624,0.7920842406746902,-1.4315902122174213,0.7562069958092791,-1.6800580496359239,0.31161105264521366,-1.3842820125466786,0.3559248408302881,-2.019244018617469,0.341115534459112,-1.8948699983636919,0.7119076398333221,-1.241421312058994,-0.15919335555971914,-1.28217929423821,-0.10004032360394328,-1.1759453704030678,-0.059783096069370566,-1.401534137395143,0.49259515442907503,-1.369476116038679,0.15656003536183324,-0.8087651709667122,0.06020418886877383,-1.250514901233494,0.09818217707162659,-0.8003096819787424,-0.05285659005692582,-0.6979331687248689,-0.1865424836877268,-0.36797179100301985,-0.11368650885766048,-0.914072386206239,0.19218778303781803,-0.5759485901898516,0.17239436085672374,-0.5761984587807951,0.04060559759351057,-0.4710782104078068,-0.14002367378687683,-0.3564215833657465],[19.864737027443166,-1.3649644904542824,-2.528489290065748,-0.8527462319497418,-2.162326316714868,-0.6075003171140019,-0.9197862450276602,0.017587612940674454,-0.5753640753793215,0.18368389090220147,-0.40376681553436206,0.319248839468707,-0.31084744081933496,-0.43784618070448417,-0.4288559378958933,0.36220848935776356,-0.29345169514571506,-0.03812979620511384,-0.13530301439169182,0.42678366391967765,0.011597515600909944,0.26645428028452117,-0.12489725030678717,-0.2687849997271217,-0.03985146609168831,-0.05946305456511576,-0.4148866397868778,-0.3488166198130144,-0.3555037387309322,0.08486434590703683,-0.41054193700425884,0.31550899461644455,0.245314735428088,-0.0645368400701963,-0.06173445671576268,0.004114069656170876,-0.31993906388902005,-0.14766000450804245,-0.2478857495599212,-0.0012208020488991022,0.09793941969067876,-0.13619959923131875,0.039194536839588345,-0.026215508115938948,-0.09152416837087546,-0.1579783361624381,-0.1480570530310866,0.06599718267127506,-0.05743521470897672],[14.397758737050056,-2.3806455927568595,-1.8633633982987101,0.442161364525543,-0.968840084819004,-0.1862577306167021,-0.6582926672813516,0.08917388547092839,0.3163436929864006,-0.04127818445456893,0.051762259000285825,0.7368548725467174,0.23923900681092594,-0.42709608525573023,-0.20966975948274497,0.15745209558708956,-0.1266046263225391,0.6575867290065801,-0.11187903068415626,0.011422642708379976,0.13157119821223873,0.04397582594754109,-0.023903373587213072,-0.3112961889910907,0.12648144316734974,-0.20647140007577686,0.010903726478307911,0.03580013121755355,0.05695775752027221,-0.1690209997923682,0.16119329614992944,0.050969351690604375,-0.12564115435347006,-0.09943156601732214,-0.08146295496168567,0.13950556704813413,0.009881786481209687,-0.1657042706640158,0.09494933749700489,0.03954403940159215,-0.03227709966192664,0.06829086982027441,-0.33018647662292017,0.2477590418932269,0.18117589441247936,0.3123557966536755,-0.1122236470307975,-0.02574146043668484,-0.01805537004605442],[-11.512342163777515,0.5070679121506136,1.3646370989493755,1.5038457723386238,-0.7109003295267744,0.3520382132401571,3.3621552907297176,1.3981221002331228,0.41404295893417997,0.9125095952722045,0.2982557255177424,0.9985402069641838,0.6693122853734141,-0.0350230918885041,0.5976976404674786,0.7013344228404318,1.039467729520944,0.47274128746339344,0.37776720086200566,0.5442395778561965,0.2818232827191867,0.43388777430270165,-0.13664568133820232,0.08961875983693462,-0.15375494449070592,-0.5565638111793053,0.38570518274528554,-0.022861900792122258,0.01918877427107987,0.22681050058553742,-0.10374921055635664,0.20762591094939178,0.321659042853315,0.1657733555923013,-0.05057879813684697,0.22123349259690128,0.08681041235331527,-0.07325954251580001,0.029877350395544698,-0.12763569264075103,-0.17992803768800247,0.16055102523447629,-0.29983754933391593,0.031303884732939886,-0.08992697180999205,-0.012888920289359488,-0.1382369162930869,0.07743354546054855,-0.16632597253767153],[9.283662938820209,-1.013319855571877,-0.6896407150235244,-0.0954478897132086,-1.0505314689883571,-0.05731052813711219,-0.9524040565577298,0.10789247022692094,0.044675184011720936,-0.6132101488697081,0.5748493571461658,-0.038635132359989964,0.12263556441046858,-0.13237963453475396,-0.11725402838672173,0.08089880151860362,0.14876144544275202,-0.013678228910261289,0.15924539480142733,-0.14330711133118193,0.2167500013564547,0.21138022818782032,0.04850728244505719,0.144218268558865,0.1010844279173306,0.021943133968331596,0.12492025722098049,0.008239248754106926,0.03859262746625251,0.09364007440656637,0.04839733292042345,0.18632610643956013,0.25763198531891246,0.054763662048415644,0.06955684622562952,0.09562469825497331,0.06860858833639054,-0.09299077227933111,0.20759452659193378,0.12316601771598648,0.17277117197836717,0.040475005652968375,0.03463096369922137,0.13345792773408918,0.002409363651908715,0.051935142375288,0.07008627327313674,-0.01763887077801762,0.19053451944238506],[-4.414758704738139,1.2596375818675367,1.2471899432059286,1.0667360620867146,0.04583611400934196,-0.11083761598175768,0.24780864135667802,0.30728445737453736,0.45380131971257176,0.02511891612431209,0.7748702200933286,0.8246034175322837,0.138427746646983,-0.38520826201631087,0.5833774198570099,0.34522044950288744,0.40481973413418443,0.24042372844153495,-0.17845524059786944,-0.18014510079652143,0.12926710617994217,-0.128141920888937,0.12480605363839396,0.30701097000293515,0.011502775738614612,0.022787288916381883,0.21849783934656394,-0.06553848288899722,-0.03415329514119494,0.05010105603111079,-0.09545089929683392,-0.08177984112499613,0.2285891563642636,-0.009774891273054429,0.13061812937942283,0.21076131386643426,0.11967736600601477,0.052230017396566755,-0.003409787066683694,-0.010053911539740177,-0.07163057481427368,0.06574891384786892,-0.07328731845371192,0.039000062541705446,-0.08903237867606148,0.1634957352780823,-0.09882847466436934,0.2333334709062526,0.07832132099123706],[-8.436805918603435,1.3407554279840301,16.58910747964074,-0.43875503015248024,-8.326080633693616,-1.627957873217064,-0.4765613702698531,0.8835383037566725,0.36333931678968784,0.5144819074249282,-0.18997945383613923,-0.3560031211197996,-0.4346119439198091,-0.6129125903580259,-0.5984388520548016,0.3224152417617871,0.035928871258964414,-0.06211450076924047,-0.5622564639363146,0.2668577500381983,0.10912374229009196,0.022701172224836322,0.15169804921102978,0.1270250387960585,0.0957466413552108,0.2516986421061763,0.16826562515063195,-0.15390092801919153,-0.014949780238963654,0.08493050434107452,0.13022555955320692,0.1492783752321744,-0.09081468732177815,-0.005453668780679714,-0.03481438565839894,-0.23281836962335542,0.09253105304092843,0.03379075492791993,0.18108293729025737,0.2392020117141888,0.0714029379290182,0.1392471266025893,-0.1340862131373506,0.017949651364652028,0.07911893688607348,0.10482003511470328,0.16870826928939323,0.08328185878491207,-0.024736858350759162],[4.077552394045801,-1.2259908284212724,1.885286835125882,-0.49416446828265415,0.7940692275351451,-0.1789923909748439,0.8202118406507063,-0.030371249780705087,0.6842958152854645,0.07940783071064653,0.8477986340497614,0.15826581941084308,0.683125367228006,0.201710610093678,0.38613352050689104,0.14013334615629672,0.4476583118178436,0.25546799954342425,0.05732062104880355,0.3713484811006618,0.048945292663185855,0.21035230013007714,-0.0448812110853439,0.16953740497294373,-0.020879029396094862,0.008937539700285466,-0.19423454361851278,0.15098763145883393,-0.18333466018704464,0.3269054381393229,-0.17576591983035297,0.22902842863291564,-0.2497442583310746,0.27404730965911944,-0.2595289955481015,0.306142791985565,-0.29097571327018373,0.21526747661357498,-0.2795898376236128,0.21003072068376005,-0.33322806118126347,0.2228007919713998,-0.4160650897138497,0.10043212554753433,-0.36061713362760317,0.07375096882599227,-0.302369321302341,0.0919277885087818,-0.372280870704147],[-11.46357787148352,-1.6982583587235998,2.2206574276933666,1.4017059229171027,1.0971455324754085,-0.5898812313379365,2.0190634118196344,-0.7957783952608897,1.5992514976985708,-0.02892252832187729,0.7213092628099306,0.4314879043835145,-0.40557456990755325,-0.35270996884447636,0.8823861318739946,-0.2619691397042789,0.28427411507488476,-0.44612028942153686,0.508629898446374,0.09665252186778789,0.9327219633682841,0.062402142644228255,0.07087802859976036,-0.049395309043130865,0.4994509493379614,0.42567835583549973,0.7924763832860289,0.18522605919272514,0.48156217512364596,0.42630001003391405,0.8268900328992326,0.4458701100641819,0.5834760443551548,0.1228340130882883,0.46332407849076795,0.14867456886135616,0.575839340836692,0.21191480391871764,0.6955719794757234,0.49069586728808395,0.797646079526516,0.23590323836801042,0.6167555246640969,0.22413978862219794,0.5624360476200844,0.4134665928307742,0.5802123083958747,0.34901530550975113,0.23961769306171232],[12.3634176727466,1.2891482029893393,-4.804142547260402,-0.8057478439498907,-0.12540171486398477,0.648567637230767,-1.5694639020379335,0.8432569443131194,-0.7521351321435288,1.119851712019965,0.1755585245867686,0.10538351790487603,-0.5648221614795832,0.3531292308769784,0.22013105180325937,0.04395692330267566,-0.9061008016935928,0.2450173923430316,-0.39846514811880984,0.48307032665654354,-0.32355390487014335,0.38862899432036385,-0.3603112544573905,0.44830555674774375,-0.32861689209302575,-0.1035636342234762,0.08485899144935412,-0.12331698673284558,-0.32802638872010176,-0.20392641481260063,-0.3573892155197682,0.23137518803553556,-0.08097073434099873,-0.1249196195415534,-0.06773455524930484,-0.06774445764412697,-0.1555819582202768,0.08961676386499262,-0.11050924769125421,-0.19000554880642526,0.17263930077831457,-0.177546897842124,-0.05308263321301851,-0.1303869948698383,0.06775121795032625,-0.04343696848214302,0.04057534497227639,-0.20388293621879985,-0.035118085360112815],[-4.899337856061423,-0.9954816330984382,-7.919783794239979,-0.28217791976999884,2.381949969550737,-0.8044198089670594,1.0588652726159802,0.8809778858839658,-0.35816719727037444,-0.049858123714386436,0.3350068613256648,-0.1352845008883871,-0.38577048433967714,-0.13978180595770467,-0.05098817918254009,-0.35173084679583044,0.12863366035812798,0.15251278125117207,-0.1308584301418229,0.005482384457196319,0.1105519300914017,0.1696399930279087,-0.21563807315452918,0.0408638621949754,-0.2374364359582925,-0.022964599778121226,0.027696149703297594,-0.08145044013898868,-0.001327452799356304,-0.3021213786173395,0.13842912937444876,0.03525939799909872,0.13795175097993673,0.07051564402744495,0.23255720898647655,-0.22193546880228804,-0.02286938998799101,0.05839146002800998,-0.15587307719013585,0.07141737451172991,0.2591987279142576,0.027915899019330184,0.14473721750699575,0.09365389979089163,0.016646731856061733,0.13140973749567794,0.08211611954838412,0.010561312009803822,0.17704577991969092],[-34.953614759929906,0.19170434246752527,4.680805328180417,-0.7306831178073968,4.5913084219859055,-0.1379305639667078,3.071040318207169,-0.16759917339020938,1.3635645360836628,-0.36109847949995555,0.4518869751821262,-0.43488633976381685,0.5912585569559865,-0.28457464574892427,0.8477482901579813,-0.5019330157040828,0.2971435019579644,-0.03107978554329601,0.1393577430561907,-0.49979160332032835,0.4811621781910434,0.028647751724042103,-0.0589992957332644,-0.10469314088758382,0.319034734031511,-0.36277928719037605,0.3155264846737329,-0.10973887221429926,0.042572573636217324,0.0599111411243218,-0.042674195493345654,-0.0577638764064701,0.16389278088898074,-0.23419717962471953,0.2663093578174472,-0.11644841867934301,0.14548209850183633,0.1381982136186005,0.23973235111208258,-0.06555205357733439,0.2888870559387815,0.07706892434760752,0.4432240506684388,-0.09128155586833954,0.2111511264674995,0.2378850986188208,-0.02319534888236213,-0.07054048360163911,0.2660250925532135],[0.270155430845942,2.07803915753609,-0.7122850555775064,1.3489574815552037,-0.6551866612445906,0.6448539633605638,-1.6655858812782554,0.8210543939397897,-0.33831210207772755,0.5704824913224086,0.08085994175070302,0.4751558514185785,-0.49642846688344644,0.5273103819523836,-0.08180414425696442,0.21210451475052905,-0.13868032360133445,0.11331155353294471,-0.39395960316408685,0.09417922369009163,-0.15243333313245164,0.07118983860034993,0.02351291696273021,0.19129269825000234,-0.11685894477086348,0.25075037305275133,0.08782450538221426,-0.17366404996085813,-0.21997926260023262,-0.2597410303267206,-0.3466845705913071,-0.29716262217794986,-0.07288337513303135,-0.24531252976828616,-0.5155244318018132,-0.04471849427893063,-0.1460054420984732,-0.2012830710724273,0.006687686693789229,0.014180393314044599,0.07856419906704737,-0.15248709584293776,-0.001786696259547264,0.042122514147441295,0.050229011298162755,0.06473337878075987,-0.03542878767394438,0.13786693931748215,-0.07199004728888193],[-2.493321992733434,2.9351911662673267,3.3164417416802325,0.962307364329186,1.934491063951486,0.9325879474130824,1.0164241688772275,-0.40979172544836734,0.12404139814374532,0.5423299438298026,0.3807192519237733,0.4327247191366067,0.3278994486016434,0.49118162568177487,-0.6078946825158071,0.4097907769857308,0.1249961159356759,0.12917832260147225,0.08242109127086063,0.04846037890001771,-0.13070407272994278,-0.026069836308801612,0.256324748944643,0.4277130476261046,-0.06651067065704205,0.2843342581032455,-0.23333444362017988,-0.002100463790845334,-0.1222639012758444,0.005772122958450073,-0.4750217672923802,-0.17952344952963178,0.19858484316288397,0.21831061389233627,-0.4075326063396798,0.14423608421094436,0.015584823605945673,0.17787880757819913,0.061653966919391366,-0.05697695625838494,-0.14171361806706154,-0.17583827375958402,-0.07255664479184658,0.03684597813100083,-0.05909024487455068,0.04300808905881287,-0.18396116043673036,-0.13041240211799876,-0.06886668372437682],[-26.010302707960722,0.057332282668644735,6.156275633236342,0.3928498496697907,2.4180919373995935,-0.23245847801600825,1.6788239382203922,-0.29640664667508854,0.23667277004053908,-0.5791891230332473,0.30835215794208004,-0.2871929592819371,0.22723421730464313,-0.02844764364614571,0.1598984251514152,-0.4240227067843312,0.1796270207424468,-0.258501359608437,0.19727269050598217,-0.1974783213024058,-0.10923370353917576,-0.01003028986927334,0.39271307183265053,-0.2752243369485207,0.2558861354583308,-0.0157925697256551,-0.0918777022450259,-0.2761868296280143,0.1567024883872829,-0.37260928540227345,-0.063198945201142,-0.1578497731495368,-0.038021916207230444,-0.12767553806206705,-0.05277987639147659,0.023153369322901327,0.13515891945520114,0.035983100676341015,0.11749847350327466,-0.03227715395787436,0.1769057348789375,0.17708530533553438,0.1059790338276279,-0.08632254545267108,0.003485529423118825,0.10020702105522222,0.0331007162620674,-0.08039220545896465,-0.07463738657225],[8.634980757974363,0.5097695015578985,-1.1161806027163097,-0.28028586199650835,-1.40300484992433,0.08902179186976507,-0.37649688746428395,0.03605965773101418,-0.642118773110201,0.04543957372403709,0.2536898479694302,0.18068950418884028,0.029627350508200992,0.0671490396920176,-0.021135937090821982,0.2446502373272314,0.3592914329751113,0.16241177552783106,0.011892669681892665,0.11253839637589379,0.0219585833027739,0.13368850794691994,-0.17585108510720251,-0.012404029836266739,0.16325577070010916,-0.1104981207218489,-0.10949563410018479,-0.046832559523548,0.07754859187826678,0.10127196779670354,0.043755031835444345,-0.11077554556323663,-0.019311695132313057,0.03917535182026325,0.00030025972548608054,-0.07445882860508068,0.029104862694693836,0.12010055935603504,0.15321736710237138,0.0804762385945314,0.0559346647275196,-0.04389959739810685,0.07427041808513649,-0.025456292795761233,0.01197570632788246,-0.019385615425099492,0.023639011874880156,0.10473790022729093,0.13061984289251052],[16.631901974462757,-0.13519880480193033,-0.6102277673036866,-1.61046196560415,1.6800113923263194,-0.7394389860891082,1.9881464474697874,-0.6772847292129855,2.6765882117211808,-0.5387736026834606,2.4402370288676187,-0.35037619389899394,2.444866364435914,-0.19717006560299555,2.110108298289046,0.11295160942006245,1.914856248485137,-0.020927197530826315,1.5244907101985643,0.3168808541127329,1.7075009752839132,0.1223501468250828,1.3660271582497714,0.13931000007525385,1.1458746644345161,0.23194849072319235,0.8445937921382216,0.2680305629857002,0.9507990469228939,0.23386331445824676,0.5353260131262201,0.416512181383817,0.6305530754208101,0.08329547698177499,0.5329810358127653,0.20624195987953817,0.3641735063102076,0.138677096943732,0.2639216788619206,0.19527901193556044,0.2000091705446179,0.20012059170304486,0.23397091037502266,0.2209208232864424,0.1187805511291118,0.26464840568618503,0.01145591428873056,0.022290389685864605,-0.10419764249099556],[-2.5231506529496754,0.029036082321091117,-4.790342787810467,0.33485221964204903,0.7448311918264912,0.6637666859423159,-1.336397511552575,-0.7509454638431795,-1.570107339706696,0.6666488927650586,-1.6894476334803754,-0.4297755408847049,-0.8718436293152915,-0.19828555576874043,-0.36288361065203356,0.004806144328947644,-0.5988383094235679,-0.32646040002742005,-0.5094665724159965,-0.15724817225361332,-0.5192754968521828,-0.09630835114136785,-0.5236225485711342,-0.19509397069423356,-0.2885313726464156,-0.21141778431582534,-0.2005981454684224,-0.012567093233488846,-0.19532160209237162,0.06903215544391993,0.09191516584258522,-0.028288856109785918,-0.17340921552653918,0.034650396365691595,0.11910463022024803,-0.09226289350107349,-0.07227194393889931,-0.05018280548510041,-0.10932538306465714,-0.09161021374075028,-0.2713035493255942,-0.04906617798801669,0.22814462398659888,0.0663176764385641,-0.00020056454636967528,0.07893110860986853,-0.00787351145610494,-0.06933276780408212,0.14581049781677505],[8.273568908155026,1.2223337204778673,3.021154115676041,-0.5108814951239868,0.3656191891690643,-0.7572894554408532,-1.3740024515880611,0.2839591954874656,-0.31324923426948276,0.31717348455165534,-0.2634064252665978,-0.8334216278518791,-0.48396228619777276,0.10493980820986645,-0.05047699870874902,-0.040370895860884846,-0.039225673689263635,0.04933265506469682,-0.123717475342202,0.3339208397670147,0.07771692093398624,0.11564918486616056,-0.150509571181153,-0.45248480056158136,-0.3066216058869341,-0.22831168720723335,0.1977179580053603,-0.21302798299888503,0.018669432667649467,0.1252954986300182,0.11707789203052349,0.2251465035389736,0.011074002149096223,0.4024338118643897,-0.11925988636437666,0.05448151720556922,-0.23388403715421438,-0.013032245205714369,-0.04954951653174249,0.008368326568739134,-0.08459743954790981,-0.04824179525559667,0.113422815687796,0.04222790170351692,0.1975725739190684,-0.1591885834166593,0.06646815246764276,0.038116193445476815,0.15021356709132863],[7.660456008674559,-1.0918992059239592,2.624695880780358,0.6496861034046494,-1.6488163329789138,-0.4570124596700503,1.0654239175812525,-0.01060441006361258,-1.3313183784264813,0.09632250260335966,0.4421882349512376,0.09295425770714995,-0.2347035498027341,0.14351990218056043,-0.29721657140372115,0.7906435182948468,-0.18749739492130185,-0.345281626580977,-0.3814856579597788,-0.17532650598537944,0.06675627388980016,0.032169110468410776,-0.6249463642562426,0.1912030506872891,0.009939926959191402,0.09546535265619728,-0.3559369744432391,-0.02269877819179748,-0.22813095076432985,0.12818836648562715,-0.10332103833079202,0.02040562611628165,-0.07705290223476358,0.0006746536271093502,-0.1139377910312964,0.16855391289875202,0.1041036650265739,-0.0641834890599362,-0.25752540421249404,0.34348381989119736,-0.22651102713509502,0.042571886681071744,0.09165094641375081,0.13094929231087996,-0.04382151387398319,0.06903969479621024,-0.05368542937118668,0.1951630281482196,-0.017946808848296754],[5.309917182134404,-0.410961148179114,4.12358891769462,-0.45689178671504554,-1.5120538053249752,0.2382167952935827,-0.7432009844856715,0.9976699512995588,0.6942377629416647,-0.0449707463771126,-0.058654075571646216,0.7835173900295697,-0.34011894308939733,0.36953197977318675,-0.30436307236871557,0.6867023311174918,-0.6024240625777937,0.8938365919889933,-0.7958504443931131,0.5637604892930791,-0.28976158978181726,0.32752967291440827,-0.6760650363107896,0.6378236402848558,-0.5512241470901729,0.36411774320744045,-0.24526263442206123,0.4108482845306193,-0.5111372381810648,0.10515382659891685,-0.46680515222868463,0.33476267743162197,-0.533863522951056,0.3553669178957299,-0.6054486605023077,-0.11207535010420172,-0.4176876104813747,0.08838826583735102,-0.13912068680495954,-0.12095966243921194,-0.5009228181943528,0.30421327338246174,-0.10416031137840238,-0.03179900770732188,-0.07051745809574235,0.03641989107360073,-0.2911553715618582,0.0466087913787654,0.03188500186616956],[-19.785357898511634,3.2252513161379746,2.090259189785716,-2.000336657898327,0.19201699575591416,-1.511531213264094,0.2541376636885855,0.09318925258151779,0.00914976040004195,0.4649159075245878,-0.028707605718631745,0.015720837853870406,-0.8173522159547962,0.16182209744669077,-0.2167083795097468,0.2501391370165241,0.06931310335825558,0.4601972119072598,-0.4947951932402498,0.046407544180664256,-0.3150758034609382,-0.2998837043178274,0.07237606814196303,0.2033098495453053,0.24622122176497702,-0.370140862719463,-0.04219864904596019,0.08911501424942116,0.2657741089275207,-0.17337542735577985,-0.2503617192051749,-0.10386904673170584,-0.10271293862891145,-0.010463448257445274,-0.1035044142298546,-0.21065262782883135,-0.24340970350661664,0.2152877001389444,0.16163089585207963,0.2541719286463714,0.04329218467087912,-0.10248343850396696,0.08173542673673014,-0.26837719767324075,0.024402113773178462,-0.03609419857859277,-0.08061405026467665,-0.008422466226636656,-0.024086203021055196],[5.004093993515027,0.3730104068304199,-0.8576050379498015,-0.5303174741811739,-0.5117972442173506,-0.44905571316453463,-0.5539931484687437,-0.35723216838713606,-0.39213082224604195,-0.05339310590846855,-0.7859193928675292,-0.07933037523656257,0.06532521558032045,0.18635966968762074,0.32021122207860264,-0.3139662382451055,-0.29040948934767696,0.05161385758815923,-0.12719848126367916,-0.03706595637963022,-0.018093817435165114,0.000981190299449134,0.07480976611316185,0.20263274940632506,-0.1439578830785083,-0.06675706810710388,0.13878975107197722,0.30973826464932175,0.20757856652490134,0.10955037845335489,0.185072993392636,-0.0379707035232287,-0.08194421774740578,0.06830694904315902,0.25735289706612624,-0.101049010679352,0.1350423565085591,0.11887981740447276,0.134400487282503,-0.05441096599846497,0.11758280924484141,-0.07576309068391988,0.3524025311194888,0.16406774165383917,0.05607207550069395,0.03193877397815845,0.11156152108336127,0.15969288927294661,-0.109484229092647],[9.558752214663285,-4.8224141620531125,-3.34295254467112,-0.9986770541511913,-3.834733860623501,-1.0236132184216398,-2.3837378169164327,-0.9716294985845684,-0.6469742617626077,-0.9784283475045531,-1.1138905426241925,-0.020887521821842288,-1.6729232899348498,-0.011635157589279087,-1.4274759973147975,-0.3896981811300983,-1.101613309026546,-0.20566654810594906,-0.46479438417207813,0.07509934097131452,-1.0702054417147293,-0.14267364435408317,-0.8353513152159258,-0.32646108042476907,-1.2953089294520035,-0.1480310838021392,-0.942212791851837,-0.5052314650728098,-0.8889960878401969,-0.5337818627112667,-1.0256462384273772,-0.5891074909847139,-0.8728096834628238,-0.08844810626107193,-0.9285844199217653,-0.38782800357901404,-0.6234771824038098,-0.3663783471146003,-0.6731392027913496,-0.28489068023275493,-0.6225584871731541,-0.33392626606245873,-0.5066036987948772,-0.33707898348346327,-0.46144490033683827,-0.14212251144683447,-0.41116002082360853,-0.44342003523444906,-0.3690299030419559],[-2.1077550281506725,1.1149644240107912,0.6549405148601819,0.7096660451433798,1.0935860756735933,0.7323625887464498,1.2364801138123234,0.04812501993616015,0.6876576450051182,0.3275091200199821,0.11830802626984861,0.011124440755110142,-0.24592898202243033,0.47649270447337333,-0.27629585372542487,0.3584913949635766,-0.4410587932080086,0.2304857389922914,-0.9903849081432057,-0.02268122651353029,-1.3266375568263995,0.09380010644231479,-1.1094638888917139,-0.0678451253511486,-1.2262688897564096,-0.23122020059195167,-1.258030567448679,-0.43886416642115494,-0.9363280539689735,-0.4450941975884984,-0.9171857265343809,-0.4749323152022978,-0.861891042416488,-0.3327226392810738,-0.5515035777187878,-0.4629827063118571,-0.3482831686884623,-0.6734065211679625,-0.20023865330764898,-0.4232481931236349,0.26884637061276595,-0.47125764132284953,0.34296929473492516,-0.3875445621422666,0.34121189445404426,-0.2889445495149715,0.6084632652956903,-0.14824399442086023,0.5187085119734997],[10.327881579641966,0.7397712119780023,-5.603379612580177,-2.0526264329747015,-1.564311917193267,-0.1847208281555083,-0.7623786838412077,-0.12263378660779468,-0.4338003828630491,0.10190596559511718,-0.720095560990132,0.6367969167575924,-0.47370994130898864,-0.13019559224024665,-0.6241689477113945,-0.72968231309877,-0.48318616397176,0.18329704932729532,0.0674913014690271,0.25302435744226986,-0.3543981066285674,-0.09409802333788918,-0.3688511487879564,-0.32015301329363194,-0.3714266009667324,0.07584469762500419,-0.2882432768302043,-0.3101788942814559,-0.23661411648505773,-0.1552899367902603,0.02119174231509835,-0.03553488647890254,-0.478671794240232,0.021316200770465914,-0.19841528826656452,-0.3326556952782891,0.0012281201017320892,0.15403638113966547,-0.047616461727907365,0.21797018232666995,-0.06845131793495174,-0.07344243052925956,0.23098152389787055,0.2524190493961569,0.17686628047630631,-0.013811747444863217,-0.11730417611167852,0.002866643280132653,-0.06304668605740298],[-1.1822605982518584,0.11680766848408379,-1.3516673031919002,-0.5447508376421997,0.0649348474209391,0.2195034348518652,1.3193706498950135,0.17787027069905892,0.0026573064193081294,0.02570916417905901,0.6304336859269989,-0.0155107415148115,0.12882957306861742,-0.22956819011915555,0.08196078449133413,0.25932920732018144,-0.18272276390580966,0.17214354706022372,0.34243918331772166,-0.1688879632981732,-0.028921642466559122,0.08038692493399564,0.2733743664244657,-0.303058398979417,0.30331262124875497,0.22862783963988448,-0.04610815944598423,-0.003268402218524858,-0.1960145476313155,0.0891212086588197,0.03973075248474652,-0.05295270154677428,0.06828746951052626,-0.161074779689662,0.16372816861492193,0.11276297120788643,0.03703844329123928,0.12308590998282046,0.050988932877224474,0.012198519510528935,0.08326461363465686,0.01570851288921566,0.10184160971167457,0.015223966578100264,0.09754777253706294,-0.039931983896667224,0.026304666399490036,-0.023585903535948728,0.011792854172980097],[-19.366929948315512,2.791775808918964,11.939046078978373,0.46378370526683915,1.2851940780413726,0.010354728083267714,0.28493420594272745,0.3594654799623632,0.7878782876441893,0.4020325650088978,0.2299510079326768,0.9645929054219591,0.4042852219922198,-0.22462500424072546,0.6325400933292629,-0.33046769600161674,0.1896499882636813,0.19327079113268952,-0.19588329104947214,0.3943773230213216,0.7150337767480068,0.11564584138111661,0.07334381208801816,-0.13382070996802786,0.4225061152713687,0.602737271193948,-0.21572029186721026,-0.17452972827927277,-0.16688000740560482,-0.07191358597043004,0.1456397728751803,-0.08676333425376383,0.13994193555864934,0.030529089976354916,-0.175073665819148,0.06463757999125816,0.11229475809601278,0.2740311343725909,-0.04042928556904734,-0.09990526942733197,-0.05927563338476207,0.03871092897555841,-0.06573387112191484,-0.006526711494403781,0.049707895204415195,0.15136931416322996,-0.10450969022566639,0.04527945337217869,-0.15676879091858373],[-9.435649218023153,-1.0433131560857383,5.034132069137863,3.538584172386011,2.8772444474417593,1.126989094886777,1.1877460914263798,0.4460983822950108,0.582601846526131,-0.06526089671618517,0.8569307531214645,0.4951759733042025,0.4491890624250205,0.6623019465375124,0.5367427874573595,-0.023863507674798307,0.4892462790760691,0.3926990897797545,0.23306464349405737,-0.28099726871498215,0.3169332187050343,0.2557326534175729,0.3070777913914375,0.09446558738261439,0.26063026797191946,0.14853500324332825,0.004202221120288101,0.050900373867631715,0.38873226430373786,0.27237788597560336,-0.0455396688237967,-0.10361773521276203,0.10665728222312945,0.14850727599674352,0.04110168827746231,0.04012702678191471,0.33484685994702684,-0.1196648499252168,0.20102450866013516,0.09164735018521486,0.21590847925128448,-0.1696215341390249,0.02888464201839262,0.03071342421839481,0.03922984808224537,0.08888924795198369,0.06281895578519167,-0.08715851780574953,0.057353593459485985],[11.316667519897434,2.4255029673686943,-0.6262387654983332,-0.7775657590245625,-0.8130315750216387,0.21073755525252932,-0.2971591221339522,0.41862189499044566,-0.15719539595625637,-0.8559494678674655,0.09280695477898045,0.9741465873166888,-0.09870911591449383,-0.2711682137483821,-0.08892074774037496,0.4735432216278158,0.5122070344649924,0.10860721282586458,0.23451473087242666,0.38566062170878024,0.028681225507716446,-0.21564046630787279,0.397378372238857,0.048716384514596124,-0.2822791074272196,-0.03665731352025055,0.08651191203151404,-0.19273539935398268,-0.10937038474457557,0.05255916838118407,-0.11929816832319551,0.1576475938848355,-0.10481591359982036,0.0668659537233303,0.10155142022785853,-0.11595826219861718,-0.08513573106617756,-0.17420301971067848,0.006878194640838132,0.18710495047204134,0.10960347653966487,-0.08867865556773981,-0.022546495451905242,-0.15961898011835976,-0.013723950211692022,-0.28390896922989306,0.0642812160450608,0.10620482425655449,-0.04721401583291764],[9.9503459197853,-2.081652778352419,-10.038408938916884,0.22601340149533486,-0.6023836583599994,-0.783008238300883,-0.10971669465281354,-0.7760354154607639,0.10401322082893294,-1.0496869782375695,-0.04192097894868894,-0.4190119111165659,-0.34941809668350904,0.4952152382441122,-0.3107159672993804,-0.20673224281022254,-0.34853082506028127,0.08157408499617014,0.41357415408604026,-0.49631633461963615,0.25883532809592114,0.0024286334622241916,0.09343298388357427,-0.15543075930773914,-0.05675660358748905,-0.08880357660467653,-0.4360740340910749,0.12393569457082211,-0.034729668805754045,-0.04726312796458854,-0.07570434860091159,0.022766818009819955,0.11585123518373104,0.1875811840353463,0.10248214963176952,-0.2521309750392886,0.0023141506862745667,-0.03222789563511804,0.08650035474238839,-0.10367700914408956,0.09418033381914219,0.01550322100279987,0.18884395903923018,-0.07064147128301948,0.06288864132685264,0.13485122016272735,0.1154556684102559,0.05837737968638219,-0.11230541040059276],[3.8682144368763556,0.8402651867707404,-5.669738597028854,-1.8043033544685059,-0.15085266535938638,0.6831321793906441,-2.6329651898312445,0.6576215484962236,-1.5226784053629787,-0.16538667407611335,-0.6469546953470258,-0.2791112539031184,-0.5211805820934728,0.021314072964421532,-0.5548344130885423,0.5154682403332553,-0.4503004254785335,0.11536585133515141,-0.23196906842760942,-0.18943911546765313,-0.33013147270440996,0.25071473414656686,0.029523495986722756,-0.06721849103078435,0.0048531852097211635,0.18469735803969523,-0.050874569281442183,0.10860523637620262,-0.19426029348719454,0.04482811896188208,0.38313582520545597,0.04113773386268181,0.08090598249076204,-0.002688645796634074,0.1622722401699448,0.013778046819196038,0.19230203020964295,-0.14021341024892883,0.32528904586044466,0.05687778150761854,0.33528810785705926,-0.13917364271301377,0.19244356245405567,-0.1459453843480154,-0.06826493307598978,0.08485108677813744,0.18347522641989147,0.031152906066142624,0.25467638361826334],[19.231248935995488,1.0135868316633099,-6.859917822422808,1.8088712494904768,-2.097165669070645,0.5709162401201862,-1.4511662502139298,0.5299127116107392,-0.7651196349401578,0.7663699663737915,-0.7649755310122834,0.7709813077499537,-0.5281794175762622,0.06388956480474486,-0.14950354959293552,0.41770533856566716,-0.44387174647907013,0.6739842323174023,-0.7412701931484731,0.509147455242951,-0.3722138798427685,0.28625547395478496,-0.2393571290935805,0.2518346894284543,-0.3293203951956701,0.29670333119438824,-0.17343319957021353,0.18950652565209808,-0.5039223003595554,0.23677949578267352,-0.7195571517774386,0.18496139841535753,-0.35636760230082,0.0015661235633186756,-0.28378397066327765,-0.001142419090174604,-0.5106515540869454,0.012484310151191292,-0.4679096386240518,0.195225027037591,-0.46064640024834846,0.0061114908344478,-0.38151738155273196,0.09568477169997118,-0.3650561925349023,-0.13334841515671184,-0.292941814781427,0.1109701837101064,-0.2093277251520025],[-1.4353229906279061,2.2134680374031324,2.55822365154119,0.5263999659708037,-0.5496731482280249,-0.1392385915453211,0.130634338689357,0.24489522491198087,-0.32447502538431805,0.538813374423597,-0.7661268435973935,-0.4819754835815195,0.15761682948912387,-0.4037071958558357,-0.8109093456600469,-0.14023066738323126,-0.8439426332107841,-0.18608401037634295,-0.5046141016637773,-0.18972498536826096,-0.6800631782253241,-0.12573681797806177,-0.09165296908025865,-0.10883635434481709,0.03196379998857277,-0.38485338662436497,-0.2837083858944503,0.0850873144302985,-0.5352620270580583,-0.03729515419477203,0.17086124625548557,-0.08396995554026952,0.013986239249429622,-0.16202592189141637,-0.28304979099951455,0.2648136474321162,-0.3469911004881809,-0.03481996848890819,-0.3285235756519986,0.006263346327843226,-0.2453087172204065,-0.2755535233527181,-0.12503527995901592,0.13864498249269774,-0.2033667922018633,-0.04076360048951969,-0.2262779907424405,-0.1781480623875725,-0.2077233961995308],[-10.659067929367971,1.3434802212719865,2.5047400400383606,-0.49865824228551464,0.09052588420494359,0.29498949191083224,0.24758535110045354,-0.14875675959356327,0.11971582188251775,0.3107946520124885,-0.20109901279993378,-0.02615314669134646,0.19630406097359493,-0.02445764098273854,-0.15328633020973323,0.09171283073957187,-0.2406929967669522,-0.0012539735431928655,-0.10391828683538497,0.07104897250293678,-0.19354734698733664,0.17296131562860903,-0.23469267147381948,0.15636438468718272,0.3679889452607354,-0.08565178000710341,-0.3900153535310951,0.232378817881217,0.10147373796163649,0.4339949370129003,0.10517084615555662,-0.06109783552647238,0.2690281698350845,0.04813551798705507,0.18958538879407888,-0.16135888503463156,0.048854537229771365,0.04624812241722804,-0.02351207759977146,0.14347245167266703,-0.03671495558737522,0.0902290169380929,0.05937462572659489,-0.004333031362723173,0.15568650306637283,-0.15383919577966293,0.18524420753297186,-0.11634092089431258,-0.03010124567627238],[9.414103998967743,-1.388027029599176,-5.094215738535625,-1.8124578766529282,0.6219746458689764,0.06101585819404014,0.5492196864491565,0.7247100925753753,-0.3161369194169063,0.4464746320313507,0.45659389870797235,0.11870877751091094,0.021040435026788512,0.15031722646552925,0.5269434474573189,0.22052633834529972,-0.3165653722683941,-0.11693271067115629,0.4597591055163683,0.35378077914643163,0.08595921709614437,0.25358241706418366,0.3604191932225408,-0.014984826107325913,-0.40358201308344216,-0.12514492725421397,-0.09960702666765665,0.28536846980793384,0.507433215767041,0.3014701466314928,0.03783227309171943,-0.10577191754891478,0.18211497113369454,0.11326859512570775,-0.15889150438929425,-0.048833751675632975,0.24840349227743067,0.14459544943719005,0.11269422176053796,0.08000309138297951,0.24050116896566795,0.14852958458533883,0.38232690993261464,0.17273228454709783,0.1542529301883709,-0.09815457665844438,0.045862620428009974,0.023471110561186087,0.03203279643108144],[0.7857621158670925,-0.6473817934148327,-7.740527545315349,1.9149783986722866,-0.0993412350807477,-0.039812408011738486,0.451928581642347,0.36814811121556223,-2.4049983094593395,0.6951173764053339,-1.2959091640994196,0.9078918825905634,-1.0395217124751903,-0.004577583336267916,-1.040526506864864,-0.2654367045418325,-0.9720660043943258,0.5329278870996352,-0.9004255282983175,0.12393555044591753,-1.0271893336205458,0.18337552951621977,-0.8089989913753762,0.1955643184803314,-0.7243950598527533,0.24156624401264026,-0.5116891637262461,-0.03172472765518528,-1.2597999992105102,-0.2594181828899578,-0.7117659175239577,-0.48932204004017155,-0.44684049295322537,0.14281741169750567,-0.7597426395960565,0.03531487388758349,-0.7781733224640517,-0.08099725426768505,-0.4543546125271607,0.07639722455565702,-0.5536615565087442,-0.2693438573729721,-0.1919327808413575,-0.08150631479930252,-0.17435204431253296,-0.03646347882244412,-0.2881356833953947,-0.12918962827012512,-0.03252082076588782],[11.545449185666635,0.1889948970800405,-0.03176752447282399,1.1115238071484008,0.6841632197119306,-0.34061140565813947,0.9706181954386881,0.29298876258308476,1.1220035425593795,0.3915131374285559,0.18741107372740692,0.2664123630827019,0.8094170028555006,-0.07729550850994475,0.2451440450016916,0.1835362355052221,0.47611787602631633,0.04784677365783841,0.33648916505562093,-0.08107662382065658,0.5564246724295837,0.20017135854728338,0.34352749798607063,-0.128626604672872,0.07396382860075282,0.3087233348043852,-0.01667890110750274,0.05269057871286979,0.09010992089816038,-0.10550268384480611,-0.18763867281569704,0.250895148962619,0.20474715954064818,0.20432513963061108,0.12923794920808535,0.012513923201063386,0.040500644291584155,0.17371291695175226,0.026560503512931752,-0.05574121003724172,-0.08016713734830218,0.013801121159543194,-0.025590813500278402,-0.11108343371444641,0.1547477592982777,0.1378821337211515,0.09988018902205474,0.11917414647766934,0.07166977450519606],[-9.995778527954512,3.057954364732927,2.675436154249291,-1.9680436481030354,-1.3158671768117478,-0.9903937785898032,-0.4834222628303862,0.05660723023429989,-0.8185624711150855,-0.20287307517838948,0.33464288406706166,0.19500069104943937,-0.06541753204615185,-0.4924735055911557,-0.3234283861474638,-0.21535585330544654,-0.07410977771874241,-0.37059790328331566,-0.037295315755065055,-0.1960974677988205,-0.10685123146471395,0.2351337888776386,0.2550037899488539,0.31834048872767357,0.323550519259115,-0.3580407881701294,-0.031137281232259277,-0.48242434087854275,-0.14244366237271722,-0.011108377596347148,0.009097758366521793,-0.055112194782631,0.030397913561210074,0.004700616602764728,-0.049226109615622723,0.07050357515841735,0.4429355848302736,-0.0719584644619711,0.13695441751750492,0.03051950804630844,0.1881561620695735,0.08701057546274567,-0.041264564789705103,0.25809580685783184,0.2944369003208284,0.0015788688779664747,0.22671456383653618,0.038166995611036744,-0.16239042157931652],[-5.131434348155498,-1.7551760796925553,4.552631004821498,1.2614460019880278,-0.61236240222829,-0.2898539479415837,0.9282853984601351,-0.5748889528735158,0.3628532533620177,0.09234307578480602,0.509905871075795,0.10979802428742128,0.0763371907467167,-0.21324615368670752,0.3918833308619745,-0.1167300733624022,0.13979800251757332,-0.074139546406856,0.4742121451171967,-0.351853594613106,0.6269325474943679,-0.26177719968548285,0.4817876150921983,-0.013897255672732482,0.6290225689730058,-0.019312226287421786,0.360806001776096,-0.2741434621861776,0.3042340570929925,-0.10060270873396122,0.8607970942965418,-0.44280441753538874,0.626681747240315,-0.13102372753951563,0.5733262942544011,-0.30837300741976376,0.4024934099242555,-0.09314089402938067,0.5604689215487443,-0.22512842005745473,0.4660862034329542,-0.13972025591556986,0.5785097309448238,-0.20246177310122884,0.700531168162917,-0.13507115212103812,0.482459119879262,-0.20897989374890774,0.49814275962566545],[-4.600100409022539,0.8140296814896024,4.766686314809944,-0.6660194525579223,-1.2164163432279236,0.17403035052205063,0.03818043880168816,0.890241864010941,0.9697620723865064,0.01283813218704615,0.3978734012541631,-0.13736933576921803,0.8769940457166704,0.12480400218998017,0.5268540914463409,0.16218222768390794,0.4979226306821653,-0.07109427573559744,0.32968593399092816,0.04133128732041151,0.39292360442573204,0.13886749779665192,0.36036989306944484,0.053008739151610786,0.417622038598715,0.07079543979369445,0.5021320055751423,0.1465235554095244,0.3587121933422202,0.02275871629618821,0.36417493611221075,0.2574199925767447,0.06590604790604608,0.10845556899716176,0.40137254082136137,0.10185806261325506,0.3571923033259179,0.13449873778378166,0.2039848321493711,0.08703695622725356,0.37200236590652636,0.04558693441831273,0.21231215538872386,-0.027756183333661685,0.3200352658539499,0.04809196886422553,0.25547751515603545,-0.0038742206036635938,0.1602687720329177],[-24.399000211291355,-1.0466613418694242,4.356815512846441,0.1342541128669573,0.6466890014602478,-0.36485044355029217,0.035323751688156504,0.47679081334847084,0.3200855084772606,-0.034391169285707575,0.10318439777566894,-0.155864612929078,0.0428769275453222,-0.07739600275742639,-0.22920802138506827,-0.07512401416646672,0.14979827098132784,0.090792968251465,0.15638440364421793,-0.22377701113678716,0.02221634372051516,0.2272348044729268,0.23610474619385946,-0.06651095888409109,0.2658478380114503,-0.09578704190824859,0.31898242444412483,-0.011118602553823739,0.12816101184962692,-0.3326050300054905,0.15334847897798495,-0.128162832719145,0.03684473551471455,0.044350604687639995,0.141851660103108,0.17511333348253175,0.18096535146552706,0.03338625946473752,0.015458000378687749,0.013780400428891885,0.28165806240210955,-0.007309219426698337,0.08186663499697293,-0.07866670137808654,0.0294049340036634,-0.05497648317982002,0.12157914912383587,-0.07563583812478233,0.039828591578315434],[-8.390706089492907,-3.9264309221671865,-1.616203901851875,1.264596191512193,1.4741762839229227,0.4669328752414826,0.08272594911281803,-0.31569964208726686,0.11890634645035483,-0.04016049663540887,0.1430063282655042,-0.1333728249292596,0.31801493111103174,-0.16268213104392878,-0.303565588149516,0.1774418950698949,0.1739339310109379,-0.5441226082607304,-0.2606402768730811,0.06731156806429527,-0.43425029109747304,0.037966879025783516,0.42496451156475773,0.06693288840276607,0.12130925723906233,0.08960639268950517,0.4019069306210234,0.02237618851021818,0.18712651448205742,0.13568157112024842,0.32186302736299155,-0.04684985974602196,0.0587670115457314,-0.20546367062942542,0.11945943575032104,-0.1897917516509398,0.25288033083201894,-0.05876347320349473,0.10196160893866534,0.14381573816854262,0.040497907212378864,0.05791518224142098,0.11598416175442089,-0.1526658002574075,-0.02511153097096268,-0.14938772299290076,0.15315458983459332,0.06905883853641125,0.034434369878180646],[14.908996776975789,-1.8712369098235595,0.193306843420702,-1.6562373688231233,0.6146373783264985,-0.4091134788793452,0.40631454854249,-0.7024987371409723,1.4725505791513738,0.09803358035939595,0.982567852905839,-0.8454742738765951,0.8614963835160367,-0.4773743508253133,0.5878175949467319,-0.23695833747929687,0.5699096029720262,-0.1525732721009505,0.59480991400558,-0.02870916445971885,0.6775773408420547,-0.16093005221694823,0.28922228801167993,-0.11819424415153856,0.4927565697191093,0.09438888619818492,0.4657694928785182,-0.23787831900851533,0.10217025856031416,0.09110323194585668,0.010202614141793782,0.19755224667184926,0.15774980880817638,-0.010576749758195732,-0.0994246037446213,0.03379465636809852,-0.06446138837050336,0.10442997426740612,-0.10792195772348435,0.0002059173617984704,-0.007367696553514298,0.0645927274686397,-0.007685616536727848,-0.0101707089202136,-0.02333380762995287,-0.02075724379913863,0.07560185778395027,0.13584960853825614,0.011753456622585163],[1.587945346815048,1.9899424941172088,-7.1654255613750735,1.2655466843349032,-0.5846262351457872,-0.19604069253049017,0.3200471621801159,0.987784429321224,-0.5027131187797191,0.5878315361210698,-0.7045877608097395,0.6094437848281089,-0.5327821324826848,-0.12231622011707721,-0.32483094412216357,0.13635385776916414,-0.5319917840205636,-0.15134897332946073,-0.23066801401493553,0.4033295990838932,0.00718447102472616,-0.05770003810956131,-0.0922829201422014,0.04555774770030253,-0.21347305330692581,0.15371753657435433,0.35891257539619076,-0.11927008523074975,0.15374670160796297,-0.1947613066362272,0.1710578170924092,-0.008238134874552373,0.2324679192958692,-0.08128242891889612,0.3924178972798388,-0.35391180569193637,0.4397523218610529,-0.27202671175574794,0.26580043551541693,-0.05009897329207934,0.2389201332554655,-0.32118926125361025,0.18867306583627028,0.027874881068900955,0.34787331239695785,-0.12534825630845975,0.40984795036476296,-0.13517178921737252,0.4372013934202209],[2.2067101908990825,-2.5749399635857793,-1.975168898214501,0.6141708246988143,1.3150385419055008,0.189345348765723,-0.14954820686085313,-0.3219507020425262,-0.08271741969915232,0.24560798970235184,0.1891993887338473,-0.023949671489356616,0.8234559812403595,0.033396157363928654,0.28318555557338315,0.3175131928806021,0.2682795425195782,-0.01062726380825169,0.3043260035115429,0.5414520532939303,0.2404338498010465,0.38220820951946466,0.16211584410933794,0.44527863898672315,0.11278607316168329,-0.03851727197895192,0.0033757104073678292,0.22746653541178913,-0.01280977985146132,0.13815898226999265,-0.025239488218901884,-0.06620755714478402,0.22050067647712224,0.18748271918719983,-0.13815991471056724,-0.02289708597055396,0.12005626120775127,-0.05245490101212742,0.010603739535653783,-0.2083281159495155,0.00924596715780469,0.06258318997958888,-0.1740779878296505,-0.10853791417833378,-0.03819167670725984,0.0785929831819899,-0.07646579768028373,0.09824712070214633,0.09439291404605883],[-16.34145466726415,1.248515619610198,4.572202233556246,-0.9179979241793443,-0.4751868417464261,-0.6728748936540576,0.5854526397943197,-0.142865131931255,-0.15471310446345712,-0.4055693848751782,-0.05881275623290109,-0.43789633540100553,0.7589964041135212,-0.34013275958302397,0.02010282532674894,-0.21035488344502098,0.03516948001022475,-0.5293383849647991,0.14347090507093016,-0.28091632896265684,0.3781519490732264,0.12331275010865564,0.29001976256271395,-0.15788570604249416,0.6558035027080475,-0.2135746503288894,0.3956448903277045,-0.21865006604390078,0.20668347570901435,-0.5030849917869447,0.4144826521308395,-0.09262580369953786,0.4809760950800836,-0.11643251364718993,0.34096008684968593,0.02003084769674334,0.31109647662886447,-0.08118333791429899,-0.013529888064119217,-0.15174371518655488,0.05763122822228639,0.034588053481215034,0.014499818746330087,-0.17549899846451578,0.07146449867687395,0.18761492362061163,-0.0007260982233708492,0.14633877793141842,0.03670977339235816],[5.577116657667604,0.6171120572089566,-5.367382813533052,0.9140793220255928,2.4937970447909574,-0.21408599748424134,0.5119142583080186,-0.20190105620104565,0.7660390556608644,-0.20085903717290754,0.1548521821811269,0.6904018104047631,0.9439921859016844,0.041095413639924465,0.90934626560777,0.17184252682452503,0.6069032254700165,-0.22759532416280412,1.201241624230657,0.0005573219697674399,1.3010681375831532,-0.15190342606673812,0.6974683862867782,0.3800857035511492,1.0006144324391284,-0.16837498204039839,0.8642730718334997,0.3456733075319932,0.9389132371420355,0.02836552668992687,0.5543363817819396,0.11805105210716693,0.7240481334301844,0.24538689418269918,0.6657953336779895,-0.08558966843967608,0.40187812707664894,-0.20765655806285122,0.6863915720825103,-0.02133924875222607,0.29059213264099076,-0.01283766833387543,0.4466592965985494,0.007840047115215385,0.4724349672208107,-0.003204502884663768,0.3889226398322566,0.20863813117784621,0.23514936726961005],[-8.400424831733849,-1.8570628612343028,-5.394732142404897,-2.519298853550787,-0.021200284588061367,-1.373843043371082,0.05854336757962233,-1.637621684467246,-0.668334079431435,-0.5382478435710034,-1.144692865986233,-1.1083176313737655,-1.0646404768767737,-0.6762750590723906,-0.7141725429568225,-0.5827131790050711,-0.08747906211964354,-0.16732643715948797,-0.4798277940503008,-0.19110595594563848,-0.7600108972218117,-0.3683542036803235,-0.18475051282705648,-0.3429145626682374,-0.38536959499177165,-0.2702190971949062,-0.06056392360814091,-0.028808730434274656,-0.23993686115509288,-0.12051386911263057,-0.3968951930003378,-0.12663636571628245,-0.3073144668384011,0.10636304820766496,-0.05599986983924637,-0.18732817633439866,-0.3389056528000023,-0.10477381764809786,-0.5204617767710962,-0.22672762332323965,-0.2900423160319627,-0.10914523699866753,-0.29414367899712107,-0.3186964295705771,-0.13030408738640648,0.12116744419624058,-0.39149729943958095,-0.17566791665824938,-0.14218679227079792],[3.2102958009239817,-2.9165374845029164,3.5906127757471835,0.4075760146229812,-1.3033004029823692,-0.5372819103326156,-1.79072169749962,-0.37784599301926397,-0.46666946196145553,-0.13460215070848044,-0.6594839787987866,-0.3193459788966643,-0.3757265216173927,0.2568782513220765,-0.46954379576507266,0.36926333990014637,-0.40931281509014206,0.035954571000851976,-0.19127035801165548,-0.11001556142197694,-0.25198404654338385,-0.4111801422423464,0.10677246710110357,-0.19610636079624266,0.022649766903961498,-0.45190458571313147,-0.1798396090218425,-0.2612437696825401,-0.12370439409933572,-0.12096773608089552,0.05922775955785902,-0.25880146449538105,-0.0763946167764797,-0.13706511089931117,-0.04400129968783774,-0.2031226898593656,-0.0449031197062559,-0.12479861568405111,-0.02214575879268343,0.2617197063946225,-0.14533288582683318,0.05221197344624839,0.014337381339643901,0.005186899548955869,0.1392206173684388,-0.083772182254416,-0.015152449565434538,-0.07963165013604698,0.006549555867121813],[22.835091693792847,-1.1934566291746698,-5.559826988652031,0.4263608591078728,0.2953416229264761,0.8692966289020768,-0.26216139556673435,0.37316693726942846,-0.45302234154224885,-0.42510816237760285,-0.09480964155730474,0.40040202545354864,-0.3734217725945491,-0.457835581508725,-0.5181745256283066,0.41687470875301985,-0.08007888930823492,-0.29565164189864923,-0.32905012842392606,-0.006111811035174104,-0.2333312425921122,-0.581906483845833,-0.31548824998876845,0.044458160926118886,0.0622205174680521,-0.4341322271390745,0.43500128992836834,-0.042667886152179604,0.09454373404610272,-0.13157378320217739,-0.1944767849020009,0.08385375208964875,0.01795099735692996,0.021519478506594036,0.21949946778203103,-0.008841798435430655,-0.16650239531079639,0.04118483973749501,-0.0631929867154574,-0.160346567551609,0.03337403346230742,-0.019725346606045482,-0.07874255803960838,-0.18840550730714617,-0.06686094869201677,-0.14208809034225234,-0.11891082621203976,-0.062044102059969915,-0.12773173510372476],[-5.289529392892622,0.6741730171571643,3.891779773763035,-0.2890790207096271,-0.07913093405614477,0.007646268158625272,-0.49314107421107206,-0.5104587892465801,0.1812799055616037,-0.17384455675076982,-0.3770210661746577,-0.6084496841812457,-0.43897514278039246,-0.20017538674074103,-0.8646570186118081,-0.06622885272369279,-0.137771818742565,-0.09941911273383117,-0.7516909865199174,-0.16559591407234944,-0.3342711635904309,-0.3289343820855981,-0.38889623800307127,0.05123163137334566,-0.3157716220758264,-0.1843833329214553,-0.2207510071923712,-0.3487046859447611,-0.3903316746905644,0.08449215872320053,-0.5245731919758126,0.04704397680470102,-0.4505256349628363,-0.03595409882543125,-0.6796600550941231,-0.23318504004645685,-0.3324789796302369,-0.21371217058971917,-0.1703289396280895,-0.13797757505669725,-0.21336778980251492,-0.030490296951930326,-0.3375515500971805,0.11005448229617357,-0.06057408678553956,-0.10522930416567632,-0.27636147107672554,-0.038796959912783965,-0.16838765203254477],[-3.381154826888688,2.597967771067965,3.6537631102625947,0.08123350141110633,2.1246066758377475,-1.0822758375038115,0.7586210398500998,0.7257554079913833,0.7165883907330661,0.2901510163665685,0.938174778079551,0.42677866104739,0.5950684784107928,-0.0924144264884291,0.3113584664490185,0.14427005303996993,0.6295168512267418,0.020322414864193495,0.6142561628064105,-0.025402945441565076,0.4946905448464463,0.5911149508045062,0.08259305934778752,0.18721184282667339,-0.04393696580548169,0.34287942701899804,0.3023956118535569,-0.02864805166351474,0.1515560772881065,0.08824405144510643,0.00012260375195911533,0.25375903913479547,-0.18945501768380618,0.01853199727264357,0.005534244087912678,0.16845085759710285,-0.14434229176693159,0.2735141710424836,-0.045999158136939534,0.21296013475865833,0.08742631736215252,-0.02043038124337607,-0.15586884192691453,0.04874442728405463,0.05526017597095103,-0.08709881448389323,-0.05786117184349803,0.06153152339079248,0.10183308253108293],[9.448150605429843,-4.664930639680655,-1.556836191226074,1.7466369335262892,-1.7102506974445606,1.00851627609979,-1.0231086844987534,0.07871788116339037,-0.10145879727609275,-0.2865638588011774,-0.38665002602508614,-0.5342554781098038,-0.2705798706178246,-0.14463728018452385,-0.2896219399952734,-0.9184442157020654,-0.22765778607437678,-0.2621639098227914,0.05942099835052621,-0.06605329897954651,0.2759224802742074,-0.12454586147598305,0.3706732600116249,-0.20099477627256748,0.07126076656759248,0.39266340980537295,0.18459041600407539,-0.02506704462796516,0.03323190491764397,0.2295844988558744,0.0012070074071214019,-0.17522323499172263,0.15626735539285527,0.05574090836436984,-0.05807928060140366,0.10348152756557914,0.19346067783771836,-0.25907654744309394,-0.12469463411478665,-0.011578911412670485,0.08449450094375635,0.005738454436319312,0.00694055550398171,0.07684558218143676,-0.0719263822034885,-0.08649240155370158,0.10128108410859922,0.0601767610304088,0.11379585758781656],[4.0434608084868,0.06772965359984225,-3.0250073600750635,-1.2916258764038089,2.216017117800377,0.47998961912888166,0.28851011837922913,0.11361383786349981,0.18773299694859527,-0.20114366244530738,0.4796039511976711,-0.49416133074798274,0.31719196446121367,-0.7553418126982138,0.5039949065232474,0.3619685385585383,-0.19498375142631333,-0.22807920963076186,0.2452817979905678,-0.04195771171865454,0.2967909886626586,-0.04321228282977473,0.1236713899168337,-0.05300098336469726,0.018037534030235065,-0.05477277004332782,0.15736085037027142,-0.41944995456946926,-0.1579055682910183,-0.29320795672972894,0.18199734892481073,0.27167432014662074,-0.2308816558038258,-0.1472461356996041,0.18474416542854258,-0.0033111430739219522,-0.09905917880512514,-0.08152458736922726,0.16900965324484885,-0.06057381055692706,-0.04936287661920678,-0.11750395606039458,0.15067125353851385,-0.08897845321760495,-0.19706753702364732,-0.004189382288020878,0.061832252109912125,-0.09167392149091914,-0.11556317327239701],[-26.70460210339873,2.671564751162337,5.330876591717079,-0.026066802768309482,-0.2680995015701551,-0.09565400972822732,-0.3246593340419785,0.05033276803575808,-0.025667001662039803,-0.18388915052902405,-0.8194966667806396,-0.7407650412245407,-0.7377449093299528,-0.11404528913989545,-0.4033723912249935,-0.1589431642653149,-0.6186914562290371,0.03754566510630821,-0.2899566293783408,0.47196266095188566,0.22067332880500146,-0.021139895089610602,0.3340844662926144,-0.1488192019593753,0.18898299485648135,0.2600630659211697,-0.057508165936863385,0.04233018404272601,0.11281469754003244,0.2442137963825753,0.04351443935635202,0.03312195134807262,0.35656051110913045,0.03169686289134745,0.18671167128285457,0.020575398757792216,0.2323356717179004,0.06041874535467402,0.2882570131139506,0.26901965093761554,0.08755870052973291,0.08350508172297913,0.1563943179443501,0.07674558755442522,-0.059742811462413066,0.025936770973606388,0.08229679114734675,-0.13785898672833158,-0.1380422207601196],[18.45741564383432,1.5835229328699052,-6.400989419916925,-0.3559441816979932,-3.3697960138155003,-0.4769108540898808,-3.1408191883260965,-0.17689861680806807,-2.5124954028524757,-0.38054749998172904,-1.711447728137908,-0.013062941368517988,-1.58999265221711,-0.3832407040083323,-1.3907795446169373,0.0881238714724233,-1.3909832944384444,0.20995007735936647,-1.2292271818938916,-0.18475075272994976,-1.1970749196458215,0.1642864660058346,-0.8455502197553971,-0.22465678242683274,-0.7835717616809231,-0.030179014561948752,-1.078278330786675,0.005982575796936665,-1.0999973566639372,0.02638319756322088,-0.6436815429807645,-0.1747377402573418,-0.5306293498004214,0.15983164720192913,-0.6805160364703183,0.017177248667613283,-0.44267305436523985,-0.08152578314143749,-0.6687034729115165,0.10030745912732701,-0.5782624162492072,-0.1948174039419754,-0.3654470439708965,-0.018868952256971883,-0.37598010072129373,-0.05962438377592934,-0.45390789385214886,0.2218752910485186,-0.027759064799561306],[-5.869557171860633,0.8348722393086678,1.4628363764601904,-0.20736092945164605,-0.4006717077342536,0.6479121879557201,-0.8625734994333857,0.4753598193732098,-0.7378227784469421,0.08694822349612169,-0.12245428067153062,0.2491886817119974,-0.44666197907501703,-0.2223449932895895,-0.4495851993595405,0.1083030473398803,-0.44981505057136384,-0.030435214691160742,-0.17083845914124152,-0.3627448729064889,-0.38610735185973055,0.11238479139188962,-0.2917285670098975,0.2282860014130039,-0.08673448107052105,0.11409995138283098,-0.4854179000350736,0.5982680824374983,-0.5928739593860134,-0.16594346279658062,-0.09386746030506975,-0.2892132031311632,-0.1280308618247619,-0.3234979275799074,0.036711080544219676,-0.15847006069479702,-0.21857920534949155,0.11308459192016662,-0.2219486429619142,-0.07889818688499352,-0.022164236066000653,-0.01230090531649866,-0.033181342935984,-0.1265423106009809,-0.40354754580518415,-0.1674620597126244,-0.00744881675159626,-0.026495458856960735,0.27764206580717166],[8.868724159838372,0.6833493001866952,-1.5437441431764574,-0.007716099877069439,-0.2325383764442073,-0.10034543439892621,-0.09367720035817326,-0.1442145180412879,0.07650281628986737,-0.13256233745875026,0.07163359069030478,0.07277622561009067,0.0988816622363557,0.280434007270127,-0.020599108112356275,0.1841892026668732,-0.039225919383877365,0.022196717962166507,0.03571766165590313,0.031976513659722205,-0.037036029718086325,0.03172372729533942,0.0611746423781003,0.07355021491038014,-0.11993611382330274,0.08564370347976061,-0.06884257532887882,-0.005614759567649296,-0.09401167550673285,0.04048845191051271,-0.04107987337827322,0.02478938444964444,-0.10893110415733205,-0.022875067373067857,-0.07727568440430473,-0.04827526686415611,-0.21891589470966832,-0.034982039340812864,-0.0943843688742417,0.0713828641305773,-0.0227779537280739,0.0031259387807718417,-0.08050394261134258,0.03836005946178065,-0.030473304380504874,-0.051816059247251516,-0.03895303866807739,-0.04414271528301333,-0.02540112743992772],[1.9829706722997806,-2.6280107565515802,4.859427836959804,-3.4144820556531865,0.47578119036038974,-0.539227464341813,0.4096987125897959,-0.5500623850964729,1.1825235521295883,-0.7010709526774591,1.0263072826870272,-0.8093118126491909,0.8796157571712475,-0.8334884143316033,0.7769560656312952,-0.1070090358562288,0.4766398046512857,-0.14479738695074829,0.7572811620827249,-0.6127806528438097,0.44027927547876217,-0.23478093286977086,0.30732597807656575,-0.24621275688822425,0.420576078989891,-0.31913194871307965,0.08393150797825906,-0.2894357054341468,0.5000895394429805,-0.30829239900247124,0.2539395510982761,-0.033406129987817446,0.11999084014923236,-0.031157070022278254,0.21005844286964995,0.18018117058873717,0.1609699183705916,0.0971054826096349,0.23561946535775607,-0.16195441887046264,0.1661835590544499,0.038606419010705884,0.0746856997480586,0.14702942054528698,0.0762095206318081,-0.0632314748261721,0.14904104823174377,0.1566527120624761,0.03224572164503821],[10.083800456784441,-2.0206879851800585,-2.247554930275513,-1.3700744210184426,1.9974392655430473,-0.19828834259886513,0.4890498551053204,0.5422470154187812,1.1226022986504876,-0.5068843640488077,0.9071429952786858,-0.23190162129175967,0.4282067322703351,0.1337911486026099,0.538364664190252,0.5249695428998826,0.3589048435989997,-0.07335830002466619,0.34954777203223986,0.26342322182252276,0.15168777288273746,0.16505013423734236,-0.04347305348664878,0.09583730374812689,0.05994861505107207,0.41456397173530457,0.44018080706759166,-0.1878833798127033,0.14376492069248936,-0.1898991254750773,0.2845578942281084,0.009980474799909649,-0.1799629328568056,-0.1104430282609429,-0.09511246038554592,0.16996912761077876,-0.2585918037715707,0.10735905351619643,-0.19537900670169625,0.022861724853387423,0.061166759138914936,0.23284391270896296,-0.05656243710791187,0.015314247572743321,0.01909844170558408,-0.04956383464207228,0.008656645773786456,-0.06913354938660089,-0.17273021779615064],[-8.502149196173946,2.126862649919396,1.5387453996914795,-1.642173018569852,-0.40207704408921535,-1.0180849664976548,0.6704016104359952,0.1831315361584259,0.05777923256630662,0.03550794444890584,0.6477408367647655,0.8228450310994038,-0.12327243635991612,-0.06635761591238737,0.6459831464377831,0.26054285716247966,0.10939975910501018,0.09780046832070288,-0.04768460190781925,-0.1997044612768173,0.05419756570520981,0.0827543092070055,-0.12726777802613123,0.01661854285796121,0.10947088422638991,0.3550005920163861,-0.2554069683225034,0.24326537613524668,0.27056639209431776,0.08102861871042746,-0.04423035615510958,-0.19923885034356337,0.14379935102991553,-0.10412650472108127,0.09142979724477124,-0.16035071921514943,-0.019953112002779293,0.2283749774369966,-0.3043765431187574,-0.09084307839231082,0.1106281243666785,0.16809198283354584,0.07395705746889643,0.10689797556445593,-0.06478986843697568,0.05965895173566404,-0.17794253129790102,0.21264943073353784,-0.023833658954913257],[4.538543878175386,-1.1034653343780256,-3.290768938049916,-1.6166240724216672,3.0468411758848575,-0.2652144677716393,0.2755824390075492,-0.6995227083700494,0.32368435298720716,-0.383336376026363,0.2843280903999431,-0.905418671376426,1.1209084356238912,0.19720183951608133,0.24534823646116471,-0.06914165028588158,0.632516895353241,-0.27525667864851516,0.5680717076272409,0.1592815547355433,0.4719370662704458,0.10395354229973865,0.3304832080630571,-0.3651219291853943,0.6113504797910168,-0.02477726914131835,0.3403054117323098,-0.11133973254979176,0.5190615973098085,0.1607846344098548,0.2826064201046493,0.04459866966304723,0.30044707719388497,-0.1629569869542503,0.5103117946596882,0.2301969535094164,0.3169873598179723,0.16294008003592586,-0.07063721101889525,-0.02080464441477724,0.3603620295553894,-0.15110570072351487,0.3408310353455828,-0.008002650686165931,0.16073416452583855,0.13043871778339666,0.16127999805416143,-0.09994991132545852,0.2302439139982819],[24.405425882300936,2.9157068116093865,-5.642031762689991,-1.0869274257266424,-3.3455744812006474,-0.10572916963474328,-1.9688837635517442,0.20571234653691137,-0.5714368812275987,0.16760600261597994,-0.03280733514783402,0.006690966835136993,-0.328651960697685,0.134981443450333,0.08372145946576648,-0.9679579974065136,-0.21370937253608027,-0.1111079334537357,0.4420689610683918,-0.621870429805715,0.2656964801367602,-0.19959811213656253,0.49672684869450745,-0.07401510785919148,0.6617906804969492,-0.11666667012563511,0.6090654754192906,-0.23731165209132477,0.4166678213003566,-0.03256912506340098,0.7709701519529073,0.327519769885797,0.5165485562450899,-0.020561521824601237,0.4812169932801177,-0.1678737673963411,0.6252079648080449,-0.0806081368419686,0.5660848458779753,0.1159319391215808,0.611344163086072,0.02677033766189719,0.743463109439968,0.26148921362416255,0.5873112323387136,-0.05516846585006706,0.5380561674311636,-0.0443090659077502,0.49112573775925494],[-8.78707687962486,2.632352857747703,6.896166273493137,2.176545766457296,-3.0639270697322383,-1.9124904453972449,0.9391401839915591,-0.8262557362586139,2.4621593128348933,1.1677811859376663,0.27964708028817314,-0.5204560386421889,0.9987382252349094,0.24799909457654373,1.4596074131292622,-0.39626268023433026,0.5484579863322036,-1.0118247298087648,0.487049166333444,0.5367088011698993,0.8971113224107816,-0.4479531983617354,1.1952432705505975,0.16785952537696464,1.181066279340039,0.27159270705636535,0.5364153155813622,-0.1503666281049401,0.9294024114489008,0.5040874313131647,1.1477296054988924,0.04660488061229718,0.23417970299651766,-0.055848436318630595,0.31278305251314437,0.17080441138533767,0.8851868938501174,0.26453217197372325,0.4364300399282597,0.050079713104865704,0.5417708408915396,0.38105869517758423,0.3205928784418339,0.3516036698955539,0.32117913219379274,0.2699521867598074,0.3750963599917556,0.2307097076565634,-0.013978191414311005],[24.03323616178251,1.1667565046188981,-13.295389165335804,-0.6306167315767401,-0.0956350014475091,0.5533728375013329,-1.4405954542282358,0.03568263319297213,-0.9564133633901193,0.3778198362290532,-1.5312052534161027,0.06564643565650492,-0.8160682389857766,0.3335900033857158,-1.2703872469377224,-0.07173866635892488,-0.16908233608483783,0.4183516036463677,-0.3058267888683508,-0.138996487803466,-0.5093021501184274,0.25428367842909033,-0.38864512620429753,-0.21733704482002175,-0.23676405108803286,0.14414006354832756,-0.2573187867856111,0.024519352492493107,0.0918120481908939,-0.014587795652841823,-0.11818062861778167,-0.0074654175265668815,0.21778145438649954,0.07183334085139498,-0.16707295364578964,-0.005403490048912503,0.006679328849596693,0.022457880636591834,-0.38010858398501035,-0.030434301841233365,-0.32658152093024206,-0.017116357470961317,0.020534382040635613,-0.055230529020833294,-0.09961537687010147,-0.12314940962931413,0.1323215781703142,0.1941599892647585,-0.05744557327619564],[7.707743744264454,1.1270960932961005,-1.8338415353001567,1.1586714724029963,1.3394055141288157,0.4497627687706474,0.26311138253896443,0.13377003164667706,0.09253540972341227,-0.08107828300741193,-0.7621474950102322,0.30728905801087125,0.12504686410195606,-0.16882414050828004,0.18070850695179366,-0.025739826707803556,0.034908460126393065,0.15356012045443207,-0.10638803585606581,-0.11284557024291553,0.053687326645651796,0.02309445617278492,0.07227961660782162,0.5752616113107092,-0.15155222905362756,-0.03677728287348121,0.11887701234255006,0.17187871496319274,-0.04998280653514195,0.08265855251686768,-0.47924293433747733,-0.11671971826996544,-0.28137706199952706,0.049896291560558026,0.1162937882821301,-0.18371909146480203,-0.10734868450584917,0.015303949117363142,-0.09478052314480308,0.01869322017014566,-0.016588811130537465,-0.22841458872648157,-0.03982177836146896,-0.09961507905341437,-0.12162605414519033,0.10209172830882862,-0.02325576394112946,0.023904670162672476,-0.0036004508645755017],[-15.420412228758527,-1.8859540090232392,4.701310247249656,-0.41424324862189355,1.6151636797229176,-0.36861340146323673,0.06308717312211479,-0.07698591834187367,-0.2901915211477906,-0.6323449133676414,-0.11226216708632404,-0.26422445190586796,-0.5064247149274488,-0.0735693063746247,-0.36005040625483314,-0.20230639162047853,-0.6008707488503882,-0.29089521692015463,-0.42297914087033556,0.1717044722760903,-0.8504406880327121,0.10671690127955026,-0.25143850715231203,-0.43384498751419814,-0.6079673692730232,-0.10901851905308005,-0.3423634907241492,0.10098609813929453,-0.16676751661903022,0.15953507415538135,-0.3699414542573698,-0.05667088674327671,-0.12636622081444585,-0.013131869989596394,-0.2617986854040268,-0.11073787812728741,-0.025270125056969947,0.03414054608695966,0.0428949414415503,0.07296713928504255,-0.08235877545374154,-0.25290050466215663,-0.2883045155921935,0.01830650314103434,0.043243732779153644,-0.18497460015860992,-0.2518929824426097,-0.11185184252516303,-0.12205245653152479],[-7.030687685368214,1.4870039692086328,3.3739942037891084,-1.1985146364601882,1.0567831799840315,0.0627151037688792,1.2091449457712604,0.7080316500790734,0.8289864781463611,0.6919492125572997,0.5082608720845098,0.6832268138633355,0.4659545083536927,0.04691998784453781,-0.1959048034050871,0.36925827202085326,0.22184964624757744,0.3213248749618951,-0.08874652122229831,0.2706980060688683,-0.04863112482277404,0.27763277886142357,-0.45304147640185405,0.3060842987379142,-0.5751309010177615,-0.2077986967040494,-0.42606937849985543,-0.27461065662947565,-0.4761463199817944,-0.07569931601560441,-0.3225931004089731,0.006349310330319532,-0.42128010162924456,0.11637427068168928,-0.5648664487427483,-0.12802007012857614,-0.25486263667676107,0.20458100159807902,-0.3115726437993952,-0.018478061809865793,-0.34894177561682116,0.12249458589677463,-0.08653047098912285,0.09860736734603312,-0.1878211167474094,-0.0849449326420186,-0.04305703165761274,0.17145143013745578,-0.13309028997724293],[-1.9480818054861324,-0.8921465743627388,4.656391712822964,3.0467342931633534,-0.2898766092437786,0.9435451701675008,-0.11658726856753315,-0.3348433499574186,0.6883957758603428,-0.4452671889568661,0.5104305354722302,0.23823862328428286,0.0036808601347732427,0.1611060075309054,0.3678189375330582,0.07072563729623464,0.17839814024056855,0.25940451268117887,0.2689411863153326,0.3245624649728828,0.1490849184448713,-0.21340276825516175,0.3105348772548227,0.049500671046833644,0.14080560192886662,-0.013348284283663284,0.19674541691509603,0.3505388723697709,0.47300413082358994,0.023385118433246067,0.1626862766780348,0.15846571411073007,-0.0012758393495534662,0.06823967739326971,0.23241664927700345,0.17593557972526874,0.10660011654730947,0.2584796117757804,-0.07073179306788731,0.20031395534682855,-0.15519851174105506,-0.10988035174303135,0.02931838632066629,-0.017426039889225305,-0.10169305499841258,-0.02073640288480785,0.13491078027977915,-0.13292570307406443,0.25610558338316436],[-7.900471560668985,-2.078192098364896,2.276934475266307,1.8932860765512418,0.8810468737262701,-0.12909412224661285,0.11710130256274301,-0.3615462666904473,-0.21425609820398522,-0.09658360498586614,0.2443725483372892,-0.2673224315426243,0.10473322932256564,-0.03608448454172913,0.16618901665474162,0.08731466559966826,-0.24726297927316135,-0.07799505568686754,0.007235072636443191,0.0379638998315415,0.07087598564743451,-0.23338474396592318,0.1827967070800117,-0.1988398728940736,0.23140200769504182,0.1676549827420931,-0.017247018831092597,0.2629271979963278,-0.07851309494087842,0.08834174887979498,-0.12340482178012742,0.1086177573991632,0.11551675562862175,0.07133612646206124,-0.2675349459906582,0.09300069171499996,-0.13752811178398583,0.24211715282066026,-0.06784696198597837,0.021402863314980203,-0.03507626544345209,-0.023285144864189943,0.0749327030849889,-0.17278502883048008,-0.03768183982098515,0.013857103875170744,-0.19415713394436485,-0.011637288656169937,-0.02267269046182497],[6.138594785115472,-0.17462577947316008,-8.345086514300302,-1.1602701188671385,1.6810115246058548,0.6744282504148604,2.5105612944776707,0.49395763828010525,1.863409823304968,0.5703814940906735,-0.16542213419184057,0.06147737929692024,-0.852510045526097,-0.1599495474516397,-0.06409456158334494,-0.5531238947372493,0.10671935909688827,-0.02077717074425225,-0.253753180128723,-0.26149323846902867,-1.0550418128057804,-0.44361813898893954,-0.6221260118744624,-0.6205314791120763,-0.4021304474319337,-0.7001226622585496,-0.3369390935424906,-0.4551700278544694,-0.4171954982771076,-0.6394142024912679,-0.5010031292871598,-0.43987864112356534,-0.2659769962408073,-0.5626652279282686,-0.21856531734044607,-0.5347408875853898,-0.1923436597881424,-0.4600438856706476,-0.42093982276368785,-0.5599257764917932,-0.7805479593364507,-0.6144600401612171,-0.40663421390562177,-0.6339610585839689,-0.3255921357735324,-0.46880649118089973,-0.38674694983381336,-0.28752483293475145,-0.28233365835228796],[-18.12899634889681,0.39087559375623826,2.730982846068485,0.07053620187615016,2.259615560521946,-0.5517870557947137,1.2677315679542591,-0.6167112667623695,1.2248605830419426,-0.2467630636025188,0.6367841759840038,-0.4750407640115378,0.9318358788277059,-0.17194132927000727,0.7411914602676715,-0.2041516969875223,1.401777964349713,-0.8369445085924587,1.3068411128405337,-0.44856660327605546,0.6437389036998519,-0.17448259873844488,1.4203288175571536,-0.007874163536007148,0.6327234794526106,-0.1341400064761836,0.6795924272159564,0.08733840640902737,0.772100100190146,0.22952581894753887,0.3853882479854996,0.3254913435357016,0.5395110867752436,0.4240878115229718,0.525956347992911,0.25721478481722004,0.26334627416993317,0.1388547260655849,0.40722444421824017,-0.06595679153581462,-0.009199710709500817,0.04755797496825226,-0.17431338022967,0.1495962662761955,-0.2804767584378895,0.04335520981444956,-0.13575768159584217,-0.08779613851119893,-0.20785900937866233],[5.620362106209121,-1.986611885382354,-4.861111896246808,-1.7039707266411015,-0.6341430156874961,-0.09071638900335412,-0.48893045162271453,0.1359189018630503,-0.558575997163765,-0.06695374531998241,-0.9076131906484362,0.23767107984993396,-0.182597471420378,-0.553040370163541,-0.40314574875308656,-0.2947216617035655,-0.054582265381671075,0.2043322863951768,-0.1997954220397399,0.054390524570929855,-0.4999647471012839,0.11160525446017583,-0.19849299222149072,-0.31672902280891074,-0.11733218212393347,0.21716820003545462,-0.24177063137956867,0.039462949184137765,0.08582418685832097,-0.07961609907362666,0.03660565661503862,0.007796891749883034,-0.21887307833148403,0.101391713891174,0.15186793198605528,-0.048216482126449335,-0.03642301867062438,-0.2071174239582022,0.02345897987947118,-0.026836293778895945,-0.14311182652428817,0.05163900904003669,-0.11736283842020405,-0.1775741394540602,0.14260941027515317,0.04300627749791199,-0.0316794472425435,-0.10835241759960428,-0.006969466208174112],[1.514916509498384,0.20328649968719265,0.4539907636222227,-0.1842736965412678,-1.5187892521679716,0.27290454677932785,0.6217949117840087,0.7118185842963679,-0.028150638959748578,-0.6612101629180056,0.7462798456425574,0.2484723293724798,0.005624099137981722,0.41650646104606076,0.3460858269167865,0.4656718097759141,0.5014957263341226,-0.13785099271252993,0.13031410373384442,0.3717343685625783,-0.20643700389072658,-0.13017750607649203,-0.40455159234663673,-0.1853876944856914,-0.12212094940764093,0.31686005218702745,-0.31063032129198553,-0.12467073975056273,-0.15460089685321837,0.01966584798521763,-0.27857874923079595,-0.14376301910006778,0.0531696287753814,-0.022151732473181727,-0.15130252726484844,0.1353065405710545,0.031255113492420064,-0.07018205049838412,0.07969149551557655,0.07374007675420925,-0.13840060848018368,-0.05981095645521048,-0.015136688322754468,-0.19172981889688248,0.03412782308585545,-0.11276315573755422,0.0021683241040144404,0.009890296493167099,0.021980966271127327],[-16.784576551984156,2.18733922453025,-0.36500611348443995,-0.5535055072798614,1.6142962838057673,1.4643587805710079,1.7149010485482175,1.258602681616307,0.30328673538264006,-0.049652515642822546,-0.6619570603155032,-0.24841471434032597,-0.4454238024853016,-0.08767078910865975,-0.440967303510839,-0.11176161847043167,-0.39011203415894985,-0.0844558504467504,-0.8758014037790077,-0.24031014131291376,-0.5839479715598264,-0.08974848925499719,-0.8749186800966889,-0.5890897309685172,-0.2577168227382667,-0.08486894783942034,-0.4728494925400076,-0.3615788052137389,-0.13177690254531965,-0.1840605219749006,-0.3476153757556969,-0.37634551850145903,-0.5816510710634375,-0.35169181300479546,-0.6482688377447752,-0.13109147283625242,-0.2631019608135657,-0.12289125220242146,-0.339960436485768,0.16643558221137514,-0.23858132178778313,0.24700502100426394,-0.6404461588805015,0.05031778256180592,-0.3083977731056762,0.05954818276251989,-0.07347539622590797,-0.019702148785598093,-0.428573538037478],[4.7315813864549785,2.2886951456457227,-0.7782427064182152,0.4987829772895577,2.4243027290472123,1.371100975092054,0.4438265029420355,0.8287079884656678,1.7715745121950535,0.36387902666892297,0.3771616164030971,-0.013311132344584005,0.7753191690997344,0.6800487117289502,1.0878809667906557,0.4737117972536368,0.5985570694318695,0.7331411279107716,0.7168734157277206,0.1754661147021733,0.1191754445647789,0.6682951160139018,0.6230618093379913,0.7797150403979567,0.7124413174130988,0.5557303239549701,-0.11214386059643419,0.6672794868222695,0.002892880949207775,0.10580886860584217,-0.0562602497312706,-0.07328228063719465,0.21197379832096233,-0.08739701873966417,0.11110352348183175,-0.1249771222754194,0.035812511674994045,0.2936206109638753,0.06431366586419765,-0.1373324004045771,0.01729953572558816,-0.22951380867788032,0.10769102102583236,0.05059945379680785,0.08628641817695198,0.13331082346087142,-0.143721164682037,-0.03201336819999885,0.03159087019122028],[4.882469630743767,-1.148346622837362,-1.1154380990755073,0.3668049234205892,1.7966168721381475,0.10078122701032095,0.8468503191663997,0.10108906842505946,-0.1750995998212868,0.23319748816737437,-0.2494578045712942,-0.31412493810192194,0.4405510809848806,0.35917495860647447,-0.11264765487883835,0.10080309800553319,-0.016963101735458112,-0.5176029007271166,0.04504546035355577,0.34466279952117695,0.2520616788304221,-0.1364043817335823,-0.08185338471321138,-0.1404418083377786,-0.17595479967405772,0.31740728668124885,-0.2424281445306762,0.26285651484844696,0.03014128219527652,0.14141889439096608,-0.13810637057020297,-0.1704419558407795,-0.20533115455338963,0.0032474309738971217,-0.05548751095680857,-0.0817270917337002,-0.2539133456033699,-0.007906608264789129,-0.13222645242032757,0.17985949467032136,-0.0625568108090335,0.14219247420584075,-0.030282862543843582,-0.009714963398825924,-0.14208769459031603,-0.09619427932336935,-0.20596696479172208,-0.022375908067531664,-0.055692899518882545],[2.2968984286085092,0.9679336609193665,2.1542630919298666,-0.6708778812691537,-0.14559169387627396,1.2956274678928863,0.4520030008073837,0.6315317681043486,0.36088171703738287,0.483513263069928,0.5176574966871202,0.006227876503850971,0.9291413773718351,-0.09539450446137265,0.3881568762109909,0.7650700361119337,0.6508501699360146,0.1257557224131957,0.49249962409584447,0.38459325100679653,0.555670853944636,0.15690521778871433,0.0943994572482867,0.1255135359379548,0.008319322712059216,-0.02179590622827315,-0.23313167506187107,0.3156128471144409,0.3107041706808104,-0.06781548126757486,-0.1712010007715799,-0.06138644931728284,0.009408674243536965,0.0855352163222349,0.015191226822746108,-0.055611303914923936,0.008435642016962999,-0.09625386841099413,0.1325894295487957,0.14544905233041455,-0.2409017356752054,0.13049358524171734,-0.18868580751843983,0.06835861695769263,-0.024389165570444522,0.14434641776826654,-0.04036760992710267,0.09700970230119677,0.04893459318203136],[-24.238376994558497,0.46057856291993726,2.667531447365895,0.8657009739358019,3.3163275148819813,1.0039623075385045,1.8820911803591678,-0.16749693738316793,0.4021718189182513,0.5887815702907288,0.8558454791039195,-0.019975166549346227,-0.03562818266592486,-0.04584595425128197,0.7412418317120821,-0.13592208888427565,0.6196695037551285,0.022888887286623815,0.44959775882006375,-0.42554336641543733,0.8383171400903395,-0.12935592727202025,0.7755346532201168,-0.1718025492009464,0.6136436673038766,-0.3633068566664105,0.9774005987193144,-0.2781313593436512,0.78542018177979,-0.334916298625545,0.6005048472299652,-0.20756068420309218,0.7661785811105194,-0.1329807876189202,0.625584412998955,-0.35046950643773195,0.3339600037660744,-0.0856340247424524,0.4600978186577065,-0.11921277785960724,0.3464996607267128,0.0643288915050063,0.5162038326681996,0.0858247812277339,0.2679421038913385,-0.061174387340441984,0.5092242834240607,0.023587016584926273,0.2331243717300576],[-13.290914043769398,0.7456046294504223,-4.563728381957902,-0.17256748220252482,-1.024477289001211,1.4522981818827874,-1.6343020028445994,-0.059607674218422405,-1.9634681758698511,0.04202370627109896,-2.0644074482175676,-0.6047359596369473,-1.4416257912701886,-0.16101529113988397,-1.016909115602884,-0.16814801259966644,-0.5505922179194964,-0.419838220142026,-0.3864304076202727,-0.5963979477849227,-0.2610110155608739,-0.24916477755593056,-0.2559785113333351,-0.5878894124038705,0.06010964379793602,-0.6106625754344078,0.11392644622700164,-0.6363559519430367,0.39341668757013604,-0.588402486098187,0.5201498029527364,-0.26278185847738184,0.3587317485681467,-0.4826551020560018,0.6509608785592426,-0.48858010231817994,0.5137163631402483,-0.4743274102950887,0.6396991619362463,-0.07844120900797323,0.619533199547795,-0.40009389870154344,0.7876753144975571,-0.39772676433839016,0.8022471246143505,-0.07361950251035924,0.5757135079318552,-0.08235294949831559,0.3859871486255291],[-18.13629725196967,2.9772086951130996,5.848264245218301,1.1576659512231766,0.016858524982676176,1.2165803386833858,0.32082610473190154,0.8776754391753243,0.9311098865735318,-0.47087056270238065,0.02482194383819647,-0.11802977238481908,-0.11363579313852085,0.3506094692661039,0.05176548548831519,0.5965315432299273,-0.10112426693228806,0.19063002513283073,0.3681760786261513,0.03771345472691528,0.2636665837139032,0.03486748307417803,0.4623191984987408,-0.08997194234281683,0.5181269458183981,0.05280916985412918,0.08312943368727999,0.1747392201280497,0.12127704988669918,0.0663770754754601,0.04323529509956012,0.07916706746059222,0.3437799714162929,0.16023666755094557,0.29129513853861644,0.1384991094339568,0.10266904989620715,0.20459870077156792,0.3542034938498693,0.15009727811005516,0.25833040826906056,0.1496361201409591,0.4417192145137082,0.08149418203710347,0.33958360130035453,0.03434392592130411,0.3105544815706354,0.12880782139964914,0.21280261950357604],[5.990626816212943,-0.33534848895576047,-4.641693865222684,-0.2563455474942391,1.9801054096758062,-0.9222650787286859,0.5281689736432954,-0.3399779124687467,-0.5674717145006111,0.4493285616299501,0.42922147765241975,-0.028614896058744614,-0.07215592293542233,-0.297114707277047,0.5745436002704936,-0.6249336499202104,-0.22487904530015054,-0.4552468078056978,-0.19059798634151623,-0.08438558947702142,0.38538207575745387,-0.031076820542843404,0.07080110307585052,-0.04218510530215491,0.335896759326251,-0.15020482513983083,0.023945718916528226,-0.1444534426327127,0.13311859786287172,0.2246127947984877,0.04360339503289941,-0.1447616054325789,0.173075216151117,0.1097057294515542,-0.13789407740780596,0.04570039935783277,0.10551453669690947,-0.06312972314967698,-0.009130866513309919,-0.13886006868651307,-0.018885342443385543,0.20368679902409792,0.24032665626386923,-0.04815415637077972,0.014024220190007925,0.02652192616262216,-0.04468810042861665,0.027224253421090475,0.004549260159428073],[41.75883796601411,-2.0016002467931195,-7.67989636770765,-0.619744351913029,-0.7108111784168548,-0.47283905569428486,-1.2420899889293324,0.5438145715364572,-0.6201053734792391,-0.35225293540867836,0.231623819709926,0.20337126270767927,-0.15221120744976047,0.3234579065759276,0.5174408762827434,-0.5385254121610439,-0.2299090953485171,-0.18563650930184114,-0.13751625557427305,-0.11075726628375454,-0.45025714576496895,0.41558796606165055,0.1511020484717248,-0.04708760830251516,-0.029955903496826992,0.0814346733310205,-0.16019982388536946,0.34157841100449476,0.03875367331779688,0.08285298670901901,-0.04055795107548351,-0.16828500025634918,-0.012255720342650898,-0.03219297617968594,0.06577796703327822,0.16822965917452845,0.03389622965943896,-0.18532335900871172,0.10348921573503235,-0.30676468279381514,-0.2609624422567845,0.3660318263242168,-0.05437867556669867,-0.11048762090616362,0.0965681510009602,0.16813883911315905,0.015873559411424022,-0.00712484457593095,-0.13606153044589778],[-1.8586167475857218,-1.5994052220511903,-0.46504089684504346,0.4171168415387222,0.043574162819783754,-0.9462724350921741,-0.754297812051284,-0.4848066943984693,-0.6784156253234743,-0.5438578070650334,-0.7345188944420735,0.15331398270628938,-0.8681902582095337,0.0896126798546674,-0.4357683872240355,-0.4254844178514666,-0.7111155620150869,-0.11915935289382373,-0.28412804413917075,-0.3708545479554677,-0.4423510124677013,0.14994063261298468,0.1387728525315296,0.2614394096632684,-0.15944785064996791,0.025505282573216728,0.21945389026607517,0.3049634285681453,0.11947241642048193,-0.09471268872342484,-0.05121953020277195,-0.05690541513864579,0.27540455744972087,0.25870508498103617,0.005288298962516103,0.2025103203596835,0.14451360110509262,0.37754133476531176,-0.0336669313493929,0.08155656351958887,0.2096772627556293,0.2150859919608355,0.08198257327091513,-0.023091915993969352,0.29205030227349255,0.14593478554154546,0.17272189233822138,0.2244477221476527,0.1137433618904951],[-10.962133073599782,-0.827115267965368,3.965829993860731,-0.2207204271628613,0.4150484053410306,-0.17101032504057384,-0.09980815013461104,0.03532970443867126,0.6637628449935867,0.13499374471658415,-0.0466044054280878,0.24560776143245902,-0.008306930047185931,-0.1740804497001551,-0.004186087932509003,-0.06146580777818355,-0.04672271714741052,-0.2565359328812775,0.08540851627219591,0.03237712158448727,-0.30912032613575413,-0.12543170679061774,0.28399662003153975,0.06950162290886942,0.19082665551055347,-0.2041301219525713,0.1110210965609091,0.2010968056306119,-0.15006282633934154,0.22352168429731978,0.07229229304881855,0.23726479482449436,0.053876444790139214,0.07912131839986462,-0.03282299078437145,0.17039721598360774,-0.04143941017289133,-0.05612743753943853,-0.15651376082997112,0.1505028662957205,0.16630777106294942,0.15668085212329558,-0.00896022759885151,0.025038497625111872,-0.014888925796005413,0.06623669251549057,-0.11056846377024185,-0.0963917390472232,-0.034227359766079296],[-33.94186903824348,3.1328539071086436,6.267171365334099,-1.2840036831775445,-1.3871664424323709,-1.055616451561248,-1.2813527872154118,-0.4455661894471558,-0.12167949384825943,-0.23047476698238115,-0.9525853744745748,-0.40645299528573753,-0.10221138454498616,-0.06950881947335245,-0.5520232325923649,0.07921019576346212,0.003714208940430233,-0.34072500401667677,-0.27185396927648803,-0.1929006721282657,0.06174970454808403,-0.16683926084493497,0.1498607443283614,0.0945900607652678,-0.11483720598134885,0.08707096121367078,0.04025178387457925,0.10334948724534083,0.07283100130988049,0.06818084417954667,0.13993819003876104,-0.2912584992539935,-0.392458764447784,-0.2899624525589546,-0.05072363440230302,0.15266374787938194,0.06444064994344056,0.21781768512844787,0.22025486198757435,0.27028454583257505,0.14878589244033838,-0.11207938830014129,-0.14432592777205075,-0.12667395609051255,-0.06558309705608083,-0.01065038680773199,-0.0680455497620334,0.004919825712362934,-0.12741377139391186],[-1.9168551400584162,-0.5492943346613455,0.40889448847066995,-1.3061589719468143,-1.2361731225269303,-0.6484475235525876,-0.9075589248243451,0.3359661145256958,-0.5463733576270028,-0.0622434533571516,-0.8537244765839261,0.12268302305241792,-0.6347807913573204,-0.0645101645464301,-0.247265580158142,0.12529009735388363,-0.3689494485510584,-0.04499885174706737,-0.39235220078297334,0.1340557391551421,-0.21334472298733814,0.10528210596719877,-0.13521030562422867,0.15595567419567868,0.046911935907292354,-0.002197484531566181,-0.16290954070933159,-0.1289661807748905,0.2729538700877677,0.04209797611429406,0.3350041601206758,-0.13350380271263587,-0.20967980546240245,-0.07707890397198539,0.38645431298725147,-0.009528560429971564,-0.0664982469666969,-0.1338197415862228,0.057566036892485795,-0.003001180120182744,0.2510267254912364,0.07086778116637463,-0.05626846625531615,0.18239777276238148,-0.027831467382336994,0.23660746029964702,0.32623724737380977,0.09002005246035423,0.3276728331672889]],"nearest_scaled_distance_max":0.6058514689405025,"training":{"train_count":384,"tune_count":96,"test_count":128,"seeds":{"train":142017,"tune":189173,"test":231119},"native_phases":129,"triangles":800,"primary_mass_solar":1.2,"primary_teff_k":6300.0,"passband":"Johnson:V","native_artifact_sha256":"dde66290f0d8f8a1562b98909e34d534dba8c175db032d04de4f69296e233d03","baseline_commit":"b7ce074c8231cc8db9bc2e4016b667071daaa845","tune_rms":0.0010109659703225425,"test_metrics":[{"regime":"all","n":128,"rms":0.0010765244010697932,"curve_rms_p95":0.0018063431866725721,"abs_p95":0.0020898943015142377,"worst_abs":0.025301453673032215},{"regime":"grazing","n":42,"rms":0.0010671235291669762,"curve_rms_p95":0.0019262284395053304,"abs_p95":0.001982696611178935,"worst_abs":0.009169286608528449},{"regime":"shallow_contact","n":23,"rms":0.0012784069172370265,"curve_rms_p95":0.0017203887518395524,"abs_p95":0.0024087972960971644,"worst_abs":0.009918346889248308},{"regime":"deep_contact","n":29,"rms":0.001146891447099439,"curve_rms_p95":0.0018059774353100795,"abs_p95":0.0024474662436348993,"worst_abs":0.011044638659025885},{"regime":"unequal_mass","n":46,"rms":0.0010749815430450575,"curve_rms_p95":0.0016701016411015993,"abs_p95":0.002058957751826862,"worst_abs":0.025301453673032215}],"old_test_metrics":[{"regime":"all","n":128,"rms":0.0020452918795599356,"curve_rms_p95":0.00393016312906818,"abs_p95":0.0044296933712471885,"worst_abs":0.02367385851144216},{"regime":"grazing","n":42,"rms":0.0022048417914313644,"curve_rms_p95":0.004035500089749699,"abs_p95":0.0047613468953603515,"worst_abs":0.008951145368800928},{"regime":"shallow_contact","n":23,"rms":0.002519933414316346,"curve_rms_p95":0.0038151005383075002,"abs_p95":0.005238370486929255,"worst_abs":0.022052387952884667},{"regime":"deep_contact","n":29,"rms":0.0024806526097668835,"curve_rms_p95":0.0040559317506171306,"abs_p95":0.00514763648153016,"worst_abs":0.02367385851144216},{"regime":"unequal_mass","n":46,"rms":0.0025403114718972226,"curve_rms_p95":0.004151966745800771,"abs_p95":0.005231607621146915,"worst_abs":0.02367385851144216}],"response_metrics":[{"anchor":0,"parameter":"inclination","native_response_rms":0.004395239517005318,"difference_rms":0.00033573670370576906,"cosine":0.9972371665083578,"relative_norm_error":0.015037577245698867},{"anchor":0,"parameter":"mass_ratio","native_response_rms":0.00346706646850713,"difference_rms":0.001488505688716323,"cosine":0.9148561830433595,"relative_norm_error":0.2310328785514424},{"anchor":0,"parameter":"fillout_factor","native_response_rms":0.002836188625338927,"difference_rms":0.000702237190970658,"cosine":0.9855374387574427,"relative_norm_error":0.19498751719372298},{"anchor":0,"parameter":"temperature_ratio","native_response_rms":0.003928401714091917,"difference_rms":0.00020919964649530884,"cosine":0.9986730639553638,"relative_norm_error":0.014883610487510655},{"anchor":1,"parameter":"inclination","native_response_rms":0.0018573033053684836,"difference_rms":0.000614603370327048,"cosine":0.9846798169368723,"relative_norm_error":0.2659215040579021},{"anchor":1,"parameter":"mass_ratio","native_response_rms":0.0014711832912181436,"difference_rms":0.0006457032399818279,"cosine":0.9425368177500789,"relative_norm_error":0.2271583136224771},{"anchor":1,"parameter":"fillout_factor","native_response_rms":0.001581638216920194,"difference_rms":0.0010125925852356487,"cosine":0.846489716422904,"relative_norm_error":0.20205037867341927},{"anchor":1,"parameter":"temperature_ratio","native_response_rms":0.009263510140423252,"difference_rms":0.0005554039990034865,"cosine":0.9982105227863465,"relative_norm_error":0.0025674125142937485},{"anchor":2,"parameter":"inclination","native_response_rms":0.0023097352135490598,"difference_rms":0.0006313395687996101,"cosine":0.9676938241279175,"relative_norm_error":0.0732650097582327},{"anchor":2,"parameter":"mass_ratio","native_response_rms":0.004820309251796688,"difference_rms":0.0007114946169899529,"cosine":0.9900984703407951,"relative_norm_error":0.055528060632248404},{"anchor":2,"parameter":"fillout_factor","native_response_rms":0.002076680226911129,"difference_rms":0.00037504726952938106,"cosine":0.9840176571787973,"relative_norm_error":0.04609719789957297},{"anchor":2,"parameter":"temperature_ratio","native_response_rms":0.009727466867462604,"difference_rms":0.00017467272387299617,"cosine":0.9998467408617454,"relative_norm_error":0.0038400493940367398},{"anchor":3,"parameter":"inclination","native_response_rms":0.005731811241844056,"difference_rms":0.00014349005075828028,"cosine":0.9996901275650507,"relative_norm_error":0.0029653064542677363},{"anchor":3,"parameter":"mass_ratio","native_response_rms":0.001398950561062697,"difference_rms":0.0008999024967515012,"cosine":0.770119136252687,"relative_norm_error":0.3128228969071001},{"anchor":3,"parameter":"fillout_factor","native_response_rms":0.0023895099772479607,"difference_rms":0.0004825741845641139,"cosine":0.979675091232266,"relative_norm_error":0.003110624970920376},{"anchor":3,"parameter":"temperature_ratio","native_response_rms":0.00743709481459411,"difference_rms":0.0001754542410164124,"cosine":0.9997557207795333,"relative_norm_error":0.008006351719632581}],"old_response_metrics":[{"anchor":0,"parameter":"inclination","native_response_rms":0.004395239517005318,"difference_rms":0.0008086211224942313,"cosine":0.983624610284823,"relative_norm_error":0.02056719995291525},{"anchor":0,"parameter":"mass_ratio","native_response_rms":0.00346706646850713,"difference_rms":0.0020778199502984483,"cosine":0.8700261752420484,"relative_norm_error":0.4707207411293417},{"anchor":0,"parameter":"fillout_factor","native_response_rms":0.002836188625338927,"difference_rms":0.0016884448492935332,"cosine":0.9272042138963149,"relative_norm_error":0.5355221319679989},{"anchor":0,"parameter":"temperature_ratio","native_response_rms":0.003928401714091917,"difference_rms":0.0014567189240589193,"cosine":0.9312916147836442,"relative_norm_error":0.0006419397677430184},{"anchor":1,"parameter":"inclination","native_response_rms":0.0018573033053684836,"difference_rms":0.0009001162854414183,"cosine":0.9623111512182655,"relative_norm_error":0.3634531832849848},{"anchor":1,"parameter":"mass_ratio","native_response_rms":0.0014711832912181436,"difference_rms":0.0008932460621607191,"cosine":0.8830782109901661,"relative_norm_error":0.2683993572194098},{"anchor":1,"parameter":"fillout_factor","native_response_rms":0.001581638216920194,"difference_rms":0.0009599336313303964,"cosine":0.8726008660736949,"relative_norm_error":0.23286269039162605},{"anchor":1,"parameter":"temperature_ratio","native_response_rms":0.009263510140423252,"difference_rms":0.0012450270949915052,"cosine":0.9944693604936947,"relative_norm_error":0.08939360980434036},{"anchor":2,"parameter":"inclination","native_response_rms":0.0023097352135490598,"difference_rms":0.0006085916646086487,"cosine":0.9681386471150624,"relative_norm_error":0.05011006672244212},{"anchor":2,"parameter":"mass_ratio","native_response_rms":0.004820309251796688,"difference_rms":0.0008518761284073309,"cosine":0.9909819711022313,"relative_norm_error":0.12424608183893923},{"anchor":2,"parameter":"fillout_factor","native_response_rms":0.002076680226911129,"difference_rms":0.000414822288736388,"cosine":0.9799415936163823,"relative_norm_error":0.03371969503398597},{"anchor":2,"parameter":"temperature_ratio","native_response_rms":0.009727466867462604,"difference_rms":0.0003666783022413983,"cosine":0.9995524299721531,"relative_norm_error":0.022486780590475007},{"anchor":3,"parameter":"inclination","native_response_rms":0.005731811241844056,"difference_rms":0.0001947892372002483,"cosine":0.9994230258155328,"relative_norm_error":0.0005584746581162824},{"anchor":3,"parameter":"mass_ratio","native_response_rms":0.001398950561062697,"difference_rms":0.0009228431937893803,"cosine":0.7552817622994632,"relative_norm_error":0.3196359933126855},{"anchor":3,"parameter":"fillout_factor","native_response_rms":0.0023895099772479607,"difference_rms":0.0006347185697458775,"cosine":0.9673060259815116,"relative_norm_error":0.1116793914445799},{"anchor":3,"parameter":"temperature_ratio","native_response_rms":0.00743709481459411,"difference_rms":0.00039297404785975065,"cosine":0.9986694592092202,"relative_norm_error":0.01019010757468836}]}}''')

In [ ]:
"""Portable interactive PHOEBE fitter used by the generated Colab notebook.

The notebook embeds this source and the frozen preview artifacts.  Native PHOEBE
curves are the only curves allowed to populate fit statistics and exports.
"""
from __future__ import annotations

import base64
import io
import json
import math
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from scipy.optimize import least_squares


GEOMETRIES = ("detached", "semi_primary", "semi_secondary", "contact")
LABELS = {
    "detached": "Detached",
    "semi_primary": "Semi-detached",
    "semi_secondary": "Semi-detached",
    "contact": "Contact",
}
DEFAULTS = {
    "geometry": "detached", "donor": "primary", "inclination": 84.5,
    "radius_primary": .21, "radius_secondary": .17, "free_fill": .55,
    "fillout_factor": .25, "temperature_ratio": .84, "mass_ratio": .75,
    "eccentricity": 0., "omega": 90., "period": 1.8, "epoch": 0.,
    "distortion": False, "irradiation": False,
}
BOUNDS = {
    "inclination": (0., 90.), "radius_primary": (.10, .32),
    "radius_secondary": (.08, .29), "free_fill": (.25, .82),
    "fillout_factor": (.10, .50), "temperature_ratio": (.58, 1.14),
    "mass_ratio": (.20, 1.50), "eccentricity": (0., .35),
    "omega": (0., 360.), "period": (.1, 100.),
}
FIT_STEPS = {
    "inclination": .025, "radius_primary": .0002, "radius_secondary": .0002,
    "free_fill": .001, "fillout_factor": .001, "temperature_ratio": .001,
    "mass_ratio": .002, "eccentricity": .0002, "omega": .1,
    "period": .0004,
}


def geometry_name(parameters):
    return str(parameters.get("geometry", "detached"))


def donor_name(parameters):
    geometry = geometry_name(parameters)
    return "secondary" if geometry == "semi_secondary" else "primary"


def allowed_fit_keys(geometry):
    if geometry == "contact":
        return ("inclination", "fillout_factor", "temperature_ratio", "mass_ratio", "period")
    if geometry.startswith("semi_"):
        return ("inclination", "free_fill", "temperature_ratio", "mass_ratio", "period")
    return ("inclination", "radius_primary", "radius_secondary", "temperature_ratio",
            "mass_ratio", "eccentricity", "omega", "period")


def validate_parameters(values):
    p = dict(DEFAULTS, **values)
    if p["geometry"] not in GEOMETRIES:
        raise ValueError("Choose Detached, Semi-detached, or Contact geometry.")
    p["donor"] = donor_name(p)
    for key, (low, high) in BOUNDS.items():
        value = float(p[key])
        if not math.isfinite(value) or not low <= value <= high:
            raise ValueError(f"{key.replace('_', ' ')} must be between {low} and {high}.")
        p[key] = value
    for key in ("distortion", "irradiation"):
        p[key] = bool(p[key])
    if p["geometry"] != "detached" and p["eccentricity"] != 0:
        raise ValueError("Semi-detached and contact modes currently require a circular orbit.")
    return p


def _rbf_preview(spec, values, model):
    scaled = []
    for key in spec["keys"]:
        low, high = spec["bounds"][key]
        value = float(values[key])
        if not low <= value <= high:
            raise ValueError(f"{key.replace('_', ' ')} is outside the trained preview range {low}–{high}.")
        scaled.append(2 * (value - low) / (high - low) - 1)
    scaled = np.asarray(scaled)
    centers = np.asarray(model["centers"])
    distance = np.sqrt(np.sum((centers - scaled) ** 2, axis=1))
    if distance.min() > model["nearest_scaled_distance_max"]:
        raise ValueError("This point is too far from a training system; run native PHOEBE.")
    coefficients = np.exp(-model["gamma"] * distance ** 2) @ np.asarray(model["weights"])
    phase = np.linspace(-.5, .5, spec["phases"] + 1)
    basis = np.cos(2 * np.pi * np.outer(phase, np.arange(spec["harmonics"] + 1)))
    flux = basis @ coefficients
    return phase, flux / np.median(flux[:-1])


def _overlap(distance, back_radius, front_radius):
    if distance >= back_radius + front_radius:
        return 0.
    if distance <= abs(back_radius - front_radius):
        return min(back_radius, front_radius) ** 2
    ca = np.clip((distance**2 + back_radius**2 - front_radius**2) /
                 (2 * distance * back_radius), -1., 1.)
    cb = np.clip((distance**2 + front_radius**2 - back_radius**2) /
                 (2 * distance * front_radius), -1., 1.)
    radical = ((back_radius + front_radius - distance) *
               (distance + back_radius - front_radius) *
               (distance - back_radius + front_radius) *
               (distance + back_radius + front_radius))
    return (back_radius**2 * np.arccos(ca) + front_radius**2 * np.arccos(cb) -
            .5 * np.sqrt(max(0., radical))) / np.pi


def _detached_preview(spec, p):
    fixed = spec["fixed"]
    for key in ("mass_ratio", "eccentricity", "omega", "period"):
        if not np.isclose(p[key], fixed[key]):
            raise ValueError(f"Detached trained preview requires {key}={fixed[key]}; run native PHOEBE.")
    if p["distortion"] or p["irradiation"]:
        raise ValueError("Detached trained preview is spherical with irradiation off; run native PHOEBE.")
    for key, (low, high) in zip(spec["keys"], spec["bounds"]):
        if not low <= p[key] <= high:
            raise ValueError(f"{key.replace('_', ' ')} is outside the trained preview range {low}–{high}.")
    table = spec["table"]
    teffs = np.asarray(table["teffs"]); gravities = np.asarray(table["gravities"])
    radii = np.asarray(table["radii"]); rings = len(radii) - 1
    values = np.frombuffer(base64.b64decode(table["log_intensity_f32le"]), dtype="<f4")
    values = values.reshape(len(teffs), len(gravities), rings)
    areas = radii[1:]**2 - radii[:-1]**2

    def interval(axis, value):
        if value < axis[0] or value > axis[-1]:
            raise ValueError("The detached preview is outside its atmosphere table.")
        index = min(np.searchsorted(axis, value, side="right") - 1, len(axis) - 2)
        index = max(0, index)
        return index, (value - axis[index]) / (axis[index + 1] - axis[index])

    def profile(teff, gravity, radius):
        i, u = interval(teffs, teff); j, v = interval(gravities, gravity)
        a = (1 - v) * values[i, j] + v * values[i, j + 1]
        b = (1 - v) * values[i + 1, j] + v * values[i + 1, j + 1]
        intensity = np.exp((1 - u) * a + u * b)
        weights = intensity - np.r_[intensity[1:], 0.]
        return {"radius": radius, "weights": weights,
                "light": float(np.sum(intensity * areas) * radius**2)}

    def blocked(distance, back, front):
        if distance >= back["radius"] + front["radius"]:
            return 0.
        if distance + back["radius"] <= front["radius"]:
            return back["light"]
        separation = distance / back["radius"]
        size = front["radius"] / back["radius"]
        fraction = sum(_overlap(separation, radii[k + 1], size) * back["weights"][k]
                       for k in range(rings))
        return fraction * back["radius"]**2

    r1, r2 = p["radius_primary"], p["radius_secondary"]
    star1 = profile(table["primary_teff"], table["base_logg"][0] - 2*np.log10(r1), r1)
    star2 = profile(table["primary_teff"] * p["temperature_ratio"],
                    table["base_logg"][1] - 2*np.log10(r2), r2)
    total = star1["light"] + star2["light"]
    n = int(spec["n"]); phase = np.linspace(-.5, .5, n + 1); flux = np.empty(n + 1)
    cosine = math.cos(math.radians(p["inclination"]))
    for j in range(n // 4 + 1):
        angle = 2 * np.pi * j / n
        distance = math.hypot(math.sin(angle), cosine * math.cos(angle))
        primary = 1 - blocked(distance, star1, star2) / total
        secondary = 1 - blocked(distance, star2, star1) / total
        flux[n//2-j] = flux[n//2+j] = primary
        flux[j] = flux[n-j] = secondary
    return phase, flux


class PreviewModels:
    def __init__(self, detached, semi, contact):
        self.detached = detached; self.semi = semi; self.contact = contact

    def curve(self, parameters):
        p = validate_parameters(parameters); geometry = p["geometry"]
        if geometry == "detached":
            return _detached_preview(self.detached, p)
        if geometry == "contact":
            fixed = self.contact["fixed"]
            if not np.isclose(p["period"], fixed["period"]) or p["irradiation"]:
                raise ValueError("Contact preview requires P=1.8 d and irradiation off; run native PHOEBE.")
            values = {key: p[key] for key in self.contact["keys"]}
            return _rbf_preview(self.contact, values, self.contact)
        fixed = self.semi["fixed"]
        if not np.isclose(p["period"], fixed["period"]) or p["irradiation"]:
            raise ValueError("Semi-detached preview requires P=1.8 d and irradiation off; run native PHOEBE.")
        values = {"inclination": p["inclination"], "mass_ratio": p["mass_ratio"],
                  "free_fill": p["free_fill"], "temperature_ratio": p["temperature_ratio"]}
        return _rbf_preview(self.semi, values, self.semi["donors"][p["donor"]])


@dataclass
class ObservationSet:
    x: np.ndarray
    y: np.ndarray
    sigma: np.ndarray | None
    x_kind: str
    y_kind: str
    name: str


def parse_csv_bytes(content, name="observations.csv"):
    if isinstance(content, str):
        content = content.encode()
    try:
        data = np.genfromtxt(io.StringIO(content.decode("utf-8-sig")), delimiter=",",
                             names=True, dtype=float, encoding=None)
    except Exception as error:
        raise ValueError(f"Could not read CSV: {error}") from error
    names = list(data.dtype.names or [])
    if len(names) < 2:
        raise ValueError("CSV needs named phase/time and flux/magnitude columns.")
    lower = {key.lower().strip(): key for key in names}
    x_key = next((lower[k] for k in ("phase", "time", "bjd", "hjd", "jd") if k in lower), None)
    y_key = next((lower[k] for k in ("flux", "normalized_flux", "brightness", "magnitude", "mag") if k in lower), None)
    if x_key is None or y_key is None:
        raise ValueError("Use a phase or time column and a flux or magnitude column.")
    sigma_key = next((lower[k] for k in ("sigma", "error", "uncertainty", "flux_error", "mag_error") if k in lower), None)
    x = np.atleast_1d(np.asarray(data[x_key], dtype=float)); y = np.atleast_1d(np.asarray(data[y_key], dtype=float))
    sigma = None if sigma_key is None else np.atleast_1d(np.asarray(data[sigma_key], dtype=float))
    if not 8 <= len(x) <= 5000 or len(y) != len(x):
        raise ValueError("Load 8–5,000 matching observations.")
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError("Observations contain missing or non-numeric values.")
    if sigma is not None and (len(sigma) != len(x) or not np.isfinite(sigma).all() or np.any(sigma <= 0)):
        raise ValueError("Uncertainties must be finite, positive, and match the observations.")
    return ObservationSet(x, y, sigma, "phase" if x_key.lower() == "phase" else "time",
                          "magnitude" if y_key.lower() in ("magnitude", "mag") else "flux", name)


def observation_arrays(observations, parameters, sort=True):
    p = validate_parameters(parameters); x = observations.x.copy(); y = observations.y.copy()
    sigma = None if observations.sigma is None else observations.sigma.copy()
    if observations.y_kind == "magnitude":
        flux = 10 ** (-.4 * (y - np.median(y)))
        if sigma is not None:
            sigma = .4 * np.log(10.) * flux * sigma
    else:
        flux = y
    if not np.isfinite(flux).all() or np.any(flux <= 0):
        raise ValueError("Flux values must be finite and positive.")
    scale = np.median(flux); flux = flux / scale
    if sigma is not None:
        sigma = sigma / scale
    phase = x if observations.x_kind == "phase" else (x - p["epoch"]) / p["period"]
    phase = (phase + .5) % 1 - .5
    if not sort:
        return phase, flux, sigma
    order = np.argsort(phase)
    return phase[order], flux[order], None if sigma is None else sigma[order]


def compare_curve(model_phase, model_flux, observations, parameters, fitted_count=0, sort=True):
    phase, flux, sigma = observation_arrays(observations, parameters, sort=sort)
    predicted = np.interp(phase, model_phase, model_flux)
    weights = np.ones_like(flux) if sigma is None else 1 / sigma**2
    normalization = float(np.sum(weights * flux * predicted) / np.sum(weights * predicted**2))
    predicted = normalization * predicted; residual = flux - predicted
    dof = len(flux) - fitted_count - 1
    reduced = None if sigma is None or dof <= 0 else float(np.sum((residual / sigma)**2) / dof)
    return {"phase": phase, "flux": flux, "sigma": sigma, "model_at_data": predicted,
            "residual": residual, "rms": float(np.sqrt(np.mean(residual**2))),
            "reduced_chi2": reduced, "normalization": normalization}


class NativeBackend:
    def __init__(self, phoebe_module=None):
        if phoebe_module is None:
            import phoebe as phoebe_module
        self.phoebe = phoebe_module; self.bundles = {}; self.surface_bundles = {}

    def _bundle(self, p, phases, triangles):
        geometry = p["geometry"]
        if geometry not in self.bundles:
            if geometry == "contact":
                b = self.phoebe.default_contact_binary()
                b.flip_constraint("mass@primary", solve_for="sma@binary")
                b.set_value("mass", component="primary", value=1.2)
                b.flip_constraint("pot@contact_envelope", solve_for="requiv@primary")
                b.flip_constraint("fillout_factor@contact_envelope", solve_for="pot@contact_envelope")
            elif geometry.startswith("semi_"):
                b = self.phoebe.default_binary(semidetached=p["donor"])
                b.flip_constraint("mass@primary", solve_for="sma@binary")
                b.set_value("mass", component="primary", value=1.2)
            else:
                b = self.phoebe.default_binary()
                b.flip_constraint("mass@primary", solve_for="sma@binary")
                b.set_value("mass", component="primary", value=1.2)
            b.add_dataset("lc", dataset="fitter_lc", compute_phases=phases, passband="Johnson:V")
            self.bundles[geometry] = b
        b = self.bundles[geometry]
        b.set_value("compute_phases", dataset="fitter_lc", context="dataset", value=phases)
        b.set_value_all("ntriangles", value=int(triangles))
        return b

    def curve(self, parameters, phases, triangles=800):
        p = validate_parameters(parameters); phases = np.asarray(phases, dtype=float)
        b = self._bundle(p, phases, triangles)
        b.set_value("period", component="binary", value=p["period"])
        b.set_value("q", component="binary", value=p["mass_ratio"])
        if p["geometry"] == "contact":
            b.set_value("fillout_factor", component="contact_envelope", context="component",
                        value=p["fillout_factor"])
        elif p["geometry"].startswith("semi_"):
            free = "secondary" if p["donor"] == "primary" else "primary"
            maximum = b.get_value("requiv_max", component=free, context="component")
            b.set_value("requiv", component=free, context="component", value=p["free_fill"] * maximum)
        else:
            separation = b.get_value("sma", component="binary", context="component")
            b.set_value("requiv", component="primary", value=p["radius_primary"] * separation)
            b.set_value("requiv", component="secondary", value=p["radius_secondary"] * separation)
            b.set_value("ecc", component="binary", value=p["eccentricity"])
            b.set_value("per0", component="binary", value=p["omega"])
            b.set_value_all("distortion_method", value="roche" if p["distortion"] else "sphere")
        b.set_value("teff", component="primary", value=6300.)
        b.set_value("teff", component="secondary", value=6300. * p["temperature_ratio"])
        b.set_value("incl", component="binary", value=p["inclination"])
        b.set_value("irrad_method", value="horvat" if p["irradiation"] else "none")
        b.run_compute(model="fitter_model", overwrite=True, progressbar=False)
        flux = np.asarray(b.get_value("fluxes", dataset="fitter_lc", model="fitter_model"), dtype=float)
        if not np.isfinite(flux).all() or np.any(flux <= 0):
            raise ValueError("Native PHOEBE did not produce a finite positive curve.")
        return phases, flux / np.median(flux)

    def surface(self, parameters, triangles=500):
        """Build a native phase-zero co-rotating constrained surface."""
        p = validate_parameters(parameters); geometry = p["geometry"]
        if geometry == "detached":
            raise ValueError("This native surface animator covers semi-detached and contact geometry.")
        if geometry not in self.surface_bundles:
            if geometry == "contact":
                b = self.phoebe.default_contact_binary()
                b.flip_constraint("mass@primary", solve_for="sma@binary")
                b.set_value("mass", component="primary", value=1.2)
                b.flip_constraint("pot@contact_envelope", solve_for="requiv@primary")
                b.flip_constraint("fillout_factor@contact_envelope", solve_for="pot@contact_envelope")
            else:
                b = self.phoebe.default_binary(semidetached=p["donor"])
                b.flip_constraint("mass@primary", solve_for="sma@binary")
                b.set_value("mass", component="primary", value=1.2)
            b.add_dataset("mesh", dataset="fitter_mesh", compute_phases=[0.], columns=["xs","ys","zs"])
            self.surface_bundles[geometry] = b
        b = self.surface_bundles[geometry]
        b.set_value("period", component="binary", value=p["period"])
        b.set_value("q", component="binary", value=p["mass_ratio"])
        if geometry == "contact":
            b.set_value("fillout_factor", component="contact_envelope", context="component", value=p["fillout_factor"])
        else:
            free = "secondary" if p["donor"] == "primary" else "primary"
            maximum = b.get_value("requiv_max", component=free, context="component")
            b.set_value("requiv", component=free, context="component", value=p["free_fill"] * maximum)
        b.set_value("teff", component="primary", value=6300.)
        b.set_value("teff", component="secondary", value=6300.*p["temperature_ratio"])
        b.set_value("incl", component="binary", value=p["inclination"])
        actual = int(triangles)
        while True:
            b.set_value_all("ntriangles", value=actual)
            try:
                b.run_compute(model="fitter_surface", overwrite=True, progressbar=False)
                break
            except TypeError as error:
                if geometry != "contact" or actual >= 1000 or "Mesh failed or incomplete" not in str(error): raise
                actual = 1000
        components=[]
        for star in b.hierarchy.get_stars():
            values=np.asarray(b.get_value("xyz_elements",component=star,dataset="fitter_mesh",model="fitter_surface"),dtype=float)
            if values.ndim!=3 or values.shape[1:]!=(3,3) or not np.isfinite(values).all():
                raise ValueError("PHOEBE returned an incomplete surface mesh.")
            components.append({"name":star,"triangles":values})
        result={"schema":1,"source":f"native PHOEBE {self.phoebe.__version__}",
                "geometry":"contact" if geometry=="contact" else "semi","donor":p["donor"],
                "frame":"primary_origin" if geometry=="contact" else "component_local",
                "phase":0.,"triangles_requested":int(triangles),"triangles_actual":actual,
                "components":components}
        if geometry=="contact":
            import libphoebe
            pot=float(b.get_value("pot",component="contact_envelope",context="component"))
            result["neck"]=float(libphoebe.roche_contact_neck_min(np.pi/2,p["mass_ratio"],1.,pot)["xmin"])
        return result


def project_surface(mesh, parameters, phase=0., view="earth"):
    """Project a cached co-rotating native mesh at an arbitrary orbit phase."""
    p=validate_parameters(parameters); angle=2*np.pi*float(phase)
    # At phase zero the companion lies along +y in the overhead convention.
    cosine=-np.sin(angle); sine=np.cos(angle)
    ci=np.cos(np.radians(p["inclination"]));si=np.sin(np.radians(p["inclination"]))
    centers=(-p["mass_ratio"]/(1+p["mass_ratio"]),1/(1+p["mass_ratio"]))
    projected=[]
    for component,part in enumerate(mesh["components"]):
        triangles=np.asarray(part["triangles"],dtype=float)
        if triangles.ndim!=3 or triangles.shape[1:]!=(3,3):raise ValueError("Invalid native surface triangles.")
        values=triangles.copy()
        if mesh["frame"]=="primary_origin": values[...,0]-=p["mass_ratio"]/(1+p["mass_ratio"]);sign=1;center=0.
        else: sign=1 if component==0 else -1;center=centers[component]
        u,v,z=values[...,0],values[...,1],values[...,2]
        across=sign*(u*cosine-v*sine);along=sign*(u*sine+v*cosine)
        if mesh["frame"]=="component_local":
            center_across=center*cosine;center_along=center*sine
            across+=center_across;along+=center_along
        if view=="above": x,y,depth=-across,along,z
        else: x,y,depth=across,-along*ci+z*si,along*si+z*ci
        points=np.stack((x,y,depth),axis=-1)
        for triangle in points: projected.append({"component":component,"points":triangle,"depth":float(triangle[:,2].mean())})
    return sorted(projected,key=lambda triangle:triangle["depth"])


def run_native_compute(backend, observations, parameters, final_phases=513, final_triangles=1500):
    p = validate_parameters(parameters); phase = np.linspace(-.5, .5, int(final_phases))
    started = time.perf_counter(); x, flux = backend.curve(p, phase, final_triangles)
    comparison = compare_curve(x, flux, observations, p)
    return {"parameters": p, "phases": x, "model": flux * comparison["normalization"],
            "comparison": comparison, "fit": None, "seconds": time.perf_counter() - started,
            "source": "native PHOEBE"}


def run_native_fit(backend, observations, parameters, keys, fit_phases=65, max_nfev=8,
                   final_phases=513, search_triangles=None, final_triangles=1500, progress=None):
    initial = validate_parameters(parameters); geometry = initial["geometry"]
    keys = tuple(keys)
    if not keys or len(set(keys)) != len(keys) or not set(keys) <= set(allowed_fit_keys(geometry)):
        raise ValueError("Choose one or more parameters supported for this geometry.")
    if "omega" in keys and initial["eccentricity"] == 0 and "eccentricity" not in keys:
        raise ValueError("Fit eccentricity with periastron angle when starting from a circular orbit.")
    count = int(fit_phases)
    if count not in (65, 129):
        raise ValueError("Fit phases must be 65 or 129; final validation always uses 513.")
    triangles = search_triangles or (800 if geometry != "detached" else 400)
    phase_grid = np.linspace(-.5, .5, count)
    lower = np.asarray([BOUNDS[key][0] for key in keys], dtype=float)
    upper = np.asarray([BOUNDS[key][1] for key in keys], dtype=float)
    if geometry == "contact":
        for key in ("inclination", "mass_ratio", "fillout_factor", "temperature_ratio"):
            if key in keys:
                low, high = initial_preview_bounds = {
                    "inclination": (65., 90.), "mass_ratio": (.45, 1.2),
                    "fillout_factor": (.1, .5), "temperature_ratio": (.78, 1.12),
                }[key]; i = keys.index(key); lower[i], upper[i] = low, high
    elif geometry.startswith("semi_"):
        preview_bounds = {"inclination": (65.,90.), "mass_ratio": (.4,1.3),
                          "free_fill": (.25,.82), "temperature_ratio": (.76,1.14)}
        for key in keys:
            if key in preview_bounds:
                i = keys.index(key); lower[i], upper[i] = preview_bounds[key]
    best = {"cost": float("inf"), "parameters": initial}; evaluations = 0
    _, _, sigma = observation_arrays(observations, initial, sort=False)

    def evaluate(values):
        nonlocal evaluations
        p = dict(initial, **dict(zip(keys, map(float, values))))
        if "omega" in keys:
            p["omega"] %= 360.
        _, model = backend.curve(p, phase_grid, triangles)
        comparison = compare_curve(phase_grid, model, observations, p, len(keys), sort=False)
        residual = comparison["residual"] if sigma is None else comparison["residual"] / sigma
        cost = float(np.dot(residual, residual)); evaluations += 1
        if cost < best["cost"]:
            best.update(cost=cost, parameters=p)
        if progress is not None:
            progress(evaluations, comparison["rms"], p)
        return residual

    def jacobian(x):
        base = evaluate(x); columns = []
        for i, key in enumerate(keys):
            probe = x.copy(); step = max(FIT_STEPS[key], 4*np.spacing(max(1., abs(x[i]))))
            probe[i] += step if x[i] + step <= upper[i] else -step
            columns.append((evaluate(probe) - base) / (probe[i] - x[i]))
        return np.column_stack(columns)

    started = time.perf_counter(); start = np.asarray([initial[key] for key in keys], dtype=float)
    evaluate(start)
    result = least_squares(evaluate, start, bounds=(lower, upper), jac=jacobian,
                           x_scale="jac", max_nfev=int(max_nfev),
                           ftol=2e-4, xtol=1e-8, gtol=2e-4)
    fitted = validate_parameters(best["parameters"]); final_grid = np.linspace(-.5, .5, int(final_phases))
    x, flux = backend.curve(fitted, final_grid, final_triangles)
    comparison = compare_curve(x, flux, observations, fitted, len(keys))
    return {"parameters": fitted, "phases": x, "model": flux * comparison["normalization"],
            "comparison": comparison, "fit": {"parameters": list(keys), "evaluations": evaluations,
            "objective_phases": count, "objective_triangles": triangles,
            "converged": bool(result.success), "message": str(result.message)},
            "seconds": time.perf_counter() - started, "source": "native PHOEBE"}


def export_result(result, destination="phoebe_fit_results.zip"):
    if not result or result.get("source") != "native PHOEBE":
        raise ValueError("Run native PHOEBE compute or fit before exporting.")
    destination = Path(destination); comparison = result["comparison"]
    summary = {"source": result["source"], "parameters": result["parameters"],
               "fit": result["fit"], "seconds": result["seconds"],
               "statistics": {"rms": comparison["rms"],
                              "reduced_chi2": comparison["reduced_chi2"],
                              "normalization": comparison["normalization"]}}
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        model = io.StringIO(); np.savetxt(model, np.c_[result["phases"], result["model"]],
                                          delimiter=",", header="phase,normalized_flux", comments="")
        residual = io.StringIO(); columns = [comparison["phase"], comparison["flux"],
                                             comparison["model_at_data"], comparison["residual"]]
        header = "phase,observed_flux,model_flux,residual"
        if comparison["sigma"] is not None:
            columns.append(comparison["sigma"]); header += ",sigma"
        np.savetxt(residual, np.column_stack(columns), delimiter=",", header=header, comments="")
        archive.writestr("native_model.csv", model.getvalue())
        archive.writestr("residuals.csv", residual.getvalue())
        archive.writestr("fit_summary.json", json.dumps(summary, indent=2))
    return destination


class BinaryFitterApp:
    """ipywidgets surface. Callback methods are public enough for integration QA."""
    def __init__(self, models, backend=None, render_plots=True, export_directory=".", auto_surface=True):
        import ipywidgets as widgets
        self.widgets = widgets; self.models = models; self.backend = backend or NativeBackend()
        self.render_plots = render_plots; self.export_directory = Path(export_directory)
        self.observations = None; self.preview_result = None; self.exact_result = None
        self.surface_mesh = None; self.surface_key = None; self.auto_surface = auto_surface
        self._suspend = False; self._observers = []; self._mode_snapshots = {}
        self._build_widgets(); self._register_callbacks(); self._update_mode(); self.load_sample()
        self._mode_snapshots["detached"] = self._snapshot_controls()

    def _build_widgets(self):
        w = self.widgets
        self.geometry = w.ToggleButtons(options=[("Detached","detached"),("Semi-detached","semi"),("Contact","contact")], description="Geometry")
        self.donor = w.ToggleButtons(options=[("Primary donor","primary"),("Secondary donor","secondary")], description="Donor")
        def slider(description, key, step):
            low, high = BOUNDS[key]
            return w.FloatSlider(description=description, min=low, max=high, step=step,
                                 value=DEFAULTS[key], continuous_update=False, readout_format=".4g",
                                 style={"description_width":"150px"}, layout=w.Layout(width="510px"))
        self.controls = {
            "inclination": slider("Inclination (deg)", "inclination", .1),
            "radius_primary": slider("Primary radius / a", "radius_primary", .001),
            "radius_secondary": slider("Secondary radius / a", "radius_secondary", .001),
            "free_fill": slider("Free-star Roche fill", "free_fill", .005),
            "fillout_factor": slider("Contact fill-out", "fillout_factor", .005),
            "temperature_ratio": slider("T2 / T1", "temperature_ratio", .005),
            "mass_ratio": slider("q = M2 / M1", "mass_ratio", .005),
            "eccentricity": slider("Eccentricity", "eccentricity", .001),
            "omega": slider("Periastron angle", "omega", 1.),
            "period": slider("Period (days)", "period", .01),
        }
        self.epoch = w.FloatText(value=0., description="Epoch", style={"description_width":"150px"})
        self.distortion = w.Checkbox(value=False, description="Native detached Roche distortion")
        self.irradiation = w.Checkbox(value=False, description="Native irradiation (Horvat)")
        self.sample_choice = w.Dropdown(options=[("Mode-appropriate sample","auto"),("Grazing sample","grazing")], description="Sample")
        self.sample_button = w.Button(description="Load sample", button_style="info")
        self.upload = w.FileUpload(accept=".csv,text/csv", multiple=False, description="Upload CSV")
        self.preview_button = w.Button(description="Refresh trained preview")
        self.native_button = w.Button(description="Compute native curve", button_style="info")
        self.fit_button = w.Button(description="Run bounded native fit", button_style="primary")
        self.export_button = w.Button(description="Export native result", disabled=True)
        self.surface_build = w.Button(description="Build/rebuild native surface", button_style="info")
        self.surface_auto = w.Checkbox(value=self.auto_surface, description="Auto-build after constrained shape changes")
        self.surface_view = w.ToggleButtons(options=[("From Earth","earth"),("Above orbit","above")], value="earth", description="View")
        self.surface_phase = w.IntSlider(value=0,min=0,max=100,step=1,description="Orbit phase",continuous_update=False)
        self.surface_play = w.Play(value=0,min=0,max=100,step=1,interval=80,description="Play")
        self.surface_speed = w.Dropdown(options=[("16 sec/orbit",160),("8 sec/orbit",80),("4 sec/orbit",40),("2 sec/orbit",20)],value=80,description="Playback")
        self.surface_status = w.HTML(); self.surface_output = w.Output()
        self.surface_link = w.jslink((self.surface_play,"value"),(self.surface_phase,"value"))
        self.fit_phases = w.Dropdown(options=[("65 (fast)",65),("129 (standard)",129)], value=65, description="Fit phases")
        self.max_evaluations = w.Dropdown(options=[4,8,12], value=8, description="Max evals")
        descriptions = {"inclination":"Inclination", "radius_primary":"Primary radius", "radius_secondary":"Secondary radius",
                        "free_fill":"Free-star fill", "fillout_factor":"Contact fill-out", "temperature_ratio":"T2 / T1",
                        "mass_ratio":"Mass ratio q", "eccentricity":"Eccentricity", "omega":"Periastron angle", "period":"Period"}
        self.fit_checks = {key:w.Checkbox(value=key in ("inclination","temperature_ratio"), description=descriptions[key], indent=False)
                           for key in FIT_STEPS}
        self.status = w.HTML(); self.statistics = w.HTML(); self.plot_output = w.Output(); self.log_output = w.Output(); self.download_output = w.Output()
        self.parameter_box = w.VBox(list(self.controls.values()) + [self.epoch, self.distortion, self.irradiation])
        self.fit_box = w.VBox(list(self.fit_checks.values()))
        self.surface_box = w.VBox([w.HTML("<h3>Native PHOEBE orbit surface</h3><p>The phase-zero co-rotating mesh is cached and rotated with orbital phase. Contact uses one connected common envelope; semi-detached uses two tidal surfaces.</p>"),
                                   w.HBox([self.surface_build,self.surface_auto]),w.HBox([self.surface_view,self.surface_speed,self.surface_play]),
                                   self.surface_phase,self.surface_status,self.surface_output])
        self.root = w.VBox([
            w.HTML("<h2>PHOEBE binary fitting utility</h2><p><b>Trained previews are approximate and kernel-side.</b> Slider updates make a Colab/Jupyter kernel round trip. Native PHOEBE is authoritative for fitting, statistics, and exports.</p>"),
            self.geometry, self.donor, self.parameter_box,
            w.HBox([self.sample_choice,self.sample_button,self.upload]),
            w.HBox([self.preview_button,self.native_button]), self.status,
            self.plot_output, self.statistics,
            self.surface_box,
            w.HTML("<h3>Bounded native fit</h3><p>Select only identifiable parameters. Photometry alone may not uniquely constrain q; joint RV fitting is not implemented.</p>"),
            self.fit_box, w.HBox([self.fit_phases,self.max_evaluations,self.fit_button]), self.log_output,
            w.HBox([self.export_button]), self.download_output,
        ])

    def _observe(self, widget, handler, name="value"):
        widget.observe(handler, names=name); self._observers.append((widget, handler, name))

    def _register_callbacks(self):
        self._observe(self.geometry, self._geometry_changed); self._observe(self.donor, self._parameter_changed)
        for widget in self.controls.values(): self._observe(widget, self._parameter_changed)
        self._observe(self.epoch, self._parameter_changed); self._observe(self.distortion, self._parameter_changed); self._observe(self.irradiation, self._parameter_changed)
        self._observe(self.upload, self._upload_changed)
        self._observe(self.surface_phase, self._surface_phase_changed); self._observe(self.surface_view, self._surface_phase_changed)
        self._observe(self.surface_speed, self._surface_speed_changed)
        self.sample_button.on_click(self._load_sample_clicked); self.preview_button.on_click(self._preview_clicked)
        self.native_button.on_click(self._native_clicked); self.fit_button.on_click(self._fit_clicked); self.export_button.on_click(self._export_clicked)
        self.surface_build.on_click(self._surface_build_clicked)

    def close(self):
        for widget, handler, name in self._observers:
            widget.unobserve(handler, names=name)
        self._observers.clear()
        for button, handler in ((self.sample_button,self._load_sample_clicked),(self.preview_button,self._preview_clicked),
                                (self.native_button,self._native_clicked),(self.fit_button,self._fit_clicked),(self.export_button,self._export_clicked),
                                (self.surface_build,self._surface_build_clicked)):
            button.on_click(handler, remove=True)
        self.surface_link.unlink()
        self.root.close()

    def parameters(self):
        geometry = self.geometry.value
        if geometry == "semi": geometry = "semi_secondary" if self.donor.value == "secondary" else "semi_primary"
        p = dict(DEFAULTS, geometry=geometry, donor=self.donor.value, epoch=float(self.epoch.value),
                 distortion=self.distortion.value, irradiation=self.irradiation.value)
        p.update({key:float(widget.value) for key,widget in self.controls.items()})
        return validate_parameters(p)

    def _set_visible(self, key, visible): self.controls[key].layout.display = "" if visible else "none"

    def _update_mode(self):
        mode = self.geometry.value; semi = mode == "semi"; contact = mode == "contact"
        self.donor.layout.display = "" if semi else "none"
        self._set_visible("radius_primary", mode == "detached")
        self._set_visible("radius_secondary", mode == "detached")
        self._set_visible("free_fill", semi); self._set_visible("fillout_factor", contact)
        self._set_visible("eccentricity", mode == "detached"); self._set_visible("omega", mode == "detached")
        self.distortion.layout.display = "" if mode == "detached" else "none"
        self.surface_box.layout.display = "none" if mode == "detached" else ""
        allowed = set(allowed_fit_keys(self.parameters()["geometry"]))
        for key, checkbox in self.fit_checks.items(): checkbox.layout.display = "" if key in allowed else "none"

    def _snapshot_controls(self):
        return {"controls":{key:widget.value for key,widget in self.controls.items()},
                "epoch":self.epoch.value,"distortion":self.distortion.value,
                "irradiation":self.irradiation.value,"donor":self.donor.value}

    def _restore_controls(self, snapshot):
        self._suspend = True
        try:
            for key,value in snapshot["controls"].items(): self.controls[key].value = value
            self.epoch.value = snapshot["epoch"]; self.distortion.value = snapshot["distortion"]
            self.irradiation.value = snapshot["irradiation"]; self.donor.value = snapshot["donor"]
        finally: self._suspend = False

    def _invalidate_exact(self, reason="Parameters or observations changed"):
        self.exact_result = None; self.export_button.disabled = True; self.statistics.value = ""
        self.status.value = f"<span style='color:#805500'>{reason}; prior native statistics/export invalidated.</span>"

    def _geometry_changed(self, change):
        if self._suspend: return
        old, new = change.get("old"), change.get("new")
        if old in ("detached","semi","contact"): self._mode_snapshots[old] = self._snapshot_controls()
        if new in self._mode_snapshots:
            self._restore_controls(self._mode_snapshots[new])
        elif new != "detached":
            snapshot = self._snapshot_controls(); snapshot["controls"].update(eccentricity=0.,omega=90.,period=1.8)
            snapshot.update(distortion=False,irradiation=False)
            self._restore_controls(snapshot)
        self._update_mode(); self._invalidate_exact("Geometry changed"); self.refresh_preview(); self._surface_parameters_changed()

    def _parameter_changed(self, change):
        if self._suspend: return
        self._invalidate_exact(); self.refresh_preview(); self._surface_parameters_changed()

    def _surface_geometry_key(self):
        p=self.parameters();geometry=p["geometry"]
        if geometry=="detached":return None
        return (geometry,p["donor"],p["mass_ratio"],p["fillout_factor"] if geometry=="contact" else p["free_fill"])

    def _surface_parameters_changed(self):
        key=self._surface_geometry_key()
        if key is None:
            self.surface_mesh=None;self.surface_key=None;self.surface_status.value="";return
        if key!=self.surface_key:
            self.surface_mesh=None;self.surface_key=key
            self.surface_status.value="<span style='color:#805500'>Native surface is stale.</span>"
            if self.surface_auto.value:self.build_surface()
        else:self._draw_surface()

    def _surface_build_clicked(self,button):self.build_surface()

    def build_surface(self):
        if self.parameters()["geometry"]=="detached":return None
        if not hasattr(self.backend,"surface"):
            self.surface_status.value="<span style='color:#a00000'>This backend does not provide native surfaces.</span>";return None
        self.surface_build.disabled=True;self.surface_status.value="<b>Building native PHOEBE phase-zero surface…</b>"
        try:
            mesh=self.backend.surface(self.parameters(),500);key=self._surface_geometry_key()
            self.surface_mesh=mesh;self.surface_key=key
            detail="connected common envelope with native neck" if mesh["geometry"]=="contact" else f"two tidal surfaces · {mesh['donor']} donor fills its Roche lobe"
            self.surface_status.value=f"<b>{mesh['source']}</b> · {detail} · {mesh['triangles_actual']}-triangle mesh target"
            self._draw_surface();return mesh
        except Exception as error:
            self.surface_mesh=None;self.surface_status.value=f"<span style='color:#a00000'><b>Surface error:</b> {error}</span>";return None
        finally:self.surface_build.disabled=False

    def _surface_speed_changed(self,change):self.surface_play.interval=int(change["new"])

    def _surface_phase_changed(self,change):
        self._draw_surface()
        current=self.exact_result or self.preview_result
        if current is not None:self._draw(current)

    def _draw_surface(self):
        if not self.render_plots or self.surface_mesh is None:return
        import matplotlib.pyplot as plt
        from matplotlib.collections import PolyCollection
        from IPython.display import display
        p=self.parameters();phase=self.surface_phase.value/100;view=self.surface_view.value
        triangles=project_surface(self.surface_mesh,p,phase,view)
        polygons=[entry["points"][:,:2] for entry in triangles]
        colors=["#db984b" if entry["component"]==0 else "#8876d0" for entry in triangles]
        with self.surface_output:
            self.surface_output.clear_output(wait=True)
            fig,ax=plt.subplots(figsize=(8.8,4.2));collection=PolyCollection(polygons,facecolors=colors,edgecolors="#5d536f55",linewidths=.25,alpha=.84)
            ax.add_collection(collection);ax.autoscale_view();ax.set_aspect("equal",adjustable="datalim");ax.margins(.08)
            ax.scatter([0],[0],marker="s",s=18,color="#5d536f",label="center of mass")
            if view=="above" and p["inclination"]>1e-9:
                ax.invert_yaxis()
                ax.annotate("Toward Earth",xy=(.5,.04),xytext=(.5,.18),xycoords="axes fraction",ha="center",arrowprops={"arrowstyle":"->"})
            title="Connected native PHOEBE contact envelope · neck visible" if self.surface_mesh["geometry"]=="contact" else f"Native PHOEBE semi-detached surfaces · {p['donor']} donor"
            ax.set_title(title);ax.set_xlabel("Projected separation");ax.set_ylabel("Projected separation");ax.text(.01,.01,f"phase {phase:.2f}",transform=ax.transAxes)
            display(fig);plt.close(fig)

    def _upload_changed(self, change):
        value = change.get("new", self.upload.value)
        if not value: return
        try:
            item = next(iter(value.values())) if isinstance(value, dict) else value[0]
            content = item["content"]; content = content.tobytes() if hasattr(content,"tobytes") else bytes(content)
            self.load_csv_bytes(content, item.get("name", "uploaded.csv"))
        except Exception as error:
            self._set_error(error)

    def load_csv_bytes(self, content, name="uploaded.csv"):
        self.observations = parse_csv_bytes(content, name); self._invalidate_exact("New observations loaded")
        self.status.value = f"<b>Loaded {len(self.observations.x)} rows from {name}.</b> Preview statistics are approximate until native compute/fit."
        self.refresh_preview(); return self.observations

    def load_sample(self):
        p = self.parameters()
        if self.sample_choice.value == "grazing": p["inclination"] = 68.
        try: phase, flux = self.models.curve(p)
        except ValueError:
            p.update(period=1.8, eccentricity=0., omega=90., irradiation=False, distortion=False, mass_ratio=.75)
            phase, flux = self.models.curve(p)
        take = np.linspace(0, len(phase)-2, 120, dtype=int); rng = np.random.default_rng(271828)
        sigma = np.full(len(take), .0015); observed = flux[take] + rng.normal(0, sigma)
        self.observations = ObservationSet(phase[take], observed, sigma, "phase", "flux", "built-in surrogate sample")
        self._invalidate_exact("Sample loaded"); self.refresh_preview(); return self.observations

    def _load_sample_clicked(self, button): self.load_sample()
    def _preview_clicked(self, button): self.refresh_preview()

    def refresh_preview(self):
        try:
            phase, flux = self.models.curve(self.parameters())
            comparison = None if self.observations is None else compare_curve(phase, flux, self.observations, self.parameters())
            self.preview_result = {"phases":phase,"model":flux,"comparison":comparison,"source":"trained preview"}
            self.status.value = "<b>Approximate trained preview.</b> Run native PHOEBE for authoritative statistics/export."
            self._draw(self.preview_result)
            return self.preview_result
        except Exception as error:
            self.preview_result = None; self.status.value = f"<span style='color:#805500'>Preview unavailable: {error}</span>"
            return None

    def _set_busy(self, message):
        self.status.value = f"<b>{message}</b>"; self.native_button.disabled = self.fit_button.disabled = True

    def _set_error(self, error):
        self.status.value = f"<span style='color:#a00000'><b>Error:</b> {error}</span>"

    def _native_clicked(self, button):
        if self.observations is None: self._set_error("Load observations first."); return
        self._set_busy("Running authoritative native PHOEBE compute…")
        try:
            result = run_native_compute(self.backend, self.observations, self.parameters())
            self._accept_exact(result)
        except Exception as error: self._set_error(error)
        finally: self.native_button.disabled = self.fit_button.disabled = False

    def selected_fit_keys(self):
        allowed = set(allowed_fit_keys(self.parameters()["geometry"]))
        return [key for key,checkbox in self.fit_checks.items() if checkbox.value and key in allowed]

    def _fit_clicked(self, button):
        if self.observations is None: self._set_error("Load observations first."); return
        keys = self.selected_fit_keys()
        if not keys: self._set_error("Select at least one supported fit parameter."); return
        self._set_busy("Running bounded native PHOEBE fit…")
        def progress(evaluation, rms, parameters):
            self.status.value = f"<b>Native fit evaluation {evaluation}</b> · current RMS {rms:.6g}"
        try:
            result = run_native_fit(self.backend, self.observations, self.parameters(), keys,
                                    self.fit_phases.value, self.max_evaluations.value, progress=progress)
            self._suspend = True
            try:
                for key, widget in self.controls.items(): widget.value = result["parameters"][key]
            finally: self._suspend = False
            self._accept_exact(result)
        except Exception as error: self._set_error(error)
        finally: self.native_button.disabled = self.fit_button.disabled = False

    def _accept_exact(self, result):
        self.exact_result = result; self.export_button.disabled = False; comparison = result["comparison"]
        reduced = "—" if comparison["reduced_chi2"] is None else f"{comparison['reduced_chi2']:.5g}"
        fit = result.get("fit"); search = "native compute" if fit is None else f"{fit['objective_phases']}-phase fit ({fit['evaluations']} evaluations)"
        self.status.value = f"<b>Authoritative native PHOEBE complete.</b> {search}; final curve: 513 phases, 1,500 triangles."
        self.statistics.value = f"<b>RMS:</b> {comparison['rms']:.7g} &nbsp; <b>Reduced χ²:</b> {reduced} &nbsp; <b>N:</b> {len(comparison['phase'])}"
        self._draw(result)

    def _export_clicked(self, button):
        try:
            path = export_result(self.exact_result, self.export_directory / "phoebe_fit_results.zip")
            with self.download_output:
                self.download_output.clear_output(wait=True)
                try:
                    from google.colab import files
                    files.download(str(path))
                except ImportError:
                    from IPython.display import FileLink, display
                    display(FileLink(str(path)))
        except Exception as error: self._set_error(error)

    def _draw(self, result):
        if not self.render_plots: return
        import matplotlib.pyplot as plt
        from IPython.display import display
        with self.plot_output:
            self.plot_output.clear_output(wait=True)
            if self.observations is None:
                fig, ax = plt.subplots(figsize=(9,3.2)); ax.plot(result["phases"],result["model"]); ax.set(xlabel="Orbital phase",ylabel="Normalized flux")
            else:
                comp = result["comparison"]
                if comp is None:
                    comp = compare_curve(result["phases"],result["model"],self.observations,self.parameters())
                curve = result["model"] if result.get("source") == "native PHOEBE" else result["model"] * comp["normalization"]
                fig,(ax,res)=plt.subplots(2,1,figsize=(9,5.3),sharex=True,gridspec_kw={"height_ratios":[3,1]})
                ax.errorbar(comp["phase"],comp["flux"],yerr=comp["sigma"],fmt=".",alpha=.65,label="observed")
                ax.plot(result["phases"],curve,label=result["source"]); ax.set_ylabel("Normalized flux"); ax.legend()
                marker=(self.surface_phase.value/100+.5)%1-.5;ax.axvline(marker,color="#97571a",lw=1,alpha=.7)
                res.axhline(0,color=".5",lw=.8); res.plot(comp["phase"],comp["residual"],"."); res.set(xlabel="Orbital phase",ylabel="Residual")
            plt.tight_layout(); display(fig); plt.close(fig)


def launch_fitter(models, backend=None, render_plots=True, export_directory=".", auto_surface=True):
    """Idempotent notebook entry point: re-running the cell closes old callbacks."""
    global _PHOEBE_FITTER_APP
    try:
        _PHOEBE_FITTER_APP.close()
    except (NameError, AttributeError):
        pass
    _PHOEBE_FITTER_APP = BinaryFitterApp(models, backend, render_plots, export_directory, auto_surface)
    try:
        from IPython.display import display
        display(_PHOEBE_FITTER_APP.root)
    except ImportError:
        pass
    return _PHOEBE_FITTER_APP


In [ ]:
MODELS = PreviewModels(DETACHED_MODEL, SEMI_MODEL, CONTACT_MODEL)
print("Embedded preview artifacts:", DETACHED_MODEL["label"], SEMI_MODEL["name"], CONTACT_MODEL["name"])


In [ ]:
# Re-run this cell after a notebook restart or whenever you want a clean interface.
app = launch_fitter(MODELS)


## Reading the output

The plot is marked as a trained preview until an explicit native action completes. After a native compute or fit, changing any parameter, donor, geometry, or observation file invalidates the statistics and disables export. Export produces a ZIP containing the authoritative 513-phase native curve, model-at-data residual table, and JSON fit summary.

For model training reproduction, independent holdouts, boundary sweeps, and response-direction validation, see `PHOEBE_Semi_Detached_Validation.ipynb` and the committed experiment reports.